In [22]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abinavprabhuv/dasonet-rsc-external-test-assets-v1/RSC_EXTERNAL_TEST_ASSETS_STATE_v1.json
/kaggle/input/datasets/abinavprabhuv/dasonet-rsc-external-test-assets-v1/DASO-Net-RSC-External-TestAssets-v1.sha256.txt
/kaggle/input/datasets/abinavprabhuv/dasonet-rsc-external-test-assets-v1/DASO-Net-RSC-External-TestAssets-v1/DASO-Net-RSC-External-TestAssets-v1/RSC_EXACT_DEDUP247_MANIFEST_v1.csv
/kaggle/input/datasets/abinavprabhuv/dasonet-rsc-external-test-assets-v1/DASO-Net-RSC-External-TestAssets-v1/DASO-Net-RSC-External-TestAssets-v1/README_RSC_EXTERNAL_TEST_ASSETS.txt
/kaggle/input/datasets/abinavprabhuv/dasonet-rsc-external-test-assets-v1/DASO-Net-RSC-External-TestAssets-v1/DASO-Net-RSC-External-TestAssets-v1/RSC_OFFICIAL_TEST250_MANIFEST_v1.csv
/kaggle/input/datasets/abinavprabhuv/dasonet-rsc-external-test-assets-v1/DASO-Net-RSC-External-TestAssets-v1/DASO-Net-RSC-External-TestAssets-v1/RSC_EXTERNAL_TEST_FILE_MANIFEST_v1.json
/kaggle/input/datasets/abinavprabhuv/das

In [11]:
# ============================================================
# DASO-Net — AFTER-SEED3 RESTORE + THREE-SEED INTEGRITY AUDIT
#
# NEW FRESH KAGGLE NOTEBOOK
#
# PURPOSE
# -------
# Restore the final three-seed handoff and verify:
#
#   - handoff package provenance
#   - every handoff-manifest file SHA256
#   - Seed 1 terminal checkpoint
#   - Seed 2 terminal checkpoint
#   - Seed 3 terminal checkpoint
#   - Seed 1/2/3 QC artifacts
#   - frozen multi-seed plan
#   - validated decoder
#   - frozen project assets
#   - all three seeds = exactly 2750 successful updates
#   - no best-seed selection
#
# NO TRAINING.
# NO INFERENCE.
# NO MODEL SELECTION.
# ============================================================

from __future__ import annotations

import hashlib
import json
import shutil
from pathlib import Path


# ============================================================
# 1. FROZEN IDENTITIES
# ============================================================

PACKAGE_NAME = (
    "DASO-Net-AFTER-SEED3-EOD-20260918"
)

EXPECTED_PACKAGE_SHA = (
    "21aca1cc30f70c3229f8f9c357e4e8eb"
    "f7e711804ffefacae325542b269c610c"
)

SEED1 = 1782656963
SEED2 = 901395089
SEED3 = 252916394

TARGET_UPDATES = 2750

EXPECTED_SEED1_SHA = (
    "e4c4b79ad4b8c6e18ba2b1e3d6cbe796"
    "c9cc301bd6fcc7c596fc290ad3a7e81d"
)

EXPECTED_SEED2_SHA = (
    "19cdb834a46240916b2c8023e8c3afe573"
    "8d9c7cd5d8f818c62ad1e289df1ad2"
)

EXPECTED_SEED3_SHA = (
    "be7b7941f2f62179e15074aa973bf6da0"
    "eb9018ec9133486512bd1f87153f447"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)


# ============================================================
# 2. INPUT ROOTS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

HANDOFF_DATASET = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-after-seed3-eod"
)

OUTER_STATE = (
    HANDOFF_DATASET
    / f"{PACKAGE_NAME}-STATE.json"
)

SHA_RECORD = (
    HANDOFF_DATASET
    / f"{PACKAGE_NAME}.sha256.txt"
)

WORKING = Path(
    "/kaggle/working"
)

RESTORE_AUDIT = (
    WORKING
    / "DASO-Net-AFTER-SEED3-RESTORE-AUDIT-v1.json"
)


# ============================================================
# 3. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(
    path: Path,
    obj,
) -> None:

    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def copy_fresh(
    src: Path,
    dst: Path,
) -> None:

    if dst.exists():

        raise RuntimeError(
            "Fresh restore destination already exists:\n"
            f"{dst}\n\n"
            "STOP rather than overwrite existing state."
        )

    if src.is_dir():

        shutil.copytree(
            src,
            dst,
        )

    else:

        dst.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            src,
            dst,
        )


def optional_pass_status(
    obj: dict,
    label: str,
) -> None:
    """
    Some older Seed-1 artifacts predate the final PASS schema.
    If a status/sanity field exists, it must be valid.
    """

    if "status" in obj:

        if obj["status"] != "PASS":

            raise RuntimeError(
                f"{label}: status is not PASS."
            )

    if "sanity_flags" in obj:

        flags = obj[
            "sanity_flags"
        ]

        if not all(
            bool(v)
            for v in flags.values()
        ):

            raise RuntimeError(
                f"{label}: sanity flag failure."
            )


# ============================================================
# 4. LOCATE AUTO-EXTRACTED HANDOFF
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — AFTER-SEED3 RESTORE + INTEGRITY AUDIT"
)

print(
    "============================================================"
)


require(
    PROJECT_ROOT,
    "Frozen project dataset",
)

require(
    HANDOFF_DATASET,
    "After-Seed3 Kaggle dataset",
)

require(
    OUTER_STATE,
    "Outer handoff state",
)

require(
    SHA_RECORD,
    "Package SHA record",
)


manifest_candidates = list(
    HANDOFF_DATASET.rglob(
        "HANDOFF_MANIFEST.json"
    )
)


if len(
    manifest_candidates
) != 1:

    raise RuntimeError(
        "Expected exactly one HANDOFF_MANIFEST.json; "
        f"found {len(manifest_candidates)}.\n"
        + "\n".join(
            str(p)
            for p in manifest_candidates
        )
    )


HANDOFF_MANIFEST = (
    manifest_candidates[0]
)

HANDOFF_ROOT = (
    HANDOFF_MANIFEST.parent
)

SOURCE_WORKSPACE = (
    HANDOFF_ROOT
    / "workspace"
)

INNER_STATE = (
    HANDOFF_ROOT
    / "AFTER_SEED3_STATE.json"
)

README = (
    HANDOFF_ROOT
    / "README_HANDOFF.txt"
)

RESTORE_SCRIPT = (
    HANDOFF_ROOT
    / "RESTORE_AFTER_SEED3.py"
)


for path, label in [
    (
        HANDOFF_ROOT,
        "Auto-extracted handoff root",
    ),
    (
        HANDOFF_MANIFEST,
        "Handoff manifest",
    ),
    (
        INNER_STATE,
        "Internal After-Seed3 state",
    ),
    (
        README,
        "Handoff README",
    ),
    (
        RESTORE_SCRIPT,
        "Restore script",
    ),
    (
        SOURCE_WORKSPACE,
        "Handoff workspace",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 5. OUTER PACKAGE PROVENANCE
# ============================================================

outer = load_json(
    OUTER_STATE
)

inner = load_json(
    INNER_STATE
)


if (
    outer[
        "package_sha256"
    ]
    != EXPECTED_PACKAGE_SHA
):

    raise RuntimeError(
        "Outer-state package SHA mismatch."
    )


sha_text = SHA_RECORD.read_text(
    encoding="utf-8"
).strip()


if EXPECTED_PACKAGE_SHA not in sha_text:

    raise RuntimeError(
        "SHA record does not contain expected "
        "After-Seed3 package SHA."
    )


if not bool(
    outer[
        "three_final_seeds_complete"
    ]
):

    raise RuntimeError(
        "Outer state does not declare all three "
        "final seeds complete."
    )


if bool(
    outer[
        "training_remaining"
    ]
):

    raise RuntimeError(
        "Outer state unexpectedly says training remains."
    )


if bool(
    outer[
        "best_seed_selection"
    ]
):

    raise RuntimeError(
        "Outer state unexpectedly indicates best-seed selection."
    )


if (
    outer[
        "seed1_terminal_sha256"
    ]
    != EXPECTED_SEED1_SHA
):

    raise RuntimeError(
        "Outer Seed-1 SHA mismatch."
    )


if (
    outer[
        "seed2_terminal_sha256"
    ]
    != EXPECTED_SEED2_SHA
):

    raise RuntimeError(
        "Outer Seed-2 SHA mismatch."
    )


if (
    outer[
        "seed3_terminal_sha256"
    ]
    != EXPECTED_SEED3_SHA
):

    raise RuntimeError(
        "Outer Seed-3 SHA mismatch."
    )


if (
    outer[
        "decoder_sha256"
    ]
    != EXPECTED_DECODER_SHA
):

    raise RuntimeError(
        "Outer decoder SHA mismatch."
    )


print()
print(
    "Original package SHA provenance: PASS"
)

print(
    "Three-final-seed completion state: PASS"
)

print(
    "No remaining final-seed training: PASS"
)

print(
    "No best-seed selection: PASS"
)


# ============================================================
# 6. INTERNAL STATE AUDIT
# ============================================================

if (
    inner[
        "final_multiseed_training"
    ]
    != "COMPLETE"
):

    raise RuntimeError(
        "Internal state does not mark training COMPLETE."
    )


if int(
    inner[
        "final_seed_count"
    ]
) != 3:

    raise RuntimeError(
        "Internal state final_seed_count != 3."
    )


expected_seed_map = {
    1:
        SEED1,

    2:
        SEED2,

    3:
        SEED3,
}


seen = set()


for record in inner[
    "seeds"
]:

    idx = int(
        record[
            "seed_index"
        ]
    )

    rng = int(
        record[
            "rng_seed"
        ]
    )

    updates = int(
        record[
            "successful_optimizer_updates"
        ]
    )


    if idx not in expected_seed_map:

        raise RuntimeError(
            f"Unexpected seed index: {idx}"
        )


    if rng != expected_seed_map[
        idx
    ]:

        raise RuntimeError(
            f"Seed-{idx} RNG mismatch."
        )


    if updates != TARGET_UPDATES:

        raise RuntimeError(
            f"Seed-{idx} does not have exactly 2750 "
            "successful optimizer updates."
        )


    if record[
        "status"
    ] != "COMPLETE":

        raise RuntimeError(
            f"Seed-{idx} status is not COMPLETE."
        )


    if record[
        "qc"
    ] != "PASS":

        raise RuntimeError(
            f"Seed-{idx} QC is not PASS."
        )


    seen.add(
        idx
    )


if seen != {
    1,
    2,
    3,
}:

    raise RuntimeError(
        "Internal three-seed state is incomplete."
    )


print(
    "Internal three-seed state: PASS"
)


# ============================================================
# 7. FULL FILE-BY-FILE MANIFEST SHA AUDIT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "HANDOFF FILE-BY-FILE SHA256 AUDIT"
)

print(
    "============================================================"
)


manifest = load_json(
    HANDOFF_MANIFEST
)

records = manifest[
    "files"
]


if int(
    manifest[
        "file_count"
    ]
) != len(
    records
):

    raise RuntimeError(
        "Manifest file count mismatch."
    )


verified = 0


for record in records:

    relative_path = record[
        "relative_path"
    ]

    path = (
        HANDOFF_ROOT
        / relative_path
    )


    require(
        path,
        f"Manifest entry {relative_path}",
    )


    expected_size = int(
        record[
            "size_bytes"
        ]
    )

    actual_size = (
        path.stat().st_size
    )


    if actual_size != expected_size:

        raise RuntimeError(
            "Manifest size mismatch:\n"
            f"{relative_path}\n"
            f"Expected: {expected_size}\n"
            f"Actual:   {actual_size}"
        )


    expected_sha = record[
        "sha256"
    ]

    actual_sha = sha256_file(
        path
    )


    if actual_sha != expected_sha:

        raise RuntimeError(
            "Manifest SHA mismatch:\n"
            f"{relative_path}\n"
            f"Expected: {expected_sha}\n"
            f"Actual:   {actual_sha}"
        )


    verified += 1


print(
    f"Manifest entries verified: {verified}"
)

print(
    "File-by-file SHA256 audit: PASS"
)


# ============================================================
# 8. SOURCE WORKSPACE CRITICAL PATHS
# ============================================================

SRC_CODE = (
    SOURCE_WORKSPACE
    / "DASO-Net-Phase5B-KaggleTraining"
)

SRC_MULTISEED = (
    SOURCE_WORKSPACE
    / "DASO-Net-FinalMultiSeed-v1"
)

SRC_FREEZE = (
    SOURCE_WORKSPACE
    / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1"
)

SRC_PLAN = (
    SRC_MULTISEED
    / "FINAL_MULTISEED_PLAN_v1.json"
)

SRC_DECODER = (
    SRC_CODE
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)


SRC_SEED1_RUN = (
    SOURCE_WORKSPACE
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
)

SRC_SEED2_RUN = (
    SOURCE_WORKSPACE
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
)

SRC_SEED3_RUN = (
    SOURCE_WORKSPACE
    / "dasonet_phase5b"
    / "dasonet_final_seed03_s252916394"
)


SRC_SEED1_TERMINAL = (
    SRC_SEED1_RUN
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

SRC_SEED2_TERMINAL = (
    SRC_SEED2_RUN
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

SRC_SEED3_TERMINAL = (
    SRC_SEED3_RUN
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)


SRC_SEED1_QC = (
    SOURCE_WORKSPACE
    / "dasonet_final_qc"
    / "seed01_s1782656963_val1000_v1"
)

SRC_SEED2_QC = (
    SOURCE_WORKSPACE
    / "dasonet_final_qc"
    / "seed02_s901395089_val1000_v3_fixedop005"
)

SRC_SEED3_QC = (
    SOURCE_WORKSPACE
    / "dasonet_final_qc"
    / "seed03_s252916394_val1000_v1_fixedop005"
)


SRC_SEED1_QC_SUMMARY = (
    SRC_SEED1_QC
    / "FINAL_SEED01_QC_SUMMARY.json"
)

SRC_SEED2_QC_SUMMARY = (
    SRC_SEED2_QC
    / "FINAL_SEED02_QC_SUMMARY.json"
)

SRC_SEED3_QC_SUMMARY = (
    SRC_SEED3_QC
    / "FINAL_SEED03_QC_SUMMARY.json"
)


critical = [
    (
        SRC_CODE,
        "Validated code tree",
    ),
    (
        SRC_MULTISEED,
        "Frozen multi-seed plan directory",
    ),
    (
        SRC_FREEZE,
        "Frozen final-training recipe",
    ),
    (
        SRC_PLAN,
        "Frozen multi-seed plan JSON",
    ),
    (
        SRC_DECODER,
        "Validated decoder",
    ),

    (
        SRC_SEED1_TERMINAL,
        "Seed-1 terminal",
    ),
    (
        SRC_SEED2_TERMINAL,
        "Seed-2 terminal",
    ),
    (
        SRC_SEED3_TERMINAL,
        "Seed-3 terminal",
    ),

    (
        SRC_SEED1_QC_SUMMARY,
        "Seed-1 QC summary",
    ),
    (
        SRC_SEED2_QC_SUMMARY,
        "Seed-2 QC summary",
    ),
    (
        SRC_SEED3_QC_SUMMARY,
        "Seed-3 QC summary",
    ),
]


for path, label in critical:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 9. TERMINAL CHECKPOINT HASHES
# ============================================================

terminal_checks = [
    (
        SRC_SEED1_TERMINAL,
        EXPECTED_SEED1_SHA,
        "Seed-1",
    ),
    (
        SRC_SEED2_TERMINAL,
        EXPECTED_SEED2_SHA,
        "Seed-2",
    ),
    (
        SRC_SEED3_TERMINAL,
        EXPECTED_SEED3_SHA,
        "Seed-3",
    ),
]


for path, expected, label in terminal_checks:

    actual = sha256_file(
        path
    )


    if actual != expected:

        raise RuntimeError(
            f"{label} terminal SHA mismatch.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


    print(
        f"{label} terminal SHA: PASS"
    )


if (
    sha256_file(
        SRC_DECODER
    )
    != EXPECTED_DECODER_SHA
):

    raise RuntimeError(
        "Validated decoder SHA mismatch."
    )


print(
    "Validated decoder SHA: PASS"
)


# ============================================================
# 10. QC SUMMARY AUDIT
# ============================================================

seed1_qc = load_json(
    SRC_SEED1_QC_SUMMARY
)

seed2_qc = load_json(
    SRC_SEED2_QC_SUMMARY
)

seed3_qc = load_json(
    SRC_SEED3_QC_SUMMARY
)


optional_pass_status(
    seed1_qc,
    "Seed-1 QC",
)

optional_pass_status(
    seed2_qc,
    "Seed-2 QC",
)

optional_pass_status(
    seed3_qc,
    "Seed-3 QC",
)


if (
    "status"
    in seed2_qc
    and seed2_qc[
        "status"
    ]
    != "PASS"
):

    raise RuntimeError(
        "Seed-2 final QC status failure."
    )


if (
    "status"
    in seed3_qc
    and seed3_qc[
        "status"
    ]
    != "PASS"
):

    raise RuntimeError(
        "Seed-3 final QC status failure."
    )


if (
    sha256_file(
        SRC_SEED2_QC_SUMMARY
    )
    != outer[
        "seed2_qc_summary_sha256"
    ]
):

    raise RuntimeError(
        "Seed-2 QC summary SHA does not match outer state."
    )


if (
    sha256_file(
        SRC_SEED3_QC_SUMMARY
    )
    != outer[
        "seed3_qc_summary_sha256"
    ]
):

    raise RuntimeError(
        "Seed-3 QC summary SHA does not match outer state."
    )


print()
print(
    "Seed-1 QC artifact: PASS"
)

print(
    "Seed-2 final QC: PASS"
)

print(
    "Seed-3 final QC: PASS"
)


# ============================================================
# 11. FROZEN MULTI-SEED PLAN AUDIT
# ============================================================

plan = load_json(
    SRC_PLAN
)


if int(
    plan[
        "frozen_training_recipe"
    ][
        "optimizer_updates"
    ]
) != TARGET_UPDATES:

    raise RuntimeError(
        "Frozen plan target is not 2750 updates."
    )


expected = {
    1:
        (
            SEED1,
            "dasonet_final_seed01_s1782656963",
        ),

    2:
        (
            SEED2,
            "dasonet_final_seed02_s901395089",
        ),

    3:
        (
            SEED3,
            "dasonet_final_seed03_s252916394",
        ),
}


plan_seen = set()


for item in plan[
    "final_seed_configs"
]:

    idx = int(
        item[
            "seed_index"
        ]
    )


    if idx not in expected:
        continue


    rng_expected, exp_expected = (
        expected[
            idx
        ]
    )


    if int(
        item[
            "seed"
        ]
    ) != rng_expected:

        raise RuntimeError(
            f"Frozen plan Seed-{idx} RNG mismatch."
        )


    if (
        item[
            "experiment_name"
        ]
        != exp_expected
    ):

        raise RuntimeError(
            f"Frozen plan Seed-{idx} experiment mismatch."
        )


    plan_seen.add(
        idx
    )


if plan_seen != {
    1,
    2,
    3,
}:

    raise RuntimeError(
        "Frozen multi-seed plan does not contain all three "
        "final seeds."
    )


if (
    sha256_file(
        SRC_PLAN
    )
    != outer[
        "multiseed_plan_sha256"
    ]
):

    raise RuntimeError(
        "Frozen multi-seed plan SHA mismatch."
    )


print()
print(
    "Frozen 2750-update recipe: PASS"
)

print(
    "Frozen Seed-1/2/3 identities: PASS"
)

print(
    "Frozen multi-seed plan SHA: PASS"
)


# ============================================================
# 12. FROZEN PROJECT-DATA PRESENCE
# ============================================================

project_required = [
    (
        PROJECT_ROOT
        / "configs"
        / "renderer_v3_1.json",
        "Renderer config",
    ),

    (
        PROJECT_ROOT
        / "src"
        / "dasonet"
        / "synth"
        / "renderer.py",
        "Renderer source",
    ),

    (
        PROJECT_ROOT
        / "data"
        / "mp_morph"
        / "v1"
        / "morphology_manifest_v1.csv",
        "MP morphology manifest",
    ),

    (
        PROJECT_ROOT
        / "data"
        / "fruit_bg"
        / "v1_final"
        / "fruitbg_rgb_v1_manifest.csv",
        "FruitBG manifest",
    ),

    (
        PROJECT_ROOT
        / "data"
        / "fruitmp_syn"
        / "v1_1_recipes"
        / "fruitmp_syn_v1_1_full_freeze_manifest.json",
        "FruitMP-Syn freeze manifest",
    ),

    (
        PROJECT_ROOT
        / "data"
        / "fruitmp_syn"
        / "v1_1_recipes"
        / "fruitmp_syn_v1_1_train_recipes.csv",
        "FruitMP-Syn train recipes",
    ),

    (
        PROJECT_ROOT
        / "data"
        / "fruitmp_syn"
        / "v1_1_recipes"
        / "fruitmp_syn_v1_1_val_recipes.csv",
        "FruitMP-Syn val recipes",
    ),

    (
        PROJECT_ROOT
        / "data"
        / "fruitmp_syn"
        / "v1_1_recipes"
        / "fruitmp_syn_v1_1_test_recipes.csv",
        "Untouched FruitMP-Syn test recipes",
    ),

    (
        PROJECT_ROOT
        / "data"
        / "public_real_mp"
        / "v1_processed"
        / "public_real_mp_v1_source_specific_manifest.csv",
        "Public-real source manifest",
    ),

    (
        PROJECT_ROOT
        / "data"
        / "public_real_mp"
        / "v1_processed"
        / "almeida_crpa_split_v1_candidate.csv",
        "Almeida CRPA manifest",
    ),
]


for path, label in project_required:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


print()
print(
    "Frozen project data required for immediate formal stage: PASS"
)


# ============================================================
# 13. ENSURE FRESH /kaggle/working RESTORE TARGET
# ============================================================

workspace_entries = sorted(
    SOURCE_WORKSPACE.iterdir(),
    key=lambda p: p.name,
)


conflicts = []


for src in workspace_entries:

    dst = (
        WORKING
        / src.name
    )

    if dst.exists():

        conflicts.append(
            dst
        )


if conflicts:

    raise RuntimeError(
        "Fresh restore cannot proceed because these "
        "/kaggle/working destinations already exist:\n"
        + "\n".join(
            str(p)
            for p in conflicts
        )
        + "\n\nRestart with a fresh notebook/session rather "
          "than deleting scientific state manually."
    )


print()
print(
    "Fresh /kaggle/working restore target: PASS"
)


# ============================================================
# 14. RESTORE WORKSPACE
# ============================================================

print()
print(
    "============================================================"
)

print(
    "RESTORING FINAL THREE-SEED WORKSPACE"
)

print(
    "============================================================"
)


for src in workspace_entries:

    dst = (
        WORKING
        / src.name
    )


    copy_fresh(
        src,
        dst,
    )


    print(
        f"PASS  restored {src.name}"
    )


# ============================================================
# 15. POST-RESTORE PATHS
# ============================================================

DST_CODE = (
    WORKING
    / "DASO-Net-Phase5B-KaggleTraining"
)

DST_PLAN = (
    WORKING
    / "DASO-Net-FinalMultiSeed-v1"
    / "FINAL_MULTISEED_PLAN_v1.json"
)

DST_DECODER = (
    DST_CODE
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

DST_SEED1_TERMINAL = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

DST_SEED2_TERMINAL = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

DST_SEED3_TERMINAL = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed03_s252916394"
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

DST_SEED1_QC = (
    WORKING
    / "dasonet_final_qc"
    / "seed01_s1782656963_val1000_v1"
    / "FINAL_SEED01_QC_SUMMARY.json"
)

DST_SEED2_QC = (
    WORKING
    / "dasonet_final_qc"
    / "seed02_s901395089_val1000_v3_fixedop005"
    / "FINAL_SEED02_QC_SUMMARY.json"
)

DST_SEED3_QC = (
    WORKING
    / "dasonet_final_qc"
    / "seed03_s252916394_val1000_v1_fixedop005"
    / "FINAL_SEED03_QC_SUMMARY.json"
)


for path, label in [
    (
        DST_CODE,
        "Restored validated code",
    ),
    (
        DST_PLAN,
        "Restored frozen plan",
    ),
    (
        DST_DECODER,
        "Restored decoder",
    ),
    (
        DST_SEED1_TERMINAL,
        "Restored Seed-1 terminal",
    ),
    (
        DST_SEED2_TERMINAL,
        "Restored Seed-2 terminal",
    ),
    (
        DST_SEED3_TERMINAL,
        "Restored Seed-3 terminal",
    ),
    (
        DST_SEED1_QC,
        "Restored Seed-1 QC",
    ),
    (
        DST_SEED2_QC,
        "Restored Seed-2 QC",
    ),
    (
        DST_SEED3_QC,
        "Restored Seed-3 QC",
    ),
]:

    require(
        path,
        label,
    )


# ============================================================
# 16. POST-RESTORE CRYPTOGRAPHIC VERIFICATION
# ============================================================

post_hashes = {
    "seed1_terminal":
        sha256_file(
            DST_SEED1_TERMINAL
        ),

    "seed2_terminal":
        sha256_file(
            DST_SEED2_TERMINAL
        ),

    "seed3_terminal":
        sha256_file(
            DST_SEED3_TERMINAL
        ),

    "decoder":
        sha256_file(
            DST_DECODER
        ),

    "multiseed_plan":
        sha256_file(
            DST_PLAN
        ),

    "seed1_qc":
        sha256_file(
            DST_SEED1_QC
        ),

    "seed2_qc":
        sha256_file(
            DST_SEED2_QC
        ),

    "seed3_qc":
        sha256_file(
            DST_SEED3_QC
        ),
}


if (
    post_hashes[
        "seed1_terminal"
    ]
    != EXPECTED_SEED1_SHA
):

    raise RuntimeError(
        "POST-RESTORE Seed-1 terminal SHA failure."
    )


if (
    post_hashes[
        "seed2_terminal"
    ]
    != EXPECTED_SEED2_SHA
):

    raise RuntimeError(
        "POST-RESTORE Seed-2 terminal SHA failure."
    )


if (
    post_hashes[
        "seed3_terminal"
    ]
    != EXPECTED_SEED3_SHA
):

    raise RuntimeError(
        "POST-RESTORE Seed-3 terminal SHA failure."
    )


if (
    post_hashes[
        "decoder"
    ]
    != EXPECTED_DECODER_SHA
):

    raise RuntimeError(
        "POST-RESTORE decoder SHA failure."
    )


if (
    post_hashes[
        "multiseed_plan"
    ]
    != outer[
        "multiseed_plan_sha256"
    ]
):

    raise RuntimeError(
        "POST-RESTORE plan SHA failure."
    )


if (
    post_hashes[
        "seed2_qc"
    ]
    != outer[
        "seed2_qc_summary_sha256"
    ]
):

    raise RuntimeError(
        "POST-RESTORE Seed-2 QC SHA failure."
    )


if (
    post_hashes[
        "seed3_qc"
    ]
    != outer[
        "seed3_qc_summary_sha256"
    ]
):

    raise RuntimeError(
        "POST-RESTORE Seed-3 QC SHA failure."
    )


print()
print(
    "Restored Seed-1 terminal SHA: PASS"
)

print(
    "Restored Seed-2 terminal SHA: PASS"
)

print(
    "Restored Seed-3 terminal SHA: PASS"
)

print(
    "Restored decoder SHA: PASS"
)

print(
    "Restored multi-seed plan SHA: PASS"
)

print(
    "Restored Seed-2 QC SHA: PASS"
)

print(
    "Restored Seed-3 QC SHA: PASS"
)


# ============================================================
# 17. WRITE RESTORE AUDIT
# ============================================================

audit = {
    "audit_name":
        "DASO-Net AFTER-SEED3 restore integrity audit v1",

    "status":
        "PASS",

    "project_root":
        str(
            PROJECT_ROOT
        ),

    "handoff_dataset":
        str(
            HANDOFF_DATASET
        ),

    "handoff_root":
        str(
            HANDOFF_ROOT
        ),

    "package_provenance": {
        "expected_original_zip_sha256":
            EXPECTED_PACKAGE_SHA,

        "state_sha_record_match":
            True,

        "manifest_file_count":
            len(
                records
            ),

        "manifest_file_by_file_sha_verified":
            True,
    },

    "final_seed_training": {
        "status":
            "COMPLETE",

        "num_seeds":
            3,

        "target_successful_optimizer_updates":
            TARGET_UPDATES,

        "seed1": {
            "rng_seed":
                SEED1,

            "terminal_sha256":
                EXPECTED_SEED1_SHA,

            "qc":
                "PASS",
        },

        "seed2": {
            "rng_seed":
                SEED2,

            "terminal_sha256":
                EXPECTED_SEED2_SHA,

            "qc":
                "PASS",
        },

        "seed3": {
            "rng_seed":
                SEED3,

            "terminal_sha256":
                EXPECTED_SEED3_SHA,

            "qc":
                "PASS",
        },
    },

    "best_seed_selection":
        False,

    "training_remaining":
        False,

    "external_data_used_for_final_seed_selection":
        False,

    "critical_post_restore_sha256":
        post_hashes,

    "next_stage":
        (
            "formal three-seed aggregation before untouched "
            "test/external evaluation"
        ),
}


save_json(
    RESTORE_AUDIT,
    audit,
)


# ============================================================
# 18. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "AFTER-SEED3 RESTORE COMPLETE"
)

print(
    "============================================================"
)

print()

print(
    "Final Seed 1:"
)

print(
    "  RNG:                     1782656963"
)

print(
    "  successful updates:      2750"
)

print(
    "  terminal SHA:            PASS"
)

print(
    "  QC artifact:             PASS"
)

print()

print(
    "Final Seed 2:"
)

print(
    "  RNG:                     901395089"
)

print(
    "  successful updates:      2750"
)

print(
    "  terminal SHA:            PASS"
)

print(
    "  QC:                      PASS"
)

print()

print(
    "Final Seed 3:"
)

print(
    "  RNG:                     252916394"
)

print(
    "  successful updates:      2750"
)

print(
    "  terminal SHA:            PASS"
)

print(
    "  QC:                      PASS"
)

print()

print(
    "All 3 final seeds preserved: YES"
)

print(
    "Best-seed selection performed: NO"
)

print(
    "Further final-seed training required: NO"
)

print()

print(
    "Restore audit:"
)

print(
    RESTORE_AUDIT
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET AFTER-SEED3 RESTORE + THREE-SEED AUDIT: PASS"
)

print(
    "============================================================"
)

print(
    "NO TRAINING OR INFERENCE WAS PERFORMED."
)

print(
    "STOP HERE."
)

print(
    "Next: formal three-seed aggregate."
)

DASO-Net — AFTER-SEED3 RESTORE + INTEGRITY AUDIT
PASS  Auto-extracted handoff root
PASS  Handoff manifest
PASS  Internal After-Seed3 state
PASS  Handoff README
PASS  Restore script
PASS  Handoff workspace

Original package SHA provenance: PASS
Three-final-seed completion state: PASS
No remaining final-seed training: PASS
No best-seed selection: PASS
Internal three-seed state: PASS

HANDOFF FILE-BY-FILE SHA256 AUDIT
Manifest entries verified: 102
File-by-file SHA256 audit: PASS
PASS  Validated code tree
PASS  Frozen multi-seed plan directory
PASS  Frozen final-training recipe
PASS  Frozen multi-seed plan JSON
PASS  Validated decoder
PASS  Seed-1 terminal
PASS  Seed-2 terminal
PASS  Seed-3 terminal
PASS  Seed-1 QC summary
PASS  Seed-2 QC summary
PASS  Seed-3 QC summary
Seed-1 terminal SHA: PASS
Seed-2 terminal SHA: PASS
Seed-3 terminal SHA: PASS
Validated decoder SHA: PASS

Seed-1 QC artifact: PASS
Seed-2 final QC: PASS
Seed-3 final QC: PASS

Frozen 2750-update recipe: PASS
Frozen Seed-1

RuntimeError: Fresh restore cannot proceed because these /kaggle/working destinations already exist:
/kaggle/working/DASO-Net-AFTER-SEED2-RESTORE-AUDIT-v1.json
/kaggle/working/DASO-Net-FinalMultiSeed-v1
/kaggle/working/DASO-Net-Phase5B-KaggleTraining
/kaggle/working/DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1
/kaggle/working/dasonet_final_qc
/kaggle/working/dasonet_phase5b
/kaggle/working/dasonet_phase5c
/kaggle/working/phase5c_baseline_curve1000updates_v1.json

Restart with a fresh notebook/session rather than deleting scientific state manually.

In [12]:
# ============================================================
# DASO-Net — FORMAL THREE-SEED SYNTHETIC-VALIDATION QC
# AGGREGATE v2
#
# FIX:
# ----
# Seed 1 uses an older FINAL_SEED01_QC_SUMMARY schema and does
# not duplicate every headline metric into that summary file.
#
# Therefore this version verifies each quantity against its
# canonical scientific source:
#
#   metrics         -> phase5c_metric_summary.json
#   oracle          -> oracle_component_diagnostic.json
#   counterfactual  -> counterfactual_center_diagnostic.json
#
# The final QC summaries are still checked for integrity/status,
# but are NOT incorrectly required to contain every metric.
#
# NO TRAINING.
# NO INFERENCE.
# NO MODEL SELECTION.
# NO FROZEN FILE MODIFICATION.
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import math
import shutil
import statistics
from pathlib import Path


# ============================================================
# 1. ROOTS
# ============================================================

WORKING = Path(
    "/kaggle/working"
)

RESTORE_AUDIT = (
    WORKING
    / "DASO-Net-AFTER-SEED3-RESTORE-AUDIT-v1.json"
)

OUT_ROOT = (
    WORKING
    / "dasonet_final_analysis"
    / "three_seed_qc_aggregate_v1"
)

PER_SEED_CSV = (
    OUT_ROOT
    / "three_seed_qc_per_seed.csv"
)

AGGREGATE_CSV = (
    OUT_ROOT
    / "three_seed_qc_mean_sd.csv"
)

AGGREGATE_JSON = (
    OUT_ROOT
    / "FINAL_THREE_SEED_QC_AGGREGATE_v1.json"
)

MARKDOWN_TABLE = (
    OUT_ROOT
    / "three_seed_qc_table.md"
)


# ============================================================
# 2. FINAL SEED SOURCE PATHS
# ============================================================

SEED_SOURCES = {
    1: {
        "label":
            "Seed 1",

        "rng_seed":
            1782656963,

        "qc_root":
            WORKING
            / "dasonet_final_qc"
            / "seed01_s1782656963_val1000_v1",

        "training_manifest":
            WORKING
            / "dasonet_phase5b"
            / "dasonet_final_seed01_s1782656963"
            / "FINAL_SEED01_TRAINING_MANIFEST.json",
    },

    2: {
        "label":
            "Seed 2",

        "rng_seed":
            901395089,

        "qc_root":
            WORKING
            / "dasonet_final_qc"
            / "seed02_s901395089_val1000_v3_fixedop005",

        "training_manifest":
            WORKING
            / "dasonet_phase5b"
            / "dasonet_final_seed02_s901395089"
            / "FINAL_SEED02_TRAINING_MANIFEST.json",
    },

    3: {
        "label":
            "Seed 3",

        "rng_seed":
            252916394,

        "qc_root":
            WORKING
            / "dasonet_final_qc"
            / "seed03_s252916394_val1000_v1_fixedop005",

        "training_manifest":
            WORKING
            / "dasonet_phase5b"
            / "dasonet_final_seed03_s252916394"
            / "FINAL_SEED03_TRAINING_MANIFEST.json",
    },
}


for idx, src in SEED_SOURCES.items():

    qc = src[
        "qc_root"
    ]

    src[
        "summary"
    ] = (
        qc
        / f"FINAL_SEED0{idx}_QC_SUMMARY.json"
    )

    src[
        "metric_json"
    ] = (
        qc
        / "metrics"
        / "phase5c_metric_summary.json"
    )

    src[
        "per_scene_csv"
    ] = (
        qc
        / "metrics"
        / "phase5c_per_scene.csv"
    )

    src[
        "oracle_json"
    ] = (
        qc
        / "oracle"
        / "oracle_component_diagnostic.json"
    )

    src[
        "oracle_csv"
    ] = (
        qc
        / "oracle"
        / "oracle_particle_diagnostic.csv"
    )

    src[
        "cf_json"
    ] = (
        qc
        / "counterfactual"
        / "counterfactual_center_diagnostic.json"
    )


# ============================================================
# 3. CANONICAL FINAL QC VALUES
#
# These are the finalized values already produced by the
# canonical QC runs.
#
# They will now be verified against the CORRECT source artifact.
# ============================================================

CANONICAL = {
    1: {
        # ----------------------------------------------------
        # Main evaluator
        # ----------------------------------------------------
        "mask_ap50":
            0.2311,

        "mask_ap75":
            0.0490,

        "mask_ap50_95":
            0.0872,

        "operating_precision_0_05":
            0.0178,

        "operating_recall_0_05":
            0.3277,

        "operating_f1_0_05":
            0.0338,

        "matched_mask_dice":
            0.8012,

        "count_mae_0_05":
            37.3340,

        "empty_fp_per_image_0_05":
            37.5436,

        "best_f1_threshold":
            0.30,

        "best_f1":
            0.3441,

        # ----------------------------------------------------
        # Oracle
        # ----------------------------------------------------
        "oracle_gt_score_mean":
            0.2280,

        "oracle_peak_recall_r4":
            0.5037,

        "oracle_peak_recall_r8":
            0.7709,

        "oracle_peak_recall_r16":
            0.8887,

        "oracle_peak_recall_r32":
            0.9372,

        "oracle_center_error_px":
            1.4670,

        "oracle_box_iou_mean":
            0.5389,

        "oracle_gt_box_mask_iou_mean":
            0.4386,

        # ----------------------------------------------------
        # Counterfactual
        # ----------------------------------------------------
        "cf_exact_clean_mean":
            0.0135,

        "cf_exact_composite_mean":
            0.2280,

        "cf_exact_lift_mean":
            0.2145,

        "cf_exact_composite_gt_clean":
            0.9749,

        "cf_local_clean_mean":
            0.0148,

        "cf_local_composite_mean":
            0.2799,

        "cf_local_lift_mean":
            0.2651,

        "cf_local_composite_gt_clean":
            0.9055,

        "cf_clean_max_peak_mean":
            0.2207,

        "cf_outside_effect_max_peak_mean":
            0.2172,

        "cf_effect_region_max_peak_mean":
            0.3412,
    },

    2: {
        # ----------------------------------------------------
        # Main evaluator
        # ----------------------------------------------------
        "mask_ap50":
            0.2606,

        "mask_ap75":
            0.0408,

        "mask_ap50_95":
            0.0876,

        "operating_precision_0_05":
            0.0146,

        "operating_recall_0_05":
            0.3785,

        "operating_f1_0_05":
            0.0280,

        "matched_mask_dice":
            0.8001,

        "count_mae_0_05":
            53.7010,

        "empty_fp_per_image_0_05":
            54.6443,

        "best_f1_threshold":
            0.35,

        "best_f1":
            0.3792,

        # ----------------------------------------------------
        # Oracle
        # ----------------------------------------------------
        "oracle_gt_score_mean":
            0.2381,

        "oracle_peak_recall_r4":
            0.5219,

        "oracle_peak_recall_r8":
            0.7384,

        "oracle_peak_recall_r16":
            0.8813,

        "oracle_peak_recall_r32":
            0.9348,

        "oracle_center_error_px":
            1.3220,

        "oracle_box_iou_mean":
            0.5746,

        "oracle_gt_box_mask_iou_mean":
            0.4682,

        # ----------------------------------------------------
        # Counterfactual
        # ----------------------------------------------------
        "cf_exact_clean_mean":
            0.0127,

        "cf_exact_composite_mean":
            0.2381,

        "cf_exact_lift_mean":
            0.2254,

        "cf_exact_composite_gt_clean":
            0.9707,

        "cf_local_clean_mean":
            0.0186,

        "cf_local_composite_mean":
            0.3029,

        "cf_local_lift_mean":
            0.2843,

        "cf_local_composite_gt_clean":
            0.8948,

        "cf_clean_max_peak_mean":
            0.2496,

        "cf_outside_effect_max_peak_mean":
            0.2416,

        "cf_effect_region_max_peak_mean":
            0.3733,
    },

    3: {
        # ----------------------------------------------------
        # Main evaluator
        # ----------------------------------------------------
        "mask_ap50":
            0.2591,

        "mask_ap75":
            0.0547,

        "mask_ap50_95":
            0.0966,

        "operating_precision_0_05":
            0.0181,

        "operating_recall_0_05":
            0.3859,

        "operating_f1_0_05":
            0.0346,

        "matched_mask_dice":
            0.8096,

        "count_mae_0_05":
            43.5730,

        "empty_fp_per_image_0_05":
            44.8725,

        "best_f1_threshold":
            0.35,

        "best_f1":
            0.3953,

        # ----------------------------------------------------
        # Oracle
        # ----------------------------------------------------
        "oracle_gt_score_mean":
            0.2663,

        "oracle_peak_recall_r4":
            0.5251,

        "oracle_peak_recall_r8":
            0.7751,

        "oracle_peak_recall_r16":
            0.9004,

        "oracle_peak_recall_r32":
            0.9437,

        "oracle_center_error_px":
            1.3270,

        "oracle_box_iou_mean":
            0.5793,

        "oracle_gt_box_mask_iou_mean":
            0.4712,

        # ----------------------------------------------------
        # Counterfactual
        # ----------------------------------------------------
        "cf_exact_clean_mean":
            0.0111,

        "cf_exact_composite_mean":
            0.2663,

        "cf_exact_lift_mean":
            0.2552,

        "cf_exact_composite_gt_clean":
            0.9744,

        "cf_local_clean_mean":
            0.0161,

        "cf_local_composite_mean":
            0.3455,

        "cf_local_lift_mean":
            0.3294,

        "cf_local_composite_gt_clean":
            0.9115,

        "cf_clean_max_peak_mean":
            0.2284,

        "cf_outside_effect_max_peak_mean":
            0.2140,

        "cf_effect_region_max_peak_mean":
            0.4248,
    },
}


# ============================================================
# 4. SOURCE CLASSIFICATION
# ============================================================

METRIC_FIELDS = [
    "mask_ap50",
    "mask_ap75",
    "mask_ap50_95",

    "operating_precision_0_05",
    "operating_recall_0_05",
    "operating_f1_0_05",

    "matched_mask_dice",
    "count_mae_0_05",
    "empty_fp_per_image_0_05",

    "best_f1_threshold",
    "best_f1",
]


ORACLE_FIELDS = [
    "oracle_gt_score_mean",

    "oracle_peak_recall_r4",
    "oracle_peak_recall_r8",
    "oracle_peak_recall_r16",
    "oracle_peak_recall_r32",

    "oracle_center_error_px",
    "oracle_box_iou_mean",
    "oracle_gt_box_mask_iou_mean",
]


CF_FIELDS = [
    "cf_exact_clean_mean",
    "cf_exact_composite_mean",
    "cf_exact_lift_mean",
    "cf_exact_composite_gt_clean",

    "cf_local_clean_mean",
    "cf_local_composite_mean",
    "cf_local_lift_mean",
    "cf_local_composite_gt_clean",

    "cf_clean_max_peak_mean",
    "cf_outside_effect_max_peak_mean",
    "cf_effect_region_max_peak_mean",
]


METRIC_ORDER = (
    METRIC_FIELDS
    + ORACLE_FIELDS
    + CF_FIELDS
)


# ============================================================
# 5. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(
    path: Path,
    obj,
) -> None:

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def collect_numbers(
    obj,
):
    """
    Recursively collect all finite numeric scalars.
    """

    numbers = []


    if isinstance(
        obj,
        bool,
    ):

        return numbers


    if isinstance(
        obj,
        (int, float),
    ):

        value = float(
            obj
        )


        if math.isfinite(
            value
        ):

            numbers.append(
                value
            )


        return numbers


    if isinstance(
        obj,
        dict,
    ):

        for value in obj.values():

            numbers.extend(
                collect_numbers(
                    value
                )
            )


    elif isinstance(
        obj,
        list,
    ):

        for value in obj:

            numbers.extend(
                collect_numbers(
                    value
                )
            )


    return numbers


def contains_value(
    numbers,
    target: float,
    tolerance: float = 5e-4,
) -> bool:

    return any(
        abs(
            float(value)
            - float(target)
        )
        <= tolerance
        for value in numbers
    )


def optional_qc_status_check(
    obj: dict,
    label: str,
) -> None:

    if "status" in obj:

        if obj[
            "status"
        ] != "PASS":

            raise RuntimeError(
                f"{label}: status != PASS"
            )


    if "sanity_flags" in obj:

        flags = obj[
            "sanity_flags"
        ]


        if not all(
            bool(v)
            for v in flags.values()
        ):

            failed = [
                key
                for key, value
                in flags.items()
                if not bool(
                    value
                )
            ]

            raise RuntimeError(
                f"{label}: failed sanity flags:\n"
                + "\n".join(
                    failed
                )
            )


def read_csv(
    path: Path,
):

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def find_scene_column(
    rows,
):

    if not rows:

        return None


    lookup = {
        str(key).lower():
            key
        for key in rows[0].keys()
        if key is not None
    }


    for candidate in [
        "scene_id",
        "scene",
        "recipe_id",
        "sample_id",
        "id",
    ]:

        if candidate in lookup:

            return lookup[
                candidate
            ]


    return None


# ============================================================
# 6. PREFLIGHT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FORMAL THREE-SEED QC AGGREGATE v2"
)

print(
    "============================================================"
)


require(
    RESTORE_AUDIT,
    "After-Seed3 restore audit",
)


restore = load_json(
    RESTORE_AUDIT
)


if restore.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "After-Seed3 restore audit is not PASS."
    )


if bool(
    restore.get(
        "best_seed_selection",
        True,
    )
):

    raise RuntimeError(
        "Restore audit indicates best-seed selection."
    )


if bool(
    restore.get(
        "training_remaining",
        True,
    )
):

    raise RuntimeError(
        "Restore audit indicates final training remains."
    )


print(
    "After-Seed3 restore audit: PASS"
)

print(
    "Best-seed selection: NO"
)

print(
    "Further final-seed training: NO"
)


# ============================================================
# 7. VERIFY SOURCE FILES + TRAINING IDENTITIES
# ============================================================

source_hashes = {}


for idx in [
    1,
    2,
    3,
]:

    src = SEED_SOURCES[
        idx
    ]


    for key, label in [
        (
            "training_manifest",
            "training manifest",
        ),
        (
            "summary",
            "final QC summary",
        ),
        (
            "metric_json",
            "metric JSON",
        ),
        (
            "per_scene_csv",
            "per-scene CSV",
        ),
        (
            "oracle_json",
            "oracle JSON",
        ),
        (
            "oracle_csv",
            "oracle CSV",
        ),
        (
            "cf_json",
            "counterfactual JSON",
        ),
    ]:

        require(
            src[
                key
            ],
            f"Seed-{idx} {label}",
        )


    training = load_json(
        src[
            "training_manifest"
        ]
    )


    if int(
        training[
            "rng_seed"
        ]
    ) != int(
        src[
            "rng_seed"
        ]
    ):

        raise RuntimeError(
            f"Seed-{idx} training RNG mismatch."
        )


    if int(
        training[
            "successful_optimizer_updates"
        ]
    ) != 2750:

        raise RuntimeError(
            f"Seed-{idx} is not at exactly 2750 "
            "successful optimizer updates."
        )


    qc_summary = load_json(
        src[
            "summary"
        ]
    )


    optional_qc_status_check(
        qc_summary,
        f"Seed-{idx} QC summary",
    )


    source_hashes[
        str(
            idx
        )
    ] = {
        "training_manifest":
            sha256_file(
                src[
                    "training_manifest"
                ]
            ),

        "qc_summary":
            sha256_file(
                src[
                    "summary"
                ]
            ),

        "metric_json":
            sha256_file(
                src[
                    "metric_json"
                ]
            ),

        "per_scene_csv":
            sha256_file(
                src[
                    "per_scene_csv"
                ]
            ),

        "oracle_json":
            sha256_file(
                src[
                    "oracle_json"
                ]
            ),

        "oracle_csv":
            sha256_file(
                src[
                    "oracle_csv"
                ]
            ),

        "counterfactual_json":
            sha256_file(
                src[
                    "cf_json"
                ]
            ),
    }


    print(
        f"Seed-{idx} source artifact integrity: PASS"
    )


# ============================================================
# 8. VERIFY IDENTICAL ORDERED 1000-SCENE QC SET
# ============================================================

scene_ids_by_seed = {}


for idx in [
    1,
    2,
    3,
]:

    rows = read_csv(
        SEED_SOURCES[
            idx
        ][
            "per_scene_csv"
        ]
    )


    if len(
        rows
    ) != 1000:

        raise RuntimeError(
            f"Seed-{idx} QC does not contain exactly "
            f"1000 scenes: {len(rows)}"
        )


    column = find_scene_column(
        rows
    )


    if column is None:

        raise RuntimeError(
            f"Could not locate scene-ID column for Seed-{idx}."
        )


    scene_ids_by_seed[
        idx
    ] = [
        str(
            row[
                column
            ]
        )
        for row in rows
    ]


if not (
    scene_ids_by_seed[
        1
    ]
    ==
    scene_ids_by_seed[
        2
    ]
    ==
    scene_ids_by_seed[
        3
    ]
):

    raise RuntimeError(
        "Ordered 1000-scene QC set differs across final seeds."
    )


print()
print(
    "Same ordered 1000-scene QC set across all 3 seeds: PASS"
)


# ============================================================
# 9. VERIFY EACH VALUE AGAINST ITS CORRECT CANONICAL SOURCE
# ============================================================

print()
print(
    "============================================================"
)

print(
    "VERIFYING CANONICAL VALUES AGAINST SOURCE ARTIFACTS"
)

print(
    "============================================================"
)


for idx in [
    1,
    2,
    3,
]:

    src = SEED_SOURCES[
        idx
    ]


    metric_numbers = collect_numbers(
        load_json(
            src[
                "metric_json"
            ]
        )
    )


    oracle_numbers = collect_numbers(
        load_json(
            src[
                "oracle_json"
            ]
        )
    )


    cf_numbers = collect_numbers(
        load_json(
            src[
                "cf_json"
            ]
        )
    )


    for field in METRIC_FIELDS:

        expected = CANONICAL[
            idx
        ][
            field
        ]


        if not contains_value(
            metric_numbers,
            expected,
        ):

            raise RuntimeError(
                f"Seed-{idx} evaluator value could not "
                "be verified in metric JSON:\n"
                f"{field} = {expected}"
            )


    for field in ORACLE_FIELDS:

        expected = CANONICAL[
            idx
        ][
            field
        ]


        if not contains_value(
            oracle_numbers,
            expected,
        ):

            raise RuntimeError(
                f"Seed-{idx} oracle value could not "
                "be verified in oracle JSON:\n"
                f"{field} = {expected}"
            )


    for field in CF_FIELDS:

        expected = CANONICAL[
            idx
        ][
            field
        ]


        if not contains_value(
            cf_numbers,
            expected,
        ):

            raise RuntimeError(
                f"Seed-{idx} counterfactual value could not "
                "be verified in counterfactual JSON:\n"
                f"{field} = {expected}"
            )


    print(
        f"Seed-{idx} canonical metric/oracle/CF values: PASS"
    )


print()
print(
    "All finalized values verified against correct sources: PASS"
)


# ============================================================
# 10. CROSS-SEED SCIENTIFIC SANITY
# ============================================================

for idx in [
    1,
    2,
    3,
]:

    values = CANONICAL[
        idx
    ]


    if not (
        values[
            "cf_exact_lift_mean"
        ]
        > 0.0
    ):

        raise RuntimeError(
            f"Seed-{idx}: exact counterfactual lift <= 0."
        )


    if not (
        values[
            "cf_local_lift_mean"
        ]
        > 0.0
    ):

        raise RuntimeError(
            f"Seed-{idx}: local counterfactual lift <= 0."
        )


    if not (
        values[
            "cf_effect_region_max_peak_mean"
        ]
        >
        values[
            "cf_outside_effect_max_peak_mean"
        ]
    ):

        raise RuntimeError(
            f"Seed-{idx}: effect response does not exceed "
            "outside-effect response."
        )


    if not (
        values[
            "cf_exact_composite_gt_clean"
        ]
        > 0.5
    ):

        raise RuntimeError(
            f"Seed-{idx}: exact composite > clean majority failed."
        )


    if not (
        values[
            "cf_local_composite_gt_clean"
        ]
        > 0.5
    ):

        raise RuntimeError(
            f"Seed-{idx}: local composite > clean majority failed."
        )


print(
    "Cross-seed mechanistic sanity: PASS"
)


# ============================================================
# 11. RESET ONLY THE DERIVED ANALYSIS DIRECTORY
#
# This is NOT training/QC state.
# It is only the aggregate output from this analysis cell.
# ============================================================

if OUT_ROOT.exists():

    shutil.rmtree(
        OUT_ROOT
    )


OUT_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 12. COMPUTE FORMAL MEAN + SAMPLE SD
# ============================================================

aggregate = {}


for metric in METRIC_ORDER:

    values = [
        float(
            CANONICAL[
                idx
            ][
                metric
            ]
        )
        for idx in [
            1,
            2,
            3,
        ]
    ]


    aggregate[
        metric
    ] = {
        "values":
            values,

        "mean":
            statistics.mean(
                values
            ),

        "sample_sd":
            statistics.stdev(
                values
            ),

        "n":
            3,
    }


# ============================================================
# 13. WRITE PER-SEED CSV
# ============================================================

with PER_SEED_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.writer(
        f
    )


    writer.writerow(
        [
            "seed_index",
            "rng_seed",
            *METRIC_ORDER,
        ]
    )


    for idx in [
        1,
        2,
        3,
    ]:

        writer.writerow(
            [
                idx,
                SEED_SOURCES[
                    idx
                ][
                    "rng_seed"
                ],
                *[
                    CANONICAL[
                        idx
                    ][
                        metric
                    ]
                    for metric in METRIC_ORDER
                ],
            ]
        )


# ============================================================
# 14. WRITE AGGREGATE CSV
# ============================================================

with AGGREGATE_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.writer(
        f
    )


    writer.writerow(
        [
            "metric",
            "seed1",
            "seed2",
            "seed3",
            "mean",
            "sample_sd",
            "n",
        ]
    )


    for metric in METRIC_ORDER:

        record = aggregate[
            metric
        ]


        writer.writerow(
            [
                metric,
                record[
                    "values"
                ][
                    0
                ],
                record[
                    "values"
                ][
                    1
                ],
                record[
                    "values"
                ][
                    2
                ],
                record[
                    "mean"
                ],
                record[
                    "sample_sd"
                ],
                3,
            ]
        )


# ============================================================
# 15. WRITE FORMAL JSON
# ============================================================

formal = {
    "analysis_name":
        (
            "DASO-Net formal three-seed synthetic-validation "
            "QC aggregate v1"
        ),

    "implementation_revision":
        (
            "v2 wrapper: source-specific metric verification "
            "for backward-compatible Seed-1 QC schema"
        ),

    "status":
        "PASS",

    "scientific_role":
        (
            "Aggregate of the three precommitted final-model "
            "synthetic-validation QC runs. These results are "
            "development/QC evidence and are not untouched-test "
            "or independent external-domain performance."
        ),

    "seed_policy": {
        "num_final_seeds":
            3,

        "all_final_seeds_retained":
            True,

        "best_seed_selection":
            False,

        "seed1_rng":
            1782656963,

        "seed2_rng":
            901395089,

        "seed3_rng":
            252916394,

        "successful_optimizer_updates_per_seed":
            2750,

        "same_ordered_qc_scenes":
            True,

        "qc_scenes_per_seed":
            1000,
    },

    "statistics": {
        "mean":
            "arithmetic mean across the 3 precommitted seeds",

        "sd":
            (
                "sample standard deviation with denominator n-1"
            ),

        "n":
            3,
    },

    "canonical_source_sha256":
        source_hashes,

    "metrics":
        aggregate,

    "interpretation_constraints": [
        (
            "Matched-mask Dice is conditional on matched "
            "predictions and must not be reported as overall "
            "segmentation Dice."
        ),
        (
            "The 0.05 operating threshold is intentionally "
            "permissive and diagnostic; it is not a deployment "
            "threshold."
        ),
        (
            "The best-F1 thresholds were obtained from validation "
            "QC and must not be retuned using untouched test or "
            "external-real test data."
        ),
        (
            "No individual final seed is selected, rejected, "
            "ranked, or discarded."
        ),
        (
            "Untouched FruitMP-Syn test and independent real-domain "
            "evaluation remain separate from this artifact."
        ),
    ],
}


save_json(
    AGGREGATE_JSON,
    formal,
)


# ============================================================
# 16. WRITE MARKDOWN HEADLINE TABLE
# ============================================================

HEADLINE = [
    (
        "Mask AP50",
        "mask_ap50",
    ),
    (
        "Mask AP75",
        "mask_ap75",
    ),
    (
        "Mask AP50:95",
        "mask_ap50_95",
    ),
    (
        "Best validation F1",
        "best_f1",
    ),
    (
        "Matched-mask Dice*",
        "matched_mask_dice",
    ),
    (
        "Count MAE @0.05",
        "count_mae_0_05",
    ),
    (
        "Empty FP/image @0.05",
        "empty_fp_per_image_0_05",
    ),
    (
        "Oracle GT-cell score",
        "oracle_gt_score_mean",
    ),
    (
        "Oracle peak recall <=8 px",
        "oracle_peak_recall_r8",
    ),
    (
        "Oracle Box IoU",
        "oracle_box_iou_mean",
    ),
    (
        "GT-box mask IoU",
        "oracle_gt_box_mask_iou_mean",
    ),
    (
        "CF exact lift",
        "cf_exact_lift_mean",
    ),
    (
        "CF local lift",
        "cf_local_lift_mean",
    ),
    (
        "CF effect-region max",
        "cf_effect_region_max_peak_mean",
    ),
]


lines = [
    (
        "| Metric | Seed 1 | Seed 2 | Seed 3 | "
        "Mean ± sample SD |"
    ),
    "|---|---:|---:|---:|---:|",
]


for label, metric in HEADLINE:

    record = aggregate[
        metric
    ]

    v1, v2, v3 = record[
        "values"
    ]


    lines.append(
        f"| {label} "
        f"| {v1:.4f} "
        f"| {v2:.4f} "
        f"| {v3:.4f} "
        f"| {record['mean']:.4f} ± "
        f"{record['sample_sd']:.4f} |"
    )


lines.extend(
    [
        "",
        (
            "*Matched-mask Dice is conditional on matched "
            "predictions; it is not overall segmentation Dice."
        ),
        "",
        (
            "These are synthetic-validation QC results, not "
            "untouched-test or independent real-domain results."
        ),
    ]
)


MARKDOWN_TABLE.write_text(
    "\n".join(
        lines
    )
    + "\n",
    encoding="utf-8",
)


# ============================================================
# 17. FINAL AGGREGATE SANITY
# ============================================================

if not (
    aggregate[
        "mask_ap50_95"
    ][
        "sample_sd"
    ]
    < 0.01
):

    raise RuntimeError(
        "Unexpectedly high AP50:95 cross-seed variability."
    )


if not (
    aggregate[
        "matched_mask_dice"
    ][
        "sample_sd"
    ]
    < 0.01
):

    raise RuntimeError(
        "Unexpectedly high matched-Dice cross-seed variability."
    )


if aggregate[
    "cf_exact_lift_mean"
][
    "mean"
] <= 0.0:

    raise RuntimeError(
        "Mean exact counterfactual lift is not positive."
    )


if aggregate[
    "cf_local_lift_mean"
][
    "mean"
] <= 0.0:

    raise RuntimeError(
        "Mean local counterfactual lift is not positive."
    )


print()
print(
    "Formal aggregate sanity checks: PASS"
)


# ============================================================
# 18. OUTPUT HASHES
# ============================================================

output_hashes = {
    "per_seed_csv":
        sha256_file(
            PER_SEED_CSV
        ),

    "aggregate_csv":
        sha256_file(
            AGGREGATE_CSV
        ),

    "aggregate_json":
        sha256_file(
            AGGREGATE_JSON
        ),

    "markdown_table":
        sha256_file(
            MARKDOWN_TABLE
        ),
}


# ============================================================
# 19. FINAL REPORT
# ============================================================

def show(
    label: str,
    metric: str,
    digits: int = 4,
) -> None:

    record = aggregate[
        metric
    ]


    print(
        f"{label:<34}"
        f"{record['mean']:.{digits}f} "
        f"± "
        f"{record['sample_sd']:.{digits}f}"
    )


print()
print(
    "============================================================"
)

print(
    "FINAL THREE-SEED SYNTHETIC-VALIDATION QC AGGREGATE"
)

print(
    "============================================================"
)

print()

show(
    "Mask AP50:",
    "mask_ap50",
)

show(
    "Mask AP75:",
    "mask_ap75",
)

show(
    "Mask AP50:95:",
    "mask_ap50_95",
)

show(
    "Best validation F1:",
    "best_f1",
)

show(
    "Matched-mask Dice*:",
    "matched_mask_dice",
)

show(
    "Count MAE @0.05:",
    "count_mae_0_05",
)

show(
    "Empty FP/image @0.05:",
    "empty_fp_per_image_0_05",
)

print()

show(
    "Operating precision @0.05:",
    "operating_precision_0_05",
)

show(
    "Operating recall @0.05:",
    "operating_recall_0_05",
)

show(
    "Operating F1 @0.05:",
    "operating_f1_0_05",
)

print()

show(
    "Oracle GT-cell score:",
    "oracle_gt_score_mean",
)

show(
    "Oracle peak recall <=4px:",
    "oracle_peak_recall_r4",
)

show(
    "Oracle peak recall <=8px:",
    "oracle_peak_recall_r8",
)

show(
    "Oracle peak recall <=16px:",
    "oracle_peak_recall_r16",
)

show(
    "Oracle peak recall <=32px:",
    "oracle_peak_recall_r32",
)

show(
    "Oracle center error px:",
    "oracle_center_error_px",
)

show(
    "Oracle Box IoU:",
    "oracle_box_iou_mean",
)

show(
    "GT-box mask IoU:",
    "oracle_gt_box_mask_iou_mean",
)

print()

show(
    "CF exact clean:",
    "cf_exact_clean_mean",
)

show(
    "CF exact composite:",
    "cf_exact_composite_mean",
)

show(
    "CF exact lift:",
    "cf_exact_lift_mean",
)

show(
    "CF exact comp>clean:",
    "cf_exact_composite_gt_clean",
)

show(
    "CF local clean:",
    "cf_local_clean_mean",
)

show(
    "CF local composite:",
    "cf_local_composite_mean",
)

show(
    "CF local lift:",
    "cf_local_lift_mean",
)

show(
    "CF local comp>clean:",
    "cf_local_composite_gt_clean",
)

show(
    "CF clean max:",
    "cf_clean_max_peak_mean",
)

show(
    "CF outside-effect max:",
    "cf_outside_effect_max_peak_mean",
)

show(
    "CF effect-region max:",
    "cf_effect_region_max_peak_mean",
)

print()

print(
    "Best-F1 validation thresholds:"
)

print(
    "  Seed 1: 0.30"
)

print(
    "  Seed 2: 0.35"
)

print(
    "  Seed 3: 0.35"
)

print()

print(
    "Aggregate JSON:"
)

print(
    AGGREGATE_JSON
)

print()

print(
    "Aggregate CSV:"
)

print(
    AGGREGATE_CSV
)

print()

print(
    "Per-seed CSV:"
)

print(
    PER_SEED_CSV
)

print()

print(
    "Markdown table:"
)

print(
    MARKDOWN_TABLE
)

print()

print(
    "Aggregate JSON SHA256:"
)

print(
    output_hashes[
        "aggregate_json"
    ]
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET FORMAL THREE-SEED QC AGGREGATE: PASS"
)

print(
    "============================================================"
)

print()

print(
    "NO TRAINING OR INFERENCE WAS PERFORMED."
)

print(
    "NO BEST-SEED SELECTION WAS PERFORMED."
)

print(
    "ALL THREE FINAL SEEDS REMAIN IN THE FINAL ANALYSIS."
)

print()

print(
    "IMPORTANT:"
)

print(
    "These are SYNTHETIC-VALIDATION QC statistics."
)

print(
    "They are NOT untouched-test performance."
)

print()

print(
    "STOP HERE."
)

print(
    "Next: freeze the formal evaluation protocol before "
    "running the untouched FruitMP-Syn test."
)

DASO-Net — FORMAL THREE-SEED QC AGGREGATE v2
After-Seed3 restore audit: PASS
Best-seed selection: NO
Further final-seed training: NO
Seed-1 source artifact integrity: PASS
Seed-2 source artifact integrity: PASS
Seed-3 source artifact integrity: PASS

Same ordered 1000-scene QC set across all 3 seeds: PASS

VERIFYING CANONICAL VALUES AGAINST SOURCE ARTIFACTS
Seed-1 canonical metric/oracle/CF values: PASS
Seed-2 canonical metric/oracle/CF values: PASS
Seed-3 canonical metric/oracle/CF values: PASS

All finalized values verified against correct sources: PASS
Cross-seed mechanistic sanity: PASS

Formal aggregate sanity checks: PASS

FINAL THREE-SEED SYNTHETIC-VALIDATION QC AGGREGATE

Mask AP50:                        0.2503 ± 0.0166
Mask AP75:                        0.0482 ± 0.0070
Mask AP50:95:                     0.0905 ± 0.0053
Best validation F1:               0.3729 ± 0.0262
Matched-mask Dice*:               0.8036 ± 0.0052
Count MAE @0.05:                  44.8693 ± 8.2601
Empty FP/i

In [13]:
# ============================================================
# DASO-Net — FORMAL EVALUATION PROTOCOL FREEZE v1
# REWRITTEN AFTER FRUITBG-SCHEMA WRAPPER FAILURE
#
# IMPORTANT
# ---------
# The previous attempt:
#   - DID NOT train
#   - DID NOT run inference
#   - DID NOT observe test predictions
#   - DID NOT create a completed protocol freeze
#
# Wrapper fix only:
# FruitBG split column is identified from its frozen split counts
# rather than assuming its column name.
#
# NO TRAINING.
# NO INFERENCE.
# NO MODEL SELECTION.
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import shutil
from collections import Counter
from pathlib import Path


# ============================================================
# 1. ROOTS
# ============================================================

WORKING = Path("/kaggle/working")

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CODE_ROOT = (
    WORKING
    / "DASO-Net-Phase5B-KaggleTraining"
)

ANALYSIS_ROOT = (
    WORKING
    / "dasonet_final_analysis"
)

AGGREGATE_ROOT = (
    ANALYSIS_ROOT
    / "three_seed_qc_aggregate_v1"
)

AGGREGATE_JSON = (
    AGGREGATE_ROOT
    / "FINAL_THREE_SEED_QC_AGGREGATE_v1.json"
)

RESTORE_AUDIT = (
    WORKING
    / "DASO-Net-AFTER-SEED3-RESTORE-AUDIT-v1.json"
)


# ============================================================
# 2. FROZEN PROJECT DATA
# ============================================================

TEST_RECIPES = (
    PROJECT_ROOT
    / "data"
    / "fruitmp_syn"
    / "v1_1_recipes"
    / "fruitmp_syn_v1_1_test_recipes.csv"
)

FULL_SYN_FREEZE = (
    PROJECT_ROOT
    / "data"
    / "fruitmp_syn"
    / "v1_1_recipes"
    / "fruitmp_syn_v1_1_full_freeze_manifest.json"
)

FRUITBG_MANIFEST = (
    PROJECT_ROOT
    / "data"
    / "fruit_bg"
    / "v1_final"
    / "fruitbg_rgb_v1_manifest.csv"
)

MORPH_MANIFEST = (
    PROJECT_ROOT
    / "data"
    / "mp_morph"
    / "v1"
    / "morphology_manifest_v1.csv"
)

RENDERER_CONFIG = (
    PROJECT_ROOT
    / "configs"
    / "renderer_v3_1.json"
)

PROJECT_RENDERER = (
    PROJECT_ROOT
    / "src"
    / "dasonet"
    / "synth"
    / "renderer.py"
)


# ============================================================
# 3. VALIDATED EVALUATION CODE
# ============================================================

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)


# ============================================================
# 4. FINAL TERMINAL CHECKPOINTS
# ============================================================

SEED1_TERMINAL = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

SEED2_TERMINAL = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

SEED3_TERMINAL = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed03_s252916394"
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)


EXPECTED_SEED1_SHA = (
    "e4c4b79ad4b8c6e18ba2b1e3d6cbe796"
    "c9cc301bd6fcc7c596fc290ad3a7e81d"
)

EXPECTED_SEED2_SHA = (
    "19cdb834a46240916b2c8023e8c3afe573"
    "8d9c7cd5d8f818c62ad1e289df1ad2"
)

EXPECTED_SEED3_SHA = (
    "be7b7941f2f62179e15074aa973bf6da0"
    "eb9018ec9133486512bd1f87153f447"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)

EXPECTED_AGGREGATE_SHA = (
    "e5fab1decdc1efc3a194ee428b44a776"
    "b5fe21478ff396196eb8f1606168676d"
)


# ============================================================
# 5. FROZEN VALIDATION-DERIVED OPERATING THRESHOLDS
# ============================================================

FINAL_SEEDS = {
    1: {
        "rng_seed": 1782656963,
        "checkpoint": SEED1_TERMINAL,
        "checkpoint_sha256": EXPECTED_SEED1_SHA,
        "operating_threshold": 0.30,
    },

    2: {
        "rng_seed": 901395089,
        "checkpoint": SEED2_TERMINAL,
        "checkpoint_sha256": EXPECTED_SEED2_SHA,
        "operating_threshold": 0.35,
    },

    3: {
        "rng_seed": 252916394,
        "checkpoint": SEED3_TERMINAL,
        "checkpoint_sha256": EXPECTED_SEED3_SHA,
        "operating_threshold": 0.35,
    },
}


# ============================================================
# 6. FIXED DECODER / DATA COUNTS
# ============================================================

DECODE_MIN_SCORE = 0.05
TOPK = 100
MASK_THRESHOLD = 0.50

EXPECTED_TEST_SCENES = 5000

EXPECTED_FRUITBG_TOTAL = 1644
EXPECTED_FRUITBG_TRAIN = 1151
EXPECTED_FRUITBG_VAL = 247
EXPECTED_FRUITBG_TEST = 246


# ============================================================
# 7. OUTPUT
# ============================================================

FREEZE_ROOT = (
    ANALYSIS_ROOT
    / "formal_evaluation_protocol_freeze_v1"
)

PROTOCOL_JSON = (
    FREEZE_ROOT
    / "FORMAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

PROTOCOL_README = (
    FREEZE_ROOT
    / "README_FORMAL_EVALUATION_PROTOCOL.txt"
)

PROTOCOL_SHA = (
    FREEZE_ROOT
    / "FORMAL_EVALUATION_PROTOCOL_FREEZE_v1.sha256.txt"
)


# ============================================================
# 8. HELPERS
# ============================================================

def require(path: Path, label: str) -> None:
    if not path.exists():
        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def load_json(path: Path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(path: Path, obj) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def read_csv(path: Path):
    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:
        return list(
            csv.DictReader(f)
        )


def detect_column(
    fieldnames,
    candidates,
):
    lookup = {
        str(name).strip().lower(): name
        for name in fieldnames
        if name is not None
    }

    for candidate in candidates:
        key = candidate.strip().lower()

        if key in lookup:
            return lookup[key]

    return None


def normalize_split_value(value):
    """
    Map common textual split spellings to:
      train / val / test

    Return None for anything else.
    """

    s = str(value).strip().lower()

    mapping = {
        "train": "train",
        "training": "train",

        "val": "val",
        "valid": "val",
        "validation": "val",
        "dev": "val",

        "test": "test",
        "testing": "test",
    }

    return mapping.get(
        s,
        None,
    )


def detect_fruitbg_split_column(rows):
    """
    Identify the actual FruitBG split column from the frozen
    row counts rather than assuming a column name.

    Required frozen counts:
      train = 1151
      val   = 247
      test  = 246
      total = 1644
    """

    if not rows:
        raise RuntimeError(
            "FruitBG manifest is empty."
        )

    fields = list(
        rows[0].keys()
    )

    candidates = []

    diagnostics = {}


    for field in fields:

        normalized = [
            normalize_split_value(
                row.get(
                    field,
                    "",
                )
            )
            for row in rows
        ]

        counts = Counter(
            value
            for value in normalized
            if value is not None
        )

        recognized = sum(
            counts.values()
        )

        diagnostics[
            str(field)
        ] = {
            "recognized_rows":
                recognized,

            "train":
                counts.get(
                    "train",
                    0,
                ),

            "val":
                counts.get(
                    "val",
                    0,
                ),

            "test":
                counts.get(
                    "test",
                    0,
                ),
        }


        if (
            recognized
            == EXPECTED_FRUITBG_TOTAL
            and counts.get(
                "train",
                0,
            )
            == EXPECTED_FRUITBG_TRAIN
            and counts.get(
                "val",
                0,
            )
            == EXPECTED_FRUITBG_VAL
            and counts.get(
                "test",
                0,
            )
            == EXPECTED_FRUITBG_TEST
        ):

            candidates.append(
                field
            )


    if len(
        candidates
    ) != 1:

        print()
        print(
            "FruitBG split-column diagnostics:"
        )

        for field, info in diagnostics.items():

            if (
                info[
                    "recognized_rows"
                ]
                > 0
            ):
                print(
                    f"  {field}: "
                    f"recognized={info['recognized_rows']} "
                    f"train={info['train']} "
                    f"val={info['val']} "
                    f"test={info['test']}"
                )


        raise RuntimeError(
            "Could not uniquely identify FruitBG split column "
            "from the frozen 1151/247/246 counts.\n"
            f"Candidates: {candidates}"
        )


    return candidates[0]


# ============================================================
# 9. PREFLIGHT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FORMAL EVALUATION PROTOCOL FREEZE v1"
)

print(
    "============================================================"
)


for path, label in [
    (
        RESTORE_AUDIT,
        "After-Seed3 restore audit",
    ),
    (
        AGGREGATE_JSON,
        "Formal three-seed QC aggregate",
    ),
    (
        TEST_RECIPES,
        "Untouched FruitMP-Syn test recipes",
    ),
    (
        FULL_SYN_FREEZE,
        "FruitMP-Syn full freeze manifest",
    ),
    (
        FRUITBG_MANIFEST,
        "FruitBG manifest",
    ),
    (
        MORPH_MANIFEST,
        "MP morphology manifest",
    ),
    (
        RENDERER_CONFIG,
        "Renderer config",
    ),
    (
        PROJECT_RENDERER,
        "Frozen project renderer",
    ),
    (
        EVALUATOR,
        "Validated evaluator",
    ),
    (
        DECODER,
        "Validated decoder",
    ),
    (
        SEED1_TERMINAL,
        "Seed-1 terminal",
    ),
    (
        SEED2_TERMINAL,
        "Seed-2 terminal",
    ),
    (
        SEED3_TERMINAL,
        "Seed-3 terminal",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 10. RESTORE + THREE-SEED AGGREGATE PROVENANCE
# ============================================================

restore = load_json(
    RESTORE_AUDIT
)


if restore.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Restore audit is not PASS."
    )


if bool(
    restore.get(
        "best_seed_selection",
        True,
    )
):

    raise RuntimeError(
        "Best-seed selection unexpectedly detected."
    )


if bool(
    restore.get(
        "training_remaining",
        True,
    )
):

    raise RuntimeError(
        "Final-seed training unexpectedly remains."
    )


actual_aggregate_sha = sha256_file(
    AGGREGATE_JSON
)


if actual_aggregate_sha != EXPECTED_AGGREGATE_SHA:

    raise RuntimeError(
        "Formal three-seed aggregate SHA mismatch.\n"
        f"Expected: {EXPECTED_AGGREGATE_SHA}\n"
        f"Actual:   {actual_aggregate_sha}"
    )


aggregate = load_json(
    AGGREGATE_JSON
)


if aggregate.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Three-seed QC aggregate is not PASS."
    )


print()
print(
    "After-Seed3 restore provenance: PASS"
)

print(
    "Three-seed aggregate SHA: PASS"
)

print(
    "No best-seed selection: PASS"
)


# ============================================================
# 11. TERMINAL MODEL CRYPTOGRAPHIC AUDIT
# ============================================================

for idx, record in FINAL_SEEDS.items():

    actual_sha = sha256_file(
        record[
            "checkpoint"
        ]
    )


    if actual_sha != record[
        "checkpoint_sha256"
    ]:

        raise RuntimeError(
            f"Seed-{idx} terminal SHA mismatch."
        )


    print(
        f"Seed-{idx} terminal SHA: PASS"
    )


if sha256_file(
    DECODER
) != EXPECTED_DECODER_SHA:

    raise RuntimeError(
        "Validated decoder SHA mismatch."
    )


print(
    "Validated decoder SHA: PASS"
)


# ============================================================
# 12. AUDIT UNTOUCHED FRUITMP-SYN TEST
# ============================================================

test_rows = read_csv(
    TEST_RECIPES
)


if len(
    test_rows
) != EXPECTED_TEST_SCENES:

    raise RuntimeError(
        "Untouched FruitMP-Syn test count mismatch.\n"
        f"Expected: {EXPECTED_TEST_SCENES}\n"
        f"Actual:   {len(test_rows)}"
    )


test_fields = list(
    test_rows[0].keys()
)


split_col = detect_column(
    test_fields,
    [
        "split",
        "subset",
        "partition",
    ],
)


if split_col is not None:

    split_values = {
        str(
            row[
                split_col
            ]
        ).strip().lower()
        for row in test_rows
    }


    if split_values != {
        "test"
    }:

        raise RuntimeError(
            "Test recipe file contains unexpected split values:\n"
            f"{sorted(split_values)}"
        )


scene_col = detect_column(
    test_fields,
    [
        "scene_id",
        "recipe_id",
        "sample_id",
        "id",
    ],
)


if scene_col is None:

    raise RuntimeError(
        "Could not identify scene-ID column in "
        "FruitMP-Syn test recipes."
    )


scene_ids = [
    str(
        row[
            scene_col
        ]
    )
    for row in test_rows
]


if len(
    set(
        scene_ids
    )
) != EXPECTED_TEST_SCENES:

    raise RuntimeError(
        "Untouched test scene IDs are not unique."
    )


fruit_col = detect_column(
    test_fields,
    [
        "fruit",
        "fruit_type",
        "fruit_class",
        "category",
    ],
)


fruit_distribution = {}


if fruit_col is not None:

    fruit_distribution = dict(
        sorted(
            Counter(
                str(
                    row[
                        fruit_col
                    ]
                )
                for row in test_rows
            ).items()
        )
    )


print()
print(
    "Untouched FruitMP-Syn test scenes: 5000 — PASS"
)

print(
    "Unique untouched test scene IDs: 5000 — PASS"
)

print(
    "No test inference has been performed by this cell."
)


if fruit_distribution:

    print(
        "Test fruit distribution:"
    )

    for key, value in fruit_distribution.items():

        print(
            f"  {key}: {value}"
        )


# ============================================================
# 13. ROBUST FRUITBG SPLIT AUDIT
# ============================================================

fruitbg_rows = read_csv(
    FRUITBG_MANIFEST
)


if len(
    fruitbg_rows
) != EXPECTED_FRUITBG_TOTAL:

    raise RuntimeError(
        "FruitBG total row count mismatch.\n"
        f"Expected: {EXPECTED_FRUITBG_TOTAL}\n"
        f"Actual:   {len(fruitbg_rows)}"
    )


fruitbg_split_col = detect_fruitbg_split_column(
    fruitbg_rows
)


normalized_fruitbg_splits = [
    normalize_split_value(
        row[
            fruitbg_split_col
        ]
    )
    for row in fruitbg_rows
]


fruitbg_counts = Counter(
    normalized_fruitbg_splits
)


if (
    fruitbg_counts[
        "train"
    ]
    != EXPECTED_FRUITBG_TRAIN
):

    raise RuntimeError(
        "FruitBG train count mismatch."
    )


if (
    fruitbg_counts[
        "val"
    ]
    != EXPECTED_FRUITBG_VAL
):

    raise RuntimeError(
        "FruitBG val count mismatch."
    )


if (
    fruitbg_counts[
        "test"
    ]
    != EXPECTED_FRUITBG_TEST
):

    raise RuntimeError(
        "FruitBG test count mismatch."
    )


fruitbg_test_rows = [
    row
    for row, split_value
    in zip(
        fruitbg_rows,
        normalized_fruitbg_splits,
    )
    if split_value
    == "test"
]


print()
print(
    "FruitBG split column detected:"
)

print(
    f"  {fruitbg_split_col}"
)

print()

print(
    "FruitBG frozen split counts:"
)

print(
    "  train: 1151"
)

print(
    "  val:    247"
)

print(
    "  test:   246"
)

print(
    "  total:  1644"
)

print()

print(
    "Held-out clean FruitBG test backgrounds: 246 — PASS"
)


# ============================================================
# 14. VERIFY VALIDATION-ONLY OPERATING THRESHOLDS
# ============================================================

expected_thresholds = {
    1: 0.30,
    2: 0.35,
    3: 0.35,
}


for idx in [
    1,
    2,
    3,
]:

    actual = FINAL_SEEDS[
        idx
    ][
        "operating_threshold"
    ]


    if actual != expected_thresholds[
        idx
    ]:

        raise RuntimeError(
            f"Seed-{idx} operating threshold mismatch."
        )


print()
print(
    "Validation-derived operating thresholds frozen:"
)

print(
    "  Seed 1: 0.30"
)

print(
    "  Seed 2: 0.35"
)

print(
    "  Seed 3: 0.35"
)

print(
    "Test-derived threshold tuning: FORBIDDEN"
)


# ============================================================
# 15. FREEZE CRITICAL HASHES
# ============================================================

critical_sha256 = {
    "three_seed_qc_aggregate":
        actual_aggregate_sha,

    "fruitmp_syn_test_recipes":
        sha256_file(
            TEST_RECIPES
        ),

    "fruitmp_syn_full_freeze_manifest":
        sha256_file(
            FULL_SYN_FREEZE
        ),

    "fruitbg_manifest":
        sha256_file(
            FRUITBG_MANIFEST
        ),

    "morphology_manifest":
        sha256_file(
            MORPH_MANIFEST
        ),

    "renderer_config":
        sha256_file(
            RENDERER_CONFIG
        ),

    "project_renderer":
        sha256_file(
            PROJECT_RENDERER
        ),

    "evaluator":
        sha256_file(
            EVALUATOR
        ),

    "decoder":
        sha256_file(
            DECODER
        ),

    "seed1_terminal":
        sha256_file(
            SEED1_TERMINAL
        ),

    "seed2_terminal":
        sha256_file(
            SEED2_TERMINAL
        ),

    "seed3_terminal":
        sha256_file(
            SEED3_TERMINAL
        ),
}


# ============================================================
# 16. PREDECLARED FORMAL EVALUATION PROTOCOL
# ============================================================

protocol = {
    "protocol_name":
        "DASO-Net Formal Evaluation Protocol Freeze v1",

    "implementation_note":
        (
            "FruitBG split column was discovered from the "
            "already-frozen 1151/247/246 counts rather than "
            "assuming a column name. This is an infrastructure/"
            "schema wrapper fix and does not change the "
            "scientific protocol."
        ),

    "status":
        "FROZEN_BEFORE_TEST_INFERENCE",

    "scientific_scope":
        (
            "Final evaluation of the three precommitted "
            "DASO-Net models for visually resolvable suspected "
            "microplastic-like particle instance segmentation/"
            "selective screening on RGB fruit surfaces."
        ),

    "claim_boundary": {
        "rgb_chemical_identification":
            False,

        "polymer_classification":
            False,

        "target":
            (
                "visually resolvable suspected microplastic-like "
                "particles / suspected surface plastic contamination"
            ),
    },

    "final_seed_policy": {
        "num_seeds":
            3,

        "all_seeds_retained":
            True,

        "best_seed_selection":
            False,

        "checkpoint_selection_after_test":
            False,

        "successful_optimizer_updates_per_seed":
            2750,

        "seeds": {
            "seed1": {
                "rng_seed":
                    1782656963,

                "operating_threshold":
                    0.30,

                "threshold_source":
                    "fixed synthetic-validation QC only",
            },

            "seed2": {
                "rng_seed":
                    901395089,

                "operating_threshold":
                    0.35,

                "threshold_source":
                    "fixed synthetic-validation QC only",
            },

            "seed3": {
                "rng_seed":
                    252916394,

                "operating_threshold":
                    0.35,

                "threshold_source":
                    "fixed synthetic-validation QC only",
            },
        },
    },

    "decoder_policy": {
        "decode_min_score":
            DECODE_MIN_SCORE,

        "topk":
            TOPK,

        "mask_threshold":
            MASK_THRESHOLD,

        "decoder_sha256":
            EXPECTED_DECODER_SHA,

        "post_test_decoder_changes":
            False,
    },

    "untouched_synthetic_test": {
        "dataset":
            "FruitMP-Syn v1.1",

        "split":
            "test",

        "num_scenes":
            5000,

        "fruit_distribution":
            fruit_distribution,

        "use_all_test_scenes":
            True,

        "subsampling":
            False,

        "test_recipe_sha256":
            critical_sha256[
                "fruitmp_syn_test_recipes"
            ],

        "primary_metrics": [
            "mask AP50",
            "mask AP75",
            "mask AP50:95",
        ],

        "secondary_operating_metrics": [
            (
                "precision at each seed's frozen "
                "validation-derived operating threshold"
            ),
            (
                "recall at each seed's frozen "
                "validation-derived operating threshold"
            ),
            (
                "F1 at each seed's frozen "
                "validation-derived operating threshold"
            ),
            "count MAE",
            "count bias",
            "zero-particle false positives per image",
        ],

        "conditional_localization_metrics": [
            "matched mask IoU",
            "matched mask Dice",
        ],

        "stratified_reporting": [
            "fruit type",
            "particle size bin using evaluator definitions unchanged",
        ],

        "threshold_sweep_policy":
            (
                "The existing evaluator threshold sweep may be "
                "retained as descriptive sensitivity analysis only. "
                "A test-best threshold must never replace the frozen "
                "validation-derived operating threshold."
            ),

        "model_selection_using_test":
            False,

        "threshold_selection_using_test":
            False,
    },

    "held_out_clean_fruitbg": {
        "dataset":
            "FruitBG-RGB v1 final",

        "manifest_total":
            1644,

        "detected_split_column":
            str(
                fruitbg_split_col
            ),

        "frozen_split_counts": {
            "train":
                1151,

            "val":
                247,

            "test":
                246,
        },

        "split":
            "test",

        "num_backgrounds":
            246,

        "purpose":
            (
                "independent held-out clean-background "
                "false-positive/confounder assessment"
            ),

        "primary_metrics": [
            "false positive instances per image",
            "fraction of images with >=1 false positive",
            "maximum detection score distribution",
        ],

        "operating_thresholds": {
            "seed1":
                0.30,

            "seed2":
                0.35,

            "seed3":
                0.35,
        },

        "fixed_0_05_diagnostic":
            True,

        "threshold_tuning":
            False,
    },

    "aggregation": {
        "report_individual_seeds":
            True,

        "report_mean":
            True,

        "report_sample_sd":
            True,

        "sample_sd_denominator":
            "n-1",

        "n":
            3,

        "seed_ranking":
            False,

        "seed_exclusion_after_results":
            False,
    },

    "interpretation_rules": [
        (
            "Matched-mask Dice and matched-mask IoU are "
            "conditional on matched detections and must not be "
            "described as overall segmentation scores."
        ),

        (
            "AP50, AP75 and AP50:95 are the primary "
            "instance-segmentation metrics on the untouched "
            "synthetic test."
        ),

        (
            "Operating-point metrics use validation-derived "
            "thresholds frozen before test inference."
        ),

        (
            "Decode minimum score 0.05 is a candidate-retention "
            "floor rather than the primary operating threshold."
        ),

        (
            "No test result may trigger retraining, checkpoint "
            "selection, renderer modification, decoder modification "
            "or threshold tuning."
        ),

        (
            "All three precommitted final seeds must be reported."
        ),
    ],

    "external_evaluation_policy": {
        "rsc":
            (
                "External detection/box evaluation only; official "
                "250-image test primary and exact-dedup 247 "
                "sensitivity. No tuning."
            ),

        "tioman":
            (
                "Limited FTIR-linked external evidence only; do "
                "not imply all microscopy images are one-to-one "
                "chemically verified."
            ),

        "talanta":
            (
                "Qualitative food/smartphone transfer only because "
                "per-image instance ground truth is unavailable."
            ),

        "external_data_used_for_model_selection":
            False,
    },

    "critical_sha256":
        critical_sha256,

    "freeze_timing":
        (
            "Created after final three-seed synthetic-validation "
            "QC aggregation and before any untouched FruitMP-Syn "
            "test inference."
        ),
}


# ============================================================
# 17. CREATE FREEZE DIRECTORY
# ============================================================

if FREEZE_ROOT.exists():

    existing_files = [
        p
        for p in FREEZE_ROOT.rglob("*")
        if p.is_file()
    ]


    if existing_files:

        raise RuntimeError(
            "Formal evaluation protocol directory already contains "
            "files:\n"
            f"{FREEZE_ROOT}\n\n"
            "Do not overwrite an existing freeze."
        )


    shutil.rmtree(
        FREEZE_ROOT
    )


FREEZE_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


save_json(
    PROTOCOL_JSON,
    protocol,
)


# ============================================================
# 18. HUMAN-READABLE README
# ============================================================

readme_text = f"""
DASO-Net FORMAL EVALUATION PROTOCOL FREEZE v1
==============================================

STATUS
------
FROZEN BEFORE ANY UNTOUCHED TEST INFERENCE

FINAL MODELS
------------
Seed 1
  RNG: 1782656963
  successful optimizer updates: 2750
  operating threshold: 0.30

Seed 2
  RNG: 901395089
  successful optimizer updates: 2750
  operating threshold: 0.35

Seed 3
  RNG: 252916394
  successful optimizer updates: 2750
  operating threshold: 0.35

NO BEST-SEED SELECTION.

UNTOUCHED SYNTHETIC TEST
------------------------
FruitMP-Syn v1.1
split: test
scenes: 5000
subsampling: NONE

Fruit distribution:
  apple:      1000
  cucumber:   1000
  grape:      1000
  strawberry: 1000
  tomato:     1000

Primary metrics:
  mask AP50
  mask AP75
  mask AP50:95

Secondary:
  precision
  recall
  F1
  count MAE
  count bias
  zero-particle FP/image

Decoder:
  decode minimum score = 0.05
  top-k = 100
  mask threshold = 0.50

Any automatically computed test "best F1 threshold" is
POST-HOC descriptive sensitivity only.

It CANNOT replace:
  Seed 1 -> 0.30
  Seed 2 -> 0.35
  Seed 3 -> 0.35

HELD-OUT CLEAN FRUITBG
----------------------
Total: 1644
Train: 1151
Val:   247
Test:  246

Detected manifest split column:
  {fruitbg_split_col}

The 246 test backgrounds are frozen held-out clean data.

STRICT RULES
------------
- No test-set tuning.
- No threshold changes after viewing test.
- No checkpoint changes after viewing test.
- No best-seed selection.
- No seed exclusion.
- No renderer modification.
- No decoder modification.
- No retraining triggered by test results.
- Report all 3 final seeds.
- Report mean +/- sample SD.
""".strip() + "\n"


PROTOCOL_README.write_text(
    readme_text,
    encoding="utf-8",
)


# ============================================================
# 19. HASH FREEZE
# ============================================================

protocol_sha = sha256_file(
    PROTOCOL_JSON
)


PROTOCOL_SHA.write_text(
    f"{protocol_sha}  {PROTOCOL_JSON.name}\n",
    encoding="utf-8",
)


# ============================================================
# 20. READ-BACK AUDIT
# ============================================================

readback = load_json(
    PROTOCOL_JSON
)


if (
    readback[
        "status"
    ]
    != "FROZEN_BEFORE_TEST_INFERENCE"
):

    raise RuntimeError(
        "Protocol read-back status failure."
    )


if (
    int(
        readback[
            "untouched_synthetic_test"
        ][
            "num_scenes"
        ]
    )
    != 5000
):

    raise RuntimeError(
        "Protocol test-scene count failure."
    )


if (
    int(
        readback[
            "held_out_clean_fruitbg"
        ][
            "num_backgrounds"
        ]
    )
    != 246
):

    raise RuntimeError(
        "Protocol FruitBG test count failure."
    )


if bool(
    readback[
        "untouched_synthetic_test"
    ][
        "threshold_selection_using_test"
    ]
):

    raise RuntimeError(
        "Protocol incorrectly allows test threshold selection."
    )


if bool(
    readback[
        "final_seed_policy"
    ][
        "best_seed_selection"
    ]
):

    raise RuntimeError(
        "Protocol incorrectly permits best-seed selection."
    )


# ============================================================
# 21. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FORMAL EVALUATION PROTOCOL FROZEN"
)

print(
    "============================================================"
)

print()

print(
    "Untouched FruitMP-Syn test scenes: 5000"
)

print(
    "Held-out clean FruitBG images:     246"
)

print()

print(
    "FruitBG detected split column:"
)

print(
    f"  {fruitbg_split_col}"
)

print()

print(
    "Frozen operating thresholds:"
)

print(
    "  Seed 1: 0.30"
)

print(
    "  Seed 2: 0.35"
)

print(
    "  Seed 3: 0.35"
)

print()

print(
    "Decoder:"
)

print(
    "  decode minimum: 0.05"
)

print(
    "  top-k:          100"
)

print(
    "  mask threshold: 0.50"
)

print()

print(
    "Primary untouched-test metrics:"
)

print(
    "  Mask AP50"
)

print(
    "  Mask AP75"
)

print(
    "  Mask AP50:95"
)

print()

print(
    "Test-set threshold tuning: NO"
)

print(
    "Best-seed selection:       NO"
)

print(
    "Seed exclusion:            NO"
)

print(
    "All 3 seeds reported:      YES"
)

print()

print(
    "Protocol JSON:"
)

print(
    PROTOCOL_JSON
)

print()

print(
    "Protocol SHA256:"
)

print(
    protocol_sha
)

print()

print(
    "Protocol README:"
)

print(
    PROTOCOL_README
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET FORMAL EVALUATION PROTOCOL FREEZE: PASS"
)

print(
    "============================================================"
)

print()

print(
    "NO TRAINING WAS PERFORMED."
)

print(
    "NO INFERENCE WAS PERFORMED."
)

print(
    "NO TEST PREDICTIONS WERE OBSERVED."
)

print()

print(
    "STOP HERE."
)

print(
    "Next: untouched 5000-scene FruitMP-Syn test "
    "for all three final seeds."
)

DASO-Net — FORMAL EVALUATION PROTOCOL FREEZE v1
PASS  After-Seed3 restore audit
PASS  Formal three-seed QC aggregate
PASS  Untouched FruitMP-Syn test recipes
PASS  FruitMP-Syn full freeze manifest
PASS  FruitBG manifest
PASS  MP morphology manifest
PASS  Renderer config
PASS  Frozen project renderer
PASS  Validated evaluator
PASS  Validated decoder
PASS  Seed-1 terminal
PASS  Seed-2 terminal
PASS  Seed-3 terminal

After-Seed3 restore provenance: PASS
Three-seed aggregate SHA: PASS
No best-seed selection: PASS
Seed-1 terminal SHA: PASS
Seed-2 terminal SHA: PASS
Seed-3 terminal SHA: PASS
Validated decoder SHA: PASS

Untouched FruitMP-Syn test scenes: 5000 — PASS
Unique untouched test scene IDs: 5000 — PASS
No test inference has been performed by this cell.
Test fruit distribution:
  apple: 1000
  cucumber: 1000
  grape: 1000
  strawberry: 1000
  tomato: 1000

FruitBG split column detected:
  fruitbg_split

FruitBG frozen split counts:
  train: 1151
  val:    247
  test:   246
  total:  1

RuntimeError: Formal evaluation protocol directory already contains files:
/kaggle/working/dasonet_final_analysis/formal_evaluation_protocol_freeze_v1

Do not overwrite an existing freeze.

In [14]:
# ============================================================
# DASO-Net — FORMAL TEST ASSET AVAILABILITY AUDIT v1
#
# PURPOSE
# -------
# Diagnose the failed first untouched-test execution.
#
# Specifically determine whether the current Kaggle project
# input contains:
#   - FruitBG train assets
#   - FruitBG val assets
#   - FruitBG held-out test assets
#
# Also preserve evidence that:
#   - Seed-1 / Seed-2 formal inference failed before metrics
#   - no test-result tuning occurred
#
# NO TRAINING.
# NO INFERENCE.
# NO FILE DELETION.
# NO FROZEN DATA MODIFICATION.
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import re
from collections import Counter, defaultdict
from pathlib import Path


# ============================================================
# 1. ROOTS
# ============================================================

WORKING = Path(
    "/kaggle/working"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

PROJECT_DATASET_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project"
)

FRUITBG_MANIFEST = (
    PROJECT_ROOT
    / "data"
    / "fruit_bg"
    / "v1_final"
    / "fruitbg_rgb_v1_manifest.csv"
)

TEST_RECIPES = (
    PROJECT_ROOT
    / "data"
    / "fruitmp_syn"
    / "v1_1_recipes"
    / "fruitmp_syn_v1_1_test_recipes.csv"
)

SLIM_MANIFEST = (
    PROJECT_DATASET_ROOT
    / "PHASE5B_SLIM_BUNDLE_MANIFEST.json"
)

FORMAL_ROOT = (
    WORKING
    / "dasonet_final_evaluation"
    / "fruitmp_syn_test_v1"
)

SEED1_LOG = (
    FORMAL_ROOT
    / "seed01_evaluation.log"
)

SEED2_LOG = (
    FORMAL_ROOT
    / "seed02_evaluation.log"
)

SEED1_OUT = (
    FORMAL_ROOT
    / "seed01_s1782656963"
)

SEED2_OUT = (
    FORMAL_ROOT
    / "seed02_s901395089"
)

AUDIT_JSON = (
    WORKING
    / "DASO-Net-FORMAL-TEST-ASSET-AUDIT-v1.json"
)


# ============================================================
# 2. EXPECTED FROZEN COUNTS
# ============================================================

EXPECTED_TOTAL = 1644
EXPECTED_TRAIN = 1151
EXPECTED_VAL = 247
EXPECTED_TEST = 246

EXPECTED_TEST_RECIPES = 5000


# ============================================================
# 3. HELPERS
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
}


def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(
    path: Path,
    obj,
) -> None:

    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def read_csv(
    path: Path,
):

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def normalize_split(
    value,
):

    s = str(
        value
    ).strip().lower()


    mapping = {
        "train":
            "train",

        "training":
            "train",

        "val":
            "val",

        "valid":
            "val",

        "validation":
            "val",

        "dev":
            "val",

        "test":
            "test",

        "testing":
            "test",
    }


    return mapping.get(
        s
    )


def looks_like_image_path(
    value,
) -> bool:

    if value is None:
        return False

    text = str(
        value
    ).strip()


    if not text:
        return False


    suffix = Path(
        text.replace(
            "\\",
            "/",
        )
    ).suffix.lower()


    return (
        suffix
        in IMAGE_EXTENSIONS
    )


def candidate_current_paths(
    raw_value,
):
    """
    Translate a stored/historical path into plausible paths under
    the current frozen Kaggle project root.

    This does NOT modify anything.
    """

    text = str(
        raw_value
    ).strip().replace(
        "\\",
        "/",
    )


    candidates = []


    def add(
        p: Path,
    ):

        key = str(
            p
        )


        if key not in {
            str(x)
            for x in candidates
        }:

            candidates.append(
                p
            )


    # --------------------------------------------------------
    # 1. Literal path
    # --------------------------------------------------------

    literal = Path(
        text
    )


    if literal.is_absolute():

        add(
            literal
        )


    # --------------------------------------------------------
    # 2. Relative to project root
    # --------------------------------------------------------

    add(
        PROJECT_ROOT
        / text.lstrip("/")
    )


    # --------------------------------------------------------
    # 3. Relative to project data/
    # --------------------------------------------------------

    if not text.startswith(
        "data/"
    ):

        add(
            PROJECT_ROOT
            / "data"
            / text.lstrip("/")
        )


    # --------------------------------------------------------
    # 4. Strip historical prefix through DASO-Net-Phase1A/
    # --------------------------------------------------------

    marker = (
        "DASO-Net-Phase1A/"
    )


    if marker in text:

        suffix = text.split(
            marker,
            1,
        )[1]


        add(
            PROJECT_ROOT
            / suffix
        )


    # --------------------------------------------------------
    # 5. Map suffix beginning at /data/
    # --------------------------------------------------------

    data_marker = (
        "/data/"
    )


    if data_marker in text:

        suffix = (
            "data/"
            + text.split(
                data_marker,
                1,
            )[1]
        )


        add(
            PROJECT_ROOT
            / suffix
        )


    return candidates


def resolve_stored_path(
    raw_value,
):

    candidates = candidate_current_paths(
        raw_value
    )


    for path in candidates:

        if path.exists():

            return {
                "exists":
                    True,

                "resolved":
                    str(
                        path
                    ),

                "candidates":
                    [
                        str(p)
                        for p in candidates
                    ],
            }


    return {
        "exists":
            False,

        "resolved":
            None,

        "candidates":
            [
                str(p)
                for p in candidates
            ],
    }


def recursively_find_values(
    obj,
    needle,
    prefix="root",
):

    hits = []


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            current = (
                f"{prefix}.{key}"
            )


            if (
                needle.lower()
                in str(
                    key
                ).lower()
            ):

                hits.append(
                    (
                        current,
                        value,
                    )
                )


            hits.extend(
                recursively_find_values(
                    value,
                    needle,
                    current,
                )
            )


    elif isinstance(
        obj,
        list,
    ):

        for idx, value in enumerate(
            obj
        ):

            hits.extend(
                recursively_find_values(
                    value,
                    needle,
                    f"{prefix}[{idx}]",
                )
            )


    return hits


# ============================================================
# 4. PREFLIGHT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FORMAL TEST ASSET AVAILABILITY AUDIT v1"
)

print(
    "============================================================"
)


for path, label in [
    (
        PROJECT_ROOT,
        "Current frozen project root",
    ),
    (
        FRUITBG_MANIFEST,
        "FruitBG manifest",
    ),
    (
        TEST_RECIPES,
        "FruitMP-Syn untouched test recipes",
    ),
    (
        FORMAL_ROOT,
        "Preserved failed formal-test root",
    ),
    (
        SEED1_LOG,
        "Preserved Seed-1 failure log",
    ),
    (
        SEED2_LOG,
        "Preserved Seed-2 failure log",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 5. VERIFY NO COMPLETED FORMAL METRIC OUTPUT EXISTS
# ============================================================

completed_summaries = []


for idx, out_dir in [
    (
        1,
        SEED1_OUT,
    ),
    (
        2,
        SEED2_OUT,
    ),
]:

    metric_summary = (
        out_dir
        / "phase5c_metric_summary.json"
    )


    if metric_summary.exists():

        completed_summaries.append(
            str(
                metric_summary
            )
        )


if completed_summaries:

    raise RuntimeError(
        "Unexpected completed formal metric summary exists.\n"
        "STOP and inspect before proceeding:\n"
        + "\n".join(
            completed_summaries
        )
    )


print()
print(
    "Seed-1 completed formal metric summary: ABSENT — PASS"
)

print(
    "Seed-2 completed formal metric summary: ABSENT — PASS"
)

print(
    "No completed untouched-test metric result was produced."
)


# ============================================================
# 6. CONFIRM FAILURE MODE FROM LOGS
# ============================================================

logs = {
    "seed1":
        SEED1_LOG.read_text(
            encoding="utf-8",
            errors="replace",
        ),

    "seed2":
        SEED2_LOG.read_text(
            encoding="utf-8",
            errors="replace",
        ),
}


for name, text in logs.items():

    if (
        "FileNotFoundError"
        not in text
    ):

        raise RuntimeError(
            f"{name} log does not contain expected "
            "FileNotFoundError."
        )


    if (
        "fruit_public_raw"
        not in text
    ):

        raise RuntimeError(
            f"{name} log does not show missing "
            "fruit_public_raw asset."
        )


print()
print(
    "Seed-1 failure mode = missing FruitBG source asset: PASS"
)

print(
    "Seed-2 failure mode = missing FruitBG source asset: PASS"
)


# ============================================================
# 7. LOAD FROZEN FRUITBG MANIFEST
# ============================================================

rows = read_csv(
    FRUITBG_MANIFEST
)


if len(
    rows
) != EXPECTED_TOTAL:

    raise RuntimeError(
        "FruitBG manifest total mismatch.\n"
        f"Expected: {EXPECTED_TOTAL}\n"
        f"Actual:   {len(rows)}"
    )


if not rows:

    raise RuntimeError(
        "FruitBG manifest is empty."
    )


fields = list(
    rows[0].keys()
)


if (
    "fruitbg_split"
    not in fields
):

    raise RuntimeError(
        "Expected previously detected FruitBG split field "
        "'fruitbg_split' is absent."
    )


split_counts = Counter(
    normalize_split(
        row[
            "fruitbg_split"
        ]
    )
    for row in rows
)


if (
    split_counts[
        "train"
    ]
    != EXPECTED_TRAIN
    or split_counts[
        "val"
    ]
    != EXPECTED_VAL
    or split_counts[
        "test"
    ]
    != EXPECTED_TEST
):

    raise RuntimeError(
        "Frozen FruitBG split counts changed."
    )


print()
print(
    "FruitBG manifest:"
)

print(
    f"  total: {len(rows)}"
)

print(
    f"  train: {split_counts['train']}"
)

print(
    f"  val:   {split_counts['val']}"
)

print(
    f"  test:  {split_counts['test']}"
)

print(
    "Frozen FruitBG split counts: PASS"
)


# ============================================================
# 8. FIND ALL IMAGE-PATH COLUMNS IN FRUITBG MANIFEST
# ============================================================

path_columns = []


for field in fields:

    n_image_like = sum(
        1
        for row in rows
        if looks_like_image_path(
            row.get(
                field
            )
        )
    )


    if n_image_like > 0:

        path_columns.append(
            {
                "column":
                    field,

                "image_like_rows":
                    n_image_like,
            }
        )


if not path_columns:

    raise RuntimeError(
        "Could not identify any image-path-like fields "
        "in FruitBG manifest."
    )


print()
print(
    "Image-path-like FruitBG manifest columns:"
)


for record in path_columns:

    print(
        f"  {record['column']}: "
        f"{record['image_like_rows']} image-like values"
    )


# ============================================================
# 9. EXISTENCE AUDIT BY SPLIT AND PATH COLUMN
# ============================================================

column_audits = {}


for record in path_columns:

    column = record[
        "column"
    ]


    stats = {
        "train": {
            "total":
                0,

            "exists":
                0,

            "missing":
                0,
        },

        "val": {
            "total":
                0,

            "exists":
                0,

            "missing":
                0,
        },

        "test": {
            "total":
                0,

            "exists":
                0,

            "missing":
                0,
        },
    }


    missing_examples = {
        "train":
            [],

        "val":
            [],

        "test":
            [],
    }


    for row in rows:

        split = normalize_split(
            row[
                "fruitbg_split"
            ]
        )


        if split not in stats:
            continue


        value = row.get(
            column
        )


        if not looks_like_image_path(
            value
        ):
            continue


        stats[
            split
        ][
            "total"
        ] += 1


        resolution = resolve_stored_path(
            value
        )


        if resolution[
            "exists"
        ]:

            stats[
                split
            ][
                "exists"
            ] += 1

        else:

            stats[
                split
            ][
                "missing"
            ] += 1


            if len(
                missing_examples[
                    split
                ]
            ) < 10:

                missing_examples[
                    split
                ].append(
                    {
                        "stored_value":
                            str(
                                value
                            ),

                        "candidate_paths":
                            resolution[
                                "candidates"
                            ],
                    }
                )


    column_audits[
        column
    ] = {
        "stats":
            stats,

        "missing_examples":
            missing_examples,
    }


print()
print(
    "============================================================"
)

print(
    "FRUITBG ASSET EXISTENCE BY SPLIT"
)

print(
    "============================================================"
)


for column, result in column_audits.items():

    print()
    print(
        f"Column: {column}"
    )


    for split in [
        "train",
        "val",
        "test",
    ]:

        stat = result[
            "stats"
        ][
            split
        ]


        print(
            f"  {split:<5} "
            f"total={stat['total']:<5} "
            f"exists={stat['exists']:<5} "
            f"missing={stat['missing']:<5}"
        )


# ============================================================
# 10. IDENTIFY LIKELY RENDERER SOURCE-IMAGE COLUMN
#
# The useful column should strongly resemble:
#   train existing ~= 1151
#   val existing   ~= 247
#   test missing   ~= 246
#
# We do not silently assume it; we report candidates.
# ============================================================

likely_source_columns = []


for column, result in column_audits.items():

    stats = result[
        "stats"
    ]


    train_exists = stats[
        "train"
    ][
        "exists"
    ]

    val_exists = stats[
        "val"
    ][
        "exists"
    ]

    test_exists = stats[
        "test"
    ][
        "exists"
    ]

    test_missing = stats[
        "test"
    ][
        "missing"
    ]


    if (
        train_exists
        >= EXPECTED_TRAIN
        * 0.95
        and val_exists
        >= EXPECTED_VAL
        * 0.95
        and test_missing
        > 0
    ):

        likely_source_columns.append(
            column
        )


print()
print(
    "Likely renderer background-source columns:"
)

if likely_source_columns:

    for column in likely_source_columns:

        print(
            f"  {column}"
        )

else:

    print(
        "  NONE auto-identified"
    )


# ============================================================
# 11. PRINT TEST MISSING EXAMPLES
# ============================================================

print()
print(
    "============================================================"
)

print(
    "HELD-OUT TEST MISSING-ASSET EXAMPLES"
)

print(
    "============================================================"
)


printed = False


for column, result in column_audits.items():

    examples = result[
        "missing_examples"
    ][
        "test"
    ]


    if not examples:
        continue


    printed = True


    print()
    print(
        f"Column: {column}"
    )


    for idx, example in enumerate(
        examples[
            :5
        ],
        start=1,
    ):

        print()
        print(
            f"  Example {idx}:"
        )

        print(
            "    stored:"
        )

        print(
            f"      {example['stored_value']}"
        )

        print(
            "    attempted mappings:"
        )


        for candidate in example[
            "candidate_paths"
        ]:

            print(
                f"      {candidate}"
            )


if not printed:

    print(
        "No missing test-path examples found in path-like columns."
    )


# ============================================================
# 12. INSPECT TEST RECIPE SCHEMA
# ============================================================

test_rows = read_csv(
    TEST_RECIPES
)


if len(
    test_rows
) != EXPECTED_TEST_RECIPES:

    raise RuntimeError(
        "Untouched test recipe count changed."
    )


test_fields = list(
    test_rows[0].keys()
)


relevant_recipe_fields = [
    field
    for field in test_fields
    if any(
        token
        in str(
            field
        ).lower()
        for token in [
            "bg",
            "background",
            "fruit",
            "image",
            "path",
            "source",
            "id",
        ]
    )
]


print()
print(
    "============================================================"
)

print(
    "UNTOUCHED TEST RECIPE RELEVANT FIELDS"
)

print(
    "============================================================"
)


for field in relevant_recipe_fields:

    print(
        f"  {field}"
    )


print()
print(
    "First untouched-test recipe relevant values:"
)


first = test_rows[
    0
]


for field in relevant_recipe_fields:

    print(
        f"  {field}: {first.get(field)}"
    )


# ============================================================
# 13. INSPECT SLIM-BUNDLE MANIFEST
# ============================================================

slim_info = {
    "exists":
        SLIM_MANIFEST.exists(),

    "sha256":
        None,

    "fruit_related_hits":
        [],
}


if SLIM_MANIFEST.exists():

    slim_info[
        "sha256"
    ] = sha256_file(
        SLIM_MANIFEST
    )


    slim = load_json(
        SLIM_MANIFEST
    )


    hits = []


    for needle in [
        "fruit",
        "background",
        "1398",
        "train",
        "val",
        "test",
    ]:

        for path, value in recursively_find_values(
            slim,
            needle,
        ):

            text = str(
                value
            )


            if len(
                text
            ) > 300:

                text = (
                    text[
                        :300
                    ]
                    + "..."
                )


            hits.append(
                {
                    "needle":
                        needle,

                    "path":
                        path,

                    "value":
                        text,
                }
            )


    # Deduplicate
    seen = set()
    dedup = []


    for item in hits:

        key = (
            item[
                "path"
            ],
            item[
                "value"
            ],
        )


        if key in seen:
            continue


        seen.add(
            key
        )

        dedup.append(
            item
        )


    slim_info[
        "fruit_related_hits"
    ] = dedup[
        :50
    ]


    print()
    print(
        "============================================================"
    )

    print(
        "PHASE5B SLIM-BUNDLE MANIFEST"
    )

    print(
        "============================================================"
    )

    print(
        f"Manifest: {SLIM_MANIFEST}"
    )

    print(
        f"SHA256:   {slim_info['sha256']}"
    )


    for item in slim_info[
        "fruit_related_hits"
    ]:

        print(
            f"  {item['path']} = {item['value']}"
        )

else:

    print()
    print(
        "PHASE5B slim-bundle manifest not found at expected "
        "top-level path."
    )


# ============================================================
# 14. SUMMARIZE TEST ASSET DEFICIT
# ============================================================

test_deficits = {}


for column, result in column_audits.items():

    stats = result[
        "stats"
    ][
        "test"
    ]


    if stats[
        "missing"
    ] > 0:

        test_deficits[
            column
        ] = {
            "test_total_path_values":
                stats[
                    "total"
                ],

            "test_existing":
                stats[
                    "exists"
                ],

            "test_missing":
                stats[
                    "missing"
                ],
        }


# ============================================================
# 15. WRITE AUDIT ARTIFACT
# ============================================================

audit = {
    "audit_name":
        "DASO-Net formal test asset availability audit v1",

    "status":
        "PASS",

    "scientific_status":
        (
            "Infrastructure/data-availability failure before "
            "formal metric completion; not a model-performance "
            "result."
        ),

    "formal_test_attempt": {
        "seed1_process_failed":
            True,

        "seed2_process_failed":
            True,

        "seed1_completed_metric_summary":
            False,

        "seed2_completed_metric_summary":
            False,

        "seed3_started":
            False,

        "failure_class":
            "FileNotFoundError",

        "failure_resource":
            "FruitBG / fruit_public_raw source image asset",

        "test_threshold_tuning_performed":
            False,

        "best_seed_selection_performed":
            False,
    },

    "fruitbg_manifest": {
        "path":
            str(
                FRUITBG_MANIFEST
            ),

        "sha256":
            sha256_file(
                FRUITBG_MANIFEST
            ),

        "total":
            len(
                rows
            ),

        "train":
            split_counts[
                "train"
            ],

        "val":
            split_counts[
                "val"
            ],

        "test":
            split_counts[
                "test"
            ],
    },

    "path_columns":
        path_columns,

    "column_asset_audits":
        column_audits,

    "likely_renderer_source_columns":
        likely_source_columns,

    "test_asset_deficits":
        test_deficits,

    "untouched_test_recipes": {
        "count":
            len(
                test_rows
            ),

        "sha256":
            sha256_file(
                TEST_RECIPES
            ),
    },

    "slim_bundle_manifest":
        slim_info,

    "policy": {
        "training_performed":
            False,

        "inference_metrics_completed":
            False,

        "frozen_data_modified":
            False,

        "failed_outputs_deleted":
            False,

        "rerun_authorized":
            False,
    },
}


save_json(
    AUDIT_JSON,
    audit,
)


# ============================================================
# 16. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FORMAL TEST ASSET AUDIT COMPLETE"
)

print(
    "============================================================"
)

print()

print(
    "Formal Seed-1 metric result completed: NO"
)

print(
    "Formal Seed-2 metric result completed: NO"
)

print(
    "Formal Seed-3 inference started:       NO"
)

print()

print(
    "Failure class:"
)

print(
    "  missing frozen FruitBG source asset"
)

print()

print(
    "FruitBG:"
)

print(
    f"  train = {split_counts['train']}"
)

print(
    f"  val   = {split_counts['val']}"
)

print(
    f"  test  = {split_counts['test']}"
)

print()

print(
    "Test-asset deficits detected:"
)


if test_deficits:

    for column, info in test_deficits.items():

        print(
            f"  {column}: "
            f"existing={info['test_existing']} "
            f"missing={info['test_missing']} "
            f"of {info['test_total_path_values']}"
        )

else:

    print(
        "  NONE in manifest path columns"
    )


print()

print(
    "Likely renderer source columns:"
)

if likely_source_columns:

    for column in likely_source_columns:

        print(
            f"  {column}"
        )

else:

    print(
        "  none auto-identified"
    )


print()

print(
    "Audit artifact:"
)

print(
    AUDIT_JSON
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET FORMAL TEST ASSET AVAILABILITY AUDIT: PASS"
)

print(
    "============================================================"
)

print()

print(
    "NO TRAINING WAS PERFORMED."
)

print(
    "NO COMPLETED FORMAL TEST METRICS WERE PRODUCED."
)

print(
    "NO FROZEN DATA WERE MODIFIED."
)

print(
    "FAILED LOGS/OUTPUT DIRECTORIES WERE PRESERVED."
)

print()

print(
    "STOP HERE."
)

print(
    "Next: provide the missing frozen held-out FruitBG test "
    "assets, then resume the formal test without changing "
    "the frozen protocol."
)

DASO-Net — FORMAL TEST ASSET AVAILABILITY AUDIT v1
PASS  Current frozen project root
PASS  FruitBG manifest
PASS  FruitMP-Syn untouched test recipes
PASS  Preserved failed formal-test root
PASS  Preserved Seed-1 failure log
PASS  Preserved Seed-2 failure log

Seed-1 completed formal metric summary: ABSENT — PASS
Seed-2 completed formal metric summary: ABSENT — PASS
No completed untouched-test metric result was produced.

Seed-1 failure mode = missing FruitBG source asset: PASS
Seed-2 failure mode = missing FruitBG source asset: PASS

FruitBG manifest:
  total: 1644
  train: 1151
  val:   247
  test:  246
Frozen FruitBG split counts: PASS

Image-path-like FruitBG manifest columns:
  source_relative_path: 1644 image-like values
  source_absolute_path: 1644 image-like values
  surface_mask_path: 1644 image-like values
  phase2d_surface_mask_path: 1644 image-like values
  frozen_surface_mask_path: 1644 image-like values

FRUITBG ASSET EXISTENCE BY SPLIT

Column: source_relative_path
  trai

In [16]:
# ============================================================
# DASO-Net — FORMAL TEST RESUME 01B
# CORRECT SURFACE-ID AUDIT + FINALIZE OVERLAY + INFERENCE
#
# Previous Resume01 stopped BEFORE inference because an
# infrastructure audit assumed the wrong FruitBG ID column.
#
# Already verified in Resume01:
#   - 492 mounted supplemental files SHA256 PASS
#   - 246 source RGB present in overlay
#   - 246 frozen masks present in overlay
#   - frozen scientific hashes PASS
#
# This cell:
#   1. preserves the partial overlay from Resume01
#   2. re-verifies its scientific integrity
#   3. determines the recipe->FruitBG ID representation by
#      exact data matching, without changing any frozen file
#   4. verifies all 5000 recipes reference held-out TEST rows
#   5. atomically finalizes the overlay
#   6. runs the same frozen formal evaluation:
#          Seed 1 -> 0.30
#          Seed 2 -> 0.35
#          Seed 3 -> 0.35
#
# NO TRAINING.
# NO THRESHOLD TUNING.
# NO MODEL SELECTION.
# NO TEST-SET ADAPTATION.
# NO FROZEN SCIENTIFIC FILE MODIFICATION.
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import shlex
import subprocess
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path


# ============================================================
# 1. FROZEN IDENTITIES
# ============================================================

EXPECTED_PROTOCOL_SHA = (
    "a98ac6c5af42c7dc84bc7e7c00b7a9ed"
    "661d45af6882bdb0029c49b8d0cb5b62"
)

EXPECTED_ASSET_PACKAGE_ZIP_SHA = (
    "7a365d1b9e89e1dee2d0c4677a87458a"
    "390e95ed19952fa48b9f809e024281e4"
)

EXPECTED_FRUITBG_MANIFEST_SHA = (
    "d0108ac720cb03dd565cc6c1fb792762"
    "e7afea808ee596f4b7b2be82e2f223fe"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)

EXPECTED_SEED1_SHA = (
    "e4c4b79ad4b8c6e18ba2b1e3d6cbe796"
    "c9cc301bd6fcc7c596fc290ad3a7e81d"
)

EXPECTED_SEED2_SHA = (
    "19cdb834a46240916b2c8023e8c3afe573"
    "8d9c7cd5d8f818c62ad1e289df1ad2"
)

EXPECTED_SEED3_SHA = (
    "be7b7941f2f62179e15074aa973bf6da0"
    "eb9018ec9133486512bd1f87153f447"
)

EXPECTED_TEST_SCENES = 5000

EXPECTED_FRUITBG_TOTAL = 1644
EXPECTED_FRUITBG_TRAIN = 1151
EXPECTED_FRUITBG_VAL = 247
EXPECTED_FRUITBG_TEST = 246

EXPECTED_SUPPLEMENTAL_RGB = 246
EXPECTED_SUPPLEMENTAL_MASKS = 246
EXPECTED_SUPPLEMENTAL_FILES = 492


# ============================================================
# 2. ROOTS
# ============================================================

WORKING = Path(
    "/kaggle/working"
)

BASE_PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CODE_ROOT = (
    WORKING
    / "DASO-Net-Phase5B-KaggleTraining"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

PROTOCOL_JSON = (
    WORKING
    / "dasonet_final_analysis"
    / "formal_evaluation_protocol_freeze_v1"
    / "FORMAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)


# ============================================================
# 3. FINAL CHECKPOINTS
# ============================================================

SEED1_CHECKPOINT = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

SEED2_CHECKPOINT = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

SEED3_CHECKPOINT = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed03_s252916394"
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)


# ============================================================
# 4. ORIGINAL FAILED FORMAL ATTEMPT
# ============================================================

FAILED_FORMAL_ROOT = (
    WORKING
    / "dasonet_final_evaluation"
    / "fruitmp_syn_test_v1"
)

FAILED_SEED1_LOG = (
    FAILED_FORMAL_ROOT
    / "seed01_evaluation.log"
)

FAILED_SEED2_LOG = (
    FAILED_FORMAL_ROOT
    / "seed02_evaluation.log"
)

FAILED_SEED1_SUMMARY = (
    FAILED_FORMAL_ROOT
    / "seed01_s1782656963"
    / "phase5c_metric_summary.json"
)

FAILED_SEED2_SUMMARY = (
    FAILED_FORMAL_ROOT
    / "seed02_s901395089"
    / "phase5c_metric_summary.json"
)

ASSET_FAILURE_AUDIT = (
    WORKING
    / "DASO-Net-FORMAL-TEST-ASSET-AUDIT-v1.json"
)


# ============================================================
# 5. SUPPLEMENTAL DATASET
# ============================================================

SUPPLEMENT_DATASET_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-fruitbg-test-assets-v1"
)

SUPPLEMENT_SHA_RECORD = (
    SUPPLEMENT_DATASET_ROOT
    / "DASO-Net-FruitBG-TestAssets-v1.sha256.txt"
)

SUPPLEMENT_TOP_STATE = (
    SUPPLEMENT_DATASET_ROOT
    / "FRUITBG_TEST_ASSETS_STATE.json"
)


# ============================================================
# 6. PARTIAL / FINAL OVERLAY
# ============================================================

OVERLAY_BUILD_ROOT = (
    WORKING
    / "DASO-Net-Phase1A-FORMALTEST-OVERLAY-v1.building"
)

OVERLAY_ROOT = (
    WORKING
    / "DASO-Net-Phase1A-FORMALTEST-OVERLAY-v1"
)

OVERLAY_AUDIT = (
    WORKING
    / "DASO-Net-FORMALTEST-OVERLAY-AUDIT-v1.json"
)


# ============================================================
# 7. RESUMED FORMAL OUTPUT
# ============================================================

RESUME_ROOT = (
    WORKING
    / "dasonet_final_evaluation"
    / "fruitmp_syn_test_v1_resume01_assets_complete"
)

RESUME_MANIFEST = (
    RESUME_ROOT
    / "FORMAL_FRUITMP_SYN_TEST_RUN_MANIFEST_v1_RESUME01.json"
)


# ============================================================
# 8. FIXED EVALUATION CONTRACT
# ============================================================

DECODE_MIN_SCORE = 0.05
TOPK = 100
MASK_THRESHOLD = 0.50
EVALUATION_SEED = 20260917


SEEDS = {
    1: {
        "rng_seed":
            1782656963,

        "checkpoint":
            SEED1_CHECKPOINT,

        "checkpoint_sha256":
            EXPECTED_SEED1_SHA,

        "operating_threshold":
            0.30,

        "gpu":
            0,
    },

    2: {
        "rng_seed":
            901395089,

        "checkpoint":
            SEED2_CHECKPOINT,

        "checkpoint_sha256":
            EXPECTED_SEED2_SHA,

        "operating_threshold":
            0.35,

        "gpu":
            1,
    },

    3: {
        "rng_seed":
            252916394,

        "checkpoint":
            SEED3_CHECKPOINT,

        "checkpoint_sha256":
            EXPECTED_SEED3_SHA,

        "operating_threshold":
            0.35,

        "gpu":
            0,
    },
}


for idx in [
    1,
    2,
    3,
]:

    SEEDS[idx]["output_dir"] = (
        RESUME_ROOT
        / f"seed0{idx}_s{SEEDS[idx]['rng_seed']}"
    )

    SEEDS[idx]["summary_json"] = (
        SEEDS[idx]["output_dir"]
        / "phase5c_metric_summary.json"
    )

    SEEDS[idx]["per_scene_csv"] = (
        SEEDS[idx]["output_dir"]
        / "phase5c_per_scene.csv"
    )

    SEEDS[idx]["log"] = (
        RESUME_ROOT
        / f"seed0{idx}_evaluation.log"
    )


# ============================================================
# 9. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open("rb") as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(block)

    return h.hexdigest()


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(
    path: Path,
    obj,
) -> None:

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def read_csv(
    path: Path,
):

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:

        return list(
            csv.DictReader(f)
        )


def normalize_split(
    value,
):

    s = str(
        value
    ).strip().lower()

    mapping = {
        "train":
            "train",

        "training":
            "train",

        "val":
            "val",

        "valid":
            "val",

        "validation":
            "val",

        "dev":
            "val",

        "test":
            "test",

        "testing":
            "test",
    }

    return mapping.get(s)


def normalize_path_text(
    value,
) -> str:

    return str(
        value
    ).strip().replace(
        "\\",
        "/",
    )


def project_rel_from_source_absolute(
    value,
) -> Path:

    text = normalize_path_text(
        value
    )

    marker = (
        "DASO-Net-Phase1A/"
    )

    if marker not in text:

        raise RuntimeError(
            "Could not resolve source_absolute_path:\n"
            f"{text}"
        )

    return Path(
        text.split(
            marker,
            1,
        )[1]
    )


def detect_column(
    fields,
    candidates,
):

    lookup = {
        str(field).strip().lower():
            field
        for field in fields
        if field is not None
    }

    for candidate in candidates:

        key = str(
            candidate
        ).strip().lower()

        if key in lookup:

            return lookup[key]

    return None


def find_scene_column(
    rows,
):

    if not rows:
        return None

    return detect_column(
        list(
            rows[0].keys()
        ),
        [
            "scene_id",
            "scene",
            "recipe_id",
            "sample_id",
            "id",
        ],
    )


def path_stem(
    value,
) -> str:

    text = normalize_path_text(
        value
    )

    if not text:
        return ""

    return Path(
        text
    ).stem


def tail_text(
    path: Path,
    lines: int = 35,
) -> str:

    if not path.exists():

        return "(log missing)"

    data = path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    return "\n".join(
        data[-lines:]
    )


# ============================================================
# 10. PREFLIGHT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FORMAL TEST RESUME 01B"
)

print(
    "CORRECT SURFACE-ID AUDIT + FINALIZE + INFERENCE"
)

print(
    "============================================================"
)


for path, label in [
    (
        BASE_PROJECT_ROOT,
        "Phase5B slim frozen project",
    ),
    (
        EVALUATOR,
        "Validated evaluator",
    ),
    (
        DECODER,
        "Validated decoder",
    ),
    (
        PROTOCOL_JSON,
        "Frozen evaluation protocol",
    ),
    (
        SEED1_CHECKPOINT,
        "Seed-1 terminal",
    ),
    (
        SEED2_CHECKPOINT,
        "Seed-2 terminal",
    ),
    (
        SEED3_CHECKPOINT,
        "Seed-3 terminal",
    ),
    (
        FAILED_FORMAL_ROOT,
        "Original failed formal-test root",
    ),
    (
        FAILED_SEED1_LOG,
        "Original failed Seed-1 log",
    ),
    (
        FAILED_SEED2_LOG,
        "Original failed Seed-2 log",
    ),
    (
        ASSET_FAILURE_AUDIT,
        "Asset failure audit",
    ),
    (
        SUPPLEMENT_DATASET_ROOT,
        "FruitBG supplemental dataset",
    ),
    (
        SUPPLEMENT_SHA_RECORD,
        "Supplement SHA record",
    ),
    (
        SUPPLEMENT_TOP_STATE,
        "Supplement state",
    ),
    (
        OVERLAY_BUILD_ROOT,
        "Resume01 partial overlay",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


if OVERLAY_ROOT.exists():

    raise RuntimeError(
        "Final overlay already exists unexpectedly:\n"
        f"{OVERLAY_ROOT}"
    )


if OVERLAY_AUDIT.exists():

    raise RuntimeError(
        "Overlay audit already exists unexpectedly:\n"
        f"{OVERLAY_AUDIT}"
    )


if RESUME_ROOT.exists():

    raise RuntimeError(
        "Formal Resume01 output already exists:\n"
        f"{RESUME_ROOT}"
    )


# ============================================================
# 11. ORIGINAL FAILED ATTEMPT STILL HAS NO RESULTS
# ============================================================

if FAILED_SEED1_SUMMARY.exists():

    raise RuntimeError(
        "Original Seed-1 failed attempt unexpectedly has "
        "a metric summary."
    )


if FAILED_SEED2_SUMMARY.exists():

    raise RuntimeError(
        "Original Seed-2 failed attempt unexpectedly has "
        "a metric summary."
    )


for label, log in [
    (
        "Seed-1",
        FAILED_SEED1_LOG,
    ),
    (
        "Seed-2",
        FAILED_SEED2_LOG,
    ),
]:

    text = log.read_text(
        encoding="utf-8",
        errors="replace",
    )

    if (
        "FileNotFoundError"
        not in text
        or "fruit_public_raw"
        not in text
    ):

        raise RuntimeError(
            f"{label} preserved failure provenance mismatch."
        )


print()
print(
    "Original failed attempt remains non-result-producing: PASS"
)

print(
    "No untouched-test metric was observed in that attempt."
)


# ============================================================
# 12. PROTOCOL / CHECKPOINT / DECODER AUDIT
# ============================================================

protocol_sha = sha256_file(
    PROTOCOL_JSON
)


if protocol_sha != EXPECTED_PROTOCOL_SHA:

    raise RuntimeError(
        "Frozen protocol SHA mismatch."
    )


protocol = load_json(
    PROTOCOL_JSON
)


if (
    protocol.get(
        "status"
    )
    != "FROZEN_BEFORE_TEST_INFERENCE"
):

    raise RuntimeError(
        "Frozen protocol status changed."
    )


if bool(
    protocol[
        "untouched_synthetic_test"
    ][
        "threshold_selection_using_test"
    ]
):

    raise RuntimeError(
        "Frozen protocol now permits test threshold tuning."
    )


if bool(
    protocol[
        "final_seed_policy"
    ][
        "best_seed_selection"
    ]
):

    raise RuntimeError(
        "Frozen protocol now permits best-seed selection."
    )


print()
print(
    "Frozen protocol SHA: PASS"
)


for idx in [
    1,
    2,
    3,
]:

    record = SEEDS[idx]

    if (
        sha256_file(
            record["checkpoint"]
        )
        != record[
            "checkpoint_sha256"
        ]
    ):

        raise RuntimeError(
            f"Seed-{idx} checkpoint SHA mismatch."
        )

    protocol_seed = (
        protocol[
            "final_seed_policy"
        ][
            "seeds"
        ][
            f"seed{idx}"
        ]
    )

    if int(
        protocol_seed[
            "rng_seed"
        ]
    ) != int(
        record[
            "rng_seed"
        ]
    ):

        raise RuntimeError(
            f"Seed-{idx} RNG mismatch."
        )

    if float(
        protocol_seed[
            "operating_threshold"
        ]
    ) != float(
        record[
            "operating_threshold"
        ]
    ):

        raise RuntimeError(
            f"Seed-{idx} threshold mismatch."
        )

    print(
        f"Seed-{idx} checkpoint + frozen threshold: PASS"
    )


if sha256_file(
    DECODER
) != EXPECTED_DECODER_SHA:

    raise RuntimeError(
        "Decoder SHA mismatch."
    )


print(
    "Validated decoder SHA: PASS"
)


# ============================================================
# 13. SUPPLEMENT PACKAGE AUDIT
# ============================================================

sha_record = SUPPLEMENT_SHA_RECORD.read_text(
    encoding="utf-8",
).strip()


if (
    EXPECTED_ASSET_PACKAGE_ZIP_SHA
    not in sha_record
):

    raise RuntimeError(
        "Supplement ZIP SHA provenance mismatch."
    )


state = load_json(
    SUPPLEMENT_TOP_STATE
)


if state.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Supplement package state != PASS."
    )


internal_hits = list(
    SUPPLEMENT_DATASET_ROOT.rglob(
        "FRUITBG_TEST_ASSETS_FILE_MANIFEST.json"
    )
)


if len(
    internal_hits
) != 1:

    raise RuntimeError(
        "Expected exactly one internal supplement manifest; "
        f"found {len(internal_hits)}."
    )


SUPPLEMENT_FILE_MANIFEST = (
    internal_hits[0]
)

SUPPLEMENT_PACKAGE_ROOT = (
    SUPPLEMENT_FILE_MANIFEST.parent
)

supp_manifest = load_json(
    SUPPLEMENT_FILE_MANIFEST
)

file_records = supp_manifest.get(
    "files",
    [],
)


if len(
    file_records
) != 492:

    raise RuntimeError(
        "Supplement manifest does not contain 492 file records."
    )


print()
print(
    "Re-verifying 492 supplemental payload files..."
)


for n, record in enumerate(
    file_records,
    start=1,
):

    package_rel = Path(
        record[
            "relative_path"
        ]
    )

    mounted = (
        SUPPLEMENT_PACKAGE_ROOT
        / package_rel
    )

    require(
        mounted,
        f"Supplement file {package_rel}",
    )

    if (
        mounted.stat().st_size
        != int(
            record[
                "size_bytes"
            ]
        )
    ):

        raise RuntimeError(
            "Supplement payload size mismatch:\n"
            f"{mounted}"
        )

    if (
        sha256_file(
            mounted
        )
        != record[
            "sha256"
        ]
    ):

        raise RuntimeError(
            "Supplement payload SHA mismatch:\n"
            f"{mounted}"
        )

    if (
        n % 100 == 0
        or n == 492
    ):

        print(
            f"  verified {n}/492"
        )


print(
    "Supplemental 492-file SHA audit: PASS"
)


# ============================================================
# 14. RE-VERIFY PARTIAL OVERLAY CONTAINS EXACT SUPPLEMENT BYTES
# ============================================================

for record in file_records:

    package_rel = Path(
        record[
            "relative_path"
        ]
    )

    parts = package_rel.parts


    if (
        not parts
        or parts[0]
        != "DASO-Net-Phase1A"
    ):

        raise RuntimeError(
            "Unexpected supplemental relative path:\n"
            f"{package_rel}"
        )


    project_rel = Path(
        *parts[1:]
    )

    overlay_file = (
        OVERLAY_BUILD_ROOT
        / project_rel
    )


    require(
        overlay_file,
        f"Partial-overlay supplemental file {project_rel}",
    )


    if (
        sha256_file(
            overlay_file
        )
        != record[
            "sha256"
        ]
    ):

        raise RuntimeError(
            "Partial overlay supplemental SHA mismatch:\n"
            f"{overlay_file}"
        )


print()
print(
    "Partial overlay contains exact 492 supplemental files: PASS"
)


# ============================================================
# 15. FROZEN FILES INSIDE PARTIAL OVERLAY
# ============================================================

FRUITBG_MANIFEST = (
    OVERLAY_BUILD_ROOT
    / "data"
    / "fruit_bg"
    / "v1_final"
    / "fruitbg_rgb_v1_manifest.csv"
)

TEST_RECIPES = (
    OVERLAY_BUILD_ROOT
    / "data"
    / "fruitmp_syn"
    / "v1_1_recipes"
    / "fruitmp_syn_v1_1_test_recipes.csv"
)

OVERLAY_RENDERER = (
    OVERLAY_BUILD_ROOT
    / "src"
    / "dasonet"
    / "synth"
    / "renderer.py"
)

OVERLAY_RENDERER_CONFIG = (
    OVERLAY_BUILD_ROOT
    / "configs"
    / "renderer_v3_1.json"
)


for path, label in [
    (
        FRUITBG_MANIFEST,
        "Partial-overlay FruitBG manifest",
    ),
    (
        TEST_RECIPES,
        "Partial-overlay test recipes",
    ),
    (
        OVERLAY_RENDERER,
        "Partial-overlay renderer",
    ),
    (
        OVERLAY_RENDERER_CONFIG,
        "Partial-overlay renderer config",
    ),
]:

    require(
        path,
        label,
    )


if (
    sha256_file(
        FRUITBG_MANIFEST
    )
    != EXPECTED_FRUITBG_MANIFEST_SHA
):

    raise RuntimeError(
        "FruitBG manifest SHA changed."
    )


critical_hashes = protocol[
    "critical_sha256"
]


if (
    sha256_file(
        TEST_RECIPES
    )
    != critical_hashes[
        "fruitmp_syn_test_recipes"
    ]
):

    raise RuntimeError(
        "Test recipes SHA changed."
    )


if (
    sha256_file(
        OVERLAY_RENDERER
    )
    != critical_hashes[
        "project_renderer"
    ]
):

    raise RuntimeError(
        "Renderer SHA changed."
    )


if (
    sha256_file(
        OVERLAY_RENDERER_CONFIG
    )
    != critical_hashes[
        "renderer_config"
    ]
):

    raise RuntimeError(
        "Renderer config SHA changed."
    )


print(
    "Partial-overlay frozen scientific hashes: PASS"
)


# ============================================================
# 16. LOAD FROZEN FRUITBG + TEST RECIPES
# ============================================================

fruitbg_rows = read_csv(
    FRUITBG_MANIFEST
)

recipe_rows = read_csv(
    TEST_RECIPES
)


if len(
    fruitbg_rows
) != EXPECTED_FRUITBG_TOTAL:

    raise RuntimeError(
        "FruitBG row count != 1644."
    )


if len(
    recipe_rows
) != EXPECTED_TEST_SCENES:

    raise RuntimeError(
        "Test recipe count != 5000."
    )


fruitbg_fields = list(
    fruitbg_rows[0].keys()
)

recipe_fields = list(
    recipe_rows[0].keys()
)


if (
    "fruitbg_split"
    not in fruitbg_fields
):

    raise RuntimeError(
        "FruitBG split field missing."
    )


split_counts = Counter(
    normalize_split(
        row[
            "fruitbg_split"
        ]
    )
    for row in fruitbg_rows
)


if (
    split_counts["train"]
    != 1151
    or split_counts["val"]
    != 247
    or split_counts["test"]
    != 246
):

    raise RuntimeError(
        "Frozen FruitBG split counts changed."
    )


test_bg_rows = [
    row
    for row in fruitbg_rows
    if normalize_split(
        row[
            "fruitbg_split"
        ]
    )
    == "test"
]


recipe_bg_col = detect_column(
    recipe_fields,
    [
        "background_surface_id",
        "fruitbg_surface_id",
        "surface_id",
    ],
)


if recipe_bg_col is None:

    raise RuntimeError(
        "Could not locate recipe background surface-ID field."
    )


recipe_surface_ids = [
    str(
        row[
            recipe_bg_col
        ]
    ).strip()
    for row in recipe_rows
]


recipe_surface_set = set(
    recipe_surface_ids
)


print()
print(
    "Recipe background ID field:"
)

print(
    f"  {recipe_bg_col}"
)

print(
    f"Unique recipe background IDs: {len(recipe_surface_set)}"
)


# ============================================================
# 17. DETERMINE CORRECT FRUITBG IDENTIFIER REPRESENTATION
#
# Do not assume that a manifest field called "surface_id" uses
# the same namespace as recipe "background_surface_id".
#
# Test every direct manifest column and every path-stem
# representation. A valid representation must map every recipe
# background ID to exactly one held-out FruitBG test row.
# ============================================================

representations = {}


for field in fruitbg_fields:

    direct_values = [
        str(
            row.get(
                field,
                "",
            )
        ).strip()
        for row in test_bg_rows
    ]


    representations[
        f"column:{field}"
    ] = direct_values


    stem_values = [
        path_stem(
            row.get(
                field,
                "",
            )
        )
        for row in test_bg_rows
    ]


    if any(
        value
        for value in stem_values
    ):

        representations[
            f"stem:{field}"
        ] = stem_values


valid_representations = {}


for name, values in representations.items():

    value_to_rows = defaultdict(
        list
    )


    for idx, value in enumerate(
        values
    ):

        if value:

            value_to_rows[
                value
            ].append(
                idx
            )


    missing_ids = [
        item
        for item in recipe_surface_set
        if item not in value_to_rows
    ]


    ambiguous_ids = [
        item
        for item in recipe_surface_set
        if len(
            value_to_rows.get(
                item,
                [],
            )
        ) != 1
    ]


    matched_unique = (
        len(
            recipe_surface_set
        )
        - len(
            missing_ids
        )
    )


    if (
        not missing_ids
        and not ambiguous_ids
    ):

        manifest_value_set = {
            value
            for value in values
            if value
        }


        valid_representations[
            name
        ] = {
            "values":
                values,

            "value_to_rows":
                value_to_rows,

            "manifest_unique_values":
                len(
                    manifest_value_set
                ),

            "exact_set_equality":
                (
                    manifest_value_set
                    == recipe_surface_set
                ),
        }


print()
print(
    "============================================================"
)

print(
    "FRUITBG / RECIPE ID-NAMESPACE AUDIT"
)

print(
    "============================================================"
)


if not valid_representations:

    # Print useful coverage before stopping.
    coverage = []


    for name, values in representations.items():

        value_set = {
            v
            for v in values
            if v
        }

        overlap = len(
            recipe_surface_set
            & value_set
        )

        if overlap > 0:

            coverage.append(
                (
                    overlap,
                    name,
                    len(
                        value_set
                    ),
                )
            )


    coverage.sort(
        reverse=True
    )


    print(
        "Highest-overlap representations:"
    )


    for overlap, name, count in coverage[
        :20
    ]:

        print(
            f"  {name:<45} "
            f"overlap={overlap} "
            f"manifest_unique={count}"
        )


    raise RuntimeError(
        "No manifest representation uniquely covers every "
        "recipe background_surface_id."
    )


for name, info in valid_representations.items():

    print(
        f"PASS  {name}"
        f"  unique={info['manifest_unique_values']}"
        f"  exact_set={info['exact_set_equality']}"
    )


# ============================================================
# 18. CHOOSE CANONICAL REPRESENTATION
#
# Prefer the frozen v1_final mask stem because this is an
# immutable per-surface artifact in the exact held-out bundle.
#
# Fall back only to another representation that independently
# gives complete one-to-one coverage.
# ============================================================

preference = [
    "stem:frozen_surface_mask_path",
    "column:background_surface_id",
    "column:fruitbg_surface_id",
    "column:surface_id",
    "stem:phase2d_surface_mask_path",
    "stem:surface_mask_path",
]


canonical_representation = None


for candidate in preference:

    if candidate in valid_representations:

        canonical_representation = candidate

        break


if canonical_representation is None:

    canonical_representation = sorted(
        valid_representations.keys()
    )[0]


canonical_info = (
    valid_representations[
        canonical_representation
    ]
)


print()
print(
    "Canonical audit representation:"
)

print(
    f"  {canonical_representation}"
)


# ============================================================
# 19. VERIFY ALL 5000 RECIPES MAP TO HELD-OUT TEST ROWS
# ============================================================

value_to_rows = canonical_info[
    "value_to_rows"
]


recipe_to_manifest_row = []


for recipe_index, recipe_id in enumerate(
    recipe_surface_ids
):

    matches = value_to_rows.get(
        recipe_id,
        [],
    )


    if len(
        matches
    ) != 1:

        raise RuntimeError(
            "Recipe background ID does not uniquely map to "
            "a held-out FruitBG test row:\n"
            f"Recipe index: {recipe_index}\n"
            f"ID:           {recipe_id}\n"
            f"Matches:      {matches}"
        )


    recipe_to_manifest_row.append(
        matches[0]
    )


if len(
    recipe_to_manifest_row
) != 5000:

    raise RuntimeError(
        "Did not map exactly 5000 recipes."
    )


print()
print(
    "All 5000 recipes uniquely map to held-out FruitBG "
    "TEST rows: PASS"
)

print(
    f"Unique held-out surfaces used: {len(recipe_surface_set)}"
)


# ============================================================
# 20. VERIFY 246 HELD-OUT SOURCE RGB + FROZEN MASKS
# ============================================================

source_count = 0
mask_count = 0


for row in test_bg_rows:

    source_rel = project_rel_from_source_absolute(
        row[
            "source_absolute_path"
        ]
    )

    mask_rel = Path(
        normalize_path_text(
            row[
                "frozen_surface_mask_path"
            ]
        )
    )


    source = (
        OVERLAY_BUILD_ROOT
        / source_rel
    )

    mask = (
        OVERLAY_BUILD_ROOT
        / mask_rel
    )


    if not source.is_file():

        raise RuntimeError(
            "Held-out FruitBG source RGB missing:\n"
            f"{source}"
        )


    if not mask.is_file():

        raise RuntimeError(
            "Held-out FruitBG frozen mask missing:\n"
            f"{mask}"
        )


    source_count += 1
    mask_count += 1


if source_count != 246:

    raise RuntimeError(
        "Held-out RGB count != 246."
    )


if mask_count != 246:

    raise RuntimeError(
        "Held-out mask count != 246."
    )


print()
print(
    "Held-out source RGB availability: 246/246 — PASS"
)

print(
    "Held-out frozen mask availability: 246/246 — PASS"
)


# ============================================================
# 21. WRITE OVERLAY AUDIT
# ============================================================

overlay_audit = {
    "audit_name":
        "DASO-Net Formal-Test Project Overlay Audit v1",

    "status":
        "PASS",

    "wrapper_revision":
        (
            "Resume01B: recipe background_surface_id was "
            "matched to the frozen FruitBG test manifest by "
            "data-derived one-to-one namespace verification "
            "rather than assuming the manifest surface_id field."
        ),

    "scientific_change":
        False,

    "base_project":
        str(
            BASE_PROJECT_ROOT
        ),

    "supplement_dataset":
        str(
            SUPPLEMENT_DATASET_ROOT
        ),

    "supplement_zip_sha256":
        EXPECTED_ASSET_PACKAGE_ZIP_SHA,

    "supplement_payload_files":
        492,

    "supplement_source_rgb":
        246,

    "supplement_frozen_masks":
        246,

    "fruitbg": {
        "manifest_sha256":
            EXPECTED_FRUITBG_MANIFEST_SHA,

        "total":
            1644,

        "train":
            1151,

        "val":
            247,

        "test":
            246,
    },

    "recipe_mapping": {
        "recipe_background_field":
            recipe_bg_col,

        "canonical_manifest_representation":
            canonical_representation,

        "valid_manifest_representations":
            sorted(
                valid_representations.keys()
            ),

        "recipe_scenes":
            5000,

        "unique_recipe_background_ids":
            len(
                recipe_surface_set
            ),

        "all_recipes_uniquely_map_to_held_out_test_rows":
            True,
    },

    "protocol": {
        "sha256":
            EXPECTED_PROTOCOL_SHA,

        "changed":
            False,
    },

    "frozen_components_changed": {
        "renderer":
            False,

        "renderer_config":
            False,

        "test_recipes":
            False,

        "fruitbg_manifest":
            False,

        "checkpoints":
            False,

        "decoder":
            False,

        "operating_thresholds":
            False,
    },
}


save_json(
    OVERLAY_AUDIT,
    overlay_audit,
)


print()
print(
    "Overlay audit artifact written: PASS"
)


# ============================================================
# 22. ATOMICALLY FINALIZE PARTIAL OVERLAY
# ============================================================

OVERLAY_BUILD_ROOT.rename(
    OVERLAY_ROOT
)


print()
print(
    "============================================================"
)

print(
    "FORMAL-TEST PROJECT OVERLAY FINALIZED"
)

print(
    "============================================================"
)

print(
    OVERLAY_ROOT
)

print()

print(
    "No frozen scientific file was modified."
)


# ============================================================
# 23. FINALIZED OVERLAY READ-BACK
# ============================================================

FINAL_TEST_RECIPES = (
    OVERLAY_ROOT
    / "data"
    / "fruitmp_syn"
    / "v1_1_recipes"
    / "fruitmp_syn_v1_1_test_recipes.csv"
)

FINAL_FRUITBG_MANIFEST = (
    OVERLAY_ROOT
    / "data"
    / "fruit_bg"
    / "v1_final"
    / "fruitbg_rgb_v1_manifest.csv"
)


if (
    sha256_file(
        FINAL_TEST_RECIPES
    )
    != critical_hashes[
        "fruitmp_syn_test_recipes"
    ]
):

    raise RuntimeError(
        "Finalized overlay test-recipe SHA mismatch."
    )


if (
    sha256_file(
        FINAL_FRUITBG_MANIFEST
    )
    != EXPECTED_FRUITBG_MANIFEST_SHA
):

    raise RuntimeError(
        "Finalized overlay FruitBG-manifest SHA mismatch."
    )


print(
    "Finalized overlay read-back audit: PASS"
)


# ============================================================
# 24. GPU PREFLIGHT
# ============================================================

import torch


if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA unavailable."
    )


gpu_count = torch.cuda.device_count()


if gpu_count < 2:

    raise RuntimeError(
        "Expected two GPUs; visible count = "
        f"{gpu_count}"
    )


gpu_names = [
    torch.cuda.get_device_name(
        idx
    )
    for idx in range(
        gpu_count
    )
]


print()
print(
    f"Visible GPUs: {gpu_names}"
)

print(
    "Two-GPU runtime: PASS"
)


# ============================================================
# 25. FRESH FORMAL OUTPUT
# ============================================================

if RESUME_ROOT.exists():

    raise RuntimeError(
        "Formal Resume01 output unexpectedly exists:\n"
        f"{RESUME_ROOT}"
    )


RESUME_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 26. COMMAND BUILDER
# ============================================================

def build_command(
    idx: int,
):

    record = SEEDS[
        idx
    ]


    return [
        sys.executable,
        str(
            EVALUATOR
        ),

        "--project-root",
        str(
            OVERLAY_ROOT
        ),

        "--checkpoint",
        str(
            record[
                "checkpoint"
            ]
        ),

        "--output-dir",
        str(
            record[
                "output_dir"
            ]
        ),

        "--split",
        "test",

        "--max-scenes",
        "5000",

        "--seed",
        str(
            EVALUATION_SEED
        ),

        "--decode-min-score",
        str(
            DECODE_MIN_SCORE
        ),

        "--operating-score",
        str(
            record[
                "operating_threshold"
            ]
        ),

        "--topk",
        str(
            TOPK
        ),

        "--mask-threshold",
        str(
            MASK_THRESHOLD
        ),
    ]


def start_eval(
    idx: int,
):

    record = SEEDS[
        idx
    ]


    record[
        "output_dir"
    ].mkdir(
        parents=True,
        exist_ok=False,
    )


    command = build_command(
        idx
    )


    env = os.environ.copy()

    env[
        "CUDA_VISIBLE_DEVICES"
    ] = str(
        record[
            "gpu"
        ]
    )


    log_handle = record[
        "log"
    ].open(
        "w",
        encoding="utf-8",
        buffering=1,
    )


    print()
    print(
        "=" * 60
    )

    print(
        f"STARTING FORMAL TEST — SEED {idx}"
    )

    print(
        "=" * 60
    )

    print(
        f"RNG seed:             "
        f"{record['rng_seed']}"
    )

    print(
        f"GPU:                  "
        f"{record['gpu']}"
    )

    print(
        f"Operating threshold:  "
        f"{record['operating_threshold']}"
    )

    print(
        "Command:"
    )

    print(
        " ".join(
            shlex.quote(
                part
            )
            for part in command
        )
    )

    print(
        f"Log: {record['log']}"
    )


    process = subprocess.Popen(
        command,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=env,
        text=True,
    )


    return {
        "process":
            process,

        "log_handle":
            log_handle,

        "start_time":
            time.time(),
    }


def wait_for_group(
    jobs,
    poll_seconds: int = 60,
):

    while True:

        alive = []


        for idx, job in jobs.items():

            if (
                job[
                    "process"
                ].poll()
                is None
            ):

                alive.append(idx)


        if not alive:

            break


        status = []


        for idx in alive:

            elapsed_min = (
                time.time()
                - jobs[
                    idx
                ][
                    "start_time"
                ]
            ) / 60.0


            status.append(
                f"Seed {idx}: "
                f"{elapsed_min:.1f} min"
            )


        print(
            "[formal test running] "
            + " | ".join(status),
            flush=True,
        )


        time.sleep(
            poll_seconds
        )


    failures = []


    for idx, job in jobs.items():

        returncode = (
            job[
                "process"
            ].wait()
        )


        job[
            "log_handle"
        ].close()


        elapsed = (
            time.time()
            - job[
                "start_time"
            ]
        )


        print()
        print(
            "=" * 60
        )

        print(
            f"SEED {idx} PROCESS FINISHED"
        )

        print(
            "=" * 60
        )

        print(
            f"Return code: {returncode}"
        )

        print(
            f"Elapsed:     "
            f"{elapsed / 60.0:.2f} min"
        )

        print()

        print(
            "--- LOG TAIL ---"
        )

        print(
            tail_text(
                SEEDS[
                    idx
                ][
                    "log"
                ]
            )
        )

        print(
            "--- END LOG TAIL ---"
        )


        if returncode != 0:

            failures.append(idx)


    if failures:

        raise RuntimeError(
            "Formal evaluator failed for seed(s): "
            + ", ".join(
                str(x)
                for x in failures
            )
            + "\n\n"
            "STOP. Preserve all outputs and logs."
        )


# ============================================================
# 27. PRINT UNCHANGED CONTRACT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "UNCHANGED FROZEN FORMAL TEST CONTRACT"
)

print(
    "============================================================"
)

print(
    "Protocol SHA:"
)

print(
    EXPECTED_PROTOCOL_SHA
)

print()

print(
    "FruitMP-Syn v1.1 TEST scenes: 5000"
)

print(
    "Subsampling:                   NONE"
)

print(
    "Decode minimum:                0.05"
)

print(
    "Top-k:                         100"
)

print(
    "Mask threshold:                0.50"
)

print()

print(
    "Seed 1 operating threshold:    0.30"
)

print(
    "Seed 2 operating threshold:    0.35"
)

print(
    "Seed 3 operating threshold:    0.35"
)

print()

print(
    "Test-set threshold tuning:     NO"
)

print(
    "Best-seed selection:           NO"
)

print(
    "Seed exclusion:                NO"
)

print(
    "Scientific protocol change:    NO"
)


# ============================================================
# 28. RUN SEEDS 1 + 2 IN PARALLEL
# ============================================================

overall_start = time.time()


jobs_12 = {
    1:
        start_eval(
            1
        ),

    2:
        start_eval(
            2
        ),
}


wait_for_group(
    jobs_12,
    poll_seconds=60,
)


# ============================================================
# 29. VERIFY SEEDS 1 + 2 COMPLETE
# ============================================================

for idx in [
    1,
    2,
]:

    require(
        SEEDS[
            idx
        ][
            "summary_json"
        ],
        f"Seed-{idx} metric summary",
    )

    require(
        SEEDS[
            idx
        ][
            "per_scene_csv"
        ],
        f"Seed-{idx} per-scene CSV",
    )


    rows = read_csv(
        SEEDS[
            idx
        ][
            "per_scene_csv"
        ]
    )


    if len(rows) != 5000:

        raise RuntimeError(
            f"Seed-{idx} formal output has "
            f"{len(rows)} scenes instead of 5000."
        )


    print(
        f"Seed-{idx} completed 5000-scene output: PASS"
    )


# ============================================================
# 30. RUN SEED 3
# ============================================================

jobs_3 = {
    3:
        start_eval(
            3
        )
}


wait_for_group(
    jobs_3,
    poll_seconds=60,
)


# ============================================================
# 31. VERIFY ALL 3 RESULTS
# ============================================================

for idx in [
    1,
    2,
    3,
]:

    require(
        SEEDS[
            idx
        ][
            "summary_json"
        ],
        f"Seed-{idx} metric summary",
    )

    require(
        SEEDS[
            idx
        ][
            "per_scene_csv"
        ],
        f"Seed-{idx} per-scene CSV",
    )


    rows = read_csv(
        SEEDS[
            idx
        ][
            "per_scene_csv"
        ]
    )


    if len(rows) != 5000:

        raise RuntimeError(
            f"Seed-{idx} scene count != 5000."
        )


print()
print(
    "All 3 final seeds produced 5000-scene outputs: PASS"
)


# ============================================================
# 32. IDENTICAL ORDERED SCENE AUDIT
# ============================================================

ordered_ids = {}


for idx in [
    1,
    2,
    3,
]:

    rows = read_csv(
        SEEDS[
            idx
        ][
            "per_scene_csv"
        ]
    )


    scene_col = find_scene_column(
        rows
    )


    if scene_col is None:

        raise RuntimeError(
            f"Seed-{idx} scene-ID column not found."
        )


    ids = [
        str(
            row[
                scene_col
            ]
        )
        for row in rows
    ]


    if len(
        ids
    ) != 5000:

        raise RuntimeError(
            f"Seed-{idx} scene-ID count != 5000."
        )


    if len(
        set(ids)
    ) != 5000:

        raise RuntimeError(
            f"Seed-{idx} contains duplicate scene IDs."
        )


    ordered_ids[idx] = ids


if not (
    ordered_ids[1]
    == ordered_ids[2]
    == ordered_ids[3]
):

    raise RuntimeError(
        "Seeds did not evaluate identical ordered scenes."
    )


print(
    "Identical ordered 5000 scenes across all seeds: PASS"
)


# ============================================================
# 33. EXACT FROZEN TEST-RECIPE SET AUDIT
# ============================================================

frozen_recipe_rows = read_csv(
    FINAL_TEST_RECIPES
)

frozen_scene_col = find_scene_column(
    frozen_recipe_rows
)


if frozen_scene_col is None:

    raise RuntimeError(
        "Frozen recipe scene-ID column not found."
    )


frozen_ids = [
    str(
        row[
            frozen_scene_col
        ]
    )
    for row in frozen_recipe_rows
]


if (
    set(frozen_ids)
    != set(
        ordered_ids[1]
    )
):

    raise RuntimeError(
        "Evaluator scene set differs from frozen recipes."
    )


print(
    "Evaluator scene set exactly matches frozen recipes: PASS"
)


# ============================================================
# 34. HASH RAW FORMAL RESULTS
# ============================================================

result_hashes = {}


for idx in [
    1,
    2,
    3,
]:

    result_hashes[
        f"seed{idx}"
    ] = {
        "metric_summary_sha256":
            sha256_file(
                SEEDS[
                    idx
                ][
                    "summary_json"
                ]
            ),

        "per_scene_csv_sha256":
            sha256_file(
                SEEDS[
                    idx
                ][
                    "per_scene_csv"
                ]
            ),

        "log_sha256":
            sha256_file(
                SEEDS[
                    idx
                ][
                    "log"
                ]
            ),
    }


# ============================================================
# 35. WRITE FORMAL RUN MANIFEST
# ============================================================

elapsed_total = (
    time.time()
    - overall_start
)


run_manifest = {
    "run_name":
        (
            "DASO-Net Formal Untouched FruitMP-Syn Test "
            "v1 Resume01 Assets Complete"
        ),

    "status":
        "PASS",

    "protocol": {
        "sha256":
            EXPECTED_PROTOCOL_SHA,

        "changed_after_failures":
            False,
    },

    "prior_infrastructure_failures": {
        "first_attempt":
            (
                "missing held-out FruitBG assets; "
                "no metric completion"
            ),

        "resume01_wrapper_attempt":
            (
                "incorrect audit identifier namespace; "
                "no inference started"
            ),
    },

    "asset_overlay": {
        "audit":
            str(
                OVERLAY_AUDIT
            ),

        "audit_sha256":
            sha256_file(
                OVERLAY_AUDIT
            ),

        "supplement_zip_sha256":
            EXPECTED_ASSET_PACKAGE_ZIP_SHA,

        "source_rgb":
            246,

        "frozen_masks":
            246,

        "payload_files":
            492,

        "regenerated":
            False,

        "scientific_files_modified":
            False,
    },

    "recipe_to_fruitbg_mapping": {
        "recipe_field":
            recipe_bg_col,

        "verified_manifest_representation":
            canonical_representation,

        "all_5000_uniquely_map_to_test_rows":
            True,
    },

    "dataset": {
        "name":
            "FruitMP-Syn v1.1",

        "split":
            "test",

        "scenes":
            5000,

        "all_scenes":
            True,

        "subsampling":
            False,

        "identical_order_across_seeds":
            True,

        "matches_frozen_recipe_set":
            True,
    },

    "decoder": {
        "decode_min_score":
            0.05,

        "topk":
            100,

        "mask_threshold":
            0.50,

        "sha256":
            EXPECTED_DECODER_SHA,
    },

    "models": {
        "seed1": {
            "rng_seed":
                1782656963,

            "checkpoint_sha256":
                EXPECTED_SEED1_SHA,

            "operating_threshold":
                0.30,
        },

        "seed2": {
            "rng_seed":
                901395089,

            "checkpoint_sha256":
                EXPECTED_SEED2_SHA,

            "operating_threshold":
                0.35,
        },

        "seed3": {
            "rng_seed":
                252916394,

            "checkpoint_sha256":
                EXPECTED_SEED3_SHA,

            "operating_threshold":
                0.35,
        },
    },

    "result_sha256":
        result_hashes,

    "policy": {
        "training":
            False,

        "checkpoint_selection":
            False,

        "best_seed_selection":
            False,

        "seed_exclusion":
            False,

        "test_threshold_tuning":
            False,

        "test_best_threshold_if_emitted":
            (
                "post-hoc descriptive sensitivity only"
            ),
    },

    "wall_time_minutes":
        elapsed_total
        / 60.0,
}


save_json(
    RESUME_MANIFEST,
    run_manifest,
)


# ============================================================
# 36. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FORMAL UNTOUCHED FRUITMP-SYN TEST RESUME COMPLETE"
)

print(
    "============================================================"
)

print()

print(
    "Protocol SHA unchanged:"
)

print(
    EXPECTED_PROTOCOL_SHA
)

print()

print(
    "Scientific protocol changed:       NO"
)

print(
    "Frozen data regenerated:           NO"
)

print(
    "Frozen scientific files modified:  NO"
)

print()

print(
    "Held-out FruitBG restored:"
)

print(
    "  source RGB:    246/246"
)

print(
    "  frozen masks:  246/246"
)

print(
    "  payload SHA:   492/492 PASS"
)

print()

print(
    "Recipe ID representation used only for audit:"
)

print(
    f"  {canonical_representation}"
)

print()

print(
    "FruitMP-Syn test:"
)

print(
    "  scenes/seed:       5000"
)

print(
    "  final seeds:       3/3"
)

print(
    "  identical order:   YES"
)

print(
    "  frozen recipe set: YES"
)

print()

print(
    "Frozen operating thresholds:"
)

print(
    "  Seed 1: 0.30"
)

print(
    "  Seed 2: 0.35"
)

print(
    "  Seed 3: 0.35"
)

print()

print(
    "Seed-1 raw summary:"
)

print(
    SEEDS[
        1
    ][
        "summary_json"
    ]
)

print()

print(
    "Seed-2 raw summary:"
)

print(
    SEEDS[
        2
    ][
        "summary_json"
    ]
)

print()

print(
    "Seed-3 raw summary:"
)

print(
    SEEDS[
        3
    ][
        "summary_json"
    ]
)

print()

print(
    "Formal run manifest:"
)

print(
    RESUME_MANIFEST
)

print()

print(
    f"Formal inference wall time: "
    f"{elapsed_total / 60.0:.2f} min"
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET FORMAL UNTOUCHED FRUITMP-SYN TEST RESUME: PASS"
)

print(
    "============================================================"
)

print()

print(
    "NO TRAINING WAS PERFORMED."
)

print(
    "NO TEST-SET THRESHOLD TUNING WAS PERFORMED."
)

print(
    "NO BEST-SEED SELECTION WAS PERFORMED."
)

print(
    "ALL THREE RAW FORMAL TEST RESULTS ARE PRESERVED."
)

print()

print(
    "STOP HERE."
)

print(
    "Next: formal three-seed untouched-test aggregation."
)

DASO-Net — FORMAL TEST RESUME 01B
CORRECT SURFACE-ID AUDIT + FINALIZE + INFERENCE
PASS  Phase5B slim frozen project
PASS  Validated evaluator
PASS  Validated decoder
PASS  Frozen evaluation protocol
PASS  Seed-1 terminal
PASS  Seed-2 terminal
PASS  Seed-3 terminal
PASS  Original failed formal-test root
PASS  Original failed Seed-1 log
PASS  Original failed Seed-2 log
PASS  Asset failure audit
PASS  FruitBG supplemental dataset
PASS  Supplement SHA record
PASS  Supplement state
PASS  Resume01 partial overlay

Original failed attempt remains non-result-producing: PASS
No untouched-test metric was observed in that attempt.

Frozen protocol SHA: PASS
Seed-1 checkpoint + frozen threshold: PASS
Seed-2 checkpoint + frozen threshold: PASS
Seed-3 checkpoint + frozen threshold: PASS
Validated decoder SHA: PASS

Re-verifying 492 supplemental payload files...
  verified 100/492
  verified 200/492
  verified 300/492
  verified 400/492
  verified 492/492
Supplemental 492-file SHA audit: PASS

Partia

In [18]:
# ============================================================
# DASO-Net — FORMAL THREE-SEED UNTOUCHED-TEST AGGREGATE v1
# RESUME-SAFE REWRITE AFTER PER-SCENE SCHEMA WRAPPER FAILURE
#
# INPUT
# -----
# Formal untouched FruitMP-Syn v1.1 TEST:
#   5000 scenes x 3 precommitted final seeds
#
# Frozen operating thresholds:
#   Seed 1 = 0.30
#   Seed 2 = 0.35
#   Seed 3 = 0.35
#
# IMPORTANT
# ---------
# Per-scene evaluator schema is:
#
#   scene_id
#   fruit
#   gt_count
#   pred_count
#   tp
#   fp
#   fn
#   count_abs_error
#   matched_iou_mean
#   matched_dice_mean
#   max_score
#   decoded_predictions
#
# We DO NOT assume pred_count is the operating-point count.
# We prove it by verifying:
#
#   pred_count == tp + fp
#   gt_count   == tp + fn
#   count_abs_error == abs(pred_count - gt_count)
#
# and by reproducing:
#
#   operating precision
#   operating recall
#   operating F1
#   count MAE
#   empty-scene FP/image
#
# from the raw per-scene rows.
#
# Any evaluator field containing "best" remains POST-HOC
# test-sweep diagnostic output and is EXCLUDED from the formal
# operating-point aggregate.
#
# NO TRAINING.
# NO INFERENCE.
# NO THRESHOLD TUNING.
# NO SEED SELECTION.
# NO FROZEN DATA MODIFICATION.
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import math
import re
import statistics
from pathlib import Path


# ============================================================
# 1. ROOTS
# ============================================================

WORKING = Path(
    "/kaggle/working"
)

FORMAL_ROOT = (
    WORKING
    / "dasonet_final_evaluation"
    / "fruitmp_syn_test_v1_resume01_assets_complete"
)

RUN_MANIFEST = (
    FORMAL_ROOT
    / "FORMAL_FRUITMP_SYN_TEST_RUN_MANIFEST_v1_RESUME01.json"
)

PROTOCOL_JSON = (
    WORKING
    / "dasonet_final_analysis"
    / "formal_evaluation_protocol_freeze_v1"
    / "FORMAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

OVERLAY_AUDIT = (
    WORKING
    / "DASO-Net-FORMALTEST-OVERLAY-AUDIT-v1.json"
)

TEST_RECIPES = (
    WORKING
    / "DASO-Net-Phase1A-FORMALTEST-OVERLAY-v1"
    / "data"
    / "fruitmp_syn"
    / "v1_1_recipes"
    / "fruitmp_syn_v1_1_test_recipes.csv"
)


# ============================================================
# 2. EXPECTED FROZEN IDENTITIES
# ============================================================

EXPECTED_PROTOCOL_SHA = (
    "a98ac6c5af42c7dc84bc7e7c00b7a9ed"
    "661d45af6882bdb0029c49b8d0cb5b62"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)

EXPECTED_SEED1_SHA = (
    "e4c4b79ad4b8c6e18ba2b1e3d6cbe796"
    "c9cc301bd6fcc7c596fc290ad3a7e81d"
)

EXPECTED_SEED2_SHA = (
    "19cdb834a46240916b2c8023e8c3afe573"
    "8d9c7cd5d8f818c62ad1e289df1ad2"
)

EXPECTED_SEED3_SHA = (
    "be7b7941f2f62179e15074aa973bf6da0"
    "eb9018ec9133486512bd1f87153f447"
)

EXPECTED_TEST_SCENES = 5000


# ============================================================
# 3. RAW FORMAL RESULT PATHS
# ============================================================

SEEDS = {
    1: {
        "rng_seed":
            1782656963,

        "operating_threshold":
            0.30,

        "checkpoint_sha256":
            EXPECTED_SEED1_SHA,

        "root":
            FORMAL_ROOT
            / "seed01_s1782656963",
    },

    2: {
        "rng_seed":
            901395089,

        "operating_threshold":
            0.35,

        "checkpoint_sha256":
            EXPECTED_SEED2_SHA,

        "root":
            FORMAL_ROOT
            / "seed02_s901395089",
    },

    3: {
        "rng_seed":
            252916394,

        "operating_threshold":
            0.35,

        "checkpoint_sha256":
            EXPECTED_SEED3_SHA,

        "root":
            FORMAL_ROOT
            / "seed03_s252916394",
    },
}


for idx in [
    1,
    2,
    3,
]:

    SEEDS[idx]["summary"] = (
        SEEDS[idx]["root"]
        / "phase5c_metric_summary.json"
    )

    SEEDS[idx]["per_scene"] = (
        SEEDS[idx]["root"]
        / "phase5c_per_scene.csv"
    )

    SEEDS[idx]["log"] = (
        FORMAL_ROOT
        / f"seed0{idx}_evaluation.log"
    )


# ============================================================
# 4. CONSOLE HEADLINE VALUES
#
# Used only as an independent source-integrity cross-check.
# Formal aggregate uses full-precision summary JSON values.
# ============================================================

CONSOLE_VALUES = {
    1: {
        "mask_ap50":
            0.2397,

        "mask_ap75":
            0.0426,

        "mask_ap50_95":
            0.0844,

        "operating_precision":
            0.5101,

        "operating_recall":
            0.2708,

        "operating_f1":
            0.3538,

        "matched_mask_dice":
            0.8019,

        "count_mae":
            1.1330,

        "empty_fp_per_image":
            0.1160,
    },

    2: {
        "mask_ap50":
            0.2616,

        "mask_ap75":
            0.0355,

        "mask_ap50_95":
            0.0831,

        "operating_precision":
            0.5879,

        "operating_recall":
            0.2874,

        "operating_f1":
            0.3861,

        "matched_mask_dice":
            0.8081,

        "count_mae":
            1.1978,

        "empty_fp_per_image":
            0.0800,
    },

    3: {
        "mask_ap50":
            0.2834,

        "mask_ap75":
            0.0410,

        "mask_ap50_95":
            0.0925,

        "operating_precision":
            0.6025,

        "operating_recall":
            0.3325,

        "operating_f1":
            0.4285,

        "matched_mask_dice":
            0.8076,

        "count_mae":
            1.0548,

        "empty_fp_per_image":
            0.0587,
    },
}


# ============================================================
# 5. OUTPUT
# ============================================================

OUT_ROOT = (
    WORKING
    / "dasonet_final_analysis"
    / "formal_fruitmp_syn_test_three_seed_v1"
)

PER_SEED_CSV = (
    OUT_ROOT
    / "formal_fruitmp_syn_test_per_seed.csv"
)

MEAN_SD_CSV = (
    OUT_ROOT
    / "formal_fruitmp_syn_test_mean_sd.csv"
)

AGGREGATE_JSON = (
    OUT_ROOT
    / "FINAL_FORMAL_FRUITMP_SYN_TEST_THREE_SEED_AGGREGATE_v1.json"
)

MARKDOWN_TABLE = (
    OUT_ROOT
    / "formal_fruitmp_syn_test_table.md"
)

SHA_RECORD = (
    OUT_ROOT
    / "FINAL_FORMAL_FRUITMP_SYN_TEST_THREE_SEED_AGGREGATE_v1.sha256.txt"
)


# ============================================================
# 6. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(
    path: Path,
    obj,
) -> None:

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def read_csv(
    path: Path,
):

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def normalize_name(
    value,
) -> str:

    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(
            value
        ).lower(),
    ).strip(
        "_"
    )


def flatten_numeric(
    obj,
    prefix="root",
):

    out = []


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            out.extend(
                flatten_numeric(
                    value,
                    f"{prefix}.{key}",
                )
            )


    elif isinstance(
        obj,
        list,
    ):

        for idx, value in enumerate(
            obj
        ):

            out.extend(
                flatten_numeric(
                    value,
                    f"{prefix}[{idx}]",
                )
            )


    elif (
        isinstance(
            obj,
            (int, float),
        )
        and not isinstance(
            obj,
            bool,
        )
    ):

        v = float(
            obj
        )

        if math.isfinite(
            v
        ):

            out.append(
                (
                    prefix,
                    v,
                )
            )


    return out


def find_scene_column(
    rows,
):

    if not rows:

        return None


    lookup = {
        normalize_name(
            key
        ):
            key
        for key in rows[
            0
        ].keys()
        if key is not None
    }


    for candidate in [
        "scene_id",
        "scene",
        "recipe_id",
        "sample_id",
        "id",
    ]:

        key = normalize_name(
            candidate
        )


        if key in lookup:

            return lookup[
                key
            ]


    return None


def resolve_summary_metric(
    summary,
    metric_name,
    expected_console_value,
):
    """
    Resolve a formal headline metric from the evaluator JSON.

    Any path containing 'best' is excluded.
    """

    flattened = flatten_numeric(
        summary
    )


    def allowed(
        path,
    ):

        p = normalize_name(
            path
        )


        if "best" in p:

            return False


        if metric_name == "mask_ap50":

            return (
                (
                    "ap50"
                    in p
                    or "ap_50"
                    in p
                )
                and "95"
                not in p
                and "75"
                not in p
                and "threshold"
                not in p
            )


        if metric_name == "mask_ap75":

            return (
                (
                    "ap75"
                    in p
                    or "ap_75"
                    in p
                )
                and "threshold"
                not in p
            )


        if metric_name == "mask_ap50_95":

            return (
                (
                    "ap50_95"
                    in p
                    or "ap_50_95"
                    in p
                    or "ap5095"
                    in p
                    or "50_95"
                    in p
                )
                and "threshold"
                not in p
            )


        if metric_name == "operating_precision":

            return (
                "precision"
                in p
                and (
                    "operating"
                    in p
                    or "op_precision"
                    in p
                )
            )


        if metric_name == "operating_recall":

            return (
                "recall"
                in p
                and (
                    "operating"
                    in p
                    or "op_recall"
                    in p
                )
            )


        if metric_name == "operating_f1":

            return (
                "f1"
                in p
                and (
                    "operating"
                    in p
                    or "op_f1"
                    in p
                )
            )


        if metric_name == "matched_mask_dice":

            return (
                "dice"
                in p
                and "matched"
                in p
            )


        if metric_name == "count_mae":

            return (
                "mae"
                in p
                and "count"
                in p
            )


        if metric_name == "empty_fp_per_image":

            return (
                "empty"
                in p
                and (
                    "fp"
                    in p
                    or "false_positive"
                    in p
                )
                and (
                    "image"
                    in p
                    or "scene"
                    in p
                )
            )


        return False


    candidates = [
        (
            path,
            value,
        )
        for path, value
        in flattened
        if allowed(
            path
        )
    ]


    matching = [
        (
            path,
            value,
        )
        for path, value
        in candidates
        if abs(
            value
            - expected_console_value
        )
        <= 5e-4
    ]


    if not matching:

        print()
        print(
            f"Unable to resolve metric: {metric_name}"
        )

        print(
            f"Expected console value: {expected_console_value}"
        )

        print(
            "Semantic candidates:"
        )


        for path, value in candidates:

            print(
                f"  {path} = {value}"
            )


        raise RuntimeError(
            f"Could not verify {metric_name} "
            "against raw summary JSON."
        )


    unique_values = []


    for _, value in matching:

        if not any(
            abs(
                value
                - prior
            )
            <= 1e-12
            for prior in unique_values
        ):

            unique_values.append(
                value
            )


    if len(
        unique_values
    ) != 1:

        print()
        print(
            f"Ambiguous source values for {metric_name}:"
        )


        for path, value in matching:

            print(
                f"  {path} = {value}"
            )


        raise RuntimeError(
            f"Ambiguous source metric: {metric_name}"
        )


    return unique_values[
        0
    ]


def collect_best_fields(
    summary,
):

    out = {}


    for path, value in flatten_numeric(
        summary
    ):

        if "best" in normalize_name(
            path
        ):

            out[
                path
            ] = value


    return out


def mean_sd(
    values,
):

    if len(
        values
    ) != 3:

        raise RuntimeError(
            "Formal aggregate requires exactly 3 seeds."
        )


    return (
        statistics.mean(
            values
        ),
        statistics.stdev(
            values
        ),
    )


def safe_ratio(
    numerator,
    denominator,
):

    if denominator == 0:

        return 0.0


    return (
        numerator
        / denominator
    )


# ============================================================
# 7. PREFLIGHT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FORMAL THREE-SEED UNTOUCHED-TEST AGGREGATE v1"
)

print(
    "============================================================"
)


for path, label in [
    (
        RUN_MANIFEST,
        "Formal run manifest",
    ),
    (
        PROTOCOL_JSON,
        "Frozen evaluation protocol",
    ),
    (
        OVERLAY_AUDIT,
        "Formal-test overlay audit",
    ),
    (
        TEST_RECIPES,
        "Frozen 5000-scene test recipes",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


for idx in [
    1,
    2,
    3,
]:

    for key, label in [
        (
            "summary",
            "metric summary",
        ),
        (
            "per_scene",
            "per-scene CSV",
        ),
        (
            "log",
            "evaluation log",
        ),
    ]:

        require(
            SEEDS[idx][
                key
            ],
            f"Seed-{idx} {label}",
        )


print(
    "All three raw formal result sets present: PASS"
)


# ============================================================
# 8. RESUME-SAFE OUTPUT DIRECTORY
# ============================================================

if OUT_ROOT.exists():

    existing_files = [
        p
        for p in OUT_ROOT.rglob(
            "*"
        )
        if p.is_file()
    ]


    if existing_files:

        raise RuntimeError(
            "Formal aggregate output directory already "
            "contains files.\n"
            "Do not overwrite an existing formal aggregate:\n"
            f"{OUT_ROOT}\n\n"
            + "\n".join(
                str(
                    p
                )
                for p in existing_files[
                    :30
                ]
            )
        )


    print()
    print(
        "Existing empty derived aggregate directory detected."
    )

    print(
        "Resume mode: REUSE EMPTY DIRECTORY"
    )

else:

    OUT_ROOT.mkdir(
        parents=True,
        exist_ok=False,
    )

    print()
    print(
        "Fresh formal aggregate directory created."
    )


# ============================================================
# 9. PROTOCOL / RUN PROVENANCE
# ============================================================

if (
    sha256_file(
        PROTOCOL_JSON
    )
    != EXPECTED_PROTOCOL_SHA
):

    raise RuntimeError(
        "Frozen formal protocol SHA mismatch."
    )


protocol = load_json(
    PROTOCOL_JSON
)


if (
    protocol.get(
        "status"
    )
    != "FROZEN_BEFORE_TEST_INFERENCE"
):

    raise RuntimeError(
        "Formal protocol status changed."
    )


run_manifest = load_json(
    RUN_MANIFEST
)


if run_manifest.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Formal run manifest is not PASS."
    )


if (
    run_manifest.get(
        "protocol",
        {}
    ).get(
        "sha256"
    )
    != EXPECTED_PROTOCOL_SHA
):

    raise RuntimeError(
        "Run-manifest protocol SHA mismatch."
    )


policy = run_manifest.get(
    "policy",
    {}
)


if bool(
    policy.get(
        "training",
        False,
    )
):

    raise RuntimeError(
        "Formal run unexpectedly indicates training."
    )


if bool(
    policy.get(
        "best_seed_selection",
        False,
    )
):

    raise RuntimeError(
        "Formal run unexpectedly indicates best-seed selection."
    )


if bool(
    policy.get(
        "test_threshold_tuning",
        False,
    )
):

    raise RuntimeError(
        "Formal run unexpectedly indicates test threshold tuning."
    )


print()
print(
    "Frozen protocol SHA: PASS"
)

print(
    "Formal run manifest: PASS"
)

print(
    "Training during test: NO"
)

print(
    "Test-threshold tuning: NO"
)

print(
    "Best-seed selection: NO"
)


# ============================================================
# 10. VERIFY RAW RESULT HASHES
# ============================================================

manifest_result_hashes = run_manifest.get(
    "result_sha256",
    {}
)


for idx in [
    1,
    2,
    3,
]:

    expected = manifest_result_hashes.get(
        f"seed{idx}",
        {}
    )


    actual_summary_sha = sha256_file(
        SEEDS[idx][
            "summary"
        ]
    )

    actual_scene_sha = sha256_file(
        SEEDS[idx][
            "per_scene"
        ]
    )

    actual_log_sha = sha256_file(
        SEEDS[idx][
            "log"
        ]
    )


    if (
        expected.get(
            "metric_summary_sha256"
        )
        != actual_summary_sha
    ):

        raise RuntimeError(
            f"Seed-{idx} summary SHA mismatch."
        )


    if (
        expected.get(
            "per_scene_csv_sha256"
        )
        != actual_scene_sha
    ):

        raise RuntimeError(
            f"Seed-{idx} per-scene CSV SHA mismatch."
        )


    if (
        expected.get(
            "log_sha256"
        )
        != actual_log_sha
    ):

        raise RuntimeError(
            f"Seed-{idx} log SHA mismatch."
        )


    print(
        f"Seed-{idx} raw result SHA integrity: PASS"
    )


# ============================================================
# 11. VERIFY FROZEN OPERATING THRESHOLDS
# ============================================================

expected_thresholds = {
    1:
        0.30,

    2:
        0.35,

    3:
        0.35,
}


for idx in [
    1,
    2,
    3,
]:

    protocol_seed = (
        protocol[
            "final_seed_policy"
        ][
            "seeds"
        ][
            f"seed{idx}"
        ]
    )


    protocol_threshold = float(
        protocol_seed[
            "operating_threshold"
        ]
    )


    if (
        protocol_threshold
        != expected_thresholds[
            idx
        ]
    ):

        raise RuntimeError(
            f"Seed-{idx} frozen threshold changed."
        )


    if (
        float(
            SEEDS[idx][
                "operating_threshold"
            ]
        )
        != protocol_threshold
    ):

        raise RuntimeError(
            f"Seed-{idx} aggregate threshold mismatch."
        )


print()
print(
    "Frozen operating thresholds:"
)

print(
    "  Seed 1: 0.30 — PASS"
)

print(
    "  Seed 2: 0.35 — PASS"
)

print(
    "  Seed 3: 0.35 — PASS"
)


# ============================================================
# 12. VERIFY SAME ORDERED 5000 TEST SCENES
# ============================================================

scene_rows = {}

ordered_scene_ids = {}


for idx in [
    1,
    2,
    3,
]:

    rows = read_csv(
        SEEDS[idx][
            "per_scene"
        ]
    )


    if len(
        rows
    ) != EXPECTED_TEST_SCENES:

        raise RuntimeError(
            f"Seed-{idx} per-scene count != 5000."
        )


    scene_col = find_scene_column(
        rows
    )


    if scene_col is None:

        raise RuntimeError(
            f"Seed-{idx} scene-ID field not found."
        )


    ids = [
        str(
            row[
                scene_col
            ]
        )
        for row in rows
    ]


    if len(
        set(
            ids
        )
    ) != 5000:

        raise RuntimeError(
            f"Seed-{idx} contains duplicate scene IDs."
        )


    scene_rows[
        idx
    ] = rows

    ordered_scene_ids[
        idx
    ] = ids


if not (
    ordered_scene_ids[
        1
    ]
    ==
    ordered_scene_ids[
        2
    ]
    ==
    ordered_scene_ids[
        3
    ]
):

    raise RuntimeError(
        "The three formal seeds do not contain "
        "identical ordered test scenes."
    )


frozen_recipe_rows = read_csv(
    TEST_RECIPES
)


frozen_scene_col = find_scene_column(
    frozen_recipe_rows
)


if frozen_scene_col is None:

    raise RuntimeError(
        "Frozen test recipe scene-ID field not found."
    )


frozen_scene_ids = [
    str(
        row[
            frozen_scene_col
        ]
    )
    for row in frozen_recipe_rows
]


if (
    set(
        frozen_scene_ids
    )
    != set(
        ordered_scene_ids[
            1
        ]
    )
):

    raise RuntimeError(
        "Formal scene set differs from frozen recipes."
    )


print()
print(
    "Same ordered 5000 scenes across all seeds: PASS"
)

print(
    "Exact set matches frozen test recipes: PASS"
)


# ============================================================
# 13. EXTRACT FULL-PRECISION HEADLINE SUMMARY METRICS
# ============================================================

metric_names = [
    "mask_ap50",
    "mask_ap75",
    "mask_ap50_95",
    "operating_precision",
    "operating_recall",
    "operating_f1",
    "matched_mask_dice",
    "count_mae",
    "empty_fp_per_image",
]


per_seed = {}

posthoc_best_fields = {}


print()
print(
    "============================================================"
)

print(
    "VERIFYING HEADLINE METRICS AGAINST RAW SUMMARY JSON"
)

print(
    "============================================================"
)


for idx in [
    1,
    2,
    3,
]:

    summary = load_json(
        SEEDS[idx][
            "summary"
        ]
    )


    row = {
        "seed_index":
            idx,

        "rng_seed":
            SEEDS[idx][
                "rng_seed"
            ],

        "operating_threshold":
            SEEDS[idx][
                "operating_threshold"
            ],
    }


    for metric in metric_names:

        row[
            metric
        ] = resolve_summary_metric(
            summary=summary,
            metric_name=metric,
            expected_console_value=(
                CONSOLE_VALUES[
                    idx
                ][
                    metric
                ]
            ),
        )


    per_seed[
        idx
    ] = row


    posthoc_best_fields[
        f"seed{idx}"
    ] = collect_best_fields(
        summary
    )


    print(
        f"Seed-{idx} headline source verification: PASS"
    )


print()
print(
    "All formal headline metrics verified: PASS"
)


# ============================================================
# 14. PER-SCENE OPERATING-POINT ALGEBRA AUDIT
#
# This is the key schema correction.
#
# We accept pred_count only after proving it is consistent with
# the operating-point TP / FP / FN and summary metrics.
# ============================================================

print()
print(
    "============================================================"
)

print(
    "PER-SCENE OPERATING-POINT SCHEMA VERIFICATION"
)

print(
    "============================================================"
)


REQUIRED_PER_SCENE_COLUMNS = {
    "gt_count",
    "pred_count",
    "tp",
    "fp",
    "fn",
    "count_abs_error",
}


for idx in [
    1,
    2,
    3,
]:

    rows = scene_rows[
        idx
    ]


    fields = set(
        rows[
            0
        ].keys()
    )


    missing = sorted(
        REQUIRED_PER_SCENE_COLUMNS
        - fields
    )


    if missing:

        raise RuntimeError(
            f"Seed-{idx} missing required per-scene fields:\n"
            + "\n".join(
                missing
            )
        )


    gt_counts = []
    pred_counts = []

    total_tp = 0
    total_fp = 0
    total_fn = 0


    for row_number, row in enumerate(
        rows,
        start=1,
    ):

        gt = int(
            float(
                row[
                    "gt_count"
                ]
            )
        )

        pred = int(
            float(
                row[
                    "pred_count"
                ]
            )
        )

        tp = int(
            float(
                row[
                    "tp"
                ]
            )
        )

        fp = int(
            float(
                row[
                    "fp"
                ]
            )
        )

        fn = int(
            float(
                row[
                    "fn"
                ]
            )
        )

        abs_error = float(
            row[
                "count_abs_error"
            ]
        )


        # ----------------------------------------------------
        # Core identity proving pred_count represents the
        # predictions being scored by operating TP/FP/FN.
        # ----------------------------------------------------

        if pred != (
            tp
            + fp
        ):

            raise RuntimeError(
                f"Seed-{idx} row {row_number}: "
                "pred_count != tp + fp.\n"
                f"pred_count={pred}, tp={tp}, fp={fp}"
            )


        if gt != (
            tp
            + fn
        ):

            raise RuntimeError(
                f"Seed-{idx} row {row_number}: "
                "gt_count != tp + fn.\n"
                f"gt_count={gt}, tp={tp}, fn={fn}"
            )


        if abs(
            abs_error
            - abs(
                pred
                - gt
            )
        ) > 1e-12:

            raise RuntimeError(
                f"Seed-{idx} row {row_number}: "
                "count_abs_error mismatch."
            )


        gt_counts.append(
            gt
        )

        pred_counts.append(
            pred
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn


    # --------------------------------------------------------
    # Reproduce formal operating precision/recall/F1
    # --------------------------------------------------------

    derived_precision = safe_ratio(
        total_tp,
        total_tp
        + total_fp,
    )

    derived_recall = safe_ratio(
        total_tp,
        total_tp
        + total_fn,
    )

    derived_f1 = safe_ratio(
        2.0
        * derived_precision
        * derived_recall,
        derived_precision
        + derived_recall,
    )


    # --------------------------------------------------------
    # Count metrics
    # --------------------------------------------------------

    derived_mae = statistics.mean(
        abs(
            pred
            - gt
        )
        for pred, gt
        in zip(
            pred_counts,
            gt_counts,
        )
    )


    derived_bias = statistics.mean(
        pred
        - gt
        for pred, gt
        in zip(
            pred_counts,
            gt_counts,
        )
    )


    empty_predictions = [
        pred
        for pred, gt
        in zip(
            pred_counts,
            gt_counts,
        )
        if gt == 0
    ]


    if not empty_predictions:

        raise RuntimeError(
            f"Seed-{idx}: no empty test scenes."
        )


    derived_empty_fp = statistics.mean(
        empty_predictions
    )


    # --------------------------------------------------------
    # Cross-check against full-precision summary.
    # --------------------------------------------------------

    tolerance = 5e-8


    checks = [
        (
            "operating precision",
            derived_precision,
            per_seed[
                idx
            ][
                "operating_precision"
            ],
        ),

        (
            "operating recall",
            derived_recall,
            per_seed[
                idx
            ][
                "operating_recall"
            ],
        ),

        (
            "operating F1",
            derived_f1,
            per_seed[
                idx
            ][
                "operating_f1"
            ],
        ),

        (
            "count MAE",
            derived_mae,
            per_seed[
                idx
            ][
                "count_mae"
            ],
        ),

        (
            "empty FP/image",
            derived_empty_fp,
            per_seed[
                idx
            ][
                "empty_fp_per_image"
            ],
        ),
    ]


    for name, derived, summary_value in checks:

        if abs(
            derived
            - summary_value
        ) > tolerance:

            raise RuntimeError(
                f"Seed-{idx} {name} mismatch.\n"
                f"Derived from per-scene: {derived}\n"
                f"Formal summary JSON:    {summary_value}"
            )


    # --------------------------------------------------------
    # We have now proven pred_count belongs to the operating
    # TP/FP/FN accounting used by the formal summary.
    # --------------------------------------------------------

    per_seed[
        idx
    ][
        "count_bias"
    ] = derived_bias

    per_seed[
        idx
    ][
        "empty_scene_count"
    ] = len(
        empty_predictions
    )

    per_seed[
        idx
    ][
        "total_tp"
    ] = total_tp

    per_seed[
        idx
    ][
        "total_fp"
    ] = total_fp

    per_seed[
        idx
    ][
        "total_fn"
    ] = total_fn


    print(
        f"Seed-{idx}:"
    )

    print(
        "  pred_count == tp + fp:            PASS"
    )

    print(
        "  gt_count == tp + fn:              PASS"
    )

    print(
        "  count_abs_error algebra:          PASS"
    )

    print(
        "  operating precision reproduction: PASS"
    )

    print(
        "  operating recall reproduction:    PASS"
    )

    print(
        "  operating F1 reproduction:        PASS"
    )

    print(
        "  count MAE reproduction:           PASS"
    )

    print(
        "  empty FP/image reproduction:      PASS"
    )

    print(
        f"  empty test scenes:                "
        f"{len(empty_predictions)}"
    )

    print(
        f"  count bias:                       "
        f"{derived_bias:.6f}"
    )


print()
print(
    "Per-scene pred_count confirmed as formal operating-point "
    "prediction count: PASS"
)


# ============================================================
# 15. AGGREGATE ALL THREE PRECOMMITTED SEEDS
# ============================================================

aggregate_metrics = [
    "mask_ap50",
    "mask_ap75",
    "mask_ap50_95",
    "operating_precision",
    "operating_recall",
    "operating_f1",
    "matched_mask_dice",
    "count_mae",
    "count_bias",
    "empty_fp_per_image",
]


aggregate = {}


for metric in aggregate_metrics:

    values = [
        float(
            per_seed[
                idx
            ][
                metric
            ]
        )
        for idx in [
            1,
            2,
            3,
        ]
    ]


    mean, sd = mean_sd(
        values
    )


    aggregate[
        metric
    ] = {
        "values":
            values,

        "mean":
            mean,

        "sample_sd":
            sd,

        "n":
            3,
    }


# ============================================================
# 16. CONSOLE-ROUNDED AGGREGATE SANITY CHECK
# ============================================================

expected_console_aggregate = {
    "mask_ap50":
        (
            0.2616,
            0.0219,
        ),

    "mask_ap75":
        (
            0.0397,
            0.0037,
        ),

    "mask_ap50_95":
        (
            0.0867,
            0.0051,
        ),

    "operating_precision":
        (
            0.5668,
            0.0497,
        ),

    "operating_recall":
        (
            0.2969,
            0.0319,
        ),

    "operating_f1":
        (
            0.3895,
            0.0375,
        ),

    "matched_mask_dice":
        (
            0.8059,
            0.0034,
        ),

    "count_mae":
        (
            1.1285,
            0.0716,
        ),

    "empty_fp_per_image":
        (
            0.0849,
            0.0290,
        ),
}


for metric, (
    expected_mean,
    expected_sd,
) in expected_console_aggregate.items():

    actual_mean = aggregate[
        metric
    ][
        "mean"
    ]

    actual_sd = aggregate[
        metric
    ][
        "sample_sd"
    ]


    if abs(
        actual_mean
        - expected_mean
    ) > 6e-4:

        raise RuntimeError(
            f"{metric} mean differs unexpectedly "
            "from rounded-console sanity expectation."
        )


    if abs(
        actual_sd
        - expected_sd
    ) > 6e-4:

        raise RuntimeError(
            f"{metric} SD differs unexpectedly "
            "from rounded-console sanity expectation."
        )


print()
print(
    "Formal aggregate sanity checks: PASS"
)


# ============================================================
# 17. WRITE PER-SEED CSV
# ============================================================

per_seed_fields = [
    "seed_index",
    "rng_seed",
    "operating_threshold",
    "mask_ap50",
    "mask_ap75",
    "mask_ap50_95",
    "operating_precision",
    "operating_recall",
    "operating_f1",
    "matched_mask_dice",
    "count_mae",
    "count_bias",
    "empty_fp_per_image",
    "empty_scene_count",
    "total_tp",
    "total_fp",
    "total_fn",
]


with PER_SEED_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=per_seed_fields,
    )

    writer.writeheader()


    for idx in [
        1,
        2,
        3,
    ]:

        writer.writerow(
            {
                field:
                    per_seed[
                        idx
                    ].get(
                        field,
                        "",
                    )
                for field in per_seed_fields
            }
        )


# ============================================================
# 18. WRITE MEAN / SAMPLE-SD CSV
# ============================================================

with MEAN_SD_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "metric",
            "mean",
            "sample_sd",
            "n",
        ],
    )

    writer.writeheader()


    for metric in aggregate_metrics:

        writer.writerow(
            {
                "metric":
                    metric,

                "mean":
                    aggregate[
                        metric
                    ][
                        "mean"
                    ],

                "sample_sd":
                    aggregate[
                        metric
                    ][
                        "sample_sd"
                    ],

                "n":
                    3,
            }
        )


# ============================================================
# 19. FORMAL AGGREGATE JSON
# ============================================================

aggregate_obj = {
    "aggregate_name":
        (
            "DASO-Net Formal FruitMP-Syn Untouched Test "
            "Three-Seed Aggregate v1"
        ),

    "status":
        "PASS",

    "scientific_role":
        (
            "Final three-seed aggregate of the untouched "
            "FruitMP-Syn v1.1 synthetic test split."
        ),

    "wrapper_revision_note":
        (
            "Per-scene evaluator output uses the generic field "
            "'pred_count'. It was accepted as the frozen "
            "operating-point prediction count only after "
            "row-wise verification that pred_count=tp+fp, "
            "gt_count=tp+fn, count_abs_error=abs(pred_count-"
            "gt_count), and aggregate TP/FP/FN/count statistics "
            "reproduced the formal summary JSON."
        ),

    "dataset": {
        "name":
            "FruitMP-Syn v1.1",

        "split":
            "test",

        "scenes_per_seed":
            5000,

        "all_scenes_used":
            True,

        "same_ordered_scenes_across_seeds":
            True,

        "exact_frozen_recipe_set":
            True,
    },

    "protocol": {
        "sha256":
            EXPECTED_PROTOCOL_SHA,

        "changed_after_test":
            False,
    },

    "decoder": {
        "sha256":
            EXPECTED_DECODER_SHA,

        "decode_min_score":
            0.05,

        "topk":
            100,

        "mask_threshold":
            0.50,
    },

    "seed_policy": {
        "n":
            3,

        "all_precommitted_seeds_included":
            True,

        "seed_selection":
            False,

        "seed_ranking":
            False,

        "seed_exclusion":
            False,

        "aggregation":
            "arithmetic mean plus sample SD (n-1)",
    },

    "operating_threshold_policy": {
        "seed1":
            0.30,

        "seed2":
            0.35,

        "seed3":
            0.35,

        "source":
            (
                "synthetic validation only; "
                "frozen before test exposure"
            ),

        "test_threshold_tuning":
            False,
    },

    "per_scene_schema_verification": {
        "gt_count_field":
            "gt_count",

        "operating_prediction_count_field":
            "pred_count",

        "tp_field":
            "tp",

        "fp_field":
            "fp",

        "fn_field":
            "fn",

        "count_abs_error_field":
            "count_abs_error",

        "pred_count_equals_tp_plus_fp_all_rows":
            True,

        "gt_count_equals_tp_plus_fn_all_rows":
            True,

        "count_abs_error_verified_all_rows":
            True,

        "operating_precision_reproduced":
            True,

        "operating_recall_reproduced":
            True,

        "operating_f1_reproduced":
            True,

        "count_mae_reproduced":
            True,

        "empty_fp_per_image_reproduced":
            True,
    },

    "per_seed": {
        f"seed{idx}":
            per_seed[
                idx
            ]
        for idx in [
            1,
            2,
            3,
        ]
    },

    "aggregate":
        aggregate,

    "conditional_metric_warning": {
        "matched_mask_dice":
            (
                "Conditional on matched detections. "
                "Must not be described as an overall "
                "segmentation Dice score."
            ),
    },

    "posthoc_test_sweep_policy": {
        "excluded_from_formal_operating_metrics":
            True,

        "reason":
            (
                "Any evaluator best-threshold/best-F1 field "
                "from the formal test run is post-hoc "
                "test-sweep diagnostic output and cannot "
                "replace the validation-derived frozen "
                "operating thresholds."
            ),

        "evaluator_best_fields_preserved_for_transparency":
            posthoc_best_fields,
    },

    "interpretation_boundary": {
        "these_results_are":
            "untouched internal synthetic-test performance",

        "these_results_are_not": [
            "real-domain external performance",
            "chemical microplastic identification performance",
            "polymer-classification performance",
        ],

        "real_domain_external_evaluation_still_required":
            True,
    },

    "provenance": {
        "formal_run_manifest":
            str(
                RUN_MANIFEST
            ),

        "formal_run_manifest_sha256":
            sha256_file(
                RUN_MANIFEST
            ),

        "overlay_audit":
            str(
                OVERLAY_AUDIT
            ),

        "overlay_audit_sha256":
            sha256_file(
                OVERLAY_AUDIT
            ),

        "seed1_summary_sha256":
            sha256_file(
                SEEDS[
                    1
                ][
                    "summary"
                ]
            ),

        "seed2_summary_sha256":
            sha256_file(
                SEEDS[
                    2
                ][
                    "summary"
                ]
            ),

        "seed3_summary_sha256":
            sha256_file(
                SEEDS[
                    3
                ][
                    "summary"
                ]
            ),

        "seed1_per_scene_sha256":
            sha256_file(
                SEEDS[
                    1
                ][
                    "per_scene"
                ]
            ),

        "seed2_per_scene_sha256":
            sha256_file(
                SEEDS[
                    2
                ][
                    "per_scene"
                ]
            ),

        "seed3_per_scene_sha256":
            sha256_file(
                SEEDS[
                    3
                ][
                    "per_scene"
                ]
            ),
    },

    "policy": {
        "training_performed":
            False,

        "inference_performed_by_this_aggregate_cell":
            False,

        "test_threshold_tuning":
            False,

        "checkpoint_selection":
            False,

        "best_seed_selection":
            False,

        "frozen_data_modified":
            False,
    },
}


save_json(
    AGGREGATE_JSON,
    aggregate_obj,
)


# ============================================================
# 20. MARKDOWN TABLE
# ============================================================

def fmt(
    value,
):

    return f"{float(value):.4f}"


def fmt_pm(
    metric,
):

    return (
        f"{aggregate[metric]['mean']:.4f} "
        f"± {aggregate[metric]['sample_sd']:.4f}"
    )


markdown = f"""# DASO-Net Formal Untouched FruitMP-Syn Test

All 3 precommitted final seeds; 5,000 frozen test scenes per seed.

| Metric | Seed 1 | Seed 2 | Seed 3 | Mean ± sample SD |
|---|---:|---:|---:|---:|
| Mask AP50 | {fmt(per_seed[1]['mask_ap50'])} | {fmt(per_seed[2]['mask_ap50'])} | {fmt(per_seed[3]['mask_ap50'])} | {fmt_pm('mask_ap50')} |
| Mask AP75 | {fmt(per_seed[1]['mask_ap75'])} | {fmt(per_seed[2]['mask_ap75'])} | {fmt(per_seed[3]['mask_ap75'])} | {fmt_pm('mask_ap75')} |
| Mask AP50:95 | {fmt(per_seed[1]['mask_ap50_95'])} | {fmt(per_seed[2]['mask_ap50_95'])} | {fmt(per_seed[3]['mask_ap50_95'])} | {fmt_pm('mask_ap50_95')} |
| Operating precision | {fmt(per_seed[1]['operating_precision'])} | {fmt(per_seed[2]['operating_precision'])} | {fmt(per_seed[3]['operating_precision'])} | {fmt_pm('operating_precision')} |
| Operating recall | {fmt(per_seed[1]['operating_recall'])} | {fmt(per_seed[2]['operating_recall'])} | {fmt(per_seed[3]['operating_recall'])} | {fmt_pm('operating_recall')} |
| Operating F1 | {fmt(per_seed[1]['operating_f1'])} | {fmt(per_seed[2]['operating_f1'])} | {fmt(per_seed[3]['operating_f1'])} | {fmt_pm('operating_f1')} |
| Matched-mask Dice* | {fmt(per_seed[1]['matched_mask_dice'])} | {fmt(per_seed[2]['matched_mask_dice'])} | {fmt(per_seed[3]['matched_mask_dice'])} | {fmt_pm('matched_mask_dice')} |
| Count MAE | {fmt(per_seed[1]['count_mae'])} | {fmt(per_seed[2]['count_mae'])} | {fmt(per_seed[3]['count_mae'])} | {fmt_pm('count_mae')} |
| Count bias | {fmt(per_seed[1]['count_bias'])} | {fmt(per_seed[2]['count_bias'])} | {fmt(per_seed[3]['count_bias'])} | {fmt_pm('count_bias')} |
| Empty-scene FP/image | {fmt(per_seed[1]['empty_fp_per_image'])} | {fmt(per_seed[2]['empty_fp_per_image'])} | {fmt(per_seed[3]['empty_fp_per_image'])} | {fmt_pm('empty_fp_per_image')} |

Frozen operating thresholds: Seed 1 = 0.30, Seed 2 = 0.35, Seed 3 = 0.35.

*Matched-mask Dice is conditional on matched detections and is not an overall segmentation Dice score.

The evaluator's per-scene `pred_count` field was verified algebraically against `tp`, `fp`, `fn`, `count_abs_error`, and the formal operating-point summary metrics before being used.

Any evaluator `best*` threshold/F1 values generated on this test split are post-hoc sensitivity diagnostics only and are excluded from the formal operating-point results.
"""


MARKDOWN_TABLE.write_text(
    markdown,
    encoding="utf-8",
)


# ============================================================
# 21. HASH FORMAL AGGREGATE
# ============================================================

aggregate_sha = sha256_file(
    AGGREGATE_JSON
)


SHA_RECORD.write_text(
    (
        f"{aggregate_sha}  "
        f"{AGGREGATE_JSON.name}\n"
    ),
    encoding="utf-8",
)


# ============================================================
# 22. READ-BACK AUDIT
# ============================================================

readback = load_json(
    AGGREGATE_JSON
)


if readback.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Formal aggregate read-back status failure."
    )


if bool(
    readback[
        "seed_policy"
    ][
        "seed_selection"
    ]
):

    raise RuntimeError(
        "Aggregate unexpectedly indicates seed selection."
    )


if bool(
    readback[
        "operating_threshold_policy"
    ][
        "test_threshold_tuning"
    ]
):

    raise RuntimeError(
        "Aggregate unexpectedly indicates test-threshold tuning."
    )


if int(
    readback[
        "seed_policy"
    ][
        "n"
    ]
) != 3:

    raise RuntimeError(
        "Aggregate seed count != 3."
    )


if not bool(
    readback[
        "per_scene_schema_verification"
    ][
        "pred_count_equals_tp_plus_fp_all_rows"
    ]
):

    raise RuntimeError(
        "Per-scene pred_count verification missing."
    )


# ============================================================
# 23. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FINAL FORMAL THREE-SEED UNTOUCHED-TEST AGGREGATE"
)

print(
    "============================================================"
)

print()

print(
    f"Mask AP50:              "
    f"{fmt_pm('mask_ap50')}"
)

print(
    f"Mask AP75:              "
    f"{fmt_pm('mask_ap75')}"
)

print(
    f"Mask AP50:95:           "
    f"{fmt_pm('mask_ap50_95')}"
)

print()

print(
    f"Operating precision:    "
    f"{fmt_pm('operating_precision')}"
)

print(
    f"Operating recall:       "
    f"{fmt_pm('operating_recall')}"
)

print(
    f"Operating F1:           "
    f"{fmt_pm('operating_f1')}"
)

print()

print(
    f"Matched-mask Dice*:     "
    f"{fmt_pm('matched_mask_dice')}"
)

print(
    f"Count MAE:              "
    f"{fmt_pm('count_mae')}"
)

print(
    f"Count bias:             "
    f"{fmt_pm('count_bias')}"
)

print(
    f"Empty FP/image:         "
    f"{fmt_pm('empty_fp_per_image')}"
)

print()

print(
    "Frozen operating thresholds:"
)

print(
    "  Seed 1: 0.30"
)

print(
    "  Seed 2: 0.35"
)

print(
    "  Seed 3: 0.35"
)

print()

print(
    "Per-scene pred_count interpretation:"
)

print(
    "  pred_count == tp + fp:       PASS for all 15,000 rows"
)

print(
    "  gt_count == tp + fn:         PASS for all 15,000 rows"
)

print(
    "  summary metrics reproduced:  PASS"
)

print()

print(
    "Post-hoc evaluator best* fields:"
)

print(
    "  PRESERVED as diagnostics"
)

print(
    "  EXCLUDED from formal operating metrics"
)

print()

print(
    "Aggregate JSON:"
)

print(
    AGGREGATE_JSON
)

print()

print(
    "Aggregate SHA256:"
)

print(
    aggregate_sha
)

print()

print(
    "Per-seed CSV:"
)

print(
    PER_SEED_CSV
)

print()

print(
    "Mean/SD CSV:"
)

print(
    MEAN_SD_CSV
)

print()

print(
    "Markdown table:"
)

print(
    MARKDOWN_TABLE
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET FORMAL THREE-SEED UNTOUCHED-TEST AGGREGATE: PASS"
)

print(
    "============================================================"
)

print()

print(
    "NO TRAINING WAS PERFORMED."
)

print(
    "NO INFERENCE WAS PERFORMED."
)

print(
    "NO TEST-SET THRESHOLD TUNING WAS PERFORMED."
)

print(
    "NO BEST-SEED SELECTION WAS PERFORMED."
)

print(
    "ALL THREE PRECOMMITTED FINAL SEEDS WERE INCLUDED."
)

print()

print(
    "* Matched-mask Dice is conditional on matched detections."
)

print()

print(
    "IMPORTANT:"
)

print(
    "These are final INTERNAL SYNTHETIC untouched-test results."
)

print(
    "They are not real-domain external performance."
)

print()

print(
    "STOP HERE."
)

print(
    "Next: freeze/preserve this formal aggregate, then run "
    "the predeclared held-out clean FruitBG false-positive "
    "evaluation."
)

DASO-Net — FORMAL THREE-SEED UNTOUCHED-TEST AGGREGATE v1
PASS  Formal run manifest
PASS  Frozen evaluation protocol
PASS  Formal-test overlay audit
PASS  Frozen 5000-scene test recipes
All three raw formal result sets present: PASS

Existing empty derived aggregate directory detected.
Resume mode: REUSE EMPTY DIRECTORY

Frozen protocol SHA: PASS
Formal run manifest: PASS
Training during test: NO
Test-threshold tuning: NO
Best-seed selection: NO
Seed-1 raw result SHA integrity: PASS
Seed-2 raw result SHA integrity: PASS
Seed-3 raw result SHA integrity: PASS

Frozen operating thresholds:
  Seed 1: 0.30 — PASS
  Seed 2: 0.35 — PASS
  Seed 3: 0.35 — PASS

Same ordered 5000 scenes across all seeds: PASS
Exact set matches frozen test recipes: PASS

VERIFYING HEADLINE METRICS AGAINST RAW SUMMARY JSON
Seed-1 headline source verification: PASS
Seed-2 headline source verification: PASS
Seed-3 headline source verification: PASS

All formal headline metrics verified: PASS

PER-SCENE OPERATING-POIN

In [19]:
# ============================================================
# DASO-Net — PRESERVE FORMAL SYNTHETIC TEST
# + CLEAN FRUITBG IMPLEMENTATION AUDIT v1
#
# PURPOSE
# -------
# 1. Cryptographically preserve the completed formal synthetic
#    untouched-test evidence.
#
# 2. Audit the exact available model/evaluator/preprocessing
#    interfaces before running the predeclared 246-image
#    held-out clean FruitBG false-positive evaluation.
#
# NO TRAINING.
# NO MODEL INFERENCE.
# NO THRESHOLD CHANGES.
# NO FROZEN SCIENTIFIC FILE MODIFICATION.
# ============================================================

from __future__ import annotations

import ast
import csv
import hashlib
import json
import re
import shutil
import subprocess
import sys
import zipfile
from collections import Counter
from pathlib import Path

import cv2


# ============================================================
# 1. ROOTS
# ============================================================

WORKING = Path(
    "/kaggle/working"
)

CODE_ROOT = (
    WORKING
    / "DASO-Net-Phase5B-KaggleTraining"
)

OVERLAY_ROOT = (
    WORKING
    / "DASO-Net-Phase1A-FORMALTEST-OVERLAY-v1"
)

FORMAL_TEST_ROOT = (
    WORKING
    / "dasonet_final_evaluation"
    / "fruitmp_syn_test_v1_resume01_assets_complete"
)

ANALYSIS_ROOT = (
    WORKING
    / "dasonet_final_analysis"
)

FORMAL_AGG_ROOT = (
    ANALYSIS_ROOT
    / "formal_fruitmp_syn_test_three_seed_v1"
)

FORMAL_AGG_JSON = (
    FORMAL_AGG_ROOT
    / "FINAL_FORMAL_FRUITMP_SYN_TEST_THREE_SEED_AGGREGATE_v1.json"
)

FORMAL_AGG_SHA_RECORD = (
    FORMAL_AGG_ROOT
    / "FINAL_FORMAL_FRUITMP_SYN_TEST_THREE_SEED_AGGREGATE_v1.sha256.txt"
)

FORMAL_PER_SEED_CSV = (
    FORMAL_AGG_ROOT
    / "formal_fruitmp_syn_test_per_seed.csv"
)

FORMAL_MEAN_SD_CSV = (
    FORMAL_AGG_ROOT
    / "formal_fruitmp_syn_test_mean_sd.csv"
)

FORMAL_TABLE = (
    FORMAL_AGG_ROOT
    / "formal_fruitmp_syn_test_table.md"
)

PROTOCOL_ROOT = (
    ANALYSIS_ROOT
    / "formal_evaluation_protocol_freeze_v1"
)

PROTOCOL_JSON = (
    PROTOCOL_ROOT
    / "FORMAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

PROTOCOL_README = (
    PROTOCOL_ROOT
    / "README_FORMAL_EVALUATION_PROTOCOL.txt"
)

PROTOCOL_SHA_RECORD = (
    PROTOCOL_ROOT
    / "FORMAL_EVALUATION_PROTOCOL_FREEZE_v1.sha256.txt"
)

RUN_MANIFEST = (
    FORMAL_TEST_ROOT
    / "FORMAL_FRUITMP_SYN_TEST_RUN_MANIFEST_v1_RESUME01.json"
)

OVERLAY_AUDIT = (
    WORKING
    / "DASO-Net-FORMALTEST-OVERLAY-AUDIT-v1.json"
)

ASSET_FAILURE_AUDIT = (
    WORKING
    / "DASO-Net-FORMAL-TEST-ASSET-AUDIT-v1.json"
)


# ============================================================
# 2. KNOWN FROZEN HASHES
# ============================================================

EXPECTED_AGG_SHA = (
    "88ab1d74ebc6ff6ee9b957d17392587f"
    "4907315a48b94cd4652ecd023d2fd6df"
)

EXPECTED_PROTOCOL_SHA = (
    "a98ac6c5af42c7dc84bc7e7c00b7a9ed"
    "661d45af6882bdb0029c49b8d0cb5b62"
)

EXPECTED_FRUITBG_MANIFEST_SHA = (
    "d0108ac720cb03dd565cc6c1fb792762"
    "e7afea808ee596f4b7b2be82e2f223fe"
)


# ============================================================
# 3. CLEAN FRUITBG
# ============================================================

FRUITBG_MANIFEST = (
    OVERLAY_ROOT
    / "data"
    / "fruit_bg"
    / "v1_final"
    / "fruitbg_rgb_v1_manifest.csv"
)

EXPECTED_FRUITBG_TEST = 246


# ============================================================
# 4. SOURCE FILES TO AUDIT
# ============================================================

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

FRUITMP_DATASET = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "datasets"
    / "fruitmp_syn.py"
)

TRAINING_DATA = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "training"
    / "data.py"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

MODEL_DIR = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
)

SCRIPTS_DIR = (
    CODE_ROOT
    / "scripts"
)


# ============================================================
# 5. PRESERVATION PACKAGE
# ============================================================

PACKAGE_NAME = (
    "DASO-Net-FORMAL-SYNTEST-COMPLETE-20260919"
)

PACKAGE_ROOT = (
    WORKING
    / PACKAGE_NAME
)

PACKAGE_MANIFEST = (
    PACKAGE_ROOT
    / "FORMAL_SYNTEST_PRESERVATION_MANIFEST.json"
)

AUDIT_JSON = (
    PACKAGE_ROOT
    / "CLEAN_FRUITBG_IMPLEMENTATION_AUDIT_v1.json"
)

README = (
    PACKAGE_ROOT
    / "README.txt"
)

ZIP_PATH = (
    WORKING
    / f"{PACKAGE_NAME}.zip"
)

ZIP_SHA_RECORD = (
    WORKING
    / f"{PACKAGE_NAME}.sha256.txt"
)


# ============================================================
# 6. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(
    path: Path,
    obj,
) -> None:

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def read_csv(
    path: Path,
):

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def normalize_split(
    value,
):

    s = str(
        value
    ).strip().lower()

    mapping = {
        "train":
            "train",

        "training":
            "train",

        "val":
            "val",

        "valid":
            "val",

        "validation":
            "val",

        "dev":
            "val",

        "test":
            "test",

        "testing":
            "test",
    }

    return mapping.get(
        s
    )


def normalize_path(
    value,
) -> str:

    return str(
        value
    ).strip().replace(
        "\\",
        "/",
    )


def project_relative_source_path(
    value,
) -> Path:

    text = normalize_path(
        value
    )

    marker = (
        "DASO-Net-Phase1A/"
    )


    if marker not in text:

        raise RuntimeError(
            "Could not map source_absolute_path:\n"
            f"{text}"
        )


    return Path(
        text.split(
            marker,
            1,
        )[1]
    )


def extract_ast_symbols(
    path: Path,
):

    source = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    tree = ast.parse(
        source
    )

    functions = []
    classes = []


    for node in ast.walk(
        tree
    ):

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        ):

            functions.append(
                {
                    "name":
                        node.name,

                    "line":
                        int(
                            node.lineno
                        ),
                }
            )


        elif isinstance(
            node,
            ast.ClassDef,
        ):

            classes.append(
                {
                    "name":
                        node.name,

                    "line":
                        int(
                            node.lineno
                        ),
                }
            )


    functions = sorted(
        functions,
        key=lambda x: x[
            "line"
        ],
    )

    classes = sorted(
        classes,
        key=lambda x: x[
            "line"
        ],
    )


    return {
        "functions":
            functions,

        "classes":
            classes,
    }


def source_keyword_hits(
    path: Path,
    keywords,
):

    lines = path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    hits = []


    for line_number, line in enumerate(
        lines,
        start=1,
    ):

        lower = line.lower()


        if any(
            keyword.lower()
            in lower
            for keyword in keywords
        ):

            hits.append(
                {
                    "line":
                        line_number,

                    "text":
                        line.strip(),
                }
            )


    return hits


# ============================================================
# 7. PREFLIGHT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — PRESERVE FORMAL SYNTHETIC TEST"
)

print(
    "+ CLEAN FRUITBG IMPLEMENTATION AUDIT v1"
)

print(
    "============================================================"
)


for path, label in [
    (
        FORMAL_AGG_JSON,
        "Formal synthetic-test aggregate",
    ),
    (
        FORMAL_AGG_SHA_RECORD,
        "Formal aggregate SHA record",
    ),
    (
        FORMAL_PER_SEED_CSV,
        "Formal per-seed table",
    ),
    (
        FORMAL_MEAN_SD_CSV,
        "Formal mean/SD table",
    ),
    (
        FORMAL_TABLE,
        "Formal Markdown table",
    ),
    (
        PROTOCOL_JSON,
        "Frozen formal protocol",
    ),
    (
        PROTOCOL_README,
        "Frozen protocol README",
    ),
    (
        RUN_MANIFEST,
        "Formal synthetic-test run manifest",
    ),
    (
        OVERLAY_AUDIT,
        "Formal project-overlay audit",
    ),
    (
        ASSET_FAILURE_AUDIT,
        "Prior asset-failure provenance",
    ),
    (
        FRUITBG_MANIFEST,
        "Frozen FruitBG manifest",
    ),
    (
        EVALUATOR,
        "Validated evaluator",
    ),
    (
        FRUITMP_DATASET,
        "FruitMP synthetic dataset implementation",
    ),
    (
        TRAINING_DATA,
        "Training data implementation",
    ),
    (
        DECODER,
        "Validated decoder",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 8. VERIFY FINAL FORMAL SYNTHETIC AGGREGATE
# ============================================================

actual_aggregate_sha = sha256_file(
    FORMAL_AGG_JSON
)


if (
    actual_aggregate_sha
    != EXPECTED_AGG_SHA
):

    raise RuntimeError(
        "Formal synthetic-test aggregate SHA mismatch.\n"
        f"Expected: {EXPECTED_AGG_SHA}\n"
        f"Actual:   {actual_aggregate_sha}"
    )


aggregate = load_json(
    FORMAL_AGG_JSON
)


if aggregate.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Formal synthetic-test aggregate != PASS."
    )


if bool(
    aggregate[
        "seed_policy"
    ][
        "seed_selection"
    ]
):

    raise RuntimeError(
        "Formal aggregate unexpectedly indicates seed selection."
    )


if bool(
    aggregate[
        "operating_threshold_policy"
    ][
        "test_threshold_tuning"
    ]
):

    raise RuntimeError(
        "Formal aggregate unexpectedly indicates "
        "test threshold tuning."
    )


print()
print(
    "Formal untouched synthetic-test aggregate SHA: PASS"
)

print(
    "All 3 precommitted seeds retained: PASS"
)

print(
    "Test threshold tuning: NO"
)


# ============================================================
# 9. VERIFY PROTOCOL AGAIN
# ============================================================

if (
    sha256_file(
        PROTOCOL_JSON
    )
    != EXPECTED_PROTOCOL_SHA
):

    raise RuntimeError(
        "Frozen protocol SHA mismatch."
    )


protocol = load_json(
    PROTOCOL_JSON
)


clean_protocol = protocol.get(
    "held_out_clean_fruitbg",
    {}
)


if int(
    clean_protocol.get(
        "num_backgrounds",
        -1,
    )
) != 246:

    raise RuntimeError(
        "Frozen protocol does not specify 246 clean "
        "FruitBG test backgrounds."
    )


if bool(
    clean_protocol.get(
        "threshold_tuning",
        True,
    )
):

    raise RuntimeError(
        "Frozen protocol unexpectedly permits clean-set "
        "threshold tuning."
    )


print(
    "Frozen clean-FruitBG evaluation clause: PASS"
)


# ============================================================
# 10. VERIFY FRUITBG TEST ASSETS
# ============================================================

if (
    sha256_file(
        FRUITBG_MANIFEST
    )
    != EXPECTED_FRUITBG_MANIFEST_SHA
):

    raise RuntimeError(
        "FruitBG manifest SHA mismatch."
    )


fruitbg_rows = read_csv(
    FRUITBG_MANIFEST
)


test_rows = [
    row
    for row in fruitbg_rows
    if normalize_split(
        row[
            "fruitbg_split"
        ]
    )
    == "test"
]


if len(
    test_rows
) != EXPECTED_FRUITBG_TEST:

    raise RuntimeError(
        "FruitBG test row count != 246."
    )


source_paths = []
mask_paths = []


for row in test_rows:

    source_rel = project_relative_source_path(
        row[
            "source_absolute_path"
        ]
    )

    mask_rel = Path(
        normalize_path(
            row[
                "frozen_surface_mask_path"
            ]
        )
    )


    source = (
        OVERLAY_ROOT
        / source_rel
    )

    mask = (
        OVERLAY_ROOT
        / mask_rel
    )


    if not source.is_file():

        raise RuntimeError(
            "Clean FruitBG test source missing:\n"
            f"{source}"
        )


    if not mask.is_file():

        raise RuntimeError(
            "Clean FruitBG frozen mask missing:\n"
            f"{mask}"
        )


    source_paths.append(
        source
    )

    mask_paths.append(
        mask
    )


if len(
    set(
        source_paths
    )
) != 246:

    raise RuntimeError(
        "Expected 246 unique clean FruitBG source images."
    )


if len(
    set(
        mask_paths
    )
) != 246:

    raise RuntimeError(
        "Expected 246 unique clean FruitBG masks."
    )


print()
print(
    "Held-out clean FruitBG source RGB: 246/246 — PASS"
)

print(
    "Held-out clean FruitBG frozen masks: 246/246 — PASS"
)


# ============================================================
# 11. AUDIT CLEAN IMAGE DIMENSIONS
#
# Reading images is NOT model inference.
# ============================================================

shape_counts = Counter()

channel_counts = Counter()

failed_reads = []


for path in source_paths:

    image = cv2.imread(
        str(
            path
        ),
        cv2.IMREAD_COLOR,
    )


    if image is None:

        failed_reads.append(
            str(
                path
            )
        )

        continue


    shape_counts[
        tuple(
            image.shape[
                :2
            ]
        )
    ] += 1

    channel_counts[
        int(
            image.shape[
                2
            ]
        )
    ] += 1


if failed_reads:

    raise RuntimeError(
        "Could not read clean FruitBG test images:\n"
        + "\n".join(
            failed_reads[
                :20
            ]
        )
    )


mask_shape_counts = Counter()


for path in mask_paths:

    mask = cv2.imread(
        str(
            path
        ),
        cv2.IMREAD_GRAYSCALE,
    )


    if mask is None:

        raise RuntimeError(
            "Could not read clean FruitBG mask:\n"
            f"{path}"
        )


    mask_shape_counts[
        tuple(
            mask.shape[
                :2
            ]
        )
    ] += 1


print()
print(
    "Clean FruitBG RGB shape distribution:"
)


for shape, count in sorted(
    shape_counts.items()
):

    print(
        f"  {shape}: {count}"
    )


print()
print(
    "Clean FruitBG mask shape distribution:"
)


for shape, count in sorted(
    mask_shape_counts.items()
):

    print(
        f"  {shape}: {count}"
    )


# ============================================================
# 12. MANIFEST FIELD AUDIT
# ============================================================

manifest_fields = list(
    fruitbg_rows[
        0
    ].keys()
)


print()
print(
    "============================================================"
)

print(
    "FRUITBG MANIFEST FIELDS"
)

print(
    "============================================================"
)


for field in manifest_fields:

    print(
        f"  {field}"
    )


print()
print(
    "Representative TEST row:"
)


first_test_row = test_rows[
    0
]


for field in manifest_fields:

    value = first_test_row.get(
        field,
        ""
    )


    if len(
        str(
            value
        )
    ) > 200:

        value = (
            str(
                value
            )[
                :200
            ]
            + "..."
        )


    print(
        f"  {field}: {value}"
    )


# ============================================================
# 13. AUDIT AVAILABLE EVALUATION SCRIPTS
# ============================================================

script_files = sorted(
    p
    for p in SCRIPTS_DIR.glob(
        "*.py"
    )
)


print()
print(
    "============================================================"
)

print(
    "AVAILABLE EVALUATION / INFERENCE SCRIPTS"
)

print(
    "============================================================"
)


relevant_scripts = []


for path in script_files:

    name = path.name.lower()


    if any(
        token
        in name
        for token in [
            "eval",
            "infer",
            "predict",
            "clean",
            "fruit",
            "phase5",
        ]
    ):

        relevant_scripts.append(
            path
        )

        print(
            f"  {path.name}"
        )


# ============================================================
# 14. EVALUATOR --HELP
#
# No model loading/inference occurs.
# ============================================================

print()
print(
    "============================================================"
)

print(
    "VALIDATED EVALUATOR CLI"
)

print(
    "============================================================"
)


help_proc = subprocess.run(
    [
        sys.executable,
        str(
            EVALUATOR
        ),
        "--help",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    timeout=60,
)


print(
    help_proc.stdout
)


if help_proc.returncode != 0:

    raise RuntimeError(
        "Evaluator --help failed."
    )


# ============================================================
# 15. AST SYMBOL AUDIT
# ============================================================

source_files = {
    "evaluate_phase5c":
        EVALUATOR,

    "fruitmp_syn":
        FRUITMP_DATASET,

    "training_data":
        TRAINING_DATA,

    "decoder":
        DECODER,
}


ast_audit = {}


print()
print(
    "============================================================"
)

print(
    "RELEVANT PYTHON SYMBOLS"
)

print(
    "============================================================"
)


for label, path in source_files.items():

    symbols = extract_ast_symbols(
        path
    )


    ast_audit[
        label
    ] = symbols


    print()
    print(
        f"[{label}]"
    )


    for record in symbols[
        "classes"
    ]:

        name_lower = record[
            "name"
        ].lower()


        if any(
            token
            in name_lower
            for token in [
                "dataset",
                "model",
                "eval",
                "infer",
                "render",
                "decoder",
            ]
        ):

            print(
                f"  CLASS "
                f"{record['name']} "
                f"@ line {record['line']}"
            )


    for record in symbols[
        "functions"
    ]:

        name_lower = record[
            "name"
        ].lower()


        if any(
            token
            in name_lower
            for token in [
                "model",
                "checkpoint",
                "load",
                "decode",
                "infer",
                "eval",
                "dataset",
                "batch",
                "image",
                "scene",
                "config",
            ]
        ):

            print(
                f"  FUNC  "
                f"{record['name']} "
                f"@ line {record['line']}"
            )


# ============================================================
# 16. SOURCE KEYWORD AUDIT
#
# Print only implementation-relevant lines.
# ============================================================

keyword_sets = {
    "evaluate_phase5c": [
        "checkpoint",
        "load_state_dict",
        "DASONet",
        "decode",
        "DataLoader",
        "FruitMPSyn",
        "project_root",
        "torch.load",
        "model(",
        ".eval(",
    ],

    "fruitmp_syn": [
        "clean",
        "composite",
        "image",
        "build_scene",
        "renderer",
        "transform",
        "resize",
        "background",
    ],

    "training_data": [
        "image",
        "normalize",
        "resize",
        "tensor",
        "dataset",
        "collate",
    ],
}


keyword_audit = {}


print()
print(
    "============================================================"
)

print(
    "IMPLEMENTATION KEYWORD HITS"
)

print(
    "============================================================"
)


for label, keywords in keyword_sets.items():

    path = source_files[
        label
    ]

    hits = source_keyword_hits(
        path,
        keywords,
    )


    keyword_audit[
        label
    ] = hits


    print()
    print(
        f"[{label}]"
    )


    for hit in hits[
        :120
    ]:

        print(
            f"  L{hit['line']}: "
            f"{hit['text']}"
        )


# ============================================================
# 17. MODEL MODULE INVENTORY
# ============================================================

model_python_files = sorted(
    p
    for p in MODEL_DIR.rglob(
        "*.py"
    )
)


print()
print(
    "============================================================"
)

print(
    "MODEL MODULE INVENTORY"
)

print(
    "============================================================"
)


model_inventory = []


for path in model_python_files:

    rel = path.relative_to(
        CODE_ROOT
    )

    symbols = extract_ast_symbols(
        path
    )


    classes = [
        item[
            "name"
        ]
        for item in symbols[
            "classes"
        ]
    ]


    functions = [
        item[
            "name"
        ]
        for item in symbols[
            "functions"
        ]
    ]


    model_inventory.append(
        {
            "path":
                str(
                    rel
                ),

            "classes":
                classes,

            "functions":
                functions,
        }
    )


    print(
        f"  {rel}"
    )


    if classes:

        print(
            "    classes: "
            + ", ".join(
                classes
            )
        )


# ============================================================
# 18. CREATE PRESERVATION PACKAGE
# ============================================================

if PACKAGE_ROOT.exists():

    raise RuntimeError(
        "Preservation package already exists:\n"
        f"{PACKAGE_ROOT}\n"
        "Do not overwrite it."
    )


if ZIP_PATH.exists():

    raise RuntimeError(
        "Preservation ZIP already exists:\n"
        f"{ZIP_PATH}"
    )


if ZIP_SHA_RECORD.exists():

    raise RuntimeError(
        "Preservation ZIP SHA record already exists:\n"
        f"{ZIP_SHA_RECORD}"
    )


PACKAGE_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 19. COPY FORMAL EVIDENCE
# ============================================================

copy_items = [
    (
        FORMAL_AGG_JSON,
        "formal_aggregate/"
        + FORMAL_AGG_JSON.name,
    ),

    (
        FORMAL_AGG_SHA_RECORD,
        "formal_aggregate/"
        + FORMAL_AGG_SHA_RECORD.name,
    ),

    (
        FORMAL_PER_SEED_CSV,
        "formal_aggregate/"
        + FORMAL_PER_SEED_CSV.name,
    ),

    (
        FORMAL_MEAN_SD_CSV,
        "formal_aggregate/"
        + FORMAL_MEAN_SD_CSV.name,
    ),

    (
        FORMAL_TABLE,
        "formal_aggregate/"
        + FORMAL_TABLE.name,
    ),

    (
        PROTOCOL_JSON,
        "protocol/"
        + PROTOCOL_JSON.name,
    ),

    (
        PROTOCOL_README,
        "protocol/"
        + PROTOCOL_README.name,
    ),

    (
        PROTOCOL_SHA_RECORD,
        "protocol/"
        + PROTOCOL_SHA_RECORD.name,
    ),

    (
        RUN_MANIFEST,
        "formal_run/"
        + RUN_MANIFEST.name,
    ),

    (
        OVERLAY_AUDIT,
        "provenance/"
        + OVERLAY_AUDIT.name,
    ),

    (
        ASSET_FAILURE_AUDIT,
        "provenance/"
        + ASSET_FAILURE_AUDIT.name,
    ),
]


for idx in [
    1,
    2,
    3,
]:

    source_root = (
        FORMAL_TEST_ROOT
        / f"seed0{idx}_s{SEEDS_RNG}"
    ) if False else None


# Explicit paths to avoid accidental dynamic mistakes.
seed_items = [
    (
        FORMAL_TEST_ROOT
        / "seed01_s1782656963"
        / "phase5c_metric_summary.json",

        "raw_formal_test/"
        "seed01_s1782656963/"
        "phase5c_metric_summary.json",
    ),

    (
        FORMAL_TEST_ROOT
        / "seed01_s1782656963"
        / "phase5c_per_scene.csv",

        "raw_formal_test/"
        "seed01_s1782656963/"
        "phase5c_per_scene.csv",
    ),

    (
        FORMAL_TEST_ROOT
        / "seed01_evaluation.log",

        "raw_formal_test/"
        "seed01_evaluation.log",
    ),

    (
        FORMAL_TEST_ROOT
        / "seed02_s901395089"
        / "phase5c_metric_summary.json",

        "raw_formal_test/"
        "seed02_s901395089/"
        "phase5c_metric_summary.json",
    ),

    (
        FORMAL_TEST_ROOT
        / "seed02_s901395089"
        / "phase5c_per_scene.csv",

        "raw_formal_test/"
        "seed02_s901395089/"
        "phase5c_per_scene.csv",
    ),

    (
        FORMAL_TEST_ROOT
        / "seed02_evaluation.log",

        "raw_formal_test/"
        "seed02_evaluation.log",
    ),

    (
        FORMAL_TEST_ROOT
        / "seed03_s252916394"
        / "phase5c_metric_summary.json",

        "raw_formal_test/"
        "seed03_s252916394/"
        "phase5c_metric_summary.json",
    ),

    (
        FORMAL_TEST_ROOT
        / "seed03_s252916394"
        / "phase5c_per_scene.csv",

        "raw_formal_test/"
        "seed03_s252916394/"
        "phase5c_per_scene.csv",
    ),

    (
        FORMAL_TEST_ROOT
        / "seed03_evaluation.log",

        "raw_formal_test/"
        "seed03_evaluation.log",
    ),
]


copy_items.extend(
    seed_items
)


for src, rel in copy_items:

    require(
        src,
        f"Preservation source {src.name}",
    )


    dst = (
        PACKAGE_ROOT
        / rel
    )


    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    shutil.copy2(
        src,
        dst,
    )


# ============================================================
# 20. CLEAN IMPLEMENTATION AUDIT JSON
# ============================================================

clean_audit = {
    "audit_name":
        (
            "DASO-Net Clean FruitBG "
            "Implementation Audit v1"
        ),

    "status":
        "PASS",

    "model_inference_performed":
        False,

    "training_performed":
        False,

    "formal_synthetic_aggregate_sha256":
        EXPECTED_AGG_SHA,

    "formal_protocol_sha256":
        EXPECTED_PROTOCOL_SHA,

    "clean_fruitbg": {
        "test_rows":
            246,

        "unique_source_rgb":
            len(
                set(
                    str(
                        p
                    )
                    for p in source_paths
                )
            ),

        "unique_frozen_masks":
            len(
                set(
                    str(
                        p
                    )
                    for p in mask_paths
                )
            ),

        "rgb_shape_distribution":
            {
                str(
                    shape
                ):
                    count
                for shape, count
                in sorted(
                    shape_counts.items()
                )
            },

        "mask_shape_distribution":
            {
                str(
                    shape
                ):
                    count
                for shape, count
                in sorted(
                    mask_shape_counts.items()
                )
            },

        "manifest_fields":
            manifest_fields,
    },

    "frozen_clean_evaluation_policy":
        clean_protocol,

    "available_relevant_scripts": [
        str(
            p.relative_to(
                CODE_ROOT
            )
        )
        for p in relevant_scripts
    ],

    "source_ast":
        ast_audit,

    "source_keyword_hits":
        keyword_audit,

    "model_inventory":
        model_inventory,
}


save_json(
    AUDIT_JSON,
    clean_audit,
)


# ============================================================
# 21. PRESERVATION MANIFEST
# ============================================================

preserved_files = []


for path in sorted(
    p
    for p in PACKAGE_ROOT.rglob(
        "*"
    )
    if p.is_file()
):

    preserved_files.append(
        {
            "relative_path":
                path.relative_to(
                    PACKAGE_ROOT
                ).as_posix(),

            "size_bytes":
                path.stat().st_size,

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "package_name":
        PACKAGE_NAME,

    "status":
        "PASS",

    "scientific_state":
        (
            "Formal internal synthetic untouched-test "
            "evaluation complete."
        ),

    "formal_synthetic_test": {
        "aggregate_sha256":
            EXPECTED_AGG_SHA,

        "protocol_sha256":
            EXPECTED_PROTOCOL_SHA,

        "all_three_final_seeds":
            True,

        "training_after_final_seeds":
            False,

        "test_threshold_tuning":
            False,

        "best_seed_selection":
            False,
    },

    "next_stage":
        (
            "held-out clean FruitBG false-positive evaluation"
        ),

    "preserved_files":
        preserved_files,
}


save_json(
    PACKAGE_MANIFEST,
    manifest,
)


# ============================================================
# 22. README
# ============================================================

README.write_text(
    f"""
DASO-Net Formal Synthetic-Test Complete — 2026-09-19
=====================================================

Formal internal synthetic untouched-test aggregate:
  SHA256:
  {EXPECTED_AGG_SHA}

Formal evaluation protocol:
  SHA256:
  {EXPECTED_PROTOCOL_SHA}

Formal internal test:
  FruitMP-Syn v1.1
  split = test
  5000 scenes per seed
  3 precommitted final seeds
  no seed selection
  no threshold tuning

Primary aggregate:
  Mask AP50    = 0.2616 +/- 0.0219
  Mask AP75    = 0.0397 +/- 0.0037
  Mask AP50:95 = 0.0867 +/- 0.0051

Frozen operating-point aggregate:
  Precision = 0.5668 +/- 0.0497
  Recall    = 0.2969 +/- 0.0319
  F1        = 0.3894 +/- 0.0375

Counting:
  MAE  = 1.1285 +/- 0.0716
  Bias = -1.0331 +/- 0.0696

Empty synthetic scenes:
  FP/image = 0.0849 +/- 0.0290

Matched Dice is CONDITIONAL on matched detections and is
not an overall Dice score.

Next scientific stage:
  246 held-out clean FruitBG test backgrounds
  false-positive / confounder evaluation

No model inference is performed by the accompanying clean
implementation audit.
""".strip()
    + "\n",
    encoding="utf-8",
)


# ============================================================
# 23. REBUILD FINAL FILE MANIFEST INCLUDING README/MANIFEST
# ============================================================

final_records = []


for path in sorted(
    p
    for p in PACKAGE_ROOT.rglob(
        "*"
    )
    if p.is_file()
    and p != PACKAGE_MANIFEST
):

    final_records.append(
        {
            "relative_path":
                path.relative_to(
                    PACKAGE_ROOT
                ).as_posix(),

            "size_bytes":
                path.stat().st_size,

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest[
    "preserved_files"
] = final_records


save_json(
    PACKAGE_MANIFEST,
    manifest,
)


# ============================================================
# 24. PACKAGE READ-BACK SHA AUDIT
# ============================================================

readback = load_json(
    PACKAGE_MANIFEST
)


if readback.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Preservation manifest status != PASS."
    )


for record in readback[
    "preserved_files"
]:

    path = (
        PACKAGE_ROOT
        / record[
            "relative_path"
        ]
    )


    require(
        path,
        "Preserved artifact",
    )


    if (
        path.stat().st_size
        != int(
            record[
                "size_bytes"
            ]
        )
    ):

        raise RuntimeError(
            "Preserved artifact size mismatch:\n"
            f"{path}"
        )


    if (
        sha256_file(
            path
        )
        != record[
            "sha256"
        ]
    ):

        raise RuntimeError(
            "Preserved artifact SHA mismatch:\n"
            f"{path}"
        )


print()
print(
    "Preservation package file-by-file SHA audit: PASS"
)


# ============================================================
# 25. CREATE ZIP
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:

    for path in sorted(
        p
        for p in PACKAGE_ROOT.rglob(
            "*"
        )
        if p.is_file()
    ):

        arcname = (
            Path(
                PACKAGE_NAME
            )
            / path.relative_to(
                PACKAGE_ROOT
            )
        )


        zf.write(
            path,
            arcname=str(
                arcname
            ),
        )


with zipfile.ZipFile(
    ZIP_PATH,
    "r",
) as zf:

    bad = zf.testzip()


if bad is not None:

    raise RuntimeError(
        f"Preservation ZIP CRC failure: {bad}"
    )


zip_sha = sha256_file(
    ZIP_PATH
)


ZIP_SHA_RECORD.write_text(
    f"{zip_sha}  {ZIP_PATH.name}\n",
    encoding="utf-8",
)


# ============================================================
# 26. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FORMAL SYNTHETIC TEST PRESERVED"
)

print(
    "============================================================"
)

print()

print(
    "Formal aggregate SHA256:"
)

print(
    EXPECTED_AGG_SHA
)

print()

print(
    "Preservation ZIP:"
)

print(
    ZIP_PATH
)

print()

print(
    "ZIP size:"
)

print(
    f"{ZIP_PATH.stat().st_size / (1024 ** 2):.2f} MiB"
)

print()

print(
    "ZIP SHA256:"
)

print(
    zip_sha
)

print()

print(
    "Preservation manifest:"
)

print(
    PACKAGE_MANIFEST
)

print()

print(
    "Clean implementation audit:"
)

print(
    AUDIT_JSON
)

print()

print(
    "============================================================"
)

print(
    "CLEAN FRUITBG PRE-INFERENCE AUDIT"
)

print(
    "============================================================"
)

print()

print(
    "Held-out clean FruitBG rows:      246"
)

print(
    "Source RGB available:             246/246"
)

print(
    "Frozen masks available:           246/246"
)

print()

print(
    "RGB shape distribution:"
)


for shape, count in sorted(
    shape_counts.items()
):

    print(
        f"  {shape}: {count}"
    )


print()

print(
    "Relevant evaluator/model interfaces were audited."
)

print(
    "NO MODEL INFERENCE WAS PERFORMED."
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET FORMAL SYNTEST PRESERVATION + "
    "CLEAN-EVAL AUDIT: PASS"
)

print(
    "============================================================"
)

print()

print(
    "STOP HERE."
)

print(
    "Next: construct the exact 246-image held-out clean "
    "FruitBG inference cell using the audited native "
    "model/preprocessing interface."
)

DASO-Net — PRESERVE FORMAL SYNTHETIC TEST
+ CLEAN FRUITBG IMPLEMENTATION AUDIT v1
PASS  Formal synthetic-test aggregate
PASS  Formal aggregate SHA record
PASS  Formal per-seed table
PASS  Formal mean/SD table
PASS  Formal Markdown table
PASS  Frozen formal protocol
PASS  Frozen protocol README
PASS  Formal synthetic-test run manifest
PASS  Formal project-overlay audit
PASS  Prior asset-failure provenance
PASS  Frozen FruitBG manifest
PASS  Validated evaluator
PASS  FruitMP synthetic dataset implementation
PASS  Training data implementation
PASS  Validated decoder

Formal untouched synthetic-test aggregate SHA: PASS
All 3 precommitted seeds retained: PASS
Test threshold tuning: NO
Frozen clean-FruitBG evaluation clause: PASS

Held-out clean FruitBG source RGB: 246/246 — PASS
Held-out clean FruitBG frozen masks: 246/246 — PASS

Clean FruitBG RGB shape distribution:
  (1660, 4000): 43
  (1872, 1872): 1
  (1882, 1882): 2
  (1892, 2365): 1
  (1893, 2554): 1
  (1895, 1895): 1
  (1971, 1971):

In [20]:
# ============================================================
# DASO-Net — FORMAL HELD-OUT CLEAN FRUITBG EVALUATION v1
#
# SCIENTIFIC ROLE
# ---------------
# False-positive / confounder evaluation on all 246 frozen
# held-out FruitBG test surfaces.
#
# INPUT CONSTRUCTION
# ------------------
# For every unique held-out background_surface_id:
#
#   - use its FIRST occurrence in the immutable ordered
#     FruitMP-Syn v1.1 test recipe file
#   - obtain sample["clean"] through the exact
#     FruitMPSynTrainAdapter / renderer path
#   - do NOT use the composite image
#   - do NOT add synthetic particles
#
# This gives exactly:
#   246 backgrounds -> 246 deterministic clean 1024 tiles
#
# The tile-selection rule and all clean-input hashes are frozen
# BEFORE model inference.
#
# MODELS
# ------
# All 3 precommitted terminal checkpoints.
#
# Frozen operating thresholds:
#   Seed 1 = 0.30
#   Seed 2 = 0.35
#   Seed 3 = 0.35
#
# Decoder:
#   min score      = 0.05
#   top-k          = 100
#   mask threshold = 0.50
#   NMS IoU        = inherited from validated evaluator default
#
# PRIMARY CLEAN METRICS / SEED
# ----------------------------
#   FP/image at frozen operating threshold
#   fraction images with >=1 FP at frozen threshold
#   decoded max-score distribution
#
# DIAGNOSTIC
# ----------
#   FP/image at decoder floor 0.05
#   fraction images with >=1 candidate at 0.05
#
# NO TRAINING.
# NO THRESHOLD SWEEP.
# NO THRESHOLD TUNING.
# NO MODEL SELECTION.
# NO SEED EXCLUSION.
# NO FROZEN FILE MODIFICATION.
# ============================================================

from __future__ import annotations

import ast
import csv
import gc
import hashlib
import importlib.util
import inspect
import json
import math
import os
import statistics
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch


# ============================================================
# 1. ROOTS
# ============================================================

WORKING = Path(
    "/kaggle/working"
)

CODE_ROOT = (
    WORKING
    / "DASO-Net-Phase5B-KaggleTraining"
)

SRC_ROOT = (
    CODE_ROOT
    / "src"
)

OVERLAY_ROOT = (
    WORKING
    / "DASO-Net-Phase1A-FORMALTEST-OVERLAY-v1"
)

ANALYSIS_ROOT = (
    WORKING
    / "dasonet_final_analysis"
)

PROTOCOL_JSON = (
    ANALYSIS_ROOT
    / "formal_evaluation_protocol_freeze_v1"
    / "FORMAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

FORMAL_SYN_AGGREGATE = (
    ANALYSIS_ROOT
    / "formal_fruitmp_syn_test_three_seed_v1"
    / "FINAL_FORMAL_FRUITMP_SYN_TEST_THREE_SEED_AGGREGATE_v1.json"
)

TEST_RECIPES = (
    OVERLAY_ROOT
    / "data"
    / "fruitmp_syn"
    / "v1_1_recipes"
    / "fruitmp_syn_v1_1_test_recipes.csv"
)

FRUITBG_MANIFEST = (
    OVERLAY_ROOT
    / "data"
    / "fruit_bg"
    / "v1_final"
    / "fruitbg_rgb_v1_manifest.csv"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)


# ============================================================
# 2. FINAL CHECKPOINTS
# ============================================================

SEED1_CHECKPOINT = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

SEED2_CHECKPOINT = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

SEED3_CHECKPOINT = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed03_s252916394"
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)


# ============================================================
# 3. EXPECTED FROZEN IDENTITIES
# ============================================================

EXPECTED_PROTOCOL_SHA = (
    "a98ac6c5af42c7dc84bc7e7c00b7a9ed"
    "661d45af6882bdb0029c49b8d0cb5b62"
)

EXPECTED_SYN_AGG_SHA = (
    "88ab1d74ebc6ff6ee9b957d17392587f"
    "4907315a48b94cd4652ecd023d2fd6df"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)

EXPECTED_SEED1_SHA = (
    "e4c4b79ad4b8c6e18ba2b1e3d6cbe796"
    "c9cc301bd6fcc7c596fc290ad3a7e81d"
)

EXPECTED_SEED2_SHA = (
    "19cdb834a46240916b2c8023e8c3afe573"
    "8d9c7cd5d8f818c62ad1e289df1ad2"
)

EXPECTED_SEED3_SHA = (
    "be7b7941f2f62179e15074aa973bf6da0"
    "eb9018ec9133486512bd1f87153f447"
)

EXPECTED_TEST_RECIPES = 5000
EXPECTED_CLEAN_BACKGROUNDS = 246


# ============================================================
# 4. FROZEN DECODER / OPERATING CONTRACT
# ============================================================

DECODE_MIN_SCORE = 0.05
TOPK = 100
MASK_THRESHOLD = 0.50


SEEDS = {
    1: {
        "rng_seed":
            1782656963,

        "checkpoint":
            SEED1_CHECKPOINT,

        "checkpoint_sha256":
            EXPECTED_SEED1_SHA,

        "operating_threshold":
            0.30,
    },

    2: {
        "rng_seed":
            901395089,

        "checkpoint":
            SEED2_CHECKPOINT,

        "checkpoint_sha256":
            EXPECTED_SEED2_SHA,

        "operating_threshold":
            0.35,
    },

    3: {
        "rng_seed":
            252916394,

        "checkpoint":
            SEED3_CHECKPOINT,

        "checkpoint_sha256":
            EXPECTED_SEED3_SHA,

        "operating_threshold":
            0.35,
    },
}


# ============================================================
# 5. OUTPUT
# ============================================================

OUT_ROOT = (
    WORKING
    / "dasonet_final_evaluation"
    / "clean_fruitbg_test_v1"
)

CLEAN_FREEZE_JSON = (
    OUT_ROOT
    / "CLEAN_FRUITBG_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

CLEAN_FREEZE_SHA = (
    OUT_ROOT
    / "CLEAN_FRUITBG_EVALUATION_PROTOCOL_FREEZE_v1.sha256.txt"
)

FINAL_AGGREGATE_JSON = (
    OUT_ROOT
    / "FINAL_CLEAN_FRUITBG_THREE_SEED_AGGREGATE_v1.json"
)

PER_SEED_CSV = (
    OUT_ROOT
    / "clean_fruitbg_per_seed.csv"
)

PER_FRUIT_CSV = (
    OUT_ROOT
    / "clean_fruitbg_per_fruit.csv"
)

README = (
    OUT_ROOT
    / "README_CLEAN_FRUITBG_EVALUATION.txt"
)


for idx in [
    1,
    2,
    3,
]:

    SEEDS[idx]["result_csv"] = (
        OUT_ROOT
        / f"seed0{idx}_clean_per_image.csv"
    )

    SEEDS[idx]["summary_json"] = (
        OUT_ROOT
        / f"seed0{idx}_clean_summary.json"
    )


# ============================================================
# 6. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(
    data: bytes,
) -> str:

    return hashlib.sha256(
        data
    ).hexdigest()


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(
    path: Path,
    obj,
) -> None:

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def read_csv(
    path: Path,
):

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def tensor_sha256(
    tensor: torch.Tensor,
) -> str:

    array = (
        tensor
        .detach()
        .cpu()
        .contiguous()
        .numpy()
    )

    header = (
        str(
            array.dtype
        )
        + "|"
        + repr(
            tuple(
                array.shape
            )
        )
        + "|"
    ).encode(
        "utf-8"
    )

    return sha256_bytes(
        header
        + array.tobytes(
            order="C"
        )
    )


def percentile(
    values,
    q,
):

    if not values:

        return 0.0

    return float(
        np.quantile(
            np.asarray(
                values,
                dtype=np.float64,
            ),
            q,
        )
    )


def mean_sd(
    values,
):

    if len(
        values
    ) != 3:

        raise RuntimeError(
            "Expected exactly three seeds."
        )

    return {
        "mean":
            statistics.mean(
                values
            ),

        "sample_sd":
            statistics.stdev(
                values
            ),
    }


# ============================================================
# 7. EXTRACT ARGPARSE DEFAULT FROM VALIDATED EVALUATOR
# ============================================================

def extract_argparse_default(
    script_path: Path,
    flag: str,
):

    source = script_path.read_text(
        encoding="utf-8"
    )

    tree = ast.parse(
        source
    )


    for node in ast.walk(
        tree
    ):

        if not isinstance(
            node,
            ast.Call,
        ):

            continue


        func = node.func


        if not (
            isinstance(
                func,
                ast.Attribute,
            )
            and func.attr
            == "add_argument"
        ):

            continue


        string_args = []


        for arg in node.args:

            if (
                isinstance(
                    arg,
                    ast.Constant,
                )
                and isinstance(
                    arg.value,
                    str,
                )
            ):

                string_args.append(
                    arg.value
                )


        if flag not in string_args:

            continue


        for kw in node.keywords:

            if kw.arg == "default":

                try:

                    return ast.literal_eval(
                        kw.value
                    )

                except Exception:

                    return None


        return None


    raise RuntimeError(
        f"Could not locate argparse option: {flag}"
    )


# ============================================================
# 8. MODEL BUILD CALL
# ============================================================

def call_build_model(
    build_model_fn,
    checkpoint,
    device,
):

    signature = inspect.signature(
        build_model_fn
    )

    kwargs = {}


    for name, parameter in (
        signature.parameters.items()
    ):

        lname = name.lower()


        if (
            "checkpoint"
            in lname
            or lname
            in {
                "ckpt",
                "state",
                "bundle",
            }
        ):

            kwargs[
                name
            ] = checkpoint


        elif lname == "device":

            kwargs[
                name
            ] = device


        elif (
            parameter.default
            is not inspect._empty
        ):

            continue


        else:

            raise RuntimeError(
                "Cannot safely satisfy required build_model "
                f"parameter {name!r}.\n"
                f"Signature: {signature}"
            )


    return build_model_fn(
        **kwargs
    )


# ============================================================
# 9. DECODER CALL
# ============================================================

def call_decoder(
    decode_fn,
    model_output,
    h,
    w,
    nms_iou,
):

    signature = inspect.signature(
        decode_fn
    )

    parameters = list(
        signature.parameters.items()
    )


    if not parameters:

        raise RuntimeError(
            "decode_instances has no parameters."
        )


    # First argument is the model output.
    first_name = parameters[
        0
    ][
        0
    ]


    kwargs = {}


    for name, parameter in parameters[
        1:
    ]:

        lname = name.lower()


        if lname in {
            "score_threshold",
            "score_thresh",
            "score_thr",
            "min_score",
            "decode_min_score",
        }:

            kwargs[
                name
            ] = DECODE_MIN_SCORE


        elif lname in {
            "topk",
            "top_k",
            "max_detections",
            "max_instances",
        }:

            kwargs[
                name
            ] = TOPK


        elif lname in {
            "nms_iou",
            "nms_iou_threshold",
            "nms_threshold",
            "iou_threshold",
        }:

            kwargs[
                name
            ] = nms_iou


        elif lname in {
            "mask_threshold",
            "mask_thresh",
            "mask_thr",
        }:

            kwargs[
                name
            ] = MASK_THRESHOLD


        elif lname in {
            "image_size",
            "input_size",
            "output_size",
        }:

            kwargs[
                name
            ] = (
                h,
                w,
            )


        elif lname in {
            "image_hw",
            "input_hw",
            "shape",
        }:

            kwargs[
                name
            ] = (
                h,
                w,
            )


        elif lname in {
            "height",
            "image_height",
            "h",
        }:

            kwargs[
                name
            ] = h


        elif lname in {
            "width",
            "image_width",
            "w",
        }:

            kwargs[
                name
            ] = w


        elif (
            parameter.default
            is not inspect._empty
        ):

            # Preserve decoder's own validated default.
            continue


        else:

            raise RuntimeError(
                "Cannot safely satisfy required decoder "
                f"parameter {name!r}.\n"
                f"Signature: {signature}"
            )


    return decode_fn(
        model_output,
        **kwargs,
    )


# ============================================================
# 10. ROBUST SCORE EXTRACTION
# ============================================================

def to_float_list(
    value,
):

    if value is None:

        return []


    if torch.is_tensor(
        value
    ):

        return [
            float(
                x
            )
            for x in (
                value
                .detach()
                .cpu()
                .reshape(
                    -1
                )
                .tolist()
            )
        ]


    if isinstance(
        value,
        np.ndarray,
    ):

        return [
            float(
                x
            )
            for x in value.reshape(
                -1
            ).tolist()
        ]


    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):

        return [
            float(
                value
            )
        ]


    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        out = []


        for item in value:

            out.extend(
                to_float_list(
                    item
                )
            )


        return out


    return []


def extract_detection_scores(
    decoded,
):
    """
    Extract only detection confidence scores.

    Supported common decoder outputs:
      - list[dict(score=...)]
      - dict(scores=tensor)
      - dict(instances=...)
      - object.score / object.scores
      - outer batch wrapper with one element
    """

    if decoded is None:

        return []


    # --------------------------------------------------------
    # Dictionary
    # --------------------------------------------------------

    if isinstance(
        decoded,
        dict,
    ):

        for key in [
            "scores",
            "score",
            "detection_scores",
        ]:

            if key in decoded:

                scores = to_float_list(
                    decoded[
                        key
                    ]
                )

                if scores:

                    return scores


        for key in [
            "instances",
            "predictions",
            "detections",
        ]:

            if key in decoded:

                return extract_detection_scores(
                    decoded[
                        key
                    ]
                )


        return []


    # --------------------------------------------------------
    # Sequence
    # --------------------------------------------------------

    if isinstance(
        decoded,
        (
            list,
            tuple,
        ),
    ):

        if len(
            decoded
        ) == 0:

            return []


        # Batch-size-one wrapper around an instance collection.
        if (
            len(
                decoded
            )
            == 1
            and isinstance(
                decoded[
                    0
                ],
                (
                    list,
                    tuple,
                    dict,
                ),
            )
        ):

            nested = extract_detection_scores(
                decoded[
                    0
                ]
            )


            if nested:

                return nested


        scores = []


        for item in decoded:

            if isinstance(
                item,
                dict,
            ):

                if "score" in item:

                    scores.extend(
                        to_float_list(
                            item[
                                "score"
                            ]
                        )
                    )

                elif "scores" in item:

                    scores.extend(
                        to_float_list(
                            item[
                                "scores"
                            ]
                        )
                    )

                else:

                    nested = (
                        extract_detection_scores(
                            item
                        )
                    )

                    scores.extend(
                        nested
                    )


            elif hasattr(
                item,
                "score",
            ):

                scores.extend(
                    to_float_list(
                        getattr(
                            item,
                            "score",
                        )
                    )
                )


            elif hasattr(
                item,
                "scores",
            ):

                scores.extend(
                    to_float_list(
                        getattr(
                            item,
                            "scores",
                        )
                    )
                )


        return scores


    # --------------------------------------------------------
    # Object
    # --------------------------------------------------------

    if hasattr(
        decoded,
        "scores",
    ):

        return to_float_list(
            decoded.scores
        )


    if hasattr(
        decoded,
        "score",
    ):

        return to_float_list(
            decoded.score
        )


    return []


# ============================================================
# 11. PREFLIGHT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FORMAL HELD-OUT CLEAN FRUITBG EVALUATION v1"
)

print(
    "============================================================"
)


for path, label in [
    (
        CODE_ROOT,
        "Validated code root",
    ),
    (
        OVERLAY_ROOT,
        "Formal-test complete project overlay",
    ),
    (
        PROTOCOL_JSON,
        "Frozen formal evaluation protocol",
    ),
    (
        FORMAL_SYN_AGGREGATE,
        "Frozen formal synthetic-test aggregate",
    ),
    (
        TEST_RECIPES,
        "Frozen FruitMP-Syn test recipes",
    ),
    (
        FRUITBG_MANIFEST,
        "Frozen FruitBG manifest",
    ),
    (
        EVALUATOR,
        "Validated evaluator",
    ),
    (
        DECODER,
        "Validated decoder",
    ),
    (
        SEED1_CHECKPOINT,
        "Seed-1 terminal checkpoint",
    ),
    (
        SEED2_CHECKPOINT,
        "Seed-2 terminal checkpoint",
    ),
    (
        SEED3_CHECKPOINT,
        "Seed-3 terminal checkpoint",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 12. OUTPUT MUST BE FRESH
# ============================================================

if OUT_ROOT.exists():

    raise RuntimeError(
        "Clean FruitBG formal output already exists:\n"
        f"{OUT_ROOT}\n\n"
        "Do not overwrite a formal clean evaluation."
    )


OUT_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 13. CRYPTOGRAPHIC PROVENANCE
# ============================================================

if (
    sha256_file(
        PROTOCOL_JSON
    )
    != EXPECTED_PROTOCOL_SHA
):

    raise RuntimeError(
        "Formal evaluation protocol SHA mismatch."
    )


if (
    sha256_file(
        FORMAL_SYN_AGGREGATE
    )
    != EXPECTED_SYN_AGG_SHA
):

    raise RuntimeError(
        "Formal synthetic aggregate SHA mismatch."
    )


if (
    sha256_file(
        DECODER
    )
    != EXPECTED_DECODER_SHA
):

    raise RuntimeError(
        "Validated decoder SHA mismatch."
    )


for idx in [
    1,
    2,
    3,
]:

    if (
        sha256_file(
            SEEDS[
                idx
            ][
                "checkpoint"
            ]
        )
        != SEEDS[
            idx
        ][
            "checkpoint_sha256"
        ]
    ):

        raise RuntimeError(
            f"Seed-{idx} checkpoint SHA mismatch."
        )


print()
print(
    "Formal protocol SHA: PASS"
)

print(
    "Formal synthetic aggregate SHA: PASS"
)

print(
    "Validated decoder SHA: PASS"
)

print(
    "All 3 terminal checkpoint SHAs: PASS"
)


# ============================================================
# 14. VERIFY FROZEN CLEAN-EVALUATION CLAUSE
# ============================================================

protocol = load_json(
    PROTOCOL_JSON
)

clean_clause = protocol[
    "held_out_clean_fruitbg"
]


if int(
    clean_clause[
        "num_backgrounds"
    ]
) != 246:

    raise RuntimeError(
        "Frozen clean-background count != 246."
    )


if bool(
    clean_clause[
        "threshold_tuning"
    ]
):

    raise RuntimeError(
        "Frozen clean protocol unexpectedly allows "
        "threshold tuning."
    )


for idx in [
    1,
    2,
    3,
]:

    frozen = float(
        clean_clause[
            "operating_thresholds"
        ][
            f"seed{idx}"
        ]
    )

    if (
        frozen
        != SEEDS[
            idx
        ][
            "operating_threshold"
        ]
    ):

        raise RuntimeError(
            f"Seed-{idx} clean threshold mismatch."
        )


print(
    "Frozen clean-FruitBG protocol clause: PASS"
)


# ============================================================
# 15. EXTRACT VALIDATED EVALUATOR NMS DEFAULT
# ============================================================

NMS_IOU = extract_argparse_default(
    EVALUATOR,
    "--nms-iou",
)


if NMS_IOU is None:

    raise RuntimeError(
        "Could not recover evaluator --nms-iou default."
    )


NMS_IOU = float(
    NMS_IOU
)


print()
print(
    f"Validated evaluator NMS IoU default: {NMS_IOU}"
)


# ============================================================
# 16. IMPORT EXACT PROJECT INTERFACES
# ============================================================

if str(
    SRC_ROOT
) not in sys.path:

    sys.path.insert(
        0,
        str(
            SRC_ROOT
        ),
    )


spec = importlib.util.spec_from_file_location(
    "dasonet_formal_evaluator_module",
    EVALUATOR,
)


if spec is None or spec.loader is None:

    raise RuntimeError(
        "Could not import validated evaluator module."
    )


eval_module = (
    importlib.util.module_from_spec(
        spec
    )
)

spec.loader.exec_module(
    eval_module
)


if not hasattr(
    eval_module,
    "build_model",
):

    raise RuntimeError(
        "Validated evaluator build_model() missing."
    )


if not hasattr(
    eval_module,
    "decode_instances",
):

    raise RuntimeError(
        "Validated evaluator decode_instances missing."
    )


if not hasattr(
    eval_module,
    "FruitMPSynTrainAdapter",
):

    raise RuntimeError(
        "Validated evaluator FruitMPSynTrainAdapter missing."
    )


build_model_fn = (
    eval_module.build_model
)

decode_fn = (
    eval_module.decode_instances
)

AdapterClass = (
    eval_module.FruitMPSynTrainAdapter
)


print()
print(
    "Exact validated interfaces imported: PASS"
)

print(
    f"build_model signature:     "
    f"{inspect.signature(build_model_fn)}"
)

print(
    f"decode_instances signature:"
    f" {inspect.signature(decode_fn)}"
)

print(
    f"adapter signature:         "
    f"{inspect.signature(AdapterClass)}"
)


# ============================================================
# 17. INSTANTIATE EXACT TEST ADAPTER
#
# This is DATA CONSTRUCTION ONLY.
# No model exists yet.
# ============================================================

try:

    dataset = AdapterClass(
        OVERLAY_ROOT,
        split="test",
    )

except Exception as exc:

    raise RuntimeError(
        "Could not instantiate FruitMPSynTrainAdapter using "
        "the same project_root + split=test interface as the "
        "validated evaluator.\n"
        f"Adapter signature: {inspect.signature(AdapterClass)}\n"
        f"Original error: {exc}"
    ) from exc


if len(
    dataset
) != EXPECTED_TEST_RECIPES:

    raise RuntimeError(
        "Test adapter length != 5000.\n"
        f"Actual: {len(dataset)}"
    )


print()
print(
    "FruitMPSynTrainAdapter(split=test): PASS"
)

print(
    "Adapter scenes: 5000"
)


# ============================================================
# 18. SELECT EXACTLY ONE CLEAN TILE PER HELD-OUT BACKGROUND
#
# PREDECLARED RULE:
# first occurrence in immutable ordered test recipe file.
#
# No model/predictions are consulted.
# ============================================================

recipe_rows = read_csv(
    TEST_RECIPES
)


if len(
    recipe_rows
) != 5000:

    raise RuntimeError(
        "Frozen test recipe count != 5000."
    )


required_recipe_fields = {
    "scene_id",
    "fruit",
    "background_surface_id",
}


missing_recipe_fields = (
    required_recipe_fields
    - set(
        recipe_rows[
            0
        ].keys()
    )
)


if missing_recipe_fields:

    raise RuntimeError(
        "Frozen test recipes missing fields:\n"
        + "\n".join(
            sorted(
                missing_recipe_fields
            )
        )
    )


first_index_by_surface = {}


for index, row in enumerate(
    recipe_rows
):

    surface_id = str(
        row[
            "background_surface_id"
        ]
    ).strip()


    if surface_id not in first_index_by_surface:

        first_index_by_surface[
            surface_id
        ] = index


if len(
    first_index_by_surface
) != 246:

    raise RuntimeError(
        "Expected 246 unique held-out background IDs "
        "in frozen test recipes.\n"
        f"Actual: {len(first_index_by_surface)}"
    )


selected = []


for surface_id, index in (
    first_index_by_surface.items()
):

    row = recipe_rows[
        index
    ]


    selected.append(
        {
            "recipe_index":
                index,

            "scene_id":
                str(
                    row[
                        "scene_id"
                    ]
                ),

            "fruit":
                str(
                    row[
                        "fruit"
                    ]
                ),

            "background_surface_id":
                surface_id,
        }
    )


selected.sort(
    key=lambda x: x[
        "recipe_index"
    ]
)


print()
print(
    "Clean tile selection rule:"
)

print(
    "  FIRST frozen test-recipe occurrence per "
    "background_surface_id"
)

print(
    "Selected held-out surfaces: 246 — PASS"
)


# ============================================================
# 19. VERIFY SELECTED IDs ARE EXACTLY FRUITBG TEST SURFACES
# ============================================================

fruit_rows = read_csv(
    FRUITBG_MANIFEST
)


test_fruit_rows = [
    row
    for row in fruit_rows
    if str(
        row[
            "fruitbg_split"
        ]
    ).strip().lower()
    == "test"
]


if len(
    test_fruit_rows
) != 246:

    raise RuntimeError(
        "FruitBG test rows != 246."
    )


# The recipe namespace was previously verified to equal
# phase2d_surface_id / frozen mask stem.
manifest_test_ids = {
    str(
        row[
            "phase2d_surface_id"
        ]
    ).strip()
    for row in test_fruit_rows
}


selected_ids = {
    item[
        "background_surface_id"
    ]
    for item in selected
}


if (
    selected_ids
    != manifest_test_ids
):

    raise RuntimeError(
        "Selected 246 recipe backgrounds do not exactly "
        "match the 246 held-out FruitBG test surfaces."
    )


print(
    "Selected set exactly matches held-out FruitBG test set: PASS"
)


# ============================================================
# 20. PRE-RENDER + HASH ALL 246 CLEAN MODEL INPUTS
#
# IMPORTANT:
# still NO model loading and NO inference.
#
# We hash each selected clean tensor now so every seed can be
# required to consume byte-identical model inputs.
# ============================================================

clean_input_hashes = []

clean_shapes = Counter()

clean_min = float(
    "inf"
)

clean_max = float(
    "-inf"
)


print()
print(
    "============================================================"
)

print(
    "FREEZING 246 CLEAN MODEL INPUTS BEFORE INFERENCE"
)

print(
    "============================================================"
)


for n, item in enumerate(
    selected,
    start=1,
):

    sample = dataset[
        item[
            "recipe_index"
        ]
    ]


    if (
        not isinstance(
            sample,
            dict,
        )
        or "clean"
        not in sample
    ):

        raise RuntimeError(
            "Adapter sample does not contain 'clean'."
        )


    clean = sample[
        "clean"
    ]


    if not torch.is_tensor(
        clean
    ):

        raise RuntimeError(
            "Adapter clean input is not a torch.Tensor.\n"
            f"Type: {type(clean)}"
        )


    if clean.ndim != 3:

        raise RuntimeError(
            "Clean tensor is not CHW.\n"
            f"Shape: {tuple(clean.shape)}"
        )


    if int(
        clean.shape[
            0
        ]
    ) != 3:

        raise RuntimeError(
            "Clean tensor does not have 3 RGB channels.\n"
            f"Shape: {tuple(clean.shape)}"
        )


    clean = (
        clean
        .detach()
        .cpu()
        .contiguous()
    )


    shape = tuple(
        int(
            x
        )
        for x in clean.shape
    )


    clean_shapes[
        shape
    ] += 1


    this_min = float(
        clean.min().item()
    )

    this_max = float(
        clean.max().item()
    )


    clean_min = min(
        clean_min,
        this_min,
    )

    clean_max = max(
        clean_max,
        this_max,
    )


    # Exact model-input range expected from _to_chw_float.
    if (
        this_min
        < -1e-6
        or this_max
        > 1.000001
    ):

        raise RuntimeError(
            "Clean tensor values are outside [0,1].\n"
            f"Scene: {item['scene_id']}\n"
            f"Range: {this_min} .. {this_max}"
        )


    clean_hash = tensor_sha256(
        clean
    )


    item[
        "clean_tensor_sha256"
    ] = clean_hash

    item[
        "clean_shape"
    ] = list(
        shape
    )


    clean_input_hashes.append(
        clean_hash
    )


    if (
        n % 25 == 0
        or n == 246
    ):

        print(
            f"  frozen clean inputs {n}/246"
        )


if len(
    clean_input_hashes
) != 246:

    raise RuntimeError(
        "Did not freeze exactly 246 clean inputs."
    )


clean_collection_sha = sha256_bytes(
    "\n".join(
        clean_input_hashes
    ).encode(
        "utf-8"
    )
)


print()
print(
    "Clean tensor shape distribution:"
)


for shape, count in sorted(
    clean_shapes.items()
):

    print(
        f"  {shape}: {count}"
    )


print(
    f"Clean value range: {clean_min:.6f} .. {clean_max:.6f}"
)

print(
    "All 246 clean model-input hashes frozen: PASS"
)

print(
    "Clean input collection SHA256:"
)

print(
    clean_collection_sha
)


# ============================================================
# 21. REQUIRE NATIVE 1024 INPUT
# ============================================================

if set(
    clean_shapes.keys()
) != {
    (
        3,
        1024,
        1024,
    )
}:

    raise RuntimeError(
        "Selected clean inputs are not uniformly native "
        "3x1024x1024 model tiles.\n"
        f"Observed: {dict(clean_shapes)}"
    )


print(
    "Native clean input size 3x1024x1024: PASS"
)


# ============================================================
# 22. FREEZE CLEAN EVALUATION PROTOCOL BEFORE MODEL LOADING
# ============================================================

clean_freeze = {
    "protocol_name":
        "DASO-Net Held-Out Clean FruitBG Evaluation v1",

    "status":
        "FROZEN_BEFORE_CLEAN_MODEL_INFERENCE",

    "parent_formal_protocol_sha256":
        EXPECTED_PROTOCOL_SHA,

    "formal_synthetic_aggregate_sha256":
        EXPECTED_SYN_AGG_SHA,

    "dataset": {
        "name":
            "FruitBG-RGB v1 final",

        "split":
            "test",

        "held_out_surfaces":
            246,

        "all_surfaces_used":
            True,

        "surface_identifier":
            "phase2d_surface_id / background_surface_id",
    },

    "input_construction": {
        "implementation":
            "FruitMPSynTrainAdapter(split=test)",

        "input_key":
            "clean",

        "selection_rule":
            (
                "For each unique held-out background_surface_id, "
                "use its first occurrence in the immutable ordered "
                "FruitMP-Syn v1.1 test recipe file."
            ),

        "selected_tiles":
            246,

        "tile_shape":
            [
                3,
                1024,
                1024,
            ],

        "clean_only":
            True,

        "synthetic_particle_overlay":
            False,

        "full_roi_tiling":
            False,

        "surface_mask_post_filtering":
            False,

        "selection_uses_model_predictions":
            False,

        "clean_input_collection_sha256":
            clean_collection_sha,

        "selected":
            selected,
    },

    "decoder": {
        "decode_min_score":
            0.05,

        "topk":
            100,

        "mask_threshold":
            0.50,

        "nms_iou":
            NMS_IOU,

        "nms_iou_source":
            (
                "validated evaluate_phase5c.py argparse default"
            ),

        "decoder_sha256":
            EXPECTED_DECODER_SHA,
    },

    "operating_thresholds": {
        "seed1":
            0.30,

        "seed2":
            0.35,

        "seed3":
            0.35,

        "source":
            "frozen synthetic validation only",
    },

    "primary_metrics": [
        "false positive instances per clean tile at frozen operating threshold",
        "fraction of clean tiles with >=1 false positive at frozen operating threshold",
        "decoded maximum-score distribution",
    ],

    "diagnostic_metrics": [
        "false positive instances per clean tile at decoder floor 0.05",
        "fraction of clean tiles with >=1 decoded candidate at 0.05",
    ],

    "aggregation": {
        "all_three_precommitted_seeds":
            True,

        "mean":
            True,

        "sample_sd":
            True,

        "seed_selection":
            False,
    },

    "prohibitions": {
        "threshold_sweep":
            True,

        "threshold_tuning":
            True,

        "checkpoint_selection":
            True,

        "seed_exclusion":
            True,

        "model_retraining":
            True,
    },
}


save_json(
    CLEAN_FREEZE_JSON,
    clean_freeze,
)


clean_freeze_sha = sha256_file(
    CLEAN_FREEZE_JSON
)


CLEAN_FREEZE_SHA.write_text(
    (
        f"{clean_freeze_sha}  "
        f"{CLEAN_FREEZE_JSON.name}\n"
    ),
    encoding="utf-8",
)


print()
print(
    "============================================================"
)

print(
    "CLEAN FRUITBG EVALUATION PROTOCOL FROZEN"
)

print(
    "============================================================"
)

print(
    f"Backgrounds:          246"
)

print(
    f"Clean tiles:          246"
)

print(
    f"Input size:           3x1024x1024"
)

print(
    f"Decode min score:     0.05"
)

print(
    f"Top-k:                100"
)

print(
    f"Mask threshold:       0.50"
)

print(
    f"NMS IoU:              {NMS_IOU}"
)

print(
    "Seed thresholds:      0.30 / 0.35 / 0.35"
)

print(
    "Threshold sweep:      NO"
)

print(
    "Model selection:      NO"
)

print()

print(
    "Clean protocol SHA256:"
)

print(
    clean_freeze_sha
)

print()

print(
    "NO MODEL INFERENCE HAS OCCURRED YET."
)


# ============================================================
# 23. CUDA PREFLIGHT
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA unavailable."
    )


device = torch.device(
    "cuda:0"
)


print()
print(
    f"CUDA device: {torch.cuda.get_device_name(0)}"
)


# ============================================================
# 24. RUN ONE FINAL SEED
# ============================================================

def run_clean_seed(
    idx: int,
):

    record = SEEDS[
        idx
    ]


    print()
    print(
        "=" * 60
    )

    print(
        f"CLEAN FRUITBG — SEED {idx}"
    )

    print(
        "=" * 60
    )

    print(
        f"RNG seed:            {record['rng_seed']}"
    )

    print(
        f"Frozen threshold:    "
        f"{record['operating_threshold']}"
    )


    # --------------------------------------------------------
    # Load exact terminal checkpoint.
    # --------------------------------------------------------

    try:

        checkpoint = torch.load(
            record[
                "checkpoint"
            ],
            map_location="cpu",
            weights_only=False,
        )

    except TypeError:

        checkpoint = torch.load(
            record[
                "checkpoint"
            ],
            map_location="cpu",
        )


    model = call_build_model(
        build_model_fn,
        checkpoint,
        device,
    )


    model.eval()


    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats(
        device
    )


    rows_out = []

    start = time.time()


    with torch.inference_mode():

        for n, item in enumerate(
            selected,
            start=1,
        ):

            sample = dataset[
                item[
                    "recipe_index"
                ]
            ]


            clean = sample[
                "clean"
            ]


            if not torch.is_tensor(
                clean
            ):

                raise RuntimeError(
                    f"Seed-{idx}: clean input ceased to "
                    "be a tensor."
                )


            clean = (
                clean
                .detach()
                .cpu()
                .contiguous()
            )


            # ------------------------------------------------
            # Every seed must consume byte-identical input
            # frozen BEFORE inference.
            # ------------------------------------------------

            observed_clean_sha = tensor_sha256(
                clean
            )


            if (
                observed_clean_sha
                != item[
                    "clean_tensor_sha256"
                ]
            ):

                raise RuntimeError(
                    "Clean renderer input changed between "
                    "pre-inference freeze and model evaluation.\n"
                    f"Seed:  {idx}\n"
                    f"Scene: {item['scene_id']}"
                )


            x = (
                clean
                .unsqueeze(
                    0
                )
                .to(
                    device,
                    non_blocking=False,
                )
            )


            h = int(
                x.shape[
                    -2
                ]
            )

            w = int(
                x.shape[
                    -1
                ]
            )


            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=True,
            ):

                output = model(
                    x
                )


            decoded = call_decoder(
                decode_fn=decode_fn,
                model_output=output,
                h=h,
                w=w,
                nms_iou=NMS_IOU,
            )


            scores = extract_detection_scores(
                decoded
            )


            scores = [
                float(
                    score
                )
                for score in scores
            ]


            if any(
                not math.isfinite(
                    score
                )
                for score in scores
            ):

                raise RuntimeError(
                    f"Seed-{idx}: non-finite decoded score."
                )


            # Decoder is frozen with candidate floor = 0.05.
            if any(
                score
                < (
                    DECODE_MIN_SCORE
                    - 1e-6
                )
                for score in scores
            ):

                raise RuntimeError(
                    "Decoder emitted score below frozen "
                    "minimum-score floor."
                )


            if len(
                scores
            ) > TOPK:

                raise RuntimeError(
                    "Decoder emitted more than frozen top-k."
                )


            fp_005 = len(
                scores
            )


            op_threshold = float(
                record[
                    "operating_threshold"
                ]
            )


            fp_op = sum(
                1
                for score in scores
                if score
                >= op_threshold
            )


            max_score = (
                max(
                    scores
                )
                if scores
                else 0.0
            )


            rows_out.append(
                {
                    "background_surface_id":
                        item[
                            "background_surface_id"
                        ],

                    "selected_scene_id":
                        item[
                            "scene_id"
                        ],

                    "selected_recipe_index":
                        item[
                            "recipe_index"
                        ],

                    "fruit":
                        item[
                            "fruit"
                        ],

                    "clean_tensor_sha256":
                        item[
                            "clean_tensor_sha256"
                        ],

                    "decoded_count_at_0_05":
                        fp_005,

                    "false_positive_count_operating":
                        fp_op,

                    "max_decoded_score":
                        max_score,
                }
            )


            if (
                n % 25 == 0
                or n == 246
            ):

                elapsed = (
                    time.time()
                    - start
                ) / 60.0


                print(
                    f"clean eval {n}/246 "
                    f"elapsed={elapsed:.1f}m"
                )


    runtime = (
        time.time()
        - start
    )


    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    op_counts = [
        int(
            row[
                "false_positive_count_operating"
            ]
        )
        for row in rows_out
    ]


    low_counts = [
        int(
            row[
                "decoded_count_at_0_05"
            ]
        )
        for row in rows_out
    ]


    max_scores = [
        float(
            row[
                "max_decoded_score"
            ]
        )
        for row in rows_out
    ]


    fp_per_image_op = statistics.mean(
        op_counts
    )

    frac_ge1_op = statistics.mean(
        1.0
        if value >= 1
        else 0.0
        for value in op_counts
    )


    fp_per_image_005 = statistics.mean(
        low_counts
    )

    frac_ge1_005 = statistics.mean(
        1.0
        if value >= 1
        else 0.0
        for value in low_counts
    )


    max_score_summary = {
        "mean":
            statistics.mean(
                max_scores
            ),

        "median":
            statistics.median(
                max_scores
            ),

        "p90":
            percentile(
                max_scores,
                0.90,
            ),

        "p95":
            percentile(
                max_scores,
                0.95,
            ),

        "p99":
            percentile(
                max_scores,
                0.99,
            ),

        "max":
            max(
                max_scores
            ),
    }


    # --------------------------------------------------------
    # Per-fruit descriptive breakdown.
    # --------------------------------------------------------

    per_fruit = {}


    grouped = defaultdict(
        list
    )


    for row in rows_out:

        grouped[
            row[
                "fruit"
            ]
        ].append(
            row
        )


    for fruit, fruit_rows in sorted(
        grouped.items()
    ):

        fruit_op = [
            int(
                row[
                    "false_positive_count_operating"
                ]
            )
            for row in fruit_rows
        ]


        fruit_max = [
            float(
                row[
                    "max_decoded_score"
                ]
            )
            for row in fruit_rows
        ]


        per_fruit[
            fruit
        ] = {
            "n":
                len(
                    fruit_rows
                ),

            "fp_per_image_operating":
                statistics.mean(
                    fruit_op
                ),

            "fraction_images_ge1_fp_operating":
                statistics.mean(
                    1.0
                    if value >= 1
                    else 0.0
                    for value in fruit_op
                ),

            "max_score_mean":
                statistics.mean(
                    fruit_max
                ),

            "max_score_median":
                statistics.median(
                    fruit_max
                ),
        }


    summary = {
        "status":
            "PASS",

        "seed_index":
            idx,

        "rng_seed":
            int(
                record[
                    "rng_seed"
                ]
            ),

        "checkpoint_sha256":
            record[
                "checkpoint_sha256"
            ],

        "clean_protocol_sha256":
            clean_freeze_sha,

        "num_clean_backgrounds":
            246,

        "operating_threshold":
            float(
                record[
                    "operating_threshold"
                ]
            ),

        "primary": {
            "false_positive_instances_per_image":
                fp_per_image_op,

            "fraction_images_with_ge1_false_positive":
                frac_ge1_op,

            "max_decoded_score_distribution":
                max_score_summary,
        },

        "diagnostic_at_0_05": {
            "false_positive_instances_per_image":
                fp_per_image_005,

            "fraction_images_with_ge1_candidate":
                frac_ge1_005,
        },

        "per_fruit":
            per_fruit,

        "runtime_minutes":
            runtime
            / 60.0,

        "peak_reserved_vram_gb":
            torch.cuda.max_memory_reserved(
                device
            )
            / (
                1024 ** 3
            ),
    }


    # --------------------------------------------------------
    # Write per-image result.
    # --------------------------------------------------------

    fieldnames = [
        "background_surface_id",
        "selected_scene_id",
        "selected_recipe_index",
        "fruit",
        "clean_tensor_sha256",
        "decoded_count_at_0_05",
        "false_positive_count_operating",
        "max_decoded_score",
    ]


    with record[
        "result_csv"
    ].open(
        "w",
        encoding="utf-8",
        newline="",
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        writer.writerows(
            rows_out
        )


    save_json(
        record[
            "summary_json"
        ],
        summary,
    )


    print()
    print(
        f"Seed-{idx} clean evaluation:"
    )

    print(
        f"  FP/image @ frozen op: "
        f"{fp_per_image_op:.6f}"
    )

    print(
        f"  Images >=1 FP:        "
        f"{frac_ge1_op:.6f}"
    )

    print(
        f"  FP/image @0.05:       "
        f"{fp_per_image_005:.6f}"
    )

    print(
        f"  Max score median:     "
        f"{max_score_summary['median']:.6f}"
    )

    print(
        f"  Max score p95:        "
        f"{max_score_summary['p95']:.6f}"
    )

    print(
        f"  Runtime:              "
        f"{runtime / 60.0:.2f} min"
    )


    # --------------------------------------------------------
    # Free GPU before next seed.
    # --------------------------------------------------------

    del model

    del checkpoint

    gc.collect()

    torch.cuda.empty_cache()


    return summary


# ============================================================
# 25. RUN ALL THREE PRECOMMITTED SEEDS
# ============================================================

all_start = time.time()


summaries = {}


for idx in [
    1,
    2,
    3,
]:

    summaries[
        idx
    ] = run_clean_seed(
        idx
    )


# ============================================================
# 26. VERIFY ALL THREE USED SAME 246 HASHED CLEAN INPUTS
# ============================================================

for idx in [
    1,
    2,
    3,
]:

    rows = read_csv(
        SEEDS[
            idx
        ][
            "result_csv"
        ]
    )


    if len(
        rows
    ) != 246:

        raise RuntimeError(
            f"Seed-{idx} clean result row count != 246."
        )


    observed_hashes = [
        row[
            "clean_tensor_sha256"
        ]
        for row in rows
    ]


    if (
        observed_hashes
        != clean_input_hashes
    ):

        raise RuntimeError(
            f"Seed-{idx} clean input hashes do not "
            "exactly match frozen pre-inference inputs."
        )


print()
print(
    "All 3 seeds used identical pre-frozen 246 clean inputs: PASS"
)


# ============================================================
# 27. THREE-SEED AGGREGATE
# ============================================================

primary_fp = [
    summaries[
        idx
    ][
        "primary"
    ][
        "false_positive_instances_per_image"
    ]
    for idx in [
        1,
        2,
        3,
    ]
]


primary_frac = [
    summaries[
        idx
    ][
        "primary"
    ][
        "fraction_images_with_ge1_false_positive"
    ]
    for idx in [
        1,
        2,
        3,
    ]
]


diag_fp = [
    summaries[
        idx
    ][
        "diagnostic_at_0_05"
    ][
        "false_positive_instances_per_image"
    ]
    for idx in [
        1,
        2,
        3,
    ]
]


diag_frac = [
    summaries[
        idx
    ][
        "diagnostic_at_0_05"
    ][
        "fraction_images_with_ge1_candidate"
    ]
    for idx in [
        1,
        2,
        3,
    ]
]


max_score_mean = [
    summaries[
        idx
    ][
        "primary"
    ][
        "max_decoded_score_distribution"
    ][
        "mean"
    ]
    for idx in [
        1,
        2,
        3,
    ]
]


max_score_median = [
    summaries[
        idx
    ][
        "primary"
    ][
        "max_decoded_score_distribution"
    ][
        "median"
    ]
    for idx in [
        1,
        2,
        3,
    ]
]


max_score_p95 = [
    summaries[
        idx
    ][
        "primary"
    ][
        "max_decoded_score_distribution"
    ][
        "p95"
    ]
    for idx in [
        1,
        2,
        3,
    ]
]


aggregate = {
    "aggregate_name":
        "DASO-Net Held-Out Clean FruitBG Three-Seed Aggregate v1",

    "status":
        "PASS",

    "clean_protocol_sha256":
        clean_freeze_sha,

    "parent_formal_protocol_sha256":
        EXPECTED_PROTOCOL_SHA,

    "formal_synthetic_aggregate_sha256":
        EXPECTED_SYN_AGG_SHA,

    "dataset": {
        "name":
            "FruitBG-RGB v1 final",

        "split":
            "test",

        "held_out_backgrounds":
            246,

        "evaluation_units":
            (
                "one deterministic clean native 1024 tile "
                "per held-out background surface"
            ),

        "clean_input_collection_sha256":
            clean_collection_sha,
    },

    "seed_policy": {
        "all_three_precommitted_seeds":
            True,

        "seed_selection":
            False,

        "seed_exclusion":
            False,

        "threshold_tuning":
            False,
    },

    "per_seed": {
        f"seed{idx}":
            summaries[
                idx
            ]
        for idx in [
            1,
            2,
            3,
        ]
    },

    "aggregate": {
        "false_positive_instances_per_image_operating":
            mean_sd(
                primary_fp
            ),

        "fraction_images_with_ge1_false_positive_operating":
            mean_sd(
                primary_frac
            ),

        "false_positive_instances_per_image_at_0_05":
            mean_sd(
                diag_fp
            ),

        "fraction_images_with_ge1_candidate_at_0_05":
            mean_sd(
                diag_frac
            ),

        "max_decoded_score_mean":
            mean_sd(
                max_score_mean
            ),

        "max_decoded_score_median":
            mean_sd(
                max_score_median
            ),

        "max_decoded_score_p95":
            mean_sd(
                max_score_p95
            ),
    },

    "interpretation": {
        "all_predictions_are_false_positives":
            True,

        "chemical_identification_claim":
            False,

        "full_raw_roi_evaluation":
            False,

        "surface_mask_post_filtering":
            False,

        "note":
            (
                "This measures false-positive behavior on one "
                "deterministically frozen native 1024 clean tile "
                "from each of the 246 held-out FruitBG test "
                "surfaces, using the exact synthetic evaluator "
                "input construction pathway."
            ),
    },
}


save_json(
    FINAL_AGGREGATE_JSON,
    aggregate,
)


# ============================================================
# 28. PER-SEED TABLE
# ============================================================

with PER_SEED_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    fieldnames = [
        "seed_index",
        "rng_seed",
        "operating_threshold",
        "fp_per_image_operating",
        "fraction_images_ge1_fp_operating",
        "fp_per_image_0_05",
        "fraction_images_ge1_candidate_0_05",
        "max_score_mean",
        "max_score_median",
        "max_score_p95",
        "runtime_minutes",
    ]


    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames,
    )

    writer.writeheader()


    for idx in [
        1,
        2,
        3,
    ]:

        summary = summaries[
            idx
        ]


        writer.writerow(
            {
                "seed_index":
                    idx,

                "rng_seed":
                    summary[
                        "rng_seed"
                    ],

                "operating_threshold":
                    summary[
                        "operating_threshold"
                    ],

                "fp_per_image_operating":
                    summary[
                        "primary"
                    ][
                        "false_positive_instances_per_image"
                    ],

                "fraction_images_ge1_fp_operating":
                    summary[
                        "primary"
                    ][
                        "fraction_images_with_ge1_false_positive"
                    ],

                "fp_per_image_0_05":
                    summary[
                        "diagnostic_at_0_05"
                    ][
                        "false_positive_instances_per_image"
                    ],

                "fraction_images_ge1_candidate_0_05":
                    summary[
                        "diagnostic_at_0_05"
                    ][
                        "fraction_images_with_ge1_candidate"
                    ],

                "max_score_mean":
                    summary[
                        "primary"
                    ][
                        "max_decoded_score_distribution"
                    ][
                        "mean"
                    ],

                "max_score_median":
                    summary[
                        "primary"
                    ][
                        "max_decoded_score_distribution"
                    ][
                        "median"
                    ],

                "max_score_p95":
                    summary[
                        "primary"
                    ][
                        "max_decoded_score_distribution"
                    ][
                        "p95"
                    ],

                "runtime_minutes":
                    summary[
                        "runtime_minutes"
                    ],
            }
        )


# ============================================================
# 29. PER-FRUIT TABLE
# ============================================================

with PER_FRUIT_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    fieldnames = [
        "seed_index",
        "fruit",
        "n",
        "fp_per_image_operating",
        "fraction_images_ge1_fp_operating",
        "max_score_mean",
        "max_score_median",
    ]


    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames,
    )

    writer.writeheader()


    for idx in [
        1,
        2,
        3,
    ]:

        for fruit, stats in sorted(
            summaries[
                idx
            ][
                "per_fruit"
            ].items()
        ):

            writer.writerow(
                {
                    "seed_index":
                        idx,

                    "fruit":
                        fruit,

                    "n":
                        stats[
                            "n"
                        ],

                    "fp_per_image_operating":
                        stats[
                            "fp_per_image_operating"
                        ],

                    "fraction_images_ge1_fp_operating":
                        stats[
                            "fraction_images_ge1_fp_operating"
                        ],

                    "max_score_mean":
                        stats[
                            "max_score_mean"
                        ],

                    "max_score_median":
                        stats[
                            "max_score_median"
                        ],
                }
            )


# ============================================================
# 30. README
# ============================================================

README.write_text(
    f"""
DASO-Net Formal Held-Out Clean FruitBG Evaluation v1
=====================================================

Held-out dataset:
  FruitBG-RGB v1 final
  test surfaces = 246

Evaluation unit:
  one deterministic clean native 1024 tile per held-out surface

Selection rule:
  first occurrence of each background_surface_id in the frozen
  ordered FruitMP-Syn v1.1 test recipe file

Input implementation:
  FruitMPSynTrainAdapter(split=test)
  sample["clean"]

No synthetic contamination is supplied to the model.

Clean input collection SHA256:
  {clean_collection_sha}

Clean evaluation protocol SHA256:
  {clean_freeze_sha}

Frozen operating thresholds:
  Seed 1 = 0.30
  Seed 2 = 0.35
  Seed 3 = 0.35

Decoder:
  minimum score = 0.05
  top-k         = 100
  mask threshold= 0.50
  NMS IoU       = {NMS_IOU}

Primary:
  FP/image at frozen operating threshold
  fraction images with >=1 FP
  decoded maximum-score distribution

Diagnostic:
  FP/image at 0.05 decoder floor
  fraction images with >=1 candidate at 0.05

No threshold tuning.
No seed selection.
No retraining.
""".strip()
    + "\n",
    encoding="utf-8",
)


# ============================================================
# 31. FINAL HASHES
# ============================================================

final_aggregate_sha = sha256_file(
    FINAL_AGGREGATE_JSON
)


total_runtime = (
    time.time()
    - all_start
)


# ============================================================
# 32. FINAL REPORT
# ============================================================

def pm(
    key,
):

    record = aggregate[
        "aggregate"
    ][
        key
    ]

    return (
        f"{record['mean']:.4f} "
        f"± {record['sample_sd']:.4f}"
    )


print()
print(
    "============================================================"
)

print(
    "FINAL HELD-OUT CLEAN FRUITBG THREE-SEED RESULT"
)

print(
    "============================================================"
)

print()

print(
    "Held-out backgrounds:       246"
)

print(
    "Clean native tiles/seed:    246"
)

print(
    "Final seeds evaluated:      3 / 3"
)

print()

print(
    "Frozen operating thresholds:"
)

print(
    "  Seed 1: 0.30"
)

print(
    "  Seed 2: 0.35"
)

print(
    "  Seed 3: 0.35"
)

print()

print(
    "PRIMARY CLEAN FALSE-POSITIVE METRICS"
)

print()

print(
    "FP/image @ frozen operating point:"
)

print(
    "  "
    + pm(
        "false_positive_instances_per_image_operating"
    )
)

print()

print(
    "Fraction images with >=1 FP:"
)

print(
    "  "
    + pm(
        "fraction_images_with_ge1_false_positive_operating"
    )
)

print()

print(
    "Max decoded score mean:"
)

print(
    "  "
    + pm(
        "max_decoded_score_mean"
    )
)

print()

print(
    "Max decoded score median:"
)

print(
    "  "
    + pm(
        "max_decoded_score_median"
    )
)

print()

print(
    "Max decoded score p95:"
)

print(
    "  "
    + pm(
        "max_decoded_score_p95"
    )
)

print()

print(
    "DIAGNOSTIC @ 0.05"
)

print()

print(
    "FP/image:"
)

print(
    "  "
    + pm(
        "false_positive_instances_per_image_at_0_05"
    )
)

print()

print(
    "Fraction images with >=1 candidate:"
)

print(
    "  "
    + pm(
        "fraction_images_with_ge1_candidate_at_0_05"
    )
)

print()

print(
    "Clean protocol SHA256:"
)

print(
    clean_freeze_sha
)

print()

print(
    "Clean aggregate SHA256:"
)

print(
    final_aggregate_sha
)

print()

print(
    "Aggregate JSON:"
)

print(
    FINAL_AGGREGATE_JSON
)

print()

print(
    "Per-seed CSV:"
)

print(
    PER_SEED_CSV
)

print()

print(
    "Per-fruit CSV:"
)

print(
    PER_FRUIT_CSV
)

print()

print(
    f"Total clean inference runtime: "
    f"{total_runtime / 60.0:.2f} min"
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET FORMAL HELD-OUT CLEAN FRUITBG EVALUATION: PASS"
)

print(
    "============================================================"
)

print()

print(
    "NO TRAINING WAS PERFORMED."
)

print(
    "NO THRESHOLD SWEEP WAS PERFORMED."
)

print(
    "NO TEST-SET THRESHOLD TUNING WAS PERFORMED."
)

print(
    "NO BEST-SEED SELECTION WAS PERFORMED."
)

print(
    "ALL THREE PRECOMMITTED FINAL SEEDS WERE INCLUDED."
)

print()

print(
    "IMPORTANT:"
)

print(
    "Every prediction on these clean tiles is a false positive."
)

print(
    "This is one deterministic native 1024 clean tile per "
    "held-out FruitBG test surface, not exhaustive full-ROI "
    "sliding-window evaluation."
)

print()

print(
    "STOP HERE."
)

print(
    "Next: preserve the clean-background evidence, then move "
    "to the untouched RSC external real-particle evaluation."
)

DASO-Net — FORMAL HELD-OUT CLEAN FRUITBG EVALUATION v1
PASS  Validated code root
PASS  Formal-test complete project overlay
PASS  Frozen formal evaluation protocol
PASS  Frozen formal synthetic-test aggregate
PASS  Frozen FruitMP-Syn test recipes
PASS  Frozen FruitBG manifest
PASS  Validated evaluator
PASS  Validated decoder
PASS  Seed-1 terminal checkpoint
PASS  Seed-2 terminal checkpoint
PASS  Seed-3 terminal checkpoint

Formal protocol SHA: PASS
Formal synthetic aggregate SHA: PASS
Validated decoder SHA: PASS
All 3 terminal checkpoint SHAs: PASS
Frozen clean-FruitBG protocol clause: PASS

Validated evaluator NMS IoU default: 0.5

Exact validated interfaces imported: PASS
build_model signature:     (checkpoint, device)
decode_instances signature: (outputs: 'dict', input_size: 'tuple[int, int]', stride: 'int' = 4, score_threshold: 'float' = 0.25, topk: 'int' = 100, nms_iou: 'float' = 0.5, mask_threshold: 'float' = 0.5, local_peak_kernel: 'int' = 3) -> 'List[dict[str, torch.Tensor]]'
a

In [21]:
# ============================================================
# DASO-Net — PRESERVE CLEAN FRUITBG EVIDENCE
# + RSC EXTERNAL PRE-INFERENCE ASSET AUDIT v1
#
# PURPOSE
# -------
# 1. Cryptographically preserve the completed formal held-out
#    clean FruitBG evaluation.
#
# 2. Audit the RSC 2025 external dataset implementation and
#    assets BEFORE any external inference.
#
# Scientific role of RSC:
#   - untouched real-domain external test
#   - OFFICIAL authors' test split: 250 images = PRIMARY
#   - exact-dedup sensitivity subset: 247 images = SENSITIVITY
#   - detection / box evaluation only
#   - NEVER used for training, tuning, calibration, threshold
#     selection, checkpoint selection, or CRPA
#
# NO TRAINING.
# NO MODEL INFERENCE.
# NO EXTERNAL TUNING.
# NO THRESHOLD CHANGES.
# NO FROZEN DATA MODIFICATION.
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import shutil
import zipfile
from collections import Counter
from pathlib import Path

import cv2


# ============================================================
# 1. ROOTS
# ============================================================

WORKING = Path(
    "/kaggle/working"
)

OVERLAY_ROOT = (
    WORKING
    / "DASO-Net-Phase1A-FORMALTEST-OVERLAY-v1"
)

CLEAN_ROOT = (
    WORKING
    / "dasonet_final_evaluation"
    / "clean_fruitbg_test_v1"
)

FORMAL_ANALYSIS_ROOT = (
    WORKING
    / "dasonet_final_analysis"
)

FORMAL_SYN_AGGREGATE = (
    FORMAL_ANALYSIS_ROOT
    / "formal_fruitmp_syn_test_three_seed_v1"
    / "FINAL_FORMAL_FRUITMP_SYN_TEST_THREE_SEED_AGGREGATE_v1.json"
)

PARENT_PROTOCOL = (
    FORMAL_ANALYSIS_ROOT
    / "formal_evaluation_protocol_freeze_v1"
    / "FORMAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)


# ============================================================
# 2. COMPLETED CLEAN EVIDENCE
# ============================================================

CLEAN_PROTOCOL = (
    CLEAN_ROOT
    / "CLEAN_FRUITBG_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

CLEAN_PROTOCOL_SHA_RECORD = (
    CLEAN_ROOT
    / "CLEAN_FRUITBG_EVALUATION_PROTOCOL_FREEZE_v1.sha256.txt"
)

CLEAN_AGGREGATE = (
    CLEAN_ROOT
    / "FINAL_CLEAN_FRUITBG_THREE_SEED_AGGREGATE_v1.json"
)

CLEAN_PER_SEED = (
    CLEAN_ROOT
    / "clean_fruitbg_per_seed.csv"
)

CLEAN_PER_FRUIT = (
    CLEAN_ROOT
    / "clean_fruitbg_per_fruit.csv"
)

CLEAN_README = (
    CLEAN_ROOT
    / "README_CLEAN_FRUITBG_EVALUATION.txt"
)


# ============================================================
# 3. EXPECTED COMPLETED HASHES
# ============================================================

EXPECTED_PARENT_PROTOCOL_SHA = (
    "a98ac6c5af42c7dc84bc7e7c00b7a9ed"
    "661d45af6882bdb0029c49b8d0cb5b62"
)

EXPECTED_SYN_AGGREGATE_SHA = (
    "88ab1d74ebc6ff6ee9b957d17392587f"
    "4907315a48b94cd4652ecd023d2fd6df"
)

EXPECTED_CLEAN_PROTOCOL_SHA = (
    "b69efbc0d0ee7c951cfa975ee4ffd362"
    "4fe3666371451dd1f0af3f8e1ba4fdc1"
)

EXPECTED_CLEAN_AGGREGATE_SHA = (
    "0cda57a966ad76cc10558f8165173bc9"
    "28a7a7de4a7c242b6b2410765cb6741e"
)


# ============================================================
# 4. RSC ROOTS
# ============================================================

PUBLIC_REAL_ROOT = (
    OVERLAY_ROOT
    / "data"
    / "public_real_mp"
)

KNOWN_RSC_EXACT_DEDUP = (
    PUBLIC_REAL_ROOT
    / "v1_processed"
    / "rsc_test_exact_dedup_v1.csv"
)


# ============================================================
# 5. PRESERVATION PACKAGE
# ============================================================

PACKAGE_NAME = (
    "DASO-Net-CLEAN-FRUITBG-COMPLETE-20260919"
)

PACKAGE_ROOT = (
    WORKING
    / PACKAGE_NAME
)

PACKAGE_MANIFEST = (
    PACKAGE_ROOT
    / "CLEAN_FRUITBG_PRESERVATION_MANIFEST.json"
)

RSC_AUDIT_JSON = (
    PACKAGE_ROOT
    / "RSC_EXTERNAL_PREINFERENCE_ASSET_AUDIT_v1.json"
)

PACKAGE_README = (
    PACKAGE_ROOT
    / "README.txt"
)

ZIP_PATH = (
    WORKING
    / f"{PACKAGE_NAME}.zip"
)

ZIP_SHA_RECORD = (
    WORKING
    / f"{PACKAGE_NAME}.sha256.txt"
)


# ============================================================
# 6. HELPERS
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
}

LABEL_EXTENSIONS = {
    ".txt",
    ".csv",
    ".json",
    ".xml",
}


def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(
    path: Path,
    obj,
) -> None:

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def read_csv(
    path: Path,
):

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def normalize_path_text(
    value,
) -> str:

    return str(
        value
    ).strip().replace(
        "\\",
        "/",
    )


def is_image_like(
    value,
) -> bool:

    text = normalize_path_text(
        value
    )


    if not text:

        return False


    return (
        Path(
            text
        ).suffix.lower()
        in IMAGE_EXTENSIONS
    )


def is_label_like(
    value,
) -> bool:

    text = normalize_path_text(
        value
    )


    if not text:

        return False


    return (
        Path(
            text
        ).suffix.lower()
        in LABEL_EXTENSIONS
    )


def candidate_paths(
    raw_value,
):

    text = normalize_path_text(
        raw_value
    )


    candidates = []


    def add(
        path: Path,
    ):

        key = str(
            path
        )


        if key not in {
            str(
                p
            )
            for p in candidates
        }:

            candidates.append(
                path
            )


    # --------------------------------------------------------
    # Absolute POSIX path
    # --------------------------------------------------------

    literal = Path(
        text
    )


    if literal.is_absolute():

        add(
            literal
        )


    # --------------------------------------------------------
    # Project-relative
    # --------------------------------------------------------

    add(
        OVERLAY_ROOT
        / text.lstrip(
            "/"
        )
    )


    # --------------------------------------------------------
    # public_real_mp-relative
    # --------------------------------------------------------

    add(
        PUBLIC_REAL_ROOT
        / text.lstrip(
            "/"
        )
    )


    # --------------------------------------------------------
    # Historical project suffix
    # --------------------------------------------------------

    marker = (
        "DASO-Net-Phase1A/"
    )


    if marker in text:

        suffix = text.split(
            marker,
            1,
        )[1]


        add(
            OVERLAY_ROOT
            / suffix
        )


    # --------------------------------------------------------
    # Suffix beginning data/
    # --------------------------------------------------------

    data_marker = (
        "/data/"
    )


    if data_marker in text:

        suffix = (
            "data/"
            + text.split(
                data_marker,
                1,
            )[1]
        )


        add(
            OVERLAY_ROOT
            / suffix
        )


    return candidates


def resolve_path(
    raw_value,
):

    candidates = candidate_paths(
        raw_value
    )


    for path in candidates:

        if path.exists():

            return {
                "exists":
                    True,

                "resolved":
                    str(
                        path
                    ),

                "candidates":
                    [
                        str(
                            p
                        )
                        for p in candidates
                    ],
            }


    return {
        "exists":
            False,

        "resolved":
            None,

        "candidates":
            [
                str(
                    p
                )
                for p in candidates
            ],
    }


def summarize_csv(
    path: Path,
):

    rows = read_csv(
        path
    )


    fields = (
        list(
            rows[
                0
            ].keys()
        )
        if rows
        else []
    )


    image_columns = []

    label_columns = []


    for field in fields:

        image_like = sum(
            1
            for row in rows
            if is_image_like(
                row.get(
                    field
                )
            )
        )


        label_like = sum(
            1
            for row in rows
            if is_label_like(
                row.get(
                    field
                )
            )
        )


        if image_like > 0:

            image_columns.append(
                {
                    "column":
                        field,

                    "image_like_rows":
                        image_like,
                }
            )


        if label_like > 0:

            label_columns.append(
                {
                    "column":
                        field,

                    "label_like_rows":
                        label_like,
                }
            )


    return {
        "path":
            str(
                path
            ),

        "sha256":
            sha256_file(
                path
            ),

        "rows":
            len(
                rows
            ),

        "fields":
            fields,

        "image_columns":
            image_columns,

        "label_columns":
            label_columns,
    }


# ============================================================
# 7. PREFLIGHT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — PRESERVE CLEAN FRUITBG EVIDENCE"
)

print(
    "+ RSC EXTERNAL PRE-INFERENCE ASSET AUDIT v1"
)

print(
    "============================================================"
)


for path, label in [
    (
        CLEAN_PROTOCOL,
        "Clean FruitBG protocol freeze",
    ),
    (
        CLEAN_PROTOCOL_SHA_RECORD,
        "Clean FruitBG protocol SHA record",
    ),
    (
        CLEAN_AGGREGATE,
        "Clean FruitBG aggregate",
    ),
    (
        CLEAN_PER_SEED,
        "Clean FruitBG per-seed CSV",
    ),
    (
        CLEAN_PER_FRUIT,
        "Clean FruitBG per-fruit CSV",
    ),
    (
        CLEAN_README,
        "Clean FruitBG README",
    ),
    (
        PARENT_PROTOCOL,
        "Parent formal evaluation protocol",
    ),
    (
        FORMAL_SYN_AGGREGATE,
        "Formal synthetic-test aggregate",
    ),
    (
        PUBLIC_REAL_ROOT,
        "Public-real-data root",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 8. VERIFY COMPLETED CLEAN EVIDENCE
# ============================================================

if (
    sha256_file(
        PARENT_PROTOCOL
    )
    != EXPECTED_PARENT_PROTOCOL_SHA
):

    raise RuntimeError(
        "Parent formal protocol SHA mismatch."
    )


if (
    sha256_file(
        FORMAL_SYN_AGGREGATE
    )
    != EXPECTED_SYN_AGGREGATE_SHA
):

    raise RuntimeError(
        "Formal synthetic aggregate SHA mismatch."
    )


if (
    sha256_file(
        CLEAN_PROTOCOL
    )
    != EXPECTED_CLEAN_PROTOCOL_SHA
):

    raise RuntimeError(
        "Clean FruitBG protocol SHA mismatch."
    )


if (
    sha256_file(
        CLEAN_AGGREGATE
    )
    != EXPECTED_CLEAN_AGGREGATE_SHA
):

    raise RuntimeError(
        "Clean FruitBG aggregate SHA mismatch."
    )


clean_aggregate = load_json(
    CLEAN_AGGREGATE
)


if clean_aggregate.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Clean FruitBG aggregate status != PASS."
    )


if bool(
    clean_aggregate[
        "seed_policy"
    ][
        "seed_selection"
    ]
):

    raise RuntimeError(
        "Clean aggregate unexpectedly indicates seed selection."
    )


if bool(
    clean_aggregate[
        "seed_policy"
    ][
        "threshold_tuning"
    ]
):

    raise RuntimeError(
        "Clean aggregate unexpectedly indicates threshold tuning."
    )


print()
print(
    "Completed clean FruitBG aggregate SHA: PASS"
)

print(
    "Completed clean FruitBG protocol SHA: PASS"
)

print(
    "No threshold tuning: PASS"
)

print(
    "No seed selection: PASS"
)


# ============================================================
# 9. PRESERVATION OUTPUT MUST BE FRESH
# ============================================================

if PACKAGE_ROOT.exists():

    raise RuntimeError(
        "Clean preservation package already exists:\n"
        f"{PACKAGE_ROOT}"
    )


if ZIP_PATH.exists():

    raise RuntimeError(
        "Clean preservation ZIP already exists:\n"
        f"{ZIP_PATH}"
    )


if ZIP_SHA_RECORD.exists():

    raise RuntimeError(
        "Clean preservation ZIP SHA record already exists:\n"
        f"{ZIP_SHA_RECORD}"
    )


PACKAGE_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 10. COPY COMPLETED CLEAN EVIDENCE
# ============================================================

copy_items = [
    CLEAN_PROTOCOL,
    CLEAN_PROTOCOL_SHA_RECORD,
    CLEAN_AGGREGATE,
    CLEAN_PER_SEED,
    CLEAN_PER_FRUIT,
    CLEAN_README,
]


for idx in [
    1,
    2,
    3,
]:

    copy_items.extend(
        [
            (
                CLEAN_ROOT
                / f"seed0{idx}_clean_per_image.csv"
            ),

            (
                CLEAN_ROOT
                / f"seed0{idx}_clean_summary.json"
            ),
        ]
    )


for src in copy_items:

    require(
        src,
        f"Clean evidence {src.name}",
    )


    dst = (
        PACKAGE_ROOT
        / "clean_evaluation"
        / src.name
    )


    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    shutil.copy2(
        src,
        dst,
    )


print()
print(
    "Completed clean FruitBG evidence copied: PASS"
)


# ============================================================
# 11. DISCOVER RSC-RELATED FILES
# ============================================================

rsc_named_files = sorted(
    p
    for p in PUBLIC_REAL_ROOT.rglob(
        "*"
    )
    if p.is_file()
    and "rsc"
    in p.name.lower()
)


print()
print(
    "============================================================"
)

print(
    "RSC-RELATED FILE DISCOVERY"
)

print(
    "============================================================"
)

print(
    f"RSC-named files found: {len(rsc_named_files)}"
)


for path in rsc_named_files[
    :100
]:

    print(
        "  "
        + str(
            path.relative_to(
                OVERLAY_ROOT
            )
        )
    )


if len(
    rsc_named_files
) > 100:

    print(
        f"  ... {len(rsc_named_files) - 100} more"
    )


# ============================================================
# 12. DISCOVER RSC CSV MANIFESTS
# ============================================================

rsc_csvs = sorted(
    p
    for p in PUBLIC_REAL_ROOT.rglob(
        "*.csv"
    )
    if "rsc"
    in p.name.lower()
    or "rsc"
    in str(
        p.parent
    ).lower()
)


csv_summaries = []


print()
print(
    "============================================================"
)

print(
    "RSC CSV / MANIFEST INVENTORY"
)

print(
    "============================================================"
)


for path in rsc_csvs:

    summary = summarize_csv(
        path
    )


    csv_summaries.append(
        summary
    )


    print()
    print(
        f"CSV: "
        f"{path.relative_to(OVERLAY_ROOT)}"
    )

    print(
        f"  rows:   {summary['rows']}"
    )

    print(
        f"  sha256: {summary['sha256']}"
    )

    print(
        "  fields:"
    )


    for field in summary[
        "fields"
    ]:

        print(
            f"    {field}"
        )


    if summary[
        "image_columns"
    ]:

        print(
            "  image-like columns:"
        )


        for record in summary[
            "image_columns"
        ]:

            print(
                f"    {record['column']}: "
                f"{record['image_like_rows']}"
            )


    if summary[
        "label_columns"
    ]:

        print(
            "  label-like columns:"
        )


        for record in summary[
            "label_columns"
        ]:

            print(
                f"    {record['column']}: "
                f"{record['label_like_rows']}"
            )


# ============================================================
# 13. VERIFY KNOWN EXACT-DEDUP SENSITIVITY MANIFEST
# ============================================================

exact_dedup_info = {
    "path":
        str(
            KNOWN_RSC_EXACT_DEDUP
        ),

    "exists":
        KNOWN_RSC_EXACT_DEDUP.is_file(),
}


if KNOWN_RSC_EXACT_DEDUP.is_file():

    exact_rows = read_csv(
        KNOWN_RSC_EXACT_DEDUP
    )


    exact_dedup_info[
        "rows"
    ] = len(
        exact_rows
    )

    exact_dedup_info[
        "sha256"
    ] = sha256_file(
        KNOWN_RSC_EXACT_DEDUP
    )

    exact_dedup_info[
        "fields"
    ] = (
        list(
            exact_rows[
                0
            ].keys()
        )
        if exact_rows
        else []
    )


    if len(
        exact_rows
    ) != 247:

        raise RuntimeError(
            "Known RSC exact-dedup sensitivity manifest "
            "does not contain 247 rows."
        )


    print()
    print(
        "Known exact-dedup RSC sensitivity manifest: PASS"
    )

    print(
        "Rows: 247"
    )

    print(
        f"SHA256: "
        f"{exact_dedup_info['sha256']}"
    )

else:

    print()
    print(
        "WARNING:"
    )

    print(
        "Known rsc_test_exact_dedup_v1.csv is not present "
        "in the current overlay."
    )


# ============================================================
# 14. FIND 250-ROW PRIMARY OFFICIAL-TEST CANDIDATES
# ============================================================

official_250_candidates = [
    record
    for record in csv_summaries
    if int(
        record[
            "rows"
        ]
    )
    == 250
]


print()
print(
    "============================================================"
)

print(
    "OFFICIAL RSC TEST-250 MANIFEST CANDIDATES"
)

print(
    "============================================================"
)


if official_250_candidates:

    for record in official_250_candidates:

        print(
            "  "
            + record[
                "path"
            ]
        )

else:

    print(
        "  NONE found among RSC CSV manifests"
    )


# ============================================================
# 15. PATH-EXISTENCE AUDIT FOR 250 / 247 CANDIDATES
# ============================================================

manifest_asset_audits = []


candidate_manifest_paths = []


for record in official_250_candidates:

    candidate_manifest_paths.append(
        Path(
            record[
                "path"
            ]
        )
    )


if KNOWN_RSC_EXACT_DEDUP.is_file():

    candidate_manifest_paths.append(
        KNOWN_RSC_EXACT_DEDUP
    )


# Deduplicate paths.
candidate_manifest_paths = list(
    dict.fromkeys(
        candidate_manifest_paths
    )
)


print()
print(
    "============================================================"
)

print(
    "RSC MANIFEST ASSET-PATH EXISTENCE AUDIT"
)

print(
    "============================================================"
)


for manifest_path in candidate_manifest_paths:

    rows = read_csv(
        manifest_path
    )


    fields = (
        list(
            rows[
                0
            ].keys()
        )
        if rows
        else []
    )


    result = {
        "manifest":
            str(
                manifest_path
            ),

        "rows":
            len(
                rows
            ),

        "columns":
            {},
    }


    print()
    print(
        f"Manifest:"
    )

    print(
        f"  {manifest_path}"
    )

    print(
        f"Rows: {len(rows)}"
    )


    for field in fields:

        image_values = [
            row.get(
                field
            )
            for row in rows
            if is_image_like(
                row.get(
                    field
                )
            )
        ]


        label_values = [
            row.get(
                field
            )
            for row in rows
            if is_label_like(
                row.get(
                    field
                )
            )
        ]


        if not image_values and not label_values:

            continue


        values = (
            image_values
            if image_values
            else label_values
        )


        exists = 0

        missing = 0

        resolved_examples = []

        missing_examples = []


        for value in values:

            resolution = resolve_path(
                value
            )


            if resolution[
                "exists"
            ]:

                exists += 1


                if len(
                    resolved_examples
                ) < 3:

                    resolved_examples.append(
                        resolution[
                            "resolved"
                        ]
                    )

            else:

                missing += 1


                if len(
                    missing_examples
                ) < 3:

                    missing_examples.append(
                        {
                            "stored":
                                str(
                                    value
                                ),

                            "candidates":
                                resolution[
                                    "candidates"
                                ],
                        }
                    )


        result[
            "columns"
        ][
            field
        ] = {
            "values":
                len(
                    values
                ),

            "exists":
                exists,

            "missing":
                missing,

            "resolved_examples":
                resolved_examples,

            "missing_examples":
                missing_examples,
        }


        print(
            f"  {field}: "
            f"values={len(values)} "
            f"exists={exists} "
            f"missing={missing}"
        )


    manifest_asset_audits.append(
        result
    )


# ============================================================
# 16. DISCOVER RSC-LIKE IMAGE / TXT FILES IN TREE
#
# This catches datasets whose manifest stores identifiers rather
# than explicit paths.
# ============================================================

rsc_tree_files = [
    p
    for p in PUBLIC_REAL_ROOT.rglob(
        "*"
    )
    if p.is_file()
    and (
        "rsc"
        in str(
            p
        ).lower()
        or "mobile"
        in str(
            p
        ).lower()
    )
]


rsc_tree_images = [
    p
    for p in rsc_tree_files
    if p.suffix.lower()
    in IMAGE_EXTENSIONS
]


rsc_tree_txt = [
    p
    for p in rsc_tree_files
    if p.suffix.lower()
    == ".txt"
]


print()
print(
    "============================================================"
)

print(
    "RSC-LIKE ASSET TREE DISCOVERY"
)

print(
    "============================================================"
)

print(
    f"Image files discovered: {len(rsc_tree_images)}"
)

print(
    f"TXT annotation files:   {len(rsc_tree_txt)}"
)


if rsc_tree_images:

    print()
    print(
        "Sample images:"
    )


    for path in rsc_tree_images[
        :10
    ]:

        print(
            "  "
            + str(
                path.relative_to(
                    OVERLAY_ROOT
                )
            )
        )


if rsc_tree_txt:

    print()
    print(
        "Sample TXT annotations:"
    )


    for path in rsc_tree_txt[
        :10
    ]:

        print(
            "  "
            + str(
                path.relative_to(
                    OVERLAY_ROOT
                )
            )
        )


# ============================================================
# 17. READ SAMPLE RSC IMAGES IF AVAILABLE
#
# Reading dimensions only. NO model inference.
# ============================================================

image_shape_counts = Counter()

failed_image_reads = []


for path in rsc_tree_images[
    :250
]:

    image = cv2.imread(
        str(
            path
        ),
        cv2.IMREAD_COLOR,
    )


    if image is None:

        failed_image_reads.append(
            str(
                path
            )
        )

        continue


    image_shape_counts[
        tuple(
            image.shape[
                :2
            ]
        )
    ] += 1


print()
print(
    "RSC discovered image shape distribution "
    "(up to first 250 files):"
)


if image_shape_counts:

    for shape, count in sorted(
        image_shape_counts.items()
    ):

        print(
            f"  {shape}: {count}"
        )

else:

    print(
        "  No RSC image files available to inspect."
    )


if failed_image_reads:

    print()
    print(
        f"Unreadable discovered images: "
        f"{len(failed_image_reads)}"
    )


# ============================================================
# 18. INSPECT YOLO-LIKE TXT ANNOTATION FORMAT
#
# Read-only. No conversion yet.
# ============================================================

txt_examples = []


for path in rsc_tree_txt[
    :10
]:

    lines = path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()


    nonempty = [
        line.strip()
        for line in lines
        if line.strip()
    ]


    txt_examples.append(
        {
            "path":
                str(
                    path
                ),

            "line_count":
                len(
                    nonempty
                ),

            "first_lines":
                nonempty[
                    :5
                ],
        }
    )


if txt_examples:

    print()
    print(
        "Annotation-format samples:"
    )


    for record in txt_examples[
        :5
    ]:

        print()
        print(
            f"  {record['path']}"
        )

        print(
            f"  non-empty lines: "
            f"{record['line_count']}"
        )


        for line in record[
            "first_lines"
        ]:

            print(
                f"    {line}"
            )


# ============================================================
# 19. DETERMINE READINESS — WITHOUT GUESSING
# ============================================================

has_247_manifest = (
    KNOWN_RSC_EXACT_DEDUP.is_file()
    and exact_dedup_info.get(
        "rows"
    )
    == 247
)

has_250_manifest = (
    len(
        official_250_candidates
    )
    >= 1
)


# A dataset can still be ready even if explicit paths are not
# stored in a manifest, so this is an informational readiness
# classification rather than a destructive failure.
rsc_asset_files_present = (
    len(
        rsc_tree_images
    )
    > 0
)


if (
    has_247_manifest
    and has_250_manifest
    and rsc_asset_files_present
):

    readiness = (
        "READY_FOR_FINAL_PROTOCOL_CONSTRUCTION"
    )

elif (
    has_247_manifest
    and has_250_manifest
):

    readiness = (
        "MANIFESTS_PRESENT_BUT_IMAGE_ASSETS_NOT_DISCOVERED"
    )

elif has_247_manifest:

    readiness = (
        "SENSITIVITY_247_PRESENT_PRIMARY_250_NEEDS_RESOLUTION"
    )

else:

    readiness = (
        "RSC_INPUTS_INCOMPLETE_IN_CURRENT_OVERLAY"
    )


# ============================================================
# 20. WRITE RSC PRE-INFERENCE AUDIT
# ============================================================

rsc_audit = {
    "audit_name":
        "DASO-Net RSC External Pre-Inference Asset Audit v1",

    "status":
        "PASS",

    "model_inference_performed":
        False,

    "training_performed":
        False,

    "external_tuning_performed":
        False,

    "scientific_role": {
        "rsc_primary":
            (
                "official authors' 250-image test split; "
                "untouched real-domain external detection test"
            ),

        "rsc_sensitivity":
            (
                "247-image exact-deduplicated sensitivity subset"
            ),

        "task":
            "object detection / bounding-box evaluation",

        "segmentation_mask_claim":
            False,

        "training_use":
            False,

        "crpa_use":
            False,

        "threshold_tuning_use":
            False,
    },

    "clean_evidence_preserved": {
        "aggregate_sha256":
            EXPECTED_CLEAN_AGGREGATE_SHA,

        "protocol_sha256":
            EXPECTED_CLEAN_PROTOCOL_SHA,
    },

    "public_real_root":
        str(
            PUBLIC_REAL_ROOT
        ),

    "rsc_named_files":
        [
            str(
                p
            )
            for p in rsc_named_files
        ],

    "csv_manifests":
        csv_summaries,

    "official_250_candidates":
        official_250_candidates,

    "known_exact_dedup_247":
        exact_dedup_info,

    "manifest_asset_audits":
        manifest_asset_audits,

    "rsc_tree": {
        "image_files":
            len(
                rsc_tree_images
            ),

        "txt_annotation_files":
            len(
                rsc_tree_txt
            ),

        "image_shape_distribution_first_250":
            {
                str(
                    shape
                ):
                    count
                for shape, count
                in sorted(
                    image_shape_counts.items()
                )
            },

        "annotation_examples":
            txt_examples,
    },

    "readiness":
        readiness,

    "policy": {
        "formal_rsc_protocol_frozen":
            False,

        "rsc_model_inference_allowed_by_this_cell":
            False,

        "reason":
            (
                "Asset/schema discovery must complete before "
                "freezing the RSC external evaluation protocol."
            ),
    },
}


save_json(
    RSC_AUDIT_JSON,
    rsc_audit,
)


# ============================================================
# 21. PRESERVATION README
# ============================================================

PACKAGE_README.write_text(
    f"""
DASO-Net — Clean FruitBG Complete / RSC Pre-Inference State
===========================================================

Completed held-out clean FruitBG evaluation:

  backgrounds = 246
  final seeds = 3

Primary:
  FP/image @ frozen operating threshold
    = 0.1301 +/- 0.0215

  fraction images with >=1 FP
    = 0.1125 +/- 0.0143

Decoded maximum-score distribution:
  mean   = 0.2279 +/- 0.0154
  median = 0.2139 +/- 0.0124
  p95    = 0.4033 +/- 0.0399

Diagnostic decoder floor 0.05:
  FP/image
    = 36.3008 +/- 6.4094

  fraction images with >=1 candidate
    = 0.9986 +/- 0.0023

Clean protocol SHA256:
  {EXPECTED_CLEAN_PROTOCOL_SHA}

Clean aggregate SHA256:
  {EXPECTED_CLEAN_AGGREGATE_SHA}

Next external stage:
  RSC 2025 mobile microscopy

Scientific rules:
  - official 250-image authors' test split = PRIMARY
  - exact-dedup 247 subset = SENSITIVITY
  - detection / bounding-box evaluation only
  - no external tuning
  - no threshold calibration on RSC
  - no checkpoint selection
  - no seed selection
""".strip()
    + "\n",
    encoding="utf-8",
)


# ============================================================
# 22. PRESERVATION MANIFEST
# ============================================================

records = []


for path in sorted(
    p
    for p in PACKAGE_ROOT.rglob(
        "*"
    )
    if p.is_file()
    and p != PACKAGE_MANIFEST
):

    records.append(
        {
            "relative_path":
                path.relative_to(
                    PACKAGE_ROOT
                ).as_posix(),

            "size_bytes":
                path.stat().st_size,

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


preservation_manifest = {
    "package_name":
        PACKAGE_NAME,

    "status":
        "PASS",

    "scientific_state":
        (
            "Held-out clean FruitBG evaluation complete; "
            "RSC external evaluation has not begun."
        ),

    "clean_fruitbg": {
        "protocol_sha256":
            EXPECTED_CLEAN_PROTOCOL_SHA,

        "aggregate_sha256":
            EXPECTED_CLEAN_AGGREGATE_SHA,

        "all_three_final_seeds":
            True,

        "threshold_tuning":
            False,

        "best_seed_selection":
            False,
    },

    "rsc": {
        "model_inference_performed":
            False,

        "external_tuning_performed":
            False,

        "audit":
            RSC_AUDIT_JSON.name,

        "readiness":
            readiness,
    },

    "files":
        records,
}


save_json(
    PACKAGE_MANIFEST,
    preservation_manifest,
)


# ============================================================
# 23. READ-BACK FILE SHA AUDIT
# ============================================================

readback = load_json(
    PACKAGE_MANIFEST
)


if readback.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Preservation manifest status != PASS."
    )


for record in readback[
    "files"
]:

    path = (
        PACKAGE_ROOT
        / record[
            "relative_path"
        ]
    )


    require(
        path,
        "Preserved artifact",
    )


    if (
        path.stat().st_size
        != int(
            record[
                "size_bytes"
            ]
        )
    ):

        raise RuntimeError(
            "Preserved artifact size mismatch:\n"
            f"{path}"
        )


    if (
        sha256_file(
            path
        )
        != record[
            "sha256"
        ]
    ):

        raise RuntimeError(
            "Preserved artifact SHA mismatch:\n"
            f"{path}"
        )


print()
print(
    "Clean evidence preservation SHA audit: PASS"
)


# ============================================================
# 24. CREATE ZIP
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:

    for path in sorted(
        p
        for p in PACKAGE_ROOT.rglob(
            "*"
        )
        if p.is_file()
    ):

        arcname = (
            Path(
                PACKAGE_NAME
            )
            / path.relative_to(
                PACKAGE_ROOT
            )
        )


        zf.write(
            path,
            arcname=str(
                arcname
            ),
        )


with zipfile.ZipFile(
    ZIP_PATH,
    "r",
) as zf:

    bad = zf.testzip()


if bad is not None:

    raise RuntimeError(
        f"ZIP CRC failure: {bad}"
    )


zip_sha = sha256_file(
    ZIP_PATH
)


ZIP_SHA_RECORD.write_text(
    f"{zip_sha}  {ZIP_PATH.name}\n",
    encoding="utf-8",
)


# ============================================================
# 25. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "CLEAN FRUITBG EVIDENCE PRESERVED"
)

print(
    "============================================================"
)

print()

print(
    "Clean protocol SHA256:"
)

print(
    EXPECTED_CLEAN_PROTOCOL_SHA
)

print()

print(
    "Clean aggregate SHA256:"
)

print(
    EXPECTED_CLEAN_AGGREGATE_SHA
)

print()

print(
    "Preservation ZIP:"
)

print(
    ZIP_PATH
)

print()

print(
    "ZIP size:"
)

print(
    f"{ZIP_PATH.stat().st_size / (1024 ** 2):.2f} MiB"
)

print()

print(
    "ZIP SHA256:"
)

print(
    zip_sha
)

print()

print(
    "============================================================"
)

print(
    "RSC EXTERNAL PRE-INFERENCE AUDIT"
)

print(
    "============================================================"
)

print()

print(
    f"Known exact-dedup 247 manifest: "
    f"{'YES' if has_247_manifest else 'NO'}"
)

print(
    f"250-row primary candidates:     "
    f"{len(official_250_candidates)}"
)

print(
    f"RSC-like image files found:     "
    f"{len(rsc_tree_images)}"
)

print(
    f"RSC-like TXT labels found:      "
    f"{len(rsc_tree_txt)}"
)

print()

print(
    "Readiness:"
)

print(
    f"  {readiness}"
)

print()

print(
    "RSC audit JSON:"
)

print(
    RSC_AUDIT_JSON
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET CLEAN PRESERVATION + RSC PRE-INFERENCE AUDIT: PASS"
)

print(
    "============================================================"
)

print()

print(
    "NO TRAINING WAS PERFORMED."
)

print(
    "NO MODEL INFERENCE WAS PERFORMED."
)

print(
    "NO RSC THRESHOLD TUNING WAS PERFORMED."
)

print(
    "NO RSC DATA WERE USED FOR MODEL SELECTION."
)

print()

print(
    "STOP HERE."
)

print(
    "Next: use the discovered RSC schema/assets to freeze the "
    "official-250 primary + exact-dedup-247 sensitivity "
    "external evaluation protocol before inference."
)

DASO-Net — PRESERVE CLEAN FRUITBG EVIDENCE
+ RSC EXTERNAL PRE-INFERENCE ASSET AUDIT v1
PASS  Clean FruitBG protocol freeze
PASS  Clean FruitBG protocol SHA record
PASS  Clean FruitBG aggregate
PASS  Clean FruitBG per-seed CSV
PASS  Clean FruitBG per-fruit CSV
PASS  Clean FruitBG README
PASS  Parent formal evaluation protocol
PASS  Formal synthetic-test aggregate
PASS  Public-real-data root

Completed clean FruitBG aggregate SHA: PASS
Completed clean FruitBG protocol SHA: PASS
No threshold tuning: PASS
No seed selection: PASS

Completed clean FruitBG evidence copied: PASS

RSC-RELATED FILE DISCOVERY
RSC-named files found: 0

RSC CSV / MANIFEST INVENTORY

Known rsc_test_exact_dedup_v1.csv is not present in the current overlay.

OFFICIAL RSC TEST-250 MANIFEST CANDIDATES
  NONE found among RSC CSV manifests

RSC MANIFEST ASSET-PATH EXISTENCE AUDIT

RSC-LIKE ASSET TREE DISCOVERY
Image files discovered: 0
TXT annotation files:   0

RSC discovered image shape distribution (up to first 250 fil

In [23]:
# ============================================================
# DASO-Net — RSC EXTERNAL TEST ASSETS
# KAGGLE PRE-INFERENCE IMMUTABILITY / READINESS AUDIT v1
#
# PURPOSE
# -------
# READ-ONLY audit of the newly attached PRIVATE Kaggle RSC
# external-test dataset.
#
# This cell:
#   - performs NO training
#   - performs NO model inference
#   - performs NO threshold tuning
#   - performs NO calibration
#   - performs NO checkpoint selection
#   - performs NO seed selection
#   - modifies NO scientific input assets
#
# It verifies:
#   1. frozen local ZIP SHA record
#   2. Kaggle-mounted package structure
#   3. top-level vs internal state JSON identity
#   4. internal package file-manifest hashes/sizes
#   5. exact scientific payload = 250 images + 250 labels
#   6. official test-250 manifest
#   7. exact-dedup sensitivity-247 manifest
#   8. image SHA + dimensions
#   9. YOLO labels read-only
#  10. 539 GT boxes
#  11. class IDs = {0}
#  12. four legitimate zero-box images
#  13. exact dedup exclusions = 182/183/184
#  14. only test_113 zero-box image remains in dedup-247
#  15. frozen provenance manifest SHAs
#  16. no-RSC-training/tuning policy flags
#
# Only derived output:
#   /kaggle/working/dasonet_final_analysis/
#       rsc_external_preinference_asset_audit_v1/
#
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import math
import traceback
from collections import Counter
from pathlib import Path

from PIL import Image


# ============================================================
# 1. FROZEN EXPECTATIONS
# ============================================================

DATASET_ROOT = Path(
    "/kaggle/input/datasets/abinavprabhuv/"
    "dasonet-rsc-external-test-assets-v1"
)

EXPECTED_PACKAGE_NAME = "DASO-Net-RSC-External-TestAssets-v1"

EXPECTED_ZIP_SHA256 = (
    "71fba0d782f5588e3c1ad19827436f849cb58f8485c9c5d08bb1f4321647ae72"
)

EXPECTED_ZIP_FILENAME = (
    "DASO-Net-RSC-External-TestAssets-v1.zip"
)

EXPECTED_RSC_SOURCE_ID = (
    "rsc2025_mobile_consumer"
)

EXPECTED_TOTAL_ROWS = 2490
EXPECTED_TRAIN_ROWS = 1990
EXPECTED_VAL_ROWS = 250
EXPECTED_TEST_ROWS = 250
EXPECTED_DEDUP_ROWS = 247

EXPECTED_GT_BOXES = 539

EXPECTED_ZERO_BOX_FILENAMES = {
    "test_113.jpg",
    "test_182.jpg",
    "test_183.jpg",
    "test_184.jpg",
}

EXPECTED_DEDUP_EXCLUSIONS = {
    "test_182.jpg",
    "test_183.jpg",
    "test_184.jpg",
}

EXPECTED_DEDUP_ZERO_BOX_FILENAMES = {
    "test_113.jpg",
}

EXPECTED_SOURCE_SPECIFIC_SHA256 = (
    "1bc39ae0ecb03436b7948138a6a7f285"
    "47d0d610e116442328d7e5dad0a46301"
)

EXPECTED_IMAGE_MANIFEST_SHA256 = (
    "8396b044322f3fc25c653c2b78bff83a"
    "d930b71d4c70a8b2fed554a4ea47e8f8"
)

EXPECTED_DEDUP_MANIFEST_SHA256 = (
    "09562a1414f68b7c3088198145a004d72"
    "c8308abced91c4e0aee624861e57bc5"
)

EXPECTED_SHAPE_DISTRIBUTION = {
    (960, 885): 3,
    (960, 888): 244,
    (961, 889): 3,
}

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
}


# ============================================================
# 2. DERIVED OUTPUT
# ============================================================

AUDIT_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_external_preinference_asset_audit_v1"
)

AUDIT_JSON = (
    AUDIT_DIR
    / "RSC_EXTERNAL_PREINFERENCE_ASSET_AUDIT_v1.json"
)

AUDIT_SHA_RECORD = (
    AUDIT_DIR
    / "RSC_EXTERNAL_PREINFERENCE_ASSET_AUDIT_v1.sha256.txt"
)


# ============================================================
# 3. HELPERS
# ============================================================

checks = []


def record(
    name: str,
    status: str = "PASS",
    **details,
) -> None:

    item = {
        "check": name,
        "status": status,
    }

    item.update(details)

    checks.append(item)

    print(
        f"{status:<5} {name}"
    )


def require(
    condition: bool,
    message: str,
) -> None:

    if not condition:
        raise RuntimeError(
            message
        )


def require_path(
    path: Path,
    label: str,
) -> None:

    require(
        path.exists(),
        f"{label} missing:\n{path}",
    )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def read_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path: Path,
    obj,
) -> None:

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
        )
        + "\n",
        encoding="utf-8",
    )


def read_csv_rows(
    path: Path,
):

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def parse_bool(
    value,
) -> bool:

    text = str(
        value
    ).strip().lower()

    if text in {
        "true",
        "1",
        "yes",
        "y",
    }:

        return True

    if text in {
        "false",
        "0",
        "no",
        "n",
    }:

        return False

    raise RuntimeError(
        "Could not parse boolean value: "
        f"{value!r}"
    )


def as_int(
    value,
    label: str,
) -> int:

    try:

        number = float(
            value
        )

    except Exception as exc:

        raise RuntimeError(
            f"Could not parse integer field {label}: "
            f"{value!r}"
        ) from exc

    require(
        math.isfinite(
            number
        ),
        f"Non-finite integer field {label}: {value!r}",
    )

    require(
        number.is_integer(),
        f"Non-integral integer field {label}: {value!r}",
    )

    return int(
        number
    )


# ============================================================
# 4. BANNER
# ============================================================

print(
    "=" * 76
)

print(
    "DASO-Net — RSC KAGGLE PRE-INFERENCE ASSET AUDIT v1"
)

print(
    "=" * 76
)

print()

print(
    "Scientific action:"
)

print(
    "  READ-ONLY external-test asset verification"
)

print(
    "  NO training / inference / tuning / calibration / selection"
)

print()


# ============================================================
# 5. CREATE VERSIONED AUDIT OUTPUT
#
# Never overwrite an earlier attempt.
# ============================================================

if AUDIT_DIR.exists():

    raise RuntimeError(
        "Audit output directory already exists.\n"
        "Preserve the previous audit rather than overwriting it:\n"
        f"{AUDIT_DIR}"
    )


AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


audit = {
    "audit_name":
        "DASO-Net RSC External Pre-Inference Asset Audit v1",

    "status":
        "RUNNING",

    "scientific_action":
        (
            "Read-only verification of frozen RSC external-test "
            "assets before any RSC model inference."
        ),

    "training_performed":
        False,

    "model_inference_performed":
        False,

    "threshold_tuning_performed":
        False,

    "calibration_performed":
        False,

    "checkpoint_selection_performed":
        False,

    "seed_selection_performed":
        False,

    "scientific_assets_modified":
        False,

    "checks":
        checks,
}


write_json(
    AUDIT_JSON,
    audit,
)


# ============================================================
# 6. MAIN AUDIT WITH FAILURE PRESERVATION
# ============================================================

try:

    # ========================================================
    # 6A. DATASET ROOT
    # ========================================================

    require_path(
        DATASET_ROOT,
        "Attached RSC Kaggle dataset root",
    )

    require(
        DATASET_ROOT.is_dir(),
        "RSC Kaggle dataset root is not a directory:\n"
        f"{DATASET_ROOT}",
    )

    record(
        "Attached private RSC dataset root",
        path=str(
            DATASET_ROOT
        ),
    )


    # ========================================================
    # 6B. TOP-LEVEL SHA RECORD
    # ========================================================

    top_sha_record = (
        DATASET_ROOT
        / "DASO-Net-RSC-External-TestAssets-v1.sha256.txt"
    )

    require_path(
        top_sha_record,
        "Top-level frozen ZIP SHA record",
    )

    sha_lines = [
        line.strip()
        for line in top_sha_record.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

    require(
        len(
            sha_lines
        ) == 1,
        "Expected exactly one non-empty line in ZIP SHA record.\n"
        f"Found: {len(sha_lines)}",
    )

    sha_parts = (
        sha_lines[
            0
        ]
        .replace(
            "*",
            " ",
        )
        .split()
    )

    require(
        len(
            sha_parts
        ) >= 2,
        "Could not parse ZIP SHA record:\n"
        f"{sha_lines[0]}",
    )

    recorded_zip_sha = (
        sha_parts[
            0
        ].strip().lower()
    )

    recorded_zip_filename = (
        sha_parts[
            -1
        ].strip()
    )

    require(
        recorded_zip_sha
        == EXPECTED_ZIP_SHA256,
        "Frozen ZIP SHA record differs from local packaging result.\n"
        f"Expected: {EXPECTED_ZIP_SHA256}\n"
        f"Actual:   {recorded_zip_sha}",
    )

    require(
        recorded_zip_filename
        == EXPECTED_ZIP_FILENAME,
        "Frozen ZIP SHA record filename mismatch.\n"
        f"Expected: {EXPECTED_ZIP_FILENAME}\n"
        f"Actual:   {recorded_zip_filename}",
    )

    record(
        "Frozen local ZIP SHA record",
        sha256=recorded_zip_sha,
        filename=recorded_zip_filename,
    )


    # ========================================================
    # 6C. CHECK WHETHER KAGGLE EXPOSES ORIGINAL ZIP BYTES
    #
    # Kaggle often auto-extracts uploaded ZIPs and does not
    # mount the ZIP itself. Do not reconstruct it.
    # ========================================================

    zip_candidates = sorted(
        DATASET_ROOT.rglob(
            EXPECTED_ZIP_FILENAME
        )
    )

    require(
        len(
            zip_candidates
        ) <= 1,
        "Multiple original RSC ZIP files unexpectedly mounted:\n"
        + "\n".join(
            str(
                p
            )
            for p in zip_candidates
        ),
    )

    zip_bytes_reverified = False

    if len(
        zip_candidates
    ) == 1:

        mounted_zip = (
            zip_candidates[
                0
            ]
        )

        mounted_zip_sha = sha256_file(
            mounted_zip
        )

        require(
            mounted_zip_sha
            == EXPECTED_ZIP_SHA256,
            "Mounted RSC ZIP SHA mismatch.\n"
            f"Expected: {EXPECTED_ZIP_SHA256}\n"
            f"Actual:   {mounted_zip_sha}",
        )

        zip_bytes_reverified = True

        record(
            "Mounted original ZIP byte SHA",
            path=str(
                mounted_zip
            ),
            sha256=mounted_zip_sha,
        )

    else:

        print(
            "INFO  Kaggle did not expose the original ZIP bytes."
        )

        print(
            "INFO  Kaggle mounted the extracted ZIP contents instead."
        )

        print(
            "INFO  ZIP will NOT be reconstructed or re-zipped."
        )

        record(
            "Kaggle ZIP handling",
            mode="auto-extracted contents",
            original_zip_bytes_exposed=False,
            frozen_sha_record_verified=True,
        )


    # ========================================================
    # 6D. IDENTIFY EXACT INTERNAL PACKAGE ROOT
    # ========================================================

    primary_candidates = sorted(
        DATASET_ROOT.rglob(
            "RSC_OFFICIAL_TEST250_MANIFEST_v1.csv"
        )
    )

    package_root_candidates = []

    for manifest_path in primary_candidates:

        candidate = (
            manifest_path.parent
        )

        required_names = {
            "RSC_OFFICIAL_TEST250_MANIFEST_v1.csv",
            "RSC_EXACT_DEDUP247_MANIFEST_v1.csv",
            "RSC_EXTERNAL_TEST_FILE_MANIFEST_v1.json",
            "RSC_EXTERNAL_TEST_ASSETS_STATE_v1.json",
            "README_RSC_EXTERNAL_TEST_ASSETS.txt",
        }

        present_names = {
            p.name
            for p in candidate.iterdir()
            if p.is_file()
        }

        if required_names.issubset(
            present_names
        ):

            package_root_candidates.append(
                candidate
            )


    package_root_candidates = sorted(
        set(
            package_root_candidates
        )
    )

    require(
        len(
            package_root_candidates
        ) == 1,
        "Expected exactly one valid internal RSC package root; "
        f"found {len(package_root_candidates)}.\n"
        + "\n".join(
            str(
                p
            )
            for p in package_root_candidates
        ),
    )

    PACKAGE_ROOT = (
        package_root_candidates[
            0
        ]
    )

    PACKAGE_PROJECT_ROOT = (
        PACKAGE_ROOT
        / "DASO-Net-Phase1A"
    )

    require_path(
        PACKAGE_PROJECT_ROOT,
        "Packaged DASO-Net project payload root",
    )

    record(
        "Unique internal RSC package root",
        path=str(
            PACKAGE_ROOT
        ),
    )


    # ========================================================
    # 6E. TOP-LEVEL STATE VS INTERNAL STATE
    # ========================================================

    top_state_path = (
        DATASET_ROOT
        / "RSC_EXTERNAL_TEST_ASSETS_STATE_v1.json"
    )

    internal_state_path = (
        PACKAGE_ROOT
        / "RSC_EXTERNAL_TEST_ASSETS_STATE_v1.json"
    )

    require_path(
        top_state_path,
        "Top-level RSC state JSON",
    )

    require_path(
        internal_state_path,
        "Internal RSC state JSON",
    )

    top_state_sha = sha256_file(
        top_state_path
    )

    internal_state_sha = sha256_file(
        internal_state_path
    )

    require(
        top_state_sha
        == internal_state_sha,
        "Top-level and internal RSC state JSON differ.\n"
        f"Top-level SHA: {top_state_sha}\n"
        f"Internal SHA:  {internal_state_sha}",
    )

    record(
        "Top-level/internal state JSON byte identity",
        sha256=top_state_sha,
    )


    # ========================================================
    # 6F. STATE CONTRACT
    # ========================================================

    state = read_json(
        internal_state_path
    )

    require(
        state.get(
            "status"
        ) == "PASS",
        "RSC package state status is not PASS.",
    )

    require(
        state.get(
            "package_name"
        ) == EXPECTED_PACKAGE_NAME,
        "RSC package_name mismatch.",
    )

    require(
        state.get(
            "rsc_source_id"
        ) == EXPECTED_RSC_SOURCE_ID,
        "RSC source ID mismatch.",
    )

    author_split = state.get(
        "author_split",
        {},
    )

    require(
        author_split.get(
            "total"
        ) == EXPECTED_TOTAL_ROWS,
        "State author total != 2490.",
    )

    require(
        author_split.get(
            "train"
        ) == EXPECTED_TRAIN_ROWS,
        "State author train != 1990.",
    )

    require(
        author_split.get(
            "val"
        ) == EXPECTED_VAL_ROWS,
        "State author val != 250.",
    )

    require(
        author_split.get(
            "test"
        ) == EXPECTED_TEST_ROWS,
        "State author test != 250.",
    )

    payload_state = state.get(
        "packaged_payload",
        {},
    )

    require(
        payload_state.get(
            "official_test_images"
        ) == 250,
        "State official_test_images != 250.",
    )

    require(
        payload_state.get(
            "official_test_labels"
        ) == 250,
        "State official_test_labels != 250.",
    )

    require(
        payload_state.get(
            "files"
        ) == 500,
        "State scientific payload files != 500.",
    )

    require(
        payload_state.get(
            "total_gt_boxes"
        ) == EXPECTED_GT_BOXES,
        "State total GT boxes != 539.",
    )

    require(
        payload_state.get(
            "zero_box_label_count"
        ) == 4,
        "State zero-box label count != 4.",
    )

    require(
        set(
            payload_state.get(
                "zero_box_filenames",
                [],
            )
        )
        == EXPECTED_ZERO_BOX_FILENAMES,
        "State zero-box filename set mismatch.",
    )

    state_dedup = state.get(
        "exact_dedup_sensitivity",
        {},
    )

    require(
        state_dedup.get(
            "rows"
        ) == EXPECTED_DEDUP_ROWS,
        "State exact-dedup rows != 247.",
    )

    require(
        set(
            state_dedup.get(
                "excluded_official_test_filenames",
                [],
            )
        )
        == EXPECTED_DEDUP_EXCLUSIONS,
        "State dedup exclusion set mismatch.",
    )

    require(
        set(
            state_dedup.get(
                "zero_box_filenames",
                [],
            )
        )
        == EXPECTED_DEDUP_ZERO_BOX_FILENAMES,
        "State dedup zero-box set mismatch.",
    )

    require(
        set(
            state_dedup.get(
                "excluded_zero_box_filenames",
                [],
            )
        )
        == EXPECTED_DEDUP_EXCLUSIONS,
        "State excluded zero-box set mismatch.",
    )

    record(
        "Internal RSC package state contract",
        source_id=EXPECTED_RSC_SOURCE_ID,
        test_rows=250,
        dedup_rows=247,
        gt_boxes=539,
        zero_box_labels=4,
    )


    # ========================================================
    # 6G. SCIENTIFIC POLICY FLAGS
    # ========================================================

    policy = state.get(
        "policy",
        {},
    )

    expected_false_policy = [
        "rsc_training_use",
        "rsc_crpa_use",
        "rsc_calibration_use",
        "rsc_threshold_tuning",
        "rsc_checkpoint_selection",
        "rsc_seed_selection",
        "model_inference_performed",
        "scientific_assets_regenerated",
        "scientific_assets_modified",
    ]

    for key in expected_false_policy:

        require(
            policy.get(
                key
            ) is False,
            f"State policy flag must be False: {key}",
        )

    record(
        "No-training/no-tuning/no-selection package policy",
        flags_verified=expected_false_policy,
    )


    # ========================================================
    # 6H. FROZEN PROVENANCE SHAs FROM STATE
    # ========================================================

    frozen_state = state.get(
        "frozen_processed_manifests",
        {},
    )

    require(
        frozen_state.get(
            "source_specific",
            {},
        ).get(
            "sha256"
        )
        == EXPECTED_SOURCE_SPECIFIC_SHA256,
        "State source-specific manifest SHA mismatch.",
    )

    require(
        frozen_state.get(
            "image_manifest",
            {},
        ).get(
            "sha256"
        )
        == EXPECTED_IMAGE_MANIFEST_SHA256,
        "State image-manifest SHA mismatch.",
    )

    require(
        frozen_state.get(
            "exact_dedup_test",
            {},
        ).get(
            "sha256"
        )
        == EXPECTED_DEDUP_MANIFEST_SHA256,
        "State exact-dedup manifest SHA mismatch.",
    )

    record(
        "Frozen provenance SHA identities in state JSON"
    )


    # ========================================================
    # 6I. INTERNAL FILE MANIFEST
    # ========================================================

    file_manifest_path = (
        PACKAGE_ROOT
        / "RSC_EXTERNAL_TEST_FILE_MANIFEST_v1.json"
    )

    require_path(
        file_manifest_path,
        "Internal package file manifest",
    )

    file_manifest = read_json(
        file_manifest_path
    )

    require(
        file_manifest.get(
            "status"
        ) == "PASS",
        "Internal package file manifest status != PASS.",
    )

    require(
        file_manifest.get(
            "official_primary_test_images"
        ) == 250,
        "File manifest official image count != 250.",
    )

    require(
        file_manifest.get(
            "official_primary_test_labels"
        ) == 250,
        "File manifest official label count != 250.",
    )

    require(
        file_manifest.get(
            "official_primary_zero_box_labels"
        ) == 4,
        "File manifest zero-box count != 4.",
    )

    require(
        file_manifest.get(
            "exact_dedup_sensitivity_rows"
        ) == 247,
        "File manifest dedup count != 247.",
    )

    require(
        file_manifest.get(
            "scientific_payload_files"
        ) == 500,
        "File manifest scientific payload count != 500.",
    )

    file_records = file_manifest.get(
        "files",
        [],
    )

    require(
        isinstance(
            file_records,
            list
        ),
        "Internal file-manifest files field is not a list.",
    )

    require(
        len(
            file_records
        ) > 0,
        "Internal package file manifest contains no file records.",
    )


    # ========================================================
    # 6J. VERIFY EVERY FILE RECORD SIZE + SHA
    # ========================================================

    manifest_relative_paths = set()

    for index, record_item in enumerate(
        file_records,
        start=1,
    ):

        relative_path = str(
            record_item[
                "relative_path"
            ]
        )

        require(
            relative_path
            not in manifest_relative_paths,
            "Duplicate relative path in internal file manifest:\n"
            f"{relative_path}",
        )

        manifest_relative_paths.add(
            relative_path
        )

        mounted_path = (
            PACKAGE_ROOT
            / Path(
                relative_path
            )
        )

        require_path(
            mounted_path,
            "File-manifest member",
        )

        expected_size = int(
            record_item[
                "size_bytes"
            ]
        )

        actual_size = (
            mounted_path
            .stat()
            .st_size
        )

        require(
            actual_size
            == expected_size,
            "Mounted package file size mismatch.\n"
            f"File:     {mounted_path}\n"
            f"Expected: {expected_size}\n"
            f"Actual:   {actual_size}",
        )

        expected_sha = str(
            record_item[
                "sha256"
            ]
        ).strip().lower()

        actual_sha = sha256_file(
            mounted_path
        )

        require(
            actual_sha
            == expected_sha,
            "Mounted package file SHA mismatch.\n"
            f"File:     {mounted_path}\n"
            f"Expected: {expected_sha}\n"
            f"Actual:   {actual_sha}",
        )

        if (
            index % 50 == 0
            or index == len(
                file_records
            )
        ):

            print(
                f"  verified package file hashes "
                f"{index}/{len(file_records)}"
            )

    record(
        "Internal file-manifest size/SHA verification",
        files_verified=len(
            file_records
        ),
    )


    # ========================================================
    # 6K. VERIFY COMPLETE PACKAGE FILE SET
    #
    # FILE_MANIFEST was generated before these three metadata
    # files were written/closed into the package:
    #   - file manifest itself
    #   - state JSON
    #   - README
    #
    # No other untracked file is allowed.
    # ========================================================

    actual_package_files = {
        p.relative_to(
            PACKAGE_ROOT
        ).as_posix()
        for p in PACKAGE_ROOT.rglob(
            "*"
        )
        if p.is_file()
    }

    allowed_unmanifested = {
        "RSC_EXTERNAL_TEST_FILE_MANIFEST_v1.json",
        "RSC_EXTERNAL_TEST_ASSETS_STATE_v1.json",
        "README_RSC_EXTERNAL_TEST_ASSETS.txt",
    }

    expected_complete_package_files = (
        manifest_relative_paths
        | allowed_unmanifested
    )

    unexpected_files = sorted(
        actual_package_files
        - expected_complete_package_files
    )

    missing_files = sorted(
        expected_complete_package_files
        - actual_package_files
    )

    require(
        not unexpected_files,
        "Unexpected untracked files inside RSC package:\n"
        + "\n".join(
            unexpected_files
        ),
    )

    require(
        not missing_files,
        "Expected RSC package files missing:\n"
        + "\n".join(
            missing_files
        ),
    )

    record(
        "Complete internal package file-set closure",
        total_files=len(
            actual_package_files
        ),
    )


    # ========================================================
    # 6L. EXACT SCIENTIFIC PAYLOAD
    # ========================================================

    payload_files = sorted(
        p
        for p in PACKAGE_PROJECT_ROOT.rglob(
            "*"
        )
        if p.is_file()
    )

    payload_images = [
        p
        for p in payload_files
        if p.suffix.lower()
        in IMAGE_EXTENSIONS
    ]

    payload_txt = [
        p
        for p in payload_files
        if p.suffix.lower()
        == ".txt"
    ]

    require(
        len(
            payload_images
        ) == 250,
        "Mounted scientific payload image count != 250.\n"
        f"Actual: {len(payload_images)}",
    )

    require(
        len(
            payload_txt
        ) == 250,
        "Mounted scientific payload TXT count != 250.\n"
        f"Actual: {len(payload_txt)}",
    )

    require(
        len(
            payload_files
        ) == 500,
        "Mounted scientific payload total files != 500.\n"
        f"Actual: {len(payload_files)}",
    )

    record(
        "Exact mounted scientific payload",
        images=250,
        labels=250,
        total_files=500,
    )


    # ========================================================
    # 6M. PRIMARY OFFICIAL TEST-250 MANIFEST
    # ========================================================

    primary_manifest_path = (
        PACKAGE_ROOT
        / "RSC_OFFICIAL_TEST250_MANIFEST_v1.csv"
    )

    require_path(
        primary_manifest_path,
        "Primary official test-250 manifest",
    )

    primary_rows = read_csv_rows(
        primary_manifest_path
    )

    require(
        len(
            primary_rows
        ) == 250,
        "Primary official test manifest row count != 250.\n"
        f"Actual: {len(primary_rows)}",
    )

    filenames = [
        str(
            row[
                "filename"
            ]
        ).strip()
        for row in primary_rows
    ]

    require(
        len(
            set(
                filenames
            )
        ) == 250,
        "Primary official-test filenames are not unique.",
    )

    official_indices = [
        as_int(
            row[
                "official_test_index"
            ],
            "official_test_index",
        )
        for row in primary_rows
    ]

    require(
        official_indices
        == list(
            range(
                1,
                251,
            )
        ),
        "Primary official-test index order is not exactly 1..250.",
    )

    for row in primary_rows:

        require(
            str(
                row[
                    "source_id"
                ]
            ).strip()
            == EXPECTED_RSC_SOURCE_ID,
            "Primary manifest contains unexpected source_id.",
        )

        require(
            str(
                row[
                    "source_split"
                ]
            ).strip().lower()
            == "test",
            "Primary manifest contains non-test source_split.",
        )

    record(
        "Primary official authors' test manifest",
        rows=250,
        unique_filenames=250,
    )


    # ========================================================
    # 6N. VERIFY EVERY PRIMARY IMAGE/LABEL PAIR
    # ========================================================

    observed_gt_boxes = 0

    observed_class_ids = Counter()

    observed_zero_box_filenames = set()

    observed_shapes = Counter()

    primary_by_filename = {}

    for index, row in enumerate(
        primary_rows,
        start=1,
    ):

        filename = str(
            row[
                "filename"
            ]
        ).strip()

        primary_by_filename[
            filename
        ] = row

        image_rel = Path(
            str(
                row[
                    "project_image_path"
                ]
            ).strip()
        )

        label_rel = Path(
            str(
                row[
                    "project_label_path"
                ]
            ).strip()
        )

        image_path = (
            PACKAGE_PROJECT_ROOT
            / image_rel
        )

        label_path = (
            PACKAGE_PROJECT_ROOT
            / label_rel
        )

        require_path(
            image_path,
            "Primary RSC image",
        )

        require_path(
            label_path,
            "Primary RSC YOLO label",
        )


        # ----------------------------------------------------
        # Image SHA
        # ----------------------------------------------------

        expected_image_sha = str(
            row[
                "image_sha256"
            ]
        ).strip().lower()

        actual_image_sha = sha256_file(
            image_path
        )

        require(
            actual_image_sha
            == expected_image_sha,
            "Primary image SHA mismatch.\n"
            f"Image:    {image_path}\n"
            f"Expected: {expected_image_sha}\n"
            f"Actual:   {actual_image_sha}",
        )


        # ----------------------------------------------------
        # Label SHA
        # ----------------------------------------------------

        expected_label_sha = str(
            row[
                "label_sha256"
            ]
        ).strip().lower()

        actual_label_sha = sha256_file(
            label_path
        )

        require(
            actual_label_sha
            == expected_label_sha,
            "Primary label SHA mismatch.\n"
            f"Label:    {label_path}\n"
            f"Expected: {expected_label_sha}\n"
            f"Actual:   {actual_label_sha}",
        )


        # ----------------------------------------------------
        # Image decode + dimensions
        # ----------------------------------------------------

        try:

            with Image.open(
                image_path
            ) as img:

                img.load()

                width, height = (
                    img.size
                )

        except Exception as exc:

            raise RuntimeError(
                "Could not decode RSC image:\n"
                f"{image_path}"
            ) from exc

        expected_width = as_int(
            row[
                "width"
            ],
            "width",
        )

        expected_height = as_int(
            row[
                "height"
            ],
            "height",
        )

        require(
            width
            == expected_width
            and height
            == expected_height,
            "Primary image dimension mismatch.\n"
            f"Image:    {image_path}\n"
            f"Manifest: {expected_width}x{expected_height}\n"
            f"Actual:   {width}x{height}",
        )

        observed_shapes[
            (
                height,
                width,
            )
        ] += 1


        # ----------------------------------------------------
        # YOLO annotation validation
        # ----------------------------------------------------

        raw_lines = label_path.read_text(
            encoding="utf-8",
            errors="strict",
        ).splitlines()

        lines = [
            line.strip()
            for line in raw_lines
            if line.strip()
        ]

        local_box_count = 0

        for line_number, line in enumerate(
            lines,
            start=1,
        ):

            parts = line.split()

            require(
                len(
                    parts
                ) == 5,
                "Unexpected YOLO annotation format.\n"
                f"File: {label_path}\n"
                f"Line {line_number}: {line}",
            )

            try:

                class_float = float(
                    parts[
                        0
                    ]
                )

                xc = float(
                    parts[
                        1
                    ]
                )

                yc = float(
                    parts[
                        2
                    ]
                )

                bw = float(
                    parts[
                        3
                    ]
                )

                bh = float(
                    parts[
                        4
                    ]
                )

            except Exception as exc:

                raise RuntimeError(
                    "Non-numeric YOLO annotation.\n"
                    f"File: {label_path}\n"
                    f"Line {line_number}: {line}"
                ) from exc

            numeric_values = [
                class_float,
                xc,
                yc,
                bw,
                bh,
            ]

            require(
                all(
                    math.isfinite(
                        value
                    )
                    for value
                    in numeric_values
                ),
                "Non-finite YOLO annotation value.\n"
                f"File: {label_path}\n"
                f"Line {line_number}: {line}",
            )

            require(
                class_float.is_integer(),
                "YOLO class ID is not integral.\n"
                f"File: {label_path}\n"
                f"Line {line_number}: {line}",
            )

            class_id = int(
                class_float
            )

            require(
                0.0 <= xc <= 1.0
                and 0.0 <= yc <= 1.0
                and 0.0 < bw <= 1.0
                and 0.0 < bh <= 1.0,
                "YOLO normalized coordinate outside valid range.\n"
                f"File: {label_path}\n"
                f"Line {line_number}: {line}",
            )

            x1 = (
                xc
                - bw / 2.0
            ) * width

            y1 = (
                yc
                - bh / 2.0
            ) * height

            x2 = (
                xc
                + bw / 2.0
            ) * width

            y2 = (
                yc
                + bh / 2.0
            ) * height

            epsilon = 1e-6

            require(
                x1 >= -epsilon
                and y1 >= -epsilon
                and x2 <= width + epsilon
                and y2 <= height + epsilon
                and x2 > x1
                and y2 > y1,
                "Converted YOLO box invalid.\n"
                f"File: {label_path}\n"
                f"Line {line_number}: {line}\n"
                f"Converted: {(x1, y1, x2, y2)}",
            )

            observed_class_ids[
                class_id
            ] += 1

            local_box_count += 1

        expected_box_count = as_int(
            row[
                "gt_box_count"
            ],
            "gt_box_count",
        )

        require(
            local_box_count
            == expected_box_count,
            "Primary manifest GT-box count differs from label content.\n"
            f"File:     {filename}\n"
            f"Manifest: {expected_box_count}\n"
            f"Actual:   {local_box_count}",
        )

        if local_box_count == 0:

            observed_zero_box_filenames.add(
                filename
            )

        observed_gt_boxes += (
            local_box_count
        )


        if (
            index % 25 == 0
            or index == 250
        ):

            print(
                f"  verified primary pair "
                f"{index}/250"
            )


    require(
        observed_gt_boxes
        == EXPECTED_GT_BOXES,
        "Observed total RSC GT boxes != 539.\n"
        f"Actual: {observed_gt_boxes}",
    )

    require(
        dict(
            observed_class_ids
        )
        == {
            0: 539
        },
        "Observed RSC class-ID distribution mismatch.\n"
        f"Actual: {dict(observed_class_ids)}",
    )

    require(
        observed_zero_box_filenames
        == EXPECTED_ZERO_BOX_FILENAMES,
        "Observed zero-box official-test set mismatch.\n"
        f"Expected: {sorted(EXPECTED_ZERO_BOX_FILENAMES)}\n"
        f"Actual:   {sorted(observed_zero_box_filenames)}",
    )

    require(
        dict(
            observed_shapes
        )
        == EXPECTED_SHAPE_DISTRIBUTION,
        "Observed image-shape distribution mismatch.\n"
        f"Expected: {EXPECTED_SHAPE_DISTRIBUTION}\n"
        f"Actual:   {dict(observed_shapes)}",
    )

    record(
        "All 250 primary image/label pairs",
        gt_boxes=observed_gt_boxes,
        class_ids=dict(
            observed_class_ids
        ),
        zero_box_filenames=sorted(
            observed_zero_box_filenames
        ),
        shape_distribution={
            str(
                key
            ):
                value
            for key, value
            in sorted(
                observed_shapes.items()
            )
        },
    )


    # ========================================================
    # 6O. PRIMARY DEDUP MEMBERSHIP FLAG
    # ========================================================

    primary_dedup_filenames = {
        str(
            row[
                "filename"
            ]
        ).strip()
        for row in primary_rows
        if parse_bool(
            row[
                "exact_dedup247_member"
            ]
        )
    }

    require(
        len(
            primary_dedup_filenames
        ) == 247,
        "Primary manifest exact_dedup247_member count != 247.\n"
        f"Actual: {len(primary_dedup_filenames)}",
    )

    primary_exclusions = (
        set(
            filenames
        )
        - primary_dedup_filenames
    )

    require(
        primary_exclusions
        == EXPECTED_DEDUP_EXCLUSIONS,
        "Primary-manifest dedup exclusion set mismatch.\n"
        f"Expected: {sorted(EXPECTED_DEDUP_EXCLUSIONS)}\n"
        f"Actual:   {sorted(primary_exclusions)}",
    )

    record(
        "Primary-manifest exact-dedup membership",
        members=247,
        exclusions=sorted(
            primary_exclusions
        ),
    )


    # ========================================================
    # 6P. EXACT-DEDUP 247 MANIFEST
    # ========================================================

    dedup_manifest_path = (
        PACKAGE_ROOT
        / "RSC_EXACT_DEDUP247_MANIFEST_v1.csv"
    )

    require_path(
        dedup_manifest_path,
        "Exact-dedup sensitivity manifest",
    )

    dedup_rows = read_csv_rows(
        dedup_manifest_path
    )

    require(
        len(
            dedup_rows
        ) == 247,
        "Enriched exact-dedup manifest row count != 247.\n"
        f"Actual: {len(dedup_rows)}",
    )

    dedup_filenames = [
        str(
            row[
                "filename"
            ]
        ).strip()
        for row in dedup_rows
    ]

    require(
        len(
            set(
                dedup_filenames
            )
        ) == 247,
        "Exact-dedup manifest filenames are not unique.",
    )

    dedup_filename_set = set(
        dedup_filenames
    )

    require(
        dedup_filename_set
        == primary_dedup_filenames,
        "Dedup-247 manifest set differs from primary membership flags.",
    )

    require(
        dedup_filename_set.issubset(
            set(
                filenames
            )
        ),
        "Dedup-247 manifest is not a subset of official test-250.",
    )

    dedup_exclusions = (
        set(
            filenames
        )
        - dedup_filename_set
    )

    require(
        dedup_exclusions
        == EXPECTED_DEDUP_EXCLUSIONS,
        "Exact-dedup manifest exclusion set mismatch.",
    )

    dedup_zero_box = (
        dedup_filename_set
        & observed_zero_box_filenames
    )

    require(
        dedup_zero_box
        == EXPECTED_DEDUP_ZERO_BOX_FILENAMES,
        "Dedup-247 zero-box membership mismatch.\n"
        f"Expected: {sorted(EXPECTED_DEDUP_ZERO_BOX_FILENAMES)}\n"
        f"Actual:   {sorted(dedup_zero_box)}",
    )

    dedup_gt_total = 0

    for row in dedup_rows:

        filename = str(
            row[
                "filename"
            ]
        ).strip()

        require(
            filename
            in primary_by_filename,
            "Dedup row absent from primary manifest:\n"
            f"{filename}",
        )

        primary = primary_by_filename[
            filename
        ]

        require(
            str(
                row[
                    "image_sha256"
                ]
            ).strip().lower()
            == str(
                primary[
                    "image_sha256"
                ]
            ).strip().lower(),
            "Dedup image SHA differs from primary manifest.\n"
            f"Filename: {filename}",
        )

        require(
            str(
                row[
                    "label_sha256"
                ]
            ).strip().lower()
            == str(
                primary[
                    "label_sha256"
                ]
            ).strip().lower(),
            "Dedup label SHA differs from primary manifest.\n"
            f"Filename: {filename}",
        )

        require(
            as_int(
                row[
                    "gt_box_count"
                ],
                "dedup gt_box_count",
            )
            == as_int(
                primary[
                    "gt_box_count"
                ],
                "primary gt_box_count",
            ),
            "Dedup GT-box count differs from primary manifest.\n"
            f"Filename: {filename}",
        )

        dedup_gt_total += as_int(
            row[
                "gt_box_count"
            ],
            "dedup gt_box_count",
        )

    # Three excluded rows are all zero-box rows, therefore
    # dedup-247 retains all 539 positive GT boxes.
    require(
        dedup_gt_total
        == EXPECTED_GT_BOXES,
        "Dedup-247 GT-box total != 539.\n"
        f"Actual: {dedup_gt_total}",
    )

    record(
        "Exact-dedup sensitivity manifest",
        rows=247,
        gt_boxes=dedup_gt_total,
        exclusions=sorted(
            dedup_exclusions
        ),
        zero_box_retained=sorted(
            dedup_zero_box
        ),
    )


    # ========================================================
    # 6Q. ACTUAL FROZEN PROVENANCE FILE SHAs
    # ========================================================

    provenance_root = (
        PACKAGE_ROOT
        / "provenance"
    )

    require_path(
        provenance_root,
        "RSC package provenance directory",
    )

    provenance_expectations = {
        "public_real_mp_v1_source_specific_manifest.csv":
            EXPECTED_SOURCE_SPECIFIC_SHA256,

        "public_real_mp_v1_image_manifest.csv":
            EXPECTED_IMAGE_MANIFEST_SHA256,

        "rsc_test_exact_dedup_v1.csv":
            EXPECTED_DEDUP_MANIFEST_SHA256,
    }

    provenance_observed = {}

    for filename, expected_sha in (
        provenance_expectations.items()
    ):

        path = (
            provenance_root
            / filename
        )

        require_path(
            path,
            "Frozen provenance manifest",
        )

        actual_sha = sha256_file(
            path
        )

        require(
            actual_sha
            == expected_sha,
            "Frozen provenance manifest SHA mismatch.\n"
            f"File:     {path}\n"
            f"Expected: {expected_sha}\n"
            f"Actual:   {actual_sha}",
        )

        provenance_observed[
            filename
        ] = actual_sha

    record(
        "Actual mounted frozen provenance manifest SHAs",
        manifests=provenance_observed,
    )


    # ========================================================
    # 6R. FINAL CROSS-CHECK AGAINST STATE SHAPE DISTRIBUTION
    # ========================================================

    state_shape_distribution = (
        payload_state.get(
            "image_shape_distribution",
            {},
        )
    )

    observed_shape_distribution_for_state = {
        str(
            shape
        ):
            count
        for shape, count
        in sorted(
            observed_shapes.items()
        )
    }

    require(
        state_shape_distribution
        == observed_shape_distribution_for_state,
        "State image-shape distribution differs from mounted images.\n"
        f"State:    {state_shape_distribution}\n"
        f"Observed: {observed_shape_distribution_for_state}",
    )

    record(
        "State vs mounted scientific-content cross-check"
    )


    # ========================================================
    # 6S. FINAL PASS STATE
    # ========================================================

    audit.update(
        {
            "status":
                "PASS",

            "dataset_root":
                str(
                    DATASET_ROOT
                ),

            "package_root":
                str(
                    PACKAGE_ROOT
                ),

            "package_project_root":
                str(
                    PACKAGE_PROJECT_ROOT
                ),

            "frozen_local_zip": {
                "filename":
                    EXPECTED_ZIP_FILENAME,

                "sha256":
                    EXPECTED_ZIP_SHA256,

                "sha_record_verified":
                    True,

                "original_zip_bytes_exposed_by_kaggle":
                    zip_bytes_reverified,

                "original_zip_bytes_rehashed_on_kaggle":
                    zip_bytes_reverified,
            },

            "primary_official_test": {
                "rows":
                    250,

                "images":
                    250,

                "labels":
                    250,

                "gt_boxes":
                    observed_gt_boxes,

                "class_ids":
                    dict(
                        observed_class_ids
                    ),

                "zero_box_label_count":
                    len(
                        observed_zero_box_filenames
                    ),

                "zero_box_filenames":
                    sorted(
                        observed_zero_box_filenames
                    ),
            },

            "exact_dedup_sensitivity": {
                "rows":
                    247,

                "gt_boxes":
                    dedup_gt_total,

                "excluded_official_test_filenames":
                    sorted(
                        dedup_exclusions
                    ),

                "zero_box_label_count":
                    len(
                        dedup_zero_box
                    ),

                "zero_box_filenames":
                    sorted(
                        dedup_zero_box
                    ),
            },

            "scientific_payload": {
                "images":
                    len(
                        payload_images
                    ),

                "labels":
                    len(
                        payload_txt
                    ),

                "files":
                    len(
                        payload_files
                    ),

                "modified":
                    False,
            },

            "policy": {
                "training":
                    False,

                "model_inference":
                    False,

                "threshold_tuning":
                    False,

                "calibration":
                    False,

                "checkpoint_selection":
                    False,

                "seed_selection":
                    False,
            },

            "readiness":
                "READY_TO_FREEZE_RSC_EXTERNAL_EVALUATION_PROTOCOL",

            "checks":
                checks,
        }
    )

    write_json(
        AUDIT_JSON,
        audit,
    )

    audit_sha = sha256_file(
        AUDIT_JSON
    )

    AUDIT_SHA_RECORD.write_text(
        f"{audit_sha}  {AUDIT_JSON.name}\n",
        encoding="utf-8",
    )


    # ========================================================
    # 7. FINAL REPORT
    # ========================================================

    print()
    print(
        "=" * 76
    )

    print(
        "DASO-NET RSC KAGGLE PRE-INFERENCE ASSET AUDIT: PASS"
    )

    print(
        "=" * 76
    )

    print()

    print(
        f"Dataset root:\n  {DATASET_ROOT}"
    )

    print()

    print(
        f"Internal package root:\n  {PACKAGE_ROOT}"
    )

    print()

    print(
        "Frozen local ZIP SHA256:"
    )

    print(
        f"  {EXPECTED_ZIP_SHA256}"
    )

    print()

    if zip_bytes_reverified:

        print(
            "Original ZIP bytes exposed by Kaggle: YES"
        )

        print(
            "Original ZIP SHA reverified on Kaggle: YES"
        )

    else:

        print(
            "Original ZIP bytes exposed by Kaggle: NO"
        )

        print(
            "Kaggle mounted auto-extracted contents."
        )

        print(
            "Frozen SHA record: PASS"
        )

        print(
            "Mounted file-by-file SHA verification: PASS"
        )

    print()

    print(
        "PRIMARY — official authors' test:"
    )

    print(
        "  images:          250"
    )

    print(
        "  labels:          250"
    )

    print(
        f"  GT boxes:        {observed_gt_boxes}"
    )

    print(
        f"  class IDs:       {dict(observed_class_ids)}"
    )

    print(
        "  zero-box labels: 4"
    )

    for filename in sorted(
        observed_zero_box_filenames
    ):

        print(
            f"    {filename}"
        )

    print()

    print(
        "SENSITIVITY — exact-dedup:"
    )

    print(
        "  rows:            247"
    )

    print(
        f"  GT boxes:        {dedup_gt_total}"
    )

    print(
        "  exclusions:"
    )

    for filename in sorted(
        dedup_exclusions
    ):

        print(
            f"    {filename}"
        )

    print(
        "  zero-box retained:"
    )

    for filename in sorted(
        dedup_zero_box
    ):

        print(
            f"    {filename}"
        )

    print()

    print(
        "Training performed:         NO"
    )

    print(
        "Model inference performed:  NO"
    )

    print(
        "Threshold tuning performed: NO"
    )

    print(
        "Calibration performed:      NO"
    )

    print(
        "Checkpoint selection:       NO"
    )

    print(
        "Seed selection:             NO"
    )

    print()

    print(
        "Audit JSON:"
    )

    print(
        f"  {AUDIT_JSON}"
    )

    print()

    print(
        "Audit JSON SHA256:"
    )

    print(
        f"  {audit_sha}"
    )

    print()

    print(
        "Audit SHA record:"
    )

    print(
        f"  {AUDIT_SHA_RECORD}"
    )

    print()

    print(
        "READINESS:"
    )

    print(
        "  READY_TO_FREEZE_RSC_EXTERNAL_EVALUATION_PROTOCOL"
    )

    print()

    print(
        "NO RSC INFERENCE HAS BEEN PERFORMED."
    )

    print(
        "=" * 76
    )


except Exception as exc:

    audit.update(
        {
            "status":
                "FAIL",

            "error_type":
                type(
                    exc
                ).__name__,

            "error":
                str(
                    exc
                ),

            "traceback":
                traceback.format_exc(),

            "readiness":
                "NOT_READY_FOR_RSC_INFERENCE",

            "checks":
                checks,
        }
    )

    write_json(
        AUDIT_JSON,
        audit,
    )

    fail_audit_sha = sha256_file(
        AUDIT_JSON
    )

    AUDIT_SHA_RECORD.write_text(
        f"{fail_audit_sha}  {AUDIT_JSON.name}\n",
        encoding="utf-8",
    )

    print()

    print(
        "=" * 76
    )

    print(
        "DASO-NET RSC KAGGLE PRE-INFERENCE ASSET AUDIT: FAIL"
    )

    print(
        "=" * 76
    )

    print()

    print(
        "Failure preserved at:"
    )

    print(
        f"  {AUDIT_JSON}"
    )

    print()

    print(
        "No RSC inference was performed."
    )

    print(
        "Do not delete or overwrite this failed audit."
    )

    print()

    raise

DASO-Net — RSC KAGGLE PRE-INFERENCE ASSET AUDIT v1

Scientific action:
  READ-ONLY external-test asset verification
  NO training / inference / tuning / calibration / selection

PASS  Attached private RSC dataset root
PASS  Frozen local ZIP SHA record
INFO  Kaggle did not expose the original ZIP bytes.
INFO  Kaggle mounted the extracted ZIP contents instead.
INFO  ZIP will NOT be reconstructed or re-zipped.
PASS  Kaggle ZIP handling
PASS  Unique internal RSC package root
PASS  Top-level/internal state JSON byte identity
PASS  Internal RSC package state contract
PASS  No-training/no-tuning/no-selection package policy
PASS  Frozen provenance SHA identities in state JSON
  verified package file hashes 50/506
  verified package file hashes 100/506
  verified package file hashes 150/506
  verified package file hashes 200/506
  verified package file hashes 250/506
  verified package file hashes 300/506
  verified package file hashes 350/506
  verified package file hashes 400/506
  verified p

In [24]:
# ============================================================
# DASO-Net — RSC EXTERNAL DETECTION EVALUATION
# PRE-INFERENCE PROTOCOL FREEZE v1
#
# IMPORTANT
# ---------
# THIS CELL PERFORMS NO MODEL INFERENCE.
#
# It freezes the RSC external-evaluation protocol BEFORE
# predictions from any final model are observed.
#
# PRIMARY:
#   official authors' RSC test split = 250 images
#
# SENSITIVITY:
#   exact-deduplicated subset = 247 images
#
# TASK:
#   class-agnostic bounding-box detection ONLY
#
# NO segmentation-mask external claims.
#
# NO RSC:
#   training
#   CRPA
#   calibration
#   threshold tuning
#   checkpoint selection
#   seed selection
#
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import math
import traceback
from pathlib import Path


# ============================================================
# 1. FROZEN IDENTITIES
# ============================================================

RSC_DATASET_ROOT = Path(
    "/kaggle/input/datasets/abinavprabhuv/"
    "dasonet-rsc-external-test-assets-v1"
)

RSC_PACKAGE_ROOT = (
    RSC_DATASET_ROOT
    / "DASO-Net-RSC-External-TestAssets-v1"
    / "DASO-Net-RSC-External-TestAssets-v1"
)

RSC_PROJECT_ROOT = (
    RSC_PACKAGE_ROOT
    / "DASO-Net-Phase1A"
)

RSC_PRIMARY_MANIFEST = (
    RSC_PACKAGE_ROOT
    / "RSC_OFFICIAL_TEST250_MANIFEST_v1.csv"
)

RSC_DEDUP_MANIFEST = (
    RSC_PACKAGE_ROOT
    / "RSC_EXACT_DEDUP247_MANIFEST_v1.csv"
)

RSC_STATE_JSON = (
    RSC_PACKAGE_ROOT
    / "RSC_EXTERNAL_TEST_ASSETS_STATE_v1.json"
)

RSC_FILE_MANIFEST = (
    RSC_PACKAGE_ROOT
    / "RSC_EXTERNAL_TEST_FILE_MANIFEST_v1.json"
)

RSC_PREINFERENCE_AUDIT = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_external_preinference_asset_audit_v1/"
    "RSC_EXTERNAL_PREINFERENCE_ASSET_AUDIT_v1.json"
)

EXPECTED_RSC_PREINFERENCE_AUDIT_SHA256 = (
    "c8eb566baa55e1c93906e61dbe4d9228"
    "246c7d130aac489b07102c51c610132b"
)

EXPECTED_RSC_PACKAGE_ZIP_SHA256 = (
    "71fba0d782f5588e3c1ad19827436f8"
    "49cb58f8485c9c5d08bb1f4321647ae72"
)

EXPECTED_RSC_SOURCE_ID = (
    "rsc2025_mobile_consumer"
)

EXPECTED_PRIMARY_ROWS = 250
EXPECTED_DEDUP_ROWS = 247
EXPECTED_GT_BOXES = 539

EXPECTED_ZERO_BOX_FILENAMES = {
    "test_113.jpg",
    "test_182.jpg",
    "test_183.jpg",
    "test_184.jpg",
}

EXPECTED_DEDUP_EXCLUSIONS = {
    "test_182.jpg",
    "test_183.jpg",
    "test_184.jpg",
}

EXPECTED_DEDUP_ZERO_BOX = {
    "test_113.jpg",
}


# ============================================================
# 2. FINAL MODEL IDENTITIES — FROZEN
# ============================================================

FINAL_MODELS = [
    {
        "seed_index": 1,
        "seed": 1782656963,
        "checkpoint": Path(
            "/kaggle/working/dasonet_phase5b/"
            "dasonet_final_seed01_s1782656963/"
            "snapshots/epoch_011_opt_002750.pt"
        ),
        "checkpoint_sha256":
            "e4c4b79ad4b8c6e18ba2b1e3d6cbe796"
            "c9cc301bd6fcc7c596fc290ad3a7e81d",
        "successful_optimizer_updates": 2750,
        "operating_threshold": 0.30,
    },
    {
        "seed_index": 2,
        "seed": 901395089,
        "checkpoint": Path(
            "/kaggle/working/dasonet_phase5b/"
            "dasonet_final_seed02_s901395089/"
            "snapshots/epoch_010_opt_002750.pt"
        ),
        "checkpoint_sha256":
            "19cdb834a46240916b2c8023e8c3afe57"
            "38d9c7cd5d8f818c62ad1e289df1ad2",
        "successful_optimizer_updates": 2750,
        "operating_threshold": 0.35,
    },
    {
        "seed_index": 3,
        "seed": 252916394,
        "checkpoint": Path(
            "/kaggle/working/dasonet_phase5b/"
            "dasonet_final_seed03_s252916394/"
            "snapshots/epoch_010_opt_002750.pt"
        ),
        "checkpoint_sha256":
            "be7b7941f2f62179e15074aa973bf6da0"
            "eb9018ec9133486512bd1f87153f447",
        "successful_optimizer_updates": 2750,
        "operating_threshold": 0.35,
    },
]


# ============================================================
# 3. VALIDATED MODEL LOADER / DECODER
# ============================================================

EVALUATOR_SCRIPT = Path(
    "/kaggle/working/DASO-Net-Phase5B-KaggleTraining/"
    "scripts/evaluate_phase5c.py"
)

VALIDATED_DECODER_REFERENCE_SHA256 = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)

DECODER_SPEC = {
    "source":
        "validated Phase5C decoder",

    "function":
        "decode_instances",

    "stride":
        4,

    "decode_min_score":
        0.05,

    "topk":
        100,

    "nms_iou":
        0.50,

    "mask_threshold":
        0.50,

    "local_peak_kernel":
        3,

    "validated_decoder_reference_sha256":
        VALIDATED_DECODER_REFERENCE_SHA256,

    "note":
        (
            "Mask reconstruction may occur internally in the validated "
            "decoder, but RSC scientific evaluation uses only decoded "
            "bounding boxes and scores."
        ),
}


# ============================================================
# 4. OUTPUT CONTRACT
# ============================================================

PROTOCOL_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_external_evaluation_protocol_freeze_v1"
)

PROTOCOL_JSON = (
    PROTOCOL_DIR
    / "RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

PROTOCOL_SHA_RECORD = (
    PROTOCOL_DIR
    / "RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.sha256.txt"
)

DESIGNATED_EVAL_ROOT = Path(
    "/kaggle/working/dasonet_final_evaluation/"
    "rsc_external_detection_v1"
)


# ============================================================
# 5. HELPERS
# ============================================================

checks = []


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open("rb") as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(block)

    return h.hexdigest()


def require(
    condition: bool,
    message: str,
) -> None:

    if not condition:
        raise RuntimeError(message)


def require_file(
    path: Path,
    label: str,
) -> None:

    require(
        path.is_file(),
        f"{label} missing:\n{path}",
    )


def require_dir(
    path: Path,
    label: str,
) -> None:

    require(
        path.is_dir(),
        f"{label} missing:\n{path}",
    )


def read_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path: Path,
    obj,
) -> None:

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
        )
        + "\n",
        encoding="utf-8",
    )


def read_csv_rows(
    path: Path,
):

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:

        return list(
            csv.DictReader(f)
        )


def record(
    name: str,
    **details,
) -> None:

    item = {
        "check": name,
        "status": "PASS",
    }

    item.update(details)

    checks.append(item)

    print(
        f"PASS  {name}"
    )


# ============================================================
# 6. BANNER
# ============================================================

print("=" * 76)

print(
    "DASO-Net — RSC EXTERNAL EVALUATION "
    "PRE-INFERENCE PROTOCOL FREEZE v1"
)

print("=" * 76)

print()

print(
    "THIS CELL PERFORMS NO MODEL INFERENCE."
)

print()

print(
    "Purpose:"
)

print(
    "  freeze the complete RSC detection-evaluation protocol"
)

print(
    "  before predictions from any final seed are observed"
)

print()


# ============================================================
# 7. NEVER OVERWRITE A PRIOR FREEZE
# ============================================================

if PROTOCOL_DIR.exists():

    raise RuntimeError(
        "Protocol-freeze directory already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{PROTOCOL_DIR}"
    )


if DESIGNATED_EVAL_ROOT.exists():

    raise RuntimeError(
        "Designated RSC evaluation output root already exists BEFORE "
        "this protocol freeze:\n"
        f"{DESIGNATED_EVAL_ROOT}\n\n"
        "Do not continue automatically. Preserve the existing output "
        "and determine whether any RSC inference has already occurred."
    )


PROTOCOL_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 8. VERIFY PRE-INFERENCE ASSET AUDIT
# ============================================================

require_file(
    RSC_PREINFERENCE_AUDIT,
    "RSC pre-inference asset audit",
)

audit_sha = sha256_file(
    RSC_PREINFERENCE_AUDIT
)

require(
    audit_sha
    == EXPECTED_RSC_PREINFERENCE_AUDIT_SHA256,
    "RSC pre-inference audit SHA mismatch.\n"
    f"Expected: {EXPECTED_RSC_PREINFERENCE_AUDIT_SHA256}\n"
    f"Actual:   {audit_sha}",
)

audit = read_json(
    RSC_PREINFERENCE_AUDIT
)

require(
    audit.get("status")
    == "PASS",
    "RSC pre-inference audit status is not PASS.",
)

require(
    audit.get("readiness")
    == "READY_TO_FREEZE_RSC_EXTERNAL_EVALUATION_PROTOCOL",
    "RSC pre-inference audit readiness is not correct.",
)

require(
    audit.get("model_inference_performed")
    is False,
    "Pre-inference audit says model inference was performed.",
)

require(
    audit.get("threshold_tuning_performed")
    is False,
    "Pre-inference audit says threshold tuning was performed.",
)

require(
    audit.get("calibration_performed")
    is False,
    "Pre-inference audit says calibration was performed.",
)

require(
    audit.get("checkpoint_selection_performed")
    is False,
    "Pre-inference audit says checkpoint selection was performed.",
)

require(
    audit.get("seed_selection_performed")
    is False,
    "Pre-inference audit says seed selection was performed.",
)

record(
    "RSC pre-inference asset audit identity",
    sha256=audit_sha,
)


# ============================================================
# 9. VERIFY RSC PACKAGE
# ============================================================

require_dir(
    RSC_DATASET_ROOT,
    "RSC Kaggle dataset root",
)

require_dir(
    RSC_PACKAGE_ROOT,
    "RSC package root",
)

require_dir(
    RSC_PROJECT_ROOT,
    "RSC scientific payload root",
)

require_file(
    RSC_PRIMARY_MANIFEST,
    "RSC primary test-250 manifest",
)

require_file(
    RSC_DEDUP_MANIFEST,
    "RSC dedup-247 manifest",
)

require_file(
    RSC_STATE_JSON,
    "RSC package state JSON",
)

require_file(
    RSC_FILE_MANIFEST,
    "RSC package file manifest",
)


state = read_json(
    RSC_STATE_JSON
)

require(
    state.get("status") == "PASS",
    "RSC package state != PASS.",
)

require(
    state.get("rsc_source_id")
    == EXPECTED_RSC_SOURCE_ID,
    "Unexpected RSC source ID.",
)


primary_rows = read_csv_rows(
    RSC_PRIMARY_MANIFEST
)

dedup_rows = read_csv_rows(
    RSC_DEDUP_MANIFEST
)

require(
    len(primary_rows)
    == EXPECTED_PRIMARY_ROWS,
    "Primary RSC manifest row count != 250.",
)

require(
    len(dedup_rows)
    == EXPECTED_DEDUP_ROWS,
    "Dedup RSC manifest row count != 247.",
)


primary_filenames = {
    str(
        row["filename"]
    ).strip()
    for row in primary_rows
}

dedup_filenames = {
    str(
        row["filename"]
    ).strip()
    for row in dedup_rows
}

require(
    len(primary_filenames) == 250,
    "Primary filename set is not 250 unique images.",
)

require(
    len(dedup_filenames) == 247,
    "Dedup filename set is not 247 unique images.",
)

require(
    dedup_filenames.issubset(
        primary_filenames
    ),
    "Dedup-247 is not a subset of official test-250.",
)


dedup_exclusions = (
    primary_filenames
    - dedup_filenames
)

require(
    dedup_exclusions
    == EXPECTED_DEDUP_EXCLUSIONS,
    "Unexpected dedup exclusion set.\n"
    f"Actual: {sorted(dedup_exclusions)}",
)


observed_zero_box = {
    str(
        row["filename"]
    ).strip()
    for row in primary_rows
    if int(
        float(
            row["gt_box_count"]
        )
    ) == 0
}

require(
    observed_zero_box
    == EXPECTED_ZERO_BOX_FILENAMES,
    "Unexpected official-test zero-box set.",
)


dedup_zero_box = (
    dedup_filenames
    & observed_zero_box
)

require(
    dedup_zero_box
    == EXPECTED_DEDUP_ZERO_BOX,
    "Unexpected dedup zero-box set.",
)


primary_gt_total = sum(
    int(
        float(
            row["gt_box_count"]
        )
    )
    for row in primary_rows
)

dedup_gt_total = sum(
    int(
        float(
            row["gt_box_count"]
        )
    )
    for row in dedup_rows
)

require(
    primary_gt_total
    == EXPECTED_GT_BOXES,
    "Primary GT-box total != 539.",
)

require(
    dedup_gt_total
    == EXPECTED_GT_BOXES,
    "Dedup GT-box total != 539.",
)


rsc_manifest_hashes = {
    "primary_test250_manifest_sha256":
        sha256_file(
            RSC_PRIMARY_MANIFEST
        ),

    "dedup247_manifest_sha256":
        sha256_file(
            RSC_DEDUP_MANIFEST
        ),

    "package_state_sha256":
        sha256_file(
            RSC_STATE_JSON
        ),

    "package_file_manifest_sha256":
        sha256_file(
            RSC_FILE_MANIFEST
        ),
}


record(
    "RSC primary/sensitivity scientific sets",
    primary_rows=250,
    dedup_rows=247,
    primary_gt_boxes=539,
    dedup_gt_boxes=539,
    zero_box_primary=sorted(
        observed_zero_box
    ),
    zero_box_dedup=sorted(
        dedup_zero_box
    ),
)


# ============================================================
# 10. VERIFY FINAL CHECKPOINTS
# ============================================================

verified_models = []


for model_info in FINAL_MODELS:

    checkpoint = model_info[
        "checkpoint"
    ]

    require_file(
        checkpoint,
        (
            "Final Seed-"
            f"{model_info['seed_index']} checkpoint"
        ),
    )

    actual_sha = sha256_file(
        checkpoint
    )

    expected_sha = model_info[
        "checkpoint_sha256"
    ]

    require(
        actual_sha == expected_sha,
        (
            "Final checkpoint SHA mismatch.\n"
            f"Seed index: {model_info['seed_index']}\n"
            f"Seed:       {model_info['seed']}\n"
            f"Checkpoint: {checkpoint}\n"
            f"Expected:   {expected_sha}\n"
            f"Actual:     {actual_sha}"
        ),
    )

    require(
        model_info[
            "successful_optimizer_updates"
        ] == 2750,
        "Final model successful-update count != 2750.",
    )

    verified_models.append(
        {
            "seed_index":
                model_info[
                    "seed_index"
                ],

            "seed":
                model_info[
                    "seed"
                ],

            "checkpoint":
                str(
                    checkpoint
                ),

            "checkpoint_sha256":
                actual_sha,

            "successful_optimizer_updates":
                2750,

            "operating_threshold":
                model_info[
                    "operating_threshold"
                ],
        }
    )

    record(
        (
            "Final Seed-"
            f"{model_info['seed_index']} checkpoint"
        ),
        seed=model_info["seed"],
        sha256=actual_sha,
        operating_threshold=(
            model_info[
                "operating_threshold"
            ]
        ),
    )


# ============================================================
# 11. VERIFY VALIDATED EVALUATOR SCRIPT EXISTS
#
# We record its whole-file SHA now.
#
# The prior validated-decoder reference SHA is also frozen in
# the protocol but is NOT assumed here to equal the whole-file
# evaluator SHA unless separately established.
# ============================================================

require_file(
    EVALUATOR_SCRIPT,
    "Validated Phase5C evaluator script",
)

evaluator_script_sha = sha256_file(
    EVALUATOR_SCRIPT
)

record(
    "Validated Phase5C evaluator script present",
    path=str(
        EVALUATOR_SCRIPT
    ),
    whole_file_sha256=evaluator_script_sha,
    validated_decoder_reference_sha256=(
        VALIDATED_DECODER_REFERENCE_SHA256
    ),
)


# ============================================================
# 12. FREEZE INPUT PREPROCESSING
#
# RSC images are approximately 960 x 885-889 and therefore
# already near the model's native training scale.
#
# To avoid aspect-ratio distortion and avoid inventing a resize
# protocol:
#
#   - use the full original RSC frame
#   - NO resize
#   - NO crop
#   - NO tiling
#   - NO TTA
#   - RGB float32 / 255
#   - CHW
#   - pad ONLY right/bottom to nearest multiple of 16
#   - constant padding value = 0.0
#
# Padding is minimal and exists only for stride compatibility.
#
# Evaluation remains in ORIGINAL-image pixel coordinates.
# ============================================================

INPUT_PROTOCOL = {
    "scientific_unit":
        "one complete original RSC test image",

    "full_image_inference":
        True,

    "resize":
        False,

    "crop":
        False,

    "tiling":
        False,

    "test_time_augmentation":
        False,

    "input_color_order":
        "RGB",

    "source_decode":
        (
            "read original image bytes; convert decoded "
            "BGR/OpenCV representation to RGB"
        ),

    "tensor_dtype":
        "float32",

    "tensor_scaling":
        "pixel_value / 255.0",

    "tensor_layout":
        "CHW",

    "mean_normalization":
        None,

    "std_normalization":
        None,

    "required_spatial_multiple":
        16,

    "padding": {
        "enabled":
            True,

        "placement":
            "right_and_bottom_only",

        "target":
            (
                "smallest height and width >= original dimensions "
                "that are each divisible by 16"
            ),

        "mode":
            "constant",

        "constant_value":
            0.0,

        "maximum_padding_per_dimension":
            15,
    },

    "prediction_coordinate_policy": {
        "decoder_coordinate_system":
            "padded-input pixel coordinates",

        "scientific_evaluation_coordinate_system":
            "original-image pixel coordinates",

        "prediction_center_filter":
            (
                "discard decoded predictions whose box center lies "
                "outside the original image extent"
            ),

        "box_clipping":
            (
                "clip retained predicted boxes to original image "
                "bounds"
            ),

        "degenerate_after_clipping":
            "discard",

        "rescaling":
            False,
    },

    "ground_truth_coordinate_policy": {
        "source_format":
            "YOLO normalized class xc yc width height",

        "conversion":
            (
                "convert directly to floating-point xyxy using the "
                "original image width and height"
            ),

        "integer_rounding":
            False,

        "class_handling":
            "class-agnostic",
    },
}


# ============================================================
# 13. FREEZE BOX-METRIC PROTOCOL
# ============================================================

BOX_IOU_THRESHOLDS = [
    round(
        0.50 + 0.05 * i,
        2,
    )
    for i in range(10)
]

require(
    BOX_IOU_THRESHOLDS
    == [
        0.50,
        0.55,
        0.60,
        0.65,
        0.70,
        0.75,
        0.80,
        0.85,
        0.90,
        0.95,
    ],
    "Internal IoU-threshold construction failure.",
)


METRIC_PROTOCOL = {
    "task":
        "class-agnostic bounding-box detection",

    "segmentation_metrics":
        False,

    "primary_scientific_metrics": [
        "Box AP50",
        "Box AP75",
        "Box AP50:95",
    ],

    "ap_definition": {
        "prediction_pool":
            (
                "all decoded predictions surviving the frozen "
                "decoder floor and NMS"
            ),

        "decoder_score_floor":
            0.05,

        "ranking":
            "descending prediction score",

        "deterministic_tie_break":
            (
                "image official_test_index ascending, then decoder "
                "prediction order ascending"
            ),

        "matching":
            (
                "score-ordered one-to-one greedy matching within "
                "each image to the unmatched GT box having maximum IoU"
            ),

        "tp_condition":
            (
                "best unmatched GT IoU >= evaluated IoU threshold"
            ),

        "fp_condition":
            (
                "no unmatched GT in the same image reaches the "
                "evaluated IoU threshold"
            ),

        "zero_gt_images":
            (
                "every retained prediction is a false positive"
            ),

        "interpolation":
            "101-point interpolated precision",

        "recall_grid":
            "0.00 to 1.00 inclusive in increments of 0.01",

        "iou_thresholds":
            BOX_IOU_THRESHOLDS,

        "AP50":
            0.50,

        "AP75":
            0.75,

        "AP50_95":
            (
                "arithmetic mean of AP at IoU thresholds "
                "0.50:0.05:0.95"
            ),
    },

    "frozen_operating_metrics": {
        "matching_iou":
            0.50,

        "seed_specific_score_thresholds": {
            "seed_1":
                0.30,

            "seed_2":
                0.35,

            "seed_3":
                0.35,
        },

        "metrics": [
            "precision",
            "recall",
            "F1",
            "TP",
            "FP",
            "FN",
            "mean matched box IoU conditional on TP",
            "count MAE",
            "count bias",
            "zero-GT FP/image",
            "zero-GT fraction with >=1 FP",
        ],

        "count_definition":
            (
                "number of decoded predictions retained at the "
                "seed-specific frozen operating threshold"
            ),

        "count_bias_definition":
            "predicted count minus GT count",

        "zero_gt_note":
            (
                "Primary test has only four zero-GT images; "
                "zero-GT FP metrics are descriptive and must be "
                "reported with n=4."
            ),
    },

    "per_image_output_required":
        True,

    "per_image_required_fields": [
        "official_test_index",
        "filename",
        "subset_primary",
        "subset_dedup247",
        "gt_count",
        "pred_count_operating",
        "tp_operating_iou50",
        "fp_operating_iou50",
        "fn_operating_iou50",
        "count_error",
        "count_abs_error",
        "max_decoded_score",
    ],
}


# ============================================================
# 14. FREEZE SCIENTIFIC SET ROLES
# ============================================================

SET_PROTOCOL = {
    "primary": {
        "name":
            "RSC official authors' test split",

        "rows":
            250,

        "gt_boxes":
            539,

        "zero_gt_images":
            4,

        "role":
            "PRIMARY real-domain external detection evaluation",

        "manifest":
            str(
                RSC_PRIMARY_MANIFEST
            ),

        "manifest_sha256":
            rsc_manifest_hashes[
                "primary_test250_manifest_sha256"
            ],
    },

    "sensitivity": {
        "name":
            "RSC exact-deduplicated test subset",

        "rows":
            247,

        "gt_boxes":
            539,

        "zero_gt_images":
            1,

        "role":
            "SENSITIVITY analysis only",

        "manifest":
            str(
                RSC_DEDUP_MANIFEST
            ),

        "manifest_sha256":
            rsc_manifest_hashes[
                "dedup247_manifest_sha256"
            ],

        "excluded_primary_filenames":
            sorted(
                EXPECTED_DEDUP_EXCLUSIONS
            ),
    },

    "reporting_order": [
        "PRIMARY official authors' test-250",
        "SENSITIVITY exact-dedup-247",
    ],

    "prohibition":
        (
            "The exact-dedup 247 result must not replace the "
            "official test-250 result as the primary external result."
        ),
}


# ============================================================
# 15. FREEZE NO-TUNING / NO-SELECTION RULES
# ============================================================

SCIENTIFIC_POLICY = {
    "rsc_training_use":
        False,

    "rsc_crpa_use":
        False,

    "rsc_calibration_use":
        False,

    "rsc_threshold_tuning":
        False,

    "rsc_checkpoint_selection":
        False,

    "rsc_seed_selection":
        False,

    "best_seed_selection":
        False,

    "seed_exclusion_based_on_rsc":
        False,

    "checkpoint_exclusion_based_on_rsc":
        False,

    "decoder_change_after_rsc_results":
        False,

    "input_preprocessing_change_after_rsc_results":
        False,

    "metric_definition_change_after_rsc_results":
        False,

    "primary_subset_change_after_rsc_results":
        False,

    "posthoc_operating_threshold_replacement":
        False,

    "all_three_final_seeds_required":
        True,

    "aggregate_reporting":
        "mean ± sample SD across all three final seeds",

    "operating_threshold_source":
        (
            "synthetic validation only; frozen before formal "
            "synthetic test and before RSC external inference"
        ),

    "rsc_threshold_sweep":
        (
            "not part of the formal external protocol; if any "
            "later exploratory score-threshold sensitivity is "
            "computed, it must be explicitly post-hoc/descriptive "
            "and must never replace the frozen operating thresholds"
        ),

    "phash_policy":
        (
            "review/report only; no automatic sample removal"
        ),
}


# ============================================================
# 16. FREEZE INFERENCE EXECUTION POLICY
# ============================================================

INFERENCE_PROTOCOL = {
    "model_mode":
        "eval",

    "gradient_tracking":
        False,

    "inference_dtype":
        "float32",

    "mixed_precision":
        False,

    "batching":
        (
            "batch size 1 per image is the reference execution; "
            "larger batching is permitted only if numerically "
            "byte-equivalent prediction ordering is preserved"
        ),

    "device":
        "CUDA when available",

    "random_augmentation":
        False,

    "test_time_augmentation":
        False,

    "stochastic_model_layers":
        "disabled via model.eval()",

    "seed_order":
        [1, 2, 3],

    "image_order":
        (
            "ascending official_test_index from the frozen "
            "primary manifest"
        ),

    "resume_policy":
        (
            "seed-level or image-level resume is permitted only "
            "from preserved deterministic outputs; completed "
            "results must never be silently overwritten"
        ),
}


# ============================================================
# 17. BUILD THE COMPLETE PROTOCOL FREEZE
# ============================================================

protocol = {
    "protocol_name":
        "DASO-Net RSC External Detection Evaluation Protocol v1",

    "status":
        "FROZEN_BEFORE_RSC_INFERENCE",

    "scientific_scope":
        (
            "Real-domain external class-agnostic bounding-box "
            "detection evaluation on RSC 2025 mobile/consumer "
            "microscopy."
        ),

    "external_claim_limit":
        (
            "RSC supports bounding-box detection evaluation only. "
            "No RSC segmentation-mask performance claim."
        ),

    "pre_inference_proof": {
        "asset_audit_path":
            str(
                RSC_PREINFERENCE_AUDIT
            ),

        "asset_audit_sha256":
            audit_sha,

        "asset_audit_status":
            "PASS",

        "model_inference_performed_before_freeze":
            False,

        "threshold_tuning_performed_before_freeze":
            False,

        "calibration_performed_before_freeze":
            False,

        "checkpoint_selection_before_freeze":
            False,

        "seed_selection_before_freeze":
            False,
    },

    "rsc_package": {
        "dataset_root":
            str(
                RSC_DATASET_ROOT
            ),

        "package_root":
            str(
                RSC_PACKAGE_ROOT
            ),

        "project_payload_root":
            str(
                RSC_PROJECT_ROOT
            ),

        "rsc_source_id":
            EXPECTED_RSC_SOURCE_ID,

        "original_local_package_zip_sha256":
            EXPECTED_RSC_PACKAGE_ZIP_SHA256,

        **rsc_manifest_hashes,
    },

    "scientific_sets":
        SET_PROTOCOL,

    "models":
        verified_models,

    "validated_evaluator": {
        "script":
            str(
                EVALUATOR_SCRIPT
            ),

        "whole_file_sha256":
            evaluator_script_sha,

        "decoder":
            DECODER_SPEC,
    },

    "input_protocol":
        INPUT_PROTOCOL,

    "metric_protocol":
        METRIC_PROTOCOL,

    "inference_protocol":
        INFERENCE_PROTOCOL,

    "scientific_policy":
        SCIENTIFIC_POLICY,

    "formal_reporting": {
        "per_seed_required":
            True,

        "three_seed_mean_required":
            True,

        "three_seed_sample_sd_required":
            True,

        "primary_before_sensitivity":
            True,

        "primary_metrics": [
            "Box AP50",
            "Box AP75",
            "Box AP50:95",
        ],

        "operating_metrics_use_frozen_thresholds":
            True,

        "conditional_metric_wording":
            (
                "Any mean matched box IoU must be explicitly "
                "described as conditional on successful IoU>=0.50 "
                "matches."
            ),
    },

    "designated_future_output_root":
        str(
            DESIGNATED_EVAL_ROOT
        ),

    "checks":
        checks,
}


# ============================================================
# 18. WRITE FREEZE
# ============================================================

write_json(
    PROTOCOL_JSON,
    protocol,
)

protocol_sha = sha256_file(
    PROTOCOL_JSON
)

PROTOCOL_SHA_RECORD.write_text(
    (
        f"{protocol_sha}  "
        f"{PROTOCOL_JSON.name}\n"
    ),
    encoding="utf-8",
)


# ============================================================
# 19. READ-BACK VERIFICATION
# ============================================================

protocol_readback = read_json(
    PROTOCOL_JSON
)

require(
    protocol_readback.get("status")
    == "FROZEN_BEFORE_RSC_INFERENCE",
    "Protocol read-back status failure.",
)

require(
    protocol_readback[
        "scientific_sets"
    ][
        "primary"
    ][
        "rows"
    ] == 250,
    "Protocol read-back primary rows failure.",
)

require(
    protocol_readback[
        "scientific_sets"
    ][
        "sensitivity"
    ][
        "rows"
    ] == 247,
    "Protocol read-back sensitivity rows failure.",
)

require(
    [
        model[
            "operating_threshold"
        ]
        for model in protocol_readback[
            "models"
        ]
    ]
    == [
        0.30,
        0.35,
        0.35,
    ],
    "Protocol read-back operating thresholds failure.",
)

require(
    protocol_readback[
        "input_protocol"
    ][
        "resize"
    ] is False,
    "Protocol unexpectedly permits RSC resize.",
)

require(
    protocol_readback[
        "metric_protocol"
    ][
        "segmentation_metrics"
    ] is False,
    "Protocol unexpectedly enables RSC segmentation metrics.",
)

require(
    protocol_readback[
        "scientific_policy"
    ][
        "rsc_threshold_tuning"
    ] is False,
    "Protocol unexpectedly permits RSC threshold tuning.",
)

require(
    protocol_readback[
        "scientific_policy"
    ][
        "all_three_final_seeds_required"
    ] is True,
    "Protocol does not require all three final seeds.",
)

readback_sha = sha256_file(
    PROTOCOL_JSON
)

require(
    readback_sha == protocol_sha,
    "Protocol SHA changed during read-back.",
)

record(
    "Protocol JSON read-back identity",
    sha256=protocol_sha,
)


# Update the checks list once more after the final record.

protocol[
    "checks"
] = checks

write_json(
    PROTOCOL_JSON,
    protocol,
)

protocol_sha = sha256_file(
    PROTOCOL_JSON
)

PROTOCOL_SHA_RECORD.write_text(
    (
        f"{protocol_sha}  "
        f"{PROTOCOL_JSON.name}\n"
    ),
    encoding="utf-8",
)


# Final byte read-back.

require(
    sha256_file(
        PROTOCOL_JSON
    )
    == protocol_sha,
    "Final protocol byte SHA read-back failure.",
)


# ============================================================
# 20. FINAL REPORT
# ============================================================

print()

print("=" * 76)

print(
    "DASO-NET RSC EXTERNAL EVALUATION PROTOCOL: FROZEN"
)

print("=" * 76)

print()

print(
    "Status:"
)

print(
    "  FROZEN_BEFORE_RSC_INFERENCE"
)

print()

print(
    "PRIMARY:"
)

print(
    "  official authors' test images: 250"
)

print(
    "  GT boxes:                     539"
)

print(
    "  zero-GT images:                 4"
)

print()

print(
    "SENSITIVITY:"
)

print(
    "  exact-dedup images:            247"
)

print(
    "  GT boxes:                      539"
)

print(
    "  zero-GT images:                  1"
)

print()

print(
    "External task:"
)

print(
    "  class-agnostic BOUNDING-BOX detection only"
)

print()

print(
    "Primary metrics:"
)

print(
    "  Box AP50"
)

print(
    "  Box AP75"
)

print(
    "  Box AP50:95"
)

print()

print(
    "Frozen operating thresholds:"
)

for model in verified_models:

    print(
        "  "
        f"Seed{model['seed_index']}: "
        f"{model['operating_threshold']:.2f}"
    )

print()

print(
    "Decoder:"
)

print(
    "  minimum score: 0.05"
)

print(
    "  top-k:         100"
)

print(
    "  NMS IoU:       0.50"
)

print(
    "  local peak:    3x3"
)

print()

print(
    "RSC input:"
)

print(
    "  original full image"
)

print(
    "  NO resize"
)

print(
    "  NO crop"
)

print(
    "  NO tiling"
)

print(
    "  RGB float32 / 255"
)

print(
    "  right/bottom zero-pad only to nearest multiple of 16"
)

print()

print(
    "RSC training:             NO"
)

print(
    "RSC CRPA:                 NO"
)

print(
    "RSC calibration:          NO"
)

print(
    "RSC threshold tuning:     NO"
)

print(
    "Checkpoint selection:     NO"
)

print(
    "Seed selection:           NO"
)

print(
    "Best-seed selection:      NO"
)

print()

print(
    "All 3 final seeds required: YES"
)

print(
    "Report mean ± sample SD:     YES"
)

print()

print(
    "Protocol JSON:"
)

print(
    f"  {PROTOCOL_JSON}"
)

print()

print(
    "Protocol SHA256:"
)

print(
    f"  {protocol_sha}"
)

print()

print(
    "SHA record:"
)

print(
    f"  {PROTOCOL_SHA_RECORD}"
)

print()

print(
    "Future RSC evaluation root:"
)

print(
    f"  {DESIGNATED_EVAL_ROOT}"
)

print()

print(
    "NO RSC MODEL INFERENCE HAS BEEN PERFORMED BY THIS CELL."
)

print()

print(
    "NEXT READINESS:"
)

print(
    "  READY_TO_IMPLEMENT_AND_AUDIT_RSC_EVALUATOR"
)

print("=" * 76)

DASO-Net — RSC EXTERNAL EVALUATION PRE-INFERENCE PROTOCOL FREEZE v1

THIS CELL PERFORMS NO MODEL INFERENCE.

Purpose:
  freeze the complete RSC detection-evaluation protocol
  before predictions from any final seed are observed

PASS  RSC pre-inference asset audit identity
PASS  RSC primary/sensitivity scientific sets
PASS  Final Seed-1 checkpoint
PASS  Final Seed-2 checkpoint
PASS  Final Seed-3 checkpoint
PASS  Validated Phase5C evaluator script present
PASS  Protocol JSON read-back identity

DASO-NET RSC EXTERNAL EVALUATION PROTOCOL: FROZEN

Status:
  FROZEN_BEFORE_RSC_INFERENCE

PRIMARY:
  official authors' test images: 250
  GT boxes:                     539
  zero-GT images:                 4

SENSITIVITY:
  exact-dedup images:            247
  GT boxes:                      539
  zero-GT images:                  1

External task:
  class-agnostic BOUNDING-BOX detection only

Primary metrics:
  Box AP50
  Box AP75
  Box AP50:95

Frozen operating thresholds:
  Seed1: 0.30
  Seed2: 

In [25]:
%%writefile /kaggle/working/evaluate_rsc_external_v1.py
# ============================================================
# DASO-Net — RSC EXTERNAL DETECTION EVALUATOR v1
#
# Scientific task:
#   class-agnostic bounding-box detection only
#
# Frozen protocol:
#   RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.json
#
# Modes:
#   self-test : synthetic metric-engine unit tests, NO RSC inference
#   smoke     : structural two-image RSC inference only, NO metrics
#   evaluate  : formal per-seed RSC evaluation
#
# IMPORTANT
# ---------
# RSC is NEVER used for:
#   training
#   CRPA
#   calibration
#   threshold tuning
#   checkpoint selection
#   seed selection
#
# PRIMARY:
#   official authors' test-250
#
# SENSITIVITY:
#   exact-dedup-247
#
# ============================================================

from __future__ import annotations

import argparse
import csv
import hashlib
import importlib.util
import inspect
import json
import math
import sys
from collections import defaultdict
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import torch


# ============================================================
# 1. FROZEN PROTOCOL
# ============================================================

DEFAULT_PROTOCOL = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_external_evaluation_protocol_freeze_v1/"
    "RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "e3c87caebafe87c7671b5745a480c0ed"
    "f67f444feac4278633da6addae4d1f7e"
)

SMOKE_OUTPUT_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_v1_smoke_audit"
)

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
}


# ============================================================
# 2. GENERIC HELPERS
# ============================================================

def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open("rb") as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(block)

    return h.hexdigest()


def require(
    condition: bool,
    message: str,
) -> None:

    if not condition:
        raise RuntimeError(message)


def require_file(
    path: Path,
    label: str,
) -> None:

    require(
        path.is_file(),
        f"{label} missing:\n{path}",
    )


def require_dir(
    path: Path,
    label: str,
) -> None:

    require(
        path.is_dir(),
        f"{label} missing:\n{path}",
    )


def read_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path: Path,
    obj,
) -> None:

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def read_csv_rows(
    path: Path,
):

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:

        return list(
            csv.DictReader(f)
        )


def as_int(
    value,
    label: str,
) -> int:

    try:

        number = float(value)

    except Exception as exc:

        raise RuntimeError(
            f"Could not parse integer field "
            f"{label}: {value!r}"
        ) from exc

    require(
        math.isfinite(number),
        f"Non-finite integer field "
        f"{label}: {value!r}",
    )

    require(
        number.is_integer(),
        f"Non-integral integer field "
        f"{label}: {value!r}",
    )

    return int(number)


def parse_bool(
    value,
) -> bool:

    text = str(
        value
    ).strip().lower()

    if text in {
        "true",
        "1",
        "yes",
        "y",
    }:

        return True

    if text in {
        "false",
        "0",
        "no",
        "n",
    }:

        return False

    raise RuntimeError(
        f"Could not parse boolean: {value!r}"
    )


def safe_float(
    value,
    label: str,
) -> float:

    try:

        number = float(value)

    except Exception as exc:

        raise RuntimeError(
            f"Could not parse float field "
            f"{label}: {value!r}"
        ) from exc

    require(
        math.isfinite(number),
        f"Non-finite float field "
        f"{label}: {value!r}",
    )

    return number


# ============================================================
# 3. LOAD + VERIFY FROZEN PROTOCOL
# ============================================================

def load_protocol(
    path: Path,
):

    require_file(
        path,
        "Frozen RSC evaluation protocol",
    )

    actual_sha = sha256_file(path)

    require(
        actual_sha
        == EXPECTED_PROTOCOL_SHA256,
        "Frozen RSC protocol SHA mismatch.\n"
        f"Expected: {EXPECTED_PROTOCOL_SHA256}\n"
        f"Actual:   {actual_sha}",
    )

    protocol = read_json(path)

    require(
        protocol.get("status")
        == "FROZEN_BEFORE_RSC_INFERENCE",
        "RSC protocol is not marked "
        "FROZEN_BEFORE_RSC_INFERENCE.",
    )

    require(
        protocol[
            "scientific_policy"
        ][
            "rsc_threshold_tuning"
        ] is False,
        "Frozen protocol unexpectedly permits "
        "RSC threshold tuning.",
    )

    require(
        protocol[
            "metric_protocol"
        ][
            "segmentation_metrics"
        ] is False,
        "Frozen protocol unexpectedly enables "
        "RSC segmentation metrics.",
    )

    require(
        protocol[
            "input_protocol"
        ][
            "resize"
        ] is False,
        "Frozen protocol unexpectedly permits resize.",
    )

    return protocol, actual_sha


# ============================================================
# 4. LOAD VALIDATED PHASE5C MODEL/DECODER MODULE
# ============================================================

def load_phase5c_module(
    protocol,
):

    script = Path(
        protocol[
            "validated_evaluator"
        ][
            "script"
        ]
    )

    expected_sha = (
        protocol[
            "validated_evaluator"
        ][
            "whole_file_sha256"
        ]
    )

    require_file(
        script,
        "Validated Phase5C evaluator script",
    )

    actual_sha = sha256_file(script)

    require(
        actual_sha == expected_sha,
        "Validated Phase5C evaluator whole-file "
        "SHA changed.\n"
        f"Expected: {expected_sha}\n"
        f"Actual:   {actual_sha}",
    )

    training_root = (
        script.parent.parent
    )

    for candidate in [
        training_root,
        script.parent,
        training_root / "src",
    ]:

        text = str(candidate)

        if (
            candidate.exists()
            and text not in sys.path
        ):

            sys.path.insert(
                0,
                text,
            )

    module_name = (
        "dasonet_phase5c_frozen_evaluator"
    )

    spec = (
        importlib.util
        .spec_from_file_location(
            module_name,
            script,
        )
    )

    require(
        spec is not None
        and spec.loader is not None,
        "Could not create import specification "
        "for evaluate_phase5c.py.",
    )

    module = (
        importlib.util
        .module_from_spec(spec)
    )

    spec.loader.exec_module(module)

    require(
        hasattr(
            module,
            "build_model",
        ),
        "Validated evaluator does not expose "
        "build_model.",
    )

    require(
        hasattr(
            module,
            "decode_instances",
        ),
        "Validated evaluator does not expose "
        "decode_instances.",
    )

    build_model = module.build_model
    decode_instances = module.decode_instances

    decoder_signature = inspect.signature(
        decode_instances
    )

    required_decoder_parameters = {
        "outputs",
        "input_size",
        "stride",
        "score_threshold",
        "topk",
        "nms_iou",
        "mask_threshold",
        "local_peak_kernel",
    }

    missing = (
        required_decoder_parameters
        - set(
            decoder_signature.parameters
        )
    )

    require(
        not missing,
        "Validated decoder signature is missing "
        "required parameters:\n"
        + "\n".join(
            sorted(missing)
        ),
    )

    return {
        "module":
            module,

        "build_model":
            build_model,

        "decode_instances":
            decode_instances,

        "script":
            script,

        "script_sha256":
            actual_sha,

        "decoder_signature":
            str(
                decoder_signature
            ),
    }


# ============================================================
# 5. MODEL IDENTITY
# ============================================================

def get_model_info(
    protocol,
    seed_index: int,
):

    candidates = [
        model
        for model
        in protocol[
            "models"
        ]
        if int(
            model[
                "seed_index"
            ]
        )
        == seed_index
    ]

    require(
        len(candidates) == 1,
        "Expected exactly one frozen model for "
        f"seed index {seed_index}.",
    )

    model_info = candidates[0]

    checkpoint = Path(
        model_info[
            "checkpoint"
        ]
    )

    require_file(
        checkpoint,
        (
            "Frozen final checkpoint for "
            f"Seed-{seed_index}"
        ),
    )

    actual_sha = sha256_file(
        checkpoint
    )

    expected_sha = (
        model_info[
            "checkpoint_sha256"
        ]
    )

    require(
        actual_sha == expected_sha,
        "Checkpoint SHA mismatch.\n"
        f"Seed index: {seed_index}\n"
        f"Expected:   {expected_sha}\n"
        f"Actual:     {actual_sha}",
    )

    require(
        int(
            model_info[
                "successful_optimizer_updates"
            ]
        )
        == 2750,
        "Frozen checkpoint successful-update "
        "count != 2750.",
    )

    return {
        **model_info,
        "checkpoint":
            checkpoint,
        "checkpoint_sha256":
            actual_sha,
    }


# ============================================================
# 6. RSC MANIFEST / GT LOADING
# ============================================================

def load_rsc_rows(
    protocol,
):

    package_root = Path(
        protocol[
            "rsc_package"
        ][
            "package_root"
        ]
    )

    project_root = Path(
        protocol[
            "rsc_package"
        ][
            "project_payload_root"
        ]
    )

    primary_manifest = Path(
        protocol[
            "scientific_sets"
        ][
            "primary"
        ][
            "manifest"
        ]
    )

    dedup_manifest = Path(
        protocol[
            "scientific_sets"
        ][
            "sensitivity"
        ][
            "manifest"
        ]
    )

    require_dir(
        package_root,
        "RSC package root",
    )

    require_dir(
        project_root,
        "RSC scientific payload root",
    )

    require_file(
        primary_manifest,
        "RSC primary manifest",
    )

    require_file(
        dedup_manifest,
        "RSC dedup manifest",
    )

    require(
        sha256_file(
            primary_manifest
        )
        == protocol[
            "scientific_sets"
        ][
            "primary"
        ][
            "manifest_sha256"
        ],
        "Primary manifest SHA changed.",
    )

    require(
        sha256_file(
            dedup_manifest
        )
        == protocol[
            "scientific_sets"
        ][
            "sensitivity"
        ][
            "manifest_sha256"
        ],
        "Dedup manifest SHA changed.",
    )

    primary_rows = read_csv_rows(
        primary_manifest
    )

    dedup_rows = read_csv_rows(
        dedup_manifest
    )

    require(
        len(primary_rows) == 250,
        "Primary manifest rows != 250.",
    )

    require(
        len(dedup_rows) == 247,
        "Dedup manifest rows != 247.",
    )

    primary_rows = sorted(
        primary_rows,
        key=lambda row: as_int(
            row[
                "official_test_index"
            ],
            "official_test_index",
        ),
    )

    expected_order = list(
        range(
            1,
            251,
        )
    )

    actual_order = [
        as_int(
            row[
                "official_test_index"
            ],
            "official_test_index",
        )
        for row in primary_rows
    ]

    require(
        actual_order == expected_order,
        "Primary manifest official_test_index "
        "is not exactly 1..250.",
    )

    dedup_set = {
        str(
            row[
                "filename"
            ]
        ).strip()
        for row in dedup_rows
    }

    enriched = []

    total_gt = 0

    for row in primary_rows:

        filename = str(
            row[
                "filename"
            ]
        ).strip()

        image_path = (
            project_root
            / Path(
                str(
                    row[
                        "project_image_path"
                    ]
                ).strip()
            )
        )

        label_path = (
            project_root
            / Path(
                str(
                    row[
                        "project_label_path"
                    ]
                ).strip()
            )
        )

        require_file(
            image_path,
            "RSC image",
        )

        require_file(
            label_path,
            "RSC label",
        )

        width = as_int(
            row[
                "width"
            ],
            "width",
        )

        height = as_int(
            row[
                "height"
            ],
            "height",
        )

        gt_boxes = read_yolo_gt_boxes(
            label_path=label_path,
            width=width,
            height=height,
        )

        manifest_gt_count = as_int(
            row[
                "gt_box_count"
            ],
            "gt_box_count",
        )

        require(
            len(gt_boxes)
            == manifest_gt_count,
            "GT label content differs from frozen "
            "manifest count.\n"
            f"Filename: {filename}",
        )

        total_gt += len(gt_boxes)

        enriched.append(
            {
                "official_test_index":
                    as_int(
                        row[
                            "official_test_index"
                        ],
                        "official_test_index",
                    ),

                "filename":
                    filename,

                "image_path":
                    image_path,

                "label_path":
                    label_path,

                "width":
                    width,

                "height":
                    height,

                "gt_boxes":
                    gt_boxes,

                "dedup247_member":
                    filename
                    in dedup_set,

                "frozen_image_sha256":
                    str(
                        row[
                            "image_sha256"
                        ]
                    ).strip(),

                "frozen_label_sha256":
                    str(
                        row[
                            "label_sha256"
                        ]
                    ).strip(),
            }
        )

    require(
        total_gt == 539,
        "RSC GT-box total != 539.",
    )

    zero_gt = {
        row[
            "filename"
        ]
        for row in enriched
        if len(
            row[
                "gt_boxes"
            ]
        ) == 0
    }

    require(
        zero_gt
        == {
            "test_113.jpg",
            "test_182.jpg",
            "test_183.jpg",
            "test_184.jpg",
        },
        "Unexpected RSC zero-GT image set.",
    )

    return enriched


def read_yolo_gt_boxes(
    label_path: Path,
    width: int,
    height: int,
):

    lines = [
        line.strip()
        for line
        in label_path.read_text(
            encoding="utf-8",
            errors="strict",
        ).splitlines()
        if line.strip()
    ]

    boxes = []

    for line_number, line in enumerate(
        lines,
        start=1,
    ):

        parts = line.split()

        require(
            len(parts) == 5,
            "Unexpected YOLO label format.\n"
            f"File: {label_path}\n"
            f"Line {line_number}: {line}",
        )

        values = [
            safe_float(
                part,
                (
                    "YOLO value "
                    f"{line_number}"
                ),
            )
            for part in parts
        ]

        class_float, xc, yc, bw, bh = values

        require(
            class_float.is_integer(),
            "Non-integral YOLO class ID.\n"
            f"File: {label_path}\n"
            f"Line: {line}",
        )

        require(
            0.0 <= xc <= 1.0
            and 0.0 <= yc <= 1.0
            and 0.0 < bw <= 1.0
            and 0.0 < bh <= 1.0,
            "Invalid normalized YOLO box.\n"
            f"File: {label_path}\n"
            f"Line: {line}",
        )

        x1 = (
            xc - bw / 2.0
        ) * width

        y1 = (
            yc - bh / 2.0
        ) * height

        x2 = (
            xc + bw / 2.0
        ) * width

        y2 = (
            yc + bh / 2.0
        ) * height

        epsilon = 1e-6

        require(
            x1 >= -epsilon
            and y1 >= -epsilon
            and x2 <= width + epsilon
            and y2 <= height + epsilon
            and x2 > x1
            and y2 > y1,
            "Converted GT box invalid.\n"
            f"File: {label_path}\n"
            f"Line: {line}",
        )

        boxes.append(
            [
                float(x1),
                float(y1),
                float(x2),
                float(y2),
            ]
        )

    return boxes


# ============================================================
# 7. FROZEN INPUT PREPROCESSING
# ============================================================

def preprocess_rsc_image(
    image_path: Path,
):

    bgr = cv2.imread(
        str(image_path),
        cv2.IMREAD_COLOR,
    )

    require(
        bgr is not None,
        "Could not decode RSC image:\n"
        f"{image_path}",
    )

    original_h, original_w = (
        bgr.shape[:2]
    )

    rgb = cv2.cvtColor(
        bgr,
        cv2.COLOR_BGR2RGB,
    )

    rgb = (
        rgb.astype(
            np.float32
        )
        / 255.0
    )

    padded_h = (
        (
            original_h + 15
        )
        // 16
        * 16
    )

    padded_w = (
        (
            original_w + 15
        )
        // 16
        * 16
    )

    pad_bottom = (
        padded_h
        - original_h
    )

    pad_right = (
        padded_w
        - original_w
    )

    require(
        0 <= pad_bottom <= 15,
        "Invalid bottom padding.",
    )

    require(
        0 <= pad_right <= 15,
        "Invalid right padding.",
    )

    if (
        pad_bottom > 0
        or pad_right > 0
    ):

        rgb = np.pad(
            rgb,
            (
                (
                    0,
                    pad_bottom,
                ),
                (
                    0,
                    pad_right,
                ),
                (
                    0,
                    0,
                ),
            ),
            mode="constant",
            constant_values=0.0,
        )

    tensor = (
        torch.from_numpy(rgb)
        .permute(
            2,
            0,
            1,
        )
        .contiguous()
        .unsqueeze(0)
    )

    require(
        tensor.dtype
        == torch.float32,
        "RSC tensor is not float32.",
    )

    require(
        tuple(
            tensor.shape
        )
        == (
            1,
            3,
            padded_h,
            padded_w,
        ),
        "Unexpected RSC tensor shape.",
    )

    require(
        float(
            tensor.min()
        ) >= 0.0
        and float(
            tensor.max()
        ) <= 1.0,
        "RSC tensor outside [0,1].",
    )

    return {
        "tensor":
            tensor,

        "original_h":
            original_h,

        "original_w":
            original_w,

        "padded_h":
            padded_h,

        "padded_w":
            padded_w,

        "pad_bottom":
            pad_bottom,

        "pad_right":
            pad_right,
    }


# ============================================================
# 8. DECODER OUTPUT NORMALIZATION
# ============================================================

def to_numpy(
    value,
):

    if isinstance(
        value,
        torch.Tensor,
    ):

        return (
            value.detach()
            .cpu()
            .numpy()
        )

    return np.asarray(value)


def find_first_key(
    mapping: dict,
    candidates,
):

    for key in candidates:

        if key in mapping:

            return key

    return None


def normalize_decoded_instances(
    decoded,
):

    # --------------------------------------------------------
    # Case A:
    # {
    #   "boxes": Nx4,
    #   "scores": N,
    #   ...
    # }
    # --------------------------------------------------------

    if isinstance(
        decoded,
        dict,
    ):

        nested_key = find_first_key(
            decoded,
            [
                "instances",
                "detections",
                "predictions",
            ],
        )

        if nested_key is not None:

            return normalize_decoded_instances(
                decoded[
                    nested_key
                ]
            )

        box_key = find_first_key(
            decoded,
            [
                "boxes",
                "bboxes",
                "xyxy",
            ],
        )

        score_key = find_first_key(
            decoded,
            [
                "scores",
                "confidences",
                "confidence",
            ],
        )

        if (
            box_key is not None
            and score_key is not None
        ):

            boxes = to_numpy(
                decoded[
                    box_key
                ]
            )

            scores = to_numpy(
                decoded[
                    score_key
                ]
            ).reshape(-1)

            if boxes.size == 0:

                boxes = np.empty(
                    (
                        0,
                        4,
                    ),
                    dtype=np.float32,
                )

            boxes = np.asarray(
                boxes
            ).reshape(
                -1,
                4,
            )

            require(
                len(boxes)
                == len(scores),
                "Decoder boxes/scores length mismatch.",
            )

            return [
                {
                    "box":
                        boxes[
                            i
                        ].astype(
                            float
                        ).tolist(),

                    "score":
                        float(
                            scores[
                                i
                            ]
                        ),

                    "decoder_order":
                        i,
                }
                for i in range(
                    len(boxes)
                )
            ]

        # Single-instance dictionary.
        single_box_key = find_first_key(
            decoded,
            [
                "box",
                "bbox",
                "bbox_xyxy",
                "xyxy",
            ],
        )

        single_score_key = find_first_key(
            decoded,
            [
                "score",
                "confidence",
                "conf",
            ],
        )

        if (
            single_box_key is not None
            and single_score_key is not None
        ):

            box = to_numpy(
                decoded[
                    single_box_key
                ]
            ).reshape(-1)

            require(
                len(box) == 4,
                "Single decoder box does not "
                "contain four coordinates.",
            )

            return [
                {
                    "box":
                        box.astype(
                            float
                        ).tolist(),

                    "score":
                        float(
                            decoded[
                                single_score_key
                            ]
                        ),

                    "decoder_order":
                        0,
                }
            ]

        raise RuntimeError(
            "Unsupported dictionary decoder output.\n"
            f"Keys: {sorted(decoded.keys())}"
        )


    # --------------------------------------------------------
    # Case B:
    # list of instance dictionaries
    # --------------------------------------------------------

    if isinstance(
        decoded,
        (list, tuple),
    ):

        if len(decoded) == 0:

            return []

        # Some decoders return one batch element:
        # [ {"boxes":..., "scores":...} ]
        if (
            len(decoded) == 1
            and isinstance(
                decoded[0],
                dict,
            )
        ):

            first = decoded[0]

            if (
                find_first_key(
                    first,
                    [
                        "boxes",
                        "bboxes",
                    ],
                )
                is not None
            ):

                return normalize_decoded_instances(
                    first
                )

        # Tuple-like:
        # (boxes, scores, ...)
        if (
            len(decoded) >= 2
            and not isinstance(
                decoded[0],
                dict,
            )
        ):

            try:

                boxes = to_numpy(
                    decoded[0]
                )

                scores = to_numpy(
                    decoded[1]
                ).reshape(-1)

                if (
                    boxes.ndim >= 2
                    and boxes.shape[-1] == 4
                ):

                    boxes = boxes.reshape(
                        -1,
                        4,
                    )

                    require(
                        len(boxes)
                        == len(scores),
                        "Tuple decoder output length mismatch.",
                    )

                    return [
                        {
                            "box":
                                boxes[
                                    i
                                ].astype(
                                    float
                                ).tolist(),

                            "score":
                                float(
                                    scores[
                                        i
                                    ]
                                ),

                            "decoder_order":
                                i,
                        }
                        for i in range(
                            len(boxes)
                        )
                    ]

            except Exception:
                pass

        instances = []

        for index, item in enumerate(
            decoded
        ):

            if isinstance(
                item,
                dict,
            ):

                box_key = find_first_key(
                    item,
                    [
                        "box",
                        "bbox",
                        "bbox_xyxy",
                        "xyxy",
                    ],
                )

                score_key = find_first_key(
                    item,
                    [
                        "score",
                        "confidence",
                        "conf",
                    ],
                )

                require(
                    box_key is not None,
                    "Decoder instance dictionary "
                    "has no recognized box key.\n"
                    f"Keys: {sorted(item.keys())}",
                )

                require(
                    score_key is not None,
                    "Decoder instance dictionary "
                    "has no recognized score key.\n"
                    f"Keys: {sorted(item.keys())}",
                )

                box = to_numpy(
                    item[
                        box_key
                    ]
                ).reshape(-1)

                require(
                    len(box) == 4,
                    "Decoder instance box does not "
                    "contain four coordinates.",
                )

                instances.append(
                    {
                        "box":
                            box.astype(
                                float
                            ).tolist(),

                        "score":
                            float(
                                item[
                                    score_key
                                ]
                            ),

                        "decoder_order":
                            index,
                    }
                )

                continue

            # Attribute-based instance object.
            box_value = None
            score_value = None

            for attr in [
                "box",
                "bbox",
                "bbox_xyxy",
                "xyxy",
            ]:

                if hasattr(
                    item,
                    attr,
                ):

                    box_value = getattr(
                        item,
                        attr,
                    )

                    break

            for attr in [
                "score",
                "confidence",
                "conf",
            ]:

                if hasattr(
                    item,
                    attr,
                ):

                    score_value = getattr(
                        item,
                        attr,
                    )

                    break

            require(
                box_value is not None
                and score_value is not None,
                "Unsupported decoder instance object "
                f"type: {type(item)}",
            )

            box = to_numpy(
                box_value
            ).reshape(-1)

            require(
                len(box) == 4,
                "Attribute-based decoder instance "
                "box length != 4.",
            )

            instances.append(
                {
                    "box":
                        box.astype(
                            float
                        ).tolist(),

                    "score":
                        float(
                            score_value
                        ),

                    "decoder_order":
                        index,
                }
            )

        return instances


    raise RuntimeError(
        "Unsupported decoder output type: "
        f"{type(decoded)}"
    )


# ============================================================
# 9. ORIGINAL-IMAGE COORDINATE POLICY
# ============================================================

def apply_original_image_box_policy(
    predictions,
    original_w: int,
    original_h: int,
):

    retained = []

    for item in predictions:

        box = np.asarray(
            item[
                "box"
            ],
            dtype=np.float64,
        ).reshape(-1)

        require(
            len(box) == 4,
            "Decoded prediction box length != 4.",
        )

        score = float(
            item[
                "score"
            ]
        )

        require(
            np.all(
                np.isfinite(box)
            ),
            "Non-finite decoded box.",
        )

        require(
            math.isfinite(score),
            "Non-finite decoded score.",
        )

        x1, y1, x2, y2 = [
            float(value)
            for value in box
        ]

        require(
            x2 > x1
            and y2 > y1,
            "Decoded box is degenerate before "
            "original-image clipping.",
        )

        cx = (
            x1 + x2
        ) / 2.0

        cy = (
            y1 + y2
        ) / 2.0

        # Frozen policy:
        # discard boxes whose CENTER is outside
        # original image extent.
        if not (
            0.0 <= cx < original_w
            and 0.0 <= cy < original_h
        ):

            continue

        x1 = max(
            0.0,
            min(
                float(original_w),
                x1,
            ),
        )

        y1 = max(
            0.0,
            min(
                float(original_h),
                y1,
            ),
        )

        x2 = max(
            0.0,
            min(
                float(original_w),
                x2,
            ),
        )

        y2 = max(
            0.0,
            min(
                float(original_h),
                y2,
            ),
        )

        if (
            x2 <= x1
            or y2 <= y1
        ):

            continue

        retained.append(
            {
                "box":
                    [
                        x1,
                        y1,
                        x2,
                        y2,
                    ],

                "score":
                    score,

                "decoder_order":
                    int(
                        item[
                            "decoder_order"
                        ]
                    ),
            }
        )

    retained.sort(
        key=lambda item: (
            -item[
                "score"
            ],
            item[
                "decoder_order"
            ],
        )
    )

    return retained


# ============================================================
# 10. ONE-IMAGE INFERENCE
# ============================================================

def infer_one(
    model,
    decode_instances,
    row,
    device: torch.device,
    decoder_spec,
):

    prep = preprocess_rsc_image(
        row[
            "image_path"
        ]
    )

    require(
        prep[
            "original_w"
        ]
        == row[
            "width"
        ]
        and prep[
            "original_h"
        ]
        == row[
            "height"
        ],
        "Decoded image dimensions differ from "
        "frozen manifest.",
    )

    tensor = (
        prep[
            "tensor"
        ]
        .to(
            device=device,
            dtype=torch.float32,
            non_blocking=False,
        )
    )

    with torch.inference_mode():

        outputs = model(tensor)

        decoded = decode_instances(
            outputs=outputs,
            input_size=(
                prep[
                    "padded_h"
                ],
                prep[
                    "padded_w"
                ],
            ),
            stride=int(
                decoder_spec[
                    "stride"
                ]
            ),
            score_threshold=float(
                decoder_spec[
                    "decode_min_score"
                ]
            ),
            topk=int(
                decoder_spec[
                    "topk"
                ]
            ),
            nms_iou=float(
                decoder_spec[
                    "nms_iou"
                ]
            ),
            mask_threshold=float(
                decoder_spec[
                    "mask_threshold"
                ]
            ),
            local_peak_kernel=int(
                decoder_spec[
                    "local_peak_kernel"
                ]
            ),
        )

    normalized = (
        normalize_decoded_instances(
            decoded
        )
    )

    predictions = (
        apply_original_image_box_policy(
            predictions=normalized,
            original_w=prep[
                "original_w"
            ],
            original_h=prep[
                "original_h"
            ],
        )
    )

    decoder_floor = float(
        decoder_spec[
            "decode_min_score"
        ]
    )

    for prediction in predictions:

        require(
            prediction[
                "score"
            ]
            >= decoder_floor
            - 1e-8,
            "Decoded prediction fell below frozen "
            "decoder score floor.",
        )

    return predictions, prep


# ============================================================
# 11. BOX IOU
# ============================================================

def box_iou_single(
    box_a,
    box_b,
) -> float:

    ax1, ay1, ax2, ay2 = [
        float(x)
        for x in box_a
    ]

    bx1, by1, bx2, by2 = [
        float(x)
        for x in box_b
    ]

    inter_x1 = max(
        ax1,
        bx1,
    )

    inter_y1 = max(
        ay1,
        by1,
    )

    inter_x2 = min(
        ax2,
        bx2,
    )

    inter_y2 = min(
        ay2,
        by2,
    )

    inter_w = max(
        0.0,
        inter_x2 - inter_x1,
    )

    inter_h = max(
        0.0,
        inter_y2 - inter_y1,
    )

    intersection = (
        inter_w
        * inter_h
    )

    area_a = max(
        0.0,
        ax2 - ax1,
    ) * max(
        0.0,
        ay2 - ay1,
    )

    area_b = max(
        0.0,
        bx2 - bx1,
    ) * max(
        0.0,
        by2 - by1,
    )

    union = (
        area_a
        + area_b
        - intersection
    )

    if union <= 0.0:

        return 0.0

    return float(
        intersection
        / union
    )


# ============================================================
# 12. MATCH ONE IMAGE AT ONE IOU THRESHOLD
# ============================================================

def match_image_predictions(
    predictions,
    gt_boxes,
    iou_threshold: float,
):

    matched_gt = set()

    results = []

    for prediction in sorted(
        predictions,
        key=lambda item: (
            -item[
                "score"
            ],
            item[
                "decoder_order"
            ],
        ),
    ):

        best_iou = -1.0
        best_gt_index = None

        for gt_index, gt_box in enumerate(
            gt_boxes
        ):

            if gt_index in matched_gt:
                continue

            iou = box_iou_single(
                prediction[
                    "box"
                ],
                gt_box,
            )

            if iou > best_iou:

                best_iou = iou
                best_gt_index = gt_index

        if (
            best_gt_index is not None
            and best_iou
            >= iou_threshold
        ):

            matched_gt.add(
                best_gt_index
            )

            results.append(
                {
                    "prediction":
                        prediction,

                    "tp":
                        1,

                    "fp":
                        0,

                    "matched_gt_index":
                        best_gt_index,

                    "matched_iou":
                        float(
                            best_iou
                        ),
                }
            )

        else:

            results.append(
                {
                    "prediction":
                        prediction,

                    "tp":
                        0,

                    "fp":
                        1,

                    "matched_gt_index":
                        None,

                    "matched_iou":
                        None,
                }
            )

    fn = (
        len(gt_boxes)
        - len(matched_gt)
    )

    return results, fn


# ============================================================
# 13. GLOBAL AP
# ============================================================

def compute_ap_at_iou(
    records,
    iou_threshold: float,
):

    total_gt = sum(
        len(
            record[
                "gt_boxes"
            ]
        )
        for record in records
    )

    require(
        total_gt > 0,
        "AP evaluation has zero total GT boxes.",
    )

    pooled_predictions = []

    for record in records:

        for prediction in record[
            "predictions"
        ]:

            pooled_predictions.append(
                {
                    "official_test_index":
                        record[
                            "official_test_index"
                        ],

                    "filename":
                        record[
                            "filename"
                        ],

                    "gt_boxes":
                        record[
                            "gt_boxes"
                        ],

                    "prediction":
                        prediction,
                }
            )

    pooled_predictions.sort(
        key=lambda item: (
            -item[
                "prediction"
            ][
                "score"
            ],
            item[
                "official_test_index"
            ],
            item[
                "prediction"
            ][
                "decoder_order"
            ],
        )
    )

    matched_by_image = defaultdict(
        set
    )

    tp_flags = []

    fp_flags = []

    for item in pooled_predictions:

        filename = item[
            "filename"
        ]

        prediction = item[
            "prediction"
        ]

        gt_boxes = item[
            "gt_boxes"
        ]

        best_iou = -1.0
        best_gt_index = None

        for gt_index, gt_box in enumerate(
            gt_boxes
        ):

            if (
                gt_index
                in matched_by_image[
                    filename
                ]
            ):

                continue

            iou = box_iou_single(
                prediction[
                    "box"
                ],
                gt_box,
            )

            if iou > best_iou:

                best_iou = iou
                best_gt_index = gt_index

        if (
            best_gt_index is not None
            and best_iou
            >= iou_threshold
        ):

            matched_by_image[
                filename
            ].add(
                best_gt_index
            )

            tp_flags.append(
                1.0
            )

            fp_flags.append(
                0.0
            )

        else:

            tp_flags.append(
                0.0
            )

            fp_flags.append(
                1.0
            )

    if len(tp_flags) == 0:

        return {
            "iou_threshold":
                float(
                    iou_threshold
                ),

            "ap":
                0.0,

            "total_gt":
                total_gt,

            "total_predictions":
                0,
        }

    cumulative_tp = np.cumsum(
        np.asarray(
            tp_flags,
            dtype=np.float64,
        )
    )

    cumulative_fp = np.cumsum(
        np.asarray(
            fp_flags,
            dtype=np.float64,
        )
    )

    recall = (
        cumulative_tp
        / float(total_gt)
    )

    precision = (
        cumulative_tp
        / np.maximum(
            cumulative_tp
            + cumulative_fp,
            1e-12,
        )
    )

    recall_grid = np.linspace(
        0.0,
        1.0,
        101,
    )

    interpolated = []

    for recall_target in recall_grid:

        eligible = (
            recall
            >= recall_target
        )

        if np.any(
            eligible
        ):

            interpolated.append(
                float(
                    np.max(
                        precision[
                            eligible
                        ]
                    )
                )
            )

        else:

            interpolated.append(
                0.0
            )

    ap = float(
        np.mean(
            interpolated
        )
    )

    return {
        "iou_threshold":
            float(
                iou_threshold
            ),

        "ap":
            ap,

        "total_gt":
            total_gt,

        "total_predictions":
            len(
                pooled_predictions
            ),
    }


def compute_all_ap(
    records,
):

    thresholds = [
        round(
            0.50 + 0.05 * i,
            2,
        )
        for i in range(10)
    ]

    results = {}

    for threshold in thresholds:

        metric = compute_ap_at_iou(
            records=records,
            iou_threshold=threshold,
        )

        results[
            f"{threshold:.2f}"
        ] = metric

    ap50 = results[
        "0.50"
    ][
        "ap"
    ]

    ap75 = results[
        "0.75"
    ][
        "ap"
    ]

    ap5095 = float(
        np.mean(
            [
                results[
                    f"{threshold:.2f}"
                ][
                    "ap"
                ]
                for threshold
                in thresholds
            ]
        )
    )

    return {
        "box_ap50":
            ap50,

        "box_ap75":
            ap75,

        "box_ap50_95":
            ap5095,

        "by_iou":
            results,
    }


# ============================================================
# 14. FROZEN OPERATING METRICS
# ============================================================

def compute_operating_metrics(
    records,
    operating_threshold: float,
):

    total_tp = 0
    total_fp = 0
    total_fn = 0

    matched_ious = []

    count_errors = []
    count_abs_errors = []

    zero_gt_fp = 0
    zero_gt_images = 0
    zero_gt_with_fp = 0

    per_image = []

    for record in records:

        predictions = [
            prediction
            for prediction
            in record[
                "predictions"
            ]
            if prediction[
                "score"
            ]
            >= operating_threshold
        ]

        matches, fn = (
            match_image_predictions(
                predictions=predictions,
                gt_boxes=record[
                    "gt_boxes"
                ],
                iou_threshold=0.50,
            )
        )

        tp = sum(
            item[
                "tp"
            ]
            for item in matches
        )

        fp = sum(
            item[
                "fp"
            ]
            for item in matches
        )

        for item in matches:

            if (
                item[
                    "matched_iou"
                ]
                is not None
            ):

                matched_ious.append(
                    float(
                        item[
                            "matched_iou"
                        ]
                    )
                )

        gt_count = len(
            record[
                "gt_boxes"
            ]
        )

        pred_count = len(
            predictions
        )

        count_error = (
            pred_count
            - gt_count
        )

        count_abs_error = abs(
            count_error
        )

        count_errors.append(
            float(
                count_error
            )
        )

        count_abs_errors.append(
            float(
                count_abs_error
            )
        )

        if gt_count == 0:

            zero_gt_images += 1
            zero_gt_fp += fp

            if fp >= 1:
                zero_gt_with_fp += 1

        total_tp += tp
        total_fp += fp
        total_fn += fn

        max_decoded_score = (
            max(
                [
                    prediction[
                        "score"
                    ]
                    for prediction
                    in record[
                        "predictions"
                    ]
                ]
            )
            if record[
                "predictions"
            ]
            else None
        )

        per_image.append(
            {
                "official_test_index":
                    record[
                        "official_test_index"
                    ],

                "filename":
                    record[
                        "filename"
                    ],

                "subset_primary":
                    True,

                "subset_dedup247":
                    bool(
                        record[
                            "dedup247_member"
                        ]
                    ),

                "gt_count":
                    gt_count,

                "pred_count_operating":
                    pred_count,

                "tp_operating_iou50":
                    tp,

                "fp_operating_iou50":
                    fp,

                "fn_operating_iou50":
                    fn,

                "count_error":
                    count_error,

                "count_abs_error":
                    count_abs_error,

                "max_decoded_score":
                    max_decoded_score,
            }
        )

    precision = (
        total_tp
        / (
            total_tp
            + total_fp
        )
        if (
            total_tp
            + total_fp
        )
        > 0
        else 0.0
    )

    recall = (
        total_tp
        / (
            total_tp
            + total_fn
        )
        if (
            total_tp
            + total_fn
        )
        > 0
        else 0.0
    )

    f1 = (
        2.0
        * precision
        * recall
        / (
            precision
            + recall
        )
        if (
            precision
            + recall
        )
        > 0
        else 0.0
    )

    mean_matched_iou = (
        float(
            np.mean(
                matched_ious
            )
        )
        if matched_ious
        else None
    )

    count_mae = float(
        np.mean(
            count_abs_errors
        )
    )

    count_bias = float(
        np.mean(
            count_errors
        )
    )

    zero_gt_fp_per_image = (
        float(
            zero_gt_fp
            / zero_gt_images
        )
        if zero_gt_images > 0
        else None
    )

    zero_gt_fraction_with_fp = (
        float(
            zero_gt_with_fp
            / zero_gt_images
        )
        if zero_gt_images > 0
        else None
    )

    require(
        total_tp
        + total_fn
        == sum(
            len(
                record[
                    "gt_boxes"
                ]
            )
            for record
            in records
        ),
        "Operating TP+FN does not equal GT total.",
    )

    return {
        "operating_threshold":
            float(
                operating_threshold
            ),

        "matching_iou":
            0.50,

        "tp":
            int(
                total_tp
            ),

        "fp":
            int(
                total_fp
            ),

        "fn":
            int(
                total_fn
            ),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1":
            float(
                f1
            ),

        "mean_matched_box_iou_conditional":
            mean_matched_iou,

        "count_mae":
            count_mae,

        "count_bias":
            count_bias,

        "zero_gt_images":
            int(
                zero_gt_images
            ),

        "zero_gt_fp_total":
            int(
                zero_gt_fp
            ),

        "zero_gt_fp_per_image":
            zero_gt_fp_per_image,

        "zero_gt_fraction_with_ge1_fp":
            zero_gt_fraction_with_fp,

        "per_image":
            per_image,
    }


# ============================================================
# 15. EVALUATE ONE SUBSET FROM ALREADY-COMPUTED PREDICTIONS
# ============================================================

def evaluate_subset(
    records,
    operating_threshold: float,
):

    ap = compute_all_ap(
        records
    )

    operating = (
        compute_operating_metrics(
            records=records,
            operating_threshold=(
                operating_threshold
            ),
        )
    )

    return {
        **ap,

        "operating":
            {
                key:
                    value
                for key, value
                in operating.items()
                if key != "per_image"
            },

        "per_image":
            operating[
                "per_image"
            ],
    }


# ============================================================
# 16. WRITE CSV
# ============================================================

def write_csv(
    path: Path,
    rows,
) -> None:

    require(
        len(rows) > 0,
        "Cannot write empty CSV.",
    )

    fieldnames = list(
        rows[0].keys()
    )

    with path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        for row in rows:

            writer.writerow(row)


# ============================================================
# 17. SYNTHETIC SELF-TESTS
# ============================================================

def run_self_tests():

    print(
        "=" * 72
    )

    print(
        "DASO-Net — RSC EVALUATOR SYNTHETIC SELF-TEST"
    )

    print(
        "=" * 72
    )

    # --------------------------------------------------------
    # IoU
    # --------------------------------------------------------

    require(
        abs(
            box_iou_single(
                [0, 0, 10, 10],
                [0, 0, 10, 10],
            )
            - 1.0
        )
        < 1e-12,
        "IoU identity test failed.",
    )

    require(
        box_iou_single(
            [0, 0, 10, 10],
            [20, 20, 30, 30],
        )
        == 0.0,
        "IoU disjoint test failed.",
    )

    print(
        "PASS  Box IoU unit tests"
    )

    # --------------------------------------------------------
    # Perfect AP
    # --------------------------------------------------------

    perfect_records = [
        {
            "official_test_index":
                1,

            "filename":
                "toy_1",

            "gt_boxes":
                [
                    [
                        0.0,
                        0.0,
                        10.0,
                        10.0,
                    ]
                ],

            "predictions":
                [
                    {
                        "box":
                            [
                                0.0,
                                0.0,
                                10.0,
                                10.0,
                            ],

                        "score":
                            0.90,

                        "decoder_order":
                            0,
                    }
                ],

            "dedup247_member":
                True,
        },

        {
            "official_test_index":
                2,

            "filename":
                "toy_2",

            "gt_boxes":
                [
                    [
                        20.0,
                        20.0,
                        40.0,
                        40.0,
                    ]
                ],

            "predictions":
                [
                    {
                        "box":
                            [
                                20.0,
                                20.0,
                                40.0,
                                40.0,
                            ],

                        "score":
                            0.80,

                        "decoder_order":
                            0,
                    }
                ],

            "dedup247_member":
                True,
        },
    ]

    perfect_ap = compute_all_ap(
        perfect_records
    )

    require(
        abs(
            perfect_ap[
                "box_ap50"
            ]
            - 1.0
        )
        < 1e-12,
        "Perfect AP50 test failed.",
    )

    require(
        abs(
            perfect_ap[
                "box_ap75"
            ]
            - 1.0
        )
        < 1e-12,
        "Perfect AP75 test failed.",
    )

    require(
        abs(
            perfect_ap[
                "box_ap50_95"
            ]
            - 1.0
        )
        < 1e-12,
        "Perfect AP50:95 test failed.",
    )

    print(
        "PASS  101-point AP unit test"
    )

    # --------------------------------------------------------
    # Operating metrics with a zero-GT image.
    # --------------------------------------------------------

    op_records = [
        {
            "official_test_index":
                1,

            "filename":
                "positive",

            "gt_boxes":
                [
                    [
                        0.0,
                        0.0,
                        10.0,
                        10.0,
                    ]
                ],

            "predictions":
                [
                    {
                        "box":
                            [
                                0.0,
                                0.0,
                                10.0,
                                10.0,
                            ],

                        "score":
                            0.90,

                        "decoder_order":
                            0,
                    }
                ],

            "dedup247_member":
                True,
        },

        {
            "official_test_index":
                2,

            "filename":
                "negative",

            "gt_boxes":
                [],

            "predictions":
                [
                    {
                        "box":
                            [
                                1.0,
                                1.0,
                                5.0,
                                5.0,
                            ],

                        "score":
                            0.80,

                        "decoder_order":
                            0,
                    }
                ],

            "dedup247_member":
                True,
        },
    ]

    operating = compute_operating_metrics(
        records=op_records,
        operating_threshold=0.50,
    )

    require(
        operating[
            "tp"
        ] == 1,
        "Synthetic operating TP test failed.",
    )

    require(
        operating[
            "fp"
        ] == 1,
        "Synthetic operating FP test failed.",
    )

    require(
        operating[
            "fn"
        ] == 0,
        "Synthetic operating FN test failed.",
    )

    require(
        abs(
            operating[
                "precision"
            ]
            - 0.5
        )
        < 1e-12,
        "Synthetic precision test failed.",
    )

    require(
        abs(
            operating[
                "recall"
            ]
            - 1.0
        )
        < 1e-12,
        "Synthetic recall test failed.",
    )

    require(
        abs(
            operating[
                "f1"
            ]
            - (
                2.0
                / 3.0
            )
        )
        < 1e-12,
        "Synthetic F1 test failed.",
    )

    require(
        operating[
            "zero_gt_images"
        ] == 1,
        "Synthetic zero-GT image count test failed.",
    )

    require(
        operating[
            "zero_gt_fp_per_image"
        ] == 1.0,
        "Synthetic zero-GT FP/image test failed.",
    )

    require(
        operating[
            "zero_gt_fraction_with_ge1_fp"
        ] == 1.0,
        "Synthetic zero-GT trigger-fraction "
        "test failed.",
    )

    print(
        "PASS  Frozen operating-metric unit tests"
    )

    print()

    print(
        "SELF-TEST STATUS: PASS"
    )


# ============================================================
# 18. STRUCTURAL SMOKE AUDIT
# ============================================================

def run_smoke(
    protocol,
    protocol_sha: str,
    phase5c,
    seed_index: int,
    device: torch.device,
):

    if SMOKE_OUTPUT_DIR.exists():

        raise RuntimeError(
            "Smoke output directory already exists.\n"
            "Preserve it rather than overwriting it:\n"
            f"{SMOKE_OUTPUT_DIR}"
        )

    SMOKE_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=False,
    )

    evaluator_path = Path(
        __file__
    ).resolve()

    evaluator_sha = sha256_file(
        evaluator_path
    )

    rows = load_rsc_rows(
        protocol
    )

    positive_candidates = [
        row
        for row in rows
        if len(
            row[
                "gt_boxes"
            ]
        ) > 0
    ]

    zero_candidates = [
        row
        for row in rows
        if row[
            "filename"
        ]
        == "test_113.jpg"
    ]

    require(
        len(
            positive_candidates
        ) > 0,
        "No positive-GT RSC image available "
        "for smoke audit.",
    )

    require(
        len(
            zero_candidates
        ) == 1,
        "Could not uniquely identify "
        "test_113.jpg zero-GT smoke case.",
    )

    smoke_rows = [
        positive_candidates[0],
        zero_candidates[0],
    ]

    model_info = get_model_info(
        protocol=protocol,
        seed_index=seed_index,
    )

    model = phase5c[
        "build_model"
    ](
        model_info[
            "checkpoint"
        ],
        device,
    )

    model.eval()

    decoder_spec = (
        protocol[
            "validated_evaluator"
        ][
            "decoder"
        ]
    )

    image_checks = []

    for row in smoke_rows:

        predictions, prep = infer_one(
            model=model,
            decode_instances=(
                phase5c[
                    "decode_instances"
                ]
            ),
            row=row,
            device=device,
            decoder_spec=decoder_spec,
        )

        # IMPORTANT:
        # Do not save or print scores/counts.
        # Structural validation only.
        for prediction in predictions:

            box = prediction[
                "box"
            ]

            require(
                len(box) == 4,
                "Smoke prediction box length != 4.",
            )

            x1, y1, x2, y2 = box

            require(
                0.0 <= x1
                < x2
                <= row[
                    "width"
                ],
                "Smoke prediction X coordinates "
                "outside original image bounds.",
            )

            require(
                0.0 <= y1
                < y2
                <= row[
                    "height"
                ],
                "Smoke prediction Y coordinates "
                "outside original image bounds.",
            )

            require(
                math.isfinite(
                    prediction[
                        "score"
                    ]
                ),
                "Smoke prediction has "
                "non-finite score.",
            )

        image_checks.append(
            {
                "filename":
                    row[
                        "filename"
                    ],

                "gt_case":
                    (
                        "positive_gt"
                        if len(
                            row[
                                "gt_boxes"
                            ]
                        ) > 0
                        else "zero_gt"
                    ),

                "original_shape":
                    [
                        prep[
                            "original_h"
                        ],
                        prep[
                            "original_w"
                        ],
                    ],

                "padded_shape":
                    [
                        prep[
                            "padded_h"
                        ],
                        prep[
                            "padded_w"
                        ],
                    ],

                "decoder_return_normalized":
                    True,

                "prediction_values_finite":
                    True,

                "original_coordinate_policy_valid":
                    True,

                # Deliberately no prediction count,
                # scores, AP, or operating metrics.
            }
        )

    smoke = {
        "audit_name":
            "DASO-Net RSC Evaluator Structural Smoke Audit v1",

        "status":
            "PASS",

        "purpose":
            (
                "Validate frozen RSC preprocessing, "
                "checkpoint loading, model forward pass, "
                "validated decoder integration, decoder "
                "output normalization, and original-image "
                "box-coordinate policy without reporting "
                "scientific RSC performance."
            ),

        "protocol_sha256":
            protocol_sha,

        "evaluator_script":
            str(
                evaluator_path
            ),

        "evaluator_sha256":
            evaluator_sha,

        "phase5c_evaluator_sha256":
            phase5c[
                "script_sha256"
            ],

        "phase5c_decoder_signature":
            phase5c[
                "decoder_signature"
            ],

        "seed_index_used_for_structural_smoke":
            seed_index,

        "checkpoint_sha256":
            model_info[
                "checkpoint_sha256"
            ],

        "images_checked":
            len(
                smoke_rows
            ),

        "positive_gt_case_checked":
            True,

        "zero_gt_case_checked":
            True,

        "scientific_metrics_computed":
            False,

        "threshold_sweep_performed":
            False,

        "threshold_tuning_performed":
            False,

        "calibration_performed":
            False,

        "checkpoint_selection_performed":
            False,

        "seed_selection_performed":
            False,

        "prediction_counts_saved":
            False,

        "prediction_scores_saved":
            False,

        "image_checks":
            image_checks,

        "readiness":
            "READY_TO_FREEZE_RSC_EVALUATOR_IMPLEMENTATION",
    }

    smoke_json = (
        SMOKE_OUTPUT_DIR
        / "RSC_EVALUATOR_STRUCTURAL_SMOKE_AUDIT_v1.json"
    )

    write_json(
        smoke_json,
        smoke,
    )

    smoke_sha = sha256_file(
        smoke_json
    )

    smoke_sha_record = (
        SMOKE_OUTPUT_DIR
        / "RSC_EVALUATOR_STRUCTURAL_SMOKE_AUDIT_v1.sha256.txt"
    )

    smoke_sha_record.write_text(
        f"{smoke_sha}  {smoke_json.name}\n",
        encoding="utf-8",
    )

    print(
        "=" * 72
    )

    print(
        "DASO-Net — RSC EVALUATOR STRUCTURAL SMOKE AUDIT"
    )

    print(
        "=" * 72
    )

    print()

    print(
        "PASS  Frozen protocol identity"
    )

    print(
        "PASS  Frozen checkpoint identity"
    )

    print(
        "PASS  Phase5C evaluator identity"
    )

    print(
        "PASS  Phase5C decoder signature"
    )

    print(
        "PASS  Full-original-image preprocessing"
    )

    print(
        "PASS  Minimal stride-16 padding"
    )

    print(
        "PASS  Model forward pass"
    )

    print(
        "PASS  Validated decoder integration"
    )

    print(
        "PASS  Decoder output normalization"
    )

    print(
        "PASS  Original-image coordinate policy"
    )

    print(
        "PASS  Positive-GT structural case"
    )

    print(
        "PASS  Zero-GT structural case"
    )

    print()

    print(
        "Scientific RSC metrics computed: NO"
    )

    print(
        "Prediction counts reported/saved: NO"
    )

    print(
        "Prediction scores reported/saved: NO"
    )

    print(
        "Threshold tuning: NO"
    )

    print(
        "Calibration: NO"
    )

    print(
        "Checkpoint selection: NO"
    )

    print(
        "Seed selection: NO"
    )

    print()

    print(
        "Evaluator SHA256:"
    )

    print(
        f"  {evaluator_sha}"
    )

    print()

    print(
        "Smoke audit JSON:"
    )

    print(
        f"  {smoke_json}"
    )

    print()

    print(
        "Smoke audit SHA256:"
    )

    print(
        f"  {smoke_sha}"
    )

    print()

    print(
        "READINESS:"
    )

    print(
        "  READY_TO_FREEZE_RSC_EVALUATOR_IMPLEMENTATION"
    )

    print(
        "=" * 72
    )


# ============================================================
# 19. FORMAL PER-SEED EVALUATION
# ============================================================

def run_formal_evaluation(
    protocol,
    protocol_sha: str,
    phase5c,
    seed_index: int,
    device: torch.device,
):

    rows = load_rsc_rows(
        protocol
    )

    model_info = get_model_info(
        protocol=protocol,
        seed_index=seed_index,
    )

    operating_threshold = float(
        model_info[
            "operating_threshold"
        ]
    )

    evaluator_path = Path(
        __file__
    ).resolve()

    evaluator_sha = sha256_file(
        evaluator_path
    )

    designated_root = Path(
        protocol[
            "designated_future_output_root"
        ]
    )

    designated_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    seed_dir = (
        designated_root
        / (
            f"seed{seed_index:02d}_"
            f"s{model_info['seed']}"
        )
    )

    if seed_dir.exists():

        raise RuntimeError(
            "Formal RSC seed output directory already exists.\n"
            "Preserve it rather than overwriting it:\n"
            f"{seed_dir}"
        )

    seed_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    run_state_path = (
        seed_dir
        / "RSC_RUN_STATE.json"
    )

    run_state = {
        "status":
            "RUNNING",

        "seed_index":
            seed_index,

        "seed":
            int(
                model_info[
                    "seed"
                ]
            ),

        "checkpoint_sha256":
            model_info[
                "checkpoint_sha256"
            ],

        "protocol_sha256":
            protocol_sha,

        "evaluator_sha256":
            evaluator_sha,

        "operating_threshold":
            operating_threshold,

        "threshold_tuning":
            False,

        "calibration":
            False,

        "checkpoint_selection":
            False,

        "seed_selection":
            False,
    }

    write_json(
        run_state_path,
        run_state,
    )

    model = phase5c[
        "build_model"
    ](
        model_info[
            "checkpoint"
        ],
        device,
    )

    model.eval()

    decoder_spec = (
        protocol[
            "validated_evaluator"
        ][
            "decoder"
        ]
    )

    records = []

    predictions_jsonl = (
        seed_dir
        / "rsc_predictions_decoder_floor.jsonl"
    )

    with predictions_jsonl.open(
        "w",
        encoding="utf-8",
    ) as prediction_file:

        for index, row in enumerate(
            rows,
            start=1,
        ):

            predictions, prep = infer_one(
                model=model,
                decode_instances=(
                    phase5c[
                        "decode_instances"
                    ]
                ),
                row=row,
                device=device,
                decoder_spec=decoder_spec,
            )

            record = {
                "official_test_index":
                    row[
                        "official_test_index"
                    ],

                "filename":
                    row[
                        "filename"
                    ],

                "width":
                    row[
                        "width"
                    ],

                "height":
                    row[
                        "height"
                    ],

                "gt_boxes":
                    row[
                        "gt_boxes"
                    ],

                "dedup247_member":
                    row[
                        "dedup247_member"
                    ],

                "predictions":
                    predictions,
            }

            records.append(
                record
            )

            prediction_file.write(
                json.dumps(
                    {
                        "official_test_index":
                            record[
                                "official_test_index"
                            ],

                        "filename":
                            record[
                                "filename"
                            ],

                        "width":
                            record[
                                "width"
                            ],

                        "height":
                            record[
                                "height"
                            ],

                        "dedup247_member":
                            record[
                                "dedup247_member"
                            ],

                        "gt_boxes":
                            record[
                                "gt_boxes"
                            ],

                        "predictions_decoder_floor":
                            record[
                                "predictions"
                            ],
                    },
                    allow_nan=False,
                )
                + "\n"
            )

            if (
                index % 25 == 0
                or index == 250
            ):

                print(
                    f"  inferred {index}/250"
                )

    primary_records = records

    dedup_records = [
        record
        for record in records
        if record[
            "dedup247_member"
        ]
    ]

    require(
        len(primary_records) == 250,
        "Formal primary record count != 250.",
    )

    require(
        len(dedup_records) == 247,
        "Formal dedup record count != 247.",
    )

    require(
        sum(
            len(
                record[
                    "gt_boxes"
                ]
            )
            for record
            in primary_records
        )
        == 539,
        "Formal primary GT total != 539.",
    )

    require(
        sum(
            len(
                record[
                    "gt_boxes"
                ]
            )
            for record
            in dedup_records
        )
        == 539,
        "Formal dedup GT total != 539.",
    )

    primary_result = evaluate_subset(
        records=primary_records,
        operating_threshold=(
            operating_threshold
        ),
    )

    dedup_result = evaluate_subset(
        records=dedup_records,
        operating_threshold=(
            operating_threshold
        ),
    )

    primary_per_image = (
        primary_result.pop(
            "per_image"
        )
    )

    dedup_per_image = (
        dedup_result.pop(
            "per_image"
        )
    )

    primary_csv = (
        seed_dir
        / "rsc_primary250_per_image.csv"
    )

    dedup_csv = (
        seed_dir
        / "rsc_dedup247_per_image.csv"
    )

    write_csv(
        primary_csv,
        primary_per_image,
    )

    write_csv(
        dedup_csv,
        dedup_per_image,
    )

    summary = {
        "evaluation_name":
            "DASO-Net RSC External Detection v1",

        "status":
            "PASS",

        "scientific_task":
            "class-agnostic bounding-box detection",

        "segmentation_metrics":
            False,

        "seed_index":
            seed_index,

        "seed":
            int(
                model_info[
                    "seed"
                ]
            ),

        "checkpoint":
            str(
                model_info[
                    "checkpoint"
                ]
            ),

        "checkpoint_sha256":
            model_info[
                "checkpoint_sha256"
            ],

        "successful_optimizer_updates":
            2750,

        "protocol_sha256":
            protocol_sha,

        "evaluator_sha256":
            evaluator_sha,

        "phase5c_evaluator_sha256":
            phase5c[
                "script_sha256"
            ],

        "decoder":
            decoder_spec,

        "input_protocol":
            protocol[
                "input_protocol"
            ],

        "operating_threshold":
            operating_threshold,

        "primary_official_test250":
            {
                "images":
                    250,

                "gt_boxes":
                    539,

                "zero_gt_images":
                    4,

                **primary_result,
            },

        "sensitivity_exact_dedup247":
            {
                "images":
                    247,

                "gt_boxes":
                    539,

                "zero_gt_images":
                    1,

                **dedup_result,
            },

        "scientific_policy": {
            "rsc_training":
                False,

            "rsc_crpa":
                False,

            "rsc_calibration":
                False,

            "rsc_threshold_tuning":
                False,

            "checkpoint_selection":
                False,

            "seed_selection":
                False,

            "best_seed_selection":
                False,
        },

        "artifacts": {
            "predictions_decoder_floor":
                str(
                    predictions_jsonl
                ),

            "primary_per_image":
                str(
                    primary_csv
                ),

            "dedup_per_image":
                str(
                    dedup_csv
                ),
        },
    }

    summary_path = (
        seed_dir
        / "RSC_EXTERNAL_DETECTION_SUMMARY.json"
    )

    write_json(
        summary_path,
        summary,
    )

    summary_sha = sha256_file(
        summary_path
    )

    summary_sha_record = (
        seed_dir
        / "RSC_EXTERNAL_DETECTION_SUMMARY.sha256.txt"
    )

    summary_sha_record.write_text(
        f"{summary_sha}  {summary_path.name}\n",
        encoding="utf-8",
    )

    run_state.update(
        {
            "status":
                "PASS",

            "summary":
                str(
                    summary_path
                ),

            "summary_sha256":
                summary_sha,
        }
    )

    write_json(
        run_state_path,
        run_state,
    )

    print()

    print(
        "=" * 72
    )

    print(
        "DASO-NET RSC FORMAL PER-SEED EVALUATION: PASS"
    )

    print(
        "=" * 72
    )

    print()

    print(
        f"Seed index: {seed_index}"
    )

    print(
        f"Seed:       {model_info['seed']}"
    )

    print(
        f"Frozen operating threshold: "
        f"{operating_threshold:.2f}"
    )

    print()

    print(
        "PRIMARY official test-250:"
    )

    print(
        "  Box AP50:    "
        f"{primary_result['box_ap50']:.6f}"
    )

    print(
        "  Box AP75:    "
        f"{primary_result['box_ap75']:.6f}"
    )

    print(
        "  Box AP50:95: "
        f"{primary_result['box_ap50_95']:.6f}"
    )

    print(
        "  Precision:   "
        f"{primary_result['operating']['precision']:.6f}"
    )

    print(
        "  Recall:      "
        f"{primary_result['operating']['recall']:.6f}"
    )

    print(
        "  F1:          "
        f"{primary_result['operating']['f1']:.6f}"
    )

    print()

    print(
        "SENSITIVITY exact-dedup-247:"
    )

    print(
        "  Box AP50:    "
        f"{dedup_result['box_ap50']:.6f}"
    )

    print(
        "  Box AP75:    "
        f"{dedup_result['box_ap75']:.6f}"
    )

    print(
        "  Box AP50:95: "
        f"{dedup_result['box_ap50_95']:.6f}"
    )

    print()

    print(
        "Threshold tuning: NO"
    )

    print(
        "Calibration: NO"
    )

    print(
        "Checkpoint selection: NO"
    )

    print(
        "Seed selection: NO"
    )

    print()

    print(
        "Summary JSON:"
    )

    print(
        f"  {summary_path}"
    )

    print()

    print(
        "Summary SHA256:"
    )

    print(
        f"  {summary_sha}"
    )

    print(
        "=" * 72
    )


# ============================================================
# 20. CLI
# ============================================================

def main():

    parser = argparse.ArgumentParser(
        description=(
            "DASO-Net frozen RSC external "
            "bounding-box evaluator v1"
        )
    )

    parser.add_argument(
        "--mode",
        required=True,
        choices=[
            "self-test",
            "smoke",
            "evaluate",
        ],
    )

    parser.add_argument(
        "--protocol",
        type=Path,
        default=DEFAULT_PROTOCOL,
    )

    parser.add_argument(
        "--seed-index",
        type=int,
        choices=[
            1,
            2,
            3,
        ],
        default=1,
    )

    parser.add_argument(
        "--device",
        default="cuda",
        choices=[
            "cuda",
            "cpu",
        ],
    )

    args = parser.parse_args()

    if args.mode == "self-test":

        run_self_tests()

        return

    protocol, protocol_sha = (
        load_protocol(
            args.protocol
        )
    )

    phase5c = load_phase5c_module(
        protocol
    )

    if args.device == "cuda":

        require(
            torch.cuda.is_available(),
            "CUDA requested but not available.",
        )

    device = torch.device(
        args.device
    )

    torch.manual_seed(0)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(0)

    torch.backends.cudnn.benchmark = False

    try:

        torch.backends.cudnn.deterministic = True

    except Exception:
        pass

    try:

        torch.use_deterministic_algorithms(
            True,
            warn_only=True,
        )

    except Exception:
        pass

    if args.mode == "smoke":

        run_smoke(
            protocol=protocol,
            protocol_sha=protocol_sha,
            phase5c=phase5c,
            seed_index=args.seed_index,
            device=device,
        )

        return

    if args.mode == "evaluate":

        run_formal_evaluation(
            protocol=protocol,
            protocol_sha=protocol_sha,
            phase5c=phase5c,
            seed_index=args.seed_index,
            device=device,
        )

        return

    raise RuntimeError(
        f"Unhandled mode: {args.mode}"
    )


if __name__ == "__main__":

    main()

Writing /kaggle/working/evaluate_rsc_external_v1.py


In [26]:
from pathlib import Path
import hashlib
import py_compile
import subprocess
import sys

SCRIPT = Path(
    "/kaggle/working/evaluate_rsc_external_v1.py"
)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)
    return h.hexdigest()

print("=" * 72)
print("DASO-Net — RSC EVALUATOR STATIC + SYNTHETIC AUDIT")
print("=" * 72)
print()

py_compile.compile(
    str(SCRIPT),
    doraise=True,
)

print("PASS  Python compile")
print(f"Evaluator SHA256: {sha256_file(SCRIPT)}")
print()

result = subprocess.run(
    [
        sys.executable,
        str(SCRIPT),
        "--mode",
        "self-test",
    ],
    check=False,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr.strip():
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Evaluator synthetic self-test failed "
        f"with exit code {result.returncode}"
    )

print()
print("READINESS:")
print("  READY_FOR_STRUCTURAL_RSC_SMOKE")

DASO-Net — RSC EVALUATOR STATIC + SYNTHETIC AUDIT

PASS  Python compile
Evaluator SHA256: 6163e461d7566869a273a4b963049175fa88bd15bf973aba3b62d945374406d6

DASO-Net — RSC EVALUATOR SYNTHETIC SELF-TEST
PASS  Box IoU unit tests
PASS  101-point AP unit test
PASS  Frozen operating-metric unit tests

SELF-TEST STATUS: PASS


READINESS:
  READY_FOR_STRUCTURAL_RSC_SMOKE


In [28]:
%%writefile /kaggle/working/evaluate_rsc_external_v1.py
# ============================================================
# DASO-Net — RSC EXTERNAL DETECTION EVALUATOR v1
# FIXED CHECKPOINT-LOADING / SMOKE RESUME01
#
# Scientific task:
#   class-agnostic bounding-box detection only
#
# Frozen protocol:
#   RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.json
#
# Modes:
#   self-test : synthetic metric-engine unit tests; NO RSC inference
#   smoke     : structural two-image RSC inference only; NO metrics
#   evaluate  : formal per-seed RSC evaluation
#
# IMPORTANT
# ---------
# RSC is NEVER used for:
#   training
#   CRPA
#   calibration
#   threshold tuning
#   checkpoint selection
#   seed selection
#
# PRIMARY:
#   official authors' test-250
#
# SENSITIVITY:
#   exact-dedup-247
#
# FIX IN THIS VERSION
# -------------------
# evaluate_phase5c.py exposes:
#
#     build_model(checkpoint, device)
#
# where checkpoint is the already-loaded checkpoint mapping.
#
# The previous evaluator incorrectly passed a pathlib.Path directly
# to build_model().  That failed before model construction because
# build_model() calls checkpoint.get(...).
#
# This version:
#   1. loads the frozen checkpoint with torch.load(...)
#   2. verifies that the result is mapping-like
#   3. passes that loaded object to build_model(...)
#   4. preserves the previous failed smoke directory
#   5. writes the resumed smoke audit to a new output directory
#
# No scientific protocol is changed.
# ============================================================

from __future__ import annotations

import argparse
import csv
import hashlib
import importlib.util
import inspect
import json
import math
import sys
import traceback
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import torch


# ============================================================
# 1. FROZEN IDENTITIES
# ============================================================

DEFAULT_PROTOCOL = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_external_evaluation_protocol_freeze_v1/"
    "RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "e3c87caebafe87c7671b5745a480c0edf67f444feac4278633da6addae4d1f7e"
)

PRIOR_FAILED_SMOKE_OUTPUT_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_v1_smoke_audit"
)

SMOKE_OUTPUT_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_v1_smoke_audit_resume01"
)


# ============================================================
# 2. GENERIC HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)


def require_file(path: Path, label: str) -> None:
    require(
        path.is_file(),
        f"{label} missing:\n{path}",
    )


def require_dir(path: Path, label: str) -> None:
    require(
        path.is_dir(),
        f"{label} missing:\n{path}",
    )


def read_json(path: Path):
    return json.loads(
        path.read_text(encoding="utf-8")
    )


def write_json(path: Path, obj) -> None:
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def read_csv_rows(path: Path):
    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:
        return list(csv.DictReader(f))


def as_int(value, label: str) -> int:
    try:
        number = float(value)
    except Exception as exc:
        raise RuntimeError(
            f"Could not parse integer field {label}: {value!r}"
        ) from exc

    require(
        math.isfinite(number),
        f"Non-finite integer field {label}: {value!r}",
    )

    require(
        number.is_integer(),
        f"Non-integral integer field {label}: {value!r}",
    )

    return int(number)


def safe_float(value, label: str) -> float:
    try:
        number = float(value)
    except Exception as exc:
        raise RuntimeError(
            f"Could not parse float field {label}: {value!r}"
        ) from exc

    require(
        math.isfinite(number),
        f"Non-finite float field {label}: {value!r}",
    )

    return number


# ============================================================
# 3. LOAD + VERIFY FROZEN PROTOCOL
# ============================================================

def load_protocol(path: Path):
    require_file(
        path,
        "Frozen RSC evaluation protocol",
    )

    actual_sha = sha256_file(path)

    require(
        actual_sha == EXPECTED_PROTOCOL_SHA256,
        "Frozen RSC protocol SHA mismatch.\n"
        f"Expected: {EXPECTED_PROTOCOL_SHA256}\n"
        f"Actual:   {actual_sha}",
    )

    protocol = read_json(path)

    require(
        protocol.get("status")
        == "FROZEN_BEFORE_RSC_INFERENCE",
        "RSC protocol is not marked "
        "FROZEN_BEFORE_RSC_INFERENCE.",
    )

    require(
        protocol["scientific_policy"]["rsc_threshold_tuning"]
        is False,
        "Frozen protocol unexpectedly permits "
        "RSC threshold tuning.",
    )

    require(
        protocol["metric_protocol"]["segmentation_metrics"]
        is False,
        "Frozen protocol unexpectedly enables "
        "RSC segmentation metrics.",
    )

    require(
        protocol["input_protocol"]["resize"]
        is False,
        "Frozen protocol unexpectedly permits resize.",
    )

    return protocol, actual_sha


# ============================================================
# 4. LOAD VALIDATED PHASE5C MODULE
# ============================================================

def load_phase5c_module(protocol):
    script = Path(
        protocol[
            "validated_evaluator"
        ][
            "script"
        ]
    )

    expected_sha = (
        protocol[
            "validated_evaluator"
        ][
            "whole_file_sha256"
        ]
    )

    require_file(
        script,
        "Validated Phase5C evaluator script",
    )

    actual_sha = sha256_file(script)

    require(
        actual_sha == expected_sha,
        "Validated Phase5C evaluator whole-file SHA changed.\n"
        f"Expected: {expected_sha}\n"
        f"Actual:   {actual_sha}",
    )

    training_root = script.parent.parent

    for candidate in [
        training_root,
        script.parent,
        training_root / "src",
    ]:
        if candidate.exists():
            candidate_text = str(candidate)

            if candidate_text not in sys.path:
                sys.path.insert(
                    0,
                    candidate_text,
                )

    module_name = "dasonet_phase5c_frozen_evaluator"

    spec = importlib.util.spec_from_file_location(
        module_name,
        script,
    )

    require(
        spec is not None
        and spec.loader is not None,
        "Could not create import specification for "
        "evaluate_phase5c.py.",
    )

    module = importlib.util.module_from_spec(spec)

    # Some imported modules/classes expect their module to be
    # registered while the file is executing.
    sys.modules[module_name] = module

    spec.loader.exec_module(module)

    require(
        hasattr(module, "build_model"),
        "Validated evaluator does not expose build_model.",
    )

    require(
        hasattr(module, "decode_instances"),
        "Validated evaluator does not expose decode_instances.",
    )

    build_model = module.build_model
    decode_instances = module.decode_instances

    build_signature = inspect.signature(
        build_model
    )

    decoder_signature = inspect.signature(
        decode_instances
    )

    required_decoder_parameters = {
        "outputs",
        "input_size",
        "stride",
        "score_threshold",
        "topk",
        "nms_iou",
        "mask_threshold",
        "local_peak_kernel",
    }

    missing = (
        required_decoder_parameters
        - set(decoder_signature.parameters)
    )

    require(
        not missing,
        "Validated decoder signature is missing "
        "required parameters:\n"
        + "\n".join(sorted(missing)),
    )

    return {
        "module": module,
        "build_model": build_model,
        "decode_instances": decode_instances,
        "script": script,
        "script_sha256": actual_sha,
        "build_model_signature": str(build_signature),
        "decoder_signature": str(decoder_signature),
    }


# ============================================================
# 5. MODEL / CHECKPOINT IDENTITY
# ============================================================

def get_model_info(
    protocol,
    seed_index: int,
):
    candidates = [
        model
        for model in protocol["models"]
        if int(model["seed_index"]) == seed_index
    ]

    require(
        len(candidates) == 1,
        "Expected exactly one frozen model for "
        f"seed index {seed_index}.",
    )

    model_info = candidates[0]

    checkpoint = Path(
        model_info["checkpoint"]
    )

    require_file(
        checkpoint,
        (
            "Frozen final checkpoint for "
            f"Seed-{seed_index}"
        ),
    )

    actual_sha = sha256_file(
        checkpoint
    )

    expected_sha = model_info[
        "checkpoint_sha256"
    ]

    require(
        actual_sha == expected_sha,
        "Checkpoint SHA mismatch.\n"
        f"Seed index: {seed_index}\n"
        f"Expected:   {expected_sha}\n"
        f"Actual:     {actual_sha}",
    )

    require(
        int(
            model_info[
                "successful_optimizer_updates"
            ]
        )
        == 2750,
        "Frozen checkpoint successful-update count != 2750.",
    )

    return {
        **model_info,
        "checkpoint": checkpoint,
        "checkpoint_sha256": actual_sha,
    }


def load_checkpoint_mapping(
    checkpoint_path: Path,
):
    """
    Load the checkpoint object expected by
    evaluate_phase5c.py::build_model(checkpoint, device).

    The previous RSC evaluator incorrectly passed checkpoint_path
    itself into build_model().  build_model() expects the loaded
    mapping and calls checkpoint.get(...).
    """

    require_file(
        checkpoint_path,
        "Frozen checkpoint",
    )

    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False,
        )

    except TypeError:
        # Compatibility with PyTorch versions predating the
        # weights_only argument.
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
        )

    require(
        hasattr(checkpoint, "get"),
        "Loaded checkpoint is not mapping-like and therefore "
        "cannot be passed to Phase5C build_model().\n"
        f"Checkpoint: {checkpoint_path}\n"
        f"Loaded type: {type(checkpoint)}",
    )

    return checkpoint


def build_frozen_model(
    phase5c,
    model_info,
    device: torch.device,
):
    """
    Correct Phase5C loader contract:

        checkpoint_object = torch.load(...)
        model = build_model(checkpoint_object, device)
    """

    checkpoint_object = load_checkpoint_mapping(
        model_info["checkpoint"]
    )

    model = phase5c["build_model"](
        checkpoint_object,
        device,
    )

    # Be conservative if a future loader returns (model, metadata).
    if isinstance(model, tuple):
        require(
            len(model) >= 1,
            "Phase5C build_model returned an empty tuple.",
        )

        model = model[0]

    require(
        hasattr(model, "eval"),
        "Phase5C build_model did not return a model-like object.\n"
        f"Returned type: {type(model)}",
    )

    model.eval()

    return model


# ============================================================
# 6. YOLO GT LOADING
# ============================================================

def read_yolo_gt_boxes(
    label_path: Path,
    width: int,
    height: int,
):
    lines = [
        line.strip()
        for line in label_path.read_text(
            encoding="utf-8",
            errors="strict",
        ).splitlines()
        if line.strip()
    ]

    boxes = []

    for line_number, line in enumerate(
        lines,
        start=1,
    ):
        parts = line.split()

        require(
            len(parts) == 5,
            "Unexpected YOLO label format.\n"
            f"File: {label_path}\n"
            f"Line {line_number}: {line}",
        )

        values = [
            safe_float(
                part,
                f"YOLO value line {line_number}",
            )
            for part in parts
        ]

        class_float, xc, yc, bw, bh = values

        require(
            class_float.is_integer(),
            "Non-integral YOLO class ID.\n"
            f"File: {label_path}\n"
            f"Line: {line}",
        )

        require(
            0.0 <= xc <= 1.0
            and 0.0 <= yc <= 1.0
            and 0.0 < bw <= 1.0
            and 0.0 < bh <= 1.0,
            "Invalid normalized YOLO box.\n"
            f"File: {label_path}\n"
            f"Line: {line}",
        )

        x1 = (xc - bw / 2.0) * width
        y1 = (yc - bh / 2.0) * height
        x2 = (xc + bw / 2.0) * width
        y2 = (yc + bh / 2.0) * height

        epsilon = 1e-6

        require(
            x1 >= -epsilon
            and y1 >= -epsilon
            and x2 <= width + epsilon
            and y2 <= height + epsilon
            and x2 > x1
            and y2 > y1,
            "Converted GT box invalid.\n"
            f"File: {label_path}\n"
            f"Line: {line}",
        )

        boxes.append(
            [
                float(x1),
                float(y1),
                float(x2),
                float(y2),
            ]
        )

    return boxes


# ============================================================
# 7. LOAD FROZEN RSC ROWS
# ============================================================

def load_rsc_rows(protocol):
    package_root = Path(
        protocol[
            "rsc_package"
        ][
            "package_root"
        ]
    )

    project_root = Path(
        protocol[
            "rsc_package"
        ][
            "project_payload_root"
        ]
    )

    primary_manifest = Path(
        protocol[
            "scientific_sets"
        ][
            "primary"
        ][
            "manifest"
        ]
    )

    dedup_manifest = Path(
        protocol[
            "scientific_sets"
        ][
            "sensitivity"
        ][
            "manifest"
        ]
    )

    require_dir(
        package_root,
        "RSC package root",
    )

    require_dir(
        project_root,
        "RSC scientific payload root",
    )

    require_file(
        primary_manifest,
        "RSC primary manifest",
    )

    require_file(
        dedup_manifest,
        "RSC dedup manifest",
    )

    require(
        sha256_file(primary_manifest)
        == protocol[
            "scientific_sets"
        ][
            "primary"
        ][
            "manifest_sha256"
        ],
        "Primary manifest SHA changed.",
    )

    require(
        sha256_file(dedup_manifest)
        == protocol[
            "scientific_sets"
        ][
            "sensitivity"
        ][
            "manifest_sha256"
        ],
        "Dedup manifest SHA changed.",
    )

    primary_rows = read_csv_rows(
        primary_manifest
    )

    dedup_rows = read_csv_rows(
        dedup_manifest
    )

    require(
        len(primary_rows) == 250,
        "Primary manifest rows != 250.",
    )

    require(
        len(dedup_rows) == 247,
        "Dedup manifest rows != 247.",
    )

    primary_rows = sorted(
        primary_rows,
        key=lambda row: as_int(
            row["official_test_index"],
            "official_test_index",
        ),
    )

    expected_order = list(
        range(1, 251)
    )

    actual_order = [
        as_int(
            row["official_test_index"],
            "official_test_index",
        )
        for row in primary_rows
    ]

    require(
        actual_order == expected_order,
        "Primary manifest official_test_index "
        "is not exactly 1..250.",
    )

    dedup_set = {
        str(row["filename"]).strip()
        for row in dedup_rows
    }

    enriched = []
    total_gt = 0

    for row in primary_rows:
        filename = str(
            row["filename"]
        ).strip()

        image_path = (
            project_root
            / Path(
                str(
                    row["project_image_path"]
                ).strip()
            )
        )

        label_path = (
            project_root
            / Path(
                str(
                    row["project_label_path"]
                ).strip()
            )
        )

        require_file(
            image_path,
            "RSC image",
        )

        require_file(
            label_path,
            "RSC label",
        )

        width = as_int(
            row["width"],
            "width",
        )

        height = as_int(
            row["height"],
            "height",
        )

        gt_boxes = read_yolo_gt_boxes(
            label_path=label_path,
            width=width,
            height=height,
        )

        manifest_gt_count = as_int(
            row["gt_box_count"],
            "gt_box_count",
        )

        require(
            len(gt_boxes) == manifest_gt_count,
            "GT label content differs from frozen "
            "manifest count.\n"
            f"Filename: {filename}",
        )

        total_gt += len(gt_boxes)

        enriched.append(
            {
                "official_test_index":
                    as_int(
                        row["official_test_index"],
                        "official_test_index",
                    ),

                "filename":
                    filename,

                "image_path":
                    image_path,

                "label_path":
                    label_path,

                "width":
                    width,

                "height":
                    height,

                "gt_boxes":
                    gt_boxes,

                "dedup247_member":
                    filename in dedup_set,

                "frozen_image_sha256":
                    str(
                        row["image_sha256"]
                    ).strip(),

                "frozen_label_sha256":
                    str(
                        row["label_sha256"]
                    ).strip(),
            }
        )

    require(
        total_gt == 539,
        "RSC GT-box total != 539.",
    )

    zero_gt = {
        row["filename"]
        for row in enriched
        if len(row["gt_boxes"]) == 0
    }

    require(
        zero_gt
        == {
            "test_113.jpg",
            "test_182.jpg",
            "test_183.jpg",
            "test_184.jpg",
        },
        "Unexpected RSC zero-GT image set.",
    )

    require(
        sum(
            1
            for row in enriched
            if row["dedup247_member"]
        )
        == 247,
        "RSC dedup membership count != 247.",
    )

    return enriched


# ============================================================
# 8. FROZEN INPUT PREPROCESSING
# ============================================================

def preprocess_rsc_image(
    image_path: Path,
):
    bgr = cv2.imread(
        str(image_path),
        cv2.IMREAD_COLOR,
    )

    require(
        bgr is not None,
        "Could not decode RSC image:\n"
        f"{image_path}",
    )

    original_h, original_w = (
        bgr.shape[:2]
    )

    rgb = cv2.cvtColor(
        bgr,
        cv2.COLOR_BGR2RGB,
    )

    rgb = (
        rgb.astype(np.float32)
        / 255.0
    )

    padded_h = (
        (original_h + 15)
        // 16
        * 16
    )

    padded_w = (
        (original_w + 15)
        // 16
        * 16
    )

    pad_bottom = (
        padded_h
        - original_h
    )

    pad_right = (
        padded_w
        - original_w
    )

    require(
        0 <= pad_bottom <= 15,
        "Invalid bottom padding.",
    )

    require(
        0 <= pad_right <= 15,
        "Invalid right padding.",
    )

    if (
        pad_bottom > 0
        or pad_right > 0
    ):
        rgb = np.pad(
            rgb,
            (
                (0, pad_bottom),
                (0, pad_right),
                (0, 0),
            ),
            mode="constant",
            constant_values=0.0,
        )

    tensor = (
        torch.from_numpy(rgb)
        .permute(2, 0, 1)
        .contiguous()
        .unsqueeze(0)
    )

    require(
        tensor.dtype == torch.float32,
        "RSC tensor is not float32.",
    )

    require(
        tuple(tensor.shape)
        == (
            1,
            3,
            padded_h,
            padded_w,
        ),
        "Unexpected RSC tensor shape.",
    )

    require(
        float(tensor.min()) >= 0.0
        and float(tensor.max()) <= 1.0,
        "RSC tensor outside [0,1].",
    )

    return {
        "tensor": tensor,
        "original_h": original_h,
        "original_w": original_w,
        "padded_h": padded_h,
        "padded_w": padded_w,
        "pad_bottom": pad_bottom,
        "pad_right": pad_right,
    }


# ============================================================
# 9. DECODER OUTPUT NORMALIZATION
# ============================================================

def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return (
            value.detach()
            .cpu()
            .numpy()
        )

    return np.asarray(value)


def find_first_key(
    mapping: dict,
    candidates,
):
    for key in candidates:
        if key in mapping:
            return key

    return None


def normalize_decoded_instances(decoded):
    # --------------------------------------------------------
    # Dictionary output
    # --------------------------------------------------------

    if isinstance(decoded, dict):
        nested_key = find_first_key(
            decoded,
            [
                "instances",
                "detections",
                "predictions",
            ],
        )

        if nested_key is not None:
            return normalize_decoded_instances(
                decoded[nested_key]
            )

        box_key = find_first_key(
            decoded,
            [
                "boxes",
                "bboxes",
                "xyxy",
            ],
        )

        score_key = find_first_key(
            decoded,
            [
                "scores",
                "confidences",
                "confidence",
            ],
        )

        if (
            box_key is not None
            and score_key is not None
        ):
            boxes = to_numpy(
                decoded[box_key]
            )

            scores = to_numpy(
                decoded[score_key]
            ).reshape(-1)

            if boxes.size == 0:
                boxes = np.empty(
                    (0, 4),
                    dtype=np.float32,
                )

            boxes = np.asarray(
                boxes
            ).reshape(-1, 4)

            require(
                len(boxes) == len(scores),
                "Decoder boxes/scores length mismatch.",
            )

            return [
                {
                    "box":
                        boxes[
                            i
                        ].astype(
                            float
                        ).tolist(),

                    "score":
                        float(
                            scores[i]
                        ),

                    "decoder_order":
                        i,
                }
                for i in range(
                    len(boxes)
                )
            ]

        single_box_key = find_first_key(
            decoded,
            [
                "box",
                "bbox",
                "bbox_xyxy",
                "xyxy",
            ],
        )

        single_score_key = find_first_key(
            decoded,
            [
                "score",
                "confidence",
                "conf",
            ],
        )

        if (
            single_box_key is not None
            and single_score_key is not None
        ):
            box = to_numpy(
                decoded[
                    single_box_key
                ]
            ).reshape(-1)

            require(
                len(box) == 4,
                "Single decoder box does not "
                "contain four coordinates.",
            )

            score_value = decoded[
                single_score_key
            ]

            if isinstance(
                score_value,
                torch.Tensor,
            ):
                score_value = (
                    score_value
                    .detach()
                    .cpu()
                    .item()
                )

            return [
                {
                    "box":
                        box.astype(
                            float
                        ).tolist(),

                    "score":
                        float(
                            score_value
                        ),

                    "decoder_order":
                        0,
                }
            ]

        raise RuntimeError(
            "Unsupported dictionary decoder output.\n"
            f"Keys: {sorted(decoded.keys())}"
        )

    # --------------------------------------------------------
    # List / tuple output
    # --------------------------------------------------------

    if isinstance(
        decoded,
        (list, tuple),
    ):
        if len(decoded) == 0:
            return []

        # Batch wrapper around one result.
        if (
            len(decoded) == 1
            and isinstance(
                decoded[0],
                (list, tuple),
            )
        ):
            return normalize_decoded_instances(
                decoded[0]
            )

        if (
            len(decoded) == 1
            and isinstance(
                decoded[0],
                dict,
            )
        ):
            first = decoded[0]

            if (
                find_first_key(
                    first,
                    [
                        "boxes",
                        "bboxes",
                    ],
                )
                is not None
            ):
                return normalize_decoded_instances(
                    first
                )

        # Tuple-like:
        # (boxes, scores, ...)
        if (
            len(decoded) >= 2
            and not isinstance(
                decoded[0],
                dict,
            )
        ):
            try:
                boxes = to_numpy(
                    decoded[0]
                )

                scores = to_numpy(
                    decoded[1]
                ).reshape(-1)

                if (
                    boxes.ndim >= 2
                    and boxes.shape[-1] == 4
                ):
                    boxes = boxes.reshape(
                        -1,
                        4,
                    )

                    require(
                        len(boxes)
                        == len(scores),
                        "Tuple decoder output length mismatch.",
                    )

                    return [
                        {
                            "box":
                                boxes[
                                    i
                                ].astype(
                                    float
                                ).tolist(),

                            "score":
                                float(
                                    scores[i]
                                ),

                            "decoder_order":
                                i,
                        }
                        for i in range(
                            len(boxes)
                        )
                    ]

            except Exception:
                pass

        instances = []

        for index, item in enumerate(decoded):
            if isinstance(item, dict):
                box_key = find_first_key(
                    item,
                    [
                        "box",
                        "bbox",
                        "bbox_xyxy",
                        "xyxy",
                    ],
                )

                score_key = find_first_key(
                    item,
                    [
                        "score",
                        "confidence",
                        "conf",
                    ],
                )

                require(
                    box_key is not None,
                    "Decoder instance dictionary "
                    "has no recognized box key.\n"
                    f"Keys: {sorted(item.keys())}",
                )

                require(
                    score_key is not None,
                    "Decoder instance dictionary "
                    "has no recognized score key.\n"
                    f"Keys: {sorted(item.keys())}",
                )

                box = to_numpy(
                    item[box_key]
                ).reshape(-1)

                require(
                    len(box) == 4,
                    "Decoder instance box does not "
                    "contain four coordinates.",
                )

                score_value = item[
                    score_key
                ]

                if isinstance(
                    score_value,
                    torch.Tensor,
                ):
                    score_value = (
                        score_value
                        .detach()
                        .cpu()
                        .item()
                    )

                instances.append(
                    {
                        "box":
                            box.astype(
                                float
                            ).tolist(),

                        "score":
                            float(
                                score_value
                            ),

                        "decoder_order":
                            index,
                    }
                )

                continue

            # Attribute-style instance object.
            box_value = None
            score_value = None

            for attr in [
                "box",
                "bbox",
                "bbox_xyxy",
                "xyxy",
            ]:
                if hasattr(item, attr):
                    box_value = getattr(
                        item,
                        attr,
                    )
                    break

            for attr in [
                "score",
                "confidence",
                "conf",
            ]:
                if hasattr(item, attr):
                    score_value = getattr(
                        item,
                        attr,
                    )
                    break

            require(
                box_value is not None
                and score_value is not None,
                "Unsupported decoder instance object "
                f"type: {type(item)}",
            )

            box = to_numpy(
                box_value
            ).reshape(-1)

            require(
                len(box) == 4,
                "Attribute-based decoder instance "
                "box length != 4.",
            )

            if isinstance(
                score_value,
                torch.Tensor,
            ):
                score_value = (
                    score_value
                    .detach()
                    .cpu()
                    .item()
                )

            instances.append(
                {
                    "box":
                        box.astype(
                            float
                        ).tolist(),

                    "score":
                        float(
                            score_value
                        ),

                    "decoder_order":
                        index,
                }
            )

        return instances

    raise RuntimeError(
        "Unsupported decoder output type: "
        f"{type(decoded)}"
    )


# ============================================================
# 10. ORIGINAL-IMAGE COORDINATE POLICY
# ============================================================

def apply_original_image_box_policy(
    predictions,
    original_w: int,
    original_h: int,
):
    retained = []

    for item in predictions:
        box = np.asarray(
            item["box"],
            dtype=np.float64,
        ).reshape(-1)

        require(
            len(box) == 4,
            "Decoded prediction box length != 4.",
        )

        score = float(
            item["score"]
        )

        require(
            np.all(np.isfinite(box)),
            "Non-finite decoded box.",
        )

        require(
            math.isfinite(score),
            "Non-finite decoded score.",
        )

        x1, y1, x2, y2 = [
            float(value)
            for value in box
        ]

        require(
            x2 > x1
            and y2 > y1,
            "Decoded box is degenerate before "
            "original-image clipping.",
        )

        cx = (
            x1 + x2
        ) / 2.0

        cy = (
            y1 + y2
        ) / 2.0

        # Frozen protocol:
        # predictions centered wholly in the padding region
        # are not scientific predictions for the original image.
        if not (
            0.0 <= cx < original_w
            and 0.0 <= cy < original_h
        ):
            continue

        x1 = max(
            0.0,
            min(
                float(original_w),
                x1,
            ),
        )

        y1 = max(
            0.0,
            min(
                float(original_h),
                y1,
            ),
        )

        x2 = max(
            0.0,
            min(
                float(original_w),
                x2,
            ),
        )

        y2 = max(
            0.0,
            min(
                float(original_h),
                y2,
            ),
        )

        if (
            x2 <= x1
            or y2 <= y1
        ):
            continue

        retained.append(
            {
                "box":
                    [
                        x1,
                        y1,
                        x2,
                        y2,
                    ],

                "score":
                    score,

                "decoder_order":
                    int(
                        item["decoder_order"]
                    ),
            }
        )

    retained.sort(
        key=lambda item: (
            -item["score"],
            item["decoder_order"],
        )
    )

    return retained


# ============================================================
# 11. ONE-IMAGE MODEL INFERENCE
# ============================================================

def infer_one(
    model,
    decode_instances,
    row,
    device: torch.device,
    decoder_spec,
):
    prep = preprocess_rsc_image(
        row["image_path"]
    )

    require(
        prep["original_w"]
        == row["width"]
        and prep["original_h"]
        == row["height"],
        "Decoded image dimensions differ "
        "from frozen manifest.",
    )

    tensor = (
        prep["tensor"]
        .to(
            device=device,
            dtype=torch.float32,
            non_blocking=False,
        )
    )

    with torch.inference_mode():
        outputs = model(tensor)

        decoded = decode_instances(
            outputs=outputs,
            input_size=(
                prep["padded_h"],
                prep["padded_w"],
            ),
            stride=int(
                decoder_spec["stride"]
            ),
            score_threshold=float(
                decoder_spec[
                    "decode_min_score"
                ]
            ),
            topk=int(
                decoder_spec["topk"]
            ),
            nms_iou=float(
                decoder_spec["nms_iou"]
            ),
            mask_threshold=float(
                decoder_spec[
                    "mask_threshold"
                ]
            ),
            local_peak_kernel=int(
                decoder_spec[
                    "local_peak_kernel"
                ]
            ),
        )

    normalized = normalize_decoded_instances(
        decoded
    )

    predictions = (
        apply_original_image_box_policy(
            predictions=normalized,
            original_w=prep[
                "original_w"
            ],
            original_h=prep[
                "original_h"
            ],
        )
    )

    decoder_floor = float(
        decoder_spec[
            "decode_min_score"
        ]
    )

    for prediction in predictions:
        require(
            prediction["score"]
            >= decoder_floor - 1e-8,
            "Decoded prediction fell below frozen "
            "decoder score floor.",
        )

    return predictions, prep


# ============================================================
# 12. BOX IoU
# ============================================================

def box_iou_single(
    box_a,
    box_b,
) -> float:
    ax1, ay1, ax2, ay2 = [
        float(x)
        for x in box_a
    ]

    bx1, by1, bx2, by2 = [
        float(x)
        for x in box_b
    ]

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(
        0.0,
        inter_x2 - inter_x1,
    )

    inter_h = max(
        0.0,
        inter_y2 - inter_y1,
    )

    intersection = (
        inter_w
        * inter_h
    )

    area_a = max(
        0.0,
        ax2 - ax1,
    ) * max(
        0.0,
        ay2 - ay1,
    )

    area_b = max(
        0.0,
        bx2 - bx1,
    ) * max(
        0.0,
        by2 - by1,
    )

    union = (
        area_a
        + area_b
        - intersection
    )

    if union <= 0.0:
        return 0.0

    return float(
        intersection / union
    )


# ============================================================
# 13. ONE-IMAGE MATCHING
# ============================================================

def match_image_predictions(
    predictions,
    gt_boxes,
    iou_threshold: float,
):
    matched_gt = set()
    results = []

    ordered_predictions = sorted(
        predictions,
        key=lambda item: (
            -item["score"],
            item["decoder_order"],
        ),
    )

    for prediction in ordered_predictions:
        best_iou = -1.0
        best_gt_index = None

        for gt_index, gt_box in enumerate(
            gt_boxes
        ):
            if gt_index in matched_gt:
                continue

            iou = box_iou_single(
                prediction["box"],
                gt_box,
            )

            if iou > best_iou:
                best_iou = iou
                best_gt_index = gt_index

        if (
            best_gt_index is not None
            and best_iou >= iou_threshold
        ):
            matched_gt.add(
                best_gt_index
            )

            results.append(
                {
                    "prediction":
                        prediction,

                    "tp":
                        1,

                    "fp":
                        0,

                    "matched_gt_index":
                        best_gt_index,

                    "matched_iou":
                        float(best_iou),
                }
            )

        else:
            results.append(
                {
                    "prediction":
                        prediction,

                    "tp":
                        0,

                    "fp":
                        1,

                    "matched_gt_index":
                        None,

                    "matched_iou":
                        None,
                }
            )

    fn = (
        len(gt_boxes)
        - len(matched_gt)
    )

    return results, fn


# ============================================================
# 14. GLOBAL 101-POINT AP
# ============================================================

def compute_ap_at_iou(
    records,
    iou_threshold: float,
):
    total_gt = sum(
        len(record["gt_boxes"])
        for record in records
    )

    require(
        total_gt > 0,
        "AP evaluation has zero total GT boxes.",
    )

    pooled_predictions = []

    for record in records:
        for prediction in record[
            "predictions"
        ]:
            pooled_predictions.append(
                {
                    "official_test_index":
                        record[
                            "official_test_index"
                        ],

                    "filename":
                        record["filename"],

                    "gt_boxes":
                        record["gt_boxes"],

                    "prediction":
                        prediction,
                }
            )

    pooled_predictions.sort(
        key=lambda item: (
            -item[
                "prediction"
            ][
                "score"
            ],
            item[
                "official_test_index"
            ],
            item[
                "prediction"
            ][
                "decoder_order"
            ],
        )
    )

    matched_by_image = defaultdict(set)

    tp_flags = []
    fp_flags = []

    for item in pooled_predictions:
        filename = item["filename"]
        prediction = item["prediction"]
        gt_boxes = item["gt_boxes"]

        best_iou = -1.0
        best_gt_index = None

        for gt_index, gt_box in enumerate(
            gt_boxes
        ):
            if (
                gt_index
                in matched_by_image[
                    filename
                ]
            ):
                continue

            iou = box_iou_single(
                prediction["box"],
                gt_box,
            )

            if iou > best_iou:
                best_iou = iou
                best_gt_index = gt_index

        if (
            best_gt_index is not None
            and best_iou >= iou_threshold
        ):
            matched_by_image[
                filename
            ].add(
                best_gt_index
            )

            tp_flags.append(1.0)
            fp_flags.append(0.0)

        else:
            tp_flags.append(0.0)
            fp_flags.append(1.0)

    if len(tp_flags) == 0:
        return {
            "iou_threshold":
                float(iou_threshold),

            "ap":
                0.0,

            "total_gt":
                total_gt,

            "total_predictions":
                0,
        }

    cumulative_tp = np.cumsum(
        np.asarray(
            tp_flags,
            dtype=np.float64,
        )
    )

    cumulative_fp = np.cumsum(
        np.asarray(
            fp_flags,
            dtype=np.float64,
        )
    )

    recall = (
        cumulative_tp
        / float(total_gt)
    )

    precision = (
        cumulative_tp
        / np.maximum(
            cumulative_tp
            + cumulative_fp,
            1e-12,
        )
    )

    recall_grid = np.linspace(
        0.0,
        1.0,
        101,
    )

    interpolated = []

    for recall_target in recall_grid:
        eligible = (
            recall >= recall_target
        )

        if np.any(eligible):
            interpolated.append(
                float(
                    np.max(
                        precision[
                            eligible
                        ]
                    )
                )
            )
        else:
            interpolated.append(
                0.0
            )

    ap = float(
        np.mean(interpolated)
    )

    return {
        "iou_threshold":
            float(iou_threshold),

        "ap":
            ap,

        "total_gt":
            total_gt,

        "total_predictions":
            len(
                pooled_predictions
            ),
    }


def compute_all_ap(records):
    thresholds = [
        round(
            0.50 + 0.05 * i,
            2,
        )
        for i in range(10)
    ]

    results = {}

    for threshold in thresholds:
        metric = compute_ap_at_iou(
            records=records,
            iou_threshold=threshold,
        )

        results[
            f"{threshold:.2f}"
        ] = metric

    ap50 = results["0.50"]["ap"]
    ap75 = results["0.75"]["ap"]

    ap5095 = float(
        np.mean(
            [
                results[
                    f"{threshold:.2f}"
                ][
                    "ap"
                ]
                for threshold in thresholds
            ]
        )
    )

    return {
        "box_ap50": ap50,
        "box_ap75": ap75,
        "box_ap50_95": ap5095,
        "by_iou": results,
    }


# ============================================================
# 15. FROZEN OPERATING METRICS
# ============================================================

def compute_operating_metrics(
    records,
    operating_threshold: float,
):
    total_tp = 0
    total_fp = 0
    total_fn = 0

    matched_ious = []

    count_errors = []
    count_abs_errors = []

    zero_gt_fp = 0
    zero_gt_images = 0
    zero_gt_with_fp = 0

    per_image = []

    for record in records:
        predictions = [
            prediction
            for prediction
            in record["predictions"]
            if prediction["score"]
            >= operating_threshold
        ]

        matches, fn = (
            match_image_predictions(
                predictions=predictions,
                gt_boxes=record[
                    "gt_boxes"
                ],
                iou_threshold=0.50,
            )
        )

        tp = sum(
            item["tp"]
            for item in matches
        )

        fp = sum(
            item["fp"]
            for item in matches
        )

        for item in matches:
            if (
                item["matched_iou"]
                is not None
            ):
                matched_ious.append(
                    float(
                        item[
                            "matched_iou"
                        ]
                    )
                )

        gt_count = len(
            record["gt_boxes"]
        )

        pred_count = len(
            predictions
        )

        count_error = (
            pred_count
            - gt_count
        )

        count_abs_error = abs(
            count_error
        )

        count_errors.append(
            float(count_error)
        )

        count_abs_errors.append(
            float(count_abs_error)
        )

        if gt_count == 0:
            zero_gt_images += 1
            zero_gt_fp += fp

            if fp >= 1:
                zero_gt_with_fp += 1

        total_tp += tp
        total_fp += fp
        total_fn += fn

        max_decoded_score = (
            max(
                prediction["score"]
                for prediction
                in record["predictions"]
            )
            if record["predictions"]
            else None
        )

        per_image.append(
            {
                "official_test_index":
                    record[
                        "official_test_index"
                    ],

                "filename":
                    record["filename"],

                "subset_primary":
                    True,

                "subset_dedup247":
                    bool(
                        record[
                            "dedup247_member"
                        ]
                    ),

                "gt_count":
                    gt_count,

                "pred_count_operating":
                    pred_count,

                "tp_operating_iou50":
                    tp,

                "fp_operating_iou50":
                    fp,

                "fn_operating_iou50":
                    fn,

                "count_error":
                    count_error,

                "count_abs_error":
                    count_abs_error,

                "max_decoded_score":
                    max_decoded_score,
            }
        )

    precision = (
        total_tp
        / (
            total_tp
            + total_fp
        )
        if (
            total_tp
            + total_fp
        ) > 0
        else 0.0
    )

    recall = (
        total_tp
        / (
            total_tp
            + total_fn
        )
        if (
            total_tp
            + total_fn
        ) > 0
        else 0.0
    )

    f1 = (
        2.0
        * precision
        * recall
        / (
            precision
            + recall
        )
        if (
            precision
            + recall
        ) > 0
        else 0.0
    )

    mean_matched_iou = (
        float(
            np.mean(
                matched_ious
            )
        )
        if matched_ious
        else None
    )

    count_mae = float(
        np.mean(
            count_abs_errors
        )
    )

    count_bias = float(
        np.mean(
            count_errors
        )
    )

    zero_gt_fp_per_image = (
        float(
            zero_gt_fp
            / zero_gt_images
        )
        if zero_gt_images > 0
        else None
    )

    zero_gt_fraction_with_fp = (
        float(
            zero_gt_with_fp
            / zero_gt_images
        )
        if zero_gt_images > 0
        else None
    )

    require(
        total_tp + total_fn
        == sum(
            len(record["gt_boxes"])
            for record in records
        ),
        "Operating TP+FN does not equal GT total.",
    )

    return {
        "operating_threshold":
            float(
                operating_threshold
            ),

        "matching_iou":
            0.50,

        "tp":
            int(total_tp),

        "fp":
            int(total_fp),

        "fn":
            int(total_fn),

        "precision":
            float(precision),

        "recall":
            float(recall),

        "f1":
            float(f1),

        "mean_matched_box_iou_conditional":
            mean_matched_iou,

        "count_mae":
            count_mae,

        "count_bias":
            count_bias,

        "zero_gt_images":
            int(zero_gt_images),

        "zero_gt_fp_total":
            int(zero_gt_fp),

        "zero_gt_fp_per_image":
            zero_gt_fp_per_image,

        "zero_gt_fraction_with_ge1_fp":
            zero_gt_fraction_with_fp,

        "per_image":
            per_image,
    }


# ============================================================
# 16. EVALUATE SUBSET
# ============================================================

def evaluate_subset(
    records,
    operating_threshold: float,
):
    ap = compute_all_ap(records)

    operating = (
        compute_operating_metrics(
            records=records,
            operating_threshold=(
                operating_threshold
            ),
        )
    )

    return {
        **ap,

        "operating":
            {
                key: value
                for key, value
                in operating.items()
                if key != "per_image"
            },

        "per_image":
            operating["per_image"],
    }


# ============================================================
# 17. CSV WRITER
# ============================================================

def write_csv(
    path: Path,
    rows,
) -> None:
    require(
        len(rows) > 0,
        "Cannot write empty CSV.",
    )

    fieldnames = list(
        rows[0].keys()
    )

    with path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        for row in rows:
            writer.writerow(row)


# ============================================================
# 18. SYNTHETIC METRIC SELF-TESTS
# ============================================================

def run_self_tests():
    print("=" * 72)
    print(
        "DASO-Net — RSC EVALUATOR SYNTHETIC SELF-TEST"
    )
    print("=" * 72)

    require(
        abs(
            box_iou_single(
                [0, 0, 10, 10],
                [0, 0, 10, 10],
            )
            - 1.0
        )
        < 1e-12,
        "IoU identity test failed.",
    )

    require(
        box_iou_single(
            [0, 0, 10, 10],
            [20, 20, 30, 30],
        )
        == 0.0,
        "IoU disjoint test failed.",
    )

    print(
        "PASS  Box IoU unit tests"
    )

    perfect_records = [
        {
            "official_test_index": 1,
            "filename": "toy_1",
            "gt_boxes": [
                [0.0, 0.0, 10.0, 10.0]
            ],
            "predictions": [
                {
                    "box":
                        [
                            0.0,
                            0.0,
                            10.0,
                            10.0,
                        ],
                    "score": 0.90,
                    "decoder_order": 0,
                }
            ],
            "dedup247_member": True,
        },
        {
            "official_test_index": 2,
            "filename": "toy_2",
            "gt_boxes": [
                [
                    20.0,
                    20.0,
                    40.0,
                    40.0,
                ]
            ],
            "predictions": [
                {
                    "box":
                        [
                            20.0,
                            20.0,
                            40.0,
                            40.0,
                        ],
                    "score": 0.80,
                    "decoder_order": 0,
                }
            ],
            "dedup247_member": True,
        },
    ]

    perfect_ap = compute_all_ap(
        perfect_records
    )

    require(
        abs(
            perfect_ap["box_ap50"]
            - 1.0
        )
        < 1e-12,
        "Perfect AP50 test failed.",
    )

    require(
        abs(
            perfect_ap["box_ap75"]
            - 1.0
        )
        < 1e-12,
        "Perfect AP75 test failed.",
    )

    require(
        abs(
            perfect_ap["box_ap50_95"]
            - 1.0
        )
        < 1e-12,
        "Perfect AP50:95 test failed.",
    )

    print(
        "PASS  101-point AP unit test"
    )

    op_records = [
        {
            "official_test_index": 1,
            "filename": "positive",
            "gt_boxes": [
                [0.0, 0.0, 10.0, 10.0]
            ],
            "predictions": [
                {
                    "box":
                        [
                            0.0,
                            0.0,
                            10.0,
                            10.0,
                        ],
                    "score": 0.90,
                    "decoder_order": 0,
                }
            ],
            "dedup247_member": True,
        },
        {
            "official_test_index": 2,
            "filename": "negative",
            "gt_boxes": [],
            "predictions": [
                {
                    "box":
                        [
                            1.0,
                            1.0,
                            5.0,
                            5.0,
                        ],
                    "score": 0.80,
                    "decoder_order": 0,
                }
            ],
            "dedup247_member": True,
        },
    ]

    operating = (
        compute_operating_metrics(
            records=op_records,
            operating_threshold=0.50,
        )
    )

    require(
        operating["tp"] == 1,
        "Synthetic operating TP test failed.",
    )

    require(
        operating["fp"] == 1,
        "Synthetic operating FP test failed.",
    )

    require(
        operating["fn"] == 0,
        "Synthetic operating FN test failed.",
    )

    require(
        abs(
            operating["precision"]
            - 0.5
        )
        < 1e-12,
        "Synthetic precision test failed.",
    )

    require(
        abs(
            operating["recall"]
            - 1.0
        )
        < 1e-12,
        "Synthetic recall test failed.",
    )

    require(
        abs(
            operating["f1"]
            - (2.0 / 3.0)
        )
        < 1e-12,
        "Synthetic F1 test failed.",
    )

    require(
        operating["zero_gt_images"]
        == 1,
        "Synthetic zero-GT image count test failed.",
    )

    require(
        operating["zero_gt_fp_per_image"]
        == 1.0,
        "Synthetic zero-GT FP/image test failed.",
    )

    require(
        operating[
            "zero_gt_fraction_with_ge1_fp"
        ]
        == 1.0,
        "Synthetic zero-GT trigger-fraction "
        "test failed.",
    )

    print(
        "PASS  Frozen operating-metric unit tests"
    )

    print()
    print(
        "SELF-TEST STATUS: PASS"
    )


# ============================================================
# 19. STRUCTURAL SMOKE AUDIT — RESUME01
# ============================================================

def run_smoke(
    protocol,
    protocol_sha: str,
    phase5c,
    seed_index: int,
    device: torch.device,
):
    # Preserve previous failed smoke attempt.
    if PRIOR_FAILED_SMOKE_OUTPUT_DIR.exists():
        print(
            "INFO  Prior failed smoke directory preserved:"
        )
        print(
            f"      {PRIOR_FAILED_SMOKE_OUTPUT_DIR}"
        )

    if SMOKE_OUTPUT_DIR.exists():
        raise RuntimeError(
            "Resume01 smoke output directory already exists.\n"
            "Preserve it rather than overwriting it:\n"
            f"{SMOKE_OUTPUT_DIR}"
        )

    SMOKE_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=False,
    )

    evaluator_path = Path(
        __file__
    ).resolve()

    evaluator_sha = sha256_file(
        evaluator_path
    )

    smoke_json = (
        SMOKE_OUTPUT_DIR
        / "RSC_EVALUATOR_STRUCTURAL_SMOKE_AUDIT_RESUME01.json"
    )

    smoke_sha_record = (
        SMOKE_OUTPUT_DIR
        / "RSC_EVALUATOR_STRUCTURAL_SMOKE_AUDIT_RESUME01.sha256.txt"
    )

    completed_forward_passes = 0

    try:
        rows = load_rsc_rows(
            protocol
        )

        positive_candidates = [
            row
            for row in rows
            if len(row["gt_boxes"]) > 0
        ]

        zero_candidates = [
            row
            for row in rows
            if row["filename"]
            == "test_113.jpg"
        ]

        require(
            len(positive_candidates) > 0,
            "No positive-GT RSC image available "
            "for smoke audit.",
        )

        require(
            len(zero_candidates) == 1,
            "Could not uniquely identify "
            "test_113.jpg zero-GT smoke case.",
        )

        smoke_rows = [
            positive_candidates[0],
            zero_candidates[0],
        ]

        model_info = get_model_info(
            protocol=protocol,
            seed_index=seed_index,
        )

        # ----------------------------------------------------
        # FIXED INTERFACE:
        #
        # torch.load(checkpoint path)
        #       ↓
        # checkpoint mapping
        #       ↓
        # build_model(checkpoint mapping, device)
        # ----------------------------------------------------

        model = build_frozen_model(
            phase5c=phase5c,
            model_info=model_info,
            device=device,
        )

        decoder_spec = (
            protocol[
                "validated_evaluator"
            ][
                "decoder"
            ]
        )

        image_checks = []

        for row in smoke_rows:
            predictions, prep = infer_one(
                model=model,
                decode_instances=(
                    phase5c[
                        "decode_instances"
                    ]
                ),
                row=row,
                device=device,
                decoder_spec=decoder_spec,
            )

            completed_forward_passes += 1

            # Structural validation only.
            # Do not print or save scores/counts.
            for prediction in predictions:
                box = prediction["box"]

                require(
                    len(box) == 4,
                    "Smoke prediction box length != 4.",
                )

                x1, y1, x2, y2 = box

                require(
                    0.0 <= x1
                    < x2
                    <= row["width"],
                    "Smoke prediction X coordinates "
                    "outside original image bounds.",
                )

                require(
                    0.0 <= y1
                    < y2
                    <= row["height"],
                    "Smoke prediction Y coordinates "
                    "outside original image bounds.",
                )

                require(
                    math.isfinite(
                        prediction["score"]
                    ),
                    "Smoke prediction has "
                    "non-finite score.",
                )

            image_checks.append(
                {
                    "filename":
                        row["filename"],

                    "gt_case":
                        (
                            "positive_gt"
                            if len(
                                row["gt_boxes"]
                            ) > 0
                            else "zero_gt"
                        ),

                    "original_shape":
                        [
                            prep["original_h"],
                            prep["original_w"],
                        ],

                    "padded_shape":
                        [
                            prep["padded_h"],
                            prep["padded_w"],
                        ],

                    "decoder_return_normalized":
                        True,

                    "prediction_values_finite":
                        True,

                    "original_coordinate_policy_valid":
                        True,

                    # Deliberately absent:
                    # prediction count
                    # scores
                    # AP
                    # F1
                    # any scientific metric
                }
            )

        smoke = {
            "audit_name":
                (
                    "DASO-Net RSC Evaluator "
                    "Structural Smoke Audit Resume01"
                ),

            "status":
                "PASS",

            "reason_for_resume01":
                (
                    "Previous smoke failed before model "
                    "construction because a PosixPath was "
                    "passed to Phase5C build_model() where "
                    "a loaded checkpoint mapping was required."
                ),

            "prior_failure_scientific_status":
                (
                    "Infrastructure/interface failure before "
                    "model forward pass; no prior RSC "
                    "prediction was produced."
                ),

            "prior_failed_smoke_directory":
                str(
                    PRIOR_FAILED_SMOKE_OUTPUT_DIR
                ),

            "prior_failed_smoke_directory_preserved":
                PRIOR_FAILED_SMOKE_OUTPUT_DIR.exists(),

            "purpose":
                (
                    "Validate frozen RSC preprocessing, "
                    "checkpoint loading, Phase5C model "
                    "construction, model forward pass, "
                    "validated decoder integration, decoder "
                    "output normalization, and original-image "
                    "box-coordinate policy without reporting "
                    "scientific RSC performance."
                ),

            "protocol_sha256":
                protocol_sha,

            "evaluator_script":
                str(evaluator_path),

            "evaluator_sha256":
                evaluator_sha,

            "phase5c_evaluator_sha256":
                phase5c[
                    "script_sha256"
                ],

            "phase5c_build_model_signature":
                phase5c[
                    "build_model_signature"
                ],

            "phase5c_decoder_signature":
                phase5c[
                    "decoder_signature"
                ],

            "checkpoint_loading_contract":
                (
                    "torch.load(checkpoint_path) -> "
                    "loaded mapping -> "
                    "build_model(loaded_mapping, device)"
                ),

            "seed_index_used_for_structural_smoke":
                seed_index,

            "checkpoint_sha256":
                model_info[
                    "checkpoint_sha256"
                ],

            "images_checked":
                len(smoke_rows),

            "model_forward_passes":
                completed_forward_passes,

            "positive_gt_case_checked":
                True,

            "zero_gt_case_checked":
                True,

            "scientific_metrics_computed":
                False,

            "threshold_sweep_performed":
                False,

            "threshold_tuning_performed":
                False,

            "calibration_performed":
                False,

            "checkpoint_selection_performed":
                False,

            "seed_selection_performed":
                False,

            "prediction_counts_saved":
                False,

            "prediction_scores_saved":
                False,

            "image_checks":
                image_checks,

            "readiness":
                (
                    "READY_TO_FREEZE_"
                    "RSC_EVALUATOR_IMPLEMENTATION"
                ),
        }

        write_json(
            smoke_json,
            smoke,
        )

        smoke_sha = sha256_file(
            smoke_json
        )

        smoke_sha_record.write_text(
            (
                f"{smoke_sha}  "
                f"{smoke_json.name}\n"
            ),
            encoding="utf-8",
        )

        print("=" * 72)
        print(
            "DASO-Net — RSC EVALUATOR "
            "STRUCTURAL SMOKE AUDIT RESUME01"
        )
        print("=" * 72)
        print()

        print(
            "PASS  Prior failed smoke preserved"
        )

        print(
            "PASS  Frozen protocol identity"
        )

        print(
            "PASS  Frozen checkpoint identity"
        )

        print(
            "PASS  Checkpoint loaded as mapping"
        )

        print(
            "PASS  Phase5C build_model interface"
        )

        print(
            "PASS  Phase5C evaluator identity"
        )

        print(
            "PASS  Phase5C decoder signature"
        )

        print(
            "PASS  Full-original-image preprocessing"
        )

        print(
            "PASS  Minimal stride-16 padding"
        )

        print(
            "PASS  Model forward pass"
        )

        print(
            "PASS  Validated decoder integration"
        )

        print(
            "PASS  Decoder output normalization"
        )

        print(
            "PASS  Original-image coordinate policy"
        )

        print(
            "PASS  Positive-GT structural case"
        )

        print(
            "PASS  Zero-GT structural case"
        )

        print()

        print(
            "Scientific RSC metrics computed: NO"
        )

        print(
            "Prediction counts reported/saved: NO"
        )

        print(
            "Prediction scores reported/saved: NO"
        )

        print(
            "Threshold tuning: NO"
        )

        print(
            "Calibration: NO"
        )

        print(
            "Checkpoint selection: NO"
        )

        print(
            "Seed selection: NO"
        )

        print()

        print(
            "Evaluator SHA256:"
        )

        print(
            f"  {evaluator_sha}"
        )

        print()

        print(
            "Smoke audit JSON:"
        )

        print(
            f"  {smoke_json}"
        )

        print()

        print(
            "Smoke audit SHA256:"
        )

        print(
            f"  {smoke_sha}"
        )

        print()

        print(
            "READINESS:"
        )

        print(
            "  READY_TO_FREEZE_"
            "RSC_EVALUATOR_IMPLEMENTATION"
        )

        print("=" * 72)

    except Exception as exc:
        failure = {
            "audit_name":
                (
                    "DASO-Net RSC Evaluator "
                    "Structural Smoke Audit Resume01"
                ),

            "status":
                "FAIL",

            "protocol_sha256":
                protocol_sha,

            "evaluator_sha256":
                evaluator_sha,

            "completed_model_forward_passes":
                completed_forward_passes,

            "scientific_metrics_computed":
                False,

            "threshold_tuning_performed":
                False,

            "calibration_performed":
                False,

            "checkpoint_selection_performed":
                False,

            "seed_selection_performed":
                False,

            "error_type":
                type(exc).__name__,

            "error":
                str(exc),

            "traceback":
                traceback.format_exc(),

            "readiness":
                "NOT_READY_FOR_FORMAL_RSC_EVALUATION",
        }

        write_json(
            smoke_json,
            failure,
        )

        failure_sha = sha256_file(
            smoke_json
        )

        smoke_sha_record.write_text(
            (
                f"{failure_sha}  "
                f"{smoke_json.name}\n"
            ),
            encoding="utf-8",
        )

        print()
        print("=" * 72)
        print(
            "RSC STRUCTURAL SMOKE RESUME01: FAIL"
        )
        print("=" * 72)
        print()

        print(
            "Failure preserved:"
        )

        print(
            f"  {smoke_json}"
        )

        print()

        print(
            "Completed model forward passes:"
        )

        print(
            f"  {completed_forward_passes}"
        )

        print()

        print(
            "Scientific metrics computed: NO"
        )

        print(
            "Threshold tuning: NO"
        )

        raise


# ============================================================
# 20. FORMAL PER-SEED EVALUATION
# ============================================================

def run_formal_evaluation(
    protocol,
    protocol_sha: str,
    phase5c,
    seed_index: int,
    device: torch.device,
):
    rows = load_rsc_rows(
        protocol
    )

    model_info = get_model_info(
        protocol=protocol,
        seed_index=seed_index,
    )

    operating_threshold = float(
        model_info[
            "operating_threshold"
        ]
    )

    evaluator_path = Path(
        __file__
    ).resolve()

    evaluator_sha = sha256_file(
        evaluator_path
    )

    designated_root = Path(
        protocol[
            "designated_future_output_root"
        ]
    )

    designated_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    seed_dir = (
        designated_root
        / (
            f"seed{seed_index:02d}_"
            f"s{model_info['seed']}"
        )
    )

    if seed_dir.exists():
        raise RuntimeError(
            "Formal RSC seed output directory already exists.\n"
            "Preserve it rather than overwriting it:\n"
            f"{seed_dir}"
        )

    seed_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    run_state_path = (
        seed_dir
        / "RSC_RUN_STATE.json"
    )

    run_state = {
        "status":
            "RUNNING",

        "seed_index":
            seed_index,

        "seed":
            int(model_info["seed"]),

        "checkpoint_sha256":
            model_info[
                "checkpoint_sha256"
            ],

        "protocol_sha256":
            protocol_sha,

        "evaluator_sha256":
            evaluator_sha,

        "operating_threshold":
            operating_threshold,

        "images_completed":
            0,

        "threshold_tuning":
            False,

        "calibration":
            False,

        "checkpoint_selection":
            False,

        "seed_selection":
            False,
    }

    write_json(
        run_state_path,
        run_state,
    )

    try:
        # Correct loaded-checkpoint contract.
        model = build_frozen_model(
            phase5c=phase5c,
            model_info=model_info,
            device=device,
        )

        decoder_spec = (
            protocol[
                "validated_evaluator"
            ][
                "decoder"
            ]
        )

        records = []

        predictions_jsonl = (
            seed_dir
            / "rsc_predictions_decoder_floor.jsonl"
        )

        with predictions_jsonl.open(
            "w",
            encoding="utf-8",
        ) as prediction_file:
            for index, row in enumerate(
                rows,
                start=1,
            ):
                predictions, prep = infer_one(
                    model=model,
                    decode_instances=(
                        phase5c[
                            "decode_instances"
                        ]
                    ),
                    row=row,
                    device=device,
                    decoder_spec=decoder_spec,
                )

                record = {
                    "official_test_index":
                        row[
                            "official_test_index"
                        ],

                    "filename":
                        row["filename"],

                    "width":
                        row["width"],

                    "height":
                        row["height"],

                    "gt_boxes":
                        row["gt_boxes"],

                    "dedup247_member":
                        row[
                            "dedup247_member"
                        ],

                    "predictions":
                        predictions,
                }

                records.append(record)

                prediction_file.write(
                    json.dumps(
                        {
                            "official_test_index":
                                record[
                                    "official_test_index"
                                ],

                            "filename":
                                record[
                                    "filename"
                                ],

                            "width":
                                record["width"],

                            "height":
                                record["height"],

                            "dedup247_member":
                                record[
                                    "dedup247_member"
                                ],

                            "gt_boxes":
                                record[
                                    "gt_boxes"
                                ],

                            "predictions_decoder_floor":
                                record[
                                    "predictions"
                                ],
                        },
                        allow_nan=False,
                    )
                    + "\n"
                )

                run_state[
                    "images_completed"
                ] = index

                if (
                    index % 25 == 0
                    or index == 250
                ):
                    write_json(
                        run_state_path,
                        run_state,
                    )

                    print(
                        f"  inferred {index}/250"
                    )

        primary_records = records

        dedup_records = [
            record
            for record in records
            if record[
                "dedup247_member"
            ]
        ]

        require(
            len(primary_records) == 250,
            "Formal primary record count != 250.",
        )

        require(
            len(dedup_records) == 247,
            "Formal dedup record count != 247.",
        )

        require(
            sum(
                len(record["gt_boxes"])
                for record
                in primary_records
            )
            == 539,
            "Formal primary GT total != 539.",
        )

        require(
            sum(
                len(record["gt_boxes"])
                for record
                in dedup_records
            )
            == 539,
            "Formal dedup GT total != 539.",
        )

        primary_result = evaluate_subset(
            records=primary_records,
            operating_threshold=(
                operating_threshold
            ),
        )

        dedup_result = evaluate_subset(
            records=dedup_records,
            operating_threshold=(
                operating_threshold
            ),
        )

        primary_per_image = (
            primary_result.pop(
                "per_image"
            )
        )

        dedup_per_image = (
            dedup_result.pop(
                "per_image"
            )
        )

        primary_csv = (
            seed_dir
            / "rsc_primary250_per_image.csv"
        )

        dedup_csv = (
            seed_dir
            / "rsc_dedup247_per_image.csv"
        )

        write_csv(
            primary_csv,
            primary_per_image,
        )

        write_csv(
            dedup_csv,
            dedup_per_image,
        )

        summary = {
            "evaluation_name":
                "DASO-Net RSC External Detection v1",

            "status":
                "PASS",

            "scientific_task":
                "class-agnostic bounding-box detection",

            "segmentation_metrics":
                False,

            "seed_index":
                seed_index,

            "seed":
                int(model_info["seed"]),

            "checkpoint":
                str(
                    model_info["checkpoint"]
                ),

            "checkpoint_sha256":
                model_info[
                    "checkpoint_sha256"
                ],

            "successful_optimizer_updates":
                2750,

            "protocol_sha256":
                protocol_sha,

            "evaluator_sha256":
                evaluator_sha,

            "phase5c_evaluator_sha256":
                phase5c[
                    "script_sha256"
                ],

            "checkpoint_loading_contract":
                (
                    "torch.load(checkpoint_path) -> "
                    "loaded mapping -> "
                    "build_model(loaded_mapping, device)"
                ),

            "decoder":
                decoder_spec,

            "input_protocol":
                protocol[
                    "input_protocol"
                ],

            "operating_threshold":
                operating_threshold,

            "primary_official_test250":
                {
                    "images":
                        250,

                    "gt_boxes":
                        539,

                    "zero_gt_images":
                        4,

                    **primary_result,
                },

            "sensitivity_exact_dedup247":
                {
                    "images":
                        247,

                    "gt_boxes":
                        539,

                    "zero_gt_images":
                        1,

                    **dedup_result,
                },

            "scientific_policy":
                {
                    "rsc_training":
                        False,

                    "rsc_crpa":
                        False,

                    "rsc_calibration":
                        False,

                    "rsc_threshold_tuning":
                        False,

                    "checkpoint_selection":
                        False,

                    "seed_selection":
                        False,

                    "best_seed_selection":
                        False,
                },

            "artifacts":
                {
                    "predictions_decoder_floor":
                        str(
                            predictions_jsonl
                        ),

                    "primary_per_image":
                        str(
                            primary_csv
                        ),

                    "dedup_per_image":
                        str(
                            dedup_csv
                        ),
                },
        }

        summary_path = (
            seed_dir
            / "RSC_EXTERNAL_DETECTION_SUMMARY.json"
        )

        write_json(
            summary_path,
            summary,
        )

        summary_sha = sha256_file(
            summary_path
        )

        summary_sha_record = (
            seed_dir
            / "RSC_EXTERNAL_DETECTION_SUMMARY.sha256.txt"
        )

        summary_sha_record.write_text(
            (
                f"{summary_sha}  "
                f"{summary_path.name}\n"
            ),
            encoding="utf-8",
        )

        run_state.update(
            {
                "status":
                    "PASS",

                "images_completed":
                    250,

                "summary":
                    str(
                        summary_path
                    ),

                "summary_sha256":
                    summary_sha,
            }
        )

        write_json(
            run_state_path,
            run_state,
        )

        print()
        print("=" * 72)
        print(
            "DASO-NET RSC FORMAL "
            "PER-SEED EVALUATION: PASS"
        )
        print("=" * 72)
        print()

        print(
            f"Seed index: {seed_index}"
        )

        print(
            f"Seed:       {model_info['seed']}"
        )

        print(
            "Frozen operating threshold: "
            f"{operating_threshold:.2f}"
        )

        print()

        print(
            "PRIMARY official test-250:"
        )

        print(
            "  Box AP50:    "
            f"{primary_result['box_ap50']:.6f}"
        )

        print(
            "  Box AP75:    "
            f"{primary_result['box_ap75']:.6f}"
        )

        print(
            "  Box AP50:95: "
            f"{primary_result['box_ap50_95']:.6f}"
        )

        print(
            "  Precision:   "
            f"{primary_result['operating']['precision']:.6f}"
        )

        print(
            "  Recall:      "
            f"{primary_result['operating']['recall']:.6f}"
        )

        print(
            "  F1:          "
            f"{primary_result['operating']['f1']:.6f}"
        )

        print()

        print(
            "SENSITIVITY exact-dedup-247:"
        )

        print(
            "  Box AP50:    "
            f"{dedup_result['box_ap50']:.6f}"
        )

        print(
            "  Box AP75:    "
            f"{dedup_result['box_ap75']:.6f}"
        )

        print(
            "  Box AP50:95: "
            f"{dedup_result['box_ap50_95']:.6f}"
        )

        print()

        print(
            "Threshold tuning: NO"
        )

        print(
            "Calibration: NO"
        )

        print(
            "Checkpoint selection: NO"
        )

        print(
            "Seed selection: NO"
        )

        print()

        print(
            "Summary JSON:"
        )

        print(
            f"  {summary_path}"
        )

        print()

        print(
            "Summary SHA256:"
        )

        print(
            f"  {summary_sha}"
        )

        print("=" * 72)

    except Exception as exc:
        run_state.update(
            {
                "status":
                    "FAIL",

                "error_type":
                    type(exc).__name__,

                "error":
                    str(exc),

                "traceback":
                    traceback.format_exc(),

                "threshold_tuning":
                    False,

                "calibration":
                    False,

                "checkpoint_selection":
                    False,

                "seed_selection":
                    False,
            }
        )

        write_json(
            run_state_path,
            run_state,
        )

        print()
        print("=" * 72)
        print(
            "DASO-NET RSC FORMAL "
            "PER-SEED EVALUATION: FAIL"
        )
        print("=" * 72)
        print()

        print(
            "Failed run state preserved:"
        )

        print(
            f"  {run_state_path}"
        )

        print()

        print(
            "Completed images:"
        )

        print(
            f"  {run_state['images_completed']}"
        )

        print()

        raise


# ============================================================
# 21. CLI
# ============================================================

def main():
    parser = argparse.ArgumentParser(
        description=(
            "DASO-Net frozen RSC external "
            "bounding-box evaluator v1"
        )
    )

    parser.add_argument(
        "--mode",
        required=True,
        choices=[
            "self-test",
            "smoke",
            "evaluate",
        ],
    )

    parser.add_argument(
        "--protocol",
        type=Path,
        default=DEFAULT_PROTOCOL,
    )

    parser.add_argument(
        "--seed-index",
        type=int,
        choices=[
            1,
            2,
            3,
        ],
        default=1,
    )

    parser.add_argument(
        "--device",
        default="cuda",
        choices=[
            "cuda",
            "cpu",
        ],
    )

    args = parser.parse_args()

    if args.mode == "self-test":
        run_self_tests()
        return

    protocol, protocol_sha = load_protocol(
        args.protocol
    )

    phase5c = load_phase5c_module(
        protocol
    )

    if args.device == "cuda":
        require(
            torch.cuda.is_available(),
            "CUDA requested but not available.",
        )

    device = torch.device(
        args.device
    )

    torch.manual_seed(0)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(0)

    torch.backends.cudnn.benchmark = False

    try:
        torch.backends.cudnn.deterministic = True
    except Exception:
        pass

    try:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True,
        )
    except Exception:
        pass

    if args.mode == "smoke":
        run_smoke(
            protocol=protocol,
            protocol_sha=protocol_sha,
            phase5c=phase5c,
            seed_index=args.seed_index,
            device=device,
        )
        return

    if args.mode == "evaluate":
        run_formal_evaluation(
            protocol=protocol,
            protocol_sha=protocol_sha,
            phase5c=phase5c,
            seed_index=args.seed_index,
            device=device,
        )
        return

    raise RuntimeError(
        f"Unhandled mode: {args.mode}"
    )


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/evaluate_rsc_external_v1.py


In [29]:
from pathlib import Path
import hashlib
import py_compile
import subprocess
import sys

SCRIPT = Path(
    "/kaggle/working/evaluate_rsc_external_v1.py"
)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


print("=" * 72)
print("DASO-Net — RSC EVALUATOR STATIC + SYNTHETIC AUDIT RESUME01")
print("=" * 72)
print()

py_compile.compile(
    str(SCRIPT),
    doraise=True,
)

print("PASS  Python compile")
print(f"Evaluator SHA256: {sha256_file(SCRIPT)}")
print()

result = subprocess.run(
    [
        sys.executable,
        str(SCRIPT),
        "--mode",
        "self-test",
    ],
    check=False,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr.strip():
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "Evaluator synthetic self-test failed "
        f"with exit code {result.returncode}"
    )

print()
print("READINESS:")
print("  READY_FOR_STRUCTURAL_RSC_SMOKE_RESUME01")

DASO-Net — RSC EVALUATOR STATIC + SYNTHETIC AUDIT RESUME01

PASS  Python compile
Evaluator SHA256: 59a3de31f2a5244ba32c53b705b050e033b5d17c8de7efb36c17dbc49f0a7723

DASO-Net — RSC EVALUATOR SYNTHETIC SELF-TEST
PASS  Box IoU unit tests
PASS  101-point AP unit test
PASS  Frozen operating-metric unit tests

SELF-TEST STATUS: PASS


READINESS:
  READY_FOR_STRUCTURAL_RSC_SMOKE_RESUME01


In [30]:
from pathlib import Path
import subprocess
import sys

SCRIPT = Path(
    "/kaggle/working/evaluate_rsc_external_v1.py"
)

PRIOR_FAILED = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_v1_smoke_audit"
)

RESUME01 = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_v1_smoke_audit_resume01"
)

print("=" * 72)
print("DASO-Net — RSC EVALUATOR STRUCTURAL SMOKE RESUME01")
print("=" * 72)
print()

print("Prior failed smoke directory:")
print(f"  {PRIOR_FAILED}")
print(f"  exists: {PRIOR_FAILED.exists()}")
print()

print("New smoke output directory:")
print(f"  {RESUME01}")
print()

print("This will:")
print("  - preserve the previous failed smoke attempt")
print("  - load the frozen Seed-1 checkpoint as a checkpoint mapping")
print("  - pass that mapping to Phase5C build_model(checkpoint, device)")
print("  - infer exactly two RSC images")
print("  - check one positive-GT case")
print("  - check test_113.jpg zero-GT case")
print("  - report NO prediction scores")
print("  - report NO prediction counts")
print("  - compute NO AP/F1")
print("  - perform NO threshold tuning")
print()

result = subprocess.run(
    [
        sys.executable,
        str(SCRIPT),
        "--mode",
        "smoke",
        "--seed-index",
        "1",
        "--device",
        "cuda",
    ],
    check=False,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr.strip():
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "RSC structural smoke Resume01 failed "
        f"with exit code {result.returncode}"
    )

print()
print("Do NOT run --mode evaluate yet.")

DASO-Net — RSC EVALUATOR STRUCTURAL SMOKE RESUME01

Prior failed smoke directory:
  /kaggle/working/dasonet_final_analysis/rsc_evaluator_v1_smoke_audit
  exists: True

New smoke output directory:
  /kaggle/working/dasonet_final_analysis/rsc_evaluator_v1_smoke_audit_resume01

This will:
  - preserve the previous failed smoke attempt
  - load the frozen Seed-1 checkpoint as a checkpoint mapping
  - pass that mapping to Phase5C build_model(checkpoint, device)
  - infer exactly two RSC images
  - check one positive-GT case
  - check test_113.jpg zero-GT case
  - report NO prediction scores
  - report NO prediction counts
  - compute NO AP/F1
  - perform NO threshold tuning

INFO  Prior failed smoke directory preserved:
      /kaggle/working/dasonet_final_analysis/rsc_evaluator_v1_smoke_audit
DASO-Net — RSC EVALUATOR STRUCTURAL SMOKE AUDIT RESUME01

PASS  Prior failed smoke preserved
PASS  Frozen protocol identity
PASS  Frozen checkpoint identity
PASS  Checkpoint loaded as mapping
PASS  Pha

In [31]:
# ============================================================
# DASO-Net — RSC EVALUATOR IMPLEMENTATION FREEZE v1
#
# PURPOSE
# -------
# Freeze the exact evaluator implementation AFTER:
#   - synthetic metric-engine self-test PASS
#   - structural positive-GT smoke PASS
#   - structural zero-GT smoke PASS
#
# and BEFORE:
#   - any formal 250-image RSC evaluation
#   - any RSC metric result
#
# THIS CELL PERFORMS NO RSC MODEL INFERENCE.
#
# It:
#   1. verifies the frozen scientific protocol
#   2. verifies the successful Resume01 smoke audit
#   3. verifies the previous failed smoke is preserved
#   4. verifies the evaluator SHA
#   5. verifies all 3 final checkpoint identities
#   6. verifies the validated Phase5C evaluator identity
#   7. verifies no formal RSC output root exists yet
#   8. copies the evaluator byte-for-byte to a frozen path
#   9. reruns ONLY synthetic self-tests on the frozen copy
#  10. freezes all identities in a SHA-protected JSON
#
# No scientific protocol is modified.
# ============================================================

from __future__ import annotations

import hashlib
import json
import shutil
import subprocess
import sys
import traceback
from pathlib import Path


# ============================================================
# 1. FROZEN EXPECTATIONS
# ============================================================

EXPECTED_PROTOCOL_SHA256 = (
    "e3c87caebafe87c7671b5745a480c0ed"
    "f67f444feac4278633da6addae4d1f7e"
)

EXPECTED_EVALUATOR_SHA256 = (
    "59a3de31f2a5244ba32c53b705b050e0"
    "33b5d17c8de7efb36c17dbc49f0a7723"
)

EXPECTED_SMOKE_AUDIT_SHA256 = (
    "87ebc2ab8a9fad8799269b58284ad88a"
    "0617093c22f9f0cbceeff368aba8ef4a"
)

EXPECTED_THRESHOLDS = [
    0.30,
    0.35,
    0.35,
]

EXPECTED_SEEDS = [
    1782656963,
    901395089,
    252916394,
]

EXPECTED_CHECKPOINT_SHAS = [
    (
        "e4c4b79ad4b8c6e18ba2b1e3d6cbe796"
        "c9cc301bd6fcc7c596fc290ad3a7e81d"
    ),
    (
        "19cdb834a46240916b2c8023e8c3afe57"
        "38d9c7cd5d8f818c62ad1e289df1ad2"
    ),
    (
        "be7b7941f2f62179e15074aa973bf6da0"
        "eb9018ec9133486512bd1f87153f447"
    ),
]


# ============================================================
# 2. INPUT PATHS
# ============================================================

PROTOCOL_JSON = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_external_evaluation_protocol_freeze_v1/"
    "RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

SOURCE_EVALUATOR = Path(
    "/kaggle/working/evaluate_rsc_external_v1.py"
)

PRIOR_FAILED_SMOKE_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_v1_smoke_audit"
)

SUCCESSFUL_SMOKE_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_v1_smoke_audit_resume01"
)

SMOKE_JSON = (
    SUCCESSFUL_SMOKE_DIR
    / "RSC_EVALUATOR_STRUCTURAL_SMOKE_AUDIT_RESUME01.json"
)

SMOKE_SHA_RECORD = (
    SUCCESSFUL_SMOKE_DIR
    / "RSC_EVALUATOR_STRUCTURAL_SMOKE_AUDIT_RESUME01.sha256.txt"
)

FORMAL_RSC_OUTPUT_ROOT = Path(
    "/kaggle/working/dasonet_final_evaluation/"
    "rsc_external_detection_v1"
)


# ============================================================
# 3. FREEZE OUTPUT
# ============================================================

FREEZE_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_implementation_freeze_v1"
)

FROZEN_EVALUATOR = (
    FREEZE_DIR
    / "evaluate_rsc_external_v1_FROZEN.py"
)

FROZEN_PROTOCOL_COPY = (
    FREEZE_DIR
    / "RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

FROZEN_SMOKE_COPY = (
    FREEZE_DIR
    / "RSC_EVALUATOR_STRUCTURAL_SMOKE_AUDIT_RESUME01.json"
)

SELFTEST_LOG = (
    FREEZE_DIR
    / "RSC_FROZEN_EVALUATOR_SYNTHETIC_SELFTEST.txt"
)

EVALUATOR_SHA_RECORD = (
    FREEZE_DIR
    / "RSC_FROZEN_EVALUATOR.sha256.txt"
)

FREEZE_JSON = (
    FREEZE_DIR
    / "RSC_EVALUATOR_IMPLEMENTATION_FREEZE_v1.json"
)

FREEZE_SHA_RECORD = (
    FREEZE_DIR
    / "RSC_EVALUATOR_IMPLEMENTATION_FREEZE_v1.sha256.txt"
)


# ============================================================
# 4. HELPERS
# ============================================================

checks = []


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def require(
    condition: bool,
    message: str,
) -> None:
    if not condition:
        raise RuntimeError(message)


def require_file(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_file(),
        f"{label} missing:\n{path}",
    )


def require_dir(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_dir(),
        f"{label} missing:\n{path}",
    )


def read_json(path: Path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path: Path,
    obj,
) -> None:
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def record(
    name: str,
    **details,
) -> None:
    item = {
        "check": name,
        "status": "PASS",
    }

    item.update(details)

    checks.append(item)

    print(f"PASS  {name}")


# ============================================================
# 5. BANNER
# ============================================================

print("=" * 76)
print(
    "DASO-Net — RSC EVALUATOR IMPLEMENTATION "
    "PRE-FORMAL FREEZE v1"
)
print("=" * 76)
print()

print(
    "THIS CELL PERFORMS NO RSC MODEL INFERENCE."
)
print()

print(
    "Scientific status:"
)
print(
    "  protocol frozen"
)
print(
    "  assets audited"
)
print(
    "  evaluator structurally smoke-tested"
)
print(
    "  formal RSC metrics not yet computed"
)
print()


# ============================================================
# 6. REFUSE OVERWRITE
# ============================================================

if FREEZE_DIR.exists():
    raise RuntimeError(
        "Evaluator freeze directory already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{FREEZE_DIR}"
    )


# Absolutely no formal RSC run should exist yet.

if FORMAL_RSC_OUTPUT_ROOT.exists():
    raise RuntimeError(
        "Formal RSC evaluation output root already exists BEFORE "
        "evaluator implementation freeze:\n"
        f"{FORMAL_RSC_OUTPUT_ROOT}\n\n"
        "Do not continue automatically. Preserve that directory "
        "and determine whether formal RSC inference has already "
        "occurred."
    )


FREEZE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


attempt = {
    "freeze_name":
        "DASO-Net RSC Evaluator Implementation Freeze v1",

    "status":
        "RUNNING",

    "formal_rsc_inference_performed_by_this_cell":
        False,

    "scientific_metrics_computed_by_this_cell":
        False,

    "threshold_tuning_performed":
        False,

    "calibration_performed":
        False,

    "checkpoint_selection_performed":
        False,

    "seed_selection_performed":
        False,

    "checks":
        checks,
}


# ============================================================
# 7. AUDIT / FREEZE
# ============================================================

try:

    # --------------------------------------------------------
    # 7A. Frozen scientific protocol
    # --------------------------------------------------------

    require_file(
        PROTOCOL_JSON,
        "Frozen RSC scientific protocol",
    )

    protocol_sha = sha256_file(
        PROTOCOL_JSON
    )

    require(
        protocol_sha
        == EXPECTED_PROTOCOL_SHA256,
        "Frozen RSC protocol SHA mismatch.\n"
        f"Expected: {EXPECTED_PROTOCOL_SHA256}\n"
        f"Actual:   {protocol_sha}",
    )

    protocol = read_json(
        PROTOCOL_JSON
    )

    require(
        protocol.get("status")
        == "FROZEN_BEFORE_RSC_INFERENCE",
        "Frozen protocol status changed.",
    )

    require(
        protocol[
            "scientific_policy"
        ][
            "rsc_threshold_tuning"
        ]
        is False,
        "Frozen protocol unexpectedly permits RSC threshold tuning.",
    )

    require(
        protocol[
            "scientific_policy"
        ][
            "all_three_final_seeds_required"
        ]
        is True,
        "Frozen protocol no longer requires all three final seeds.",
    )

    require(
        protocol[
            "metric_protocol"
        ][
            "segmentation_metrics"
        ]
        is False,
        "Frozen protocol unexpectedly enables segmentation metrics.",
    )

    require(
        protocol[
            "scientific_sets"
        ][
            "primary"
        ][
            "rows"
        ]
        == 250,
        "Primary RSC set is no longer 250 rows.",
    )

    require(
        protocol[
            "scientific_sets"
        ][
            "sensitivity"
        ][
            "rows"
        ]
        == 247,
        "Sensitivity RSC set is no longer 247 rows.",
    )

    require(
        Path(
            protocol[
                "designated_future_output_root"
            ]
        )
        == FORMAL_RSC_OUTPUT_ROOT,
        "Frozen protocol formal-output root changed.",
    )

    record(
        "Frozen scientific protocol identity",
        sha256=protocol_sha,
    )


    # --------------------------------------------------------
    # 7B. Successful structural smoke identity
    # --------------------------------------------------------

    require_dir(
        SUCCESSFUL_SMOKE_DIR,
        "Successful Resume01 smoke directory",
    )

    require_file(
        SMOKE_JSON,
        "Successful Resume01 smoke JSON",
    )

    require_file(
        SMOKE_SHA_RECORD,
        "Successful Resume01 smoke SHA record",
    )

    smoke_sha = sha256_file(
        SMOKE_JSON
    )

    require(
        smoke_sha
        == EXPECTED_SMOKE_AUDIT_SHA256,
        "Successful smoke JSON SHA mismatch.\n"
        f"Expected: {EXPECTED_SMOKE_AUDIT_SHA256}\n"
        f"Actual:   {smoke_sha}",
    )

    smoke = read_json(
        SMOKE_JSON
    )

    require(
        smoke.get("status")
        == "PASS",
        "Resume01 smoke status is not PASS.",
    )

    require(
        smoke.get("readiness")
        == "READY_TO_FREEZE_RSC_EVALUATOR_IMPLEMENTATION",
        "Resume01 smoke readiness changed.",
    )

    require(
        smoke.get("protocol_sha256")
        == EXPECTED_PROTOCOL_SHA256,
        "Smoke audit references an unexpected protocol SHA.",
    )

    require(
        smoke.get("evaluator_sha256")
        == EXPECTED_EVALUATOR_SHA256,
        "Smoke audit references an unexpected evaluator SHA.",
    )

    require(
        smoke.get("model_forward_passes")
        == 2,
        "Resume01 smoke did not record exactly two model forward passes.",
    )

    require(
        smoke.get("positive_gt_case_checked")
        is True,
        "Resume01 smoke did not verify positive-GT case.",
    )

    require(
        smoke.get("zero_gt_case_checked")
        is True,
        "Resume01 smoke did not verify zero-GT case.",
    )

    require(
        smoke.get("scientific_metrics_computed")
        is False,
        "Resume01 smoke unexpectedly computed scientific metrics.",
    )

    require(
        smoke.get("prediction_counts_saved")
        is False,
        "Resume01 smoke unexpectedly saved prediction counts.",
    )

    require(
        smoke.get("prediction_scores_saved")
        is False,
        "Resume01 smoke unexpectedly saved prediction scores.",
    )

    require(
        smoke.get("threshold_sweep_performed")
        is False,
        "Resume01 smoke unexpectedly performed a threshold sweep.",
    )

    require(
        smoke.get("threshold_tuning_performed")
        is False,
        "Resume01 smoke unexpectedly performed threshold tuning.",
    )

    require(
        smoke.get("calibration_performed")
        is False,
        "Resume01 smoke unexpectedly performed calibration.",
    )

    require(
        smoke.get("checkpoint_selection_performed")
        is False,
        "Resume01 smoke unexpectedly performed checkpoint selection.",
    )

    require(
        smoke.get("seed_selection_performed")
        is False,
        "Resume01 smoke unexpectedly performed seed selection.",
    )


    smoke_sha_lines = [
        line.strip()
        for line in SMOKE_SHA_RECORD.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

    require(
        len(smoke_sha_lines) == 1,
        "Unexpected Resume01 smoke SHA-record format.",
    )

    require(
        smoke_sha_lines[0].split()[0].lower()
        == EXPECTED_SMOKE_AUDIT_SHA256,
        "Resume01 smoke SHA record does not match smoke JSON.",
    )

    record(
        "Successful Resume01 structural smoke identity",
        sha256=smoke_sha,
        model_forward_passes=2,
        scientific_metrics_computed=False,
    )


    # --------------------------------------------------------
    # 7C. Preserve failed pre-fix smoke provenance
    # --------------------------------------------------------

    require_dir(
        PRIOR_FAILED_SMOKE_DIR,
        "Prior failed pre-fix smoke directory",
    )

    record(
        "Prior failed pre-fix smoke preserved",
        path=str(
            PRIOR_FAILED_SMOKE_DIR
        ),
    )


    # --------------------------------------------------------
    # 7D. Source evaluator identity
    # --------------------------------------------------------

    require_file(
        SOURCE_EVALUATOR,
        "Corrected RSC evaluator",
    )

    source_evaluator_sha = sha256_file(
        SOURCE_EVALUATOR
    )

    require(
        source_evaluator_sha
        == EXPECTED_EVALUATOR_SHA256,
        "Corrected evaluator SHA changed.\n"
        f"Expected: {EXPECTED_EVALUATOR_SHA256}\n"
        f"Actual:   {source_evaluator_sha}",
    )

    # In-memory syntax compile only; does not create a .pyc.
    evaluator_source_text = SOURCE_EVALUATOR.read_text(
        encoding="utf-8"
    )

    compile(
        evaluator_source_text,
        str(SOURCE_EVALUATOR),
        "exec",
    )

    record(
        "Corrected evaluator source identity",
        sha256=source_evaluator_sha,
    )

    record(
        "Corrected evaluator syntax compile"
    )


    # --------------------------------------------------------
    # 7E. All three final models remain exactly frozen
    # --------------------------------------------------------

    models = protocol[
        "models"
    ]

    require(
        len(models) == 3,
        "Frozen protocol does not contain exactly three models.",
    )

    observed_seeds = [
        int(
            model["seed"]
        )
        for model in models
    ]

    observed_thresholds = [
        float(
            model["operating_threshold"]
        )
        for model in models
    ]

    observed_checkpoint_shas = [
        str(
            model["checkpoint_sha256"]
        ).lower()
        for model in models
    ]

    require(
        observed_seeds
        == EXPECTED_SEEDS,
        "Frozen final-seed order changed.\n"
        f"Expected: {EXPECTED_SEEDS}\n"
        f"Actual:   {observed_seeds}",
    )

    require(
        observed_thresholds
        == EXPECTED_THRESHOLDS,
        "Frozen operating thresholds changed.\n"
        f"Expected: {EXPECTED_THRESHOLDS}\n"
        f"Actual:   {observed_thresholds}",
    )

    require(
        observed_checkpoint_shas
        == EXPECTED_CHECKPOINT_SHAS,
        "Frozen checkpoint-SHA order changed.",
    )


    verified_models = []

    for model in models:
        seed_index = int(
            model[
                "seed_index"
            ]
        )

        checkpoint = Path(
            model[
                "checkpoint"
            ]
        )

        require_file(
            checkpoint,
            f"Final Seed-{seed_index} checkpoint",
        )

        actual_checkpoint_sha = sha256_file(
            checkpoint
        )

        require(
            actual_checkpoint_sha
            == str(
                model[
                    "checkpoint_sha256"
                ]
            ).lower(),
            "Final checkpoint SHA mismatch.\n"
            f"Seed index: {seed_index}\n"
            f"Checkpoint: {checkpoint}",
        )

        require(
            int(
                model[
                    "successful_optimizer_updates"
                ]
            )
            == 2750,
            f"Seed-{seed_index} successful-update count != 2750.",
        )

        verified_models.append(
            {
                "seed_index":
                    seed_index,

                "seed":
                    int(
                        model[
                            "seed"
                        ]
                    ),

                "checkpoint":
                    str(
                        checkpoint
                    ),

                "checkpoint_sha256":
                    actual_checkpoint_sha,

                "successful_optimizer_updates":
                    2750,

                "operating_threshold":
                    float(
                        model[
                            "operating_threshold"
                        ]
                    ),
            }
        )

        record(
            f"Final Seed-{seed_index} checkpoint identity",
            seed=int(
                model[
                    "seed"
                ]
            ),
            sha256=actual_checkpoint_sha,
            operating_threshold=float(
                model[
                    "operating_threshold"
                ]
            ),
        )


    # --------------------------------------------------------
    # 7F. Validated Phase5C evaluator identity
    # --------------------------------------------------------

    phase5c_script = Path(
        protocol[
            "validated_evaluator"
        ][
            "script"
        ]
    )

    expected_phase5c_sha = (
        protocol[
            "validated_evaluator"
        ][
            "whole_file_sha256"
        ]
    )

    require_file(
        phase5c_script,
        "Validated Phase5C evaluator",
    )

    actual_phase5c_sha = sha256_file(
        phase5c_script
    )

    require(
        actual_phase5c_sha
        == expected_phase5c_sha,
        "Validated Phase5C evaluator changed.\n"
        f"Expected: {expected_phase5c_sha}\n"
        f"Actual:   {actual_phase5c_sha}",
    )

    record(
        "Validated Phase5C evaluator identity",
        sha256=actual_phase5c_sha,
    )


    # --------------------------------------------------------
    # 7G. Formal result directory must STILL NOT EXIST
    # --------------------------------------------------------

    require(
        not FORMAL_RSC_OUTPUT_ROOT.exists(),
        "Formal RSC output root unexpectedly exists before "
        "implementation freeze:\n"
        f"{FORMAL_RSC_OUTPUT_ROOT}",
    )

    record(
        "No formal RSC evaluation output exists",
        path=str(
            FORMAL_RSC_OUTPUT_ROOT
        ),
    )


    # --------------------------------------------------------
    # 7H. Create byte-identical frozen copies
    # --------------------------------------------------------

    shutil.copyfile(
        SOURCE_EVALUATOR,
        FROZEN_EVALUATOR,
    )

    shutil.copyfile(
        PROTOCOL_JSON,
        FROZEN_PROTOCOL_COPY,
    )

    shutil.copyfile(
        SMOKE_JSON,
        FROZEN_SMOKE_COPY,
    )


    frozen_evaluator_sha = sha256_file(
        FROZEN_EVALUATOR
    )

    frozen_protocol_sha = sha256_file(
        FROZEN_PROTOCOL_COPY
    )

    frozen_smoke_sha = sha256_file(
        FROZEN_SMOKE_COPY
    )


    require(
        frozen_evaluator_sha
        == EXPECTED_EVALUATOR_SHA256,
        "Frozen evaluator copy is not byte-identical.",
    )

    require(
        frozen_protocol_sha
        == EXPECTED_PROTOCOL_SHA256,
        "Frozen protocol copy is not byte-identical.",
    )

    require(
        frozen_smoke_sha
        == EXPECTED_SMOKE_AUDIT_SHA256,
        "Frozen smoke-audit copy is not byte-identical.",
    )

    record(
        "Frozen evaluator byte identity",
        sha256=frozen_evaluator_sha,
    )

    record(
        "Frozen protocol-copy byte identity",
        sha256=frozen_protocol_sha,
    )

    record(
        "Frozen smoke-copy byte identity",
        sha256=frozen_smoke_sha,
    )


    # --------------------------------------------------------
    # 7I. Rerun ONLY synthetic metric self-tests on frozen copy
    #
    # NO protocol load.
    # NO checkpoint load.
    # NO RSC image load.
    # NO RSC inference.
    # --------------------------------------------------------

    selftest = subprocess.run(
        [
            sys.executable,
            str(
                FROZEN_EVALUATOR
            ),
            "--mode",
            "self-test",
        ],
        check=False,
        text=True,
        capture_output=True,
    )

    selftest_text = (
        "STDOUT\n"
        "======\n"
        + selftest.stdout
        + "\n"
        "STDERR\n"
        "======\n"
        + selftest.stderr
    )

    SELFTEST_LOG.write_text(
        selftest_text,
        encoding="utf-8",
    )

    require(
        selftest.returncode == 0,
        "Frozen evaluator synthetic self-test failed.\n"
        f"Return code: {selftest.returncode}\n"
        f"Log: {SELFTEST_LOG}",
    )

    require(
        "SELF-TEST STATUS: PASS"
        in selftest.stdout,
        "Frozen evaluator did not emit expected self-test PASS.",
    )

    selftest_sha = sha256_file(
        SELFTEST_LOG
    )

    record(
        "Frozen evaluator synthetic self-test",
        sha256=selftest_sha,
        rsc_inference=False,
    )


    # --------------------------------------------------------
    # 7J. Frozen evaluator SHA record
    # --------------------------------------------------------

    EVALUATOR_SHA_RECORD.write_text(
        (
            f"{frozen_evaluator_sha}  "
            f"{FROZEN_EVALUATOR.name}\n"
        ),
        encoding="utf-8",
    )

    record(
        "Frozen evaluator SHA record"
    )


    # --------------------------------------------------------
    # 7K. Build immutable implementation-freeze metadata
    # --------------------------------------------------------

    freeze = {
        "freeze_name":
            "DASO-Net RSC Evaluator Implementation Freeze v1",

        "status":
            "FROZEN_BEFORE_FORMAL_RSC_EVALUATION",

        "scientific_status":
            {
                "rsc_assets_audited":
                    True,

                "rsc_scientific_protocol_frozen":
                    True,

                "synthetic_metric_self_test_passed":
                    True,

                "structural_positive_gt_smoke_passed":
                    True,

                "structural_zero_gt_smoke_passed":
                    True,

                "formal_rsc_evaluation_started":
                    False,

                "formal_rsc_metrics_computed":
                    False,

                "threshold_tuning_performed":
                    False,

                "calibration_performed":
                    False,

                "checkpoint_selection_performed":
                    False,

                "seed_selection_performed":
                    False,

                "best_seed_selection_performed":
                    False,
            },

        "frozen_protocol":
            {
                "source_path":
                    str(
                        PROTOCOL_JSON
                    ),

                "frozen_copy":
                    str(
                        FROZEN_PROTOCOL_COPY
                    ),

                "sha256":
                    frozen_protocol_sha,
            },

        "frozen_evaluator":
            {
                "source_path":
                    str(
                        SOURCE_EVALUATOR
                    ),

                "frozen_execution_copy":
                    str(
                        FROZEN_EVALUATOR
                    ),

                "sha256":
                    frozen_evaluator_sha,

                "formal_execution_rule":
                    (
                        "All formal RSC seed evaluations must execute "
                        "this exact frozen evaluator copy and verify "
                        "this SHA before execution."
                    ),
            },

        "structural_smoke":
            {
                "successful_smoke_source":
                    str(
                        SMOKE_JSON
                    ),

                "frozen_copy":
                    str(
                        FROZEN_SMOKE_COPY
                    ),

                "sha256":
                    frozen_smoke_sha,

                "model_forward_passes":
                    2,

                "positive_gt_case":
                    True,

                "zero_gt_case":
                    True,

                "scientific_metrics_computed":
                    False,

                "prediction_counts_saved":
                    False,

                "prediction_scores_saved":
                    False,
            },

        "prior_failed_smoke":
            {
                "path":
                    str(
                        PRIOR_FAILED_SMOKE_DIR
                    ),

                "preserved":
                    True,

                "scientific_interpretation":
                    (
                        "Infrastructure/interface failure before "
                        "model construction and before any model "
                        "forward pass."
                    ),
            },

        "frozen_models":
            verified_models,

        "validated_phase5c_evaluator":
            {
                "path":
                    str(
                        phase5c_script
                    ),

                "sha256":
                    actual_phase5c_sha,

                "decoder_reference_sha256":
                    protocol[
                        "validated_evaluator"
                    ][
                        "decoder"
                    ][
                        "validated_decoder_reference_sha256"
                    ],
            },

        "formal_scientific_sets":
            {
                "primary":
                    {
                        "name":
                            "official authors' RSC test-250",

                        "images":
                            250,

                        "gt_boxes":
                            539,

                        "zero_gt_images":
                            4,
                    },

                "sensitivity":
                    {
                        "name":
                            "exact-dedup-247",

                        "images":
                            247,

                        "gt_boxes":
                            539,

                        "zero_gt_images":
                            1,
                    },
            },

        "formal_metrics":
            {
                "task":
                    "class-agnostic bounding-box detection",

                "primary_metrics":
                    [
                        "Box AP50",
                        "Box AP75",
                        "Box AP50:95",
                    ],

                "operating_thresholds":
                    {
                        "seed_1":
                            0.30,

                        "seed_2":
                            0.35,

                        "seed_3":
                            0.35,
                    },

                "segmentation_metrics":
                    False,
            },

        "formal_execution":
            {
                "required_seed_indices":
                    [
                        1,
                        2,
                        3,
                    ],

                "required_seeds":
                    EXPECTED_SEEDS,

                "all_three_required":
                    True,

                "formal_output_root":
                    str(
                        FORMAL_RSC_OUTPUT_ROOT
                    ),

                "formal_output_root_exists_at_freeze":
                    False,

                "aggregate_rule":
                    "mean ± sample SD across all three final seeds",

                "primary_before_sensitivity":
                    True,

                "rsc_threshold_tuning":
                    False,

                "rsc_checkpoint_selection":
                    False,

                "rsc_seed_selection":
                    False,
            },

        "frozen_self_test":
            {
                "log":
                    str(
                        SELFTEST_LOG
                    ),

                "log_sha256":
                    selftest_sha,

                "rsc_inference":
                    False,
            },

        "checks":
            checks,
    }


    # --------------------------------------------------------
    # 7L. Write freeze metadata exactly once
    # --------------------------------------------------------

    write_json(
        FREEZE_JSON,
        freeze,
    )

    freeze_sha = sha256_file(
        FREEZE_JSON
    )

    FREEZE_SHA_RECORD.write_text(
        (
            f"{freeze_sha}  "
            f"{FREEZE_JSON.name}\n"
        ),
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # 7M. Final read-back
    # --------------------------------------------------------

    freeze_readback = read_json(
        FREEZE_JSON
    )

    require(
        freeze_readback.get("status")
        == "FROZEN_BEFORE_FORMAL_RSC_EVALUATION",
        "Freeze read-back status failure.",
    )

    require(
        freeze_readback[
            "frozen_evaluator"
        ][
            "sha256"
        ]
        == EXPECTED_EVALUATOR_SHA256,
        "Freeze read-back evaluator SHA failure.",
    )

    require(
        freeze_readback[
            "structural_smoke"
        ][
            "sha256"
        ]
        == EXPECTED_SMOKE_AUDIT_SHA256,
        "Freeze read-back smoke SHA failure.",
    )

    require(
        freeze_readback[
            "frozen_protocol"
        ][
            "sha256"
        ]
        == EXPECTED_PROTOCOL_SHA256,
        "Freeze read-back protocol SHA failure.",
    )

    require(
        freeze_readback[
            "formal_execution"
        ][
            "all_three_required"
        ]
        is True,
        "Freeze read-back does not require all 3 seeds.",
    )

    require(
        freeze_readback[
            "scientific_status"
        ][
            "formal_rsc_metrics_computed"
        ]
        is False,
        "Freeze read-back unexpectedly says metrics were computed.",
    )

    require(
        sha256_file(
            FREEZE_JSON
        )
        == freeze_sha,
        "Freeze JSON byte identity changed during read-back.",
    )

    require(
        sha256_file(
            FROZEN_EVALUATOR
        )
        == EXPECTED_EVALUATOR_SHA256,
        "Frozen evaluator changed after freeze write.",
    )


    # ========================================================
    # 8. FINAL REPORT
    # ========================================================

    print()
    print("=" * 76)
    print(
        "DASO-NET RSC EVALUATOR IMPLEMENTATION: FROZEN"
    )
    print("=" * 76)
    print()

    print("Status:")
    print(
        "  FROZEN_BEFORE_FORMAL_RSC_EVALUATION"
    )
    print()

    print("Scientific protocol SHA256:")
    print(
        f"  {EXPECTED_PROTOCOL_SHA256}"
    )
    print()

    print("Frozen evaluator SHA256:")
    print(
        f"  {EXPECTED_EVALUATOR_SHA256}"
    )
    print()

    print("Structural smoke SHA256:")
    print(
        f"  {EXPECTED_SMOKE_AUDIT_SHA256}"
    )
    print()

    print("Frozen evaluator execution copy:")
    print(
        f"  {FROZEN_EVALUATOR}"
    )
    print()

    print("Final models:")
    for model in verified_models:
        print(
            "  "
            f"Seed{model['seed_index']} "
            f"s{model['seed']} "
            f"threshold={model['operating_threshold']:.2f} "
            f"updates={model['successful_optimizer_updates']}"
        )

    print()

    print("Formal scientific sets:")
    print(
        "  PRIMARY     official authors' test-250"
    )
    print(
        "  SENSITIVITY exact-dedup-247"
    )
    print()

    print("Formal task:")
    print(
        "  class-agnostic bounding-box detection only"
    )
    print()

    print("Formal RSC output exists: NO")
    print("Formal RSC metrics computed: NO")
    print("RSC threshold tuning:        NO")
    print("RSC calibration:             NO")
    print("Checkpoint selection:        NO")
    print("Seed selection:              NO")
    print("Best-seed selection:         NO")
    print()

    print("Frozen synthetic self-test:")
    print(
        "  PASS"
    )
    print()

    print("Implementation freeze JSON:")
    print(
        f"  {FREEZE_JSON}"
    )
    print()

    print("Implementation freeze SHA256:")
    print(
        f"  {freeze_sha}"
    )
    print()

    print("Implementation freeze SHA record:")
    print(
        f"  {FREEZE_SHA_RECORD}"
    )
    print()

    print("NEXT READINESS:")
    print(
        "  READY_FOR_ALL_THREE_FORMAL_RSC_SEEDS"
    )
    print()

    print(
        "DO NOT MODIFY THE FROZEN EVALUATOR AFTER THIS POINT."
    )
    print("=" * 76)


except Exception as exc:

    failure = {
        "freeze_name":
            "DASO-Net RSC Evaluator Implementation Freeze v1",

        "status":
            "FAIL",

        "formal_rsc_inference_performed_by_this_cell":
            False,

        "scientific_metrics_computed_by_this_cell":
            False,

        "threshold_tuning_performed":
            False,

        "calibration_performed":
            False,

        "checkpoint_selection_performed":
            False,

        "seed_selection_performed":
            False,

        "error_type":
            type(exc).__name__,

        "error":
            str(exc),

        "traceback":
            traceback.format_exc(),

        "checks":
            checks,

        "readiness":
            "NOT_READY_FOR_FORMAL_RSC_EVALUATION",
    }

    write_json(
        FREEZE_JSON,
        failure,
    )

    failure_sha = sha256_file(
        FREEZE_JSON
    )

    FREEZE_SHA_RECORD.write_text(
        (
            f"{failure_sha}  "
            f"{FREEZE_JSON.name}\n"
        ),
        encoding="utf-8",
    )

    print()
    print("=" * 76)
    print(
        "DASO-NET RSC EVALUATOR IMPLEMENTATION FREEZE: FAIL"
    )
    print("=" * 76)
    print()

    print("Failure preserved:")
    print(
        f"  {FREEZE_JSON}"
    )
    print()

    print(
        "No formal RSC inference was performed by this cell."
    )
    print()

    raise

DASO-Net — RSC EVALUATOR IMPLEMENTATION PRE-FORMAL FREEZE v1

THIS CELL PERFORMS NO RSC MODEL INFERENCE.

Scientific status:
  protocol frozen
  assets audited
  evaluator structurally smoke-tested
  formal RSC metrics not yet computed

PASS  Frozen scientific protocol identity
PASS  Successful Resume01 structural smoke identity
PASS  Prior failed pre-fix smoke preserved
PASS  Corrected evaluator source identity
PASS  Corrected evaluator syntax compile
PASS  Final Seed-1 checkpoint identity
PASS  Final Seed-2 checkpoint identity
PASS  Final Seed-3 checkpoint identity
PASS  Validated Phase5C evaluator identity
PASS  No formal RSC evaluation output exists
PASS  Frozen evaluator byte identity
PASS  Frozen protocol-copy byte identity
PASS  Frozen smoke-copy byte identity
PASS  Frozen evaluator synthetic self-test
PASS  Frozen evaluator SHA record

DASO-NET RSC EVALUATOR IMPLEMENTATION: FROZEN

Status:
  FROZEN_BEFORE_FORMAL_RSC_EVALUATION

Scientific protocol SHA256:
  e3c87caebafe87c7671b

In [33]:
# ============================================================
# DASO-Net — RSC FORMAL RUN FAILURE DIAGNOSTIC v1
#
# READ-ONLY.
#
# PURPOSE
# -------
# Diagnose why formal Seed-1 and Seed-2 failed while preserving:
#   - all formal prediction outputs
#   - all failed run states
#   - successful Seed-3 outputs
#   - locked execution contract
#
# IMPORTANT
# ---------
# NO model inference
# NO metric recomputation
# NO threshold tuning
# NO calibration
# NO checkpoint selection
# NO seed selection
# NO deletion
# NO overwrite
#
# Seed-3 scientific metrics are intentionally NOT displayed.
# ============================================================

from __future__ import annotations

import hashlib
import json
from pathlib import Path


# ============================================================
# 1. PATHS
# ============================================================

EXECUTION_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_formal_three_seed_execution_v1"
)

LOG_DIR = (
    EXECUTION_DIR
    / "logs"
)

EXECUTION_CONTRACT = (
    EXECUTION_DIR
    / "RSC_FORMAL_EXECUTION_CONTRACT_v1.json"
)

EXECUTION_STATE = (
    EXECUTION_DIR
    / "RSC_FORMAL_EXECUTION_STATE_v1.json"
)

FORMAL_ROOT = Path(
    "/kaggle/working/dasonet_final_evaluation/"
    "rsc_external_detection_v1"
)

SEEDS = {
    1: {
        "seed": 1782656963,
        "dir":
            FORMAL_ROOT
            / "seed01_s1782656963",
    },

    2: {
        "seed": 901395089,
        "dir":
            FORMAL_ROOT
            / "seed02_s901395089",
    },

    3: {
        "seed": 252916394,
        "dir":
            FORMAL_ROOT
            / "seed03_s252916394",
    },
}


# ============================================================
# 2. HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def read_text_if_exists(path: Path) -> str | None:
    if not path.is_file():
        return None

    return path.read_text(
        encoding="utf-8",
        errors="replace",
    )


def read_json_if_exists(path: Path):
    if not path.is_file():
        return None

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def print_file_identity(
    path: Path,
    label: str,
) -> None:

    print(label)
    print(f"  path:   {path}")
    print(f"  exists: {path.is_file()}")

    if path.is_file():
        print(
            f"  bytes:  {path.stat().st_size}"
        )
        print(
            f"  sha256: {sha256_file(path)}"
        )


def list_seed_directory(
    seed_index: int,
    seed_dir: Path,
) -> None:

    print()
    print("-" * 76)
    print(
        f"SEED-{seed_index} DIRECTORY INVENTORY"
    )
    print("-" * 76)

    print(f"path:   {seed_dir}")
    print(f"exists: {seed_dir.is_dir()}")

    if not seed_dir.is_dir():
        return

    files = sorted(
        [
            path
            for path in seed_dir.rglob("*")
            if path.is_file()
        ],
        key=lambda path: str(path),
    )

    print(
        f"files:  {len(files)}"
    )

    for path in files:
        relative = path.relative_to(
            seed_dir
        )

        print(
            f"  {relative}"
            f"    bytes={path.stat().st_size}"
        )


# ============================================================
# 3. BANNER
# ============================================================

print("=" * 76)
print(
    "DASO-Net — RSC FORMAL RUN FAILURE "
    "DIAGNOSTIC v1"
)
print("=" * 76)
print()

print("Scientific action:")
print("  READ-ONLY failure diagnosis")
print()

print("NO inference")
print("NO metric recomputation")
print("NO tuning")
print("NO calibration")
print("NO selection")
print("NO deletion")
print()


# ============================================================
# 4. EXECUTION-CONTRACT / STATE IDENTITIES
# ============================================================

print_file_identity(
    EXECUTION_CONTRACT,
    "Formal execution contract:",
)

print()

print_file_identity(
    EXECUTION_STATE,
    "Formal execution state:",
)


state = read_json_if_exists(
    EXECUTION_STATE
)

if state is not None:

    print()
    print(
        "Execution-state status:"
    )

    print(
        "  "
        + str(
            state.get(
                "status"
            )
        )
    )

    print()

    print(
        "Recorded subprocess runs:"
    )

    for run in state.get(
        "runs",
        [],
    ):

        print(
            "  "
            f"Seed-{run.get('seed_index')}: "
            f"status={run.get('status')} "
            f"returncode={run.get('returncode')}"
        )


# ============================================================
# 5. DIRECTORY INVENTORIES
# ============================================================

for seed_index in [
    1,
    2,
    3,
]:

    list_seed_directory(
        seed_index,
        SEEDS[
            seed_index
        ][
            "dir"
        ],
    )


# ============================================================
# 6. SEED-1 / SEED-2 FAILURE STATES
# ============================================================

for seed_index in [
    1,
    2,
]:

    seed_info = SEEDS[
        seed_index
    ]

    seed_dir = seed_info[
        "dir"
    ]

    print()
    print("=" * 76)

    print(
        f"SEED-{seed_index} FAILURE DIAGNOSIS"
    )

    print("=" * 76)
    print()

    stdout_path = (
        LOG_DIR
        / (
            f"seed{seed_index:02d}_"
            "formal_stdout.txt"
        )
    )

    stderr_path = (
        LOG_DIR
        / (
            f"seed{seed_index:02d}_"
            "formal_stderr.txt"
        )
    )

    run_state_path = (
        seed_dir
        / "RSC_RUN_STATE.json"
    )


    # --------------------------------------------------------
    # Run state
    # --------------------------------------------------------

    print(
        "RUN STATE"
    )

    print(
        "---------"
    )

    print(
        f"path: {run_state_path}"
    )

    if run_state_path.is_file():

        print(
            f"sha256: {sha256_file(run_state_path)}"
        )

        run_state = read_json_if_exists(
            run_state_path
        )

        # Print the full failed run state.
        # It may contain traceback/error information but does
        # not require inspecting successful Seed-3 metrics.

        print(
            json.dumps(
                run_state,
                indent=2,
                sort_keys=False,
            )
        )

    else:

        print(
            "MISSING"
        )


    # --------------------------------------------------------
    # STDERR
    # --------------------------------------------------------

    print()
    print(
        "STDERR"
    )

    print(
        "------"
    )

    print(
        f"path: {stderr_path}"
    )

    stderr_text = read_text_if_exists(
        stderr_path
    )

    if stderr_text is None:

        print(
            "MISSING"
        )

    else:

        print(
            f"sha256: {sha256_file(stderr_path)}"
        )

        print()

        print(
            stderr_text
        )


    # --------------------------------------------------------
    # STDOUT
    #
    # Failed runs only.
    # If a failure occurred after metrics were printed, those
    # lines may be present; preserve the complete output rather
    # than selectively editing it.
    # --------------------------------------------------------

    print()
    print(
        "STDOUT"
    )

    print(
        "------"
    )

    print(
        f"path: {stdout_path}"
    )

    stdout_text = read_text_if_exists(
        stdout_path
    )

    if stdout_text is None:

        print(
            "MISSING"
        )

    else:

        print(
            f"sha256: {sha256_file(stdout_path)}"
        )

        print()

        print(
            stdout_text
        )


# ============================================================
# 7. SEED-3 STATUS ONLY — WITHHOLD SCIENTIFIC METRICS
# ============================================================

print()
print("=" * 76)
print(
    "SEED-3 SUCCESS PROVENANCE CHECK"
)
print("=" * 76)
print()

seed3_dir = SEEDS[
    3
][
    "dir"
]

seed3_state_path = (
    seed3_dir
    / "RSC_RUN_STATE.json"
)

seed3_summary_path = (
    seed3_dir
    / "RSC_EXTERNAL_DETECTION_SUMMARY.json"
)

seed3_predictions_path = (
    seed3_dir
    / "rsc_predictions_decoder_floor.jsonl"
)


seed3_state = read_json_if_exists(
    seed3_state_path
)

if seed3_state is None:

    print(
        "Seed-3 run state: MISSING"
    )

else:

    print(
        "Seed-3 run state: "
        f"{seed3_state.get('status')}"
    )

    print(
        "Seed-3 images_completed: "
        f"{seed3_state.get('images_completed')}"
    )

    print(
        "Seed-3 threshold tuning: "
        f"{seed3_state.get('threshold_tuning')}"
    )

    print(
        "Seed-3 calibration: "
        f"{seed3_state.get('calibration')}"
    )

    print(
        "Seed-3 checkpoint selection: "
        f"{seed3_state.get('checkpoint_selection')}"
    )

    print(
        "Seed-3 seed selection: "
        f"{seed3_state.get('seed_selection')}"
    )


print()

print_file_identity(
    seed3_summary_path,
    (
        "Seed-3 formal summary "
        "(identity only; metrics NOT displayed):"
    ),
)

print()

print_file_identity(
    seed3_predictions_path,
    (
        "Seed-3 decoder-floor predictions "
        "(identity only; contents NOT displayed):"
    ),
)


# ============================================================
# 8. END
# ============================================================

print()
print("=" * 76)

print(
    "DIAGNOSTIC COMPLETE"
)

print("=" * 76)

print()

print(
    "Do NOT rerun any formal seed yet."
)

print()

print(
    "Send the COMPLETE output of this diagnostic cell."
)

DASO-Net — RSC FORMAL RUN FAILURE DIAGNOSTIC v1

Scientific action:
  READ-ONLY failure diagnosis

NO inference
NO metric recomputation
NO tuning
NO calibration
NO selection
NO deletion

Formal execution contract:
  path:   /kaggle/working/dasonet_final_analysis/rsc_formal_three_seed_execution_v1/RSC_FORMAL_EXECUTION_CONTRACT_v1.json
  exists: True
  bytes:  2899
  sha256: ec433786bececd29c787155bc116e6641efb36aafaff46ac69898c6823d4873d

Formal execution state:
  path:   /kaggle/working/dasonet_final_analysis/rsc_formal_three_seed_execution_v1/RSC_FORMAL_EXECUTION_STATE_v1.json
  exists: True
  bytes:  2048
  sha256: e5bbc8806b9e47c8e20a54f4a33c85f472f72c6be1072493110bd101ffc0c162

Execution-state status:
  FAIL_ONE_OR_MORE_FORMAL_SEEDS

Recorded subprocess runs:
  Seed-1: status=FAIL returncode=1
  Seed-2: status=FAIL returncode=1
  Seed-3: status=PASS returncode=0

----------------------------------------------------------------------------
SEED-1 DIRECTORY INVENTORY
----------------

In [36]:
# ============================================================
# DASO-Net — RSC DEGENERATE-BOX FAILURE ROOT-CAUSE AUDIT
# RESUME02
#
# READ-ONLY.
#
# WHY RESUME02 EXISTS
# -------------------
# Resume01 correctly verified the frozen RSC wrapper but then
# assumed decode_instances was physically defined as a function
# inside evaluate_phase5c.py.
#
# Runtime evidence shows evaluate_phase5c.py exposes the symbol,
# but AST inspection found no local FunctionDef. Therefore this
# audit resolves the ACTUAL runtime-bound decoder object and uses
# inspect.getsourcefile/getsource/getsourcelines rather than
# assuming where its definition lives.
#
# NO model inference
# NO RSC metric computation
# NO threshold tuning
# NO calibration
# NO checkpoint selection
# NO seed selection
# NO frozen-file modification
# NO deletion / overwrite
#
# ============================================================

from __future__ import annotations

import ast
import csv
import hashlib
import importlib.util
import inspect
import json
import sys
import traceback
from pathlib import Path


# ============================================================
# 1. FROZEN PATHS / IDENTITIES
# ============================================================

IMPLEMENTATION_FREEZE_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_implementation_freeze_v1"
)

FROZEN_EVALUATOR = (
    IMPLEMENTATION_FREEZE_DIR
    / "evaluate_rsc_external_v1_FROZEN.py"
)

EXPECTED_EVALUATOR_SHA256 = (
    "59a3de31f2a5244ba32c53b705b050e0"
    "33b5d17c8de7efb36c17dbc49f0a7723"
)

FROZEN_PROTOCOL = (
    IMPLEMENTATION_FREEZE_DIR
    / "RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "e3c87caebafe87c7671b5745a480c0ed"
    "f67f444feac4278633da6addae4d1f7e"
)

EXECUTION_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_formal_three_seed_execution_v1"
)

EXECUTION_CONTRACT = (
    EXECUTION_DIR
    / "RSC_FORMAL_EXECUTION_CONTRACT_v1.json"
)

EXPECTED_EXECUTION_CONTRACT_SHA256 = (
    "ec433786bececd29c787155bc116e664"
    "1efb36aafaff46ac69898c6823d4873d"
)

FORMAL_ROOT = Path(
    "/kaggle/working/dasonet_final_evaluation/"
    "rsc_external_detection_v1"
)

SEED1_DIR = (
    FORMAL_ROOT
    / "seed01_s1782656963"
)

SEED2_DIR = (
    FORMAL_ROOT
    / "seed02_s901395089"
)

SEED3_DIR = (
    FORMAL_ROOT
    / "seed03_s252916394"
)

SEED1_STATE = (
    SEED1_DIR
    / "RSC_RUN_STATE.json"
)

SEED2_STATE = (
    SEED2_DIR
    / "RSC_RUN_STATE.json"
)

SEED3_STATE = (
    SEED3_DIR
    / "RSC_RUN_STATE.json"
)

SEED1_PREDICTIONS = (
    SEED1_DIR
    / "rsc_predictions_decoder_floor.jsonl"
)

SEED2_PREDICTIONS = (
    SEED2_DIR
    / "rsc_predictions_decoder_floor.jsonl"
)

SEED3_SUMMARY = (
    SEED3_DIR
    / "RSC_EXTERNAL_DETECTION_SUMMARY.json"
)


# ============================================================
# 2. FAILED AUDITS — MUST REMAIN PRESERVED
# ============================================================

FAILED_AUDIT_V1 = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_degenerate_box_root_cause_audit_v1"
)

FAILED_AUDIT_RESUME01 = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_degenerate_box_root_cause_audit_v1_resume01"
)

AUDIT_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_degenerate_box_root_cause_audit_v1_resume02"
)

AUDIT_JSON = (
    AUDIT_DIR
    / "RSC_DEGENERATE_BOX_ROOT_CAUSE_AUDIT_RESUME02.json"
)

AUDIT_SHA_RECORD = (
    AUDIT_DIR
    / "RSC_DEGENERATE_BOX_ROOT_CAUSE_AUDIT_RESUME02.sha256.txt"
)


# ============================================================
# 3. EXPECTED FAILURE
# ============================================================

EXPECTED_DEGENERATE_MESSAGE = (
    "Decoded box is degenerate before "
    "original-image clipping."
)


# ============================================================
# 4. HELPERS
# ============================================================

checks = []


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def require(
    condition: bool,
    message: str,
) -> None:
    if not condition:
        raise RuntimeError(message)


def require_file(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_file(),
        f"{label} missing:\n{path}",
    )


def require_dir(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_dir(),
        f"{label} missing:\n{path}",
    )


def read_json(path: Path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path: Path,
    obj,
) -> None:
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def record(
    name: str,
    **details,
) -> None:
    item = {
        "check": name,
        "status": "PASS",
    }

    item.update(details)

    checks.append(item)

    print(f"PASS  {name}")


def get_local_function_details(
    source_path: Path,
    function_name: str,
):
    text = source_path.read_text(
        encoding="utf-8"
    )

    lines = text.splitlines()

    tree = ast.parse(
        text,
        filename=str(source_path),
    )

    matches = [
        node
        for node in ast.walk(tree)
        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        )
        and node.name == function_name
    ]

    require(
        len(matches) == 1,
        (
            f"Expected exactly one local function named "
            f"{function_name!r} in:\n"
            f"{source_path}\n"
            f"Found: {len(matches)}"
        ),
    )

    node = matches[0]

    require(
        getattr(
            node,
            "end_lineno",
            None,
        )
        is not None,
        f"No end_lineno for {function_name}.",
    )

    start = int(node.lineno)
    end = int(node.end_lineno)

    source = "\n".join(
        lines[
            start - 1:
            end
        ]
    )

    constants = [
        child.value
        for child in ast.walk(node)
        if isinstance(
            child,
            ast.Constant,
        )
        and isinstance(
            child.value,
            str,
        )
    ]

    return {
        "start_line":
            start,

        "end_line":
            end,

        "source":
            source,

        "source_sha256":
            sha256_text(source),

        "string_constants":
            constants,
    }


def count_prediction_prefix(
    path: Path,
):
    require_file(
        path,
        "Partial prediction JSONL",
    )

    indices = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        for line_number, line in enumerate(
            f,
            start=1,
        ):
            text = line.strip()

            if not text:
                continue

            try:
                obj = json.loads(text)

            except Exception as exc:
                raise RuntimeError(
                    "Invalid JSONL.\n"
                    f"Path: {path}\n"
                    f"Line: {line_number}"
                ) from exc

            indices.append(
                int(
                    obj[
                        "official_test_index"
                    ]
                )
            )

    return {
        "count":
            len(indices),

        "exact_prefix":
            indices
            == list(
                range(
                    1,
                    len(indices) + 1,
                )
            ),

        "last_index":
            (
                indices[-1]
                if indices
                else None
            ),
    }


# ============================================================
# 5. BANNER / NON-OVERWRITE
# ============================================================

print("=" * 78)

print(
    "DASO-Net — RSC DEGENERATE-BOX FAILURE "
    "ROOT-CAUSE AUDIT RESUME02"
)

print("=" * 78)

print()

print("Scientific action:")
print("  READ-ONLY runtime-symbol/source diagnosis")
print()

print("NO inference")
print("NO metric computation")
print("NO threshold tuning")
print("NO calibration")
print("NO model/seed selection")
print("NO frozen-file modification")
print()


require_dir(
    FAILED_AUDIT_V1,
    "Failed root-cause audit v1",
)

require_dir(
    FAILED_AUDIT_RESUME01,
    "Failed root-cause audit Resume01",
)

record(
    "Failed audit v1 preserved",
    path=str(
        FAILED_AUDIT_V1
    ),
)

record(
    "Failed audit Resume01 preserved",
    path=str(
        FAILED_AUDIT_RESUME01
    ),
)


if AUDIT_DIR.exists():
    raise RuntimeError(
        "Resume02 audit directory already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{AUDIT_DIR}"
    )


AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 6. MAIN AUDIT
# ============================================================

try:

    # --------------------------------------------------------
    # 6A. Frozen identities
    # --------------------------------------------------------

    require_file(
        FROZEN_EVALUATOR,
        "Frozen RSC evaluator",
    )

    evaluator_sha = sha256_file(
        FROZEN_EVALUATOR
    )

    require(
        evaluator_sha
        == EXPECTED_EVALUATOR_SHA256,
        "Frozen evaluator SHA mismatch.",
    )

    record(
        "Frozen RSC evaluator identity",
        sha256=evaluator_sha,
    )


    require_file(
        FROZEN_PROTOCOL,
        "Frozen RSC protocol",
    )

    protocol_sha = sha256_file(
        FROZEN_PROTOCOL
    )

    require(
        protocol_sha
        == EXPECTED_PROTOCOL_SHA256,
        "Frozen protocol SHA mismatch.",
    )

    protocol = read_json(
        FROZEN_PROTOCOL
    )

    record(
        "Frozen scientific protocol identity",
        sha256=protocol_sha,
    )


    require_file(
        EXECUTION_CONTRACT,
        "Formal execution contract",
    )

    execution_contract_sha = sha256_file(
        EXECUTION_CONTRACT
    )

    require(
        execution_contract_sha
        == EXPECTED_EXECUTION_CONTRACT_SHA256,
        "Formal execution-contract SHA mismatch.",
    )

    record(
        "Formal execution-contract identity",
        sha256=execution_contract_sha,
    )


    # --------------------------------------------------------
    # 6B. Formal run states
    # --------------------------------------------------------

    for path, label in [
        (
            SEED1_STATE,
            "Seed-1 run state",
        ),
        (
            SEED2_STATE,
            "Seed-2 run state",
        ),
        (
            SEED3_STATE,
            "Seed-3 run state",
        ),
    ]:
        require_file(
            path,
            label,
        )


    seed1_state = read_json(
        SEED1_STATE
    )

    seed2_state = read_json(
        SEED2_STATE
    )

    seed3_state = read_json(
        SEED3_STATE
    )


    require(
        seed1_state.get("status")
        == "FAIL",
        "Seed-1 status changed.",
    )

    require(
        seed2_state.get("status")
        == "FAIL",
        "Seed-2 status changed.",
    )

    require(
        seed3_state.get("status")
        == "PASS",
        "Seed-3 status changed.",
    )

    require(
        seed1_state.get("error")
        == EXPECTED_DEGENERATE_MESSAGE,
        "Seed-1 failure reason changed.",
    )

    require(
        seed2_state.get("error")
        == EXPECTED_DEGENERATE_MESSAGE,
        "Seed-2 failure reason changed.",
    )

    require(
        int(
            seed1_state[
                "images_completed"
            ]
        )
        == 207,
        "Seed-1 completed-image count changed.",
    )

    require(
        int(
            seed2_state[
                "images_completed"
            ]
        )
        == 208,
        "Seed-2 completed-image count changed.",
    )

    require(
        int(
            seed3_state[
                "images_completed"
            ]
        )
        == 250,
        "Seed-3 completed-image count changed.",
    )


    record(
        "Seed-1 formal failure identity",
        images_completed=207,
    )

    record(
        "Seed-2 formal failure identity",
        images_completed=208,
    )

    record(
        "Seed-3 formal completion identity",
        images_completed=250,
    )


    # --------------------------------------------------------
    # 6C. Partial prefix closure
    # --------------------------------------------------------

    seed1_prefix = count_prediction_prefix(
        SEED1_PREDICTIONS
    )

    seed2_prefix = count_prediction_prefix(
        SEED2_PREDICTIONS
    )


    require(
        seed1_prefix[
            "count"
        ]
        == 207,
        "Seed-1 partial prediction count != 207.",
    )

    require(
        seed2_prefix[
            "count"
        ]
        == 208,
        "Seed-2 partial prediction count != 208.",
    )

    require(
        seed1_prefix[
            "exact_prefix"
        ]
        is True,
        "Seed-1 predictions are not exact prefix 1..207.",
    )

    require(
        seed2_prefix[
            "exact_prefix"
        ]
        is True,
        "Seed-2 predictions are not exact prefix 1..208.",
    )

    record(
        "Seed-1 prediction prefix 1..207"
    )

    record(
        "Seed-2 prediction prefix 1..208"
    )


    # --------------------------------------------------------
    # 6D. Failure-image identities
    # --------------------------------------------------------

    primary_manifest = Path(
        protocol[
            "scientific_sets"
        ][
            "primary"
        ][
            "manifest"
        ]
    )

    require_file(
        primary_manifest,
        "Frozen primary manifest",
    )

    require(
        sha256_file(
            primary_manifest
        )
        == protocol[
            "scientific_sets"
        ][
            "primary"
        ][
            "manifest_sha256"
        ],
        "Primary manifest SHA changed.",
    )


    with primary_manifest.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:

        manifest_rows = list(
            csv.DictReader(f)
        )


    manifest_by_index = {
        int(
            float(
                row[
                    "official_test_index"
                ]
            )
        ):
            row
        for row in manifest_rows
    }


    seed1_failure_index = 208
    seed2_failure_index = 209

    seed1_failure_filename = str(
        manifest_by_index[
            seed1_failure_index
        ][
            "filename"
        ]
    ).strip()

    seed2_failure_filename = str(
        manifest_by_index[
            seed2_failure_index
        ][
            "filename"
        ]
    ).strip()


    require(
        seed1_failure_filename
        == "test_60.jpg",
        "Seed-1 failure filename changed.",
    )

    require(
        seed2_failure_filename
        == "test_61.jpg",
        "Seed-2 failure filename changed.",
    )


    record(
        "Failure-image identities",
        seed1_index=208,
        seed1_filename=(
            seed1_failure_filename
        ),
        seed2_index=209,
        seed2_filename=(
            seed2_failure_filename
        ),
    )


    # ========================================================
    # 7. VERIFY FROZEN WRAPPER
    # ========================================================

    wrapper = get_local_function_details(
        FROZEN_EVALUATOR,
        "apply_original_image_box_policy",
    )


    require(
        EXPECTED_DEGENERATE_MESSAGE
        in wrapper[
            "string_constants"
        ],
        "Fatal pre-clipping message missing from "
        "wrapper AST constants.",
    )


    require(
        "x2 > x1"
        in wrapper[
            "source"
        ],
        "Expected pre-clipping x2>x1 assertion absent.",
    )

    require(
        "y2 > y1"
        in wrapper[
            "source"
        ],
        "Expected pre-clipping y2>y1 assertion absent.",
    )

    require(
        "x2 <= x1"
        in wrapper[
            "source"
        ]
        and "y2 <= y1"
        in wrapper[
            "source"
        ],
        "Expected post-clipping degeneracy filter absent.",
    )


    record(
        "Frozen wrapper pre-clipping fatal assertion"
    )

    record(
        "Frozen wrapper post-clipping discard rule"
    )


    print()

    print("=" * 78)

    print(
        "FROZEN WRAPPER — "
        "apply_original_image_box_policy"
    )

    print("=" * 78)

    print()

    print(
        f"Path: {FROZEN_EVALUATOR}"
    )

    print(
        f"Function lines: "
        f"{wrapper['start_line']}-"
        f"{wrapper['end_line']}"
    )

    print(
        f"Function-source SHA256: "
        f"{wrapper['source_sha256']}"
    )

    print()

    print(
        wrapper[
            "source"
        ]
    )


    # ========================================================
    # 8. RESOLVE RUNTIME-BOUND decode_instances
    #
    # This is the key correction from Resume01.
    # ========================================================

    phase5c_path = Path(
        protocol[
            "validated_evaluator"
        ][
            "script"
        ]
    )

    require_file(
        phase5c_path,
        "Validated Phase5C evaluator",
    )

    expected_phase5c_sha = (
        protocol[
            "validated_evaluator"
        ][
            "whole_file_sha256"
        ]
    )

    actual_phase5c_sha = sha256_file(
        phase5c_path
    )

    require(
        actual_phase5c_sha
        == expected_phase5c_sha,
        "Validated Phase5C evaluator SHA changed.",
    )

    record(
        "Validated Phase5C evaluator identity",
        sha256=actual_phase5c_sha,
    )


    # --------------------------------------------------------
    # Add same import roots used by frozen evaluator.
    # --------------------------------------------------------

    training_root = (
        phase5c_path.parent.parent
    )

    import_roots = [
        training_root,
        phase5c_path.parent,
        training_root / "src",
    ]

    added_import_roots = []

    for candidate in import_roots:

        if candidate.exists():

            candidate_text = str(
                candidate
            )

            if candidate_text not in sys.path:

                sys.path.insert(
                    0,
                    candidate_text,
                )

                added_import_roots.append(
                    candidate_text
                )


    # --------------------------------------------------------
    # Runtime import only.
    #
    # Importing the evaluator module does not perform RSC
    # inference. Its callable main() is guarded by __main__.
    # --------------------------------------------------------

    module_name = (
        "dasonet_phase5c_decoder_"
        "rootcause_resume02"
    )

    spec = (
        importlib.util
        .spec_from_file_location(
            module_name,
            phase5c_path,
        )
    )

    require(
        spec is not None
        and spec.loader is not None,
        "Could not create import specification for "
        "validated Phase5C evaluator.",
    )

    phase5c_module = (
        importlib.util
        .module_from_spec(
            spec
        )
    )

    sys.modules[
        module_name
    ] = phase5c_module

    spec.loader.exec_module(
        phase5c_module
    )


    require(
        hasattr(
            phase5c_module,
            "decode_instances",
        ),
        "Runtime-loaded Phase5C evaluator does not expose "
        "decode_instances.",
    )


    decoder_object = getattr(
        phase5c_module,
        "decode_instances",
    )


    require(
        callable(
            decoder_object
        ),
        "Runtime decode_instances is not callable.",
    )


    decoder_signature = str(
        inspect.signature(
            decoder_object
        )
    )


    # --------------------------------------------------------
    # Locate actual defining module/file.
    # --------------------------------------------------------

    decoder_module_name = getattr(
        decoder_object,
        "__module__",
        None,
    )

    decoder_qualname = getattr(
        decoder_object,
        "__qualname__",
        None,
    )

    decoder_name = getattr(
        decoder_object,
        "__name__",
        None,
    )


    decoder_source_file_raw = (
        inspect.getsourcefile(
            decoder_object
        )
    )

    if decoder_source_file_raw is None:
        decoder_source_file_raw = (
            inspect.getfile(
                decoder_object
            )
        )


    require(
        decoder_source_file_raw
        is not None,
        "Could not resolve source file for runtime "
        "decode_instances.",
    )


    decoder_source_file = Path(
        decoder_source_file_raw
    ).resolve()


    require_file(
        decoder_source_file,
        "Runtime decode_instances source file",
    )


    decoder_source_file_sha = sha256_file(
        decoder_source_file
    )


    decoder_source_lines, decoder_start_line = (
        inspect.getsourcelines(
            decoder_object
        )
    )

    decoder_source = "".join(
        decoder_source_lines
    ).rstrip("\n")

    decoder_end_line = (
        int(
            decoder_start_line
        )
        + len(
            decoder_source_lines
        )
        - 1
    )

    decoder_source_sha = sha256_text(
        decoder_source
    )


    require(
        len(
            decoder_source.strip()
        )
        > 0,
        "Resolved decoder source is empty.",
    )


    required_decoder_parameters = {
        "outputs",
        "input_size",
        "stride",
        "score_threshold",
        "topk",
        "nms_iou",
        "mask_threshold",
        "local_peak_kernel",
    }


    decoder_parameters = set(
        inspect.signature(
            decoder_object
        ).parameters
    )


    require(
        required_decoder_parameters.issubset(
            decoder_parameters
        ),
        "Runtime decoder signature is missing expected "
        "frozen decoder parameters.",
    )


    record(
        "Runtime decode_instances symbol resolved",
        module=decoder_module_name,
        qualname=decoder_qualname,
    )

    record(
        "Runtime decoder source file captured",
        path=str(
            decoder_source_file
        ),
        sha256=decoder_source_file_sha,
    )

    record(
        "Runtime decoder function source captured",
        start_line=int(
            decoder_start_line
        ),
        end_line=decoder_end_line,
        source_sha256=decoder_source_sha,
    )


    print()

    print("=" * 78)

    print(
        "RUNTIME-BOUND VALIDATED DECODER — "
        "decode_instances"
    )

    print("=" * 78)

    print()

    print(
        f"Exposed by evaluator:"
    )

    print(
        f"  {phase5c_path}"
    )

    print()

    print(
        f"Runtime object module:"
    )

    print(
        f"  {decoder_module_name}"
    )

    print()

    print(
        f"Runtime object name:"
    )

    print(
        f"  {decoder_name}"
    )

    print()

    print(
        f"Runtime object qualname:"
    )

    print(
        f"  {decoder_qualname}"
    )

    print()

    print(
        "Signature:"
    )

    print(
        f"  {decoder_signature}"
    )

    print()

    print(
        "Actual defining source file:"
    )

    print(
        f"  {decoder_source_file}"
    )

    print()

    print(
        "Defining source-file SHA256:"
    )

    print(
        f"  {decoder_source_file_sha}"
    )

    print()

    print(
        "Function lines:"
    )

    print(
        f"  {decoder_start_line}-"
        f"{decoder_end_line}"
    )

    print()

    print(
        "Function-source SHA256:"
    )

    print(
        f"  {decoder_source_sha}"
    )

    print()

    print(
        decoder_source
    )


    # ========================================================
    # 9. RECORD HOW evaluate_phase5c BINDS THE SYMBOL
    #
    # We do not require a particular syntax. We simply record
    # source lines mentioning decode_instances for provenance.
    # ========================================================

    phase5c_text = phase5c_path.read_text(
        encoding="utf-8"
    )

    phase5c_lines = (
        phase5c_text.splitlines()
    )

    binding_mentions = []

    for line_number, line in enumerate(
        phase5c_lines,
        start=1,
    ):

        if "decode_instances" in line:

            binding_mentions.append(
                {
                    "line_number":
                        line_number,

                    "text":
                        line,
                }
            )


    require(
        len(binding_mentions) >= 1,
        (
            "evaluate_phase5c.py exposes decode_instances at "
            "runtime but contains no textual symbol mention. "
            "Unexpected provenance condition."
        ),
    )


    print()

    print("=" * 78)

    print(
        "evaluate_phase5c.py — "
        "decode_instances SYMBOL MENTIONS"
    )

    print("=" * 78)

    print()

    for mention in binding_mentions:

        print(
            f"{mention['line_number']}: "
            f"{mention['text']}"
        )


    # ========================================================
    # 10. FROZEN COORDINATE POLICY
    # ========================================================

    coordinate_policy = (
        protocol[
            "input_protocol"
        ][
            "prediction_coordinate_policy"
        ]
    )


    require(
        coordinate_policy[
            "degenerate_after_clipping"
        ]
        == "discard",
        (
            "Frozen protocol no longer specifies "
            "degenerate_after_clipping='discard'."
        ),
    )


    record(
        "Frozen coordinate policy identity",
        degenerate_after_clipping="discard",
    )


    print()

    print("=" * 78)

    print(
        "ALREADY-FROZEN PREDICTION "
        "COORDINATE POLICY"
    )

    print("=" * 78)

    print()

    print(
        json.dumps(
            coordinate_policy,
            indent=2,
            sort_keys=False,
        )
    )


    # ========================================================
    # 11. SEED-3 IDENTITY ONLY
    # ========================================================

    require_file(
        SEED3_SUMMARY,
        "Seed-3 formal summary",
    )

    seed3_summary_sha = sha256_file(
        SEED3_SUMMARY
    )


    print()

    print(
        "Seed-3 formal summary remains preserved "
        "without displaying scientific metrics."
    )

    print(
        f"Seed-3 summary SHA256: "
        f"{seed3_summary_sha}"
    )


    # ========================================================
    # 12. STATIC CLASSIFICATION
    #
    # This classification remains deliberately narrow.
    # We do NOT yet prescribe the recovery edit here.
    # ========================================================

    classification = {
        "observed_formal_failures":
            {
                "seed1":
                    {
                        "images_completed":
                            207,

                        "failure_index":
                            208,

                        "failure_filename":
                            seed1_failure_filename,

                        "error":
                            EXPECTED_DEGENERATE_MESSAGE,
                    },

                "seed2":
                    {
                        "images_completed":
                            208,

                        "failure_index":
                            209,

                        "failure_filename":
                            seed2_failure_filename,

                        "error":
                            EXPECTED_DEGENERATE_MESSAGE,
                    },
            },

        "seed3_completed":
            True,

        "frozen_wrapper_preclip_degeneracy":
            "fatal",

        "frozen_wrapper_postclip_degeneracy":
            "discard",

        "frozen_protocol_postclip_degeneracy":
            "discard",

        "failure_class":
            (
                "external-evaluation wrapper robustness / "
                "decoded-geometry corner case"
            ),

        "model_change_indicated":
            False,

        "checkpoint_change_indicated":
            False,

        "seed_change_indicated":
            False,

        "threshold_change_indicated":
            False,

        "calibration_change_indicated":
            False,

        "validated_decoder_change_indicated":
            False,

        "recovery_policy_not_yet_executed":
            True,
    }


    record(
        "Failure classification documented"
    )


    # ========================================================
    # 13. WRITE RESUME02 AUDIT
    # ========================================================

    audit = {
        "audit_name":
            (
                "DASO-Net RSC Degenerate-Box Failure "
                "Root-Cause Audit Resume02"
            ),

        "status":
            "PASS",

        "reason_for_resume02":
            (
                "Resume01 assumed decode_instances had a local "
                "FunctionDef in evaluate_phase5c.py. Runtime "
                "inspection instead resolves the actual bound "
                "callable and its defining source."
            ),

        "failed_audits_preserved":
            [
                str(
                    FAILED_AUDIT_V1
                ),
                str(
                    FAILED_AUDIT_RESUME01
                ),
            ],

        "scientific_action":
            "READ_ONLY_RUNTIME_SYMBOL_SOURCE_DIAGNOSIS",

        "inference_performed":
            False,

        "metrics_computed":
            False,

        "threshold_tuning":
            False,

        "calibration":
            False,

        "checkpoint_selection":
            False,

        "seed_selection":
            False,

        "identities":
            {
                "frozen_evaluator_sha256":
                    evaluator_sha,

                "frozen_protocol_sha256":
                    protocol_sha,

                "formal_execution_contract_sha256":
                    execution_contract_sha,

                "validated_phase5c_evaluator_sha256":
                    actual_phase5c_sha,

                "frozen_wrapper_function_source_sha256":
                    wrapper[
                        "source_sha256"
                    ],

                "runtime_decoder_source_file_sha256":
                    decoder_source_file_sha,

                "runtime_decoder_function_source_sha256":
                    decoder_source_sha,

                "seed3_summary_sha256":
                    seed3_summary_sha,
            },

        "formal_failure_state":
            {
                "seed1":
                    {
                        "status":
                            "FAIL",

                        "images_completed":
                            207,

                        "failure_index":
                            208,

                        "failure_filename":
                            seed1_failure_filename,

                        "partial_prediction_prefix":
                            "1..207",
                    },

                "seed2":
                    {
                        "status":
                            "FAIL",

                        "images_completed":
                            208,

                        "failure_index":
                            209,

                        "failure_filename":
                            seed2_failure_filename,

                        "partial_prediction_prefix":
                            "1..208",
                    },

                "seed3":
                    {
                        "status":
                            "PASS",

                        "images_completed":
                            250,

                        "scientific_metrics_displayed_by_this_audit":
                            False,
                    },
            },

        "frozen_wrapper":
            {
                "function":
                    "apply_original_image_box_policy",

                "path":
                    str(
                        FROZEN_EVALUATOR
                    ),

                "start_line":
                    wrapper[
                        "start_line"
                    ],

                "end_line":
                    wrapper[
                        "end_line"
                    ],

                "source_sha256":
                    wrapper[
                        "source_sha256"
                    ],

                "fatal_preclip_degenerate_assertion":
                    True,

                "postclip_degenerate_discard":
                    True,
            },

        "runtime_decoder":
            {
                "symbol_exposed_by":
                    str(
                        phase5c_path
                    ),

                "runtime_module":
                    decoder_module_name,

                "runtime_name":
                    decoder_name,

                "runtime_qualname":
                    decoder_qualname,

                "signature":
                    decoder_signature,

                "actual_source_file":
                    str(
                        decoder_source_file
                    ),

                "source_file_sha256":
                    decoder_source_file_sha,

                "start_line":
                    int(
                        decoder_start_line
                    ),

                "end_line":
                    decoder_end_line,

                "function_source_sha256":
                    decoder_source_sha,

                "evaluate_phase5c_symbol_mentions":
                    binding_mentions,
            },

        "frozen_prediction_coordinate_policy":
            coordinate_policy,

        "classification":
            classification,

        "checks":
            checks,

        "readiness":
            (
                "READY_TO_FREEZE_"
                "DEGENERATE_BOX_RECOVERY_POLICY"
            ),
    }


    write_json(
        AUDIT_JSON,
        audit,
    )

    audit_sha = sha256_file(
        AUDIT_JSON
    )

    AUDIT_SHA_RECORD.write_text(
        (
            f"{audit_sha}  "
            f"{AUDIT_JSON.name}\n"
        ),
        encoding="utf-8",
    )


    # ========================================================
    # 14. READ-BACK
    # ========================================================

    readback = read_json(
        AUDIT_JSON
    )

    require(
        readback.get("status")
        == "PASS",
        "Resume02 audit read-back status failure.",
    )

    require(
        readback.get("readiness")
        == (
            "READY_TO_FREEZE_"
            "DEGENERATE_BOX_RECOVERY_POLICY"
        ),
        "Resume02 readiness failure.",
    )

    require(
        readback[
            "inference_performed"
        ]
        is False,
        "Resume02 audit unexpectedly claims inference.",
    )

    require(
        readback[
            "metrics_computed"
        ]
        is False,
        "Resume02 audit unexpectedly claims metric computation.",
    )

    require(
        sha256_file(
            AUDIT_JSON
        )
        == audit_sha,
        "Resume02 audit changed during read-back.",
    )


    # ========================================================
    # 15. FINAL REPORT
    # ========================================================

    print()

    print("=" * 78)

    print(
        "ROOT-CAUSE AUDIT RESUME02: PASS"
    )

    print("=" * 78)

    print()

    print(
        "Seed-1:"
    )

    print(
        "  completed: 207/250"
    )

    print(
        "  failure image: "
        f"208 / {seed1_failure_filename}"
    )

    print()

    print(
        "Seed-2:"
    )

    print(
        "  completed: 208/250"
    )

    print(
        "  failure image: "
        f"209 / {seed2_failure_filename}"
    )

    print()

    print(
        "Seed-3:"
    )

    print(
        "  completed: 250/250"
    )

    print(
        "  metrics displayed here: NO"
    )

    print()

    print(
        "Runtime decoder module:"
    )

    print(
        f"  {decoder_module_name}"
    )

    print()

    print(
        "Runtime decoder source file:"
    )

    print(
        f"  {decoder_source_file}"
    )

    print()

    print(
        "Runtime decoder source-file SHA256:"
    )

    print(
        f"  {decoder_source_file_sha}"
    )

    print()

    print(
        "Runtime decoder function-source SHA256:"
    )

    print(
        f"  {decoder_source_sha}"
    )

    print()

    print(
        "Failure classification:"
    )

    print(
        "  external-evaluation wrapper robustness / "
        "decoded-geometry corner case"
    )

    print()

    print(
        "Model modification indicated:      NO"
    )

    print(
        "Validated decoder modification:     NO"
    )

    print(
        "Checkpoint modification/selection:  NO"
    )

    print(
        "Seed modification/selection:        NO"
    )

    print(
        "Threshold modification/tuning:      NO"
    )

    print(
        "Calibration:                        NO"
    )

    print()

    print(
        "Resume02 audit JSON:"
    )

    print(
        f"  {AUDIT_JSON}"
    )

    print()

    print(
        "Resume02 audit SHA256:"
    )

    print(
        f"  {audit_sha}"
    )

    print()

    print(
        "READINESS:"
    )

    print(
        "  READY_TO_FREEZE_"
        "DEGENERATE_BOX_RECOVERY_POLICY"
    )

    print("=" * 78)


except Exception as exc:

    failure = {
        "audit_name":
            (
                "DASO-Net RSC Degenerate-Box Failure "
                "Root-Cause Audit Resume02"
            ),

        "status":
            "FAIL",

        "failed_prior_audits_preserved":
            (
                FAILED_AUDIT_V1.exists()
                and FAILED_AUDIT_RESUME01.exists()
            ),

        "scientific_action":
            "READ_ONLY_RUNTIME_SYMBOL_SOURCE_DIAGNOSIS",

        "inference_performed":
            False,

        "metrics_computed":
            False,

        "threshold_tuning":
            False,

        "calibration":
            False,

        "checkpoint_selection":
            False,

        "seed_selection":
            False,

        "error_type":
            type(exc).__name__,

        "error":
            str(exc),

        "traceback":
            traceback.format_exc(),

        "checks":
            checks,

        "readiness":
            "NOT_READY_FOR_RECOVERY_POLICY_FREEZE",
    }


    write_json(
        AUDIT_JSON,
        failure,
    )

    failure_sha = sha256_file(
        AUDIT_JSON
    )

    AUDIT_SHA_RECORD.write_text(
        (
            f"{failure_sha}  "
            f"{AUDIT_JSON.name}\n"
        ),
        encoding="utf-8",
    )

    print()

    print("=" * 78)

    print(
        "ROOT-CAUSE AUDIT RESUME02: FAIL"
    )

    print("=" * 78)

    print()

    print(
        "Failure preserved:"
    )

    print(
        f"  {AUDIT_JSON}"
    )

    print()

    print(
        "No RSC inference or metric computation "
        "was performed."
    )

    print()

    raise

DASO-Net — RSC DEGENERATE-BOX FAILURE ROOT-CAUSE AUDIT RESUME02

Scientific action:
  READ-ONLY runtime-symbol/source diagnosis

NO inference
NO metric computation
NO threshold tuning
NO calibration
NO model/seed selection
NO frozen-file modification

PASS  Failed audit v1 preserved
PASS  Failed audit Resume01 preserved
PASS  Frozen RSC evaluator identity
PASS  Frozen scientific protocol identity
PASS  Formal execution-contract identity
PASS  Seed-1 formal failure identity
PASS  Seed-2 formal failure identity
PASS  Seed-3 formal completion identity
PASS  Seed-1 prediction prefix 1..207
PASS  Seed-2 prediction prefix 1..208
PASS  Failure-image identities
PASS  Frozen wrapper pre-clipping fatal assertion
PASS  Frozen wrapper post-clipping discard rule

FROZEN WRAPPER — apply_original_image_box_policy

Path: /kaggle/working/dasonet_final_analysis/rsc_evaluator_implementation_freeze_v1/evaluate_rsc_external_v1_FROZEN.py
Function lines: 1471-1597
Function-source SHA256: 012a73ad333151526013

In [38]:
# ============================================================
# DASO-Net — RSC DEGENERATE-BOX RECOVERY POLICY FREEZE v1
#
# THIS CELL PERFORMS NO MODEL INFERENCE.
# THIS CELL COMPUTES NO SCIENTIFIC PERFORMANCE METRICS.
#
# PURPOSE
# -------
# Freeze a narrow, provenance-preserving recovery policy for the
# formal RSC run after:
#
#   Seed-1 failed after 207 completed images
#   Seed-2 failed after 208 completed images
#   Seed-3 completed 250/250
#
# Root cause:
#   The validated decoder can return a finite but geometrically
#   degenerate decoded box.  The original external wrapper treated
#   that condition as fatal before its original-image clipping step.
#
# Recovery:
#   Discard finite decoded boxes with non-positive area:
#
#       x2 <= x1 OR y2 <= y1
#
#   before the existing original-image center/clipping policy.
#
# Nothing else changes.
#
# ABSOLUTE PROHIBITIONS
# ---------------------
# NO model change
# NO decoder change
# NO NMS change
# NO score change
# NO threshold change
# NO calibration
# NO checkpoint selection
# NO seed selection
# NO test-set tuning
# NO sample removal
# NO primary/sensitivity membership change
#
# Recovery execution will later rerun ALL THREE seeds from image 1
# in a NEW output root. Existing formal outputs remain preserved.
# ============================================================

from __future__ import annotations

import hashlib
import json
import math
import traceback
from pathlib import Path


# ============================================================
# 1. FROZEN IDENTITIES
# ============================================================

ROOT_CAUSE_AUDIT_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_degenerate_box_root_cause_audit_v1_resume02"
)

ROOT_CAUSE_AUDIT_JSON = (
    ROOT_CAUSE_AUDIT_DIR
    / "RSC_DEGENERATE_BOX_ROOT_CAUSE_AUDIT_RESUME02.json"
)

EXPECTED_ROOT_CAUSE_AUDIT_SHA256 = (
    "8c150f118b193d2272a15387ccca74810"
    "dda0c71e4601a80ce6516b1128f078b"
)


IMPLEMENTATION_FREEZE_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_implementation_freeze_v1"
)

FROZEN_EVALUATOR = (
    IMPLEMENTATION_FREEZE_DIR
    / "evaluate_rsc_external_v1_FROZEN.py"
)

EXPECTED_FROZEN_EVALUATOR_SHA256 = (
    "59a3de31f2a5244ba32c53b705b050e0"
    "33b5d17c8de7efb36c17dbc49f0a7723"
)

FROZEN_PROTOCOL = (
    IMPLEMENTATION_FREEZE_DIR
    / "RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

EXPECTED_FROZEN_PROTOCOL_SHA256 = (
    "e3c87caebafe87c7671b5745a480c0ed"
    "f67f444feac4278633da6addae4d1f7e"
)


ORIGINAL_FORMAL_EXECUTION_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_formal_three_seed_execution_v1"
)

ORIGINAL_FORMAL_CONTRACT = (
    ORIGINAL_FORMAL_EXECUTION_DIR
    / "RSC_FORMAL_EXECUTION_CONTRACT_v1.json"
)

EXPECTED_ORIGINAL_FORMAL_CONTRACT_SHA256 = (
    "ec433786bececd29c787155bc116e664"
    "1efb36aafaff46ac69898c6823d4873d"
)

ORIGINAL_FORMAL_STATE = (
    ORIGINAL_FORMAL_EXECUTION_DIR
    / "RSC_FORMAL_EXECUTION_STATE_v1.json"
)


ORIGINAL_FORMAL_ROOT = Path(
    "/kaggle/working/dasonet_final_evaluation/"
    "rsc_external_detection_v1"
)

SEED1_OLD_DIR = (
    ORIGINAL_FORMAL_ROOT
    / "seed01_s1782656963"
)

SEED2_OLD_DIR = (
    ORIGINAL_FORMAL_ROOT
    / "seed02_s901395089"
)

SEED3_OLD_DIR = (
    ORIGINAL_FORMAL_ROOT
    / "seed03_s252916394"
)


SEED1_OLD_STATE = (
    SEED1_OLD_DIR
    / "RSC_RUN_STATE.json"
)

SEED2_OLD_STATE = (
    SEED2_OLD_DIR
    / "RSC_RUN_STATE.json"
)

SEED3_OLD_STATE = (
    SEED3_OLD_DIR
    / "RSC_RUN_STATE.json"
)


SEED3_OLD_SUMMARY = (
    SEED3_OLD_DIR
    / "RSC_EXTERNAL_DETECTION_SUMMARY.json"
)

SEED3_OLD_PREDICTIONS = (
    SEED3_OLD_DIR
    / "rsc_predictions_decoder_floor.jsonl"
)

SEED3_OLD_PRIMARY_CSV = (
    SEED3_OLD_DIR
    / "rsc_primary250_per_image.csv"
)

SEED3_OLD_DEDUP_CSV = (
    SEED3_OLD_DIR
    / "rsc_dedup247_per_image.csv"
)


EXPECTED_SEED3_OLD_SUMMARY_SHA256 = (
    "332334dcce91ffb0f54d9bc6875d04a2"
    "0f07e7d0f0e137c45e444a71c9fe4577"
)

EXPECTED_SEED3_OLD_PREDICTIONS_SHA256 = (
    "7d2e53aa90dec26497f6e96f811af356"
    "4257813e8b38ddf24ba879d951f61f1b"
)


EXPECTED_WRAPPER_FUNCTION_SOURCE_SHA256 = (
    "012a73ad3331515260136ccd3cdbf801"
    "579185b3761cd689f17e81c803dd896e"
)

EXPECTED_DECODER_FUNCTION_SOURCE_SHA256 = (
    "c6cc5ffeb08eeb0c9dfce89176cd8698"
    "693a5f3011071fa55d48cb66e990bedd"
)


# ============================================================
# 2. NEW RECOVERY POLICY OUTPUT
# ============================================================

RECOVERY_POLICY_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_degenerate_box_recovery_policy_freeze_v1"
)

RECOVERY_POLICY_JSON = (
    RECOVERY_POLICY_DIR
    / "RSC_DEGENERATE_BOX_RECOVERY_POLICY_FREEZE_v1.json"
)

RECOVERY_POLICY_SHA_RECORD = (
    RECOVERY_POLICY_DIR
    / "RSC_DEGENERATE_BOX_RECOVERY_POLICY_FREEZE_v1.sha256.txt"
)


# Future recovery paths are frozen here BEFORE recovery inference.

FUTURE_RECOVERY_EVALUATOR_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_recovery_evaluator_implementation_freeze_v1"
)

FUTURE_RECOVERY_OUTPUT_ROOT = Path(
    "/kaggle/working/dasonet_final_evaluation/"
    "rsc_external_detection_v1_recovery01"
)

FUTURE_RECOVERY_EXECUTION_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_formal_three_seed_recovery_execution_v1"
)


# ============================================================
# 3. FIXED FINAL MODEL CONTRACT
# ============================================================

EXPECTED_MODELS = [
    {
        "seed_index": 1,
        "seed": 1782656963,
        "operating_threshold": 0.30,
        "successful_optimizer_updates": 2750,
        "checkpoint_sha256":
            (
                "e4c4b79ad4b8c6e18ba2b1e3d6cbe796"
                "c9cc301bd6fcc7c596fc290ad3a7e81d"
            ),
    },
    {
        "seed_index": 2,
        "seed": 901395089,
        "operating_threshold": 0.35,
        "successful_optimizer_updates": 2750,
        "checkpoint_sha256":
            (
                "19cdb834a46240916b2c8023e8c3afe57"
                "38d9c7cd5d8f818c62ad1e289df1ad2"
            ),
    },
    {
        "seed_index": 3,
        "seed": 252916394,
        "operating_threshold": 0.35,
        "successful_optimizer_updates": 2750,
        "checkpoint_sha256":
            (
                "be7b7941f2f62179e15074aa973bf6da0"
                "eb9018ec9133486512bd1f87153f447"
            ),
    },
]


# ============================================================
# 4. HELPERS
# ============================================================

checks = []


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def require(
    condition: bool,
    message: str,
) -> None:
    if not condition:
        raise RuntimeError(message)


def require_file(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_file(),
        f"{label} missing:\n{path}",
    )


def require_dir(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_dir(),
        f"{label} missing:\n{path}",
    )


def read_json(path: Path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path: Path,
    obj,
) -> None:
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def record(
    name: str,
    **details,
) -> None:
    item = {
        "check": name,
        "status": "PASS",
    }

    item.update(details)

    checks.append(item)

    print(
        f"PASS  {name}"
    )


# ============================================================
# 5. BANNER / NON-OVERWRITE
# ============================================================

print("=" * 78)

print(
    "DASO-Net — RSC DEGENERATE-BOX "
    "RECOVERY POLICY FREEZE v1"
)

print("=" * 78)

print()

print(
    "THIS CELL PERFORMS NO MODEL INFERENCE."
)

print(
    "THIS CELL COMPUTES NO SCIENTIFIC PERFORMANCE METRICS."
)

print()

print(
    "Purpose:"
)

print(
    "  freeze one narrow invalid-geometry sanitation rule"
)

print(
    "  before any RSC recovery inference is performed"
)

print()


if RECOVERY_POLICY_DIR.exists():
    raise RuntimeError(
        "Recovery-policy freeze directory already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{RECOVERY_POLICY_DIR}"
    )


require(
    not FUTURE_RECOVERY_EVALUATOR_DIR.exists(),
    "Future recovery-evaluator freeze directory already exists:\n"
    f"{FUTURE_RECOVERY_EVALUATOR_DIR}",
)

require(
    not FUTURE_RECOVERY_OUTPUT_ROOT.exists(),
    "Future recovery formal-output root already exists:\n"
    f"{FUTURE_RECOVERY_OUTPUT_ROOT}",
)

require(
    not FUTURE_RECOVERY_EXECUTION_DIR.exists(),
    "Future recovery execution directory already exists:\n"
    f"{FUTURE_RECOVERY_EXECUTION_DIR}",
)


RECOVERY_POLICY_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 6. VERIFY ROOT-CAUSE AUDIT
# ============================================================

try:

    require_file(
        ROOT_CAUSE_AUDIT_JSON,
        "Root-cause audit Resume02",
    )

    root_cause_sha = sha256_file(
        ROOT_CAUSE_AUDIT_JSON
    )

    require(
        root_cause_sha
        == EXPECTED_ROOT_CAUSE_AUDIT_SHA256,
        "Root-cause audit SHA mismatch.\n"
        f"Expected: {EXPECTED_ROOT_CAUSE_AUDIT_SHA256}\n"
        f"Actual:   {root_cause_sha}",
    )

    root_cause = read_json(
        ROOT_CAUSE_AUDIT_JSON
    )

    require(
        root_cause.get("status")
        == "PASS",
        "Root-cause audit is not PASS.",
    )

    require(
        root_cause.get("readiness")
        == (
            "READY_TO_FREEZE_"
            "DEGENERATE_BOX_RECOVERY_POLICY"
        ),
        "Root-cause audit readiness changed.",
    )

    require(
        root_cause[
            "inference_performed"
        ]
        is False,
        "Root-cause audit unexpectedly performed inference.",
    )

    require(
        root_cause[
            "metrics_computed"
        ]
        is False,
        "Root-cause audit unexpectedly computed metrics.",
    )


    classification = root_cause[
        "classification"
    ]

    require(
        classification[
            "failure_class"
        ]
        == (
            "external-evaluation wrapper robustness / "
            "decoded-geometry corner case"
        ),
        "Unexpected failure classification.",
    )

    for key in [
        "model_change_indicated",
        "checkpoint_change_indicated",
        "seed_change_indicated",
        "threshold_change_indicated",
        "calibration_change_indicated",
        "validated_decoder_change_indicated",
    ]:
        require(
            classification[key]
            is False,
            (
                "Root-cause audit unexpectedly indicates "
                f"a scientific change: {key}"
            ),
        )


    identities = root_cause[
        "identities"
    ]

    require(
        identities[
            "frozen_evaluator_sha256"
        ]
        == EXPECTED_FROZEN_EVALUATOR_SHA256,
        "Root-cause audit evaluator identity changed.",
    )

    require(
        identities[
            "frozen_protocol_sha256"
        ]
        == EXPECTED_FROZEN_PROTOCOL_SHA256,
        "Root-cause audit protocol identity changed.",
    )

    require(
        identities[
            "formal_execution_contract_sha256"
        ]
        == EXPECTED_ORIGINAL_FORMAL_CONTRACT_SHA256,
        "Root-cause audit formal-contract identity changed.",
    )

    require(
        identities[
            "frozen_wrapper_function_source_sha256"
        ]
        == EXPECTED_WRAPPER_FUNCTION_SOURCE_SHA256,
        "Root-cause audit wrapper function SHA changed.",
    )

    require(
        identities[
            "runtime_decoder_function_source_sha256"
        ]
        == EXPECTED_DECODER_FUNCTION_SOURCE_SHA256,
        "Root-cause audit decoder function SHA changed.",
    )


    record(
        "Root-cause audit Resume02 identity",
        sha256=root_cause_sha,
    )


    # ========================================================
    # 7. VERIFY ORIGINAL FROZEN EVALUATOR / PROTOCOL
    # ========================================================

    require_file(
        FROZEN_EVALUATOR,
        "Original frozen RSC evaluator",
    )

    original_evaluator_sha = sha256_file(
        FROZEN_EVALUATOR
    )

    require(
        original_evaluator_sha
        == EXPECTED_FROZEN_EVALUATOR_SHA256,
        "Original frozen evaluator changed.",
    )

    record(
        "Original frozen evaluator preserved",
        sha256=original_evaluator_sha,
    )


    require_file(
        FROZEN_PROTOCOL,
        "Frozen RSC scientific protocol",
    )

    protocol_sha = sha256_file(
        FROZEN_PROTOCOL
    )

    require(
        protocol_sha
        == EXPECTED_FROZEN_PROTOCOL_SHA256,
        "Frozen RSC protocol changed.",
    )

    protocol = read_json(
        FROZEN_PROTOCOL
    )

    require(
        protocol.get("status")
        == "FROZEN_BEFORE_RSC_INFERENCE",
        "Frozen protocol status changed.",
    )

    record(
        "Frozen scientific protocol preserved",
        sha256=protocol_sha,
    )


    # ========================================================
    # 8. VERIFY ALREADY-FROZEN COORDINATE POLICY
    # ========================================================

    coordinate_policy = (
        protocol[
            "input_protocol"
        ][
            "prediction_coordinate_policy"
        ]
    )

    require(
        coordinate_policy[
            "prediction_center_filter"
        ]
        == (
            "discard decoded predictions whose box center lies "
            "outside the original image extent"
        ),
        "Frozen prediction-center policy changed.",
    )

    require(
        coordinate_policy[
            "box_clipping"
        ]
        == (
            "clip retained predicted boxes to original image "
            "bounds"
        ),
        "Frozen box-clipping policy changed.",
    )

    require(
        coordinate_policy[
            "degenerate_after_clipping"
        ]
        == "discard",
        "Frozen post-clipping degeneracy policy changed.",
    )

    require(
        coordinate_policy[
            "rescaling"
        ]
        is False,
        "Frozen coordinate policy unexpectedly enables rescaling.",
    )

    record(
        "Frozen original-coordinate policy preserved"
    )


    # ========================================================
    # 9. VERIFY DECODER REMAINS THE VALIDATED DECODER
    # ========================================================

    decoder = (
        protocol[
            "validated_evaluator"
        ][
            "decoder"
        ]
    )

    require(
        float(
            decoder[
                "decode_min_score"
            ]
        )
        == 0.05,
        "Frozen decoder floor changed.",
    )

    require(
        int(
            decoder[
                "topk"
            ]
        )
        == 100,
        "Frozen decoder top-k changed.",
    )

    require(
        float(
            decoder[
                "nms_iou"
            ]
        )
        == 0.50,
        "Frozen decoder NMS IoU changed.",
    )

    require(
        float(
            decoder[
                "mask_threshold"
            ]
        )
        == 0.50,
        "Frozen decoder mask threshold changed.",
    )

    require(
        int(
            decoder[
                "local_peak_kernel"
            ]
        )
        == 3,
        "Frozen decoder local-peak kernel changed.",
    )

    record(
        "Validated decoder settings preserved",
        decode_min_score=0.05,
        topk=100,
        nms_iou=0.50,
        mask_threshold=0.50,
        local_peak_kernel=3,
    )


    # ========================================================
    # 10. VERIFY ALL THREE FINAL MODELS / THRESHOLDS
    # ========================================================

    protocol_models = (
        protocol[
            "models"
        ]
    )

    require(
        len(protocol_models) == 3,
        "Frozen protocol does not contain exactly 3 models.",
    )


    for expected in EXPECTED_MODELS:

        seed_index = expected[
            "seed_index"
        ]

        matches = [
            model
            for model
            in protocol_models
            if int(
                model[
                    "seed_index"
                ]
            )
            == seed_index
        ]

        require(
            len(matches) == 1,
            f"Could not uniquely resolve Seed-{seed_index}.",
        )

        model = matches[0]

        require(
            int(
                model[
                    "seed"
                ]
            )
            == expected[
                "seed"
            ],
            f"Seed-{seed_index} RNG identity changed.",
        )

        require(
            float(
                model[
                    "operating_threshold"
                ]
            )
            == expected[
                "operating_threshold"
            ],
            f"Seed-{seed_index} operating threshold changed.",
        )

        require(
            int(
                model[
                    "successful_optimizer_updates"
                ]
            )
            == 2750,
            f"Seed-{seed_index} update count changed.",
        )

        require(
            str(
                model[
                    "checkpoint_sha256"
                ]
            ).lower()
            == expected[
                "checkpoint_sha256"
            ],
            f"Seed-{seed_index} checkpoint identity changed.",
        )

        checkpoint = Path(
            model[
                "checkpoint"
            ]
        )

        require_file(
            checkpoint,
            f"Seed-{seed_index} checkpoint",
        )

        require(
            sha256_file(
                checkpoint
            )
            == expected[
                "checkpoint_sha256"
            ],
            f"Seed-{seed_index} checkpoint bytes changed.",
        )

        record(
            f"Final Seed-{seed_index} identity",
            seed=expected[
                "seed"
            ],
            operating_threshold=expected[
                "operating_threshold"
            ],
            successful_optimizer_updates=2750,
        )


    # ========================================================
    # 11. VERIFY ORIGINAL FORMAL ATTEMPT IS PRESERVED
    # ========================================================

    require_file(
        ORIGINAL_FORMAL_CONTRACT,
        "Original formal execution contract",
    )

    original_contract_sha = sha256_file(
        ORIGINAL_FORMAL_CONTRACT
    )

    require(
        original_contract_sha
        == EXPECTED_ORIGINAL_FORMAL_CONTRACT_SHA256,
        "Original formal execution contract changed.",
    )

    require_file(
        ORIGINAL_FORMAL_STATE,
        "Original formal execution state",
    )

    original_state = read_json(
        ORIGINAL_FORMAL_STATE
    )

    require(
        original_state.get("status")
        == "FAIL_ONE_OR_MORE_FORMAL_SEEDS",
        "Original formal execution status changed.",
    )

    record(
        "Original failed formal execution preserved",
        contract_sha256=original_contract_sha,
    )


    # ========================================================
    # 12. VERIFY ORIGINAL SEED STATES
    # ========================================================

    for path, label in [
        (
            SEED1_OLD_STATE,
            "Original Seed-1 failed state",
        ),
        (
            SEED2_OLD_STATE,
            "Original Seed-2 failed state",
        ),
        (
            SEED3_OLD_STATE,
            "Original Seed-3 successful state",
        ),
    ]:
        require_file(
            path,
            label,
        )


    seed1_old = read_json(
        SEED1_OLD_STATE
    )

    seed2_old = read_json(
        SEED2_OLD_STATE
    )

    seed3_old = read_json(
        SEED3_OLD_STATE
    )


    require(
        seed1_old.get("status")
        == "FAIL",
        "Original Seed-1 status changed.",
    )

    require(
        seed2_old.get("status")
        == "FAIL",
        "Original Seed-2 status changed.",
    )

    require(
        seed3_old.get("status")
        == "PASS",
        "Original Seed-3 status changed.",
    )

    require(
        int(
            seed1_old[
                "images_completed"
            ]
        )
        == 207,
        "Original Seed-1 completion count changed.",
    )

    require(
        int(
            seed2_old[
                "images_completed"
            ]
        )
        == 208,
        "Original Seed-2 completion count changed.",
    )

    require(
        int(
            seed3_old[
                "images_completed"
            ]
        )
        == 250,
        "Original Seed-3 completion count changed.",
    )

    record(
        "Original Seed-1 failure preserved",
        images_completed=207,
    )

    record(
        "Original Seed-2 failure preserved",
        images_completed=208,
    )

    record(
        "Original Seed-3 success preserved",
        images_completed=250,
    )


    # ========================================================
    # 13. FREEZE OLD SEED-3 CONSISTENCY REFERENCES
    #
    # We hash artifacts only. We do NOT print or extract its
    # scientific metric values.
    # ========================================================

    require_file(
        SEED3_OLD_SUMMARY,
        "Original Seed-3 summary",
    )

    require_file(
        SEED3_OLD_PREDICTIONS,
        "Original Seed-3 predictions",
    )

    require_file(
        SEED3_OLD_PRIMARY_CSV,
        "Original Seed-3 primary per-image CSV",
    )

    require_file(
        SEED3_OLD_DEDUP_CSV,
        "Original Seed-3 dedup per-image CSV",
    )


    seed3_summary_sha = sha256_file(
        SEED3_OLD_SUMMARY
    )

    seed3_predictions_sha = sha256_file(
        SEED3_OLD_PREDICTIONS
    )

    seed3_primary_csv_sha = sha256_file(
        SEED3_OLD_PRIMARY_CSV
    )

    seed3_dedup_csv_sha = sha256_file(
        SEED3_OLD_DEDUP_CSV
    )


    require(
        seed3_summary_sha
        == EXPECTED_SEED3_OLD_SUMMARY_SHA256,
        "Original Seed-3 summary SHA changed.",
    )

    require(
        seed3_predictions_sha
        == EXPECTED_SEED3_OLD_PREDICTIONS_SHA256,
        "Original Seed-3 predictions SHA changed.",
    )


    record(
        "Original Seed-3 consistency references frozen",
        summary_sha256=seed3_summary_sha,
        predictions_sha256=seed3_predictions_sha,
        primary_per_image_sha256=seed3_primary_csv_sha,
        dedup_per_image_sha256=seed3_dedup_csv_sha,
    )


    # ========================================================
    # 14. FREEZE RECOVERY RULE
    # ========================================================

    recovery_rule = {

        # ----------------------------------------------------
        # Exact location in inference pipeline
        # ----------------------------------------------------

        "application_point":
            (
                "after validated decode_instances output has been "
                "normalized to finite [x1,y1,x2,y2] + score records, "
                "and before the existing original-image center filter "
                "and clipping policy"
            ),

        # ----------------------------------------------------
        # Structural validation that remains fatal
        # ----------------------------------------------------

        "fatal_conditions_unchanged":
            {
                "box_coordinate_count_not_four":
                    True,

                "nonfinite_box_coordinate":
                    True,

                "nonfinite_score":
                    True,
            },

        # ----------------------------------------------------
        # Narrow recovery
        # ----------------------------------------------------

        "new_recovery_condition":
            (
                "for a finite four-coordinate decoded box, "
                "if x2 <= x1 OR y2 <= y1"
            ),

        "new_recovery_action":
            "discard_prediction",

        "new_recovery_reason":
            (
                "non-positive-area decoded geometry is not a valid "
                "bounding box and cannot define a valid positive-area "
                "detection region"
            ),

        # ----------------------------------------------------
        # Absolutely unchanged behavior
        # ----------------------------------------------------

        "score_modified":
            False,

        "score_threshold_modified":
            False,

        "decoder_modified":
            False,

        "nms_modified":
            False,

        "topk_modified":
            False,

        "mask_threshold_modified":
            False,

        "local_peak_rule_modified":
            False,

        "model_modified":
            False,

        "checkpoint_modified":
            False,

        "seed_modified":
            False,

        "input_preprocessing_modified":
            False,

        "ground_truth_modified":
            False,

        "metric_definition_modified":
            False,

        "primary_membership_modified":
            False,

        "sensitivity_membership_modified":
            False,

        # ----------------------------------------------------
        # Existing original-image policy remains exact
        # ----------------------------------------------------

        "subsequent_center_filter":
            coordinate_policy[
                "prediction_center_filter"
            ],

        "subsequent_original_image_clipping":
            coordinate_policy[
                "box_clipping"
            ],

        "subsequent_postclipping_degenerate_action":
            "discard",

        "prediction_rescaling":
            False,

        # ----------------------------------------------------
        # Ordering
        # ----------------------------------------------------

        "retained_prediction_score_order":
            "unchanged",

        "retained_prediction_decoder_order":
            "unchanged",

        "removed_predictions":
            (
                "only predictions satisfying the finite "
                "non-positive-area condition"
            ),
    }


    # ========================================================
    # 15. FREEZE RECOVERY DIAGNOSTICS
    # ========================================================

    recovery_diagnostics = {
        "per_image_required":
            True,

        "per_seed_required":
            True,

        "required_counters":
            [
                "decoded_predictions_before_geometry_sanitation",
                "preexternalclip_degenerate_predictions_discarded",
                "predictions_after_geometry_sanitation",
            ],

        "diagnostic_only":
            True,

        "may_influence_threshold":
            False,

        "may_influence_calibration":
            False,

        "may_influence_checkpoint_selection":
            False,

        "may_influence_seed_selection":
            False,

        "may_exclude_images":
            False,

        "may_change_primary_membership":
            False,

        "may_change_sensitivity_membership":
            False,
    }


    # ========================================================
    # 16. FREEZE RECOVERY EXECUTION RULE
    # ========================================================

    recovery_execution = {
        "old_formal_outputs_preserved":
            True,

        "reuse_old_failed_output_directories":
            False,

        "resume_from_seed1_image_208":
            False,

        "resume_from_seed2_image_209":
            False,

        "rerun_seed1_from_image_1":
            True,

        "rerun_seed2_from_image_1":
            True,

        "rerun_seed3_from_image_1":
            True,

        "all_three_seeds_required":
            True,

        "new_output_root":
            str(
                FUTURE_RECOVERY_OUTPUT_ROOT
            ),

        "new_execution_directory":
            str(
                FUTURE_RECOVERY_EXECUTION_DIR
            ),

        "seed_order":
            [
                1,
                2,
                3,
            ],

        "intermediate_scientific_metrics_displayed":
            False,

        "continue_remaining_seeds_if_prior_seed_fails":
            True,

        "metric_based_early_stop":
            False,

        "final_aggregate_source":
            (
                "recovery01 outputs for Seed-1, Seed-2, and Seed-3 "
                "only; do not mix original Seed-3 result with "
                "recovery Seed-1/2"
            ),

        "aggregate_rule":
            "mean ± sample SD across all three recovery-run final seeds",

        "primary_before_sensitivity":
            True,

        "rsc_threshold_tuning":
            False,

        "rsc_calibration":
            False,

        "checkpoint_selection":
            False,

        "seed_selection":
            False,

        "best_seed_selection":
            False,
    }


    # ========================================================
    # 17. FREEZE SEED-3 CONSISTENCY GATE
    #
    # Old Seed-3 passed the fatal wrapper across all 250 images.
    # Therefore deterministic recovery Seed-3 must discard zero
    # pre-external-clipping degenerate boxes.
    #
    # Its prediction stream and per-image tables should remain
    # byte-identical if the recovery evaluator changes nothing
    # else.
    # ========================================================

    seed3_consistency_gate = {
        "purpose":
            (
                "prove that the recovery changes only the failing "
                "degenerate-geometry corner case"
            ),

        "old_seed3_completed_all_250_under_fatal_wrapper":
            True,

        "recovery_seed3_required_preexternalclip_degenerate_discard_count":
            0,

        "recovery_seed3_predictions_jsonl_must_match_old_sha":
            seed3_predictions_sha,

        "recovery_seed3_primary_per_image_csv_must_match_old_sha":
            seed3_primary_csv_sha,

        "recovery_seed3_dedup_per_image_csv_must_match_old_sha":
            seed3_dedup_csv_sha,

        "summary_json_byte_identity_required":
            False,

        "summary_json_byte_identity_not_required_reason":
            (
                "recovery summary will carry a different recovery "
                "evaluator/provenance identity even when scientific "
                "values are identical"
            ),

        "scientific_metric_values_must_match_old_seed3_exactly":
            True,

        "consistency_failure_action":
            (
                "stop before accepting any three-seed recovery "
                "aggregate and diagnose; do not tune or select"
            ),
    }


    # ========================================================
    # 18. COMPLETE POLICY FREEZE
    # ========================================================

    policy = {
        "policy_name":
            (
                "DASO-Net RSC Degenerate-Box "
                "Recovery Policy v1"
            ),

        "status":
            "FROZEN_BEFORE_RECOVERY_INFERENCE",

        "scientific_interpretation":
            (
                "Infrastructure/evaluation-wrapper robustness "
                "recovery for invalid decoded geometry. "
                "This is not a model, decoder, threshold, "
                "calibration, checkpoint, seed, dataset, or "
                "metric change."
            ),

        "provenance":
            {
                "root_cause_audit":
                    str(
                        ROOT_CAUSE_AUDIT_JSON
                    ),

                "root_cause_audit_sha256":
                    root_cause_sha,

                "original_frozen_evaluator":
                    str(
                        FROZEN_EVALUATOR
                    ),

                "original_frozen_evaluator_sha256":
                    original_evaluator_sha,

                "frozen_scientific_protocol":
                    str(
                        FROZEN_PROTOCOL
                    ),

                "frozen_scientific_protocol_sha256":
                    protocol_sha,

                "original_formal_execution_contract":
                    str(
                        ORIGINAL_FORMAL_CONTRACT
                    ),

                "original_formal_execution_contract_sha256":
                    original_contract_sha,

                "wrapper_function_source_sha256":
                    EXPECTED_WRAPPER_FUNCTION_SOURCE_SHA256,

                "validated_decoder_function_source_sha256":
                    EXPECTED_DECODER_FUNCTION_SOURCE_SHA256,
            },

        "observed_failure":
            {
                "seed1":
                    {
                        "status":
                            "FAIL",

                        "completed_images":
                            207,

                        "failure_official_test_index":
                            208,

                        "failure_filename":
                            "test_60.jpg",

                        "failure":
                            (
                                "Decoded box is degenerate before "
                                "original-image clipping."
                            ),
                    },

                "seed2":
                    {
                        "status":
                            "FAIL",

                        "completed_images":
                            208,

                        "failure_official_test_index":
                            209,

                        "failure_filename":
                            "test_61.jpg",

                        "failure":
                            (
                                "Decoded box is degenerate before "
                                "original-image clipping."
                            ),
                    },

                "seed3":
                    {
                        "status":
                            "PASS",

                        "completed_images":
                            250,

                        "scientific_metrics_used_to_define_recovery":
                            False,
                    },
            },

        "decoder_semantics_observed_in_root_cause_audit":
            {
                "runtime_module":
                    "dasonet.model.decoding",

                "decoder":
                    "decode_instances",

                "decoder_function_source_sha256":
                    EXPECTED_DECODER_FUNCTION_SOURCE_SHA256,

                "decoder_clamps_boxes_to_padded_input_bounds":
                    True,

                "decoder_explicitly_filters_nonpositive_area_boxes_before_return":
                    False,

                "decoder_is_modified_by_recovery":
                    False,
            },

        "recovery_rule":
            recovery_rule,

        "recovery_diagnostics":
            recovery_diagnostics,

        "recovery_execution":
            recovery_execution,

        "seed3_consistency_gate":
            seed3_consistency_gate,

        "scientific_sets":
            {
                "primary":
                    {
                        "name":
                            "official authors' RSC test-250",

                        "images":
                            250,

                        "gt_boxes":
                            539,

                        "zero_gt_images":
                            4,

                        "role":
                            "PRIMARY",
                    },

                "sensitivity":
                    {
                        "name":
                            "exact-dedup-247",

                        "images":
                            247,

                        "gt_boxes":
                            539,

                        "zero_gt_images":
                            1,

                        "role":
                            "SENSITIVITY ONLY",
                    },
            },

        "models":
            EXPECTED_MODELS,

        "decoder_settings":
            {
                "decode_min_score":
                    0.05,

                "topk":
                    100,

                "nms_iou":
                    0.50,

                "mask_threshold":
                    0.50,

                "local_peak_kernel":
                    3,
            },

        "operating_thresholds":
            {
                "seed_1":
                    0.30,

                "seed_2":
                    0.35,

                "seed_3":
                    0.35,
            },

        "formal_metrics":
            {
                "primary":
                    [
                        "Box AP50",
                        "Box AP75",
                        "Box AP50:95",
                    ],

                "operating_matching_iou":
                    0.50,

                "definitions_changed":
                    False,
            },

        "prohibitions":
            {
                "rsc_training":
                    False,

                "rsc_crpa":
                    False,

                "rsc_calibration":
                    False,

                "rsc_threshold_tuning":
                    False,

                "checkpoint_selection":
                    False,

                "seed_selection":
                    False,

                "best_seed_selection":
                    False,

                "sample_removal":
                    False,

                "primary_set_change":
                    False,

                "sensitivity_set_change":
                    False,

                "decoder_change":
                    False,

                "model_change":
                    False,
            },

        "future_recovery_evaluator_freeze_directory":
            str(
                FUTURE_RECOVERY_EVALUATOR_DIR
            ),

        "checks":
            checks,

        "readiness":
            (
                "READY_TO_IMPLEMENT_AND_FREEZE_"
                "RSC_RECOVERY_EVALUATOR"
            ),
    }


    # ========================================================
    # 19. WRITE + HASH
    # ========================================================

    write_json(
        RECOVERY_POLICY_JSON,
        policy,
    )

    policy_sha = sha256_file(
        RECOVERY_POLICY_JSON
    )

    RECOVERY_POLICY_SHA_RECORD.write_text(
        (
            f"{policy_sha}  "
            f"{RECOVERY_POLICY_JSON.name}\n"
        ),
        encoding="utf-8",
    )


    # ========================================================
    # 20. READ-BACK VALIDATION
    # ========================================================

    readback = read_json(
        RECOVERY_POLICY_JSON
    )

    require(
        readback.get("status")
        == "FROZEN_BEFORE_RECOVERY_INFERENCE",
        "Recovery-policy read-back status failure.",
    )

    require(
        readback.get("readiness")
        == (
            "READY_TO_IMPLEMENT_AND_FREEZE_"
            "RSC_RECOVERY_EVALUATOR"
        ),
        "Recovery-policy readiness failure.",
    )

    require(
        readback[
            "recovery_rule"
        ][
            "new_recovery_action"
        ]
        == "discard_prediction",
        "Recovery action changed.",
    )

    require(
        readback[
            "recovery_execution"
        ][
            "rerun_seed1_from_image_1"
        ]
        is True,
        "Seed-1 recovery is not frozen from image 1.",
    )

    require(
        readback[
            "recovery_execution"
        ][
            "rerun_seed2_from_image_1"
        ]
        is True,
        "Seed-2 recovery is not frozen from image 1.",
    )

    require(
        readback[
            "recovery_execution"
        ][
            "rerun_seed3_from_image_1"
        ]
        is True,
        "Seed-3 recovery is not frozen from image 1.",
    )

    require(
        readback[
            "recovery_execution"
        ][
            "all_three_seeds_required"
        ]
        is True,
        "Recovery no longer requires all three seeds.",
    )

    require(
        readback[
            "seed3_consistency_gate"
        ][
            "recovery_seed3_required_preexternalclip_degenerate_discard_count"
        ]
        == 0,
        "Seed-3 zero-discard consistency gate changed.",
    )

    for key in [
        "rsc_training",
        "rsc_crpa",
        "rsc_calibration",
        "rsc_threshold_tuning",
        "checkpoint_selection",
        "seed_selection",
        "best_seed_selection",
        "sample_removal",
        "primary_set_change",
        "sensitivity_set_change",
        "decoder_change",
        "model_change",
    ]:
        require(
            readback[
                "prohibitions"
            ][
                key
            ]
            is False,
            f"Recovery-policy prohibition changed: {key}",
        )

    require(
        sha256_file(
            RECOVERY_POLICY_JSON
        )
        == policy_sha,
        "Recovery-policy JSON changed during read-back.",
    )


    # ========================================================
    # 21. FINAL REPORT
    # ========================================================

    print()

    print("=" * 78)

    print(
        "DASO-NET RSC DEGENERATE-BOX "
        "RECOVERY POLICY: FROZEN"
    )

    print("=" * 78)

    print()

    print(
        "Status:"
    )

    print(
        "  FROZEN_BEFORE_RECOVERY_INFERENCE"
    )

    print()

    print(
        "Frozen recovery condition:"
    )

    print(
        "  finite decoded box AND "
        "(x2 <= x1 OR y2 <= y1)"
    )

    print()

    print(
        "Frozen recovery action:"
    )

    print(
        "  DISCARD that invalid prediction"
    )

    print()

    print(
        "Non-finite box/score behavior:"
    )

    print(
        "  remains FATAL"
    )

    print()

    print(
        "Validated decoder modified: NO"
    )

    print(
        "Model/checkpoints modified:  NO"
    )

    print(
        "NMS/top-k modified:           NO"
    )

    print(
        "Score floor modified:         NO"
    )

    print(
        "Operating thresholds changed: NO"
    )

    print(
        "Metric definitions changed:   NO"
    )

    print(
        "Dataset membership changed:   NO"
    )

    print(
        "Calibration performed:         NO"
    )

    print(
        "Seed/checkpoint selection:     NO"
    )

    print()

    print(
        "Recovery execution:"
    )

    print(
        "  rerun Seed-1 from image 1"
    )

    print(
        "  rerun Seed-2 from image 1"
    )

    print(
        "  rerun Seed-3 from image 1"
    )

    print(
        "  use NEW recovery output root"
    )

    print(
        "  preserve ALL original outputs"
    )

    print(
        "  aggregate recovery-run Seeds 1+2+3 only"
    )

    print()

    print(
        "Seed-3 recovery consistency gate:"
    )

    print(
        "  pre-external-clip degenerate discards must be 0"
    )

    print(
        "  prediction JSONL must match original Seed-3 SHA"
    )

    print(
        "  primary/dedup per-image CSVs must match"
    )

    print(
        "  scientific metric values must match exactly"
    )

    print()

    print(
        "Original Seed-3 prediction SHA256:"
    )

    print(
        f"  {seed3_predictions_sha}"
    )

    print()

    print(
        "Original Seed-3 primary per-image CSV SHA256:"
    )

    print(
        f"  {seed3_primary_csv_sha}"
    )

    print()

    print(
        "Original Seed-3 dedup per-image CSV SHA256:"
    )

    print(
        f"  {seed3_dedup_csv_sha}"
    )

    print()

    print(
        "Recovery policy JSON:"
    )

    print(
        f"  {RECOVERY_POLICY_JSON}"
    )

    print()

    print(
        "Recovery policy SHA256:"
    )

    print(
        f"  {policy_sha}"
    )

    print()

    print(
        "NEXT READINESS:"
    )

    print(
        "  READY_TO_IMPLEMENT_AND_FREEZE_"
        "RSC_RECOVERY_EVALUATOR"
    )

    print("=" * 78)


except Exception as exc:

    failure = {
        "policy_name":
            (
                "DASO-Net RSC Degenerate-Box "
                "Recovery Policy v1"
            ),

        "status":
            "FAIL",

        "inference_performed_by_this_cell":
            False,

        "scientific_metrics_computed_by_this_cell":
            False,

        "threshold_tuning":
            False,

        "calibration":
            False,

        "checkpoint_selection":
            False,

        "seed_selection":
            False,

        "error_type":
            type(exc).__name__,

        "error":
            str(exc),

        "traceback":
            traceback.format_exc(),

        "checks":
            checks,

        "readiness":
            "NOT_READY_TO_IMPLEMENT_RECOVERY_EVALUATOR",
    }


    write_json(
        RECOVERY_POLICY_JSON,
        failure,
    )

    failure_sha = sha256_file(
        RECOVERY_POLICY_JSON
    )

    RECOVERY_POLICY_SHA_RECORD.write_text(
        (
            f"{failure_sha}  "
            f"{RECOVERY_POLICY_JSON.name}\n"
        ),
        encoding="utf-8",
    )

    print()

    print("=" * 78)

    print(
        "RSC DEGENERATE-BOX RECOVERY POLICY FREEZE: FAIL"
    )

    print("=" * 78)

    print()

    print(
        "Failure preserved:"
    )

    print(
        f"  {RECOVERY_POLICY_JSON}"
    )

    print()

    print(
        "No model inference or scientific metric "
        "computation was performed."
    )

    print()

    raise

DASO-Net — RSC DEGENERATE-BOX RECOVERY POLICY FREEZE v1

THIS CELL PERFORMS NO MODEL INFERENCE.
THIS CELL COMPUTES NO SCIENTIFIC PERFORMANCE METRICS.

Purpose:
  freeze one narrow invalid-geometry sanitation rule
  before any RSC recovery inference is performed

PASS  Root-cause audit Resume02 identity
PASS  Original frozen evaluator preserved
PASS  Frozen scientific protocol preserved
PASS  Frozen original-coordinate policy preserved
PASS  Validated decoder settings preserved
PASS  Final Seed-1 identity
PASS  Final Seed-2 identity
PASS  Final Seed-3 identity
PASS  Original failed formal execution preserved
PASS  Original Seed-1 failure preserved
PASS  Original Seed-2 failure preserved
PASS  Original Seed-3 success preserved
PASS  Original Seed-3 consistency references frozen

DASO-NET RSC DEGENERATE-BOX RECOVERY POLICY: FROZEN

Status:
  FROZEN_BEFORE_RECOVERY_INFERENCE

Frozen recovery condition:
  finite decoded box AND (x2 <= x1 OR y2 <= y1)

Frozen recovery action:
  DISCARD that 

In [39]:
%%writefile /kaggle/working/evaluate_rsc_external_recovery_v1.py
# ============================================================
# DASO-Net — RSC EXTERNAL DETECTION RECOVERY EVALUATOR v1
#
# PURPOSE
# -------
# Provenance-preserving recovery wrapper around the original
# frozen RSC evaluator after the formal run exposed a narrow
# invalid-geometry corner case.
#
# ORIGINAL SCIENTIFIC EVALUATOR:
#   evaluate_rsc_external_v1_FROZEN.py
#
# RECOVERY RULE:
#   For a finite decoded [x1,y1,x2,y2] box:
#
#       if x2 <= x1 OR y2 <= y1:
#           discard that prediction
#
#   This sanitation occurs BEFORE the already-frozen
#   original-image center filter and clipping policy.
#
# UNCHANGED
# ---------
# model
# checkpoints
# seeds
# decoder
# decoder score floor
# top-k
# NMS
# mask threshold
# local-peak rule
# preprocessing
# GT
# metrics
# operating thresholds
# PRIMARY/SENSITIVITY membership
#
# The original frozen evaluator is imported byte-for-byte and
# reused for all model loading, inference, matching, AP, operating
# metrics, output tables, and summary generation.
#
# The only scientific-output behavior changed by this recovery
# wrapper is sanitation of finite non-positive-area decoded boxes.
#
# Additional recovery diagnostics are recorded per image and per
# seed but do not influence predictions, thresholds, or metrics.
# ============================================================

from __future__ import annotations

import copy
import csv
import hashlib
import importlib.util
import json
import math
import sys
import traceback
from pathlib import Path

import numpy as np


# ============================================================
# 1. FROZEN PROVENANCE
# ============================================================

ORIGINAL_FROZEN_EVALUATOR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_implementation_freeze_v1/"
    "evaluate_rsc_external_v1_FROZEN.py"
)

EXPECTED_ORIGINAL_EVALUATOR_SHA256 = (
    "59a3de31f2a5244ba32c53b705b050e0"
    "33b5d17c8de7efb36c17dbc49f0a7723"
)

FROZEN_PROTOCOL = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_implementation_freeze_v1/"
    "RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "e3c87caebafe87c7671b5745a480c0ed"
    "f67f444feac4278633da6addae4d1f7e"
)

RECOVERY_POLICY = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_degenerate_box_recovery_policy_freeze_v1/"
    "RSC_DEGENERATE_BOX_RECOVERY_POLICY_FREEZE_v1.json"
)

EXPECTED_RECOVERY_POLICY_SHA256 = (
    "aa754df77f897c68421a065ffa0b0ffc"
    "fe6b332b8dba96b6415d089cb2216f8b"
)

ORIGINAL_FORMAL_OUTPUT_ROOT = Path(
    "/kaggle/working/dasonet_final_evaluation/"
    "rsc_external_detection_v1"
)

RECOVERY_OUTPUT_ROOT = Path(
    "/kaggle/working/dasonet_final_evaluation/"
    "rsc_external_detection_v1_recovery01"
)


# ============================================================
# 2. GLOBAL RECOVERY DIAGNOSTICS
#
# One entry is appended for each call to the external coordinate
# policy. During formal evaluation this corresponds to one image.
# ============================================================

RECOVERY_CALL_DIAGNOSTICS = []


# ============================================================
# 3. GENERIC HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def require(
    condition: bool,
    message: str,
) -> None:
    if not condition:
        raise RuntimeError(message)


def require_file(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_file(),
        f"{label} missing:\n{path}",
    )


def read_json(path: Path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path: Path,
    obj,
) -> None:
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def write_csv(
    path: Path,
    rows,
) -> None:
    require(
        len(rows) > 0,
        "Cannot write empty CSV.",
    )

    fieldnames = list(
        rows[0].keys()
    )

    with path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        for row in rows:
            writer.writerow(row)


def cli_value(
    flag: str,
    default=None,
):
    for index, token in enumerate(
        sys.argv
    ):
        if token == flag:
            require(
                index + 1 < len(sys.argv),
                f"Missing value after {flag}.",
            )

            return sys.argv[
                index + 1
            ]

    return default


# ============================================================
# 4. RECOVERY-POLICY VALIDATION
# ============================================================

def load_and_validate_recovery_policy():
    require_file(
        RECOVERY_POLICY,
        "Frozen recovery policy",
    )

    actual_sha = sha256_file(
        RECOVERY_POLICY
    )

    require(
        actual_sha
        == EXPECTED_RECOVERY_POLICY_SHA256,
        "Frozen recovery-policy SHA mismatch.\n"
        f"Expected: {EXPECTED_RECOVERY_POLICY_SHA256}\n"
        f"Actual:   {actual_sha}",
    )

    policy = read_json(
        RECOVERY_POLICY
    )

    require(
        policy.get("status")
        == "FROZEN_BEFORE_RECOVERY_INFERENCE",
        "Recovery policy is not frozen before "
        "recovery inference.",
    )

    rule = policy[
        "recovery_rule"
    ]

    require(
        rule[
            "new_recovery_action"
        ]
        == "discard_prediction",
        "Frozen recovery action changed.",
    )

    require(
        rule[
            "fatal_conditions_unchanged"
        ][
            "box_coordinate_count_not_four"
        ]
        is True,
        "Four-coordinate structural failure no longer fatal.",
    )

    require(
        rule[
            "fatal_conditions_unchanged"
        ][
            "nonfinite_box_coordinate"
        ]
        is True,
        "Non-finite box-coordinate failure no longer fatal.",
    )

    require(
        rule[
            "fatal_conditions_unchanged"
        ][
            "nonfinite_score"
        ]
        is True,
        "Non-finite score failure no longer fatal.",
    )

    execution = policy[
        "recovery_execution"
    ]

    require(
        execution[
            "rerun_seed1_from_image_1"
        ]
        is True,
        "Recovery Seed-1 is not frozen from image 1.",
    )

    require(
        execution[
            "rerun_seed2_from_image_1"
        ]
        is True,
        "Recovery Seed-2 is not frozen from image 1.",
    )

    require(
        execution[
            "rerun_seed3_from_image_1"
        ]
        is True,
        "Recovery Seed-3 is not frozen from image 1.",
    )

    require(
        execution[
            "all_three_seeds_required"
        ]
        is True,
        "Recovery policy no longer requires all 3 seeds.",
    )

    require(
        Path(
            execution[
                "new_output_root"
            ]
        )
        == RECOVERY_OUTPUT_ROOT,
        "Recovery output-root contract changed.",
    )

    return policy, actual_sha


# ============================================================
# 5. LOAD ORIGINAL FROZEN EVALUATOR
# ============================================================

def load_original_frozen_evaluator():
    require_file(
        ORIGINAL_FROZEN_EVALUATOR,
        "Original frozen RSC evaluator",
    )

    actual_sha = sha256_file(
        ORIGINAL_FROZEN_EVALUATOR
    )

    require(
        actual_sha
        == EXPECTED_ORIGINAL_EVALUATOR_SHA256,
        "Original frozen evaluator SHA mismatch.\n"
        f"Expected: {EXPECTED_ORIGINAL_EVALUATOR_SHA256}\n"
        f"Actual:   {actual_sha}",
    )

    module_name = (
        "dasonet_rsc_original_frozen_evaluator_"
        "for_recovery_v1"
    )

    spec = (
        importlib.util
        .spec_from_file_location(
            module_name,
            ORIGINAL_FROZEN_EVALUATOR,
        )
    )

    require(
        spec is not None
        and spec.loader is not None,
        "Could not create import specification for "
        "original frozen RSC evaluator.",
    )

    module = (
        importlib.util
        .module_from_spec(
            spec
        )
    )

    sys.modules[
        module_name
    ] = module

    spec.loader.exec_module(
        module
    )

    return module


RECOVERY_POLICY_OBJECT, RECOVERY_POLICY_SHA256 = (
    load_and_validate_recovery_policy()
)

BASE = load_original_frozen_evaluator()

ORIGINAL_BASE_LOAD_PROTOCOL = (
    BASE.load_protocol
)


# ============================================================
# 6. FROZEN PROTOCOL REDIRECTION
#
# The scientific protocol bytes remain unchanged.
#
# The original evaluator is instructed to read the exact frozen
# protocol copy. The only in-memory infrastructure substitution is
# designated_future_output_root -> recovery01.
#
# Returned protocol_sha remains the exact original scientific
# protocol SHA.
# ============================================================

BASE.DEFAULT_PROTOCOL = FROZEN_PROTOCOL


def recovery_load_protocol(
    path: Path,
):
    requested = Path(
        path
    ).resolve()

    frozen = FROZEN_PROTOCOL.resolve()

    require(
        requested == frozen,
        "Recovery evaluator accepts only the exact frozen "
        "scientific protocol copy.\n"
        f"Required: {frozen}\n"
        f"Received: {requested}",
    )

    protocol, protocol_sha = (
        ORIGINAL_BASE_LOAD_PROTOCOL(
            FROZEN_PROTOCOL
        )
    )

    require(
        protocol_sha
        == EXPECTED_PROTOCOL_SHA256,
        "Scientific protocol SHA changed during recovery load.",
    )

    require(
        Path(
            protocol[
                "designated_future_output_root"
            ]
        )
        == ORIGINAL_FORMAL_OUTPUT_ROOT,
        "Original scientific protocol output-root provenance "
        "does not match the frozen formal protocol.",
    )

    recovery_protocol = copy.deepcopy(
        protocol
    )

    recovery_protocol[
        "designated_future_output_root"
    ] = str(
        RECOVERY_OUTPUT_ROOT
    )

    return (
        recovery_protocol,
        protocol_sha,
    )


BASE.load_protocol = (
    recovery_load_protocol
)


# ============================================================
# 7. RECOVERY ORIGINAL-IMAGE BOX POLICY
#
# This reproduces the original frozen function exactly except for
# ONE narrow change:
#
# ORIGINAL:
#
#   require(
#       x2 > x1 and y2 > y1,
#       "Decoded box is degenerate ..."
#   )
#
# RECOVERY:
#
#   if x2 <= x1 or y2 <= y1:
#       discard
#
# Non-finite values remain fatal.
# All subsequent center filtering, clipping, post-clipping
# degeneracy handling, sorting, and retained output structure are
# unchanged.
# ============================================================

def recovery_apply_original_image_box_policy(
    predictions,
    original_w: int,
    original_h: int,
):
    retained = []

    decoded_before = len(
        predictions
    )

    preexternalclip_degenerate_discarded = 0

    center_outside_discarded = 0

    postclip_degenerate_discarded = 0

    for item in predictions:
        box = np.asarray(
            item[
                "box"
            ],
            dtype=np.float64,
        ).reshape(-1)

        require(
            len(box) == 4,
            "Decoded prediction box length != 4.",
        )

        score = float(
            item[
                "score"
            ]
        )

        require(
            np.all(
                np.isfinite(box)
            ),
            "Non-finite decoded box.",
        )

        require(
            math.isfinite(score),
            "Non-finite decoded score.",
        )

        x1, y1, x2, y2 = [
            float(value)
            for value in box
        ]

        # ----------------------------------------------------
        # RECOVERY v1 — the only changed scientific-output rule.
        # ----------------------------------------------------

        if (
            x2 <= x1
            or y2 <= y1
        ):
            preexternalclip_degenerate_discarded += 1

            continue

        # ----------------------------------------------------
        # Everything below reproduces the original frozen rule.
        # ----------------------------------------------------

        cx = (
            x1 + x2
        ) / 2.0

        cy = (
            y1 + y2
        ) / 2.0

        if not (
            0.0 <= cx < original_w
            and 0.0 <= cy < original_h
        ):
            center_outside_discarded += 1

            continue

        x1 = max(
            0.0,
            min(
                float(original_w),
                x1,
            ),
        )

        y1 = max(
            0.0,
            min(
                float(original_h),
                y1,
            ),
        )

        x2 = max(
            0.0,
            min(
                float(original_w),
                x2,
            ),
        )

        y2 = max(
            0.0,
            min(
                float(original_h),
                y2,
            ),
        )

        if (
            x2 <= x1
            or y2 <= y1
        ):
            postclip_degenerate_discarded += 1

            continue

        retained.append(
            {
                "box":
                    [
                        x1,
                        y1,
                        x2,
                        y2,
                    ],

                "score":
                    score,

                "decoder_order":
                    int(
                        item[
                            "decoder_order"
                        ]
                    ),
            }
        )

    retained.sort(
        key=lambda item: (
            -item[
                "score"
            ],
            item[
                "decoder_order"
            ],
        )
    )

    after_geometry_sanitation = (
        decoded_before
        - preexternalclip_degenerate_discarded
    )

    RECOVERY_CALL_DIAGNOSTICS.append(
        {
            "decoded_predictions_before_geometry_sanitation":
                int(
                    decoded_before
                ),

            "preexternalclip_degenerate_predictions_discarded":
                int(
                    preexternalclip_degenerate_discarded
                ),

            "predictions_after_geometry_sanitation":
                int(
                    after_geometry_sanitation
                ),

            "center_outside_original_image_discarded":
                int(
                    center_outside_discarded
                ),

            "postclip_degenerate_predictions_discarded":
                int(
                    postclip_degenerate_discarded
                ),

            "retained_predictions":
                int(
                    len(
                        retained
                    )
                ),
        }
    )

    return retained


BASE.apply_original_image_box_policy = (
    recovery_apply_original_image_box_policy
)


# ============================================================
# 8. EVALUATOR IDENTITY REBINDING
#
# The imported original evaluator's formal-run code computes its
# evaluator SHA using its module-global __file__.
#
# Rebind that identity to THIS recovery wrapper so formal recovery
# run states and summaries record the recovery evaluator SHA, not
# the superseded original evaluator SHA.
# ============================================================

BASE.__file__ = str(
    Path(
        __file__
    ).resolve()
)


# ============================================================
# 9. SYNTHETIC RECOVERY-GEOMETRY SELF-TEST
#
# NO RSC DATA.
# NO MODEL.
# NO CHECKPOINT.
# ============================================================

def run_recovery_geometry_self_test():
    print("=" * 72)

    print(
        "DASO-Net — RSC RECOVERY GEOMETRY "
        "SYNTHETIC SELF-TEST"
    )

    print("=" * 72)

    RECOVERY_CALL_DIAGNOSTICS.clear()

    # --------------------------------------------------------
    # A. Valid box retained unchanged.
    # --------------------------------------------------------

    valid = [
        {
            "box":
                [
                    10.0,
                    20.0,
                    30.0,
                    40.0,
                ],

            "score":
                0.90,

            "decoder_order":
                0,
        }
    ]

    out = (
        recovery_apply_original_image_box_policy(
            valid,
            original_w=100,
            original_h=100,
        )
    )

    require(
        len(out) == 1,
        "Valid box was not retained.",
    )

    require(
        out[0]["box"]
        == [
            10.0,
            20.0,
            30.0,
            40.0,
        ],
        "Valid box coordinates changed.",
    )

    print(
        "PASS  Positive-area box retained"
    )


    # --------------------------------------------------------
    # B. Zero-width finite box discarded.
    # --------------------------------------------------------

    out = (
        recovery_apply_original_image_box_policy(
            [
                {
                    "box":
                        [
                            10.0,
                            20.0,
                            10.0,
                            40.0,
                        ],

                    "score":
                        0.80,

                    "decoder_order":
                        0,
                }
            ],
            original_w=100,
            original_h=100,
        )
    )

    require(
        out == [],
        "Zero-width box was not discarded.",
    )

    require(
        RECOVERY_CALL_DIAGNOSTICS[
            -1
        ][
            "preexternalclip_degenerate_predictions_discarded"
        ]
        == 1,
        "Zero-width diagnostic counter incorrect.",
    )

    print(
        "PASS  Zero-width finite box discarded"
    )


    # --------------------------------------------------------
    # C. Negative-width finite box discarded.
    # --------------------------------------------------------

    out = (
        recovery_apply_original_image_box_policy(
            [
                {
                    "box":
                        [
                            30.0,
                            20.0,
                            10.0,
                            40.0,
                        ],

                    "score":
                        0.80,

                    "decoder_order":
                        0,
                }
            ],
            original_w=100,
            original_h=100,
        )
    )

    require(
        out == [],
        "Negative-width box was not discarded.",
    )

    print(
        "PASS  Negative-width finite box discarded"
    )


    # --------------------------------------------------------
    # D. Zero-height finite box discarded.
    # --------------------------------------------------------

    out = (
        recovery_apply_original_image_box_policy(
            [
                {
                    "box":
                        [
                            10.0,
                            20.0,
                            30.0,
                            20.0,
                        ],

                    "score":
                        0.80,

                    "decoder_order":
                        0,
                }
            ],
            original_w=100,
            original_h=100,
        )
    )

    require(
        out == [],
        "Zero-height box was not discarded.",
    )

    print(
        "PASS  Zero-height finite box discarded"
    )


    # --------------------------------------------------------
    # E. Negative-height finite box discarded.
    # --------------------------------------------------------

    out = (
        recovery_apply_original_image_box_policy(
            [
                {
                    "box":
                        [
                            10.0,
                            40.0,
                            30.0,
                            20.0,
                        ],

                    "score":
                        0.80,

                    "decoder_order":
                        0,
                }
            ],
            original_w=100,
            original_h=100,
        )
    )

    require(
        out == [],
        "Negative-height box was not discarded.",
    )

    print(
        "PASS  Negative-height finite box discarded"
    )


    # --------------------------------------------------------
    # F. Center-outside rule remains unchanged.
    # --------------------------------------------------------

    out = (
        recovery_apply_original_image_box_policy(
            [
                {
                    "box":
                        [
                            101.0,
                            20.0,
                            120.0,
                            40.0,
                        ],

                    "score":
                        0.80,

                    "decoder_order":
                        0,
                }
            ],
            original_w=100,
            original_h=100,
        )
    )

    require(
        out == [],
        "Center-outside prediction was not discarded.",
    )

    require(
        RECOVERY_CALL_DIAGNOSTICS[
            -1
        ][
            "center_outside_original_image_discarded"
        ]
        == 1,
        "Center-outside diagnostic counter incorrect.",
    )

    print(
        "PASS  Original center-outside rule unchanged"
    )


    # --------------------------------------------------------
    # G. Original-image clipping remains unchanged.
    # --------------------------------------------------------

    out = (
        recovery_apply_original_image_box_policy(
            [
                {
                    "box":
                        [
                            -5.0,
                            10.0,
                            20.0,
                            30.0,
                        ],

                    "score":
                        0.80,

                    "decoder_order":
                        0,
                }
            ],
            original_w=100,
            original_h=100,
        )
    )

    require(
        len(out) == 1,
        "Partially clipped valid box was discarded.",
    )

    require(
        out[0]["box"]
        == [
            0.0,
            10.0,
            20.0,
            30.0,
        ],
        "Original clipping behavior changed.",
    )

    print(
        "PASS  Original clipping rule unchanged"
    )


    # --------------------------------------------------------
    # H. Non-finite coordinate remains fatal.
    # --------------------------------------------------------

    nonfinite_box_failed = False

    try:
        recovery_apply_original_image_box_policy(
            [
                {
                    "box":
                        [
                            float("nan"),
                            10.0,
                            20.0,
                            30.0,
                        ],

                    "score":
                        0.80,

                    "decoder_order":
                        0,
                }
            ],
            original_w=100,
            original_h=100,
        )

    except RuntimeError as exc:
        require(
            str(exc)
            == "Non-finite decoded box.",
            "Unexpected non-finite box error.",
        )

        nonfinite_box_failed = True

    require(
        nonfinite_box_failed,
        "Non-finite box did not remain fatal.",
    )

    print(
        "PASS  Non-finite box remains fatal"
    )


    # --------------------------------------------------------
    # I. Non-finite score remains fatal.
    # --------------------------------------------------------

    nonfinite_score_failed = False

    try:
        recovery_apply_original_image_box_policy(
            [
                {
                    "box":
                        [
                            10.0,
                            10.0,
                            20.0,
                            30.0,
                        ],

                    "score":
                        float("nan"),

                    "decoder_order":
                        0,
                }
            ],
            original_w=100,
            original_h=100,
        )

    except RuntimeError as exc:
        require(
            str(exc)
            == "Non-finite decoded score.",
            "Unexpected non-finite score error.",
        )

        nonfinite_score_failed = True

    require(
        nonfinite_score_failed,
        "Non-finite score did not remain fatal.",
    )

    print(
        "PASS  Non-finite score remains fatal"
    )


    # --------------------------------------------------------
    # J. Four-coordinate structural contract remains fatal.
    # --------------------------------------------------------

    length_failed = False

    try:
        recovery_apply_original_image_box_policy(
            [
                {
                    "box":
                        [
                            10.0,
                            20.0,
                            30.0,
                        ],

                    "score":
                        0.80,

                    "decoder_order":
                        0,
                }
            ],
            original_w=100,
            original_h=100,
        )

    except RuntimeError as exc:
        require(
            str(exc)
            == (
                "Decoded prediction box length != 4."
            ),
            "Unexpected box-length error.",
        )

        length_failed = True

    require(
        length_failed,
        "Invalid coordinate count did not remain fatal.",
    )

    print(
        "PASS  Four-coordinate contract remains fatal"
    )


    RECOVERY_CALL_DIAGNOSTICS.clear()

    print()

    print(
        "RECOVERY GEOMETRY SELF-TEST: PASS"
    )


# ============================================================
# 10. RECOVERY DIAGNOSTIC OUTPUT
# ============================================================

def write_recovery_diagnostics(
    seed_index: int,
    run_status: str,
    run_error: str | None,
):
    protocol = read_json(
        FROZEN_PROTOCOL
    )

    models = [
        model
        for model
        in protocol[
            "models"
        ]
        if int(
            model[
                "seed_index"
            ]
        )
        == int(
            seed_index
        )
    ]

    require(
        len(models) == 1,
        "Could not uniquely resolve recovery seed model.",
    )

    model_info = models[0]

    seed = int(
        model_info[
            "seed"
        ]
    )

    seed_dir = (
        RECOVERY_OUTPUT_ROOT
        / (
            f"seed{seed_index:02d}_"
            f"s{seed}"
        )
    )

    require(
        seed_dir.is_dir(),
        "Recovery seed output directory does not exist:\n"
        f"{seed_dir}",
    )

    primary_manifest = Path(
        protocol[
            "scientific_sets"
        ][
            "primary"
        ][
            "manifest"
        ]
    )

    require_file(
        primary_manifest,
        "Frozen primary manifest",
    )

    with primary_manifest.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:
        manifest_rows = list(
            csv.DictReader(f)
        )

    manifest_rows = sorted(
        manifest_rows,
        key=lambda row: int(
            float(
                row[
                    "official_test_index"
                ]
            )
        ),
    )

    call_count = len(
        RECOVERY_CALL_DIAGNOSTICS
    )

    require(
        call_count <= len(
            manifest_rows
        ),
        "Recovery diagnostic call count exceeds "
        "primary manifest length.",
    )

    if run_status == "PASS":
        require(
            call_count == 250,
            "Successful formal recovery run did not produce "
            "exactly 250 coordinate-policy calls.",
        )

    per_image = []

    for index, diagnostic in enumerate(
        RECOVERY_CALL_DIAGNOSTICS
    ):
        manifest_row = manifest_rows[
            index
        ]

        official_index = int(
            float(
                manifest_row[
                    "official_test_index"
                ]
            )
        )

        require(
            official_index
            == index + 1,
            "Recovery diagnostic/manifest ordering mismatch.",
        )

        per_image.append(
            {
                "official_test_index":
                    official_index,

                "filename":
                    str(
                        manifest_row[
                            "filename"
                        ]
                    ).strip(),

                **diagnostic,
            }
        )

    totals = {
        "decoded_predictions_before_geometry_sanitation":
            int(
                sum(
                    row[
                        "decoded_predictions_before_geometry_sanitation"
                    ]
                    for row in per_image
                )
            ),

        "preexternalclip_degenerate_predictions_discarded":
            int(
                sum(
                    row[
                        "preexternalclip_degenerate_predictions_discarded"
                    ]
                    for row in per_image
                )
            ),

        "predictions_after_geometry_sanitation":
            int(
                sum(
                    row[
                        "predictions_after_geometry_sanitation"
                    ]
                    for row in per_image
                )
            ),

        "center_outside_original_image_discarded":
            int(
                sum(
                    row[
                        "center_outside_original_image_discarded"
                    ]
                    for row in per_image
                )
            ),

        "postclip_degenerate_predictions_discarded":
            int(
                sum(
                    row[
                        "postclip_degenerate_predictions_discarded"
                    ]
                    for row in per_image
                )
            ),

        "retained_predictions":
            int(
                sum(
                    row[
                        "retained_predictions"
                    ]
                    for row in per_image
                )
            ),
    }

    diagnostics_json = (
        seed_dir
        / "RSC_RECOVERY_GEOMETRY_DIAGNOSTICS.json"
    )

    diagnostics_csv = (
        seed_dir
        / "rsc_recovery_geometry_per_image.csv"
    )

    provenance_json = (
        seed_dir
        / "RSC_RECOVERY_PROVENANCE.json"
    )

    diagnostics = {
        "diagnostic_name":
            (
                "DASO-Net RSC Recovery Geometry "
                "Diagnostics v1"
            ),

        "status":
            run_status,

        "seed_index":
            int(
                seed_index
            ),

        "seed":
            seed,

        "images_with_coordinate_policy_call":
            call_count,

        "run_error":
            run_error,

        "recovery_policy_sha256":
            RECOVERY_POLICY_SHA256,

        "recovery_evaluator":
            str(
                Path(
                    __file__
                ).resolve()
            ),

        "recovery_evaluator_sha256":
            sha256_file(
                Path(
                    __file__
                ).resolve()
            ),

        "totals":
            totals,

        "per_image":
            per_image,

        "diagnostic_only":
            True,

        "threshold_tuning":
            False,

        "calibration":
            False,

        "checkpoint_selection":
            False,

        "seed_selection":
            False,
    }

    write_json(
        diagnostics_json,
        diagnostics,
    )

    if per_image:
        write_csv(
            diagnostics_csv,
            per_image,
        )

    provenance = {
        "provenance_name":
            (
                "DASO-Net RSC Recovery "
                "Per-Seed Provenance v1"
            ),

        "status":
            run_status,

        "seed_index":
            int(
                seed_index
            ),

        "seed":
            seed,

        "original_frozen_evaluator":
            str(
                ORIGINAL_FROZEN_EVALUATOR
            ),

        "original_frozen_evaluator_sha256":
            EXPECTED_ORIGINAL_EVALUATOR_SHA256,

        "recovery_evaluator":
            str(
                Path(
                    __file__
                ).resolve()
            ),

        "recovery_evaluator_sha256":
            sha256_file(
                Path(
                    __file__
                ).resolve()
            ),

        "frozen_scientific_protocol":
            str(
                FROZEN_PROTOCOL
            ),

        "frozen_scientific_protocol_sha256":
            EXPECTED_PROTOCOL_SHA256,

        "recovery_policy":
            str(
                RECOVERY_POLICY
            ),

        "recovery_policy_sha256":
            RECOVERY_POLICY_SHA256,

        "recovery_output_root":
            str(
                RECOVERY_OUTPUT_ROOT
            ),

        "recovery_rule":
            (
                "discard finite decoded boxes with "
                "x2 <= x1 OR y2 <= y1 before existing "
                "original-image center/clipping policy"
            ),

        "model_modified":
            False,

        "decoder_modified":
            False,

        "threshold_modified":
            False,

        "metric_definition_modified":
            False,

        "dataset_membership_modified":
            False,

        "calibration_performed":
            False,

        "checkpoint_selection":
            False,

        "seed_selection":
            False,

        "geometry_diagnostics":
            str(
                diagnostics_json
            ),

        "geometry_diagnostics_sha256":
            sha256_file(
                diagnostics_json
            ),
    }

    if diagnostics_csv.is_file():
        provenance[
            "geometry_per_image_csv"
        ] = str(
            diagnostics_csv
        )

        provenance[
            "geometry_per_image_csv_sha256"
        ] = sha256_file(
            diagnostics_csv
        )

    write_json(
        provenance_json,
        provenance,
    )

    return {
        "diagnostics_json":
            diagnostics_json,

        "diagnostics_json_sha256":
            sha256_file(
                diagnostics_json
            ),

        "diagnostics_csv":
            (
                diagnostics_csv
                if diagnostics_csv.is_file()
                else None
            ),

        "provenance_json":
            provenance_json,

        "provenance_json_sha256":
            sha256_file(
                provenance_json
            ),

        "totals":
            totals,
    }


# ============================================================
# 11. ENTRY POINT
# ============================================================

def main():
    mode = cli_value(
        "--mode"
    )

    require(
        mode is not None,
        "--mode is required.",
    )

    # --------------------------------------------------------
    # Recovery-only synthetic geometry unit test.
    # --------------------------------------------------------

    if mode == "recovery-self-test":
        run_recovery_geometry_self_test()

        return

    require(
        mode
        in {
            "self-test",
            "smoke",
            "evaluate",
        },
        "Unsupported mode for recovery evaluator:\n"
        f"{mode}",
    )

    if mode != "evaluate":
        BASE.main()

        return

    seed_index = int(
        cli_value(
            "--seed-index",
            "1",
        )
    )

    require(
        seed_index
        in {
            1,
            2,
            3,
        },
        "Recovery formal evaluation requires "
        "seed index 1, 2, or 3.",
    )

    RECOVERY_CALL_DIAGNOSTICS.clear()

    try:
        BASE.main()

    except Exception as exc:
        diagnostic_result = None

        try:
            diagnostic_result = (
                write_recovery_diagnostics(
                    seed_index=seed_index,
                    run_status="FAIL",
                    run_error=(
                        f"{type(exc).__name__}: "
                        f"{exc}"
                    ),
                )
            )

        except Exception:
            print()

            print(
                "WARNING: failed to write recovery "
                "geometry diagnostics after base-run failure."
            )

            traceback.print_exc()

        raise

    else:
        diagnostic_result = (
            write_recovery_diagnostics(
                seed_index=seed_index,
                run_status="PASS",
                run_error=None,
            )
        )

        print()

        print(
            "Recovery geometry diagnostics:"
        )

        print(
            "  "
            f"{diagnostic_result['diagnostics_json']}"
        )

        print()

        print(
            "Recovery geometry diagnostics SHA256:"
        )

        print(
            "  "
            f"{diagnostic_result['diagnostics_json_sha256']}"
        )

        print()

        print(
            "Recovery pre-external-clip degenerate "
            "predictions discarded:"
        )

        print(
            "  "
            f"{diagnostic_result['totals']['preexternalclip_degenerate_predictions_discarded']}"
        )


if __name__ == "__main__":
    main()

Writing /kaggle/working/evaluate_rsc_external_recovery_v1.py


In [40]:
# ============================================================
# DASO-Net — RSC RECOVERY EVALUATOR IMPLEMENTATION FREEZE v1
#
# THIS CELL PERFORMS NO RSC MODEL INFERENCE.
#
# It:
#   - verifies the frozen recovery policy
#   - verifies the original frozen evaluator
#   - compiles the complete recovery evaluator
#   - runs original synthetic metric self-tests
#   - runs recovery geometry synthetic self-tests
#   - freezes the exact recovery evaluator bytes
#   - reruns both synthetic tests from the frozen copy
#   - writes SHA-protected implementation provenance
#
# ============================================================

from __future__ import annotations

import hashlib
import json
import py_compile
import shutil
import subprocess
import sys
import traceback
from pathlib import Path


# ============================================================
# 1. EXPECTED INPUTS
# ============================================================

RECOVERY_POLICY = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_degenerate_box_recovery_policy_freeze_v1/"
    "RSC_DEGENERATE_BOX_RECOVERY_POLICY_FREEZE_v1.json"
)

EXPECTED_RECOVERY_POLICY_SHA256 = (
    "aa754df77f897c68421a065ffa0b0ffc"
    "fe6b332b8dba96b6415d089cb2216f8b"
)

ORIGINAL_FROZEN_EVALUATOR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_implementation_freeze_v1/"
    "evaluate_rsc_external_v1_FROZEN.py"
)

EXPECTED_ORIGINAL_EVALUATOR_SHA256 = (
    "59a3de31f2a5244ba32c53b705b050e0"
    "33b5d17c8de7efb36c17dbc49f0a7723"
)

FROZEN_PROTOCOL = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_implementation_freeze_v1/"
    "RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "e3c87caebafe87c7671b5745a480c0ed"
    "f67f444feac4278633da6addae4d1f7e"
)

DEVELOPMENT_RECOVERY_EVALUATOR = Path(
    "/kaggle/working/"
    "evaluate_rsc_external_recovery_v1.py"
)


# ============================================================
# 2. FREEZE DESTINATION
# ============================================================

FREEZE_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_recovery_evaluator_implementation_freeze_v1"
)

FROZEN_RECOVERY_EVALUATOR = (
    FREEZE_DIR
    / "evaluate_rsc_external_recovery_v1_FROZEN.py"
)

FROZEN_RECOVERY_POLICY_COPY = (
    FREEZE_DIR
    / "RSC_DEGENERATE_BOX_RECOVERY_POLICY_FREEZE_v1.json"
)

FROZEN_PROTOCOL_COPY = (
    FREEZE_DIR
    / "RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

ORIGINAL_EVALUATOR_SHA_RECORD = (
    FREEZE_DIR
    / "ORIGINAL_FROZEN_RSC_EVALUATOR.sha256.txt"
)

RECOVERY_EVALUATOR_SHA_RECORD = (
    FREEZE_DIR
    / "RSC_RECOVERY_EVALUATOR.sha256.txt"
)

ORIGINAL_SELFTEST_LOG = (
    FREEZE_DIR
    / "RSC_RECOVERY_EVALUATOR_ORIGINAL_METRIC_SELFTEST.txt"
)

RECOVERY_SELFTEST_LOG = (
    FREEZE_DIR
    / "RSC_RECOVERY_GEOMETRY_SELFTEST.txt"
)

FREEZE_JSON = (
    FREEZE_DIR
    / "RSC_RECOVERY_EVALUATOR_IMPLEMENTATION_FREEZE_v1.json"
)

FREEZE_SHA_RECORD = (
    FREEZE_DIR
    / "RSC_RECOVERY_EVALUATOR_IMPLEMENTATION_FREEZE_v1.sha256.txt"
)


# ============================================================
# 3. FUTURE RECOVERY PATHS — MUST NOT EXIST YET
# ============================================================

RECOVERY_OUTPUT_ROOT = Path(
    "/kaggle/working/dasonet_final_evaluation/"
    "rsc_external_detection_v1_recovery01"
)

RECOVERY_EXECUTION_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_formal_three_seed_recovery_execution_v1"
)


# ============================================================
# 4. HELPERS
# ============================================================

checks = []


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def require(
    condition: bool,
    message: str,
) -> None:
    if not condition:
        raise RuntimeError(message)


def require_file(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_file(),
        f"{label} missing:\n{path}",
    )


def read_json(path: Path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path: Path,
    obj,
) -> None:
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def record(
    name: str,
    **details,
) -> None:
    item = {
        "check": name,
        "status": "PASS",
    }

    item.update(details)

    checks.append(item)

    print(
        f"PASS  {name}"
    )


def run_test(
    evaluator: Path,
    mode: str,
):
    result = subprocess.run(
        [
            sys.executable,
            str(
                evaluator
            ),
            "--mode",
            mode,
        ],
        check=False,
        text=True,
        capture_output=True,
    )

    return result


# ============================================================
# 5. BANNER / NON-OVERWRITE
# ============================================================

print("=" * 78)

print(
    "DASO-Net — RSC RECOVERY EVALUATOR "
    "IMPLEMENTATION FREEZE v1"
)

print("=" * 78)

print()

print(
    "THIS CELL PERFORMS NO RSC MODEL INFERENCE."
)

print()

print(
    "Recovery policy already frozen:"
)

print(
    "  finite decoded non-positive-area box -> DISCARD"
)

print()


if FREEZE_DIR.exists():
    raise RuntimeError(
        "Recovery-evaluator freeze directory already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{FREEZE_DIR}"
    )


require(
    not RECOVERY_OUTPUT_ROOT.exists(),
    "Recovery formal-output root already exists BEFORE "
    "implementation freeze:\n"
    f"{RECOVERY_OUTPUT_ROOT}",
)

require(
    not RECOVERY_EXECUTION_DIR.exists(),
    "Recovery execution directory already exists BEFORE "
    "implementation freeze:\n"
    f"{RECOVERY_EXECUTION_DIR}",
)


# ============================================================
# 6. PRE-FREEZE VALIDATION
# ============================================================

require_file(
    RECOVERY_POLICY,
    "Frozen recovery policy",
)

policy_sha = sha256_file(
    RECOVERY_POLICY
)

require(
    policy_sha
    == EXPECTED_RECOVERY_POLICY_SHA256,
    "Recovery-policy SHA mismatch.",
)

policy = read_json(
    RECOVERY_POLICY
)

require(
    policy.get("status")
    == "FROZEN_BEFORE_RECOVERY_INFERENCE",
    "Recovery policy status changed.",
)

require(
    policy.get("readiness")
    == (
        "READY_TO_IMPLEMENT_AND_FREEZE_"
        "RSC_RECOVERY_EVALUATOR"
    ),
    "Recovery policy readiness changed.",
)

record(
    "Frozen recovery-policy identity",
    sha256=policy_sha,
)


require_file(
    ORIGINAL_FROZEN_EVALUATOR,
    "Original frozen evaluator",
)

original_evaluator_sha = sha256_file(
    ORIGINAL_FROZEN_EVALUATOR
)

require(
    original_evaluator_sha
    == EXPECTED_ORIGINAL_EVALUATOR_SHA256,
    "Original frozen evaluator changed.",
)

record(
    "Original frozen evaluator identity",
    sha256=original_evaluator_sha,
)


require_file(
    FROZEN_PROTOCOL,
    "Frozen scientific protocol",
)

protocol_sha = sha256_file(
    FROZEN_PROTOCOL
)

require(
    protocol_sha
    == EXPECTED_PROTOCOL_SHA256,
    "Frozen scientific protocol changed.",
)

record(
    "Frozen scientific protocol identity",
    sha256=protocol_sha,
)


require_file(
    DEVELOPMENT_RECOVERY_EVALUATOR,
    "Complete recovery evaluator",
)

development_sha = sha256_file(
    DEVELOPMENT_RECOVERY_EVALUATOR
)

record(
    "Recovery evaluator source present",
    sha256=development_sha,
)


# ============================================================
# 7. COMPILE
# ============================================================

py_compile.compile(
    str(
        DEVELOPMENT_RECOVERY_EVALUATOR
    ),
    doraise=True,
)

record(
    "Recovery evaluator Python compile"
)


# ============================================================
# 8. ORIGINAL METRIC SELF-TEST THROUGH RECOVERY WRAPPER
#
# NO RSC inference.
# ============================================================

original_selftest = run_test(
    DEVELOPMENT_RECOVERY_EVALUATOR,
    "self-test",
)

require(
    original_selftest.returncode
    == 0,
    "Original synthetic metric self-test failed through "
    "recovery wrapper.\n"
    f"STDOUT:\n{original_selftest.stdout}\n"
    f"STDERR:\n{original_selftest.stderr}",
)

require(
    "SELF-TEST STATUS: PASS"
    in original_selftest.stdout,
    "Original metric self-test did not emit PASS.",
)

record(
    "Original frozen metric-engine self-test"
)


# ============================================================
# 9. RECOVERY-GEOMETRY SELF-TEST
#
# Artificial boxes only.
# NO RSC inference.
# ============================================================

recovery_selftest = run_test(
    DEVELOPMENT_RECOVERY_EVALUATOR,
    "recovery-self-test",
)

require(
    recovery_selftest.returncode
    == 0,
    "Recovery geometry self-test failed.\n"
    f"STDOUT:\n{recovery_selftest.stdout}\n"
    f"STDERR:\n{recovery_selftest.stderr}",
)

require(
    "RECOVERY GEOMETRY SELF-TEST: PASS"
    in recovery_selftest.stdout,
    "Recovery geometry self-test did not emit PASS.",
)

record(
    "Recovery geometry synthetic self-test"
)


# ============================================================
# 10. CREATE FREEZE DIRECTORY
# ============================================================

FREEZE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 11. COPY EXACT IMPLEMENTATION + PROTOCOL/POLICY
# ============================================================

shutil.copyfile(
    DEVELOPMENT_RECOVERY_EVALUATOR,
    FROZEN_RECOVERY_EVALUATOR,
)

shutil.copyfile(
    RECOVERY_POLICY,
    FROZEN_RECOVERY_POLICY_COPY,
)

shutil.copyfile(
    FROZEN_PROTOCOL,
    FROZEN_PROTOCOL_COPY,
)


frozen_recovery_sha = sha256_file(
    FROZEN_RECOVERY_EVALUATOR
)

require(
    frozen_recovery_sha
    == development_sha,
    "Frozen recovery evaluator is not byte-identical "
    "to tested implementation.",
)

require(
    sha256_file(
        FROZEN_RECOVERY_POLICY_COPY
    )
    == EXPECTED_RECOVERY_POLICY_SHA256,
    "Frozen recovery-policy copy changed.",
)

require(
    sha256_file(
        FROZEN_PROTOCOL_COPY
    )
    == EXPECTED_PROTOCOL_SHA256,
    "Frozen protocol copy changed.",
)

record(
    "Frozen recovery evaluator byte identity",
    sha256=frozen_recovery_sha,
)

record(
    "Frozen recovery-policy copy identity"
)

record(
    "Frozen scientific-protocol copy identity"
)


# ============================================================
# 12. RERUN BOTH SYNTHETIC TESTS FROM FROZEN COPY
# ============================================================

frozen_original_selftest = run_test(
    FROZEN_RECOVERY_EVALUATOR,
    "self-test",
)

require(
    frozen_original_selftest.returncode
    == 0,
    "Frozen recovery evaluator original metric "
    "self-test failed.",
)

require(
    "SELF-TEST STATUS: PASS"
    in frozen_original_selftest.stdout,
    "Frozen evaluator original metric self-test "
    "did not emit PASS.",
)


frozen_recovery_selftest = run_test(
    FROZEN_RECOVERY_EVALUATOR,
    "recovery-self-test",
)

require(
    frozen_recovery_selftest.returncode
    == 0,
    "Frozen recovery geometry self-test failed.",
)

require(
    "RECOVERY GEOMETRY SELF-TEST: PASS"
    in frozen_recovery_selftest.stdout,
    "Frozen recovery geometry self-test did not "
    "emit PASS.",
)


ORIGINAL_SELFTEST_LOG.write_text(
    (
        "STDOUT\n"
        "======\n"
        + frozen_original_selftest.stdout
        + "\n"
        "STDERR\n"
        "======\n"
        + frozen_original_selftest.stderr
    ),
    encoding="utf-8",
)

RECOVERY_SELFTEST_LOG.write_text(
    (
        "STDOUT\n"
        "======\n"
        + frozen_recovery_selftest.stdout
        + "\n"
        "STDERR\n"
        "======\n"
        + frozen_recovery_selftest.stderr
    ),
    encoding="utf-8",
)


original_selftest_log_sha = sha256_file(
    ORIGINAL_SELFTEST_LOG
)

recovery_selftest_log_sha = sha256_file(
    RECOVERY_SELFTEST_LOG
)


record(
    "Frozen original metric-engine self-test",
    sha256=original_selftest_log_sha,
)

record(
    "Frozen recovery geometry self-test",
    sha256=recovery_selftest_log_sha,
)


# ============================================================
# 13. SHA RECORDS
# ============================================================

ORIGINAL_EVALUATOR_SHA_RECORD.write_text(
    (
        f"{original_evaluator_sha}  "
        f"{ORIGINAL_FROZEN_EVALUATOR.name}\n"
    ),
    encoding="utf-8",
)

RECOVERY_EVALUATOR_SHA_RECORD.write_text(
    (
        f"{frozen_recovery_sha}  "
        f"{FROZEN_RECOVERY_EVALUATOR.name}\n"
    ),
    encoding="utf-8",
)

record(
    "Evaluator SHA records"
)


# ============================================================
# 14. FREEZE IMPLEMENTATION METADATA
# ============================================================

freeze = {
    "freeze_name":
        (
            "DASO-Net RSC Recovery Evaluator "
            "Implementation Freeze v1"
        ),

    "status":
        "FROZEN_BEFORE_RECOVERY_FORMAL_INFERENCE",

    "scientific_action":
        (
            "Infrastructure recovery for finite "
            "non-positive-area decoded geometry."
        ),

    "recovery_inference_performed":
        False,

    "scientific_metrics_computed":
        False,

    "original_frozen_evaluator":
        {
            "path":
                str(
                    ORIGINAL_FROZEN_EVALUATOR
                ),

            "sha256":
                original_evaluator_sha,

            "modified":
                False,
        },

    "recovery_evaluator":
        {
            "path":
                str(
                    FROZEN_RECOVERY_EVALUATOR
                ),

            "sha256":
                frozen_recovery_sha,

            "formal_execution_rule":
                (
                    "All recovery formal RSC evaluations "
                    "must execute this exact frozen recovery "
                    "evaluator copy."
                ),
        },

    "frozen_scientific_protocol":
        {
            "path":
                str(
                    FROZEN_PROTOCOL_COPY
                ),

            "sha256":
                EXPECTED_PROTOCOL_SHA256,

            "scientific_fields_modified":
                False,
        },

    "frozen_recovery_policy":
        {
            "path":
                str(
                    FROZEN_RECOVERY_POLICY_COPY
                ),

            "sha256":
                EXPECTED_RECOVERY_POLICY_SHA256,
        },

    "recovery_rule":
        {
            "condition":
                (
                    "finite decoded box AND "
                    "(x2 <= x1 OR y2 <= y1)"
                ),

            "action":
                "discard_prediction",

            "application_point":
                (
                    "before existing original-image center "
                    "filter and clipping"
                ),

            "nonfinite_box_remains_fatal":
                True,

            "nonfinite_score_remains_fatal":
                True,

            "invalid_coordinate_count_remains_fatal":
                True,
        },

    "unchanged_scientific_components":
        {
            "model":
                True,

            "checkpoints":
                True,

            "seeds":
                True,

            "validated_decoder":
                True,

            "decode_min_score":
                0.05,

            "topk":
                100,

            "nms_iou":
                0.50,

            "mask_threshold":
                0.50,

            "local_peak_kernel":
                3,

            "operating_thresholds":
                {
                    "seed1":
                        0.30,

                    "seed2":
                        0.35,

                    "seed3":
                        0.35,
                },

            "metric_definitions":
                True,

            "primary_membership":
                True,

            "sensitivity_membership":
                True,
        },

    "recovery_execution":
        {
            "output_root":
                str(
                    RECOVERY_OUTPUT_ROOT
                ),

            "execution_directory":
                str(
                    RECOVERY_EXECUTION_DIR
                ),

            "rerun_seed1_from_image_1":
                True,

            "rerun_seed2_from_image_1":
                True,

            "rerun_seed3_from_image_1":
                True,

            "all_three_required":
                True,

            "mix_original_seed3_with_recovery_seeds":
                False,

            "aggregate_recovery_outputs_only":
                True,
        },

    "synthetic_validation":
        {
            "original_metric_engine":
                {
                    "status":
                        "PASS",

                    "log":
                        str(
                            ORIGINAL_SELFTEST_LOG
                        ),

                    "log_sha256":
                        original_selftest_log_sha,
                },

            "recovery_geometry":
                {
                    "status":
                        "PASS",

                    "log":
                        str(
                            RECOVERY_SELFTEST_LOG
                        ),

                    "log_sha256":
                        recovery_selftest_log_sha,
                },

            "rsc_inference":
                False,
        },

    "scientific_prohibitions":
        {
            "rsc_training":
                False,

            "rsc_crpa":
                False,

            "rsc_calibration":
                False,

            "rsc_threshold_tuning":
                False,

            "checkpoint_selection":
                False,

            "seed_selection":
                False,

            "best_seed_selection":
                False,

            "sample_removal":
                False,

            "decoder_change":
                False,

            "model_change":
                False,
        },

    "checks":
        checks,

    "readiness":
        "READY_FOR_ALL_THREE_RSC_RECOVERY_SEEDS",
}


write_json(
    FREEZE_JSON,
    freeze,
)

freeze_sha = sha256_file(
    FREEZE_JSON
)

FREEZE_SHA_RECORD.write_text(
    (
        f"{freeze_sha}  "
        f"{FREEZE_JSON.name}\n"
    ),
    encoding="utf-8",
)


# ============================================================
# 15. READ-BACK
# ============================================================

readback = read_json(
    FREEZE_JSON
)

require(
    readback.get("status")
    == "FROZEN_BEFORE_RECOVERY_FORMAL_INFERENCE",
    "Recovery implementation-freeze read-back "
    "status failure.",
)

require(
    readback.get("readiness")
    == "READY_FOR_ALL_THREE_RSC_RECOVERY_SEEDS",
    "Recovery implementation-freeze readiness failure.",
)

require(
    readback[
        "recovery_evaluator"
    ][
        "sha256"
    ]
    == frozen_recovery_sha,
    "Recovery evaluator SHA read-back failure.",
)

require(
    readback[
        "recovery_inference_performed"
    ]
    is False,
    "Implementation freeze unexpectedly claims "
    "recovery inference.",
)

require(
    readback[
        "scientific_metrics_computed"
    ]
    is False,
    "Implementation freeze unexpectedly claims "
    "scientific metrics.",
)

require(
    sha256_file(
        FROZEN_RECOVERY_EVALUATOR
    )
    == frozen_recovery_sha,
    "Frozen recovery evaluator changed after freeze.",
)

require(
    sha256_file(
        FREEZE_JSON
    )
    == freeze_sha,
    "Recovery implementation-freeze JSON changed "
    "during read-back.",
)


# ============================================================
# 16. FINAL REPORT
# ============================================================

print()

print("=" * 78)

print(
    "DASO-NET RSC RECOVERY EVALUATOR "
    "IMPLEMENTATION: FROZEN"
)

print("=" * 78)

print()

print(
    "Status:"
)

print(
    "  FROZEN_BEFORE_RECOVERY_FORMAL_INFERENCE"
)

print()

print(
    "Original frozen evaluator SHA256:"
)

print(
    f"  {original_evaluator_sha}"
)

print()

print(
    "Recovery evaluator SHA256:"
)

print(
    f"  {frozen_recovery_sha}"
)

print()

print(
    "Recovery policy SHA256:"
)

print(
    f"  {policy_sha}"
)

print()

print(
    "Recovery rule:"
)

print(
    "  finite box with x2 <= x1 OR y2 <= y1"
)

print(
    "  -> DISCARD prediction"
)

print()

print(
    "Original metric-engine self-test: PASS"
)

print(
    "Recovery geometry self-test:      PASS"
)

print()

print(
    "RSC recovery inference performed: NO"
)

print(
    "Scientific metrics computed:      NO"
)

print(
    "Decoder modified:                 NO"
)

print(
    "Model/checkpoints modified:       NO"
)

print(
    "Thresholds modified/tuned:        NO"
)

print(
    "Calibration:                      NO"
)

print(
    "Seed/checkpoint selection:        NO"
)

print(
    "Dataset membership changed:       NO"
)

print()

print(
    "Frozen recovery evaluator:"
)

print(
    f"  {FROZEN_RECOVERY_EVALUATOR}"
)

print()

print(
    "Implementation freeze JSON:"
)

print(
    f"  {FREEZE_JSON}"
)

print()

print(
    "Implementation freeze SHA256:"
)

print(
    f"  {freeze_sha}"
)

print()

print(
    "NEXT READINESS:"
)

print(
    "  READY_FOR_ALL_THREE_RSC_RECOVERY_SEEDS"
)

print()

print(
    "DO NOT RUN RECOVERY INFERENCE MANUALLY YET."
)

print("=" * 78)

DASO-Net — RSC RECOVERY EVALUATOR IMPLEMENTATION FREEZE v1

THIS CELL PERFORMS NO RSC MODEL INFERENCE.

Recovery policy already frozen:
  finite decoded non-positive-area box -> DISCARD

PASS  Frozen recovery-policy identity
PASS  Original frozen evaluator identity
PASS  Frozen scientific protocol identity
PASS  Recovery evaluator source present
PASS  Recovery evaluator Python compile
PASS  Original frozen metric-engine self-test
PASS  Recovery geometry synthetic self-test
PASS  Frozen recovery evaluator byte identity
PASS  Frozen recovery-policy copy identity
PASS  Frozen scientific-protocol copy identity
PASS  Frozen original metric-engine self-test
PASS  Frozen recovery geometry self-test
PASS  Evaluator SHA records

DASO-NET RSC RECOVERY EVALUATOR IMPLEMENTATION: FROZEN

Status:
  FROZEN_BEFORE_RECOVERY_FORMAL_INFERENCE

Original frozen evaluator SHA256:
  59a3de31f2a5244ba32c53b705b050e033b5d17c8de7efb36c17dbc49f0a7723

Recovery evaluator SHA256:
  d0df4ad7cdd60aae23fdff7d2c708c93

In [41]:
# ============================================================
# DASO-Net — FORMAL RSC THREE-SEED RECOVERY EXECUTION v1
#
# PURPOSE
# -------
# Execute ALL THREE final seeds from image 1 using the exact
# frozen recovery evaluator.
#
# Recovery rule already frozen BEFORE this execution:
#
#   finite decoded box AND (x2 <= x1 OR y2 <= y1)
#       -> DISCARD prediction
#
# Everything else remains scientifically unchanged.
#
# SCIENTIFIC PROTECTIONS
# ----------------------
# - exact frozen recovery evaluator
# - exact frozen scientific protocol
# - exact frozen recovery policy
# - exact 3 final checkpoints
# - exact frozen thresholds .30 / .35 / .35
# - PRIMARY = official authors' test-250
# - SENSITIVITY = exact-dedup-247
# - class-agnostic bounding-box detection only
# - no training
# - no CRPA
# - no calibration
# - no threshold tuning
# - no checkpoint selection
# - no seed selection
# - no best-seed selection
#
# EXECUTION RULE
# --------------
# Seed1 -> Seed2 -> Seed3.
#
# Intermediate scientific metrics are captured but NOT displayed.
# Every seed is attempted even if an earlier seed fails.
#
# Acceptance requires:
#   1. all 3 recovery seeds PASS
#   2. all 3 complete 250/250
#   3. recovery provenance/diagnostics close correctly
#   4. Seed-3 recovery degenerate-discard count == 0
#   5. Seed-3 prediction JSONL SHA == original Seed-3 SHA
#   6. Seed-3 primary CSV SHA == original Seed-3 SHA
#   7. Seed-3 dedup CSV SHA == original Seed-3 SHA
#   8. Seed-3 scientific metric dictionaries == original exactly
#
# Only AFTER all gates pass are scientific results displayed.
#
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import math
import statistics
import subprocess
import sys
import traceback
from pathlib import Path


# ============================================================
# 1. FROZEN RECOVERY IMPLEMENTATION
# ============================================================

RECOVERY_IMPLEMENTATION_FREEZE_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_recovery_evaluator_implementation_freeze_v1"
)

RECOVERY_IMPLEMENTATION_FREEZE_JSON = (
    RECOVERY_IMPLEMENTATION_FREEZE_DIR
    / "RSC_RECOVERY_EVALUATOR_IMPLEMENTATION_FREEZE_v1.json"
)

EXPECTED_RECOVERY_IMPLEMENTATION_FREEZE_SHA256 = (
    "dacca485aca56ce8f6c397cff30b41742"
    "e869d21a0ccaa6424e5b2890cec62e8"
)

FROZEN_RECOVERY_EVALUATOR = (
    RECOVERY_IMPLEMENTATION_FREEZE_DIR
    / "evaluate_rsc_external_recovery_v1_FROZEN.py"
)

EXPECTED_RECOVERY_EVALUATOR_SHA256 = (
    "d0df4ad7cdd60aae23fdff7d2c708c93"
    "8952510df4a8d9d7a9087eecbad85bda"
)


# ============================================================
# 2. FROZEN SCIENTIFIC PROTOCOL
# ============================================================

FROZEN_PROTOCOL = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_evaluator_implementation_freeze_v1/"
    "RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "e3c87caebafe87c7671b5745a480c0ed"
    "f67f444feac4278633da6addae4d1f7e"
)


# ============================================================
# 3. FROZEN RECOVERY POLICY
# ============================================================

RECOVERY_POLICY = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_degenerate_box_recovery_policy_freeze_v1/"
    "RSC_DEGENERATE_BOX_RECOVERY_POLICY_FREEZE_v1.json"
)

EXPECTED_RECOVERY_POLICY_SHA256 = (
    "aa754df77f897c68421a065ffa0b0ffc"
    "fe6b332b8dba96b6415d089cb2216f8b"
)


# ============================================================
# 4. ORIGINAL FAILED FORMAL RUN — PRESERVED
# ============================================================

ORIGINAL_EXECUTION_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_formal_three_seed_execution_v1"
)

ORIGINAL_EXECUTION_STATE = (
    ORIGINAL_EXECUTION_DIR
    / "RSC_FORMAL_EXECUTION_STATE_v1.json"
)

ORIGINAL_OUTPUT_ROOT = Path(
    "/kaggle/working/dasonet_final_evaluation/"
    "rsc_external_detection_v1"
)

ORIGINAL_SEED3_DIR = (
    ORIGINAL_OUTPUT_ROOT
    / "seed03_s252916394"
)

ORIGINAL_SEED3_SUMMARY = (
    ORIGINAL_SEED3_DIR
    / "RSC_EXTERNAL_DETECTION_SUMMARY.json"
)

ORIGINAL_SEED3_PREDICTIONS = (
    ORIGINAL_SEED3_DIR
    / "rsc_predictions_decoder_floor.jsonl"
)

ORIGINAL_SEED3_PRIMARY_CSV = (
    ORIGINAL_SEED3_DIR
    / "rsc_primary250_per_image.csv"
)

ORIGINAL_SEED3_DEDUP_CSV = (
    ORIGINAL_SEED3_DIR
    / "rsc_dedup247_per_image.csv"
)


# ============================================================
# 5. RECOVERY FORMAL OUTPUT
# ============================================================

RECOVERY_OUTPUT_ROOT = Path(
    "/kaggle/working/dasonet_final_evaluation/"
    "rsc_external_detection_v1_recovery01"
)

EXECUTION_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_formal_three_seed_recovery_execution_v1"
)

LOG_DIR = (
    EXECUTION_DIR
    / "logs"
)

EXECUTION_CONTRACT_JSON = (
    EXECUTION_DIR
    / "RSC_FORMAL_RECOVERY_EXECUTION_CONTRACT_v1.json"
)

EXECUTION_CONTRACT_SHA_RECORD = (
    EXECUTION_DIR
    / "RSC_FORMAL_RECOVERY_EXECUTION_CONTRACT_v1.sha256.txt"
)

EXECUTION_STATE_JSON = (
    EXECUTION_DIR
    / "RSC_FORMAL_RECOVERY_EXECUTION_STATE_v1.json"
)

PER_SEED_METRICS_CSV = (
    EXECUTION_DIR
    / "RSC_FORMAL_RECOVERY_THREE_SEED_METRICS_v1.csv"
)

AGGREGATE_JSON = (
    EXECUTION_DIR
    / "RSC_FORMAL_RECOVERY_THREE_SEED_AGGREGATE_v1.json"
)

AGGREGATE_SHA_RECORD = (
    EXECUTION_DIR
    / "RSC_FORMAL_RECOVERY_THREE_SEED_AGGREGATE_v1.sha256.txt"
)

ARTIFACT_MANIFEST_JSON = (
    EXECUTION_DIR
    / "RSC_FORMAL_RECOVERY_ARTIFACT_MANIFEST_v1.json"
)

ARTIFACT_MANIFEST_SHA_RECORD = (
    EXECUTION_DIR
    / "RSC_FORMAL_RECOVERY_ARTIFACT_MANIFEST_v1.sha256.txt"
)


# ============================================================
# 6. FROZEN FINAL MODEL CONTRACT
# ============================================================

EXPECTED_MODELS = {
    1: {
        "seed": 1782656963,
        "threshold": 0.30,
        "checkpoint_sha256":
            (
                "e4c4b79ad4b8c6e18ba2b1e3d6cbe796"
                "c9cc301bd6fcc7c596fc290ad3a7e81d"
            ),
    },

    2: {
        "seed": 901395089,
        "threshold": 0.35,
        "checkpoint_sha256":
            (
                "19cdb834a46240916b2c8023e8c3afe57"
                "38d9c7cd5d8f818c62ad1e289df1ad2"
            ),
    },

    3: {
        "seed": 252916394,
        "threshold": 0.35,
        "checkpoint_sha256":
            (
                "be7b7941f2f62179e15074aa973bf6da0"
                "eb9018ec9133486512bd1f87153f447"
            ),
    },
}


# ============================================================
# 7. HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def require(
    condition: bool,
    message: str,
) -> None:
    if not condition:
        raise RuntimeError(message)


def require_file(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_file(),
        f"{label} missing:\n{path}",
    )


def require_dir(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_dir(),
        f"{label} missing:\n{path}",
    )


def read_json(path: Path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path: Path,
    obj,
) -> None:
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def read_csv_rows(path: Path):
    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:
        return list(
            csv.DictReader(f)
        )


def write_csv_rows(
    path: Path,
    rows,
) -> None:
    require(
        len(rows) > 0,
        "Cannot write empty CSV.",
    )

    fieldnames = list(
        rows[0].keys()
    )

    with path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        for row in rows:
            writer.writerow(row)


def finite_number(
    value,
    label: str,
) -> float:
    try:
        number = float(
            value
        )

    except Exception as exc:
        raise RuntimeError(
            f"Could not parse numeric field "
            f"{label}: {value!r}"
        ) from exc

    require(
        math.isfinite(
            number
        ),
        f"Non-finite numeric field "
        f"{label}: {value!r}",
    )

    return number


def mean_sample_sd(values):
    numeric = [
        finite_number(
            value,
            "aggregate value",
        )
        for value in values
    ]

    require(
        len(numeric) == 3,
        "Three-seed aggregate requires exactly "
        "three numeric values.",
    )

    return {
        "n":
            3,

        "values":
            numeric,

        "mean":
            float(
                statistics.mean(
                    numeric
                )
            ),

        "sample_sd":
            float(
                statistics.stdev(
                    numeric
                )
            ),
    }


def optional_mean_sample_sd(values):
    numeric = [
        finite_number(
            value,
            "optional aggregate value",
        )
        for value in values
        if value is not None
    ]

    result = {
        "n_total":
            len(
                values
            ),

        "n_defined":
            len(
                numeric
            ),

        "values":
            [
                (
                    None
                    if value is None
                    else float(
                        value
                    )
                )
                for value
                in values
            ],

        "mean":
            None,

        "sample_sd":
            None,
    }

    if len(
        numeric
    ) >= 1:
        result[
            "mean"
        ] = float(
            statistics.mean(
                numeric
            )
        )

    if len(
        numeric
    ) >= 2:
        result[
            "sample_sd"
        ] = float(
            statistics.stdev(
                numeric
            )
        )

    return result


def read_prediction_jsonl(
    path: Path,
):
    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        for line_number, line in enumerate(
            f,
            start=1,
        ):

            text = line.strip()

            if not text:
                continue

            try:
                records.append(
                    json.loads(
                        text
                    )
                )

            except Exception as exc:
                raise RuntimeError(
                    "Invalid prediction JSONL.\n"
                    f"Path: {path}\n"
                    f"Line: {line_number}"
                ) from exc

    return records


# ============================================================
# 8. BANNER / NON-OVERWRITE
# ============================================================

print("=" * 80)

print(
    "DASO-Net — FORMAL RSC THREE-SEED "
    "RECOVERY EXECUTION v1"
)

print("=" * 80)

print()

print(
    "Recovery rule:"
)

print(
    "  finite decoded non-positive-area box -> DISCARD"
)

print()

print(
    "All three final seeds will rerun from image 1."
)

print(
    "Intermediate scientific metrics will remain hidden."
)

print()


if EXECUTION_DIR.exists():
    raise RuntimeError(
        "Recovery execution directory already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{EXECUTION_DIR}"
    )


if RECOVERY_OUTPUT_ROOT.exists():
    raise RuntimeError(
        "Recovery formal-output root already exists BEFORE "
        "this locked execution:\n"
        f"{RECOVERY_OUTPUT_ROOT}\n\n"
        "Preserve it rather than overwriting/reusing it."
    )


# ============================================================
# 9. PRE-FLIGHT PROVENANCE
# ============================================================

require_file(
    RECOVERY_IMPLEMENTATION_FREEZE_JSON,
    "Recovery evaluator implementation freeze",
)

implementation_freeze_sha = sha256_file(
    RECOVERY_IMPLEMENTATION_FREEZE_JSON
)

require(
    implementation_freeze_sha
    == EXPECTED_RECOVERY_IMPLEMENTATION_FREEZE_SHA256,
    "Recovery implementation-freeze SHA mismatch.",
)

implementation_freeze = read_json(
    RECOVERY_IMPLEMENTATION_FREEZE_JSON
)

require(
    implementation_freeze.get(
        "status"
    )
    == "FROZEN_BEFORE_RECOVERY_FORMAL_INFERENCE",
    "Recovery implementation-freeze status changed.",
)

require(
    implementation_freeze.get(
        "readiness"
    )
    == "READY_FOR_ALL_THREE_RSC_RECOVERY_SEEDS",
    "Recovery implementation-freeze readiness changed.",
)

require(
    implementation_freeze[
        "recovery_inference_performed"
    ]
    is False,
    "Implementation freeze unexpectedly says "
    "recovery inference had occurred.",
)


require_file(
    FROZEN_RECOVERY_EVALUATOR,
    "Frozen recovery evaluator",
)

recovery_evaluator_sha = sha256_file(
    FROZEN_RECOVERY_EVALUATOR
)

require(
    recovery_evaluator_sha
    == EXPECTED_RECOVERY_EVALUATOR_SHA256,
    "Frozen recovery evaluator SHA mismatch.",
)


require_file(
    FROZEN_PROTOCOL,
    "Frozen scientific protocol",
)

protocol_sha = sha256_file(
    FROZEN_PROTOCOL
)

require(
    protocol_sha
    == EXPECTED_PROTOCOL_SHA256,
    "Frozen scientific protocol SHA mismatch.",
)

protocol = read_json(
    FROZEN_PROTOCOL
)


require_file(
    RECOVERY_POLICY,
    "Frozen recovery policy",
)

recovery_policy_sha = sha256_file(
    RECOVERY_POLICY
)

require(
    recovery_policy_sha
    == EXPECTED_RECOVERY_POLICY_SHA256,
    "Frozen recovery-policy SHA mismatch.",
)

recovery_policy = read_json(
    RECOVERY_POLICY
)

require(
    recovery_policy.get(
        "status"
    )
    == "FROZEN_BEFORE_RECOVERY_INFERENCE",
    "Frozen recovery-policy status changed.",
)


# ============================================================
# 10. VERIFY ORIGINAL FORMAL OUTPUT IS PRESERVED
# ============================================================

require_dir(
    ORIGINAL_EXECUTION_DIR,
    "Original formal execution directory",
)

require_file(
    ORIGINAL_EXECUTION_STATE,
    "Original formal execution state",
)

original_execution_state = read_json(
    ORIGINAL_EXECUTION_STATE
)

require(
    original_execution_state.get(
        "status"
    )
    == "FAIL_ONE_OR_MORE_FORMAL_SEEDS",
    "Original failed formal execution state changed.",
)

require_dir(
    ORIGINAL_OUTPUT_ROOT,
    "Original formal output root",
)

require_file(
    ORIGINAL_SEED3_SUMMARY,
    "Original successful Seed-3 summary",
)

require_file(
    ORIGINAL_SEED3_PREDICTIONS,
    "Original successful Seed-3 predictions",
)

require_file(
    ORIGINAL_SEED3_PRIMARY_CSV,
    "Original successful Seed-3 primary CSV",
)

require_file(
    ORIGINAL_SEED3_DEDUP_CSV,
    "Original successful Seed-3 dedup CSV",
)


original_seed3_summary_sha = sha256_file(
    ORIGINAL_SEED3_SUMMARY
)

original_seed3_predictions_sha = sha256_file(
    ORIGINAL_SEED3_PREDICTIONS
)

original_seed3_primary_sha = sha256_file(
    ORIGINAL_SEED3_PRIMARY_CSV
)

original_seed3_dedup_sha = sha256_file(
    ORIGINAL_SEED3_DEDUP_CSV
)


seed3_gate = recovery_policy[
    "seed3_consistency_gate"
]


require(
    original_seed3_predictions_sha
    == seed3_gate[
        "recovery_seed3_predictions_jsonl_must_match_old_sha"
    ],
    "Original Seed-3 prediction reference SHA changed.",
)

require(
    original_seed3_primary_sha
    == seed3_gate[
        "recovery_seed3_primary_per_image_csv_must_match_old_sha"
    ],
    "Original Seed-3 primary CSV reference SHA changed.",
)

require(
    original_seed3_dedup_sha
    == seed3_gate[
        "recovery_seed3_dedup_per_image_csv_must_match_old_sha"
    ],
    "Original Seed-3 dedup CSV reference SHA changed.",
)


# ============================================================
# 11. VERIFY THREE FINAL CHECKPOINTS / THRESHOLDS
# ============================================================

for seed_index in [
    1,
    2,
    3,
]:
    expected = EXPECTED_MODELS[
        seed_index
    ]

    models = [
        model
        for model
        in protocol[
            "models"
        ]
        if int(
            model[
                "seed_index"
            ]
        )
        == seed_index
    ]

    require(
        len(models) == 1,
        f"Could not uniquely resolve Seed-{seed_index}.",
    )

    model = models[0]

    require(
        int(
            model[
                "seed"
            ]
        )
        == expected[
            "seed"
        ],
        f"Seed-{seed_index} RNG identity changed.",
    )

    require(
        float(
            model[
                "operating_threshold"
            ]
        )
        == expected[
            "threshold"
        ],
        f"Seed-{seed_index} operating threshold changed.",
    )

    require(
        int(
            model[
                "successful_optimizer_updates"
            ]
        )
        == 2750,
        f"Seed-{seed_index} update count changed.",
    )

    require(
        str(
            model[
                "checkpoint_sha256"
            ]
        ).lower()
        == expected[
            "checkpoint_sha256"
        ],
        f"Seed-{seed_index} checkpoint identity changed.",
    )

    checkpoint = Path(
        model[
            "checkpoint"
        ]
    )

    require_file(
        checkpoint,
        f"Seed-{seed_index} checkpoint",
    )

    require(
        sha256_file(
            checkpoint
        )
        == expected[
            "checkpoint_sha256"
        ],
        f"Seed-{seed_index} checkpoint bytes changed.",
    )


print(
    "PASS  Recovery implementation-freeze identity"
)

print(
    "PASS  Frozen recovery evaluator identity"
)

print(
    "PASS  Frozen scientific protocol identity"
)

print(
    "PASS  Frozen recovery-policy identity"
)

print(
    "PASS  Original failed formal execution preserved"
)

print(
    "PASS  Original Seed-3 consistency references"
)

print(
    "PASS  All 3 final checkpoint identities"
)

print(
    "PASS  Frozen thresholds .30/.35/.35"
)

print(
    "PASS  Recovery output root absent"
)

print()


# ============================================================
# 12. CREATE LOCKED RECOVERY EXECUTION CONTRACT
#
# Written BEFORE recovery inference.
# ============================================================

EXECUTION_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

LOG_DIR.mkdir(
    parents=False,
    exist_ok=False,
)


contract = {
    "contract_name":
        (
            "DASO-Net Formal RSC Three-Seed "
            "Recovery Execution Contract v1"
        ),

    "status":
        "FROZEN_BEFORE_RECOVERY_FORMAL_EXECUTION",

    "recovery_implementation_freeze":
        {
            "path":
                str(
                    RECOVERY_IMPLEMENTATION_FREEZE_JSON
                ),

            "sha256":
                implementation_freeze_sha,
        },

    "recovery_evaluator":
        {
            "path":
                str(
                    FROZEN_RECOVERY_EVALUATOR
                ),

            "sha256":
                recovery_evaluator_sha,
        },

    "scientific_protocol":
        {
            "path":
                str(
                    FROZEN_PROTOCOL
                ),

            "sha256":
                protocol_sha,
        },

    "recovery_policy":
        {
            "path":
                str(
                    RECOVERY_POLICY
                ),

            "sha256":
                recovery_policy_sha,
        },

    "recovery_rule":
        (
            "discard finite decoded prediction if "
            "x2 <= x1 OR y2 <= y1 before the existing "
            "original-image center/clipping policy"
        ),

    "recovery_output_root":
        str(
            RECOVERY_OUTPUT_ROOT
        ),

    "seed_execution_order":
        [
            1,
            2,
            3,
        ],

    "models":
        [
            {
                "seed_index":
                    seed_index,

                "seed":
                    EXPECTED_MODELS[
                        seed_index
                    ][
                        "seed"
                    ],

                "operating_threshold":
                    EXPECTED_MODELS[
                        seed_index
                    ][
                        "threshold"
                    ],

                "checkpoint_sha256":
                    EXPECTED_MODELS[
                        seed_index
                    ][
                        "checkpoint_sha256"
                    ],
            }
            for seed_index
            in [
                1,
                2,
                3,
            ]
        ],

    "scientific_sets":
        {
            "primary":
                {
                    "images":
                        250,

                    "gt_boxes":
                        539,

                    "zero_gt_images":
                        4,

                    "role":
                        "PRIMARY",
                },

            "sensitivity":
                {
                    "images":
                        247,

                    "gt_boxes":
                        539,

                    "zero_gt_images":
                        1,

                    "role":
                        "SENSITIVITY ONLY",
                },
        },

    "seed3_consistency_gate":
        {
            "required_preexternalclip_degenerate_discard_count":
                0,

            "required_prediction_jsonl_sha256":
                original_seed3_predictions_sha,

            "required_primary_csv_sha256":
                original_seed3_primary_sha,

            "required_dedup_csv_sha256":
                original_seed3_dedup_sha,

            "scientific_metric_values_must_match_original":
                True,
        },

    "execution_rules":
        {
            "rerun_all_three_from_image_1":
                True,

            "execute_all_three_seeds":
                True,

            "display_intermediate_scientific_metrics":
                False,

            "inspect_intermediate_scientific_metrics":
                False,

            "continue_if_prior_seed_fails":
                True,

            "metric_based_early_stop":
                False,

            "use_only_recovery_outputs_for_final_aggregate":
                True,

            "mix_original_seed3_into_final_aggregate":
                False,

            "rsc_training":
                False,

            "rsc_crpa":
                False,

            "rsc_calibration":
                False,

            "rsc_threshold_tuning":
                False,

            "checkpoint_selection":
                False,

            "seed_selection":
                False,

            "best_seed_selection":
                False,
        },

    "aggregate_rule":
        "mean ± sample SD across recovery Seeds 1, 2, and 3",
}


write_json(
    EXECUTION_CONTRACT_JSON,
    contract,
)

contract_sha = sha256_file(
    EXECUTION_CONTRACT_JSON
)

EXECUTION_CONTRACT_SHA_RECORD.write_text(
    (
        f"{contract_sha}  "
        f"{EXECUTION_CONTRACT_JSON.name}\n"
    ),
    encoding="utf-8",
)


state = {
    "execution_name":
        (
            "DASO-Net Formal RSC Three-Seed "
            "Recovery Execution v1"
        ),

    "status":
        "RUNNING",

    "contract_sha256":
        contract_sha,

    "intermediate_scientific_metrics_displayed":
        False,

    "rsc_threshold_tuning":
        False,

    "rsc_calibration":
        False,

    "checkpoint_selection":
        False,

    "seed_selection":
        False,

    "runs":
        [],
}


write_json(
    EXECUTION_STATE_JSON,
    state,
)


print(
    "PASS  Recovery execution contract frozen"
)

print(
    f"Contract SHA256: {contract_sha}"
)

print()

print(
    "Beginning recovery formal execution."
)

print(
    "Scientific metrics remain hidden until all "
    "three recovery seeds finish."
)

print()


# ============================================================
# 13. RUN ALL THREE RECOVERY SEEDS
# ============================================================

run_results = []


for seed_index in [
    1,
    2,
    3,
]:

    expected = EXPECTED_MODELS[
        seed_index
    ]

    print(
        f"START Recovery Seed-{seed_index}"
    )

    stdout_path = (
        LOG_DIR
        / (
            f"seed{seed_index:02d}_"
            "recovery_stdout.txt"
        )
    )

    stderr_path = (
        LOG_DIR
        / (
            f"seed{seed_index:02d}_"
            "recovery_stderr.txt"
        )
    )

    command = [
        sys.executable,
        str(
            FROZEN_RECOVERY_EVALUATOR
        ),
        "--mode",
        "evaluate",
        "--protocol",
        str(
            FROZEN_PROTOCOL
        ),
        "--seed-index",
        str(
            seed_index
        ),
        "--device",
        "cuda",
    ]

    result = subprocess.run(
        command,
        check=False,
        text=True,
        capture_output=True,
    )

    stdout_path.write_text(
        result.stdout,
        encoding="utf-8",
    )

    stderr_path.write_text(
        result.stderr,
        encoding="utf-8",
    )

    run_record = {
        "seed_index":
            seed_index,

        "seed":
            expected[
                "seed"
            ],

        "returncode":
            int(
                result.returncode
            ),

        "status":
            (
                "PASS"
                if result.returncode == 0
                else "FAIL"
            ),

        "stdout":
            str(
                stdout_path
            ),

        "stdout_sha256":
            sha256_file(
                stdout_path
            ),

        "stderr":
            str(
                stderr_path
            ),

        "stderr_sha256":
            sha256_file(
                stderr_path
            ),
    }

    run_results.append(
        run_record
    )

    state[
        "runs"
    ] = run_results

    write_json(
        EXECUTION_STATE_JSON,
        state,
    )

    print(
        f"END   Recovery Seed-{seed_index} "
        f"returncode={result.returncode}"
    )


# ============================================================
# 14. ALL THREE HAVE NOW BEEN ATTEMPTED
# ============================================================

if any(
    run[
        "returncode"
    ]
    != 0
    for run
    in run_results
):

    state[
        "status"
    ] = (
        "FAIL_ONE_OR_MORE_RECOVERY_SEEDS"
    )

    state[
        "aggregate_produced"
    ] = False

    write_json(
        EXECUTION_STATE_JSON,
        state,
    )

    print()

    print("=" * 80)

    print(
        "RSC RECOVERY EXECUTION: "
        "ONE OR MORE SEEDS FAILED"
    )

    print("=" * 80)

    print()

    for run in run_results:

        print(
            f"Recovery Seed-{run['seed_index']}: "
            f"{run['status']} "
            f"(returncode={run['returncode']})"
        )

        print(
            f"  stdout: {run['stdout']}"
        )

        print(
            f"  stderr: {run['stderr']}"
        )

    print()

    print(
        "All three frozen recovery seeds were attempted."
    )

    print(
        "No three-seed recovery aggregate was produced."
    )

    print()

    print(
        "Preserve ALL recovery outputs."
    )

    raise RuntimeError(
        "At least one formal RSC recovery seed failed. "
        "Do not delete or reuse any recovery output."
    )


print()

print(
    "PASS  All three recovery seed subprocesses completed"
)

print()


# ============================================================
# 15. AUDIT ALL THREE RECOVERY OUTPUTS
# ============================================================

flat_rows = []

seed_results = []

artifact_records = []

recovery_discard_counts = {}


for seed_index in [
    1,
    2,
    3,
]:

    expected = EXPECTED_MODELS[
        seed_index
    ]

    seed_dir = (
        RECOVERY_OUTPUT_ROOT
        / (
            f"seed{seed_index:02d}_"
            f"s{expected['seed']}"
        )
    )

    require_dir(
        seed_dir,
        f"Recovery Seed-{seed_index} output directory",
    )


    run_state_path = (
        seed_dir
        / "RSC_RUN_STATE.json"
    )

    summary_path = (
        seed_dir
        / "RSC_EXTERNAL_DETECTION_SUMMARY.json"
    )

    summary_sha_record = (
        seed_dir
        / "RSC_EXTERNAL_DETECTION_SUMMARY.sha256.txt"
    )

    predictions_path = (
        seed_dir
        / "rsc_predictions_decoder_floor.jsonl"
    )

    primary_csv = (
        seed_dir
        / "rsc_primary250_per_image.csv"
    )

    dedup_csv = (
        seed_dir
        / "rsc_dedup247_per_image.csv"
    )

    diagnostics_json = (
        seed_dir
        / "RSC_RECOVERY_GEOMETRY_DIAGNOSTICS.json"
    )

    diagnostics_csv = (
        seed_dir
        / "rsc_recovery_geometry_per_image.csv"
    )

    provenance_json = (
        seed_dir
        / "RSC_RECOVERY_PROVENANCE.json"
    )


    for path, label in [
        (
            run_state_path,
            "run state",
        ),
        (
            summary_path,
            "summary",
        ),
        (
            summary_sha_record,
            "summary SHA record",
        ),
        (
            predictions_path,
            "prediction JSONL",
        ),
        (
            primary_csv,
            "primary CSV",
        ),
        (
            dedup_csv,
            "dedup CSV",
        ),
        (
            diagnostics_json,
            "recovery diagnostics JSON",
        ),
        (
            diagnostics_csv,
            "recovery diagnostics CSV",
        ),
        (
            provenance_json,
            "recovery provenance JSON",
        ),
    ]:
        require_file(
            path,
            f"Recovery Seed-{seed_index} {label}",
        )


    # --------------------------------------------------------
    # Run state
    # --------------------------------------------------------

    run_state = read_json(
        run_state_path
    )

    require(
        run_state.get(
            "status"
        )
        == "PASS",
        f"Recovery Seed-{seed_index} run state != PASS.",
    )

    require(
        int(
            run_state[
                "images_completed"
            ]
        )
        == 250,
        f"Recovery Seed-{seed_index} images_completed != 250.",
    )

    require(
        int(
            run_state[
                "seed_index"
            ]
        )
        == seed_index,
        f"Recovery Seed-{seed_index} seed-index mismatch.",
    )

    require(
        int(
            run_state[
                "seed"
            ]
        )
        == expected[
            "seed"
        ],
        f"Recovery Seed-{seed_index} RNG seed mismatch.",
    )

    require(
        float(
            run_state[
                "operating_threshold"
            ]
        )
        == expected[
            "threshold"
        ],
        f"Recovery Seed-{seed_index} threshold mismatch.",
    )

    require(
        run_state[
            "checkpoint_sha256"
        ]
        == expected[
            "checkpoint_sha256"
        ],
        f"Recovery Seed-{seed_index} checkpoint mismatch.",
    )

    require(
        run_state[
            "protocol_sha256"
        ]
        == EXPECTED_PROTOCOL_SHA256,
        f"Recovery Seed-{seed_index} protocol SHA mismatch.",
    )

    require(
        run_state[
            "evaluator_sha256"
        ]
        == EXPECTED_RECOVERY_EVALUATOR_SHA256,
        f"Recovery Seed-{seed_index} evaluator SHA mismatch.",
    )


    # --------------------------------------------------------
    # Formal summary
    # --------------------------------------------------------

    summary_sha = sha256_file(
        summary_path
    )

    summary_sha_lines = [
        line.strip()
        for line in summary_sha_record.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

    require(
        len(
            summary_sha_lines
        )
        == 1,
        f"Recovery Seed-{seed_index} summary "
        "SHA-record format invalid.",
    )

    require(
        summary_sha_lines[
            0
        ].split()[0].lower()
        == summary_sha,
        f"Recovery Seed-{seed_index} summary "
        "SHA-record mismatch.",
    )


    summary = read_json(
        summary_path
    )

    require(
        summary.get(
            "status"
        )
        == "PASS",
        f"Recovery Seed-{seed_index} summary != PASS.",
    )

    require(
        int(
            summary[
                "seed_index"
            ]
        )
        == seed_index,
        f"Recovery Seed-{seed_index} summary "
        "seed-index mismatch.",
    )

    require(
        int(
            summary[
                "seed"
            ]
        )
        == expected[
            "seed"
        ],
        f"Recovery Seed-{seed_index} summary RNG mismatch.",
    )

    require(
        summary[
            "checkpoint_sha256"
        ]
        == expected[
            "checkpoint_sha256"
        ],
        f"Recovery Seed-{seed_index} summary "
        "checkpoint mismatch.",
    )

    require(
        int(
            summary[
                "successful_optimizer_updates"
            ]
        )
        == 2750,
        f"Recovery Seed-{seed_index} update count != 2750.",
    )

    require(
        summary[
            "protocol_sha256"
        ]
        == EXPECTED_PROTOCOL_SHA256,
        f"Recovery Seed-{seed_index} summary "
        "protocol SHA mismatch.",
    )

    require(
        summary[
            "evaluator_sha256"
        ]
        == EXPECTED_RECOVERY_EVALUATOR_SHA256,
        f"Recovery Seed-{seed_index} summary "
        "recovery evaluator SHA mismatch.",
    )

    require(
        float(
            summary[
                "operating_threshold"
            ]
        )
        == expected[
            "threshold"
        ],
        f"Recovery Seed-{seed_index} summary "
        "threshold mismatch.",
    )

    require(
        summary[
            "segmentation_metrics"
        ]
        is False,
        f"Recovery Seed-{seed_index} unexpectedly "
        "reports segmentation metrics.",
    )


    scientific_policy = (
        summary[
            "scientific_policy"
        ]
    )

    for key in [
        "rsc_training",
        "rsc_crpa",
        "rsc_calibration",
        "rsc_threshold_tuning",
        "checkpoint_selection",
        "seed_selection",
        "best_seed_selection",
    ]:

        require(
            scientific_policy[
                key
            ]
            is False,
            (
                f"Recovery Seed-{seed_index} "
                f"unexpected policy flag: {key}"
            ),
        )


    primary = (
        summary[
            "primary_official_test250"
        ]
    )

    sensitivity = (
        summary[
            "sensitivity_exact_dedup247"
        ]
    )


    require(
        primary[
            "images"
        ]
        == 250,
        f"Recovery Seed-{seed_index} primary images != 250.",
    )

    require(
        primary[
            "gt_boxes"
        ]
        == 539,
        f"Recovery Seed-{seed_index} primary GT != 539.",
    )

    require(
        primary[
            "zero_gt_images"
        ]
        == 4,
        f"Recovery Seed-{seed_index} primary zero-GT != 4.",
    )

    require(
        sensitivity[
            "images"
        ]
        == 247,
        f"Recovery Seed-{seed_index} sensitivity images != 247.",
    )

    require(
        sensitivity[
            "gt_boxes"
        ]
        == 539,
        f"Recovery Seed-{seed_index} sensitivity GT != 539.",
    )

    require(
        sensitivity[
            "zero_gt_images"
        ]
        == 1,
        f"Recovery Seed-{seed_index} sensitivity zero-GT != 1.",
    )


    # --------------------------------------------------------
    # Formal metric sanity
    # --------------------------------------------------------

    for subset_name, subset in [
        (
            "primary",
            primary,
        ),
        (
            "sensitivity",
            sensitivity,
        ),
    ]:

        for metric_name in [
            "box_ap50",
            "box_ap75",
            "box_ap50_95",
        ]:

            value = finite_number(
                subset[
                    metric_name
                ],
                (
                    f"Recovery Seed-{seed_index} "
                    f"{subset_name} {metric_name}"
                ),
            )

            require(
                0.0 <= value <= 1.0,
                (
                    f"Recovery Seed-{seed_index} "
                    f"{subset_name} {metric_name} "
                    "outside [0,1]."
                ),
            )


        operating = (
            subset[
                "operating"
            ]
        )

        for metric_name in [
            "precision",
            "recall",
            "f1",
        ]:

            value = finite_number(
                operating[
                    metric_name
                ],
                (
                    f"Recovery Seed-{seed_index} "
                    f"{subset_name} {metric_name}"
                ),
            )

            require(
                0.0 <= value <= 1.0,
                (
                    f"Recovery Seed-{seed_index} "
                    f"{subset_name} {metric_name} "
                    "outside [0,1]."
                ),
            )


        require(
            int(
                operating[
                    "tp"
                ]
            )
            + int(
                operating[
                    "fn"
                ]
            )
            == 539,
            (
                f"Recovery Seed-{seed_index} "
                f"{subset_name} TP+FN != 539."
            ),
        )


    # --------------------------------------------------------
    # Prediction JSONL closure
    # --------------------------------------------------------

    prediction_records = (
        read_prediction_jsonl(
            predictions_path
        )
    )

    require(
        len(
            prediction_records
        )
        == 250,
        f"Recovery Seed-{seed_index} prediction "
        "record count != 250.",
    )

    indices = [
        int(
            record[
                "official_test_index"
            ]
        )
        for record
        in prediction_records
    ]

    require(
        indices
        == list(
            range(
                1,
                251,
            )
        ),
        f"Recovery Seed-{seed_index} prediction "
        "index closure failure.",
    )

    require(
        sum(
            len(
                record[
                    "gt_boxes"
                ]
            )
            for record
            in prediction_records
        )
        == 539,
        f"Recovery Seed-{seed_index} JSONL GT total != 539.",
    )

    require(
        sum(
            1
            for record
            in prediction_records
            if bool(
                record[
                    "dedup247_member"
                ]
            )
        )
        == 247,
        (
            f"Recovery Seed-{seed_index} JSONL "
            "dedup membership != 247."
        ),
    )


    # --------------------------------------------------------
    # Per-image CSV closure
    # --------------------------------------------------------

    primary_rows = read_csv_rows(
        primary_csv
    )

    dedup_rows = read_csv_rows(
        dedup_csv
    )

    require(
        len(
            primary_rows
        )
        == 250,
        f"Recovery Seed-{seed_index} primary CSV rows != 250.",
    )

    require(
        len(
            dedup_rows
        )
        == 247,
        f"Recovery Seed-{seed_index} dedup CSV rows != 247.",
    )


    # --------------------------------------------------------
    # Recovery geometry diagnostics
    # --------------------------------------------------------

    diagnostics = read_json(
        diagnostics_json
    )

    require(
        diagnostics.get(
            "status"
        )
        == "PASS",
        f"Recovery Seed-{seed_index} geometry "
        "diagnostics != PASS.",
    )

    require(
        int(
            diagnostics[
                "seed_index"
            ]
        )
        == seed_index,
        f"Recovery Seed-{seed_index} diagnostic "
        "seed-index mismatch.",
    )

    require(
        int(
            diagnostics[
                "images_with_coordinate_policy_call"
            ]
        )
        == 250,
        f"Recovery Seed-{seed_index} diagnostic "
        "image count != 250.",
    )

    require(
        diagnostics[
            "recovery_policy_sha256"
        ]
        == EXPECTED_RECOVERY_POLICY_SHA256,
        f"Recovery Seed-{seed_index} diagnostic "
        "policy SHA mismatch.",
    )

    require(
        diagnostics[
            "recovery_evaluator_sha256"
        ]
        == EXPECTED_RECOVERY_EVALUATOR_SHA256,
        f"Recovery Seed-{seed_index} diagnostic "
        "evaluator SHA mismatch.",
    )


    totals = (
        diagnostics[
            "totals"
        ]
    )

    degenerate_discard_count = int(
        totals[
            "preexternalclip_degenerate_predictions_discarded"
        ]
    )

    require(
        degenerate_discard_count >= 0,
        f"Recovery Seed-{seed_index} invalid "
        "degenerate-discard count.",
    )

    recovery_discard_counts[
        seed_index
    ] = degenerate_discard_count


    diagnostic_rows = read_csv_rows(
        diagnostics_csv
    )

    require(
        len(
            diagnostic_rows
        )
        == 250,
        f"Recovery Seed-{seed_index} diagnostic CSV rows != 250.",
    )


    # --------------------------------------------------------
    # Recovery provenance
    # --------------------------------------------------------

    recovery_provenance = read_json(
        provenance_json
    )

    require(
        recovery_provenance.get(
            "status"
        )
        == "PASS",
        f"Recovery Seed-{seed_index} provenance != PASS.",
    )

    require(
        recovery_provenance[
            "recovery_evaluator_sha256"
        ]
        == EXPECTED_RECOVERY_EVALUATOR_SHA256,
        f"Recovery Seed-{seed_index} provenance "
        "evaluator SHA mismatch.",
    )

    require(
        recovery_provenance[
            "frozen_scientific_protocol_sha256"
        ]
        == EXPECTED_PROTOCOL_SHA256,
        f"Recovery Seed-{seed_index} provenance "
        "protocol SHA mismatch.",
    )

    require(
        recovery_provenance[
            "recovery_policy_sha256"
        ]
        == EXPECTED_RECOVERY_POLICY_SHA256,
        f"Recovery Seed-{seed_index} provenance "
        "policy SHA mismatch.",
    )

    require(
        recovery_provenance[
            "model_modified"
        ]
        is False,
        f"Recovery Seed-{seed_index} provenance "
        "unexpectedly indicates model modification.",
    )

    require(
        recovery_provenance[
            "decoder_modified"
        ]
        is False,
        f"Recovery Seed-{seed_index} provenance "
        "unexpectedly indicates decoder modification.",
    )

    require(
        recovery_provenance[
            "threshold_modified"
        ]
        is False,
        f"Recovery Seed-{seed_index} provenance "
        "unexpectedly indicates threshold modification.",
    )


    # --------------------------------------------------------
    # Artifact hashes
    # --------------------------------------------------------

    artifacts = {}

    for path in [
        run_state_path,
        summary_path,
        summary_sha_record,
        predictions_path,
        primary_csv,
        dedup_csv,
        diagnostics_json,
        diagnostics_csv,
        provenance_json,
    ]:

        artifact_sha = sha256_file(
            path
        )

        artifacts[
            path.name
        ] = {
            "path":
                str(
                    path
                ),

            "size_bytes":
                path.stat().st_size,

            "sha256":
                artifact_sha,
        }

        artifact_records.append(
            {
                "seed_index":
                    seed_index,

                "seed":
                    expected[
                        "seed"
                    ],

                "filename":
                    path.name,

                "path":
                    str(
                        path
                    ),

                "size_bytes":
                    path.stat().st_size,

                "sha256":
                    artifact_sha,
            }
        )


    # --------------------------------------------------------
    # Flatten metrics
    # --------------------------------------------------------

    primary_op = (
        primary[
            "operating"
        ]
    )

    sensitivity_op = (
        sensitivity[
            "operating"
        ]
    )


    flat_row = {
        "seed_index":
            seed_index,

        "seed":
            expected[
                "seed"
            ],

        "operating_threshold":
            expected[
                "threshold"
            ],

        "preexternalclip_degenerate_predictions_discarded":
            degenerate_discard_count,

        "primary_box_ap50":
            primary[
                "box_ap50"
            ],

        "primary_box_ap75":
            primary[
                "box_ap75"
            ],

        "primary_box_ap50_95":
            primary[
                "box_ap50_95"
            ],

        "primary_precision":
            primary_op[
                "precision"
            ],

        "primary_recall":
            primary_op[
                "recall"
            ],

        "primary_f1":
            primary_op[
                "f1"
            ],

        "primary_tp":
            primary_op[
                "tp"
            ],

        "primary_fp":
            primary_op[
                "fp"
            ],

        "primary_fn":
            primary_op[
                "fn"
            ],

        "primary_mean_matched_box_iou_conditional":
            primary_op[
                "mean_matched_box_iou_conditional"
            ],

        "primary_count_mae":
            primary_op[
                "count_mae"
            ],

        "primary_count_bias":
            primary_op[
                "count_bias"
            ],

        "primary_zero_gt_fp_per_image":
            primary_op[
                "zero_gt_fp_per_image"
            ],

        "primary_zero_gt_fraction_with_ge1_fp":
            primary_op[
                "zero_gt_fraction_with_ge1_fp"
            ],

        "sensitivity_box_ap50":
            sensitivity[
                "box_ap50"
            ],

        "sensitivity_box_ap75":
            sensitivity[
                "box_ap75"
            ],

        "sensitivity_box_ap50_95":
            sensitivity[
                "box_ap50_95"
            ],

        "sensitivity_precision":
            sensitivity_op[
                "precision"
            ],

        "sensitivity_recall":
            sensitivity_op[
                "recall"
            ],

        "sensitivity_f1":
            sensitivity_op[
                "f1"
            ],

        "sensitivity_tp":
            sensitivity_op[
                "tp"
            ],

        "sensitivity_fp":
            sensitivity_op[
                "fp"
            ],

        "sensitivity_fn":
            sensitivity_op[
                "fn"
            ],

        "sensitivity_mean_matched_box_iou_conditional":
            sensitivity_op[
                "mean_matched_box_iou_conditional"
            ],

        "sensitivity_count_mae":
            sensitivity_op[
                "count_mae"
            ],

        "sensitivity_count_bias":
            sensitivity_op[
                "count_bias"
            ],

        "sensitivity_zero_gt_fp_per_image":
            sensitivity_op[
                "zero_gt_fp_per_image"
            ],

        "sensitivity_zero_gt_fraction_with_ge1_fp":
            sensitivity_op[
                "zero_gt_fraction_with_ge1_fp"
            ],
    }


    flat_rows.append(
        flat_row
    )


    seed_results.append(
        {
            "seed_index":
                seed_index,

            "seed":
                expected[
                    "seed"
                ],

            "operating_threshold":
                expected[
                    "threshold"
                ],

            "preexternalclip_degenerate_predictions_discarded":
                degenerate_discard_count,

            "summary_path":
                str(
                    summary_path
                ),

            "summary_sha256":
                summary_sha,

            "primary":
                primary,

            "sensitivity":
                sensitivity,

            "artifacts":
                artifacts,
        }
    )


    print(
        f"PASS  Recovery Seed-{seed_index} "
        "formal output audit"
    )


# ============================================================
# 16. SEED-3 CONSISTENCY GATE
# ============================================================

require(
    recovery_discard_counts[
        3
    ]
    == 0,
    (
        "RECOVERY SEED-3 CONSISTENCY FAILURE:\n"
        "Seed-3 discarded one or more pre-external-clipping "
        "degenerate boxes even though original Seed-3 completed "
        "under the fatal wrapper."
    ),
)


recovery_seed3_dir = (
    RECOVERY_OUTPUT_ROOT
    / "seed03_s252916394"
)

recovery_seed3_predictions = (
    recovery_seed3_dir
    / "rsc_predictions_decoder_floor.jsonl"
)

recovery_seed3_primary_csv = (
    recovery_seed3_dir
    / "rsc_primary250_per_image.csv"
)

recovery_seed3_dedup_csv = (
    recovery_seed3_dir
    / "rsc_dedup247_per_image.csv"
)

recovery_seed3_summary = (
    recovery_seed3_dir
    / "RSC_EXTERNAL_DETECTION_SUMMARY.json"
)


require(
    sha256_file(
        recovery_seed3_predictions
    )
    == original_seed3_predictions_sha,
    (
        "RECOVERY SEED-3 CONSISTENCY FAILURE:\n"
        "Prediction JSONL differs from original Seed-3."
    ),
)

require(
    sha256_file(
        recovery_seed3_primary_csv
    )
    == original_seed3_primary_sha,
    (
        "RECOVERY SEED-3 CONSISTENCY FAILURE:\n"
        "Primary per-image CSV differs from original Seed-3."
    ),
)

require(
    sha256_file(
        recovery_seed3_dedup_csv
    )
    == original_seed3_dedup_sha,
    (
        "RECOVERY SEED-3 CONSISTENCY FAILURE:\n"
        "Dedup per-image CSV differs from original Seed-3."
    ),
)


old_seed3_summary = read_json(
    ORIGINAL_SEED3_SUMMARY
)

new_seed3_summary = read_json(
    recovery_seed3_summary
)


require(
    old_seed3_summary[
        "primary_official_test250"
    ]
    == new_seed3_summary[
        "primary_official_test250"
    ],
    (
        "RECOVERY SEED-3 CONSISTENCY FAILURE:\n"
        "Primary scientific metric dictionary differs."
    ),
)

require(
    old_seed3_summary[
        "sensitivity_exact_dedup247"
    ]
    == new_seed3_summary[
        "sensitivity_exact_dedup247"
    ],
    (
        "RECOVERY SEED-3 CONSISTENCY FAILURE:\n"
        "Sensitivity scientific metric dictionary differs."
    ),
)

require(
    float(
        old_seed3_summary[
            "operating_threshold"
        ]
    )
    == float(
        new_seed3_summary[
            "operating_threshold"
        ]
    ),
    "Recovery Seed-3 operating threshold differs.",
)


print()

print(
    "PASS  Seed-3 recovery degenerate-discard count = 0"
)

print(
    "PASS  Seed-3 prediction JSONL byte identity"
)

print(
    "PASS  Seed-3 primary per-image CSV byte identity"
)

print(
    "PASS  Seed-3 sensitivity per-image CSV byte identity"
)

print(
    "PASS  Seed-3 scientific metric-value identity"
)

print()


# ============================================================
# 17. WRITE PER-SEED TABLE
# ============================================================

write_csv_rows(
    PER_SEED_METRICS_CSV,
    flat_rows,
)

per_seed_metrics_sha = sha256_file(
    PER_SEED_METRICS_CSV
)


# ============================================================
# 18. THREE-SEED AGGREGATION
# ============================================================

def values_for(field):
    return [
        row[
            field
        ]
        for row
        in flat_rows
    ]


aggregate_primary = {
    "box_ap50":
        mean_sample_sd(
            values_for(
                "primary_box_ap50"
            )
        ),

    "box_ap75":
        mean_sample_sd(
            values_for(
                "primary_box_ap75"
            )
        ),

    "box_ap50_95":
        mean_sample_sd(
            values_for(
                "primary_box_ap50_95"
            )
        ),

    "precision":
        mean_sample_sd(
            values_for(
                "primary_precision"
            )
        ),

    "recall":
        mean_sample_sd(
            values_for(
                "primary_recall"
            )
        ),

    "f1":
        mean_sample_sd(
            values_for(
                "primary_f1"
            )
        ),

    "tp":
        mean_sample_sd(
            values_for(
                "primary_tp"
            )
        ),

    "fp":
        mean_sample_sd(
            values_for(
                "primary_fp"
            )
        ),

    "fn":
        mean_sample_sd(
            values_for(
                "primary_fn"
            )
        ),

    "mean_matched_box_iou_conditional":
        optional_mean_sample_sd(
            values_for(
                "primary_mean_matched_box_iou_conditional"
            )
        ),

    "count_mae":
        mean_sample_sd(
            values_for(
                "primary_count_mae"
            )
        ),

    "count_bias":
        mean_sample_sd(
            values_for(
                "primary_count_bias"
            )
        ),

    "zero_gt_fp_per_image":
        mean_sample_sd(
            values_for(
                "primary_zero_gt_fp_per_image"
            )
        ),

    "zero_gt_fraction_with_ge1_fp":
        mean_sample_sd(
            values_for(
                "primary_zero_gt_fraction_with_ge1_fp"
            )
        ),
}


aggregate_sensitivity = {
    "box_ap50":
        mean_sample_sd(
            values_for(
                "sensitivity_box_ap50"
            )
        ),

    "box_ap75":
        mean_sample_sd(
            values_for(
                "sensitivity_box_ap75"
            )
        ),

    "box_ap50_95":
        mean_sample_sd(
            values_for(
                "sensitivity_box_ap50_95"
            )
        ),

    "precision":
        mean_sample_sd(
            values_for(
                "sensitivity_precision"
            )
        ),

    "recall":
        mean_sample_sd(
            values_for(
                "sensitivity_recall"
            )
        ),

    "f1":
        mean_sample_sd(
            values_for(
                "sensitivity_f1"
            )
        ),

    "tp":
        mean_sample_sd(
            values_for(
                "sensitivity_tp"
            )
        ),

    "fp":
        mean_sample_sd(
            values_for(
                "sensitivity_fp"
            )
        ),

    "fn":
        mean_sample_sd(
            values_for(
                "sensitivity_fn"
            )
        ),

    "mean_matched_box_iou_conditional":
        optional_mean_sample_sd(
            values_for(
                "sensitivity_mean_matched_box_iou_conditional"
            )
        ),

    "count_mae":
        mean_sample_sd(
            values_for(
                "sensitivity_count_mae"
            )
        ),

    "count_bias":
        mean_sample_sd(
            values_for(
                "sensitivity_count_bias"
            )
        ),

    "zero_gt_fp_per_image":
        mean_sample_sd(
            values_for(
                "sensitivity_zero_gt_fp_per_image"
            )
        ),

    "zero_gt_fraction_with_ge1_fp":
        mean_sample_sd(
            values_for(
                "sensitivity_zero_gt_fraction_with_ge1_fp"
            )
        ),
}


# ============================================================
# 19. ARTIFACT MANIFEST
# ============================================================

artifact_manifest = {
    "manifest_name":
        (
            "DASO-Net Formal RSC Recovery "
            "Artifact Manifest v1"
        ),

    "status":
        "PASS",

    "execution_contract_sha256":
        contract_sha,

    "recovery_implementation_freeze_sha256":
        implementation_freeze_sha,

    "recovery_evaluator_sha256":
        recovery_evaluator_sha,

    "recovery_policy_sha256":
        recovery_policy_sha,

    "scientific_protocol_sha256":
        protocol_sha,

    "recovery_output_root":
        str(
            RECOVERY_OUTPUT_ROOT
        ),

    "artifacts":
        artifact_records,
}


write_json(
    ARTIFACT_MANIFEST_JSON,
    artifact_manifest,
)

artifact_manifest_sha = sha256_file(
    ARTIFACT_MANIFEST_JSON
)

ARTIFACT_MANIFEST_SHA_RECORD.write_text(
    (
        f"{artifact_manifest_sha}  "
        f"{ARTIFACT_MANIFEST_JSON.name}\n"
    ),
    encoding="utf-8",
)


# ============================================================
# 20. FINAL RECOVERY AGGREGATE
# ============================================================

aggregate = {
    "evaluation_name":
        (
            "DASO-Net RSC Formal Three-Seed "
            "External Detection Recovery01"
        ),

    "status":
        "PASS",

    "scientific_task":
        "class-agnostic bounding-box detection",

    "segmentation_metrics":
        False,

    "recovery_reason":
        (
            "Original formal evaluator treated finite "
            "non-positive-area decoded geometry as fatal. "
            "Recovery discards only those invalid predictions."
        ),

    "recovery_rule":
        (
            "finite decoded box with "
            "x2 <= x1 OR y2 <= y1 -> discard"
        ),

    "scientific_components_changed":
        {
            "model":
                False,

            "decoder":
                False,

            "checkpoints":
                False,

            "seeds":
                False,

            "thresholds":
                False,

            "metrics":
                False,

            "dataset_membership":
                False,

            "preprocessing":
                False,
        },

    "primary_set":
        {
            "name":
                "official authors' RSC test-250",

            "images":
                250,

            "gt_boxes":
                539,

            "zero_gt_images":
                4,

            "role":
                "PRIMARY",
        },

    "sensitivity_set":
        {
            "name":
                "exact-dedup-247",

            "images":
                247,

            "gt_boxes":
                539,

            "zero_gt_images":
                1,

            "role":
                "SENSITIVITY ONLY",
        },

    "provenance":
        {
            "recovery_implementation_freeze_sha256":
                implementation_freeze_sha,

            "recovery_evaluator_sha256":
                recovery_evaluator_sha,

            "recovery_policy_sha256":
                recovery_policy_sha,

            "scientific_protocol_sha256":
                protocol_sha,

            "execution_contract_sha256":
                contract_sha,

            "per_seed_metrics_csv":
                str(
                    PER_SEED_METRICS_CSV
                ),

            "per_seed_metrics_csv_sha256":
                per_seed_metrics_sha,

            "artifact_manifest":
                str(
                    ARTIFACT_MANIFEST_JSON
                ),

            "artifact_manifest_sha256":
                artifact_manifest_sha,

            "original_failed_execution_preserved":
                str(
                    ORIGINAL_EXECUTION_DIR
                ),

            "original_successful_seed3_summary_sha256":
                original_seed3_summary_sha,

            "original_successful_seed3_predictions_sha256":
                original_seed3_predictions_sha,
        },

    "recovery_geometry":
        {
            "seed1_preexternalclip_degenerate_discarded":
                recovery_discard_counts[
                    1
                ],

            "seed2_preexternalclip_degenerate_discarded":
                recovery_discard_counts[
                    2
                ],

            "seed3_preexternalclip_degenerate_discarded":
                recovery_discard_counts[
                    3
                ],
        },

    "seed3_consistency_gate":
        {
            "status":
                "PASS",

            "degenerate_discard_count_zero":
                True,

            "prediction_jsonl_byte_identity":
                True,

            "primary_csv_byte_identity":
                True,

            "sensitivity_csv_byte_identity":
                True,

            "scientific_metric_value_identity":
                True,
        },

    "seeds":
        seed_results,

    "aggregate_rule":
        "mean ± sample SD across all three recovery seeds",

    "primary_official_test250":
        aggregate_primary,

    "sensitivity_exact_dedup247":
        aggregate_sensitivity,

    "scientific_policy":
        {
            "rsc_training":
                False,

            "rsc_crpa":
                False,

            "rsc_calibration":
                False,

            "rsc_threshold_tuning":
                False,

            "checkpoint_selection":
                False,

            "seed_selection":
                False,

            "best_seed_selection":
                False,

            "all_three_final_seeds_included":
                True,
        },
}


write_json(
    AGGREGATE_JSON,
    aggregate,
)

aggregate_sha = sha256_file(
    AGGREGATE_JSON
)

AGGREGATE_SHA_RECORD.write_text(
    (
        f"{aggregate_sha}  "
        f"{AGGREGATE_JSON.name}\n"
    ),
    encoding="utf-8",
)


# ============================================================
# 21. FINAL EXECUTION STATE
# ============================================================

state.update(
    {
        "status":
            "PASS",

        "all_three_recovery_seeds_completed":
            True,

        "intermediate_scientific_metrics_displayed":
            False,

        "seed3_consistency_gate":
            "PASS",

        "aggregate_produced":
            True,

        "per_seed_metrics_csv":
            str(
                PER_SEED_METRICS_CSV
            ),

        "per_seed_metrics_csv_sha256":
            per_seed_metrics_sha,

        "aggregate_json":
            str(
                AGGREGATE_JSON
            ),

        "aggregate_sha256":
            aggregate_sha,

        "artifact_manifest":
            str(
                ARTIFACT_MANIFEST_JSON
            ),

        "artifact_manifest_sha256":
            artifact_manifest_sha,
    }
)


write_json(
    EXECUTION_STATE_JSON,
    state,
)


# ============================================================
# 22. DISPLAY SCIENTIFIC RESULTS
#
# Only now, after:
#   - all 3 recovery seeds complete
#   - every output audit passes
#   - Seed-3 consistency gate passes
#
# ============================================================

def fmt(value):
    return (
        f"{float(value):.6f}"
    )


def fmt_ms(metric):
    return (
        f"{metric['mean']:.6f} "
        f"± {metric['sample_sd']:.6f}"
    )


print()

print("=" * 80)

print(
    "DASO-NET FORMAL RSC THREE-SEED "
    "RECOVERY EVALUATION: PASS"
)

print("=" * 80)

print()

print(
    "All three final recovery seeds completed."
)

print(
    "Seed-3 consistency gate: PASS"
)

print(
    "No intermediate scientific result was displayed "
    "before all three completed."
)

print()


print(
    "Recovery geometry sanitation counts"
)

print(
    "-----------------------------------"
)

for seed_index in [
    1,
    2,
    3,
]:

    print(
        f"Seed{seed_index}: "
        f"{recovery_discard_counts[seed_index]} "
        "pre-external-clip degenerate prediction(s) discarded"
    )


print()

print(
    "PRIMARY — official authors' test-250"
)

print(
    "-----------------------------------"
)


for row in flat_rows:

    print(
        f"Seed{row['seed_index']} "
        f"(threshold={float(row['operating_threshold']):.2f})"
    )

    print(
        "  Box AP50:    "
        f"{fmt(row['primary_box_ap50'])}"
    )

    print(
        "  Box AP75:    "
        f"{fmt(row['primary_box_ap75'])}"
    )

    print(
        "  Box AP50:95: "
        f"{fmt(row['primary_box_ap50_95'])}"
    )

    print(
        "  Precision:   "
        f"{fmt(row['primary_precision'])}"
    )

    print(
        "  Recall:      "
        f"{fmt(row['primary_recall'])}"
    )

    print(
        "  F1:          "
        f"{fmt(row['primary_f1'])}"
    )

    print(
        "  Count MAE:   "
        f"{fmt(row['primary_count_mae'])}"
    )

    print(
        "  Count bias:  "
        f"{fmt(row['primary_count_bias'])}"
    )

    print(
        "  Zero-GT FP/image: "
        f"{fmt(row['primary_zero_gt_fp_per_image'])}"
    )

    print()


print(
    "PRIMARY — three-seed mean ± sample SD"
)

print(
    "  Box AP50:    "
    f"{fmt_ms(aggregate_primary['box_ap50'])}"
)

print(
    "  Box AP75:    "
    f"{fmt_ms(aggregate_primary['box_ap75'])}"
)

print(
    "  Box AP50:95: "
    f"{fmt_ms(aggregate_primary['box_ap50_95'])}"
)

print(
    "  Precision:   "
    f"{fmt_ms(aggregate_primary['precision'])}"
)

print(
    "  Recall:      "
    f"{fmt_ms(aggregate_primary['recall'])}"
)

print(
    "  F1:          "
    f"{fmt_ms(aggregate_primary['f1'])}"
)

print(
    "  Count MAE:   "
    f"{fmt_ms(aggregate_primary['count_mae'])}"
)

print(
    "  Count bias:  "
    f"{fmt_ms(aggregate_primary['count_bias'])}"
)

print(
    "  Zero-GT FP/image: "
    f"{fmt_ms(aggregate_primary['zero_gt_fp_per_image'])}"
)

print(
    "  Zero-GT fraction >=1 FP: "
    f"{fmt_ms(aggregate_primary['zero_gt_fraction_with_ge1_fp'])}"
)


primary_conditional = (
    aggregate_primary[
        "mean_matched_box_iou_conditional"
    ]
)

if (
    primary_conditional[
        "n_defined"
    ]
    == 3
):

    print(
        "  Conditional matched box IoU: "
        f"{primary_conditional['mean']:.6f} "
        f"± {primary_conditional['sample_sd']:.6f}"
    )

else:

    print(
        "  Conditional matched box IoU: "
        "undefined for one or more seeds; "
        f"defined n={primary_conditional['n_defined']}/3"
    )


print()

print(
    "SENSITIVITY — exact-dedup-247"
)

print(
    "------------------------------"
)


for row in flat_rows:

    print(
        f"Seed{row['seed_index']} "
        f"(threshold={float(row['operating_threshold']):.2f})"
    )

    print(
        "  Box AP50:    "
        f"{fmt(row['sensitivity_box_ap50'])}"
    )

    print(
        "  Box AP75:    "
        f"{fmt(row['sensitivity_box_ap75'])}"
    )

    print(
        "  Box AP50:95: "
        f"{fmt(row['sensitivity_box_ap50_95'])}"
    )

    print(
        "  Precision:   "
        f"{fmt(row['sensitivity_precision'])}"
    )

    print(
        "  Recall:      "
        f"{fmt(row['sensitivity_recall'])}"
    )

    print(
        "  F1:          "
        f"{fmt(row['sensitivity_f1'])}"
    )

    print()


print(
    "SENSITIVITY — three-seed mean ± sample SD"
)

print(
    "  Box AP50:    "
    f"{fmt_ms(aggregate_sensitivity['box_ap50'])}"
)

print(
    "  Box AP75:    "
    f"{fmt_ms(aggregate_sensitivity['box_ap75'])}"
)

print(
    "  Box AP50:95: "
    f"{fmt_ms(aggregate_sensitivity['box_ap50_95'])}"
)

print(
    "  Precision:   "
    f"{fmt_ms(aggregate_sensitivity['precision'])}"
)

print(
    "  Recall:      "
    f"{fmt_ms(aggregate_sensitivity['recall'])}"
)

print(
    "  F1:          "
    f"{fmt_ms(aggregate_sensitivity['f1'])}"
)


print()

print(
    "Scientific controls"
)

print(
    "-------------------"
)

print(
    "RSC training:         NO"
)

print(
    "RSC CRPA:             NO"
)

print(
    "RSC calibration:      NO"
)

print(
    "RSC threshold tuning: NO"
)

print(
    "Checkpoint selection: NO"
)

print(
    "Seed selection:       NO"
)

print(
    "Best-seed selection:  NO"
)

print(
    "All 3 seeds included: YES"
)

print(
    "Original failed run preserved: YES"
)

print()


print(
    "Recovery execution contract SHA256:"
)

print(
    f"  {contract_sha}"
)

print()

print(
    "Recovery aggregate JSON:"
)

print(
    f"  {AGGREGATE_JSON}"
)

print()

print(
    "Recovery aggregate SHA256:"
)

print(
    f"  {aggregate_sha}"
)

print()

print(
    "Recovery artifact manifest:"
)

print(
    f"  {ARTIFACT_MANIFEST_JSON}"
)

print()

print(
    "Recovery artifact-manifest SHA256:"
)

print(
    f"  {artifact_manifest_sha}"
)

print()

print(
    "Per-seed recovery metrics CSV:"
)

print(
    f"  {PER_SEED_METRICS_CSV}"
)

print()

print(
    "Per-seed recovery metrics CSV SHA256:"
)

print(
    f"  {per_seed_metrics_sha}"
)

print()

print(
    "NEXT READINESS:"
)

print(
    "  READY_TO_PRESERVE_AND_INTERPRET_"
    "FORMAL_RSC_RECOVERY_RESULTS"
)

print("=" * 80)

DASO-Net — FORMAL RSC THREE-SEED RECOVERY EXECUTION v1

Recovery rule:
  finite decoded non-positive-area box -> DISCARD

All three final seeds will rerun from image 1.
Intermediate scientific metrics will remain hidden.

PASS  Recovery implementation-freeze identity
PASS  Frozen recovery evaluator identity
PASS  Frozen scientific protocol identity
PASS  Frozen recovery-policy identity
PASS  Original failed formal execution preserved
PASS  Original Seed-3 consistency references
PASS  All 3 final checkpoint identities
PASS  Frozen thresholds .30/.35/.35
PASS  Recovery output root absent

PASS  Recovery execution contract frozen
Contract SHA256: c9d4f7336a6b5f8ba9c6d7c7e5aec36037e735aa6064ca7eb05f89691511506f

Beginning recovery formal execution.
Scientific metrics remain hidden until all three recovery seeds finish.

START Recovery Seed-1
END   Recovery Seed-1 returncode=0
START Recovery Seed-2
END   Recovery Seed-2 returncode=0
START Recovery Seed-3
END   Recovery Seed-3 returncode=0



In [42]:
# ============================================================
# DASO-Net — FINAL RSC FORMAL RECOVERY RESULTS PRESERVATION
# FREEZE v1
#
# PURPOSE
# -------
# Permanently preserve the completed formal RSC external
# evaluation and its full recovery provenance chain.
#
# THIS CELL:
#   - performs NO model inference
#   - performs NO metric recomputation
#   - performs NO threshold tuning
#   - performs NO calibration
#   - performs NO checkpoint/seed selection
#   - modifies NO existing scientific artifact
#
# It verifies and copies:
#   1. pre-inference asset audit
#   2. frozen scientific protocol
#   3. structural smoke history
#   4. original evaluator implementation freeze
#   5. original failed formal run + partial outputs
#   6. root-cause audit history
#   7. frozen degenerate-box recovery policy
#   8. frozen recovery evaluator implementation
#   9. completed three-seed recovery execution
#  10. completed recovery per-seed scientific outputs
#
# It then:
#   - verifies byte identity of every copied file
#   - writes a file-by-file SHA256 manifest
#   - writes a final preservation record
#   - creates a deterministic portable ZIP
#   - hashes the ZIP
#
# ============================================================

from __future__ import annotations

import hashlib
import json
import shutil
import zipfile
from pathlib import Path


# ============================================================
# 1. SOURCE PROVENANCE DIRECTORIES
# ============================================================

SOURCES = {

    "01_preinference_asset_audit":
        Path(
            "/kaggle/working/dasonet_final_analysis/"
            "rsc_external_preinference_asset_audit_v1"
        ),

    "02_protocol_freeze":
        Path(
            "/kaggle/working/dasonet_final_analysis/"
            "rsc_external_evaluation_protocol_freeze_v1"
        ),

    "03_original_failed_structural_smoke":
        Path(
            "/kaggle/working/dasonet_final_analysis/"
            "rsc_evaluator_v1_smoke_audit"
        ),

    "04_successful_structural_smoke":
        Path(
            "/kaggle/working/dasonet_final_analysis/"
            "rsc_evaluator_v1_smoke_audit_resume01"
        ),

    "05_original_evaluator_implementation_freeze":
        Path(
            "/kaggle/working/dasonet_final_analysis/"
            "rsc_evaluator_implementation_freeze_v1"
        ),

    "06_original_formal_execution":
        Path(
            "/kaggle/working/dasonet_final_analysis/"
            "rsc_formal_three_seed_execution_v1"
        ),

    "07_original_formal_outputs":
        Path(
            "/kaggle/working/dasonet_final_evaluation/"
            "rsc_external_detection_v1"
        ),

    "08_root_cause_audit_failed_v1":
        Path(
            "/kaggle/working/dasonet_final_analysis/"
            "rsc_degenerate_box_root_cause_audit_v1"
        ),

    "09_root_cause_audit_failed_resume01":
        Path(
            "/kaggle/working/dasonet_final_analysis/"
            "rsc_degenerate_box_root_cause_audit_v1_resume01"
        ),

    "10_root_cause_audit_pass_resume02":
        Path(
            "/kaggle/working/dasonet_final_analysis/"
            "rsc_degenerate_box_root_cause_audit_v1_resume02"
        ),

    "11_recovery_policy_freeze":
        Path(
            "/kaggle/working/dasonet_final_analysis/"
            "rsc_degenerate_box_recovery_policy_freeze_v1"
        ),

    "12_recovery_evaluator_implementation_freeze":
        Path(
            "/kaggle/working/dasonet_final_analysis/"
            "rsc_recovery_evaluator_implementation_freeze_v1"
        ),

    "13_recovery_formal_execution":
        Path(
            "/kaggle/working/dasonet_final_analysis/"
            "rsc_formal_three_seed_recovery_execution_v1"
        ),

    "14_recovery_formal_outputs":
        Path(
            "/kaggle/working/dasonet_final_evaluation/"
            "rsc_external_detection_v1_recovery01"
        ),
}


# ============================================================
# 2. FINAL FREEZE DESTINATIONS
# ============================================================

FREEZE_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_formal_recovery_results_freeze_v1"
)

COPIED_ROOT = (
    FREEZE_DIR
    / "preserved_artifacts"
)

FILE_MANIFEST_JSON = (
    FREEZE_DIR
    / "RSC_FORMAL_RECOVERY_RESULTS_FILE_MANIFEST_v1.json"
)

FILE_MANIFEST_SHA_RECORD = (
    FREEZE_DIR
    / "RSC_FORMAL_RECOVERY_RESULTS_FILE_MANIFEST_v1.sha256.txt"
)

FINAL_RECORD_JSON = (
    FREEZE_DIR
    / "RSC_FORMAL_RECOVERY_RESULTS_FREEZE_v1.json"
)

FINAL_RECORD_SHA_RECORD = (
    FREEZE_DIR
    / "RSC_FORMAL_RECOVERY_RESULTS_FREEZE_v1.sha256.txt"
)

PORTABLE_ZIP = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "RSC_FORMAL_RECOVERY_RESULTS_FREEZE_v1.zip"
)

PORTABLE_ZIP_SHA_RECORD = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "RSC_FORMAL_RECOVERY_RESULTS_FREEZE_v1.zip.sha256.txt"
)


# ============================================================
# 3. KEY SCIENTIFIC ARTIFACTS / EXPECTED IDENTITIES
# ============================================================

KEY_ARTIFACTS = {

    # --------------------------------------------------------
    # Pre-inference provenance
    # --------------------------------------------------------

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "rsc_external_preinference_asset_audit_v1/"
        "RSC_EXTERNAL_PREINFERENCE_ASSET_AUDIT_v1.json"
    ):
        (
            "c8eb566baa55e1c93906e61dbe4d9228"
            "246c7d130aac489b07102c51c610132b"
        ),

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "rsc_external_evaluation_protocol_freeze_v1/"
        "RSC_EXTERNAL_EVALUATION_PROTOCOL_FREEZE_v1.json"
    ):
        (
            "e3c87caebafe87c7671b5745a480c0ed"
            "f67f444feac4278633da6addae4d1f7e"
        ),

    # --------------------------------------------------------
    # Original evaluator
    # --------------------------------------------------------

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "rsc_evaluator_implementation_freeze_v1/"
        "evaluate_rsc_external_v1_FROZEN.py"
    ):
        (
            "59a3de31f2a5244ba32c53b705b050e0"
            "33b5d17c8de7efb36c17dbc49f0a7723"
        ),

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "rsc_evaluator_implementation_freeze_v1/"
        "RSC_EVALUATOR_IMPLEMENTATION_FREEZE_v1.json"
    ):
        (
            "60163937843aed8c893e318bfae19aa0"
            "bb9ebbcb98e461a39a9b05c8fb1b9f64"
        ),

    # --------------------------------------------------------
    # Original formal execution
    # --------------------------------------------------------

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "rsc_formal_three_seed_execution_v1/"
        "RSC_FORMAL_EXECUTION_CONTRACT_v1.json"
    ):
        (
            "ec433786bececd29c787155bc116e664"
            "1efb36aafaff46ac69898c6823d4873d"
        ),

    # --------------------------------------------------------
    # Original successful Seed-3 references
    # --------------------------------------------------------

    Path(
        "/kaggle/working/dasonet_final_evaluation/"
        "rsc_external_detection_v1/"
        "seed03_s252916394/"
        "RSC_EXTERNAL_DETECTION_SUMMARY.json"
    ):
        (
            "332334dcce91ffb0f54d9bc6875d04a2"
            "0f07e7d0f0e137c45e444a71c9fe4577"
        ),

    Path(
        "/kaggle/working/dasonet_final_evaluation/"
        "rsc_external_detection_v1/"
        "seed03_s252916394/"
        "rsc_predictions_decoder_floor.jsonl"
    ):
        (
            "7d2e53aa90dec26497f6e96f811af356"
            "4257813e8b38ddf24ba879d951f61f1b"
        ),

    Path(
        "/kaggle/working/dasonet_final_evaluation/"
        "rsc_external_detection_v1/"
        "seed03_s252916394/"
        "rsc_primary250_per_image.csv"
    ):
        (
            "6dca71a9761e10b2611521f0d3738e6"
            "c2a522dfa7504a4a59712fb4fcb417d3a"
        ),

    Path(
        "/kaggle/working/dasonet_final_evaluation/"
        "rsc_external_detection_v1/"
        "seed03_s252916394/"
        "rsc_dedup247_per_image.csv"
    ):
        (
            "003845cf1d20ebad63c106a834c3821b"
            "261a3ec8280ecf88e6e6d844797764f3"
        ),

    # --------------------------------------------------------
    # Root cause / recovery policy
    # --------------------------------------------------------

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "rsc_degenerate_box_root_cause_audit_v1_resume02/"
        "RSC_DEGENERATE_BOX_ROOT_CAUSE_AUDIT_RESUME02.json"
    ):
        (
            "8c150f118b193d2272a15387ccca74810"
            "dda0c71e4601a80ce6516b1128f078b"
        ),

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "rsc_degenerate_box_recovery_policy_freeze_v1/"
        "RSC_DEGENERATE_BOX_RECOVERY_POLICY_FREEZE_v1.json"
    ):
        (
            "aa754df77f897c68421a065ffa0b0ffc"
            "fe6b332b8dba96b6415d089cb2216f8b"
        ),

    # --------------------------------------------------------
    # Recovery evaluator
    # --------------------------------------------------------

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "rsc_recovery_evaluator_implementation_freeze_v1/"
        "evaluate_rsc_external_recovery_v1_FROZEN.py"
    ):
        (
            "d0df4ad7cdd60aae23fdff7d2c708c93"
            "8952510df4a8d9d7a9087eecbad85bda"
        ),

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "rsc_recovery_evaluator_implementation_freeze_v1/"
        "RSC_RECOVERY_EVALUATOR_IMPLEMENTATION_FREEZE_v1.json"
    ):
        (
            "dacca485aca56ce8f6c397cff30b41742"
            "e869d21a0ccaa6424e5b2890cec62e8"
        ),

    # --------------------------------------------------------
    # Final recovery execution
    # --------------------------------------------------------

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "rsc_formal_three_seed_recovery_execution_v1/"
        "RSC_FORMAL_RECOVERY_EXECUTION_CONTRACT_v1.json"
    ):
        (
            "c9d4f7336a6b5f8ba9c6d7c7e5aec360"
            "37e735aa6064ca7eb05f89691511506f"
        ),

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "rsc_formal_three_seed_recovery_execution_v1/"
        "RSC_FORMAL_RECOVERY_THREE_SEED_AGGREGATE_v1.json"
    ):
        (
            "ae63f7d86bb228bb5991678ef8706f9f"
            "aa9653d59f9664acfa0a72de49e210c0"
        ),

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "rsc_formal_three_seed_recovery_execution_v1/"
        "RSC_FORMAL_RECOVERY_ARTIFACT_MANIFEST_v1.json"
    ):
        (
            "217ac6eea153696d490aa591c157135ab"
            "fed9f9efbd12643ede483b569d5a745"
        ),

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "rsc_formal_three_seed_recovery_execution_v1/"
        "RSC_FORMAL_RECOVERY_THREE_SEED_METRICS_v1.csv"
    ):
        (
            "b396264cbeb73074144c85524203899ea"
            "df943f2e010c3d72ed5aa4165a9f36e"
        ),
}


# ============================================================
# 4. FINAL RESULT PATHS
# ============================================================

RECOVERY_EXECUTION_DIR = SOURCES[
    "13_recovery_formal_execution"
]

RECOVERY_OUTPUT_ROOT = SOURCES[
    "14_recovery_formal_outputs"
]

RECOVERY_EXECUTION_STATE = (
    RECOVERY_EXECUTION_DIR
    / "RSC_FORMAL_RECOVERY_EXECUTION_STATE_v1.json"
)

RECOVERY_AGGREGATE = (
    RECOVERY_EXECUTION_DIR
    / "RSC_FORMAL_RECOVERY_THREE_SEED_AGGREGATE_v1.json"
)

RECOVERY_ARTIFACT_MANIFEST = (
    RECOVERY_EXECUTION_DIR
    / "RSC_FORMAL_RECOVERY_ARTIFACT_MANIFEST_v1.json"
)

RECOVERY_PER_SEED_CSV = (
    RECOVERY_EXECUTION_DIR
    / "RSC_FORMAL_RECOVERY_THREE_SEED_METRICS_v1.csv"
)


# ============================================================
# 5. HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def require(
    condition: bool,
    message: str,
) -> None:
    if not condition:
        raise RuntimeError(
            message
        )


def require_file(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_file(),
        f"{label} missing:\n{path}",
    )


def require_dir(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_dir(),
        f"{label} missing:\n{path}",
    )


def read_json(
    path: Path,
):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path: Path,
    obj,
) -> None:
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def all_files(
    root: Path,
):
    return sorted(
        [
            path
            for path
            in root.rglob("*")
            if path.is_file()
        ],
        key=lambda path:
            path.as_posix(),
    )


def fmt_ms(
    metric,
) -> str:
    return (
        f"{float(metric['mean']):.6f} "
        f"± {float(metric['sample_sd']):.6f}"
    )


# ============================================================
# 6. BANNER / NON-OVERWRITE
# ============================================================

print("=" * 80)

print(
    "DASO-Net — FINAL RSC FORMAL RECOVERY "
    "RESULTS PRESERVATION FREEZE v1"
)

print("=" * 80)

print()

print(
    "THIS CELL PERFORMS NO MODEL INFERENCE."
)

print(
    "THIS CELL PERFORMS NO SCIENTIFIC METRIC RECOMPUTATION."
)

print()

print(
    "Scientific result being preserved:"
)

print(
    "  PRIMARY = official authors' RSC test-250"
)

print(
    "  SENSITIVITY = exact-dedup-247"
)

print(
    "  TASK = class-agnostic bounding-box detection"
)

print()


if FREEZE_DIR.exists():
    raise RuntimeError(
        "Final RSC preservation freeze already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{FREEZE_DIR}"
    )


if PORTABLE_ZIP.exists():
    raise RuntimeError(
        "Portable RSC preservation ZIP already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{PORTABLE_ZIP}"
    )


if PORTABLE_ZIP_SHA_RECORD.exists():
    raise RuntimeError(
        "Portable ZIP SHA record already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{PORTABLE_ZIP_SHA_RECORD}"
    )


# ============================================================
# 7. VERIFY ALL SOURCE DIRECTORIES EXIST
# ============================================================

for label, path in SOURCES.items():

    require_dir(
        path,
        label,
    )


print(
    "PASS  Complete RSC provenance directory chain present"
)


# ============================================================
# 8. VERIFY KEY ARTIFACT HASHES
# ============================================================

for path, expected_sha in KEY_ARTIFACTS.items():

    require_file(
        path,
        "Key frozen RSC artifact",
    )

    actual_sha = sha256_file(
        path
    )

    require(
        actual_sha
        == expected_sha,
        (
            "Key RSC artifact SHA mismatch.\n"
            f"Path:     {path}\n"
            f"Expected: {expected_sha}\n"
            f"Actual:   {actual_sha}"
        ),
    )


print(
    "PASS  All key frozen RSC artifact identities"
)


# ============================================================
# 9. VERIFY FINAL EXECUTION STATE
# ============================================================

require_file(
    RECOVERY_EXECUTION_STATE,
    "Final recovery execution state",
)

execution_state = read_json(
    RECOVERY_EXECUTION_STATE
)


require(
    execution_state.get(
        "status"
    )
    == "PASS",
    "Final recovery execution state != PASS.",
)

require(
    execution_state.get(
        "all_three_recovery_seeds_completed"
    )
    is True,
    "Final state does not confirm all 3 recovery seeds.",
)

require(
    execution_state.get(
        "seed3_consistency_gate"
    )
    == "PASS",
    "Final state does not confirm Seed-3 consistency gate.",
)

require(
    execution_state.get(
        "aggregate_produced"
    )
    is True,
    "Final state does not confirm aggregate production.",
)

require(
    execution_state.get(
        "intermediate_scientific_metrics_displayed"
    )
    is False,
    "Final state unexpectedly reports intermediate metrics.",
)

require(
    execution_state.get(
        "rsc_threshold_tuning"
    )
    is False,
    "Final state unexpectedly reports RSC threshold tuning.",
)

require(
    execution_state.get(
        "rsc_calibration"
    )
    is False,
    "Final state unexpectedly reports RSC calibration.",
)

require(
    execution_state.get(
        "checkpoint_selection"
    )
    is False,
    "Final state unexpectedly reports checkpoint selection.",
)

require(
    execution_state.get(
        "seed_selection"
    )
    is False,
    "Final state unexpectedly reports seed selection.",
)


print(
    "PASS  Final three-seed recovery execution state"
)


# ============================================================
# 10. VERIFY FINAL AGGREGATE
# ============================================================

require_file(
    RECOVERY_AGGREGATE,
    "Final recovery aggregate",
)

aggregate = read_json(
    RECOVERY_AGGREGATE
)


require(
    aggregate.get(
        "status"
    )
    == "PASS",
    "Final recovery aggregate != PASS.",
)

require(
    aggregate.get(
        "scientific_task"
    )
    == "class-agnostic bounding-box detection",
    "Final scientific task changed.",
)

require(
    aggregate.get(
        "segmentation_metrics"
    )
    is False,
    "RSC aggregate unexpectedly reports segmentation metrics.",
)


for key, expected in {
    "model": False,
    "decoder": False,
    "checkpoints": False,
    "seeds": False,
    "thresholds": False,
    "metrics": False,
    "dataset_membership": False,
    "preprocessing": False,
}.items():

    require(
        aggregate[
            "scientific_components_changed"
        ][
            key
        ]
        is expected,
        (
            "Unexpected final scientific component-change "
            f"flag: {key}"
        ),
    )


require(
    aggregate[
        "seed3_consistency_gate"
    ][
        "status"
    ]
    == "PASS",
    "Aggregate Seed-3 consistency gate != PASS.",
)


require(
    aggregate[
        "recovery_geometry"
    ][
        "seed1_preexternalclip_degenerate_discarded"
    ]
    == 1,
    "Final Seed-1 recovery sanitation count changed.",
)

require(
    aggregate[
        "recovery_geometry"
    ][
        "seed2_preexternalclip_degenerate_discarded"
    ]
    == 2,
    "Final Seed-2 recovery sanitation count changed.",
)

require(
    aggregate[
        "recovery_geometry"
    ][
        "seed3_preexternalclip_degenerate_discarded"
    ]
    == 0,
    "Final Seed-3 recovery sanitation count changed.",
)


for key in [
    "rsc_training",
    "rsc_crpa",
    "rsc_calibration",
    "rsc_threshold_tuning",
    "checkpoint_selection",
    "seed_selection",
    "best_seed_selection",
]:

    require(
        aggregate[
            "scientific_policy"
        ][
            key
        ]
        is False,
        (
            "Unexpected final scientific-policy flag: "
            f"{key}"
        ),
    )


require(
    aggregate[
        "scientific_policy"
    ][
        "all_three_final_seeds_included"
    ]
    is True,
    "Final aggregate does not include all three final seeds.",
)


print(
    "PASS  Final recovery aggregate scientific controls"
)


# ============================================================
# 11. VERIFY DISPLAYED RESULT VALUES
#
# This is validation against the already-computed aggregate,
# NOT metric recomputation.
# ============================================================

primary = (
    aggregate[
        "primary_official_test250"
    ]
)

sensitivity = (
    aggregate[
        "sensitivity_exact_dedup247"
    ]
)


expected_rounded_primary = {
    "box_ap50":
        (
            0.030172,
            0.037769,
        ),

    "box_ap75":
        (
            0.001027,
            0.001145,
        ),

    "box_ap50_95":
        (
            0.005761,
            0.007662,
        ),

    "precision":
        (
            0.081214,
            0.123416,
        ),

    "recall":
        (
            0.061224,
            0.104441,
        ),

    "f1":
        (
            0.067937,
            0.114737,
        ),

    "count_mae":
        (
            1.629333,
            0.745218,
        ),

    "count_bias":
        (
            -1.504000,
            0.960900,
        ),

    "zero_gt_fp_per_image":
        (
            0.000000,
            0.000000,
        ),
}


for metric_name, (
    expected_mean,
    expected_sd,
) in expected_rounded_primary.items():

    require(
        round(
            float(
                primary[
                    metric_name
                ][
                    "mean"
                ]
            ),
            6,
        )
        == expected_mean,
        (
            "Displayed PRIMARY aggregate mean changed: "
            f"{metric_name}"
        ),
    )

    require(
        round(
            float(
                primary[
                    metric_name
                ][
                    "sample_sd"
                ]
            ),
            6,
        )
        == expected_sd,
        (
            "Displayed PRIMARY aggregate sample SD changed: "
            f"{metric_name}"
        ),
    )


for metric_name in [
    "box_ap50",
    "box_ap75",
    "box_ap50_95",
    "precision",
    "recall",
    "f1",
]:

    require(
        primary[
            metric_name
        ]
        == sensitivity[
            metric_name
        ],
        (
            "PRIMARY/SENSITIVITY aggregate mismatch "
            f"for preserved observed metric: {metric_name}"
        ),
    )


print(
    "PASS  Final displayed aggregate values"
)


# ============================================================
# 12. VERIFY ALL THREE RECOVERY SEED OUTPUTS
# ============================================================

EXPECTED_SEEDS = {
    1: 1782656963,
    2: 901395089,
    3: 252916394,
}


for seed_index, seed in EXPECTED_SEEDS.items():

    seed_dir = (
        RECOVERY_OUTPUT_ROOT
        / (
            f"seed{seed_index:02d}_"
            f"s{seed}"
        )
    )

    require_dir(
        seed_dir,
        f"Recovery Seed-{seed_index}",
    )


    run_state_path = (
        seed_dir
        / "RSC_RUN_STATE.json"
    )

    summary_path = (
        seed_dir
        / "RSC_EXTERNAL_DETECTION_SUMMARY.json"
    )

    predictions_path = (
        seed_dir
        / "rsc_predictions_decoder_floor.jsonl"
    )

    primary_csv_path = (
        seed_dir
        / "rsc_primary250_per_image.csv"
    )

    dedup_csv_path = (
        seed_dir
        / "rsc_dedup247_per_image.csv"
    )

    diagnostics_path = (
        seed_dir
        / "RSC_RECOVERY_GEOMETRY_DIAGNOSTICS.json"
    )

    provenance_path = (
        seed_dir
        / "RSC_RECOVERY_PROVENANCE.json"
    )


    for path in [
        run_state_path,
        summary_path,
        predictions_path,
        primary_csv_path,
        dedup_csv_path,
        diagnostics_path,
        provenance_path,
    ]:

        require_file(
            path,
            f"Recovery Seed-{seed_index} artifact",
        )


    run_state = read_json(
        run_state_path
    )

    require(
        run_state.get(
            "status"
        )
        == "PASS",
        f"Recovery Seed-{seed_index} state != PASS.",
    )

    require(
        int(
            run_state[
                "images_completed"
            ]
        )
        == 250,
        (
            f"Recovery Seed-{seed_index} "
            "images_completed != 250."
        ),
    )


    diagnostics = read_json(
        diagnostics_path
    )

    require(
        diagnostics.get(
            "status"
        )
        == "PASS",
        f"Recovery Seed-{seed_index} diagnostics != PASS.",
    )

    require(
        int(
            diagnostics[
                "images_with_coordinate_policy_call"
            ]
        )
        == 250,
        (
            f"Recovery Seed-{seed_index} "
            "diagnostic image closure != 250."
        ),
    )


print(
    "PASS  All three recovery per-seed output closures"
)


# ============================================================
# 13. CREATE FINAL FREEZE DIRECTORY
# ============================================================

FREEZE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

COPIED_ROOT.mkdir(
    parents=False,
    exist_ok=False,
)


# ============================================================
# 14. COPY COMPLETE PROVENANCE CHAIN
# ============================================================

for label, source in SOURCES.items():

    destination = (
        COPIED_ROOT
        / label
    )

    shutil.copytree(
        source,
        destination,
        symlinks=False,
    )


print(
    "PASS  Complete RSC provenance chain copied"
)


# ============================================================
# 15. BYTE-FOR-BYTE COPY VERIFICATION
# ============================================================

manifest_entries = []

total_files = 0

total_bytes = 0


for label, source_root in SOURCES.items():

    copied_root = (
        COPIED_ROOT
        / label
    )

    source_files = all_files(
        source_root
    )

    copied_files = all_files(
        copied_root
    )


    source_relatives = [
        path.relative_to(
            source_root
        ).as_posix()
        for path
        in source_files
    ]

    copied_relatives = [
        path.relative_to(
            copied_root
        ).as_posix()
        for path
        in copied_files
    ]


    require(
        source_relatives
        == copied_relatives,
        (
            "Source/copy file inventory mismatch.\n"
            f"Source group: {label}"
        ),
    )


    for source_file in source_files:

        relative = source_file.relative_to(
            source_root
        )

        copied_file = (
            copied_root
            / relative
        )

        source_sha = sha256_file(
            source_file
        )

        copied_sha = sha256_file(
            copied_file
        )

        require(
            source_sha
            == copied_sha,
            (
                "Copied artifact SHA mismatch.\n"
                f"Source: {source_file}\n"
                f"Copy:   {copied_file}"
            ),
        )

        size_bytes = source_file.stat().st_size

        total_files += 1

        total_bytes += size_bytes

        manifest_entries.append(
            {
                "source_group":
                    label,

                "source_path":
                    str(
                        source_file
                    ),

                "bundle_relative_path":
                    (
                        Path(
                            "preserved_artifacts"
                        )
                        / label
                        / relative
                    ).as_posix(),

                "size_bytes":
                    int(
                        size_bytes
                    ),

                "sha256":
                    source_sha,

                "copy_sha256":
                    copied_sha,

                "byte_identity":
                    True,
            }
        )


print(
    "PASS  File-by-file source/copy byte identity"
)


# ============================================================
# 16. WRITE FILE MANIFEST
# ============================================================

file_manifest = {
    "manifest_name":
        (
            "DASO-Net RSC Formal Recovery Results "
            "File Manifest v1"
        ),

    "status":
        "PASS",

    "scientific_action":
        "PRESERVATION_ONLY",

    "model_inference":
        False,

    "metric_recomputation":
        False,

    "source_group_count":
        len(
            SOURCES
        ),

    "file_count":
        total_files,

    "total_source_bytes":
        total_bytes,

    "all_source_copy_sha256_pairs_match":
        True,

    "files":
        manifest_entries,
}


write_json(
    FILE_MANIFEST_JSON,
    file_manifest,
)

file_manifest_sha = sha256_file(
    FILE_MANIFEST_JSON
)

FILE_MANIFEST_SHA_RECORD.write_text(
    (
        f"{file_manifest_sha}  "
        f"{FILE_MANIFEST_JSON.name}\n"
    ),
    encoding="utf-8",
)


print(
    "PASS  Final file manifest"
)


# ============================================================
# 17. FINAL PRESERVATION RECORD
# ============================================================

final_record = {
    "freeze_name":
        (
            "DASO-Net Formal RSC Recovery "
            "Results Preservation Freeze v1"
        ),

    "status":
        "FINAL_RSC_FORMAL_RECOVERY_RESULTS_PRESERVED",

    "scientific_action":
        "PRESERVATION_ONLY",

    "model_inference_performed":
        False,

    "scientific_metrics_recomputed":
        False,

    "formal_task":
        "class-agnostic bounding-box detection",

    "primary_set":
        {
            "name":
                "official authors' RSC test-250",

            "images":
                250,

            "gt_boxes":
                539,

            "zero_gt_images":
                4,
        },

    "sensitivity_set":
        {
            "name":
                "exact-dedup-247",

            "images":
                247,

            "gt_boxes":
                539,

            "zero_gt_images":
                1,
        },

    "formal_result":
        {
            "primary_box_ap50":
                primary[
                    "box_ap50"
                ],

            "primary_box_ap75":
                primary[
                    "box_ap75"
                ],

            "primary_box_ap50_95":
                primary[
                    "box_ap50_95"
                ],

            "primary_precision":
                primary[
                    "precision"
                ],

            "primary_recall":
                primary[
                    "recall"
                ],

            "primary_f1":
                primary[
                    "f1"
                ],

            "primary_count_mae":
                primary[
                    "count_mae"
                ],

            "primary_count_bias":
                primary[
                    "count_bias"
                ],

            "primary_zero_gt_fp_per_image":
                primary[
                    "zero_gt_fp_per_image"
                ],

            "primary_zero_gt_fraction_with_ge1_fp":
                primary[
                    "zero_gt_fraction_with_ge1_fp"
                ],

            "primary_conditional_matched_box_iou":
                primary[
                    "mean_matched_box_iou_conditional"
                ],

            "sensitivity_box_ap50":
                sensitivity[
                    "box_ap50"
                ],

            "sensitivity_box_ap75":
                sensitivity[
                    "box_ap75"
                ],

            "sensitivity_box_ap50_95":
                sensitivity[
                    "box_ap50_95"
                ],

            "sensitivity_precision":
                sensitivity[
                    "precision"
                ],

            "sensitivity_recall":
                sensitivity[
                    "recall"
                ],

            "sensitivity_f1":
                sensitivity[
                    "f1"
                ],
        },

    "recovery_geometry":
        aggregate[
            "recovery_geometry"
        ],

    "seed3_consistency_gate":
        aggregate[
            "seed3_consistency_gate"
        ],

    "scientific_controls":
        {
            "rsc_training":
                False,

            "rsc_crpa":
                False,

            "rsc_calibration":
                False,

            "rsc_threshold_tuning":
                False,

            "checkpoint_selection":
                False,

            "seed_selection":
                False,

            "best_seed_selection":
                False,

            "all_three_final_seeds_included":
                True,
        },

    "recovery_scope":
        {
            "reason":
                (
                    "Original external-evaluation wrapper "
                    "treated finite non-positive-area decoded "
                    "geometry as fatal."
                ),

            "rule":
                (
                    "finite decoded prediction with "
                    "x2 <= x1 OR y2 <= y1 -> discard"
                ),

            "seed1_discard_count":
                1,

            "seed2_discard_count":
                2,

            "seed3_discard_count":
                0,

            "model_changed":
                False,

            "validated_decoder_changed":
                False,

            "thresholds_changed":
                False,

            "metric_definitions_changed":
                False,

            "dataset_membership_changed":
                False,
        },

    "key_sha256":
        {
            "scientific_protocol":
                (
                    "e3c87caebafe87c7671b5745a480c0ed"
                    "f67f444feac4278633da6addae4d1f7e"
                ),

            "original_evaluator":
                (
                    "59a3de31f2a5244ba32c53b705b050e0"
                    "33b5d17c8de7efb36c17dbc49f0a7723"
                ),

            "recovery_policy":
                (
                    "aa754df77f897c68421a065ffa0b0ffc"
                    "fe6b332b8dba96b6415d089cb2216f8b"
                ),

            "recovery_evaluator":
                (
                    "d0df4ad7cdd60aae23fdff7d2c708c93"
                    "8952510df4a8d9d7a9087eecbad85bda"
                ),

            "recovery_evaluator_implementation_freeze":
                (
                    "dacca485aca56ce8f6c397cff30b41742"
                    "e869d21a0ccaa6424e5b2890cec62e8"
                ),

            "recovery_execution_contract":
                (
                    "c9d4f7336a6b5f8ba9c6d7c7e5aec360"
                    "37e735aa6064ca7eb05f89691511506f"
                ),

            "recovery_aggregate":
                (
                    "ae63f7d86bb228bb5991678ef8706f9f"
                    "aa9653d59f9664acfa0a72de49e210c0"
                ),

            "recovery_artifact_manifest":
                (
                    "217ac6eea153696d490aa591c157135ab"
                    "fed9f9efbd12643ede483b569d5a745"
                ),

            "recovery_per_seed_metrics_csv":
                (
                    "b396264cbeb73074144c85524203899ea"
                    "df943f2e010c3d72ed5aa4165a9f36e"
                ),

            "preservation_file_manifest":
                file_manifest_sha,
        },

    "preservation":
        {
            "freeze_directory":
                str(
                    FREEZE_DIR
                ),

            "preserved_artifacts_directory":
                str(
                    COPIED_ROOT
                ),

            "source_group_count":
                len(
                    SOURCES
                ),

            "file_count":
                total_files,

            "total_bytes":
                total_bytes,

            "all_files_byte_verified":
                True,

            "file_manifest":
                str(
                    FILE_MANIFEST_JSON
                ),

            "file_manifest_sha256":
                file_manifest_sha,
        },

    "reporting_interpretation":
        {
            "primary_external_result":
                (
                    "limited zero-shot cross-domain "
                    "bounding-box detection performance"
                ),

            "seed_variability":
                "substantial",

            "seed_selection_allowed":
                False,

            "seed3_operating_true_positive_note":
                (
                    "Seed-3 had no operating-threshold true "
                    "positives; conditional matched-box IoU "
                    "therefore is undefined for that seed."
                ),

            "claim_scope":
                (
                    "RSC mobile-microscopy external-domain "
                    "detection only; not direct fresh-produce "
                    "surface deployment performance."
                ),
        },

    "readiness":
        (
            "READY_FOR_RSC_REPORTING_AND_"
            "NEXT_EXTERNAL_EVALUATION"
        ),
}


write_json(
    FINAL_RECORD_JSON,
    final_record,
)

final_record_sha = sha256_file(
    FINAL_RECORD_JSON
)

FINAL_RECORD_SHA_RECORD.write_text(
    (
        f"{final_record_sha}  "
        f"{FINAL_RECORD_JSON.name}\n"
    ),
    encoding="utf-8",
)


print(
    "PASS  Final RSC preservation record"
)


# ============================================================
# 18. READ-BACK FINAL FREEZE
# ============================================================

readback = read_json(
    FINAL_RECORD_JSON
)


require(
    readback.get(
        "status"
    )
    == "FINAL_RSC_FORMAL_RECOVERY_RESULTS_PRESERVED",
    "Final preservation status read-back failure.",
)

require(
    readback.get(
        "readiness"
    )
    == (
        "READY_FOR_RSC_REPORTING_AND_"
        "NEXT_EXTERNAL_EVALUATION"
    ),
    "Final preservation readiness read-back failure.",
)

require(
    readback[
        "model_inference_performed"
    ]
    is False,
    "Preservation record unexpectedly claims inference.",
)

require(
    readback[
        "scientific_metrics_recomputed"
    ]
    is False,
    "Preservation record unexpectedly claims metric recomputation.",
)

require(
    readback[
        "preservation"
    ][
        "all_files_byte_verified"
    ]
    is True,
    "Preservation record does not confirm byte verification.",
)

require(
    sha256_file(
        FINAL_RECORD_JSON
    )
    == final_record_sha,
    "Final preservation record changed during read-back.",
)


print(
    "PASS  Final preservation read-back"
)


# ============================================================
# 19. DETERMINISTIC PORTABLE ZIP
#
# Fixed timestamps + sorted paths provide a reproducible archive
# for this exact freeze-directory content.
# ============================================================

bundle_files = all_files(
    FREEZE_DIR
)


with zipfile.ZipFile(
    PORTABLE_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
) as zf:

    for file_path in bundle_files:

        relative = file_path.relative_to(
            FREEZE_DIR
        )

        archive_name = (
            Path(
                FREEZE_DIR.name
            )
            / relative
        ).as_posix()

        data = file_path.read_bytes()

        info = zipfile.ZipInfo(
            filename=archive_name,
            date_time=(
                1980,
                1,
                1,
                0,
                0,
                0,
            ),
        )

        info.compress_type = (
            zipfile.ZIP_DEFLATED
        )

        info.create_system = 3

        info.external_attr = (
            0o100644
            << 16
        )

        zf.writestr(
            info,
            data,
            compress_type=zipfile.ZIP_DEFLATED,
            compresslevel=9,
        )


require_file(
    PORTABLE_ZIP,
    "Portable final RSC preservation ZIP",
)

portable_zip_sha = sha256_file(
    PORTABLE_ZIP
)

PORTABLE_ZIP_SHA_RECORD.write_text(
    (
        f"{portable_zip_sha}  "
        f"{PORTABLE_ZIP.name}\n"
    ),
    encoding="utf-8",
)


print(
    "PASS  Deterministic portable RSC preservation ZIP"
)


# ============================================================
# 20. VERIFY ZIP CONTENT INVENTORY
# ============================================================

expected_zip_names = [
    (
        Path(
            FREEZE_DIR.name
        )
        / path.relative_to(
            FREEZE_DIR
        )
    ).as_posix()
    for path in bundle_files
]


with zipfile.ZipFile(
    PORTABLE_ZIP,
    mode="r",
) as zf:

    actual_zip_names = (
        zf.namelist()
    )

    require(
        actual_zip_names
        == expected_zip_names,
        "Portable ZIP inventory differs from freeze directory.",
    )

    for file_path, archive_name in zip(
        bundle_files,
        actual_zip_names,
    ):

        archived_bytes = zf.read(
            archive_name
        )

        original_bytes = file_path.read_bytes()

        require(
            archived_bytes
            == original_bytes,
            (
                "Portable ZIP member byte mismatch:\n"
                f"{archive_name}"
            ),
        )


print(
    "PASS  Portable ZIP member byte identity"
)


# ============================================================
# 21. FINAL REPORT
# ============================================================

print()

print("=" * 80)

print(
    "DASO-NET FORMAL RSC EXTERNAL EVALUATION: "
    "FINAL RESULTS PRESERVED"
)

print("=" * 80)

print()

print(
    "Status:"
)

print(
    "  FINAL_RSC_FORMAL_RECOVERY_RESULTS_PRESERVED"
)

print()

print(
    "Scientific task:"
)

print(
    "  class-agnostic bounding-box detection"
)

print()

print(
    "PRIMARY — official authors' RSC test-250"
)

print(
    "  Box AP50:    "
    f"{fmt_ms(primary['box_ap50'])}"
)

print(
    "  Box AP75:    "
    f"{fmt_ms(primary['box_ap75'])}"
)

print(
    "  Box AP50:95: "
    f"{fmt_ms(primary['box_ap50_95'])}"
)

print(
    "  Precision:   "
    f"{fmt_ms(primary['precision'])}"
)

print(
    "  Recall:      "
    f"{fmt_ms(primary['recall'])}"
)

print(
    "  F1:          "
    f"{fmt_ms(primary['f1'])}"
)

print(
    "  Count MAE:   "
    f"{fmt_ms(primary['count_mae'])}"
)

print(
    "  Count bias:  "
    f"{fmt_ms(primary['count_bias'])}"
)

print(
    "  Zero-GT FP/image: "
    f"{fmt_ms(primary['zero_gt_fp_per_image'])}"
)

print()

print(
    "Recovery sanitation:"
)

print(
    "  Seed1: 1 finite non-positive-area prediction discarded"
)

print(
    "  Seed2: 2 finite non-positive-area predictions discarded"
)

print(
    "  Seed3: 0 discarded"
)

print()

print(
    "Seed-3 consistency gate:"
)

print(
    "  PASS"
)

print(
    "  predictions byte-identical"
)

print(
    "  primary per-image table byte-identical"
)

print(
    "  sensitivity per-image table byte-identical"
)

print(
    "  scientific metric values identical"
)

print()

print(
    "Scientific controls:"
)

print(
    "  RSC training:         NO"
)

print(
    "  RSC CRPA:             NO"
)

print(
    "  RSC calibration:      NO"
)

print(
    "  RSC threshold tuning: NO"
)

print(
    "  Checkpoint selection: NO"
)

print(
    "  Seed selection:       NO"
)

print(
    "  Best-seed selection:  NO"
)

print(
    "  All 3 final seeds:    YES"
)

print()

print(
    "Preserved source groups:"
)

print(
    f"  {len(SOURCES)}"
)

print()

print(
    "Preserved files:"
)

print(
    f"  {total_files}"
)

print()

print(
    "Preserved bytes:"
)

print(
    f"  {total_bytes}"
)

print()

print(
    "File manifest:"
)

print(
    f"  {FILE_MANIFEST_JSON}"
)

print()

print(
    "File manifest SHA256:"
)

print(
    f"  {file_manifest_sha}"
)

print()

print(
    "Final preservation record:"
)

print(
    f"  {FINAL_RECORD_JSON}"
)

print()

print(
    "Final preservation-record SHA256:"
)

print(
    f"  {final_record_sha}"
)

print()

print(
    "Portable preservation ZIP:"
)

print(
    f"  {PORTABLE_ZIP}"
)

print()

print(
    "Portable ZIP SHA256:"
)

print(
    f"  {portable_zip_sha}"
)

print()

print(
    "Portable ZIP SHA record:"
)

print(
    f"  {PORTABLE_ZIP_SHA_RECORD}"
)

print()

print(
    "NEXT READINESS:"
)

print(
    "  READY_FOR_RSC_REPORTING_AND_NEXT_EXTERNAL_EVALUATION"
)

print("=" * 80)

DASO-Net — FINAL RSC FORMAL RECOVERY RESULTS PRESERVATION FREEZE v1

THIS CELL PERFORMS NO MODEL INFERENCE.
THIS CELL PERFORMS NO SCIENTIFIC METRIC RECOMPUTATION.

Scientific result being preserved:
  PRIMARY = official authors' RSC test-250
  SENSITIVITY = exact-dedup-247
  TASK = class-agnostic bounding-box detection

PASS  Complete RSC provenance directory chain present
PASS  All key frozen RSC artifact identities
PASS  Final three-seed recovery execution state
PASS  Final recovery aggregate scientific controls
PASS  Final displayed aggregate values
PASS  All three recovery per-seed output closures
PASS  Complete RSC provenance chain copied
PASS  File-by-file source/copy byte identity
PASS  Final file manifest
PASS  Final RSC preservation record
PASS  Final preservation read-back
PASS  Deterministic portable RSC preservation ZIP
PASS  Portable ZIP member byte identity

DASO-NET FORMAL RSC EXTERNAL EVALUATION: FINAL RESULTS PRESERVED

Status:
  FINAL_RSC_FORMAL_RECOVERY_RESULTS_PRESE

In [43]:
# ============================================================
# DASO-Net — TIOMAN EXTERNAL ANCHOR ASSET DISCOVERY AUDIT v1
#
# READ-ONLY.
#
# PURPOSE
# -------
# Locate and inventory the frozen Tioman FTIR-linked external
# assets before defining any Tioman inference protocol.
#
# Known scientific status carried into this audit:
#   - Tioman is a LIMITED external anchor
#   - microscopy images: expected 35
#   - spreadsheet records: expected ~150
#   - previously established unique microscopy matches: 17/35
#
# This cell does NOT:
#   - perform model inference
#   - compute detection metrics
#   - tune thresholds
#   - calibrate
#   - select checkpoints/seeds
#   - modify any project asset
#
# ============================================================

from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path


# ============================================================
# 1. FROZEN PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/abinavprabhuv/"
    "dasonet-project/DASO-Net-Phase1A"
)

AUDIT_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "tioman_external_asset_discovery_audit_v1"
)

AUDIT_JSON = (
    AUDIT_DIR
    / "TIOMAN_EXTERNAL_ASSET_DISCOVERY_AUDIT_v1.json"
)

AUDIT_SHA_RECORD = (
    AUDIT_DIR
    / "TIOMAN_EXTERNAL_ASSET_DISCOVERY_AUDIT_v1.sha256.txt"
)


# ============================================================
# 2. HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def require(
    condition: bool,
    message: str,
) -> None:
    if not condition:
        raise RuntimeError(message)


def write_json(
    path: Path,
    obj,
) -> None:
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def is_image(path: Path) -> bool:
    return path.suffix.lower() in {
        ".jpg",
        ".jpeg",
        ".png",
        ".tif",
        ".tiff",
        ".bmp",
        ".webp",
    }


def is_table(path: Path) -> bool:
    return path.suffix.lower() in {
        ".xlsx",
        ".xls",
        ".csv",
        ".tsv",
    }


# ============================================================
# 3. BANNER / NON-OVERWRITE
# ============================================================

print("=" * 78)
print(
    "DASO-Net — TIOMAN EXTERNAL ANCHOR "
    "ASSET DISCOVERY AUDIT v1"
)
print("=" * 78)
print()

print("Scientific action:")
print("  READ-ONLY frozen-asset discovery")
print()

print("NO inference")
print("NO metric computation")
print("NO threshold tuning")
print("NO calibration")
print("NO checkpoint/seed selection")
print()


require(
    PROJECT_ROOT.is_dir(),
    f"Frozen project root missing:\n{PROJECT_ROOT}",
)

if AUDIT_DIR.exists():
    raise RuntimeError(
        "Tioman discovery-audit directory already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{AUDIT_DIR}"
    )

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 4. SEARCH ENTIRE FROZEN PROJECT FOR TIOMAN / FTIR REFERENCES
# ============================================================

all_files = sorted(
    [
        path
        for path in PROJECT_ROOT.rglob("*")
        if path.is_file()
    ],
    key=lambda p: p.as_posix(),
)


keyword_hits = []

keywords = [
    "tioman",
    "ftir",
]


for path in all_files:

    lower_path = path.as_posix().lower()

    matched_keywords = [
        keyword
        for keyword in keywords
        if keyword in lower_path
    ]

    if matched_keywords:

        keyword_hits.append(
            {
                "path":
                    str(path),

                "relative_path":
                    str(
                        path.relative_to(
                            PROJECT_ROOT
                        )
                    ),

                "suffix":
                    path.suffix.lower(),

                "size_bytes":
                    path.stat().st_size,

                "matched_keywords":
                    matched_keywords,
            }
        )


print(
    f"Frozen project files scanned: {len(all_files)}"
)

print(
    f"Path-level Tioman/FTIR hits:   {len(keyword_hits)}"
)

print()


# ============================================================
# 5. IDENTIFY CANDIDATE DIRECTORIES
# ============================================================

candidate_dirs = sorted(
    {
        Path(
            hit["path"]
        ).parent
        for hit in keyword_hits
    },
    key=lambda p: p.as_posix(),
)


print("=" * 78)
print("CANDIDATE DIRECTORIES")
print("=" * 78)
print()


for directory in candidate_dirs:

    print(
        directory
    )


# ============================================================
# 6. CANDIDATE FILE INVENTORY
# ============================================================

candidate_files = []

for directory in candidate_dirs:

    for path in sorted(
        [
            p
            for p in directory.rglob("*")
            if p.is_file()
        ],
        key=lambda p: p.as_posix(),
    ):

        candidate_files.append(
            path
        )


# Deduplicate paths.
candidate_files = sorted(
    set(
        candidate_files
    ),
    key=lambda p: p.as_posix(),
)


image_files = [
    path
    for path in candidate_files
    if is_image(path)
]

table_files = [
    path
    for path in candidate_files
    if is_table(path)
]

json_files = [
    path
    for path in candidate_files
    if path.suffix.lower() == ".json"
]

text_manifest_files = [
    path
    for path in candidate_files
    if path.suffix.lower() in {
        ".txt",
        ".md",
        ".yaml",
        ".yml",
    }
]


print()
print("=" * 78)
print("CANDIDATE ASSET COUNTS")
print("=" * 78)
print()

print(
    f"Candidate files total: {len(candidate_files)}"
)

print(
    f"Candidate image files: {len(image_files)}"
)

print(
    f"Candidate table files: {len(table_files)}"
)

print(
    f"Candidate JSON files:  {len(json_files)}"
)

print(
    "Candidate text/config files: "
    f"{len(text_manifest_files)}"
)


# ============================================================
# 7. PRINT IMAGE INVENTORY
# ============================================================

print()
print("=" * 78)
print("CANDIDATE IMAGE FILES")
print("=" * 78)
print()


for index, path in enumerate(
    image_files,
    start=1,
):

    print(
        f"{index:03d}  "
        f"{path.relative_to(PROJECT_ROOT)}"
        f"    bytes={path.stat().st_size}"
    )


# ============================================================
# 8. PRINT TABLE / MANIFEST CANDIDATES
# ============================================================

print()
print("=" * 78)
print("CANDIDATE TABLE / MANIFEST FILES")
print("=" * 78)
print()


for path in (
    table_files
    + json_files
    + text_manifest_files
):

    print(
        f"{path.relative_to(PROJECT_ROOT)}"
        f"    bytes={path.stat().st_size}"
    )


# ============================================================
# 9. HASH CANDIDATE SCIENTIFIC ASSETS
# ============================================================

hashed_assets = []


for path in candidate_files:

    # Avoid hashing obviously irrelevant large binary/cache files
    # unless they are an image/table/JSON/text scientific asset.
    if not (
        is_image(path)
        or is_table(path)
        or path.suffix.lower()
        in {
            ".json",
            ".txt",
            ".md",
            ".yaml",
            ".yml",
        }
    ):
        continue

    hashed_assets.append(
        {
            "path":
                str(path),

            "relative_path":
                str(
                    path.relative_to(
                        PROJECT_ROOT
                    )
                ),

            "suffix":
                path.suffix.lower(),

            "size_bytes":
                path.stat().st_size,

            "sha256":
                sha256_file(path),
        }
    )


# ============================================================
# 10. SEARCH MANIFEST-LIKE TEXT FOR TIOMAN REFERENCES
#
# Read small textual files only.
# ============================================================

textual_reference_hits = []


text_candidates = [
    path
    for path in all_files
    if path.suffix.lower()
    in {
        ".csv",
        ".tsv",
        ".json",
        ".txt",
        ".md",
        ".yaml",
        ".yml",
    }
    and path.stat().st_size
    <= 20 * 1024 * 1024
]


for path in text_candidates:

    try:
        text = path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

    except Exception:
        continue

    lower_text = text.lower()

    if (
        "tioman" in lower_text
        or "ftir" in lower_text
    ):

        matching_lines = []

        for line_number, line in enumerate(
            text.splitlines(),
            start=1,
        ):

            lower_line = line.lower()

            if (
                "tioman" in lower_line
                or "ftir" in lower_line
            ):

                matching_lines.append(
                    {
                        "line_number":
                            line_number,

                        "text":
                            line[:500],
                    }
                )

                if len(
                    matching_lines
                ) >= 25:
                    break

        textual_reference_hits.append(
            {
                "path":
                    str(path),

                "relative_path":
                    str(
                        path.relative_to(
                            PROJECT_ROOT
                        )
                    ),

                "size_bytes":
                    path.stat().st_size,

                "matching_lines":
                    matching_lines,
            }
        )


print()
print("=" * 78)
print("TEXTUAL TIOMAN / FTIR REFERENCES")
print("=" * 78)
print()


for item in textual_reference_hits:

    print()
    print(
        item[
            "relative_path"
        ]
    )

    for match in item[
        "matching_lines"
    ]:

        print(
            f"  L{match['line_number']}: "
            f"{match['text']}"
        )


# ============================================================
# 11. DISCOVERY CLASSIFICATION
# ============================================================

expected_prior_status = {
    "microscopy_images_expected":
        35,

    "spreadsheet_records_expected_approximately":
        150,

    "previously_established_unique_microscopy_matches":
        17,

    "previously_established_total_microscopy_images":
        35,

    "scientific_role":
        "LIMITED_FTIR_LINKED_EXTERNAL_ANCHOR",
}


# Do NOT fail merely because recursive candidates contain more
# than 35 images; the next audit will resolve the exact set.
#
# This discovery cell's purpose is only to identify the frozen
# source paths and manifests without guessing them.

readiness = (
    "READY_FOR_TIOMAN_EXACT_ASSET_AND_MAPPING_AUDIT"
    if (
        len(image_files) > 0
        and (
            len(table_files) > 0
            or len(textual_reference_hits) > 0
        )
    )
    else
    "TIOMAN_SOURCE_LOCATION_REQUIRES_REVIEW"
)


# ============================================================
# 12. WRITE DISCOVERY AUDIT
# ============================================================

audit = {
    "audit_name":
        (
            "DASO-Net Tioman External Anchor "
            "Asset Discovery Audit v1"
        ),

    "status":
        "PASS",

    "scientific_action":
        "READ_ONLY_FROZEN_ASSET_DISCOVERY",

    "inference_performed":
        False,

    "metrics_computed":
        False,

    "threshold_tuning":
        False,

    "calibration":
        False,

    "checkpoint_selection":
        False,

    "seed_selection":
        False,

    "project_root":
        str(
            PROJECT_ROOT
        ),

    "expected_prior_status":
        expected_prior_status,

    "discovery":
        {
            "frozen_project_files_scanned":
                len(all_files),

            "keyword_path_hits":
                keyword_hits,

            "candidate_directories":
                [
                    str(path)
                    for path in candidate_dirs
                ],

            "candidate_file_count":
                len(candidate_files),

            "candidate_image_count":
                len(image_files),

            "candidate_table_count":
                len(table_files),

            "candidate_json_count":
                len(json_files),

            "candidate_text_config_count":
                len(text_manifest_files),

            "candidate_images":
                [
                    str(
                        path.relative_to(
                            PROJECT_ROOT
                        )
                    )
                    for path in image_files
                ],

            "candidate_tables":
                [
                    str(
                        path.relative_to(
                            PROJECT_ROOT
                        )
                    )
                    for path in table_files
                ],

            "hashed_scientific_assets":
                hashed_assets,

            "textual_reference_hits":
                textual_reference_hits,
        },

    "readiness":
        readiness,
}


write_json(
    AUDIT_JSON,
    audit,
)

audit_sha = sha256_file(
    AUDIT_JSON
)

AUDIT_SHA_RECORD.write_text(
    (
        f"{audit_sha}  "
        f"{AUDIT_JSON.name}\n"
    ),
    encoding="utf-8",
)


# ============================================================
# 13. FINAL REPORT
# ============================================================

print()
print("=" * 78)
print(
    "TIOMAN EXTERNAL ASSET DISCOVERY AUDIT: PASS"
)
print("=" * 78)
print()

print(
    f"Candidate images found: {len(image_files)}"
)

print(
    f"Candidate tables found: {len(table_files)}"
)

print(
    f"Textual references found: "
    f"{len(textual_reference_hits)}"
)

print()

print(
    "Discovery audit JSON:"
)

print(
    f"  {AUDIT_JSON}"
)

print()

print(
    "Discovery audit SHA256:"
)

print(
    f"  {audit_sha}"
)

print()

print(
    "NEXT READINESS:"
)

print(
    f"  {readiness}"
)

print("=" * 78)

DASO-Net — TIOMAN EXTERNAL ANCHOR ASSET DISCOVERY AUDIT v1

Scientific action:
  READ-ONLY frozen-asset discovery

NO inference
NO metric computation
NO threshold tuning
NO calibration
NO checkpoint/seed selection

Frozen project files scanned: 4348
Path-level Tioman/FTIR hits:   0

CANDIDATE DIRECTORIES


CANDIDATE ASSET COUNTS

Candidate files total: 0
Candidate image files: 0
Candidate table files: 0
Candidate JSON files:  0
Candidate text/config files: 0

CANDIDATE IMAGE FILES


CANDIDATE TABLE / MANIFEST FILES


TEXTUAL TIOMAN / FTIR REFERENCES


data/public_real_mp/v1_processed/public_real_mp_v1_source_specific_manifest.csv
  L1142: tioman2025_ftir,"_a001_Supporting Dataset _6fc16519\Supporting Dataset of FTIR, Microscopic_6fc16519\Microscopic Images\JUARA BEACH\JB_S01_B_F_1.jpg",JB_S01_B_F_1.jpg,3840,2160,21f2976df30e04380b48c54c0089d9bce50c5f5318e557d52dd55cffd9e1921a,,,False
  L1143: tioman2025_ftir,"_a001_Supporting Dataset _6fc16519\Supporting Dataset of FTIR, Microscopic_6f

In [44]:
# ============================================================
# DASO-Net — TIOMAN EXACT ASSET RESOLUTION AUDIT v1
#
# READ-ONLY.
#
# PURPOSE
# -------
# Resolve the exact Tioman microscopy records from the already-
# frozen public-real source-specific manifest to physical files
# currently mounted under /kaggle/input.
#
# This avoids relying on directory names containing "tioman"
# or "ftir".
#
# The audit:
#   1. verifies the previous discovery audit
#   2. parses all source-specific manifest rows
#   3. isolates source == tioman2025_ftir
#   4. expects exactly 35 microscopy records
#   5. extracts filename / dimensions / SHA256
#   6. scans mounted Kaggle inputs by exact basename
#   7. SHA-verifies any candidate image bytes
#   8. identifies Kaggle dataset roots containing exact matches
#   9. inventories spreadsheet files in those dataset roots
#  10. writes a provenance audit JSON
#
# NO model inference
# NO metric computation
# NO FTIR/image matching reconstruction yet
# NO threshold tuning
# NO calibration
# NO checkpoint/seed selection
# NO file modification
#
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import re
from collections import Counter, defaultdict
from pathlib import Path


# ============================================================
# 1. INPUTS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/abinavprabhuv/"
    "dasonet-project/DASO-Net-Phase1A"
)

SOURCE_MANIFEST = (
    PROJECT_ROOT
    / "data/public_real_mp/v1_processed/"
      "public_real_mp_v1_source_specific_manifest.csv"
)

PRIOR_DISCOVERY_AUDIT = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "tioman_external_asset_discovery_audit_v1/"
    "TIOMAN_EXTERNAL_ASSET_DISCOVERY_AUDIT_v1.json"
)

EXPECTED_PRIOR_DISCOVERY_SHA256 = (
    "68b58011a322f3860e2c58358ea08d17"
    "6857349d4371b05c2ca8e0fefcc9dc92"
)

KAGGLE_INPUT_ROOT = Path(
    "/kaggle/input"
)

TIOMAN_SOURCE_ID = (
    "tioman2025_ftir"
)


# ============================================================
# 2. OUTPUT
# ============================================================

AUDIT_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "tioman_exact_asset_resolution_audit_v1"
)

AUDIT_JSON = (
    AUDIT_DIR
    / "TIOMAN_EXACT_ASSET_RESOLUTION_AUDIT_v1.json"
)

AUDIT_SHA_RECORD = (
    AUDIT_DIR
    / "TIOMAN_EXACT_ASSET_RESOLUTION_AUDIT_v1.sha256.txt"
)


# ============================================================
# 3. HELPERS
# ============================================================

IMAGE_SUFFIXES = {
    ".jpg",
    ".jpeg",
    ".png",
    ".tif",
    ".tiff",
    ".bmp",
    ".webp",
}

SPREADSHEET_SUFFIXES = {
    ".xlsx",
    ".xls",
    ".xlsm",
    ".csv",
    ".tsv",
}

SHA256_RE = re.compile(
    r"^[0-9a-fA-F]{64}$"
)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def require(
    condition: bool,
    message: str,
) -> None:

    if not condition:
        raise RuntimeError(
            message
        )


def require_file(
    path: Path,
    label: str,
) -> None:

    require(
        path.is_file(),
        f"{label} missing:\n{path}",
    )


def require_dir(
    path: Path,
    label: str,
) -> None:

    require(
        path.is_dir(),
        f"{label} missing:\n{path}",
    )


def read_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path: Path,
    obj,
) -> None:

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def normalize_path_text(
    value: str,
) -> str:

    return (
        str(
            value
        )
        .replace(
            "\\",
            "/",
        )
        .strip()
    )


def extract_image_cells(
    row: dict,
):

    hits = []

    for column, value in row.items():

        text = str(
            value or ""
        ).strip()

        suffix = Path(
            normalize_path_text(
                text
            )
        ).suffix.lower()

        if suffix in IMAGE_SUFFIXES:

            hits.append(
                {
                    "column":
                        column,

                    "value":
                        text,

                    "basename":
                        Path(
                            normalize_path_text(
                                text
                            )
                        ).name,
                }
            )

    return hits


def extract_sha_cells(
    row: dict,
):

    hits = []

    for column, value in row.items():

        text = str(
            value or ""
        ).strip()

        if SHA256_RE.fullmatch(
            text
        ):

            hits.append(
                {
                    "column":
                        column,

                    "value":
                        text.lower(),
                }
            )

    return hits


def infer_dataset_root(
    path: Path,
):

    resolved = path.resolve()

    parts = resolved.parts

    try:
        index = parts.index(
            "input"
        )

    except ValueError:
        return None

    # /kaggle/input/<dataset-slug>/...
    if len(parts) <= index + 1:
        return None

    return Path(
        *parts[
            : index + 2
        ]
    )


# ============================================================
# 4. BANNER / NON-OVERWRITE
# ============================================================

print("=" * 80)

print(
    "DASO-Net — TIOMAN EXACT ASSET "
    "RESOLUTION AUDIT v1"
)

print("=" * 80)

print()

print(
    "Scientific action:"
)

print(
    "  READ-ONLY manifest-to-physical-asset resolution"
)

print()

print(
    "NO inference"
)

print(
    "NO metric computation"
)

print(
    "NO FTIR/image mapping reconstruction yet"
)

print(
    "NO tuning/calibration/selection"
)

print()


require_dir(
    PROJECT_ROOT,
    "Frozen DASO-Net project",
)

require_dir(
    KAGGLE_INPUT_ROOT,
    "Kaggle input root",
)

require_file(
    SOURCE_MANIFEST,
    "Frozen public-real source-specific manifest",
)

require_file(
    PRIOR_DISCOVERY_AUDIT,
    "Prior Tioman discovery audit",
)


if AUDIT_DIR.exists():

    raise RuntimeError(
        "Tioman exact-asset audit directory already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{AUDIT_DIR}"
    )


AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 5. VERIFY PRIOR DISCOVERY AUDIT
# ============================================================

prior_sha = sha256_file(
    PRIOR_DISCOVERY_AUDIT
)

require(
    prior_sha
    == EXPECTED_PRIOR_DISCOVERY_SHA256,
    "Prior Tioman discovery audit SHA mismatch.\n"
    f"Expected: {EXPECTED_PRIOR_DISCOVERY_SHA256}\n"
    f"Actual:   {prior_sha}",
)

prior = read_json(
    PRIOR_DISCOVERY_AUDIT
)

require(
    prior.get(
        "status"
    )
    == "PASS",
    "Prior Tioman discovery audit != PASS.",
)

require(
    prior.get(
        "readiness"
    )
    == "TIOMAN_SOURCE_LOCATION_REQUIRES_REVIEW",
    "Unexpected prior Tioman discovery readiness.",
)

print(
    "PASS  Prior Tioman discovery-audit identity"
)


# ============================================================
# 6. READ SOURCE-SPECIFIC MANIFEST
# ============================================================

with SOURCE_MANIFEST.open(
    "r",
    encoding="utf-8-sig",
    newline="",
) as f:

    reader = csv.DictReader(
        f
    )

    manifest_columns = list(
        reader.fieldnames
        or []
    )

    manifest_rows = list(
        reader
    )


require(
    len(
        manifest_columns
    ) > 0,
    "Source-specific manifest has no header.",
)

require(
    len(
        manifest_rows
    ) > 0,
    "Source-specific manifest has no data rows.",
)


source_columns = []

for column in manifest_columns:

    if any(
        str(
            row.get(
                column,
                ""
            )
            or ""
        ).strip()
        == TIOMAN_SOURCE_ID
        for row in manifest_rows
    ):

        source_columns.append(
            column
        )


require(
    len(
        source_columns
    )
    == 1,
    (
        "Could not uniquely identify the manifest source column.\n"
        f"Columns containing {TIOMAN_SOURCE_ID!r}: "
        f"{source_columns}"
    ),
)


source_column = (
    source_columns[
        0
    ]
)


tioman_manifest_rows = []

for zero_index, row in enumerate(
    manifest_rows
):

    if (
        str(
            row.get(
                source_column,
                ""
            )
            or ""
        ).strip()
        != TIOMAN_SOURCE_ID
    ):

        continue

    # CSV physical line:
    # header = line 1, first DictReader row = line 2.
    csv_line_number = (
        zero_index
        + 2
    )

    tioman_manifest_rows.append(
        {
            "csv_line_number":
                csv_line_number,

            "row":
                row,
        }
    )


require(
    len(
        tioman_manifest_rows
    )
    == 35,
    (
        "Expected exactly 35 Tioman source rows.\n"
        f"Observed: {len(tioman_manifest_rows)}"
    ),
)


print(
    "PASS  Exact Tioman manifest population = 35"
)

print(
    f"Manifest source column: {source_column}"
)


# ============================================================
# 7. NORMALIZE THE 35 RECORD IDENTITIES
# ============================================================

tioman_records = []


for ordinal, item in enumerate(
    tioman_manifest_rows,
    start=1,
):

    row = item[
        "row"
    ]

    image_cells = extract_image_cells(
        row
    )

    sha_cells = extract_sha_cells(
        row
    )


    require(
        len(
            image_cells
        )
        >= 1,
        (
            "Tioman manifest row contains no recognizable "
            "image-path/filename cell.\n"
            f"CSV line: {item['csv_line_number']}"
        ),
    )

    require(
        len(
            sha_cells
        )
        >= 1,
        (
            "Tioman manifest row contains no SHA256 cell.\n"
            f"CSV line: {item['csv_line_number']}"
        ),
    )


    # Prefer an image cell whose value is only a basename;
    # otherwise use the final image-valued cell.
    basename_only_cells = [
        cell
        for cell in image_cells
        if normalize_path_text(
            cell[
                "value"
            ]
        )
        == cell[
            "basename"
        ]
    ]

    if basename_only_cells:

        chosen_image_cell = (
            basename_only_cells[
                -1
            ]
        )

    else:

        chosen_image_cell = (
            image_cells[
                -1
            ]
        )


    filename = (
        chosen_image_cell[
            "basename"
        ]
    )

    expected_sha = (
        sha_cells[
            0
        ][
            "value"
        ]
    )


    original_path_candidates = [
        cell[
            "value"
        ]
        for cell in image_cells
        if (
            "/" in normalize_path_text(
                cell[
                    "value"
                ]
            )
        )
    ]

    original_source_path = (
        original_path_candidates[
            0
        ]
        if original_path_candidates
        else None
    )


    # Detect plausible width/height integers from row fields.
    numeric_cells = []

    for column, value in row.items():

        text = str(
            value or ""
        ).strip()

        try:
            numeric_value = int(
                float(
                    text
                )
            )

        except Exception:
            continue

        if numeric_value > 0:

            numeric_cells.append(
                {
                    "column":
                        column,

                    "value":
                        numeric_value,
                }
            )


    tioman_records.append(
        {
            "ordinal":
                ordinal,

            "csv_line_number":
                item[
                    "csv_line_number"
                ],

            "source_id":
                TIOMAN_SOURCE_ID,

            "filename":
                filename,

            "expected_sha256":
                expected_sha,

            "original_source_path":
                original_source_path,

            "image_cells":
                image_cells,

            "sha_cells":
                sha_cells,

            "positive_numeric_cells":
                numeric_cells,

            "raw_manifest_row":
                row,
        }
    )


filenames = [
    record[
        "filename"
    ]
    for record in tioman_records
]


require(
    len(
        set(
            filenames
        )
    )
    == 35,
    "The 35 Tioman manifest filenames are not unique.",
)


expected_shas = [
    record[
        "expected_sha256"
    ]
    for record in tioman_records
]


require(
    len(
        set(
            expected_shas
        )
    )
    == 35,
    "The 35 Tioman manifest SHA256 values are not unique.",
)


print(
    "PASS  35 unique Tioman microscopy filenames"
)

print(
    "PASS  35 unique Tioman microscopy SHA256 identities"
)


# ============================================================
# 8. PRINT EXACT MANIFEST POPULATION
# ============================================================

print()

print("=" * 80)

print(
    "FROZEN TIOMAN MICROSCOPY MANIFEST POPULATION"
)

print("=" * 80)

print()


for record in tioman_records:

    print(
        f"{record['ordinal']:02d}  "
        f"L{record['csv_line_number']}  "
        f"{record['filename']}  "
        f"sha256={record['expected_sha256']}"
    )


# ============================================================
# 9. SCAN ALL MOUNTED KAGGLE INPUTS BY BASENAME
#
# No image decoding is performed here.
# ============================================================

wanted_basenames = set(
    filenames
)

basename_candidates = defaultdict(
    list
)

all_input_file_count = 0


for current_root, dirnames, file_names in os.walk(
    KAGGLE_INPUT_ROOT
):

    dirnames.sort()
    file_names.sort()

    root_path = Path(
        current_root
    )

    for file_name in file_names:

        all_input_file_count += 1

        if file_name in wanted_basenames:

            basename_candidates[
                file_name
            ].append(
                root_path
                / file_name
            )


print()

print(
    f"Mounted /kaggle/input files scanned: "
    f"{all_input_file_count}"
)


# ============================================================
# 10. HASH-VERIFY BASENAME CANDIDATES
# ============================================================

resolution_rows = []

exact_physical_paths = []

unresolved_records = []

basename_collision_records = []


for record in tioman_records:

    filename = record[
        "filename"
    ]

    expected_sha = record[
        "expected_sha256"
    ]

    candidates = sorted(
        basename_candidates.get(
            filename,
            []
        ),
        key=lambda p: p.as_posix(),
    )

    candidate_details = []

    exact_matches = []


    for candidate in candidates:

        actual_sha = sha256_file(
            candidate
        )

        detail = {
            "path":
                str(
                    candidate
                ),

            "size_bytes":
                candidate.stat().st_size,

            "sha256":
                actual_sha,

            "sha_matches_manifest":
                actual_sha
                == expected_sha,
        }

        candidate_details.append(
            detail
        )

        if (
            actual_sha
            == expected_sha
        ):

            exact_matches.append(
                candidate
            )


    if len(
        candidates
    ) > 1:

        basename_collision_records.append(
            filename
        )


    if not exact_matches:

        unresolved_records.append(
            filename
        )


    exact_physical_paths.extend(
        exact_matches
    )


    resolution_rows.append(
        {
            "ordinal":
                record[
                    "ordinal"
                ],

            "csv_line_number":
                record[
                    "csv_line_number"
                ],

            "filename":
                filename,

            "expected_sha256":
                expected_sha,

            "basename_candidate_count":
                len(
                    candidates
                ),

            "exact_sha_match_count":
                len(
                    exact_matches
                ),

            "candidate_files":
                candidate_details,

            "resolved":
                len(
                    exact_matches
                )
                >= 1,
        }
    )


resolved_count = sum(
    1
    for row in resolution_rows
    if row[
        "resolved"
    ]
)


print()

print("=" * 80)

print(
    "TIOMAN PHYSICAL-ASSET RESOLUTION"
)

print("=" * 80)

print()


for row in resolution_rows:

    status = (
        "RESOLVED"
        if row[
            "resolved"
        ]
        else "MISSING"
    )

    print(
        f"{row['ordinal']:02d}  "
        f"{status:<8}  "
        f"{row['filename']}  "
        f"basename_candidates="
        f"{row['basename_candidate_count']}  "
        f"exact_sha_matches="
        f"{row['exact_sha_match_count']}"
    )

    for candidate in row[
        "candidate_files"
    ]:

        marker = (
            "MATCH"
            if candidate[
                "sha_matches_manifest"
            ]
            else "SHA-MISMATCH"
        )

        print(
            f"      [{marker}] "
            f"{candidate['path']}"
        )


print()

print(
    f"Exact Tioman records resolved: "
    f"{resolved_count}/35"
)


# ============================================================
# 11. RESOLVED DATASET ROOTS
# ============================================================

dataset_roots = sorted(
    {
        infer_dataset_root(
            path
        )
        for path in exact_physical_paths
        if infer_dataset_root(
            path
        )
        is not None
    },
    key=lambda p: p.as_posix(),
)


print()

print("=" * 80)

print(
    "KAGGLE DATASET ROOTS CONTAINING "
    "SHA-VERIFIED TIOMAN IMAGES"
)

print("=" * 80)

print()


if dataset_roots:

    for root in dataset_roots:

        print(
            root
        )

else:

    print(
        "NONE"
    )


# ============================================================
# 12. SPREADSHEET INVENTORY INSIDE RESOLVED DATASET ROOTS
# ============================================================

dataset_spreadsheets = []


for dataset_root in dataset_roots:

    for path in sorted(
        [
            p
            for p in dataset_root.rglob("*")
            if (
                p.is_file()
                and p.suffix.lower()
                in SPREADSHEET_SUFFIXES
            )
        ],
        key=lambda p: p.as_posix(),
    ):

        dataset_spreadsheets.append(
            {
                "dataset_root":
                    str(
                        dataset_root
                    ),

                "path":
                    str(
                        path
                    ),

                "relative_path":
                    str(
                        path.relative_to(
                            dataset_root
                        )
                    ),

                "suffix":
                    path.suffix.lower(),

                "size_bytes":
                    path.stat().st_size,

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


print()

print("=" * 80)

print(
    "SPREADSHEETS IN RESOLVED TIOMAN "
    "DATASET ROOT(S)"
)

print("=" * 80)

print()


if dataset_spreadsheets:

    for item in dataset_spreadsheets:

        print(
            f"{item['relative_path']}  "
            f"bytes={item['size_bytes']}  "
            f"sha256={item['sha256']}"
        )

else:

    print(
        "NONE"
    )


# ============================================================
# 13. FALLBACK: INVENTORY ALL EXCEL FILES UNDER /kaggle/input
#
# This is only an inventory. Workbook contents are NOT yet used
# to reconstruct FTIR/image correspondence.
# ============================================================

all_excel_files = []


for current_root, dirnames, file_names in os.walk(
    KAGGLE_INPUT_ROOT
):

    dirnames.sort()
    file_names.sort()

    root_path = Path(
        current_root
    )

    for file_name in file_names:

        suffix = Path(
            file_name
        ).suffix.lower()

        if suffix not in {
            ".xlsx",
            ".xls",
            ".xlsm",
        }:
            continue

        path = (
            root_path
            / file_name
        )

        all_excel_files.append(
            {
                "path":
                    str(
                        path
                    ),

                "size_bytes":
                    path.stat().st_size,
            }
        )


print()

print("=" * 80)

print(
    "ALL EXCEL WORKBOOKS CURRENTLY MOUNTED "
    "UNDER /kaggle/input"
)

print("=" * 80)

print()


for index, item in enumerate(
    all_excel_files,
    start=1,
):

    print(
        f"{index:03d}  "
        f"{item['path']}  "
        f"bytes={item['size_bytes']}"
    )


# ============================================================
# 14. MANIFEST SHAPE SUMMARY
# ============================================================

dimension_pairs = []


for record in tioman_records:

    row = record[
        "raw_manifest_row"
    ]

    # Look explicitly for common width/height column names.
    lower_to_original = {
        str(
            column
        ).lower():
            column
        for column in row
    }

    width_column = None
    height_column = None

    for candidate in [
        "width",
        "image_width",
        "w",
    ]:

        if candidate in lower_to_original:

            width_column = (
                lower_to_original[
                    candidate
                ]
            )

            break

    for candidate in [
        "height",
        "image_height",
        "h",
    ]:

        if candidate in lower_to_original:

            height_column = (
                lower_to_original[
                    candidate
                ]
            )

            break


    if (
        width_column is not None
        and height_column is not None
    ):

        try:

            width = int(
                float(
                    row[
                        width_column
                    ]
                )
            )

            height = int(
                float(
                    row[
                        height_column
                    ]
                )
            )

            dimension_pairs.append(
                (
                    width,
                    height,
                )
            )

        except Exception:
            pass


shape_distribution = Counter(
    dimension_pairs
)


# ============================================================
# 15. READINESS CLASSIFICATION
# ============================================================

if (
    resolved_count
    == 35
):

    if dataset_spreadsheets:

        readiness = (
            "READY_FOR_TIOMAN_FTIR_MAPPING_AUDIT"
        )

    else:

        readiness = (
            "TIOMAN_IMAGES_RESOLVED_"
            "FTIR_WORKBOOK_LOCATION_REQUIRES_REVIEW"
        )

else:

    readiness = (
        "TIOMAN_RAW_IMAGE_ASSETS_NOT_FULLY_MOUNTED"
    )


# ============================================================
# 16. WRITE AUDIT
# ============================================================

audit = {
    "audit_name":
        (
            "DASO-Net Tioman Exact Asset "
            "Resolution Audit v1"
        ),

    "status":
        "PASS",

    "scientific_action":
        "READ_ONLY_ASSET_RESOLUTION",

    "inference_performed":
        False,

    "metrics_computed":
        False,

    "ftir_image_mapping_reconstructed":
        False,

    "threshold_tuning":
        False,

    "calibration":
        False,

    "checkpoint_selection":
        False,

    "seed_selection":
        False,

    "prior_discovery_audit":
        {
            "path":
                str(
                    PRIOR_DISCOVERY_AUDIT
                ),

            "sha256":
                prior_sha,
        },

    "source_manifest":
        {
            "path":
                str(
                    SOURCE_MANIFEST
                ),

            "sha256":
                sha256_file(
                    SOURCE_MANIFEST
                ),

            "columns":
                manifest_columns,

            "source_column":
                source_column,

            "tioman_source_id":
                TIOMAN_SOURCE_ID,

            "tioman_row_count":
                len(
                    tioman_records
                ),

            "unique_filename_count":
                len(
                    set(
                        filenames
                    )
                ),

            "unique_sha256_count":
                len(
                    set(
                        expected_shas
                    )
                ),

            "shape_distribution":
                [
                    {
                        "width":
                            width,

                        "height":
                            height,

                        "count":
                            count,
                    }
                    for (
                        width,
                        height
                    ), count
                    in sorted(
                        shape_distribution.items()
                    )
                ],
        },

    "tioman_manifest_records":
        tioman_records,

    "physical_resolution":
        {
            "kaggle_input_files_scanned":
                all_input_file_count,

            "resolved_records":
                resolved_count,

            "expected_records":
                35,

            "unresolved_filenames":
                unresolved_records,

            "basenames_with_multiple_candidates":
                basename_collision_records,

            "records":
                resolution_rows,
        },

    "resolved_kaggle_dataset_roots":
        [
            str(
                root
            )
            for root in dataset_roots
        ],

    "spreadsheets_in_resolved_dataset_roots":
        dataset_spreadsheets,

    "all_mounted_excel_files":
        all_excel_files,

    "scientific_role":
        (
            "LIMITED_FTIR_LINKED_EXTERNAL_ANCHOR"
        ),

    "previously_established_mapping_status":
        {
            "microscopy_images":
                35,

            "unique_matches":
                17,

            "mapping_is_not_reconstructed_by_this_audit":
                True,
        },

    "readiness":
        readiness,
}


write_json(
    AUDIT_JSON,
    audit,
)

audit_sha = sha256_file(
    AUDIT_JSON
)

AUDIT_SHA_RECORD.write_text(
    (
        f"{audit_sha}  "
        f"{AUDIT_JSON.name}\n"
    ),
    encoding="utf-8",
)


# ============================================================
# 17. FINAL REPORT
# ============================================================

print()

print("=" * 80)

print(
    "TIOMAN EXACT ASSET RESOLUTION AUDIT: PASS"
)

print("=" * 80)

print()

print(
    "Frozen Tioman manifest records:"
)

print(
    "  35"
)

print()

print(
    "SHA-verified physical microscopy records:"
)

print(
    f"  {resolved_count}/35"
)

print()

print(
    "Resolved Kaggle dataset roots:"
)

if dataset_roots:

    for root in dataset_roots:

        print(
            f"  {root}"
        )

else:

    print(
        "  NONE"
    )

print()

print(
    "Spreadsheets in resolved Tioman dataset root(s):"
)

if dataset_spreadsheets:

    for item in dataset_spreadsheets:

        print(
            "  "
            f"{item['path']}"
        )

else:

    print(
        "  NONE"
    )

print()

print(
    "All mounted Excel workbooks:"
)

print(
    f"  {len(all_excel_files)}"
)

print()

print(
    "Exact-asset audit JSON:"
)

print(
    f"  {AUDIT_JSON}"
)

print()

print(
    "Exact-asset audit SHA256:"
)

print(
    f"  {audit_sha}"
)

print()

print(
    "NEXT READINESS:"
)

print(
    f"  {readiness}"
)

print("=" * 80)

DASO-Net — TIOMAN EXACT ASSET RESOLUTION AUDIT v1

Scientific action:
  READ-ONLY manifest-to-physical-asset resolution

NO inference
NO metric computation
NO FTIR/image mapping reconstruction yet
NO tuning/calibration/selection

PASS  Prior Tioman discovery-audit identity
PASS  Exact Tioman manifest population = 35
Manifest source column: source_id
PASS  35 unique Tioman microscopy filenames
PASS  35 unique Tioman microscopy SHA256 identities

FROZEN TIOMAN MICROSCOPY MANIFEST POPULATION

01  L1142  JB_S01_B_F_1.jpg  sha256=21f2976df30e04380b48c54c0089d9bce50c5f5318e557d52dd55cffd9e1921a
02  L1143  JB_S02_A_M_1.jpg  sha256=1b57b5f4e8f4e136f2c41d02773543262e80424a5615ecaaab2c74034fdde05e
03  L1144  JB_S05_A_F_1.jpg  sha256=a7a385fd3981d6f42bd43872a9c20a6a42e43ec5eea8af0d7026a8c1e93fa16d
04  L1145  JB_S05_A_M_1.jpg  sha256=f7c8b502c0b731ee38af9c4fee304c54e22483f614ac08a3c3697ad16513426b
05  L1146  JB_S05_B_M_1.jpg  sha256=fd5b93dc7d71bc432526183d7f336dc56ae26279fe9b25354305224c6cc7818b


In [45]:
# ============================================================
# DASO-Net — TIOMAN SOURCE PROVENANCE + ARCHIVE AUDIT v1
#
# READ-ONLY.
#
# PURPOSE
# -------
# Determine how the frozen Tioman manifest population was
# originally sourced and whether the raw 35 microscopy images
# and FTIR workbook are:
#
#   A. present inside a mounted archive, or
#   B. absent from current Kaggle inputs but recoverable from
#      frozen acquisition/source metadata.
#
# NO inference
# NO metric computation
# NO FTIR/image rematching
# NO tuning/calibration/selection
# NO extraction
# NO download
# NO modification of source data
#
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import tarfile
import zipfile
from pathlib import Path


# ============================================================
# 1. INPUTS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/abinavprabhuv/"
    "dasonet-project/DASO-Net-Phase1A"
)

KAGGLE_INPUT_ROOT = Path(
    "/kaggle/input"
)

SOURCE_MANIFEST = (
    PROJECT_ROOT
    / "data/public_real_mp/v1_processed/"
      "public_real_mp_v1_source_specific_manifest.csv"
)

PRIOR_ASSET_AUDIT = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "tioman_exact_asset_resolution_audit_v1/"
    "TIOMAN_EXACT_ASSET_RESOLUTION_AUDIT_v1.json"
)

EXPECTED_PRIOR_ASSET_AUDIT_SHA256 = (
    "cf99b32c23659182d1de879559e62655"
    "072dc68018181f0e2c623c62cc8d09f7"
)

TIOMAN_SOURCE_ID = (
    "tioman2025_ftir"
)


# ============================================================
# 2. OUTPUT
# ============================================================

AUDIT_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "tioman_source_provenance_archive_audit_v1"
)

AUDIT_JSON = (
    AUDIT_DIR
    / "TIOMAN_SOURCE_PROVENANCE_ARCHIVE_AUDIT_v1.json"
)

AUDIT_SHA_RECORD = (
    AUDIT_DIR
    / "TIOMAN_SOURCE_PROVENANCE_ARCHIVE_AUDIT_v1.sha256.txt"
)


# ============================================================
# 3. SEARCH SETTINGS
# ============================================================

TEXT_SUFFIXES = {
    ".csv",
    ".tsv",
    ".json",
    ".txt",
    ".md",
    ".yaml",
    ".yml",
    ".py",
    ".toml",
    ".ini",
}

ARCHIVE_SUFFIXES = {
    ".zip",
    ".tar",
    ".tgz",
    ".gz",
    ".bz2",
    ".xz",
    ".7z",
    ".rar",
}

MAX_TEXT_FILE_BYTES = (
    50
    * 1024
    * 1024
)

SEARCH_TERMS = [
    "tioman2025_ftir",
    "Supporting Dataset of FTIR",
    "Supporting Dataset",
    "Microscopic Images",
    "JUARA BEACH",
    "SALANG BEACH",
    "TULAI BEACH",
    "JB_S01_B_F_1.jpg",
    "TB_S05_B_F_4.jpg",
]


# ============================================================
# 4. HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def require(
    condition: bool,
    message: str,
) -> None:

    if not condition:
        raise RuntimeError(message)


def require_file(
    path: Path,
    label: str,
) -> None:

    require(
        path.is_file(),
        f"{label} missing:\n{path}",
    )


def require_dir(
    path: Path,
    label: str,
) -> None:

    require(
        path.is_dir(),
        f"{label} missing:\n{path}",
    )


def read_json(path: Path):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path: Path,
    obj,
) -> None:

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def normalize_member_name(
    value: str,
) -> str:

    return (
        str(value)
        .replace("\\", "/")
        .strip()
    )


def basename(
    value: str,
) -> str:

    return Path(
        normalize_member_name(
            value
        )
    ).name


# ============================================================
# 5. BANNER / NON-OVERWRITE
# ============================================================

print("=" * 80)

print(
    "DASO-Net — TIOMAN SOURCE PROVENANCE "
    "+ ARCHIVE AUDIT v1"
)

print("=" * 80)

print()

print(
    "Scientific action:"
)

print(
    "  READ-ONLY source-provenance / archive inspection"
)

print()

print("NO inference")
print("NO metric computation")
print("NO FTIR/image rematching")
print("NO download")
print("NO archive extraction")
print("NO tuning/calibration/selection")
print()


require_dir(
    PROJECT_ROOT,
    "Frozen DASO-Net project root",
)

require_dir(
    KAGGLE_INPUT_ROOT,
    "Kaggle input root",
)

require_file(
    SOURCE_MANIFEST,
    "Frozen public-real manifest",
)

require_file(
    PRIOR_ASSET_AUDIT,
    "Prior Tioman exact-asset audit",
)


if AUDIT_DIR.exists():

    raise RuntimeError(
        "Tioman source-provenance audit already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{AUDIT_DIR}"
    )


AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 6. VERIFY PRIOR ASSET AUDIT
# ============================================================

prior_sha = sha256_file(
    PRIOR_ASSET_AUDIT
)

require(
    prior_sha
    == EXPECTED_PRIOR_ASSET_AUDIT_SHA256,
    "Prior Tioman exact-asset audit SHA mismatch.\n"
    f"Expected: {EXPECTED_PRIOR_ASSET_AUDIT_SHA256}\n"
    f"Actual:   {prior_sha}",
)

prior = read_json(
    PRIOR_ASSET_AUDIT
)

require(
    prior.get("status")
    == "PASS",
    "Prior Tioman exact-asset audit != PASS.",
)

require(
    prior.get("readiness")
    == "TIOMAN_RAW_IMAGE_ASSETS_NOT_FULLY_MOUNTED",
    "Unexpected prior Tioman readiness.",
)

require(
    int(
        prior[
            "physical_resolution"
        ][
            "resolved_records"
        ]
    )
    == 0,
    "Prior Tioman audit no longer reports 0/35 resolved.",
)


print(
    "PASS  Prior exact-asset audit identity"
)


# ============================================================
# 7. RECOVER EXACT 35 MANIFEST FILENAMES
# ============================================================

with SOURCE_MANIFEST.open(
    "r",
    encoding="utf-8-sig",
    newline="",
) as f:

    rows = list(
        csv.DictReader(f)
    )


tioman_rows = [
    row
    for row in rows
    if str(
        row.get(
            "source_id",
            ""
        )
    ).strip()
    == TIOMAN_SOURCE_ID
]


require(
    len(tioman_rows)
    == 35,
    (
        "Expected exactly 35 Tioman manifest rows; "
        f"found {len(tioman_rows)}."
    ),
)


wanted_filenames = []

wanted_sha_by_filename = {}


for row in tioman_rows:

    image_values = []

    sha_values = []

    for value in row.values():

        text = str(
            value or ""
        ).strip()

        lower = text.lower()

        if lower.endswith(
            (
                ".jpg",
                ".jpeg",
                ".png",
                ".tif",
                ".tiff",
                ".bmp",
                ".webp",
            )
        ):
            image_values.append(
                text
            )

        if re.fullmatch(
            r"[0-9a-fA-F]{64}",
            text,
        ):
            sha_values.append(
                text.lower()
            )


    require(
        len(image_values) >= 1,
        "Tioman manifest row lacks image filename.",
    )

    require(
        len(sha_values) >= 1,
        "Tioman manifest row lacks SHA256.",
    )


    filename_only = [
        value
        for value in image_values
        if normalize_member_name(
            value
        )
        == basename(
            value
        )
    ]


    if filename_only:

        filename = basename(
            filename_only[-1]
        )

    else:

        filename = basename(
            image_values[-1]
        )


    wanted_filenames.append(
        filename
    )

    wanted_sha_by_filename[
        filename
    ] = sha_values[0]


require(
    len(set(wanted_filenames))
    == 35,
    "Tioman filename set is not uniquely 35.",
)


print(
    "PASS  Exact Tioman manifest population recovered"
)


# ============================================================
# 8. TEXTUAL PROVENANCE SEARCH IN FROZEN PROJECT
# ============================================================

textual_hits = []


for current_root, dirnames, file_names in os.walk(
    PROJECT_ROOT
):

    dirnames.sort()
    file_names.sort()

    root = Path(
        current_root
    )

    for file_name in file_names:

        path = (
            root
            / file_name
        )

        if (
            path.suffix.lower()
            not in TEXT_SUFFIXES
        ):
            continue

        if (
            path.stat().st_size
            > MAX_TEXT_FILE_BYTES
        ):
            continue


        try:

            text = path.read_text(
                encoding="utf-8",
                errors="ignore",
            )

        except Exception:

            continue


        lower_text = (
            text.lower()
        )


        matched_terms = [
            term
            for term in SEARCH_TERMS
            if term.lower()
            in lower_text
        ]


        if not matched_terms:
            continue


        line_hits = []

        for line_number, line in enumerate(
            text.splitlines(),
            start=1,
        ):

            lower_line = (
                line.lower()
            )

            line_terms = [
                term
                for term in SEARCH_TERMS
                if term.lower()
                in lower_line
            ]


            if not line_terms:
                continue


            line_hits.append(
                {
                    "line_number":
                        line_number,

                    "matched_terms":
                        line_terms,

                    "text":
                        line[:1500],
                }
            )


            if len(
                line_hits
            ) >= 100:

                break


        textual_hits.append(
            {
                "path":
                    str(path),

                "relative_path":
                    str(
                        path.relative_to(
                            PROJECT_ROOT
                        )
                    ),

                "size_bytes":
                    path.stat().st_size,

                "sha256":
                    sha256_file(path),

                "matched_terms":
                    matched_terms,

                "line_hits":
                    line_hits,
            }
        )


print()

print("=" * 80)

print(
    "FROZEN PROJECT — TIOMAN SOURCE/PROVENANCE REFERENCES"
)

print("=" * 80)

print()


for item in textual_hits:

    print(
        item[
            "relative_path"
        ]
    )

    for hit in item[
        "line_hits"
    ]:

        print(
            f"  L{hit['line_number']}: "
            f"{hit['text']}"
        )

    print()


# ============================================================
# 9. DISCOVER MOUNTED ARCHIVES
# ============================================================

archive_files = []


for current_root, dirnames, file_names in os.walk(
    KAGGLE_INPUT_ROOT
):

    dirnames.sort()
    file_names.sort()

    root = Path(
        current_root
    )

    for file_name in file_names:

        path = (
            root
            / file_name
        )

        lower_name = (
            file_name.lower()
        )


        archive_like = (
            path.suffix.lower()
            in ARCHIVE_SUFFIXES
            or lower_name.endswith(
                ".tar.gz"
            )
            or lower_name.endswith(
                ".tar.bz2"
            )
            or lower_name.endswith(
                ".tar.xz"
            )
        )


        if not archive_like:
            continue


        archive_files.append(
            {
                "path":
                    path,

                "size_bytes":
                    path.stat().st_size,
            }
        )


print()

print("=" * 80)

print(
    "MOUNTED ARCHIVE INVENTORY"
)

print("=" * 80)

print()

print(
    f"Archive-like files found: {len(archive_files)}"
)


for index, item in enumerate(
    archive_files,
    start=1,
):

    print(
        f"{index:03d}  "
        f"{item['path']}  "
        f"bytes={item['size_bytes']}"
    )


# ============================================================
# 10. INSPECT ZIP/TAR MEMBER NAMES WITHOUT EXTRACTION
# ============================================================

archive_member_hits = []

archive_errors = []


wanted_set = set(
    wanted_filenames
)


for item in archive_files:

    path = item[
        "path"
    ]

    lower_name = (
        path.name.lower()
    )

    members = None

    archive_type = None


    try:

        if zipfile.is_zipfile(
            path
        ):

            archive_type = (
                "zip"
            )

            with zipfile.ZipFile(
                path,
                "r",
            ) as zf:

                members = (
                    zf.namelist()
                )


        elif tarfile.is_tarfile(
            path
        ):

            archive_type = (
                "tar"
            )

            with tarfile.open(
                path,
                "r:*",
            ) as tf:

                members = [
                    member.name
                    for member in tf.getmembers()
                    if member.isfile()
                ]


        else:

            # .7z/.rar cannot be safely assumed readable
            # with standard library. Record but do not mutate
            # environment or install anything.
            continue


    except Exception as exc:

        archive_errors.append(
            {
                "path":
                    str(path),

                "error_type":
                    type(exc).__name__,

                "error":
                    str(exc),
            }
        )

        continue


    if members is None:
        continue


    matched_tioman_images = []

    workbook_like_members = []

    source_phrase_members = []


    for member in members:

        normalized = (
            normalize_member_name(
                member
            )
        )

        member_base = (
            basename(
                normalized
            )
        )


        if member_base in wanted_set:

            matched_tioman_images.append(
                normalized
            )


        if Path(
            normalized
        ).suffix.lower() in {
            ".xlsx",
            ".xls",
            ".xlsm",
            ".csv",
            ".tsv",
        }:

            workbook_like_members.append(
                normalized
            )


        lower_member = (
            normalized.lower()
        )

        if (
            "ftir"
            in lower_member
            or "tioman"
            in lower_member
            or "microscopic"
            in lower_member
            or "supporting dataset"
            in lower_member
        ):

            source_phrase_members.append(
                normalized
            )


    if (
        matched_tioman_images
        or source_phrase_members
    ):

        archive_member_hits.append(
            {
                "archive_path":
                    str(path),

                "archive_type":
                    archive_type,

                "archive_size_bytes":
                    path.stat().st_size,

                "archive_sha256":
                    sha256_file(path),

                "total_members":
                    len(members),

                "matched_tioman_image_members":
                    matched_tioman_images,

                "matched_tioman_image_count":
                    len(
                        set(
                            basename(value)
                            for value
                            in matched_tioman_images
                        )
                    ),

                "workbook_like_members":
                    workbook_like_members,

                "source_phrase_members":
                    source_phrase_members[
                        :500
                    ],
            }
        )


print()

print("=" * 80)

print(
    "MOUNTED ARCHIVE TIOMAN HITS"
)

print("=" * 80)

print()


if archive_member_hits:

    for hit in archive_member_hits:

        print(
            f"ARCHIVE: {hit['archive_path']}"
        )

        print(
            f"  type: {hit['archive_type']}"
        )

        print(
            "  Tioman image basenames matched: "
            f"{hit['matched_tioman_image_count']}/35"
        )

        print(
            "  workbook-like members: "
            f"{len(hit['workbook_like_members'])}"
        )


        for member in (
            hit[
                "matched_tioman_image_members"
            ]
        ):

            print(
                f"    IMAGE  {member}"
            )


        for member in (
            hit[
                "workbook_like_members"
            ]
        ):

            print(
                f"    TABLE  {member}"
            )


        for member in (
            hit[
                "source_phrase_members"
            ]
        ):

            if (
                member
                not in
                hit[
                    "matched_tioman_image_members"
                ]
                and member
                not in
                hit[
                    "workbook_like_members"
                ]
            ):

                print(
                    f"    SOURCE {member}"
                )


        print()

else:

    print(
        "NONE"
    )


# ============================================================
# 11. SEARCH FOR ACQUISITION-LIKE FILES / DIRECTORY NAMES
# ============================================================

acquisition_name_hits = []


acquisition_terms = [
    "download",
    "acquisition",
    "source",
    "public_real",
    "public-real",
    "supplement",
    "supporting",
    "zenodo",
    "figshare",
    "mendeley",
    "dryad",
    "osf",
]


for current_root, dirnames, file_names in os.walk(
    PROJECT_ROOT
):

    dirnames.sort()
    file_names.sort()

    root = Path(
        current_root
    )


    for name in (
        list(dirnames)
        + list(file_names)
    ):

        lower_name = (
            name.lower()
        )

        matched = [
            term
            for term in acquisition_terms
            if term in lower_name
        ]


        if not matched:
            continue


        full_path = (
            root
            / name
        )


        acquisition_name_hits.append(
            {
                "path":
                    str(
                        full_path
                    ),

                "relative_path":
                    str(
                        full_path.relative_to(
                            PROJECT_ROOT
                        )
                    ),

                "is_file":
                    full_path.is_file(),

                "is_directory":
                    full_path.is_dir(),

                "matched_terms":
                    matched,
            }
        )


print()

print("=" * 80)

print(
    "ACQUISITION / SOURCE-NAMED PROJECT PATHS"
)

print("=" * 80)

print()


for item in acquisition_name_hits[
    :500
]:

    kind = (
        "FILE"
        if item[
            "is_file"
        ]
        else "DIR"
    )

    print(
        f"{kind:<4}  "
        f"{item['relative_path']}"
    )


# ============================================================
# 12. CLASSIFY CURRENT SOURCE STATE
# ============================================================

maximum_archive_image_matches = 0

best_archive = None


for hit in archive_member_hits:

    count = int(
        hit[
            "matched_tioman_image_count"
        ]
    )

    if (
        count
        > maximum_archive_image_matches
    ):

        maximum_archive_image_matches = (
            count
        )

        best_archive = hit


if (
    maximum_archive_image_matches
    == 35
):

    readiness = (
        "READY_FOR_TIOMAN_ARCHIVE_BYTE_RESOLUTION_AUDIT"
    )

    source_state = (
        "ALL_35_IMAGES_FOUND_INSIDE_MOUNTED_ARCHIVE"
    )


elif (
    maximum_archive_image_matches
    > 0
):

    readiness = (
        "TIOMAN_PARTIAL_ARCHIVE_SOURCE_REQUIRES_REVIEW"
    )

    source_state = (
        "PARTIAL_TIOMAN_ASSETS_FOUND_INSIDE_MOUNTED_ARCHIVE"
    )


else:

    readiness = (
        "TIOMAN_ORIGINAL_SOURCE_PACKAGE_MUST_BE_MOUNTED"
    )

    source_state = (
        "TIOMAN_RAW_SOURCE_PACKAGE_NOT_PRESENT_IN_CURRENT_KAGGLE_INPUTS"
    )


# ============================================================
# 13. WRITE AUDIT
# ============================================================

audit = {
    "audit_name":
        (
            "DASO-Net Tioman Source Provenance "
            "+ Archive Audit v1"
        ),

    "status":
        "PASS",

    "scientific_action":
        "READ_ONLY_SOURCE_PROVENANCE_ARCHIVE_INSPECTION",

    "inference_performed":
        False,

    "metrics_computed":
        False,

    "ftir_image_mapping_reconstructed":
        False,

    "archive_extraction_performed":
        False,

    "download_performed":
        False,

    "threshold_tuning":
        False,

    "calibration":
        False,

    "checkpoint_selection":
        False,

    "seed_selection":
        False,

    "prior_exact_asset_audit":
        {
            "path":
                str(
                    PRIOR_ASSET_AUDIT
                ),

            "sha256":
                prior_sha,

            "physical_images_resolved":
                0,

            "expected_images":
                35,
        },

    "frozen_manifest":
        {
            "path":
                str(
                    SOURCE_MANIFEST
                ),

            "sha256":
                sha256_file(
                    SOURCE_MANIFEST
                ),

            "source_id":
                TIOMAN_SOURCE_ID,

            "image_count":
                35,

            "filenames":
                wanted_filenames,

            "sha256_by_filename":
                wanted_sha_by_filename,
        },

    "textual_project_provenance_hits":
        textual_hits,

    "mounted_archives":
        [
            {
                "path":
                    str(
                        item[
                            "path"
                        ]
                    ),

                "size_bytes":
                    item[
                        "size_bytes"
                    ],
            }
            for item
            in archive_files
        ],

    "archive_member_hits":
        archive_member_hits,

    "archive_inspection_errors":
        archive_errors,

    "maximum_tioman_image_matches_in_single_archive":
        maximum_archive_image_matches,

    "best_archive":
        best_archive,

    "acquisition_named_project_paths":
        acquisition_name_hits,

    "current_source_state":
        source_state,

    "scientific_role":
        "LIMITED_FTIR_LINKED_EXTERNAL_ANCHOR",

    "required_before_inference":
        {
            "all_35_manifest_images_byte_resolved":
                True,

            "image_sha256_matches_frozen_manifest":
                True,

            "ftir_workbook_resolved":
                True,

            "ftir_workbook_identity_frozen":
                True,

            "17_of_35_unique_mapping_independently_audited":
                True,

            "inference_allowed_before_these_conditions":
                False,
        },

    "readiness":
        readiness,
}


write_json(
    AUDIT_JSON,
    audit,
)

audit_sha = sha256_file(
    AUDIT_JSON
)

AUDIT_SHA_RECORD.write_text(
    (
        f"{audit_sha}  "
        f"{AUDIT_JSON.name}\n"
    ),
    encoding="utf-8",
)


# ============================================================
# 14. FINAL REPORT
# ============================================================

print()

print("=" * 80)

print(
    "TIOMAN SOURCE PROVENANCE + ARCHIVE AUDIT: PASS"
)

print("=" * 80)

print()

print(
    "Frozen Tioman population:"
)

print(
    "  35 microscopy images"
)

print()

print(
    "Physical loose files currently mounted:"
)

print(
    "  0/35"
)

print()

print(
    "Mounted archives inspected:"
)

print(
    f"  {len(archive_files)}"
)

print()

print(
    "Best archive Tioman image coverage:"
)

print(
    f"  {maximum_archive_image_matches}/35"
)

if best_archive is not None:

    print()

    print(
        "Best archive:"
    )

    print(
        f"  {best_archive['archive_path']}"
    )

print()

print(
    "Frozen-project textual provenance files:"
)

print(
    f"  {len(textual_hits)}"
)

print()

print(
    "Current source state:"
)

print(
    f"  {source_state}"
)

print()

print(
    "Audit JSON:"
)

print(
    f"  {AUDIT_JSON}"
)

print()

print(
    "Audit SHA256:"
)

print(
    f"  {audit_sha}"
)

print()

print(
    "NEXT READINESS:"
)

print(
    f"  {readiness}"
)

print("=" * 80)

DASO-Net — TIOMAN SOURCE PROVENANCE + ARCHIVE AUDIT v1

Scientific action:
  READ-ONLY source-provenance / archive inspection

NO inference
NO metric computation
NO FTIR/image rematching
NO download
NO archive extraction
NO tuning/calibration/selection

PASS  Prior exact-asset audit identity
PASS  Exact Tioman manifest population recovered

FROZEN PROJECT — TIOMAN SOURCE/PROVENANCE REFERENCES

data/public_real_mp/v1_processed/public_real_mp_v1_source_specific_manifest.csv
  L1142: tioman2025_ftir,"_a001_Supporting Dataset _6fc16519\Supporting Dataset of FTIR, Microscopic_6fc16519\Microscopic Images\JUARA BEACH\JB_S01_B_F_1.jpg",JB_S01_B_F_1.jpg,3840,2160,21f2976df30e04380b48c54c0089d9bce50c5f5318e557d52dd55cffd9e1921a,,,False
  L1143: tioman2025_ftir,"_a001_Supporting Dataset _6fc16519\Supporting Dataset of FTIR, Microscopic_6fc16519\Microscopic Images\JUARA BEACH\JB_S02_A_M_1.jpg",JB_S02_A_M_1.jpg,3840,2160,1b57b5f4e8f4e136f2c41d02773543262e80424a5615ecaaab2c74034fdde05e,,,False
  L11

In [46]:
# ============================================================
# DASO-Net — TIOMAN CANONICAL SOURCE MOUNT VERIFICATION v1
#
# RUN ONLY AFTER:
#   Mendeley Data DOI 10.17632/v4z92mndjf.1
#   has been downloaded and mounted as a NEW Kaggle dataset.
#
# READ-ONLY.
#
# PURPOSE
# -------
# 1. Recover exact 35-image population from frozen manifest.
# 2. Search ALL /kaggle/input mounts for those images.
# 3. Accept an image only by exact SHA256 identity.
# 4. Require all 35/35.
# 5. Identify the dataset root(s) containing those exact bytes.
# 6. Inventory and hash Excel/CSV/FTIR assets in those roots.
# 7. Freeze the canonical mounted-source identity.
#
# NO inference
# NO metric computation
# NO FTIR/image matching
# NO threshold tuning
# NO calibration
# NO checkpoint/seed selection
# NO source modification
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import re
from collections import defaultdict
from pathlib import Path


# ============================================================
# 1. FROZEN INPUTS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/abinavprabhuv/"
    "dasonet-project/DASO-Net-Phase1A"
)

SOURCE_MANIFEST = (
    PROJECT_ROOT
    / "data/public_real_mp/v1_processed/"
      "public_real_mp_v1_source_specific_manifest.csv"
)

PRIOR_SOURCE_AUDIT = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "tioman_source_provenance_archive_audit_v1/"
    "TIOMAN_SOURCE_PROVENANCE_ARCHIVE_AUDIT_v1.json"
)

EXPECTED_PRIOR_SOURCE_AUDIT_SHA256 = (
    "bfb2276fdac30b093c94be0ba46b9b5e"
    "41ef90d3823d2c2b4a6b7fea4511efe5"
)

KAGGLE_INPUT_ROOT = Path(
    "/kaggle/input"
)

TIOMAN_SOURCE_ID = (
    "tioman2025_ftir"
)

CANONICAL_SOURCE = {
    "repository":
        "Mendeley Data",

    "dataset_title":
        (
            "Supporting Dataset of FTIR, Microscopic Images, "
            "Sample Descriptions, and Study Map for "
            "Microplastic Analysis from Juara, Salang, and "
            "Tulai, Tioman Island, Malaysia (August 2024)"
        ),

    "version":
        1,

    "doi":
        "10.17632/v4z92mndjf.1",

    "microscopy_source_doi":
        "10.17632/yf2nnnkx6m.2",

    "excel_source_doi":
        "10.17632/b43z3m82yx.1",

    "license":
        "CC BY 4.0",
}


# ============================================================
# 2. OUTPUT
# ============================================================

AUDIT_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "tioman_canonical_source_mount_verification_v1"
)

AUDIT_JSON = (
    AUDIT_DIR
    / "TIOMAN_CANONICAL_SOURCE_MOUNT_VERIFICATION_v1.json"
)

AUDIT_SHA_RECORD = (
    AUDIT_DIR
    / "TIOMAN_CANONICAL_SOURCE_MOUNT_VERIFICATION_v1.sha256.txt"
)


# ============================================================
# 3. HELPERS
# ============================================================

IMAGE_SUFFIXES = {
    ".jpg",
    ".jpeg",
    ".png",
    ".tif",
    ".tiff",
    ".bmp",
    ".webp",
}

TABLE_SUFFIXES = {
    ".xlsx",
    ".xls",
    ".xlsm",
    ".csv",
    ".tsv",
}

FTIR_LIKE_SUFFIXES = {
    ".txt",
    ".csv",
    ".csv",
    ".spc",
    ".jdx",
    ".dx",
    ".dpt",
    ".spa",
    ".0",
}

SHA256_RE = re.compile(
    r"^[0-9a-fA-F]{64}$"
)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def require(
    condition: bool,
    message: str,
) -> None:
    if not condition:
        raise RuntimeError(message)


def require_file(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_file(),
        f"{label} missing:\n{path}",
    )


def require_dir(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_dir(),
        f"{label} missing:\n{path}",
    )


def read_json(path: Path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path: Path,
    obj,
) -> None:
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def normalize_path_text(value: str) -> str:
    return (
        str(value)
        .replace("\\", "/")
        .strip()
    )


def infer_kaggle_dataset_root(
    path: Path,
):
    resolved = path.resolve()

    parts = resolved.parts

    # Common modern mount:
    # /kaggle/input/datasets/<owner>/<slug>/...
    if (
        len(parts) >= 6
        and parts[1:4]
        == (
            "kaggle",
            "input",
            "datasets",
        )
    ):
        return Path(
            *parts[:6]
        )

    # Conventional mount:
    # /kaggle/input/<slug>/...
    if (
        len(parts) >= 4
        and parts[1:3]
        == (
            "kaggle",
            "input",
        )
    ):
        return Path(
            *parts[:4]
        )

    return None


# ============================================================
# 4. BANNER / NON-OVERWRITE
# ============================================================

print("=" * 80)

print(
    "DASO-Net — TIOMAN CANONICAL SOURCE "
    "MOUNT VERIFICATION v1"
)

print("=" * 80)

print()

print(
    "Canonical source:"
)

print(
    "  Mendeley Data DOI 10.17632/v4z92mndjf.1"
)

print(
    "  Version 1"
)

print(
    "  License: CC BY 4.0"
)

print()

print("NO inference")
print("NO metric computation")
print("NO FTIR/image mapping")
print("NO tuning/calibration/selection")
print()


require_dir(
    PROJECT_ROOT,
    "Frozen DASO-Net project",
)

require_dir(
    KAGGLE_INPUT_ROOT,
    "Kaggle input root",
)

require_file(
    SOURCE_MANIFEST,
    "Frozen public-real source manifest",
)

require_file(
    PRIOR_SOURCE_AUDIT,
    "Prior Tioman source-provenance audit",
)


if AUDIT_DIR.exists():
    raise RuntimeError(
        "Canonical-source verification directory "
        "already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{AUDIT_DIR}"
    )


AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 5. VERIFY PRIOR SOURCE AUDIT
# ============================================================

prior_sha = sha256_file(
    PRIOR_SOURCE_AUDIT
)

require(
    prior_sha
    == EXPECTED_PRIOR_SOURCE_AUDIT_SHA256,
    (
        "Prior Tioman source audit SHA mismatch.\n"
        f"Expected: {EXPECTED_PRIOR_SOURCE_AUDIT_SHA256}\n"
        f"Actual:   {prior_sha}"
    ),
)

prior = read_json(
    PRIOR_SOURCE_AUDIT
)

require(
    prior.get("status")
    == "PASS",
    "Prior source audit != PASS.",
)

require(
    prior.get("readiness")
    == "TIOMAN_ORIGINAL_SOURCE_PACKAGE_MUST_BE_MOUNTED",
    "Unexpected prior source-audit readiness.",
)

print(
    "PASS  Prior Tioman source-provenance audit identity"
)


# ============================================================
# 6. RECOVER FROZEN 35-IMAGE MANIFEST POPULATION
# ============================================================

with SOURCE_MANIFEST.open(
    "r",
    encoding="utf-8-sig",
    newline="",
) as f:
    rows = list(
        csv.DictReader(f)
    )


tioman_rows = [
    row
    for row in rows
    if str(
        row.get(
            "source_id",
            ""
        )
    ).strip()
    == TIOMAN_SOURCE_ID
]


require(
    len(tioman_rows) == 35,
    (
        "Expected exactly 35 frozen Tioman rows; "
        f"found {len(tioman_rows)}."
    ),
)


frozen_records = []


for ordinal, row in enumerate(
    tioman_rows,
    start=1,
):
    image_values = []

    sha_values = []

    for column, value in row.items():
        text = str(
            value or ""
        ).strip()

        if Path(
            normalize_path_text(text)
        ).suffix.lower() in IMAGE_SUFFIXES:
            image_values.append(
                {
                    "column":
                        column,

                    "value":
                        text,

                    "basename":
                        Path(
                            normalize_path_text(text)
                        ).name,
                }
            )

        if SHA256_RE.fullmatch(text):
            sha_values.append(
                {
                    "column":
                        column,

                    "value":
                        text.lower(),
                }
            )


    require(
        len(image_values) >= 1,
        f"Tioman row {ordinal} has no image identity.",
    )

    require(
        len(sha_values) >= 1,
        f"Tioman row {ordinal} has no SHA256 identity.",
    )


    basename_only = [
        item
        for item in image_values
        if normalize_path_text(
            item["value"]
        )
        == item["basename"]
    ]


    chosen_image = (
        basename_only[-1]
        if basename_only
        else image_values[-1]
    )


    frozen_records.append(
        {
            "ordinal":
                ordinal,

            "filename":
                chosen_image["basename"],

            "sha256":
                sha_values[0]["value"],

            "manifest_row":
                row,
        }
    )


require(
    len(
        {
            item["filename"]
            for item in frozen_records
        }
    )
    == 35,
    "Frozen Tioman filenames are not unique.",
)

require(
    len(
        {
            item["sha256"]
            for item in frozen_records
        }
    )
    == 35,
    "Frozen Tioman image hashes are not unique.",
)


print(
    "PASS  Frozen exact Tioman population = 35"
)


# ============================================================
# 7. SCAN ALL CURRENT KAGGLE INPUTS BY BASENAME
# ============================================================

wanted_names = {
    item["filename"]
    for item in frozen_records
}

paths_by_basename = defaultdict(
    list
)

all_input_files = []


for current_root, dirnames, file_names in os.walk(
    KAGGLE_INPUT_ROOT
):
    dirnames.sort()
    file_names.sort()

    root = Path(
        current_root
    )

    for file_name in file_names:
        path = (
            root
            / file_name
        )

        all_input_files.append(
            path
        )

        if file_name in wanted_names:
            paths_by_basename[
                file_name
            ].append(
                path
            )


print(
    f"Mounted input files scanned: {len(all_input_files)}"
)


# ============================================================
# 8. EXACT SHA RESOLUTION
# ============================================================

resolved_records = []

unresolved_records = []

all_exact_paths = []


for frozen in frozen_records:
    filename = frozen[
        "filename"
    ]

    expected_sha = frozen[
        "sha256"
    ]

    candidates = sorted(
        paths_by_basename.get(
            filename,
            []
        ),
        key=lambda p: p.as_posix(),
    )

    candidate_details = []

    exact_matches = []


    for candidate in candidates:
        actual_sha = sha256_file(
            candidate
        )

        is_match = (
            actual_sha
            == expected_sha
        )

        candidate_details.append(
            {
                "path":
                    str(candidate),

                "size_bytes":
                    candidate.stat().st_size,

                "sha256":
                    actual_sha,

                "matches_frozen_manifest":
                    is_match,
            }
        )

        if is_match:
            exact_matches.append(
                candidate
            )


    resolved = (
        len(exact_matches)
        >= 1
    )


    if resolved:
        all_exact_paths.extend(
            exact_matches
        )

    else:
        unresolved_records.append(
            filename
        )


    resolved_records.append(
        {
            "ordinal":
                frozen["ordinal"],

            "filename":
                filename,

            "expected_sha256":
                expected_sha,

            "basename_candidate_count":
                len(candidates),

            "exact_sha_match_count":
                len(exact_matches),

            "candidate_files":
                candidate_details,

            "resolved":
                resolved,
        }
    )


resolved_count = sum(
    1
    for item in resolved_records
    if item["resolved"]
)


print()

print("=" * 80)

print(
    "EXACT TIOMAN IMAGE BYTE RESOLUTION"
)

print("=" * 80)

print()


for item in resolved_records:
    marker = (
        "PASS"
        if item["resolved"]
        else "MISSING"
    )

    print(
        f"{item['ordinal']:02d}  "
        f"{marker:<7} "
        f"{item['filename']}  "
        f"exact_sha_matches="
        f"{item['exact_sha_match_count']}"
    )


print()

print(
    f"Exact SHA-verified Tioman images: "
    f"{resolved_count}/35"
)


require(
    resolved_count == 35,
    (
        "Canonical Tioman source mount did not resolve "
        "all frozen microscopy bytes.\n"
        f"Resolved: {resolved_count}/35\n"
        f"Missing: {unresolved_records}\n\n"
        "Do NOT proceed to Tioman inference."
    ),
)


print(
    "PASS  All 35 Tioman microscopy images "
    "byte-identical to frozen manifest"
)


# ============================================================
# 9. RESOLVE CANONICAL DATASET ROOT(S)
# ============================================================

dataset_roots = sorted(
    {
        infer_kaggle_dataset_root(
            path
        )
        for path in all_exact_paths
        if infer_kaggle_dataset_root(
            path
        )
        is not None
    },
    key=lambda p: p.as_posix(),
)


require(
    len(dataset_roots) >= 1,
    "Could not infer Kaggle dataset root "
    "for exact Tioman assets.",
)


print()

print("=" * 80)

print(
    "DATASET ROOT(S) CONTAINING EXACT TIOMAN IMAGES"
)

print("=" * 80)

print()


for root in dataset_roots:
    print(
        root
    )


# ============================================================
# 10. INVENTORY RELEVANT ASSETS IN RESOLVED ROOT(S)
# ============================================================

table_assets = []

ftir_named_assets = []

all_root_assets = []


for dataset_root in dataset_roots:

    for path in sorted(
        [
            p
            for p in dataset_root.rglob("*")
            if p.is_file()
        ],
        key=lambda p: p.as_posix(),
    ):
        relative = path.relative_to(
            dataset_root
        )

        lower_path = (
            relative.as_posix()
            .lower()
        )

        asset = {
            "dataset_root":
                str(dataset_root),

            "path":
                str(path),

            "relative_path":
                relative.as_posix(),

            "suffix":
                path.suffix.lower(),

            "size_bytes":
                path.stat().st_size,
        }

        all_root_assets.append(
            asset
        )


        if (
            path.suffix.lower()
            in TABLE_SUFFIXES
        ):
            hashed = dict(
                asset
            )

            hashed[
                "sha256"
            ] = sha256_file(
                path
            )

            table_assets.append(
                hashed
            )


        if (
            "ftir" in lower_path
            or "spectrum" in lower_path
            or "spectra" in lower_path
        ):
            hashed = dict(
                asset
            )

            hashed[
                "sha256"
            ] = sha256_file(
                path
            )

            ftir_named_assets.append(
                hashed
            )


print()

print("=" * 80)

print(
    "TABLE / WORKBOOK CANDIDATES"
)

print("=" * 80)

print()


for item in table_assets:
    print(
        f"{item['relative_path']}\n"
        f"  bytes={item['size_bytes']}\n"
        f"  sha256={item['sha256']}"
    )


print()

print("=" * 80)

print(
    "FTIR-NAMED ASSETS"
)

print("=" * 80)

print()


for item in ftir_named_assets:
    print(
        f"{item['relative_path']}\n"
        f"  bytes={item['size_bytes']}\n"
        f"  sha256={item['sha256']}"
    )


# ============================================================
# 11. REQUIRE A SOURCE TABLE / WORKBOOK
# ============================================================

excel_assets = [
    item
    for item in table_assets
    if item[
        "suffix"
    ] in {
        ".xlsx",
        ".xls",
        ".xlsm",
    }
]


require(
    len(excel_assets) >= 1,
    (
        "All 35 image bytes are present, but no Excel "
        "workbook was found in the resolved source root(s).\n"
        "Do NOT reconstruct the 17/35 linkage yet."
    ),
)


print()

print(
    "PASS  Tioman source workbook candidate(s) present"
)


# ============================================================
# 12. SOURCE-ROOT INVENTORY HASH
#
# Hash a deterministic textual inventory of file paths/sizes.
# We do NOT hash every large FTIR file here unless already
# relevant; this is merely a mount-identity record.
# ============================================================

inventory_lines = []

for item in sorted(
    all_root_assets,
    key=lambda x: (
        x["dataset_root"],
        x["relative_path"],
    ),
):
    inventory_lines.append(
        (
            f"{item['dataset_root']}\t"
            f"{item['relative_path']}\t"
            f"{item['size_bytes']}"
        )
    )


inventory_text = (
    "\n".join(
        inventory_lines
    )
    + "\n"
)

inventory_sha = hashlib.sha256(
    inventory_text.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# 13. WRITE MOUNT VERIFICATION
# ============================================================

audit = {
    "audit_name":
        (
            "DASO-Net Tioman Canonical Source "
            "Mount Verification v1"
        ),

    "status":
        "PASS",

    "scientific_action":
        "READ_ONLY_CANONICAL_SOURCE_MOUNT_VERIFICATION",

    "inference_performed":
        False,

    "metrics_computed":
        False,

    "ftir_image_mapping_reconstructed":
        False,

    "threshold_tuning":
        False,

    "calibration":
        False,

    "checkpoint_selection":
        False,

    "seed_selection":
        False,

    "canonical_source":
        CANONICAL_SOURCE,

    "prior_source_audit":
        {
            "path":
                str(
                    PRIOR_SOURCE_AUDIT
                ),

            "sha256":
                prior_sha,
        },

    "frozen_source_manifest":
        {
            "path":
                str(
                    SOURCE_MANIFEST
                ),

            "sha256":
                sha256_file(
                    SOURCE_MANIFEST
                ),

            "source_id":
                TIOMAN_SOURCE_ID,

            "expected_images":
                35,
        },

    "microscopy_byte_resolution":
        {
            "expected":
                35,

            "resolved":
                resolved_count,

            "all_35_sha256_match_frozen_manifest":
                True,

            "records":
                resolved_records,
        },

    "resolved_dataset_roots":
        [
            str(root)
            for root in dataset_roots
        ],

    "dataset_root_inventory":
        {
            "file_count":
                len(
                    all_root_assets
                ),

            "path_size_inventory_sha256":
                inventory_sha,
        },

    "table_workbook_assets":
        table_assets,

    "excel_workbook_assets":
        excel_assets,

    "ftir_named_assets":
        ftir_named_assets,

    "scientific_role":
        "LIMITED_FTIR_LINKED_EXTERNAL_ANCHOR",

    "scientific_constraints":
        {
            "tioman_training":
                False,

            "tioman_crpa":
                False,

            "tioman_calibration":
                False,

            "tioman_threshold_tuning":
                False,

            "checkpoint_selection":
                False,

            "seed_selection":
                False,

            "inference_before_mapping_audit":
                False,
        },

    "readiness":
        "READY_FOR_TIOMAN_FTIR_MAPPING_AUDIT",
}


write_json(
    AUDIT_JSON,
    audit,
)

audit_sha = sha256_file(
    AUDIT_JSON
)

AUDIT_SHA_RECORD.write_text(
    (
        f"{audit_sha}  "
        f"{AUDIT_JSON.name}\n"
    ),
    encoding="utf-8",
)


# ============================================================
# 14. FINAL REPORT
# ============================================================

print()

print("=" * 80)

print(
    "TIOMAN CANONICAL SOURCE MOUNT "
    "VERIFICATION: PASS"
)

print("=" * 80)

print()

print(
    "Canonical DOI:"
)

print(
    "  10.17632/v4z92mndjf.1"
)

print()

print(
    "License:"
)

print(
    "  CC BY 4.0"
)

print()

print(
    "Frozen microscopy images:"
)

print(
    "  35/35 exact SHA256 matches"
)

print()

print(
    "Resolved source root(s):"
)

for root in dataset_roots:
    print(
        f"  {root}"
    )

print()

print(
    "Excel workbook candidates:"
)

for item in excel_assets:
    print(
        f"  {item['path']}"
    )

    print(
        f"    SHA256: {item['sha256']}"
    )

print()

print(
    "FTIR-named assets:"
)

print(
    f"  {len(ftir_named_assets)}"
)

print()

print(
    "Mount verification JSON:"
)

print(
    f"  {AUDIT_JSON}"
)

print()

print(
    "Mount verification SHA256:"
)

print(
    f"  {audit_sha}"
)

print()

print(
    "NEXT READINESS:"
)

print(
    "  READY_FOR_TIOMAN_FTIR_MAPPING_AUDIT"
)

print("=" * 80)

DASO-Net — TIOMAN CANONICAL SOURCE MOUNT VERIFICATION v1

Canonical source:
  Mendeley Data DOI 10.17632/v4z92mndjf.1
  Version 1
  License: CC BY 4.0

NO inference
NO metric computation
NO FTIR/image mapping
NO tuning/calibration/selection

PASS  Prior Tioman source-provenance audit identity
PASS  Frozen exact Tioman population = 35
Mounted input files scanned: 5463

EXACT TIOMAN IMAGE BYTE RESOLUTION

01  MISSING JB_S01_B_F_1.jpg  exact_sha_matches=0
02  MISSING JB_S02_A_M_1.jpg  exact_sha_matches=0
03  MISSING JB_S05_A_F_1.jpg  exact_sha_matches=0
04  MISSING JB_S05_A_M_1.jpg  exact_sha_matches=0
05  MISSING JB_S05_B_M_1.jpg  exact_sha_matches=0
06  MISSING JB_S06_B_M_1.jpg  exact_sha_matches=0
07  MISSING SLB_S01_A_M_1.jpg  exact_sha_matches=0
08  MISSING SLB_S02_A_F_1.jpg  exact_sha_matches=0
09  MISSING SLB_S03_A_L_1.jpg  exact_sha_matches=0
10  MISSING SLB_S03_A_M_1.jpg  exact_sha_matches=0
11  MISSING SLB_S03_A_S_1.jpg  exact_sha_matches=0
12  MISSING SLB_S04_B_L_1.jpg  exact_s

RuntimeError: Canonical Tioman source mount did not resolve all frozen microscopy bytes.
Resolved: 0/35
Missing: ['JB_S01_B_F_1.jpg', 'JB_S02_A_M_1.jpg', 'JB_S05_A_F_1.jpg', 'JB_S05_A_M_1.jpg', 'JB_S05_B_M_1.jpg', 'JB_S06_B_M_1.jpg', 'SLB_S01_A_M_1.jpg', 'SLB_S02_A_F_1.jpg', 'SLB_S03_A_L_1.jpg', 'SLB_S03_A_M_1.jpg', 'SLB_S03_A_S_1.jpg', 'SLB_S04_B_L_1.jpg', 'SLB_S05_B_S_1.jpg', 'TB_S01_A_F_1.jpg', 'TB_S01_A_F_2.jpg', 'TB_S01_A_F_3.jpg', 'TB_S01_A_F_4.jpg', 'TB_S01_A_F_5.jpg', 'TB_S01_B_F_1.jpg', 'TB_S01_B_F_2.jpg', 'TB_S02_A_M_1.jpg', 'TB_S02_A_M_2.jpg', 'TB_S02_B_M_1.jpg', 'TB_S03_A_M_1.jpg', 'TB_S04_A_F_1.jpg', 'TB_S04_A_F_2.jpg', 'TB_S04_A_F_3.jpg', 'TB_S04_B_F_1.jpg', 'TB_S04_B_M_1.jpg', 'TB_S05_A_F_1.jpg', 'TB_S05_A_F_2.jpg', 'TB_S05_B_F_1.jpg', 'TB_S05_B_F_2.jpg', 'TB_S05_B_F_3.jpg', 'TB_S05_B_F_4.jpg']

Do NOT proceed to Tioman inference.

In [47]:
# ============================================================
# DASO-Net — PROJECT CONTINUATION CHECKPOINT FREEZE v1
#
# PURPOSE
# -------
# Create one portable continuation bundle before moving to a
# fresh Kaggle notebook.
#
# INCLUDED
# --------
# 1. Three exact final terminal checkpoints
# 2. Phase5B code tree required for model/evaluator loading
# 3. Non-checkpoint provenance files from the three final
#    training seed directories
# 4. Final preserved RSC evaluation ZIP + SHA record
# 5. Final RSC preservation record + file manifest
# 6. All Tioman audits completed so far
# 7. Frozen public-real source-specific manifest
# 8. Continuation state / provenance / file hashes
#
# EXCLUDED
# --------
# - redundant historical training snapshot .pt files
# - Kaggle input datasets themselves
# - raw RSC source dataset
# - raw Tioman source dataset (not mounted yet)
#
# THIS CELL:
#   NO model inference
#   NO scientific metric computation/recomputation
#   NO tuning
#   NO calibration
#   NO checkpoint/seed selection
#   NO modification of existing frozen artifacts
#
# ============================================================

from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import platform
import shutil
import sys
import zipfile
from pathlib import Path


# ============================================================
# 1. SOURCE PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/abinavprabhuv/"
    "dasonet-project/DASO-Net-Phase1A"
)

PUBLIC_REAL_MANIFEST = (
    PROJECT_ROOT
    / "data/public_real_mp/v1_processed/"
      "public_real_mp_v1_source_specific_manifest.csv"
)


CODE_ROOT = Path(
    "/kaggle/working/DASO-Net-Phase5B-KaggleTraining"
)


SEED_SPECS = [
    {
        "seed_index": 1,
        "seed": 1782656963,
        "seed_dir": Path(
            "/kaggle/working/dasonet_phase5b/"
            "dasonet_final_seed01_s1782656963"
        ),
        "checkpoint": Path(
            "/kaggle/working/dasonet_phase5b/"
            "dasonet_final_seed01_s1782656963/"
            "snapshots/epoch_011_opt_002750.pt"
        ),
        "checkpoint_sha256":
            (
                "e4c4b79ad4b8c6e18ba2b1e3d6cbe796"
                "c9cc301bd6fcc7c596fc290ad3a7e81d"
            ),
        "operating_threshold":
            0.30,
    },

    {
        "seed_index": 2,
        "seed": 901395089,
        "seed_dir": Path(
            "/kaggle/working/dasonet_phase5b/"
            "dasonet_final_seed02_s901395089"
        ),
        "checkpoint": Path(
            "/kaggle/working/dasonet_phase5b/"
            "dasonet_final_seed02_s901395089/"
            "snapshots/epoch_010_opt_002750.pt"
        ),
        "checkpoint_sha256":
            (
                "19cdb834a46240916b2c8023e8c3afe573"
                "8d9c7cd5d8f818c62ad1e289df1ad2"
            ),
        "operating_threshold":
            0.35,
    },

    {
        "seed_index": 3,
        "seed": 252916394,
        "seed_dir": Path(
            "/kaggle/working/dasonet_phase5b/"
            "dasonet_final_seed03_s252916394"
        ),
        "checkpoint": Path(
            "/kaggle/working/dasonet_phase5b/"
            "dasonet_final_seed03_s252916394/"
            "snapshots/epoch_010_opt_002750.pt"
        ),
        "checkpoint_sha256":
            (
                "be7b7941f2f62179e15074aa973bf6da0"
                "eb9018ec9133486512bd1f87153f447"
            ),
        "operating_threshold":
            0.35,
    },
]


# ------------------------------------------------------------
# RSC final frozen preservation
# ------------------------------------------------------------

RSC_PRESERVATION_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "rsc_formal_recovery_results_freeze_v1"
)

RSC_FINAL_RECORD = (
    RSC_PRESERVATION_DIR
    / "RSC_FORMAL_RECOVERY_RESULTS_FREEZE_v1.json"
)

EXPECTED_RSC_FINAL_RECORD_SHA256 = (
    "c560d68ffd39957bca09b94ae45b7e202"
    "43f066bdef061b58ca0a11347700942"
)

RSC_FILE_MANIFEST = (
    RSC_PRESERVATION_DIR
    / "RSC_FORMAL_RECOVERY_RESULTS_FILE_MANIFEST_v1.json"
)

EXPECTED_RSC_FILE_MANIFEST_SHA256 = (
    "9265e339eaf95c2de0ace13e8f11502a"
    "b89e306e0beaadb1f76d1f7b21b7cf87"
)

RSC_PRESERVATION_ZIP = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "RSC_FORMAL_RECOVERY_RESULTS_FREEZE_v1.zip"
)

EXPECTED_RSC_PRESERVATION_ZIP_SHA256 = (
    "ef1080bf1a18a73e14364283bc520c980"
    "549c37d911998792d36ddd969d76f5b"
)

RSC_PRESERVATION_ZIP_SHA_RECORD = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "RSC_FORMAL_RECOVERY_RESULTS_FREEZE_v1.zip.sha256.txt"
)


# ------------------------------------------------------------
# Tioman completed audit history
# ------------------------------------------------------------

TIOMAN_AUDIT_DIRS = [
    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "tioman_external_asset_discovery_audit_v1"
    ),

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "tioman_exact_asset_resolution_audit_v1"
    ),

    Path(
        "/kaggle/working/dasonet_final_analysis/"
        "tioman_source_provenance_archive_audit_v1"
    ),
]


TIOMAN_DISCOVERY_JSON = (
    TIOMAN_AUDIT_DIRS[0]
    / "TIOMAN_EXTERNAL_ASSET_DISCOVERY_AUDIT_v1.json"
)

EXPECTED_TIOMAN_DISCOVERY_SHA256 = (
    "68b58011a322f3860e2c58358ea08d176"
    "857349d4371b05c2ca8e0fefcc9dc92"
)


TIOMAN_EXACT_ASSET_JSON = (
    TIOMAN_AUDIT_DIRS[1]
    / "TIOMAN_EXACT_ASSET_RESOLUTION_AUDIT_v1.json"
)

EXPECTED_TIOMAN_EXACT_ASSET_SHA256 = (
    "cf99b32c23659182d1de879559e626550"
    "72dc68018181f0e2c623c62cc8d09f7"
)


TIOMAN_SOURCE_PROVENANCE_JSON = (
    TIOMAN_AUDIT_DIRS[2]
    / "TIOMAN_SOURCE_PROVENANCE_ARCHIVE_AUDIT_v1.json"
)

EXPECTED_TIOMAN_SOURCE_PROVENANCE_SHA256 = (
    "bfb2276fdac30b093c94be0ba46b9b5e"
    "41ef90d3823d2c2b4a6b7fea4511efe5"
)


# The failed canonical verification created this directory
# before terminating. Preserve its existence/state if present.
FAILED_TIOMAN_MOUNT_DIR = Path(
    "/kaggle/working/dasonet_final_analysis/"
    "tioman_canonical_source_mount_verification_v1"
)


# ============================================================
# 2. OUTPUT
# ============================================================

CHECKPOINT_DIR = Path(
    "/kaggle/working/dasonet_continuation_checkpoint_v1"
)

BUNDLE_ROOT = (
    CHECKPOINT_DIR
    / "bundle"
)

CHECKPOINTS_OUT = (
    BUNDLE_ROOT
    / "final_checkpoints"
)

CODE_OUT = (
    BUNDLE_ROOT
    / "phase5b_code"
)

TRAINING_PROVENANCE_OUT = (
    BUNDLE_ROOT
    / "training_seed_provenance"
)

RSC_OUT = (
    BUNDLE_ROOT
    / "rsc_final"
)

TIOMAN_OUT = (
    BUNDLE_ROOT
    / "tioman_state"
)

FROZEN_MANIFEST_OUT = (
    BUNDLE_ROOT
    / "frozen_manifests"
)

CONTINUATION_STATE_JSON = (
    BUNDLE_ROOT
    / "DASONET_CONTINUATION_STATE_v1.json"
)

FILE_MANIFEST_JSON = (
    CHECKPOINT_DIR
    / "DASONET_CONTINUATION_CHECKPOINT_FILE_MANIFEST_v1.json"
)

FILE_MANIFEST_SHA_RECORD = (
    CHECKPOINT_DIR
    / "DASONET_CONTINUATION_CHECKPOINT_FILE_MANIFEST_v1.sha256.txt"
)

FINAL_FREEZE_JSON = (
    CHECKPOINT_DIR
    / "DASONET_CONTINUATION_CHECKPOINT_FREEZE_v1.json"
)

FINAL_FREEZE_SHA_RECORD = (
    CHECKPOINT_DIR
    / "DASONET_CONTINUATION_CHECKPOINT_FREEZE_v1.sha256.txt"
)

PORTABLE_ZIP = Path(
    "/kaggle/working/"
    "DASO_NET_CONTINUATION_CHECKPOINT_v1.zip"
)

PORTABLE_ZIP_SHA_RECORD = Path(
    "/kaggle/working/"
    "DASO_NET_CONTINUATION_CHECKPOINT_v1.zip.sha256.txt"
)


# ============================================================
# 3. HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def require(
    condition: bool,
    message: str,
) -> None:
    if not condition:
        raise RuntimeError(message)


def require_file(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_file(),
        f"{label} missing:\n{path}",
    )


def require_dir(
    path: Path,
    label: str,
) -> None:
    require(
        path.is_dir(),
        f"{label} missing:\n{path}",
    )


def read_json(path: Path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path: Path,
    obj,
) -> None:
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def all_files(root: Path):
    return sorted(
        [
            path
            for path in root.rglob("*")
            if path.is_file()
        ],
        key=lambda p: p.as_posix(),
    )


def copy_file_verified(
    source: Path,
    destination: Path,
):
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        source,
        destination,
    )

    source_sha = sha256_file(
        source
    )

    destination_sha = sha256_file(
        destination
    )

    require(
        source_sha
        == destination_sha,
        (
            "Copy byte-identity failure.\n"
            f"Source:      {source}\n"
            f"Destination: {destination}"
        ),
    )

    return source_sha


def package_version(name: str):
    try:
        return importlib.metadata.version(
            name
        )
    except Exception:
        return None


# ============================================================
# 4. BANNER / NON-OVERWRITE
# ============================================================

print("=" * 80)

print(
    "DASO-Net — PROJECT CONTINUATION "
    "CHECKPOINT FREEZE v1"
)

print("=" * 80)

print()

print(
    "NO model inference"
)

print(
    "NO scientific metric computation/recomputation"
)

print(
    "NO threshold tuning"
)

print(
    "NO calibration"
)

print(
    "NO checkpoint/seed selection"
)

print()


if CHECKPOINT_DIR.exists():
    raise RuntimeError(
        "Continuation checkpoint directory already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{CHECKPOINT_DIR}"
    )


if PORTABLE_ZIP.exists():
    raise RuntimeError(
        "Continuation ZIP already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{PORTABLE_ZIP}"
    )


if PORTABLE_ZIP_SHA_RECORD.exists():
    raise RuntimeError(
        "Continuation ZIP SHA record already exists.\n"
        "Preserve it rather than overwriting it:\n"
        f"{PORTABLE_ZIP_SHA_RECORD}"
    )


# ============================================================
# 5. VERIFY REQUIRED INPUTS
# ============================================================

require_dir(
    PROJECT_ROOT,
    "Frozen project",
)

require_file(
    PUBLIC_REAL_MANIFEST,
    "Frozen public-real manifest",
)

require_dir(
    CODE_ROOT,
    "Phase5B code root",
)


for spec in SEED_SPECS:
    require_dir(
        spec["seed_dir"],
        f"Seed-{spec['seed_index']} final directory",
    )

    require_file(
        spec["checkpoint"],
        f"Seed-{spec['seed_index']} terminal checkpoint",
    )

    actual_sha = sha256_file(
        spec["checkpoint"]
    )

    require(
        actual_sha
        == spec["checkpoint_sha256"],
        (
            f"Seed-{spec['seed_index']} terminal checkpoint "
            "SHA mismatch.\n"
            f"Expected: {spec['checkpoint_sha256']}\n"
            f"Actual:   {actual_sha}"
        ),
    )


print(
    "PASS  All three terminal checkpoint identities"
)


require_file(
    RSC_FINAL_RECORD,
    "Final RSC preservation record",
)

require(
    sha256_file(
        RSC_FINAL_RECORD
    )
    == EXPECTED_RSC_FINAL_RECORD_SHA256,
    "Final RSC preservation record SHA mismatch.",
)


require_file(
    RSC_FILE_MANIFEST,
    "Final RSC preservation file manifest",
)

require(
    sha256_file(
        RSC_FILE_MANIFEST
    )
    == EXPECTED_RSC_FILE_MANIFEST_SHA256,
    "RSC preservation file-manifest SHA mismatch.",
)


require_file(
    RSC_PRESERVATION_ZIP,
    "Final RSC preservation ZIP",
)

require(
    sha256_file(
        RSC_PRESERVATION_ZIP
    )
    == EXPECTED_RSC_PRESERVATION_ZIP_SHA256,
    "Final RSC preservation ZIP SHA mismatch.",
)


require_file(
    RSC_PRESERVATION_ZIP_SHA_RECORD,
    "RSC preservation ZIP SHA record",
)


print(
    "PASS  Final RSC preservation identities"
)


for path, expected_sha, label in [
    (
        TIOMAN_DISCOVERY_JSON,
        EXPECTED_TIOMAN_DISCOVERY_SHA256,
        "Tioman discovery audit",
    ),
    (
        TIOMAN_EXACT_ASSET_JSON,
        EXPECTED_TIOMAN_EXACT_ASSET_SHA256,
        "Tioman exact-asset audit",
    ),
    (
        TIOMAN_SOURCE_PROVENANCE_JSON,
        EXPECTED_TIOMAN_SOURCE_PROVENANCE_SHA256,
        "Tioman source-provenance audit",
    ),
]:
    require_file(
        path,
        label,
    )

    require(
        sha256_file(
            path
        )
        == expected_sha,
        f"{label} SHA mismatch.",
    )


print(
    "PASS  Completed Tioman audit identities"
)


# ============================================================
# 6. CREATE BUNDLE DIRECTORIES
# ============================================================

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

BUNDLE_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

CHECKPOINTS_OUT.mkdir(
    parents=True,
    exist_ok=False,
)

CODE_OUT.mkdir(
    parents=True,
    exist_ok=False,
)

TRAINING_PROVENANCE_OUT.mkdir(
    parents=True,
    exist_ok=False,
)

RSC_OUT.mkdir(
    parents=True,
    exist_ok=False,
)

TIOMAN_OUT.mkdir(
    parents=True,
    exist_ok=False,
)

FROZEN_MANIFEST_OUT.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 7. COPY THREE EXACT TERMINAL CHECKPOINTS
# ============================================================

checkpoint_records = []


for spec in SEED_SPECS:
    destination = (
        CHECKPOINTS_OUT
        / (
            f"seed{spec['seed_index']:02d}_"
            f"s{spec['seed']}_"
            "opt002750.pt"
        )
    )

    copied_sha = copy_file_verified(
        spec["checkpoint"],
        destination,
    )

    require(
        copied_sha
        == spec["checkpoint_sha256"],
        f"Seed-{spec['seed_index']} bundle SHA mismatch.",
    )

    checkpoint_records.append(
        {
            "seed_index":
                spec["seed_index"],

            "seed":
                spec["seed"],

            "successful_optimizer_updates":
                2750,

            "operating_threshold":
                spec["operating_threshold"],

            "original_path":
                str(
                    spec["checkpoint"]
                ),

            "bundle_path":
                str(
                    destination.relative_to(
                        BUNDLE_ROOT
                    )
                ),

            "sha256":
                copied_sha,
        }
    )


print(
    "PASS  Three terminal checkpoints copied byte-identically"
)


# ============================================================
# 8. COPY COMPLETE PHASE5B CODE TREE
#
# Skip only transient Python caches/checkpoint folders.
# This code tree is required for model construction / decoder
# imports in the fresh notebook.
# ============================================================

skip_dir_names = {
    "__pycache__",
    ".ipynb_checkpoints",
    ".pytest_cache",
}

code_file_count = 0


for source in all_files(
    CODE_ROOT
):
    relative = source.relative_to(
        CODE_ROOT
    )

    if any(
        part in skip_dir_names
        for part in relative.parts
    ):
        continue

    destination = (
        CODE_OUT
        / relative
    )

    copy_file_verified(
        source,
        destination,
    )

    code_file_count += 1


require(
    code_file_count > 0,
    "No Phase5B code files were copied.",
)


print(
    f"PASS  Phase5B code tree copied "
    f"({code_file_count} files)"
)


# ============================================================
# 9. COPY FINAL-SEED NON-PT PROVENANCE FILES
#
# We deliberately do NOT duplicate historical snapshot .pt files.
# The exact terminal .pt for each seed is already copied above.
# ============================================================

seed_provenance_records = []


for spec in SEED_SPECS:
    seed_out = (
        TRAINING_PROVENANCE_OUT
        / (
            f"seed{spec['seed_index']:02d}_"
            f"s{spec['seed']}"
        )
    )

    seed_out.mkdir(
        parents=True,
        exist_ok=False,
    )

    copied_files = []


    for source in all_files(
        spec["seed_dir"]
    ):
        if source.suffix.lower() in {
            ".pt",
            ".pth",
            ".ckpt",
        }:
            continue

        relative = source.relative_to(
            spec["seed_dir"]
        )

        destination = (
            seed_out
            / relative
        )

        copied_sha = copy_file_verified(
            source,
            destination,
        )

        copied_files.append(
            {
                "relative_path":
                    relative.as_posix(),

                "sha256":
                    copied_sha,

                "size_bytes":
                    source.stat().st_size,
            }
        )


    seed_provenance_records.append(
        {
            "seed_index":
                spec["seed_index"],

            "seed":
                spec["seed"],

            "source_directory":
                str(
                    spec["seed_dir"]
                ),

            "non_checkpoint_file_count":
                len(
                    copied_files
                ),

            "files":
                copied_files,
        }
    )


print(
    "PASS  Final-seed non-checkpoint provenance copied"
)


# ============================================================
# 10. COPY FINAL RSC PRESERVATION PACKAGE
# ============================================================

copy_file_verified(
    RSC_PRESERVATION_ZIP,
    (
        RSC_OUT
        / RSC_PRESERVATION_ZIP.name
    ),
)

copy_file_verified(
    RSC_PRESERVATION_ZIP_SHA_RECORD,
    (
        RSC_OUT
        / RSC_PRESERVATION_ZIP_SHA_RECORD.name
    ),
)

copy_file_verified(
    RSC_FINAL_RECORD,
    (
        RSC_OUT
        / RSC_FINAL_RECORD.name
    ),
)

copy_file_verified(
    RSC_FILE_MANIFEST,
    (
        RSC_OUT
        / RSC_FILE_MANIFEST.name
    ),
)


print(
    "PASS  Final frozen RSC package copied"
)


# ============================================================
# 11. COPY TIOMAN AUDIT HISTORY
# ============================================================

tioman_audit_records = []


for source_dir in TIOMAN_AUDIT_DIRS:
    destination_dir = (
        TIOMAN_OUT
        / source_dir.name
    )

    shutil.copytree(
        source_dir,
        destination_dir,
        symlinks=False,
    )

    source_files = all_files(
        source_dir
    )

    copied_files = all_files(
        destination_dir
    )

    require(
        [
            path.relative_to(
                source_dir
            ).as_posix()
            for path in source_files
        ]
        ==
        [
            path.relative_to(
                destination_dir
            ).as_posix()
            for path in copied_files
        ],
        (
            "Tioman audit copy inventory mismatch:\n"
            f"{source_dir}"
        ),
    )


    for source_file in source_files:
        relative = source_file.relative_to(
            source_dir
        )

        copied_file = (
            destination_dir
            / relative
        )

        require(
            sha256_file(
                source_file
            )
            == sha256_file(
                copied_file
            ),
            (
                "Tioman audit byte-copy failure:\n"
                f"{source_file}"
            ),
        )


    tioman_audit_records.append(
        {
            "name":
                source_dir.name,

            "source_path":
                str(
                    source_dir
                ),

            "bundle_path":
                str(
                    destination_dir.relative_to(
                        BUNDLE_ROOT
                    )
                ),

            "file_count":
                len(
                    source_files
                ),
        }
    )


# Preserve evidence that the first mount-verification attempt
# failed before a verification JSON could be completed.
failed_mount_marker = (
    TIOMAN_OUT
    / "FAILED_CANONICAL_MOUNT_VERIFICATION_v1.txt"
)

failed_mount_marker.write_text(
    (
        "DASO-Net Tioman canonical source mount verification v1\n"
        "\n"
        "STATUS: INFRASTRUCTURE / MOUNT VISIBILITY FAILURE\n"
        "\n"
        "Observed in original notebook:\n"
        "  /kaggle/input files scanned: 5463\n"
        "  exact frozen Tioman image bytes resolved: 0/35\n"
        "\n"
        "The original output directory was created before the cell "
        "terminated. No Tioman inference was performed.\n"
        "\n"
        f"Original directory existed: "
        f"{FAILED_TIOMAN_MOUNT_DIR.exists()}\n"
        f"Original directory path: {FAILED_TIOMAN_MOUNT_DIR}\n"
    ),
    encoding="utf-8",
)


print(
    "PASS  Tioman audit history preserved"
)


# ============================================================
# 12. COPY FROZEN SOURCE-SPECIFIC MANIFEST
# ============================================================

public_real_manifest_sha = copy_file_verified(
    PUBLIC_REAL_MANIFEST,
    (
        FROZEN_MANIFEST_OUT
        / PUBLIC_REAL_MANIFEST.name
    ),
)


print(
    "PASS  Frozen public-real source manifest copied"
)


# ============================================================
# 13. ENVIRONMENT RECORD
# ============================================================

environment = {
    "python":
        sys.version,

    "platform":
        platform.platform(),

    "packages":
        {
            "torch":
                package_version(
                    "torch"
                ),

            "torchvision":
                package_version(
                    "torchvision"
                ),

            "numpy":
                package_version(
                    "numpy"
                ),

            "pandas":
                package_version(
                    "pandas"
                ),

            "opencv_python":
                (
                    package_version(
                        "opencv-python"
                    )
                    or package_version(
                        "opencv-python-headless"
                    )
                ),

            "pillow":
                package_version(
                    "Pillow"
                ),
        },
}


# ============================================================
# 14. CONTINUATION STATE
# ============================================================

continuation_state = {
    "checkpoint_name":
        "DASO-Net Continuation Checkpoint v1",

    "status":
        "READY_FOR_FRESH_NOTEBOOK",

    "scientific_action":
        "PORTABLE_PROJECT_CONTINUATION_CHECKPOINT",

    "inference_performed":
        False,

    "scientific_metrics_computed":
        False,

    "current_project_state":
        {
            "phase5_training":
                "COMPLETE",

            "formal_synthetic_test":
                "FROZEN_COMPLETE",

            "formal_clean_fruitbg_test":
                "FROZEN_COMPLETE",

            "formal_rsc_external_detection":
                "FINAL_PRESERVED_COMPLETE",

            "tioman_external_anchor":
                (
                    "BLOCKED_ONLY_ON_CANONICAL_SOURCE_"
                    "MOUNT_AND_MAPPING_AUDIT"
                ),
        },

    "next_action":
        {
            "do_not_run_old_pending_resume01":
                True,

            "fresh_notebook":
                True,

            "attach_frozen_project_dataset":
                True,

            "attach_this_continuation_bundle":
                True,

            "attach_canonical_tioman_source_dataset":
                True,

            "next_scientific_step":
                (
                    "verify all 35 Tioman microscopy images "
                    "against frozen SHA256 identities, then audit "
                    "the FTIR/workbook linkage before any inference"
                ),
        },

    "final_models":
        checkpoint_records,

    "decoder_contract":
        {
            "decode_min_score":
                0.05,

            "topk":
                100,

            "nms_iou":
                0.50,

            "mask_threshold":
                0.50,

            "local_peak_kernel":
                3,
        },

    "rsc_final":
        {
            "status":
                "CLOSED",

            "primary_population":
                "official authors' RSC test-250",

            "scientific_task":
                "class-agnostic bounding-box detection",

            "primary_box_ap50_mean":
                0.030172,

            "primary_box_ap50_sample_sd":
                0.037769,

            "primary_box_ap75_mean":
                0.001027,

            "primary_box_ap75_sample_sd":
                0.001145,

            "primary_box_ap50_95_mean":
                0.005761,

            "primary_box_ap50_95_sample_sd":
                0.007662,

            "primary_precision_mean":
                0.081214,

            "primary_precision_sample_sd":
                0.123416,

            "primary_recall_mean":
                0.061224,

            "primary_recall_sample_sd":
                0.104441,

            "primary_f1_mean":
                0.067937,

            "primary_f1_sample_sd":
                0.114737,

            "preservation_record_sha256":
                EXPECTED_RSC_FINAL_RECORD_SHA256,

            "preservation_file_manifest_sha256":
                EXPECTED_RSC_FILE_MANIFEST_SHA256,

            "portable_preservation_zip_sha256":
                EXPECTED_RSC_PRESERVATION_ZIP_SHA256,

            "rerun_rsc":
                False,
        },

    "tioman_state":
        {
            "scientific_role":
                "LIMITED_FTIR_LINKED_EXTERNAL_ANCHOR",

            "frozen_microscopy_population":
                35,

            "previously_established_unique_matches":
                17,

            "current_physical_resolution":
                "0/35 in old notebook mount",

            "canonical_source_required":
                True,

            "inference_performed":
                False,

            "mapping_audit_completed":
                False,

            "training_use_allowed":
                False,

            "crpa_use_allowed":
                False,

            "threshold_tuning_allowed":
                False,

            "calibration_allowed":
                False,

            "checkpoint_selection_allowed":
                False,

            "seed_selection_allowed":
                False,

            "discovery_audit_sha256":
                EXPECTED_TIOMAN_DISCOVERY_SHA256,

            "exact_asset_audit_sha256":
                EXPECTED_TIOMAN_EXACT_ASSET_SHA256,

            "source_provenance_audit_sha256":
                EXPECTED_TIOMAN_SOURCE_PROVENANCE_SHA256,
        },

    "frozen_public_real_manifest":
        {
            "source_path":
                str(
                    PUBLIC_REAL_MANIFEST
                ),

            "bundle_path":
                str(
                    (
                        FROZEN_MANIFEST_OUT
                        / PUBLIC_REAL_MANIFEST.name
                    ).relative_to(
                        BUNDLE_ROOT
                    )
                ),

            "sha256":
                public_real_manifest_sha,
        },

    "phase5b_code":
        {
            "original_path":
                str(
                    CODE_ROOT
                ),

            "bundle_path":
                str(
                    CODE_OUT.relative_to(
                        BUNDLE_ROOT
                    )
                ),

            "file_count":
                code_file_count,
        },

    "training_seed_provenance":
        seed_provenance_records,

    "tioman_audit_directories":
        tioman_audit_records,

    "environment":
        environment,
}


write_json(
    CONTINUATION_STATE_JSON,
    continuation_state,
)


print(
    "PASS  Continuation state written"
)


# ============================================================
# 15. FILE-BY-FILE BUNDLE MANIFEST
# ============================================================

bundle_files_before_manifest = all_files(
    BUNDLE_ROOT
)

file_entries = []

total_bytes = 0


for path in bundle_files_before_manifest:
    relative = path.relative_to(
        BUNDLE_ROOT
    )

    size_bytes = path.stat().st_size

    total_bytes += size_bytes

    file_entries.append(
        {
            "relative_path":
                relative.as_posix(),

            "size_bytes":
                size_bytes,

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


file_manifest = {
    "manifest_name":
        (
            "DASO-Net Continuation Checkpoint "
            "File Manifest v1"
        ),

    "status":
        "PASS",

    "bundle_root":
        str(
            BUNDLE_ROOT
        ),

    "file_count":
        len(
            file_entries
        ),

    "total_bytes":
        total_bytes,

    "files":
        file_entries,
}


write_json(
    FILE_MANIFEST_JSON,
    file_manifest,
)

file_manifest_sha = sha256_file(
    FILE_MANIFEST_JSON
)

FILE_MANIFEST_SHA_RECORD.write_text(
    (
        f"{file_manifest_sha}  "
        f"{FILE_MANIFEST_JSON.name}\n"
    ),
    encoding="utf-8",
)


print(
    "PASS  Continuation file manifest"
)


# ============================================================
# 16. FINAL CHECKPOINT FREEZE RECORD
# ============================================================

freeze = {
    "freeze_name":
        "DASO-Net Project Continuation Checkpoint Freeze v1",

    "status":
        "FROZEN_READY_FOR_DOWNLOAD_AND_FRESH_NOTEBOOK",

    "scientific_action":
        "CONTINUATION_CHECKPOINT_ONLY",

    "model_inference_performed":
        False,

    "scientific_metrics_computed":
        False,

    "threshold_tuning":
        False,

    "calibration":
        False,

    "checkpoint_selection":
        False,

    "seed_selection":
        False,

    "bundle_directory":
        str(
            BUNDLE_ROOT
        ),

    "continuation_state":
        {
            "path":
                str(
                    CONTINUATION_STATE_JSON
                ),

            "sha256":
                sha256_file(
                    CONTINUATION_STATE_JSON
                ),
        },

    "file_manifest":
        {
            "path":
                str(
                    FILE_MANIFEST_JSON
                ),

            "sha256":
                file_manifest_sha,
        },

    "terminal_checkpoints":
        checkpoint_records,

    "rsc_preservation_zip_sha256":
        EXPECTED_RSC_PRESERVATION_ZIP_SHA256,

    "tioman_current_readiness":
        (
            "START_FRESH_NOTEBOOK_AND_MOUNT_"
            "CANONICAL_TIOMAN_SOURCE"
        ),

    "readiness":
        "READY_TO_CREATE_PORTABLE_CONTINUATION_ZIP",
}


write_json(
    FINAL_FREEZE_JSON,
    freeze,
)

final_freeze_sha = sha256_file(
    FINAL_FREEZE_JSON
)

FINAL_FREEZE_SHA_RECORD.write_text(
    (
        f"{final_freeze_sha}  "
        f"{FINAL_FREEZE_JSON.name}\n"
    ),
    encoding="utf-8",
)


print(
    "PASS  Continuation checkpoint freeze record"
)


# ============================================================
# 17. CREATE DETERMINISTIC PORTABLE ZIP
# ============================================================

files_for_zip = all_files(
    CHECKPOINT_DIR
)


with zipfile.ZipFile(
    PORTABLE_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
) as zf:

    for file_path in files_for_zip:
        relative = file_path.relative_to(
            CHECKPOINT_DIR
        )

        archive_name = (
            Path(
                CHECKPOINT_DIR.name
            )
            / relative
        ).as_posix()

        data = file_path.read_bytes()

        info = zipfile.ZipInfo(
            filename=archive_name,
            date_time=(
                1980,
                1,
                1,
                0,
                0,
                0,
            ),
        )

        info.compress_type = (
            zipfile.ZIP_DEFLATED
        )

        info.create_system = 3

        info.external_attr = (
            0o100644 << 16
        )

        zf.writestr(
            info,
            data,
            compress_type=zipfile.ZIP_DEFLATED,
            compresslevel=9,
        )


portable_zip_sha = sha256_file(
    PORTABLE_ZIP
)

PORTABLE_ZIP_SHA_RECORD.write_text(
    (
        f"{portable_zip_sha}  "
        f"{PORTABLE_ZIP.name}\n"
    ),
    encoding="utf-8",
)


print(
    "PASS  Deterministic portable continuation ZIP"
)


# ============================================================
# 18. VERIFY ZIP BYTE CONTENT
# ============================================================

expected_names = [
    (
        Path(
            CHECKPOINT_DIR.name
        )
        / path.relative_to(
            CHECKPOINT_DIR
        )
    ).as_posix()
    for path in files_for_zip
]


with zipfile.ZipFile(
    PORTABLE_ZIP,
    "r",
) as zf:

    actual_names = zf.namelist()

    require(
        actual_names
        == expected_names,
        "Continuation ZIP inventory mismatch.",
    )


    for source_path, archive_name in zip(
        files_for_zip,
        actual_names,
    ):
        require(
            zf.read(
                archive_name
            )
            == source_path.read_bytes(),
            (
                "Continuation ZIP member byte mismatch:\n"
                f"{archive_name}"
            ),
        )


print(
    "PASS  Portable ZIP member byte identity"
)


# ============================================================
# 19. FINAL REPORT
# ============================================================

zip_size = PORTABLE_ZIP.stat().st_size


print()

print("=" * 80)

print(
    "DASO-NET CONTINUATION CHECKPOINT: READY"
)

print("=" * 80)

print()

print(
    "Status:"
)

print(
    "  FROZEN_READY_FOR_DOWNLOAD_AND_FRESH_NOTEBOOK"
)

print()

print(
    "Terminal checkpoints:"
)

for item in checkpoint_records:
    print(
        f"  Seed-{item['seed_index']} "
        f"s{item['seed']}"
    )

    print(
        f"    threshold: "
        f"{item['operating_threshold']:.2f}"
    )

    print(
        f"    SHA256: {item['sha256']}"
    )

print()

print(
    "RSC state:"
)

print(
    "  CLOSED / FINAL PRESERVED"
)

print()

print(
    "Tioman state:"
)

print(
    "  NO inference performed"
)

print(
    "  next step = mount canonical source in fresh notebook"
)

print()

print(
    "Bundle files:"
)

print(
    f"  {len(files_for_zip)}"
)

print()

print(
    "Portable ZIP:"
)

print(
    f"  {PORTABLE_ZIP}"
)

print()

print(
    "Portable ZIP size:"
)

print(
    f"  {zip_size} bytes "
    f"({zip_size / (1024 ** 2):.2f} MiB)"
)

print()

print(
    "Portable ZIP SHA256:"
)

print(
    f"  {portable_zip_sha}"
)

print()

print(
    "ZIP SHA record:"
)

print(
    f"  {PORTABLE_ZIP_SHA_RECORD}"
)

print()

print(
    "Checkpoint freeze JSON:"
)

print(
    f"  {FINAL_FREEZE_JSON}"
)

print()

print(
    "Checkpoint freeze SHA256:"
)

print(
    f"  {final_freeze_sha}"
)

print()

print(
    "NEXT READINESS:"
)

print(
    "  DOWNLOAD_CHECKPOINT_AND_START_FRESH_NOTEBOOK"
)

print("=" * 80)

DASO-Net — PROJECT CONTINUATION CHECKPOINT FREEZE v1

NO model inference
NO scientific metric computation/recomputation
NO threshold tuning
NO calibration
NO checkpoint/seed selection

PASS  All three terminal checkpoint identities
PASS  Final RSC preservation identities
PASS  Completed Tioman audit identities
PASS  Three terminal checkpoints copied byte-identically
PASS  Phase5B code tree copied (29 files)
PASS  Final-seed non-checkpoint provenance copied
PASS  Final frozen RSC package copied
PASS  Tioman audit history preserved
PASS  Frozen public-real source manifest copied
PASS  Continuation state written
PASS  Continuation file manifest
PASS  Continuation checkpoint freeze record
PASS  Deterministic portable continuation ZIP
PASS  Portable ZIP member byte identity

DASO-NET CONTINUATION CHECKPOINT: READY

Status:
  FROZEN_READY_FOR_DOWNLOAD_AND_FRESH_NOTEBOOK

Terminal checkpoints:
  Seed-1 s1782656963
    threshold: 0.30
    SHA256: e4c4b79ad4b8c6e18ba2b1e3d6cbe796c9cc301bd6fcc7c